# Description to Code Model

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel
import pandas as pd

/home/balaji/miniconda3/envs/Python/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv("/home/balaji/POC/POC/EasyOCR-ChatBot/diagnox.csv")
data.head()
data = data.dropna()
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62720 entries, 0 to 62719
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   code         62720 non-null  object
 1   label        62720 non-null  object
 2   description  62720 non-null  object
dtypes: object(3)
memory usage: 1.4+ MB
None


In [3]:
label_encoder = LabelEncoder()
data['code'] = label_encoder.fit_transform(data['code'])
print(data['code'])

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

0         2043
1         2043
2         2043
3         2043
4         2043
         ...  
62715    14584
62716    14584
62717    14584
62718    14584
62719    14584
Name: code, Length: 62720, dtype: int64


In [4]:
class DescriptionDataset(Dataset):
    def __init__(self, descriptions, labels, tokenizer, max_length=128):
        self.descriptions = descriptions
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, index):
        description = self.descriptions[index]
        label = self.labels[index]
        encoded = self.tokenizer(
            description,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }


In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    data['description'], data['code'], test_size=0.2, random_state=42
)
print(train_texts.shape , val_texts.shape ,train_labels.shape , val_labels.shape)

(50176,) (12544,) (50176,) (12544,)


In [6]:
train_dataset = DescriptionDataset(train_texts.tolist(), train_labels.tolist(), tokenizer)
val_dataset = DescriptionDataset(val_texts.tolist(), val_labels.tolist(), tokenizer)

print(train_dataset)
print(val_dataset)

In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(train_loader)
print(val_loader)

In [8]:
class CodePredictionModel(nn.Module):
    def __init__(self, num_labels):
        super(CodePredictionModel, self).__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        return self.fc(pooled_output)


In [9]:
num_labels = len(label_encoder.classes_)
model = CodePredictionModel(num_labels)
print(num_labels)

2025-02-05 20:59:25.224208: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-05 20:59:25.231380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738817965.239822    3375 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738817965.242350    3375 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-05 20:59:25.251473: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

14585


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
print(device)

cuda


In [11]:
epochs = 30
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_accuracy = correct / total
    print(f"Epoch {epoch + 1}, Loss: {train_loss / len(train_loader)}, Accuracy: {train_accuracy:.4f}")

Epoch 1, Loss: 6.8666348819221765, Accuracy: 0.2112
Epoch 2, Loss: 5.503486900001156, Accuracy: 0.3807
Epoch 3, Loss: 5.17549531952459, Accuracy: 0.4033
Epoch 4, Loss: 4.959977473987609, Accuracy: 0.4125
Epoch 5, Loss: 4.775890323732581, Accuracy: 0.4197
Epoch 6, Loss: 4.608661711824183, Accuracy: 0.4292
Epoch 7, Loss: 4.447500086560542, Accuracy: 0.4406
Epoch 8, Loss: 4.2917407297966434, Accuracy: 0.4535
Epoch 9, Loss: 4.138105063718193, Accuracy: 0.4689
Epoch 10, Loss: 3.990809347100404, Accuracy: 0.4844
Epoch 11, Loss: 3.8449628716524766, Accuracy: 0.4980
Epoch 12, Loss: 3.701497433745131, Accuracy: 0.5177
Epoch 13, Loss: 3.5621220727964324, Accuracy: 0.5341
Epoch 14, Loss: 3.4260943215720507, Accuracy: 0.5494
Epoch 15, Loss: 3.291352953077579, Accuracy: 0.5669
Epoch 16, Loss: 3.1645888262713444, Accuracy: 0.5837
Epoch 17, Loss: 3.037706446510797, Accuracy: 0.6007
Epoch 18, Loss: 2.915413351919578, Accuracy: 0.6170
Epoch 19, Loss: 2.793384435377559, Accuracy: 0.6339
Epoch 20, Loss: 

In [12]:
def evaluate_model_loss_and_accuracy(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Get the predicted class index with the highest probability
            _, predicted = torch.max(outputs, dim=1)

            # Count correct predictions
            correct_predictions += (predicted == labels).sum().item()
            total_predictions += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct_predictions / total_predictions * 100

    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.2f}%")

    return avg_loss, accuracy


In [13]:
val_loss, val_accuracy = evaluate_model_loss_and_accuracy(model, val_loader, criterion, device)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}%")

Average Loss: 3.1957
Validation Accuracy: 62.77%
Validation Loss: 3.195669280022991
Validation Accuracy: 62.77104591836735%


In [14]:
def predict_code(description, model, tokenizer, label_encoder, device, max_length=128):
    model.eval()
    with torch.no_grad():
        # Tokenize the input description
        encoded = tokenizer(
            description,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        # Get model output
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, dim=1)

        # Decode the predicted label to code
        predicted_code = label_encoder.inverse_transform(predicted.cpu().numpy())[0]
        return predicted_code

In [15]:
description = input()
predicted_code = predict_code(description, model, tokenizer, label_encoder, device)

print(f"Predicted Code: {predicted_code}")

Predicted Code: C801


In [16]:
torch.save(model.state_dict(), "desc_2_code_model.pth")
with open("label_encoder.pkl", "wb") as f:
    import pickle
    pickle.dump(label_encoder, f)

# Date Processing

In [3]:
import spacy 

nlp = spacy.load('en_core_web_sm')

/home/balaji/miniconda3/envs/Python/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
! python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.2 MB/s eta 0:00:00 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
nlp.component_names

['tok2vec',
 'tagger',
 'parser',
 'senter',
 'attribute_ruler',
 'lemmatizer',
 'ner']

In [5]:
nlp.get_pipe('ner').labels

('CARDINAL',
 'DATE',
 'EVENT',
 'FAC',
 'GPE',
 'LANGUAGE',
 'LAW',
 'LOC',
 'MONEY',
 'NORP',
 'ORDINAL',
 'ORG',
 'PERCENT',
 'PERSON',
 'PRODUCT',
 'QUANTITY',
 'TIME',
 'WORK_OF_ART')

In [17]:
text = "19/08/2004"

doc = nlp(text)
for i in doc.ents:
    if i.label_ == 'DATE':
        print(i.text , i.label_)

19/08/2004 DATE


In [20]:
! pip install spacy date_spacy

In [22]:
import spacy
from date_spacy import find_dates

nlp = spacy.blank('en')
nlp.add_pipe('find_dates')

doc = nlp("""Date of Birth : 29/08/2004
          Admitted Date : 7/02/2025""")

for ent in doc.ents:
    if ent.label_ == 'DATE':
        print(f'Text: {ent.text} -> Parsed Date: {ent._.date}')

Text: 29/08/2004 -> Parsed Date: 2004-08-29 00:00:00
Text: 7/02/2025 -> Parsed Date: 2025-07-02 00:00:00


In [30]:
import spacy
from date_spacy import find_dates

nlp = spacy.blank('en')
nlp.add_pipe('find_dates')

text = """Date of Birth  29/08/2004
          Admitted Date : 7/02/2025
          Passed Out : 14/02/2025"""

doc = nlp(text)

lines = text.split("\n")  # Split text into lines for context extraction

for line in lines:
    doc_line = nlp(line)  # Process each line separately
    for ent in doc_line.ents:
        if ent.label_ == 'DATE':
            label = line.split(":")[0].strip()  # Extract the label before the date
            print(f'Label: {label} -> Text: {ent.text} -> Parsed Date: {ent._.date}')


Label: Date of Birth  29/08/2004 -> Text: 29/08/2004 -> Parsed Date: 2004-08-29 00:00:00
Label: Admitted Date -> Text: 7/02/2025 -> Parsed Date: 2025-07-02 00:00:00
Label: Passed Out -> Text: 14/02/2025 -> Parsed Date: 2025-02-14 00:00:00


In [ ]:
import spacy
from date_spacy import find_dates
from datetime import datetime

nlp = spacy.blank('en')
nlp.add_pipe('find_dates')

text = """Date of Birth : 29/08/2004
          Admitted Date : 7/02/2025
          Discharge Date : 15/02/2025"""

doc = nlp(text)

date_dict = {}  # Store extracted dates with labels

# Extract Dates with Labels
for line in text.split("\n"):
    doc_line = nlp(line.strip())
    for ent in doc_line.ents:
        if ent.label_ == 'DATE':
            label = line.split(":")[0].strip()
            parsed_date = ent._.date
            if parsed_date:
                date_dict[label] = parsed_date

# Convert to datetime for analysis
date_dict = {label: datetime.strptime(str(date), "%Y-%m-%d") for label, date in date_dict.items()}

# Analyze the dates
earliest_date_label = min(date_dict, key=date_dict.get)
latest_date_label = max(date_dict, key=date_dict.get)

print("Extracted Dates with Labels:")
for label, date in date_dict.items():
    print(f"{label}: {date.strftime('%d-%m-%Y')}")

print("\nAnalysis:")
print(f"📌 Earliest Date: {earliest_date_label} -> {date_dict[earliest_date_label].strftime('%d-%m-%Y')}")
print(f"📌 Latest Date: {latest_date_label} -> {date_dict[latest_date_label].strftime('%d-%m-%Y')}")
print(f"📌 Days between Admission & Discharge: {(date_dict['Discharge Date'] - date_dict['Admitted Date']).days} days")


# Spacy Model for **Date**

In [ ]:
# Not used...

TRAIN_DATA = [
    # Legal context
    ("The contract is set to expire on January 15, 2025, and must be renewed before the due date to ensure uninterrupted service.", 
     {"entities": [(38, 54, "DATE")]}),

    ("According to clause 5, the agreement signed on 15th Jan 2025 shall remain in effect for three years unless terminated earlier.",
     {"entities": [(42, 55, "DATE")]}),

    ("The final court hearing is scheduled for 01/15/2025, and all parties must be present by 9 AM.", 
     {"entities": [(40, 50, "DATE")]}),

    ("As per our policy update, the effective date of the new rules is 2025-01-15, ensuring compliance with the latest regulations.", 
     {"entities": [(50, 60, "DATE")]}),

    ("This document must be submitted before the deadline of 15-01-25, failing which penalties may apply.", 
     {"entities": [(47, 55, "DATE")]}),

    # Medical reports
    ("The patient was diagnosed on March 5, 1990, and has been undergoing treatment since then.", 
     {"entities": [(29, 41, "DATE")]}),

    ("His last check-up was on Monday, April 3rd, and the next appointment is scheduled two months later.", 
     {"entities": [(23, 36, "DATE")]}),

    ("The blood test conducted on 10th Feb, 2024, confirmed the presence of the virus strain.", 
     {"entities": [(26, 39, "DATE")]}),

    ("His final surgery was scheduled on 2024-12-07 at 4 PM, and the follow-up consultation is after two weeks.", 
     {"entities": [(30, 40, "DATE")]}),

    ("The prescription issued on 02/10/25 recommends taking the medication for six months.", 
     {"entities": [(29, 37, "DATE")]}),

    # Business meetings
    ("Our annual general meeting is set for Dec 25, 2025, and will be held at the main office in New York.", 
     {"entities": [(33, 44, "DATE")]}),

    ("The next board discussion is planned for 05.04.2023, covering financial performance and future strategies.", 
     {"entities": [(37, 47, "DATE")]}),

    ("Tomorrow we have an important strategy meeting, where we will finalize next year's budget.", 
     {"entities": [(0, 8, "DATE")]}),

    ("The CEO announced that yesterday’s earnings report exceeded expectations, boosting investor confidence.", 
     {"entities": [(28, 37, "DATE")]}),

    ("Can we schedule a follow-up on next Monday? The client is available in the morning.", 
     {"entities": [(27, 37, "DATE")]}),

    # Historical events
    ("On July 4, 1776, the Declaration of Independence was signed, marking the birth of a new nation.", 
     {"entities": [(3, 15, "DATE")]}),

    ("The Treaty of Versailles was signed on June 28, 1919, officially ending World War I.", 
     {"entities": [(36, 49, "DATE")]}),

    ("The Wright brothers' first flight took place on Dec. 17, 1903, marking the beginning of modern aviation.", 
     {"entities": [(39, 51, "DATE")]}),

    ("The moon landing on July 20, 1969, was one of the greatest achievements in space exploration.", 
     {"entities": [(20, 32, "DATE")]}),

    ("The Berlin Wall fell on November 9, 1989, symbolizing the end of the Cold War.", 
     {"entities": [(23, 38, "DATE")]}),

    # News articles
    ("The recent cyberattack that occurred on March 10, 2024, has affected millions of users worldwide.", 
     {"entities": [(37, 50, "DATE")]}),

    ("A powerful earthquake struck California on 12th June 2023, causing significant damage.", 
     {"entities": [(40, 54, "DATE")]}),

    ("The latest GDP report, released on 2024-08-30, indicates strong economic growth.", 
     {"entities": [(37, 47, "DATE")]}),

    ("Stock markets saw a decline yesterday following global economic uncertainty.", 
     {"entities": [(25, 34, "DATE")]}),

    ("The next presidential debate is scheduled for next Friday, with candidates set to discuss key issues.", 
     {"entities": [(36, 47, "DATE")]}),

    # Travel & tourism
    ("Our family vacation starts on August 1, 2025, and we plan to visit multiple countries.", 
     {"entities": [(28, 41, "DATE")]}),

    ("The flight is booked for 10/12/2024, and we need to reach the airport by 6 AM.", 
     {"entities": [(26, 36, "DATE")]}),

    ("We arrived in Paris on 03-07-2023 and explored the city’s landmarks.", 
     {"entities": [(21, 31, "DATE")]}),

    ("The hotel reservation was confirmed for Sept 15, 2024, with a sea-facing room.", 
     {"entities": [(37, 50, "DATE")]}),

    ("His passport expires on 14-05-2026, so he must renew it before traveling.", 
     {"entities": [(24, 34, "DATE")]}),

    # Informal & mixed formats
    ("I'll see you on the 2nd of March, '24, for our reunion!", 
     {"entities": [(17, 32, "DATE")]}),

    ("Can we reschedule for 8.11.23? I have a conflict that day.", 
     {"entities": [(24, 31, "DATE")]}),

    ("Her birthday is on 09/05/97, and we plan to throw a big party.", 
     {"entities": [(19, 27, "DATE")]}),

    ("I last saw him on 2022/04/16 in Los Angeles.", 
     {"entities": [(15, 25, "DATE")]}),

    ("The new album is dropping on 5-September-2025, and fans are excited.", 
     {"entities": [(30, 45, "DATE")]}),
]


In [20]:
TRAIN_DATA = [('\r\n1. On January 15, 2025, we launched our new product line. The initial response was overwhelming, \r\n   with over 10,000 units sold within the first week. By 15th Jan 2025, we had already received \r\n   pre-orders for the next batch. The marketing campaign that started on 01/15/2025 played a \r\n   crucial role in this success. Our internal meeting scheduled for 2025-01-15 helped finalize \r\n   the strategy. The first feedback session on 15-01-25 provided valuable insights for improvement.\r\n\r\n2. The company was founded on March 5, 1990, with a vision to revolutionize the tech industry. \r\n   Initial operations started on 05/03/1990, and within a year, our revenue skyrocketed. \r\n   The team celebrated its first milestone on 1990-03-05, marking the beginning of an era. \r\n   Our success story was featured in major newspapers on 5th March 1990. \r\n   Employees remember the launch event held on 03.05.1990, as it was a defining moment.\r\n\r\n3. The annual conference is set to take place on Monday, April 3rd. The event was originally \r\n   planned for 03rd Apr 2025 but had to be rescheduled. Our previous conference was held \r\n   on 4/3/2024, where we discussed market trends. The latest report, dated 2025-04-03, \r\n   reveals a significant growth in our industry. In a retrospective session on 03-04-25, \r\n   we analyzed past performances.\r\n\r\n4. Last Friday, a crucial decision was made regarding the upcoming merger. On 10th Feb, 2024, \r\n   both companies signed an agreement. The final discussion took place on 02/10/2024, with \r\n   legal teams from both sides present. On 2024-02-10, the CEO made an official announcement \r\n   to stakeholders. The merger process officially began on 10-02-24.\r\n\r\n5. The championship game will be held on Dec 25, 2025, and fans are eagerly waiting. Tickets \r\n   went on sale on 12/25/2025, and within hours, they were sold out. The last time we witnessed \r\n   such enthusiasm was on 25-12-25, during the previous championship. Reports from 2025-12-25 \r\n   suggest record-breaking viewership. The media covered the event extensively on 25th Dec, 2025.\r\n\r\n6. The new policy changes will take effect from 05.04.2023. Employees received official \r\n   notifications on 5th April 2023. A press release on 04/05/2023 explained the reasons \r\n   behind the updates. Stakeholders discussed the implications in a meeting on 2023-04-05. \r\n   The decision-making process began on 05-04-23 and took several months to finalize.\r\n\r\n7. The doctor scheduled my next appointment for 07.08.2023. My previous visit was on \r\n   08/07/2023, where I had a detailed health check-up. The hospital records show that \r\n   my last consultation was on 2023-08-07. My insurance coverage was updated on \r\n   7th August 2023, ensuring that I receive the necessary medical care.\r\n\r\n8. The project deadline has been extended to 2026-09-15. Initial projections expected \r\n   completion by 15th Sept 2026, but unforeseen challenges led to delays. An internal \r\n   report published on 09/15/2026 highlights the revised schedule. The first draft \r\n   was submitted on 15-09-26, with additional reviews scheduled on September 15, 2026.\r\n\r\n9. The next quarterly review will be on 30/11/2025. The last review took place on \r\n   11/30/2025, where performance metrics were analyzed. A report from 2025-11-30 \r\n   suggested improvements for future strategies. Employees submitted feedback on \r\n   30th Nov 2025, highlighting key operational challenges.\r\n\r\n10. The museum recently acquired an artifact dating back to 1865. The discovery was \r\n    made on 12th Jan 1865, in an abandoned excavation site. Historical records dated \r\n    01/12/1865 describe similar findings in the region. A report published on \r\n    1865-01-12 detailed the significance of these artifacts.\r\n\r\n', {'entities': [[8, 24, 'DATE'], [158, 171, 'DATE'], [272, 282, 'DATE'], [362, 372, 'DATE'], [438, 446, 'DATE'], [524, 537, 'DATE'], [624, 634, 'DATE'], [728, 738, 'DATE'], [832, 847, 'DATE'], [897, 907, 'DATE'], [990, 1008, 'DATE'], [1051, 1064, 'DATE'], [1133, 1141, 'DATE'], [1202, 1212, 'DATE'], [1295, 1303, 'DATE'], [1422, 1436, 'DATE'], [1514, 1524, 'DATE'], [1576, 1586, 'DATE'], [1687, 1696, 'DATE'], [1741, 1753, 'DATE'], [1814, 1824, 'DATE'], [1919, 1927, 'DATE'], [1976, 1986, 'DATE'], [2071, 2086, 'DATE'], [2138, 2149, 'DATE'], [2200, 2215, 'DATE'], [2235, 2245, 'DATE'], [2349, 2360, 'DATE'], [2403, 2411, 'DATE'], [2500, 2511, 'DATE'], [2542, 2552, 'DATE'], [2658, 2669, 'DATE'], [2712, 2727, 'DATE'], [2829, 2840, 'DATE'], [2889, 2903, 'DATE'], [2983, 2993, 'DATE'], [3112, 3131, 'DATE'], [3175, 3186, 'DATE'], [3222, 3232, 'DATE'], [3289, 3299, 'DATE'], [3388, 3401, 'DATE'], [3545, 3558, 'DATE'], [3624, 3634, 'DATE'], [3704, 3714, 'DATE']]})]

In [21]:
import spacy
from spacy.tokens import DocBin

TRAIN_DATA = TRAIN_DATA


# Load blank model
nlp = spacy.blank("en")
db = DocBin()

for text, annotations in TRAIN_DATA:
    doc = nlp.make_doc(text)
    ents = []
    for start, end, label in annotations["entities"]:
        span = doc.char_span(start, end, label=label)
        if span:
            ents.append(span)
    doc.ents = ents
    db.add(doc)

# Save as train and dev dataset
db.to_disk("./train.spacy")
db.to_disk("./dev.spacy")

print("Training data saved as .spacy format!")


Training data saved as .spacy format!


In [14]:
! python -m spacy init fill-config base_config.cfg config.cfg

✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [15]:
! python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy --gpu-id 0


ℹ Saving to output directory: output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
2025-02-10 23:04:36.095004: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-10 23:04:36.102097: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739257476.109917   40962 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739257476.112256   40962 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-10 2

In [12]:
import spacy

nlp = spacy.load("/home/balaji/POC/POC/EasyOCR-ChatBot/output/model-best")
print(nlp.pipe_names) 


['transformer', 'ner']


/home/balaji/miniconda3/envs/Python/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self._model.load_sta

In [36]:
import spacy

nlp = spacy.load("/home/balaji/POC/POC/EasyOCR-ChatBot/output/model-best")

test_text = """
Illinois Eye Center; S.C. 8921 N Wood Sage Rd Peoria, IL 61615-7822 309-243-2400 Examination Name: Melvin Little Exam Date: 4/8/2024 Acct #: 151545 Date of Birth: 11/11/1941 Primary Care Physician: Kowalska, Aneta MD Complaint 82 year old male presents for existing condition, retinal edema Location: right eye Duration: longstanding Timing: constant Quality: stable since last visit Patient denies presence of: pain/discomfort, vision changes 82 year old male presents for existing condition, macular pucker Location: left eye Duration: longstanding Timing: constant Quality: stable since last visit Review Of Systems Constitution Negative Cardiovascular Negative Ears, Nose, Mouth, Throat Negative Respiratory Negative Gastrointestinal Negative Genitourinary Negative Musculoskeletal Negative Integumentary Negative Neurological Negative Psychiatric Negative Endocrine Negative Hematologic/Lymphatic Negative Allergicllmmunologic Negative Mental Assessment Time Place Person oriented to time, place, person Mood Affect mood and affect appropriate Surgical History 1/1/1900 Stomach (CA) 1/1/1900 Hernia 9/12/2017 No previous ocular surgery to EHR Both Eyes 2/4/2019 CE IOL Standard Distance Left Eye Surgeon: Hu, Edward MD 2/18/2019 CE IOL Standard Distance Right Eye Surgeon: Edward MD Patient: Little, Melvin Acct: 151545 DOS: 4/8/2024 Print Date: July 16, 2024 Page of 5 prior Hu,
"""

doc = nlp(test_text)

print("\n🔍 Extracted Dates:")
for ent in doc.ents:
    if ent.label_ == "DATE" and "/" in ent.text:
        print(f"Date: {ent.text}")


/home/balaji/miniconda3/envs/Python/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self._model.load_sta


🔍 Extracted Dates:
Date: 4/8/2024
Date: 11/11/1941
Date: 1/1/1900
Date: 1/1/1900
Date: 9/12/2017
Date: 2/4/2019
Date: 2/18/2019
Date: 4/8/2024


In [45]:
import spacy

# Load trained spaCy model
nlp = spacy.load("/home/balaji/POC/POC/EasyOCR-ChatBot/output/model-best")

# Sample text with different numeric values
test_text = """
Balaji : 08/27/01
"""

# Process text with spaCy model
doc = nlp(test_text)

def is_valid_date(date_text):
    parts = date_text.split("/")
    if len(parts) == 3:  # Must be in MM/DD/YYYY or DD/MM/YYYY format
        month, day, year = parts
        if year.isdigit() and len(year) == 4 or len(year) == 2:  # Ensure year is four digits
            if month.isdigit() and day.isdigit():
                month, day = int(month), int(day)
                return 1 <= month <= 12 and 1 <= day <= 31  # Check valid range
    return False

# Extract valid dates
print("\n🔍 Extracted Dates:")
for ent in doc.ents:
    if ent.label_ == "DATE" and "/" in ent.text and is_valid_date(ent.text):
        print(f"Date: {ent.text}")



🔍 Extracted Dates:
Date: 08/27/01


In [25]:
import spacy

# Load custom spaCy model
nlp = spacy.load("/home/balaji/POC/POC/EasyOCR-ChatBot/output/model-best")

def extract_dates_and_diagnoses(text):
    """Extract dates and diagnoses using spaCy"""
    doc = nlp(text)
    
    extracted_data = {"dates": {}, "diagnoses": []}

    for ent in doc.ents:
        if ent.label_ in ["DATE_OF_BIRTH", "ADMISSION_DATE", "VISIT_DATE"]:
            extracted_data["dates"][ent.label_] = ent.text  # Store date with label
        elif ent.label_ == "DIAGNOSIS":
            extracted_data["diagnoses"].append({"code": ent.text.split()[0], "description": " ".join(ent.text.split()[1:])})

    return extracted_data

# Example extracted text
text = """
Capsule Normal Normal Anterior Vitreous Normal Normal Posterior Exam Exam Method Examination of retina. 
...
Diagnosis And Plan H35.81 Retinal edema OD 
Assessment: Examination revealed retinal edema, Plan: AIl testing discussed with patient and the family
...
H35.372 Puckering of macula, left eye OS 
...
H43.813 Vitreous degeneration, bilateral OU 
...
Patient: Little, Melvin 
Acct: 151545 
DOS: 8/17/2023 
Print Date: July 16, 2024 
"""

# Process the text
result = extract_dates_and_diagnoses(text)

# Print extracted results
print("Extracted Dates:")
for label, date in result["dates"].items():
    print(f"{label}: {date}")

print("\nExtracted Diagnoses:")
for diagnosis in result["diagnoses"]:
    print(f"Code: {diagnosis['code']}, Description: {diagnosis['description']}")


Extracted Dates:

Extracted Diagnoses:


In [24]:
import spacy
import re

# Load custom spaCy model
nlp = spacy.load("/home/balaji/POC/POC/EasyOCR-ChatBot/output/model-best")

def extract_dates(text):
    """Extract dates and their labels from text"""
    doc = nlp(text)
    dates = {}

    for ent in doc.ents:
        if ent.label_ in ["DATE_OF_BIRTH", "ADMISSION_DATE", "VISIT_DATE"]:
            dates[ent.label_] = ent.text  # Store date with label

    return dates

def extract_diagnoses(text):
    """Extract diagnoses and their codes from text"""
    diagnoses = []
    
    # Regex for ICD-10 codes (e.g., H35.81)
    icd_pattern = r'([A-Z]\d{2}\.\d{1,2})\s+([\w\s,]+)'

    matches = re.findall(icd_pattern, text)
    for code, description in matches:
        diagnoses.append({"code": code, "description": description.strip()})

    return diagnoses

def process_text(text):
    """Process extracted text to get structured data"""
    dates = extract_dates(text)
    diagnoses = extract_diagnoses(text)

    return {"dates": dates, "diagnoses": diagnoses}

# Example extracted text (your data will come from the PDF)
text = """
Capsule Normal Normal Anterior Vitreous Normal Normal Posterior Exam Exam Method Examination of retina. 
...
Diagnosis And Plan H35.81 Retinal edema OD 
Assessment: Examination revealed retinal edema, Plan: AIl testing discussed with patient and the family
...
H35.372 Puckering of macula, left eye OS 
...
H43.813 Vitreous degeneration, bilateral OU 
...
Patient: Little, Melvin 
Acct: 151545 
DOS: 8/17/2023 
Print Date: July 16, 2024 
Page 4 of 5
"""

result = process_text(text)

# Print extracted results
print("Extracted Dates:")
for label, date in result["dates"].items():
    print(f"{label}: {date}")

print("\nExtracted Diagnoses:")
for diagnosis in result["diagnoses"]:
    print(f"Code: {diagnosis['code']}, Description: {diagnosis['description']}")


Extracted Dates:

Extracted Diagnoses:
Code: H35.81, Description: Retinal edema OD 
Assessment


In [31]:
import spacy

nlp = spacy.load('en_core_web_sm')
nlp.get_pipe('ner').labels

('CARDINAL',
 'DATE',
 'EVENT',
 'FAC',
 'GPE',
 'LANGUAGE',
 'LAW',
 'LOC',
 'MONEY',
 'NORP',
 'ORDINAL',
 'ORG',
 'PERCENT',
 'PERSON',
 'PRODUCT',
 'QUANTITY',
 'TIME',
 'WORK_OF_ART')

In [34]:
txt = """ 
4/8/2024 Acct #: 151545 Date of Birth: 11/11/1941 Primary Care Physician: Kowalska, Aneta MD Complaint 82 year old male presents for existing condition, retinal edema Location: right eye Duration: longstanding Timing: constant Quality: stable since last visit Patient denies presence of: pain/discomfort, vision changes 82 year old male presents for existing condition, macular pucker Location: left eye Duration: longstanding Timing: constant Quality: stable since last visit Review Of Systems Constitution Negative Cardiovascular Negative Ears, Nose, Mouth, Throat Negative Respiratory Negative Gastrointestinal Negative Genitourinary Negative Musculoskeletal Negative Integumentary Negative Neurological Negative Psychiatric Negative Endocrine Negative Hematologic/Lymphatic Negative Allergicllmmunologic Negative Mental Assessment Time Place Person oriented to time, place, person Mood Affect mood and affect appropriate Surgical History 1/1/1900 Stomach (CA) 1/1/1900 Hernia 9/12/2017 No previous ocular surgery to EHR Both Eyes 2/4/2019 CE IOL Standard Distance Left Eye Surgeon: Hu, Edward MD 2/18/2019 CE IOL Standard Distance Right Eye Surgeon: Edward MD Patient: Little, Melvin Acct: 151545 DOS: 4/8/2024 Print Date: July 16, 2024 Page of 5 prior Hu,
"""

doc = nlp(txt)

# Print detected entities
for ent in doc.ents:
    if ent.label_ == "DATE":
        print(f"Detected Date: {ent.text}")

Detected Date: 82 year old
Detected Date: 82 year old
Detected Date: 151545
Detected Date: July 16, 2024 Page


# **Date Classification**

In [46]:
import re
from datetime import datetime

# Sample extracted text
text = """Conclusion Normal left and right ventricular size and function...  
Mount Sinai Health System  
Report Date: 07/29/2024  
Patient: Richards, Paulette  
DOB: 09/22/1953  
MRN: 3808523  
Gender: Female  
Admit Date/Date of Service: 04/04/2023  
Discharge Date: 04/04/2023  
HAR: Page 4 of 5"""

# Step 1: Extract Dates Using Regex
date_pattern = r"\b(\d{2}/\d{2}/\d{4})\b"
date_matches = re.findall(date_pattern, text)

# Step 2: Initialize Labels
classified_dates = {
    "Date of Birth": None,
    "Admit Date": None,
    "Discharge Date": None,
    "Report Date": None
}

# Step 3: Assign Dates Based on Context
for date in date_matches:
    if "DOB" in text.split(date)[0][-15:]:  # Look for "DOB" near the date
        classified_dates["Date of Birth"] = date
    elif "Admit Date" in text.split(date)[0][-25:] or "Date of Service" in text.split(date)[0][-25:]:
        classified_dates["Admit Date"] = date
    elif "Discharge Date" in text.split(date)[0][-25:]:
        classified_dates["Discharge Date"] = date
    elif "Report Date" in text.split(date)[0][-25:]:
        classified_dates["Report Date"] = date

# Print results
print(classified_dates)


{'Date of Birth': '09/22/1953', 'Admit Date': '04/04/2023', 'Discharge Date': None, 'Report Date': '07/29/2024'}


# Pre-Trained Spacy Model

In [34]:
! pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 47.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.2.0
    Uninstalling fsspec-2025.2.0:
      Successfully uninstalled fsspec-2025.2.0


In [ ]:
from datasets import load_dataset

# Load the JSON dataset
dataset = load_dataset("json", data_files={"train": "/home/balaji/POC/POC/EasyOCR-ChatBot/medical_ner.json"})

# Define label mappings
label_list = ["O", "B-DISEASE_DISORDER", "I-DISEASE_DISORDER"]
num_labels = len(label_list)
print(num_labels)


In [38]:
import json

with open("medical_ner.json", "r") as f:
    data = json.load(f)

print(json.dumps(data[:5], indent=2))  # Print the first 5 entries for inspection


[
  {
    "tokens": [
      "Diabetes",
      "mellitus",
      "type",
      "2",
      "with",
      "complications"
    ],
    "ner_tags": [
      "B-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER"
    ]
  },
  {
    "tokens": [
      "Diabetes",
      "type",
      "2",
      "with",
      "circulation",
      "disorder"
    ],
    "ner_tags": [
      "B-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER"
    ]
  },
  {
    "tokens": [
      "Peripheral",
      "circulatory",
      "disorder",
      "associated",
      "with",
      "type",
      "2",
      "diabetes",
      "mellitus"
    ],
    "ner_tags": [
      "B-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISORDER",
      "I-DISEASE_DISOR

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "Clinical-AI-Apollo/Medical-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)


model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=3,  
    ignore_mismatched_sizes=True
)
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_list.index(label[word_idx]))
            else:
                label_ids.append(label_list.index(label[word_idx]))
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply tokenization
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./finetuned-medical-ner",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Increase if you have a powerful GPU
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    fp16=True,  # Enable mixed precision training for faster performance
)
import torch # Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
from datasets import DatasetDict

# Define split ratio (e.g., 90% train, 10% validation)
train_test_split = dataset["train"].train_test_split(test_size=0.1)

# Rename the test split as 'valid'
dataset = DatasetDict({
    "train": train_test_split["train"],
    "valid": train_test_split["test"],
})

# Tokenize again after splitting
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
from transformers import Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],  # Pass validation dataset
    data_collator=data_collator,
)

trainer.train()

# Spacy model for **Person Name**

In [1]:
import json
 
with open('/home/balaji/POC/POC/EasyOCR-ChatBot/results (1).json', 'r') as f:
    file = json.load(f)
 
file1 = [item['text'] for item in file]  # Extract all text values
 
for i in range(0, len(file1), 2):  # Iterate in steps of 2
    data = file1[i:i+2]  # Get two texts at a time
    filename = f'/home/balaji/POC/POC/EasyOCR-ChatBot/Person_date_output{i//2 + 1}.txt'  # Create file names like output_1.txt, output_2.txt, etc.
   
    with open(filename, 'w') as w:
        w.write(','.join(data))  # Write two texts in the file, separated by a newline
 
print("Files have been saved successfully!")

Files have been saved successfully!


run this command to move every txt file to folder Person_data_output.

! find . -type f -name "Person_date_output*.txt" -exec mv -t Person_date_output {} +


In [3]:
import os
import random
 
def insert_descriptions_randomly(input_folder, output_folder, insertions=100):
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
 
    # Get all .txt files from the input folder
    txt_files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]
 
    for index, file_name in enumerate(txt_files):
        input_path = os.path.join(input_folder, file_name)
        output_path = os.path.join(output_folder, f"ner_output_{index+1}.txt")
 
        # Read the file content
        with open(input_path, 'r', encoding='utf-8') as file:
            text = file.read()
 
        # Split text into characters to allow inserting {{ descriptions }} at random positions
        text_list = list(text)
 
        # Get random positions to insert
        positions = sorted(random.sample(range(len(text_list)), insertions), reverse=True)
 
        # Insert {descriptions} at selected positions
        for pos in positions:
            text_list.insert(pos, " {descriptions} ")
 
        # Convert list back to string
        modified_text = "".join(text_list)
 
        # Write to output file
        with open(output_path, 'w', encoding='utf-8') as file:
            file.write(modified_text)
 
        print(f"Processed '{file_name}' -> Saved as '{output_path}' with {insertions} insertions.")
 
# Usage: Change 'input_texts' & 'output_texts' to actual folder paths
insert_descriptions_randomly("/home/balaji/POC/POC/EasyOCR-ChatBot/Person_date_output", "/home/balaji/POC/POC/EasyOCR-ChatBot/PDO")

Processed 'Person_date_output841.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-ChatBot/PDO/ner_output_1.txt' with 100 insertions.
Processed 'Person_date_output110.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-ChatBot/PDO/ner_output_2.txt' with 100 insertions.
Processed 'Person_date_output1017.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-ChatBot/PDO/ner_output_3.txt' with 100 insertions.
Processed 'Person_date_output567.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-ChatBot/PDO/ner_output_4.txt' with 100 insertions.
Processed 'Person_date_output1154.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-ChatBot/PDO/ner_output_5.txt' with 100 insertions.
Processed 'Person_date_output513.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-ChatBot/PDO/ner_output_6.txt' with 100 insertions.
Processed 'Person_date_output441.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-ChatBot/PDO/ner_output_7.txt' with 100 insertions.
Processed 'Person_date_output155.txt' -> Saved as '/home/balaji/POC/POC/EasyOCR-C

In [4]:
import os
import re
import json
import pandas as pd
 
def process_text_files(input_folder, output_folder, csv_file, descriptions_per_file=100):
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
 
    # Load descriptions from CSV
    df = pd.read_csv(csv_file)  
    descriptions = df['Name'].str.strip().tolist()  # Clean descriptions
 
    # Get all .txt files from input folder
    txt_files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]
 
    description_index = 0  # To track used descriptions
 
    for file_index, txt_file in enumerate(txt_files):
        input_path = os.path.join(input_folder, txt_file)
        output_json_path = os.path.join(output_folder, f"data_{file_index+1}.json")
 
        # Read text file
        with open(input_path, 'r', encoding='utf-8') as file:
            text = file.read()
 
        # Extract the next batch of 100 descriptions
        batch_descriptions = descriptions[description_index:description_index + descriptions_per_file]
        description_index += descriptions_per_file  # Move index for the next file
 
        if not batch_descriptions:
            print(f"⚠️ No more descriptions left for {txt_file}. Skipping...")
            continue
 
        # Process text for replacement
        entities = []
        new_text = text  # Work on a copy
        pattern = r"\{\s*descriptions\s*\}"
 
        for desc in batch_descriptions:
            match = re.search(pattern, new_text)
            if match:
                start = match.start() + 1  # Inside `{`
                end = match.end() + 1      # Before `}`
 
                # Replace `{ descriptions }` while keeping `{}` intact
                new_text = new_text[:start] + " " + desc + " " + new_text[end:]
 
                # Store entity annotation
                entities.append([start + 1, start + 2 + len(desc), "PERSON"])
 
            else:
                break  # No more placeholders
 
        # Prepare JSON output
        output = [
            new_text,
            {"entities": entities}
        ]
 
        # Save as JSON file
        with open(output_json_path, "w", encoding="utf-8") as json_file:
            json.dump(output, json_file, indent=4, ensure_ascii=False)
 
        print(f"✅ Processed '{txt_file}' -> Saved '{output_json_path}' with {len(entities)} entities.")
 
# Usage Example
process_text_files("/home/balaji/POC/POC/EasyOCR-ChatBot/PDO", "/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated", "/home/balaji/POC/POC/EasyOCR-ChatBot/name_gender_dataset.csv")

✅ Processed 'ner_output_1310.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_1.json' with 100 entities.
✅ Processed 'ner_output_417.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_2.json' with 100 entities.
✅ Processed 'ner_output_1300.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_3.json' with 100 entities.
✅ Processed 'ner_output_201.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_4.json' with 100 entities.
✅ Processed 'ner_output_846.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_5.json' with 100 entities.
✅ Processed 'ner_output_127.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_6.json' with 100 entities.
✅ Processed 'ner_output_247.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_7.json' with 100 entities.
✅ Processed 'ner_output_585.txt' -> Saved '/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/data_8.json' with 100 entities.
✅ Processed 'ner_outpu

In [6]:
import os
import re
 
# Define input and output folder paths
input_folder = "/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated"
output_folder = "/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/Modified/"
 
# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)
 
# Get all JSON files in the input folder
json_files = [f for f in os.listdir(input_folder) if f.endswith(".json")]
 
for file_name in json_files:
    input_path = os.path.join(input_folder, file_name)
    output_path = os.path.join(output_folder, file_name)
 
    # Read JSON file as a string
    with open(input_path, "r", encoding="utf-8") as file:
        json_str = file.read()
 
    # Replace the first `[` with `(`
    json_str = re.sub(r'^\s*\[', '(', json_str, count=1)
 
    # Add `(` before the last `]`
    json_str = re.sub(r'\]\s*$', ')]', json_str, count=1)
   
    # Save the modified JSON back to the output folder
    with open(output_path, "w", encoding="utf-8") as file:
        file.write(json_str)
 
    print(f"✅ Processed: {file_name}")
 
print("🎉 All JSON files have been successfully modified!")

✅ Processed: data_1162.json
✅ Processed: data_1231.json
✅ Processed: data_320.json
✅ Processed: data_737.json
✅ Processed: data_908.json
✅ Processed: data_50.json
✅ Processed: data_735.json
✅ Processed: data_365.json
✅ Processed: data_29.json
✅ Processed: data_141.json
✅ Processed: data_388.json
✅ Processed: data_538.json
✅ Processed: data_522.json
✅ Processed: data_938.json
✅ Processed: data_120.json
✅ Processed: data_831.json
✅ Processed: data_478.json
✅ Processed: data_810.json
✅ Processed: data_394.json
✅ Processed: data_850.json
✅ Processed: data_911.json
✅ Processed: data_590.json
✅ Processed: data_982.json
✅ Processed: data_277.json
✅ Processed: data_870.json
✅ Processed: data_1359.json
✅ Processed: data_872.json
✅ Processed: data_1190.json
✅ Processed: data_1067.json
✅ Processed: data_512.json
✅ Processed: data_1200.json
✅ Processed: data_622.json
✅ Processed: data_24.json
✅ Processed: data_1125.json
✅ Processed: data_36.json
✅ Processed: data_1248.json
✅ Processed: data_1027.j

In [7]:
import os
 
# Path to the folder containing JSON files
json_folder = "/home/balaji/POC/POC/EasyOCR-ChatBot/Annotated/Modified"
 
# Process files from data_142.json to data_628.json
for i in range(2, 1398):  # Loop from 142 to 628
    file_name = f"data_{i}.json"
    file_path = os.path.join(json_folder, file_name)
 
    # Check if file exists before processing
    if not os.path.exists(file_path):
        print(f"Skipping {file_name}: File not found")
        continue
 
    # Read the content of the file
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read().strip()
 
    # Remove last "]" and replace with ","
    if content.endswith("]"):
        content = content[:-1] + ","
 
    # Remove leading "," at the beginning if it exists
    if content.startswith(","):
        content = content[1:]
 
    # Save the modified content back to the file
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(content)
 
    print(f"Processed: {file_name}")
 
print("All JSON files have been modified successfully!")
 

Processed: data_2.json
Processed: data_3.json
Processed: data_4.json
Processed: data_5.json
Processed: data_6.json
Processed: data_7.json
Processed: data_8.json
Processed: data_9.json
Processed: data_10.json
Processed: data_11.json
Processed: data_12.json
Processed: data_13.json
Processed: data_14.json
Processed: data_15.json
Processed: data_16.json
Processed: data_17.json
Processed: data_18.json
Processed: data_19.json
Processed: data_20.json
Processed: data_21.json
Processed: data_22.json
Processed: data_23.json
Processed: data_24.json
Processed: data_25.json
Processed: data_26.json
Processed: data_27.json
Processed: data_28.json
Processed: data_29.json
Processed: data_30.json
Processed: data_31.json
Processed: data_32.json
Processed: data_33.json
Processed: data_34.json
Processed: data_35.json
Processed: data_36.json
Processed: data_37.json
Processed: data_38.json
Processed: data_39.json
Processed: data_40.json
Processed: data_41.json
Processed: data_42.json
Processed: data_43.json


In [57]:
TRAIN_DATA = [(
    "vumc ad { James ult o { John ne hundred oaks white, tyrone 71 { Robert 9 thompson lane, nashville mrn: 047717 { Michael 361, dob: 7/27/1969, legal sex { William : m nashville tn 37204 visit d { Mary ate: 10/17/2023 10/17/2023 - office visit in vanderbilt one hundred oaks primary care north { David  (continued) flows { Joseph heets (continued) { Richard  p { Charles ain questions row name 10/17/23 1421 10/17/23 1424 pai { Thomas n assessment is the  { Christopher patient yes ! -dd at 10/17/23 yes ! -dd at 10/17/23 hav { Daniel ing pain 142 { Matthew 2 1426 today? pain score 9 -dd at 10 { Elizabeth /17/23 1422 9 { Patricia  -dd  { Jennifer at 10/17/23 1426 pain loc back -d { Anthony d at 10/17/23 back patient stat { George es 1422 { Linda  that back pa { Barbara in is due to a car accident -dd at 10/17/23 1426 per { Donald sonal safety office visit fro { Paul m 10/17/2023 in vanderbilt one hundred oaks prim { Mark ary care north wit { Andrew h hinton, row n { Steven ame { Kenneth  timo { Edward thy john, md pe { Joshua rsonal safet { Margaret y does anyone { Brian  no -dd  { Kevin at 10/17/23 1424 neglect, hurt, or threaten the patient? phq-2 to 9 depression scale row name { Jessica  10/17/23  { Sarah 1424 over the p { Susan ast 2 weeks, how often have you been bothere { Timothy d by a { Dorothy ny of the following proble { Jason ms? little i { Ronald nterest or n { Helen ot at all -dd at pleasure in doing 10/17/23 1424 things feeling down, not  { Ryan at all -dd at depressed, or 10/17/23 1424 hopeless phq 2 score 0 -dd at 10/17/23 1424 mood interview did ph { Jeffrey q-2 { Karen  1 -dd at 10/17/23 1424 sc { Nancy reen negative? vita { Betty l signs row name 10/17/23  { Lisa 1426 other hr/p { Jacob ulse 9 { Nicholas 5 -dd at 10/17/23 1 { Ashley 427 user key (r) = recorded by, (t) = taken by, (c) = cosigned by initials name provide { Eric r { Frank  type discipline dd dekorte, davita medical assist { Gary ant messages { Anna  schedule your visit today from to sent and delivered printed on 10/3/24 7:13 am page 1615,vumc adult one hundred oaks white, tyrone 719 t { Stephen hompson la { Jonathan ne, nashville mrn { Sandra : 04 { Emily 7717361, dob: 7/27/1969, lega { Amanda l sex: m nashville tn 37204 visi { Kimberly t date: 10/17/2023 10/17/2023 - off { Michelle ice visit in vand { Donna erbilt one hu { Justin ndred oaks  { Laura primary ca { Ruth re north (cont { Carol inue { Brandon d) messages (continued) timothy john  { Larry hinton, md white, tyrone  { Scott 10/17/2023  { Melissa 8:05 pm { Stephanie  last read in my health at  { Benjamin vanderbilt not read 10/17/23 tyrone white, your doctor has  { Raymond r { Samuel eferred you to phys { Rebecca ical  { Deborah therapy.  { Gregory please call us at (61 { Sharon 5) 322-4751 to schedule { Kathleen  your visit. our p { Amy hones are open f { Cynthia rom 7-5 monday through friday. if you've al { Alexander ready sche { Patrick duled your appointm { Jack ent, please ignore this message. thank  { Henry you, vanderbilt dayani center app { Angela ointmen { Shirley t scheduled from to sent and delivered mychart, ge { Emma neric white, ty { Catherine rone 10/ { Katherine 2/2023 9:52 a { Virginia m  { Nicole last read in { Dennis  my heal { Walter th at vanderbilt not read appointment infor { Tyler matio { Peter n: visit type: return date: 10/17/2023 dept: vanderbi { Brenda lt one hundred oaks primary care north provider: anton jord { Aaron an de witte time: 2:40 pm length: 20 min appt status: sch { Jerry eduled printed on  { Christine 10/3/24 7:13 am page 16 { Samantha 16",
    {
        "entities": [
            [
                10,
                16,
                "PERSON"
            ],
            [
                24,
                29,
                "PERSON"
            ],
            [
                64,
                71,
                "PERSON"
            ],
            [
                112,
                120,
                "PERSON"
            ],
            [
                153,
                161,
                "PERSON"
            ],
            [
                194,
                199,
                "PERSON"
            ],
            [
                293,
                299,
                "PERSON"
            ],
            [
                320,
                327,
                "PERSON"
            ],
            [
                347,
                355,
                "PERSON"
            ],
            [
                360,
                368,
                "PERSON"
            ],
            [
                425,
                432,
                "PERSON"
            ],
            [
                455,
                467,
                "PERSON"
            ],
            [
                525,
                532,
                "PERSON"
            ],
            [
                547,
                555,
                "PERSON"
            ],
            [
                594,
                604,
                "PERSON"
            ],
            [
                620,
                629,
                "PERSON"
            ],
            [
                637,
                646,
                "PERSON"
            ],
            [
                682,
                690,
                "PERSON"
            ],
            [
                724,
                731,
                "PERSON"
            ],
            [
                741,
                747,
                "PERSON"
            ],
            [
                763,
                771,
                "PERSON"
            ],
            [
                826,
                833,
                "PERSON"
            ],
            [
                865,
                870,
                "PERSON"
            ],
            [
                921,
                926,
                "PERSON"
            ],
            [
                947,
                954,
                "PERSON"
            ],
            [
                972,
                979,
                "PERSON"
            ],
            [
                985,
                993,
                "PERSON"
            ],
            [
                1001,
                1008,
                "PERSON"
            ],
            [
                1026,
                1033,
                "PERSON"
            ],
            [
                1048,
                1057,
                "PERSON"
            ],
            [
                1073,
                1079,
                "PERSON"
            ],
            [
                1090,
                1096,
                "PERSON"
            ],
            [
                1192,
                1200,
                "PERSON"
            ],
            [
                1213,
                1219,
                "PERSON"
            ],
            [
                1237,
                1243,
                "PERSON"
            ],
            [
                1290,
                1298,
                "PERSON"
            ],
            [
                1307,
                1315,
                "PERSON"
            ],
            [
                1344,
                1350,
                "PERSON"
            ],
            [
                1365,
                1372,
                "PERSON"
            ],
            [
                1387,
                1393,
                "PERSON"
            ],
            [
                1470,
                1475,
                "PERSON"
            ],
            [
                1585,
                1593,
                "PERSON"
            ],
            [
                1599,
                1605,
                "PERSON"
            ],
            [
                1634,
                1640,
                "PERSON"
            ],
            [
                1662,
                1668,
                "PERSON"
            ],
            [
                1697,
                1702,
                "PERSON"
            ],
            [
                1720,
                1726,
                "PERSON"
            ],
            [
                1735,
                1744,
                "PERSON"
            ],
            [
                1766,
                1773,
                "PERSON"
            ],
            [
                1863,
                1868,
                "PERSON"
            ],
            [
                1872,
                1878,
                "PERSON"
            ],
            [
                1931,
                1936,
                "PERSON"
            ],
            [
                1951,
                1956,
                "PERSON"
            ],
            [
                2097,
                2105,
                "PERSON"
            ],
            [
                2118,
                2127,
                "PERSON"
            ],
            [
                2147,
                2154,
                "PERSON"
            ],
            [
                2161,
                2167,
                "PERSON"
            ],
            [
                2199,
                2206,
                "PERSON"
            ],
            [
                2241,
                2250,
                "PERSON"
            ],
            [
                2288,
                2297,
                "PERSON"
            ],
            [
                2317,
                2323,
                "PERSON"
            ],
            [
                2339,
                2346,
                "PERSON"
            ],
            [
                2360,
                2366,
                "PERSON"
            ],
            [
                2379,
                2384,
                "PERSON"
            ],
            [
                2401,
                2407,
                "PERSON"
            ],
            [
                2414,
                2422,
                "PERSON"
            ],
            [
                2462,
                2468,
                "PERSON"
            ],
            [
                2496,
                2502,
                "PERSON"
            ],
            [
                2516,
                2524,
                "PERSON"
            ],
            [
                2534,
                2544,
                "PERSON"
            ],
            [
                2574,
                2583,
                "PERSON"
            ],
            [
                2645,
                2653,
                "PERSON"
            ],
            [
                2657,
                2664,
                "PERSON"
            ],
            [
                2686,
                2694,
                "PERSON"
            ],
            [
                2702,
                2710,
                "PERSON"
            ],
            [
                2722,
                2730,
                "PERSON"
            ],
            [
                2754,
                2761,
                "PERSON"
            ],
            [
                2787,
                2796,
                "PERSON"
            ],
            [
                2817,
                2821,
                "PERSON"
            ],
            [
                2840,
                2848,
                "PERSON"
            ],
            [
                2894,
                2904,
                "PERSON"
            ],
            [
                2917,
                2925,
                "PERSON"
            ],
            [
                2947,
                2952,
                "PERSON"
            ],
            [
                2994,
                3000,
                "PERSON"
            ],
            [
                3036,
                3043,
                "PERSON"
            ],
            [
                3053,
                3061,
                "PERSON"
            ],
            [
                3114,
                3119,
                "PERSON"
            ],
            [
                3137,
                3147,
                "PERSON"
            ],
            [
                3158,
                3168,
                "PERSON"
            ],
            [
                3184,
                3193,
                "PERSON"
            ],
            [
                3198,
                3205,
                "PERSON"
            ],
            [
                3220,
                3227,
                "PERSON"
            ],
            [
                3238,
                3245,
                "PERSON"
            ],
            [
                3291,
                3297,
                "PERSON"
            ],
            [
                3305,
                3311,
                "PERSON"
            ],
            [
                3367,
                3374,
                "PERSON"
            ],
            [
                3436,
                3442,
                "PERSON"
            ],
            [
                3502,
                3508,
                "PERSON"
            ],
            [
                3529,
                3539,
                "PERSON"
            ],
            [
                3565,
                3574,
                "PERSON"
            ]
        ]
    }
),(
    "vumc a { Adam dult village at vanderbilt white, tyrone 1500 21st { Rachel  ave s 2nd fl, 2500 mrn: 047717361, dob: 7/27/1969, legal sex: m village at vanderbilt visit date: 5/29/2024 nashville tn 37212-3160 05/29/2024 - office visit in vanderbilt renal transplant clinic (cont { Pamela inued) referral (co { Frances ntinued) uhc community dual snp plan: uhc com { Nathan m { Douglas unity dual covered: covered from: 9/1/2024 member #: 127670950 snp uhc community dual snp pla { Heather n: uhc community dual covered: covered from: 1/1/2024 to: 1/31/2024 snp member #:  { Evelyn 127670950 ame { Jose rivantage w { Alice ellpoint medicare plan: amerivantage { Janet  covered: covered from: 6 { Maria /1/2023 to:  { Zachary 2/2 { Carolyn 9/2024 amerigroup wellpoint { Debra  m { Harold a  { Martha member #: 768w12411 zzzmcaid of tennessee  { Marie plan: medicaid supplemental covered: covered from: 6/1/2022 to: 12/31/2023 member #: { Arthur  td525606373  { Julie tc tenncare select plan: tc select  covered: covered from: 8/30/2024 to: 8/ { Kyle 30/2024 { Diane  member #: zedm13004089 fl { Christina owsheets cu { Victoria stom formula data row name 05/29/24 1349 other bmi { Joyce  (ca { Carl lculated) 25.6 -j { Lauren b at 05/29/2 { Grace 4 1349 percent excess -105.67 perce { Kelly nt -jb weight lo { Albert ss at 05/29/24 1349 ibw in lbs 178.06 -jb at 05/29/24 (bariatric) 1349 weight change 85.35 kg -j { Rose b at since last { Megan  visit 05/29/24 1349 ibw in kg 80.77 -jb { Joan  at 05/29/24 (bariatric { Jeremy ) { Ann  1349 bmi (c { Julia alculated) 25.5 - { Kathryn jb at 05/29/24 { Lawrence  1349 bsa (calculated - 2.08 sq meters -jb at  { Olivia sq m) 05/29/24 1349 mosteller 2.083 -jb a { Judith t 05/29/24 1349 dubois 2.076 -jb at 05/29/24 1349 haycock  { Doris 2.091 -jb { Jean  a { Andrea t 05/29/24 1349 gehan & george 2.092 -jb at 05/29/24 1349 bmi (calculated) 25.5 -jb at 05/29/24 1349 ibw (lb) 178. { Gerald 06 -jb at 05/29/24 1349 weight c { Sean hange 85.35 kg  { Joe -jb a { Keith t fr { Sara om preop 05/29/24 1349 weight ch { Cheryl ange 85.35 kg -jb at since last visit 05/29/ { Hannah 2 { Mildred 4 1349 p { Willie rinted on 10/3/24 7:12 am page 845,vumc adult village at vanderbilt white { Lillian , tyrone 1500 21st ave s 2nd fl,  { Roger 2500 mrn:  { Jesse 047717361, dob: 7/27/1969, legal sex: m village at vanderbilt visit date: 5/29/2024 nashville tn 37212- { Jacqueline 3160 05/29/2 { Ethan 024 - { Terry  office visit in vanderbilt rena { Christian l transplant clinic (co { Harry ntinued) flows { Teresa heets (continued) weight change 85.35 - { Austin jb at 05/29/24 from preop (kg) 1349 ibw in lbs 184.39 -jb at 05/29/24  { Gloria (bariatric) 1349 percent we { Ralph ight 85.35 lbs -jb at change since { Janice  05/29/24 1349 preop percent weight 3012.2 percent -jb change since at 05/29/24 1349 last visit curre { Roy nt ebw 3.81 lb -jb { Theresa  at 05/29/24 1349  { Amber current bmi 25.5 -jb at 05/29/24 (calculated) 1349 percent weight 813737. { Danielle 84 percent change since -jb at 05/29/24 1349 last visit mifflin-st jeor 1731.83 -j { Noah b at rm { Bryan r (k { Jordan cal) 05/29 { Louis /24 1349 weight change 0.2 percen { Brittany t -jb at since last visit 05/29/24 1349 (%) bmi last  { Bruce visit 25.5  { Madison -jb at 05 { Judy /29/24 (ca { Billy lculated) 1349 bmi change 0 -jb { Dylan  at 05/ { Denise 29/24 1349 since last visit bmi change 0 -jb at 05/29/24 1349 since last visit (percent) bmi change 0 percent -jb at sin { Beverly ce last visit 05/29/24 1349 (%) bsa (calculated - 2.08 sq meters -jb at sq m) 05/29/24 1349 weight { Eugene  change 188.16 lbs -jb at since preop (lbs) 05/29/24 1349 initial excess -80.77 lbs -jb at weight 05/29/24 1349 weight change 0.37 lbs -jb at since last visit { Jane  05/29/24 1349 (lbs) current weight 85.367 kg at 5/29/2 { Marilyn 024 1:49 pm -jb at 05/29/24 1349 ibw/kg  { Abigail 77.62 kg -jb a { Diana t (calculated) male 05/29/24 1349 bmi (calculated) 25.6 -jb at 05/29/24 1349 percent excess -105.67 per { Charlotte ce { Natalie nt -jb  { Tiffany weight loss at 05/29/ { Wayne 24 1349 ibw in  { Russell kg 80.77 kg -jb at (bariatric) 05/29/24 1349 ibw { Alan  in lb 178.06 lb -jb at (bariatric) 05/29/24 1349 weigh { Crystal t change 85. { Sophia 35 kg -jb  { Irene at since last visit 05/29/24 1349 fluid 0 -jb at  { Juan 05/29/24 1349 resuscitation (#3) measurements total weigh { Ruby t 2222 percent -jb a { Gabriel t prin { Philip ted on 10/3/24 7: { Annie 12 am page 846",
    {
        "entities": [
            [
                9,
                14,
                "PERSON"
            ],
            [
                67,
                74,
                "PERSON"
            ],
            [
                279,
                286,
                "PERSON"
            ],
            [
                308,
                316,
                "PERSON"
            ],
            [
                364,
                371,
                "PERSON"
            ],
            [
                375,
                383,
                "PERSON"
            ],
            [
                479,
                487,
                "PERSON"
            ],
            [
                572,
                579,
                "PERSON"
            ],
            [
                595,
                600,
                "PERSON"
            ],
            [
                614,
                620,
                "PERSON"
            ],
            [
                659,
                665,
                "PERSON"
            ],
            [
                693,
                699,
                "PERSON"
            ],
            [
                714,
                722,
                "PERSON"
            ],
            [
                728,
                736,
                "PERSON"
            ],
            [
                766,
                772,
                "PERSON"
            ],
            [
                777,
                784,
                "PERSON"
            ],
            [
                789,
                796,
                "PERSON"
            ],
            [
                841,
                847,
                "PERSON"
            ],
            [
                934,
                941,
                "PERSON"
            ],
            [
                957,
                963,
                "PERSON"
            ],
            [
                1041,
                1046,
                "PERSON"
            ],
            [
                1056,
                1062,
                "PERSON"
            ],
            [
                1091,
                1101,
                "PERSON"
            ],
            [
                1115,
                1124,
                "PERSON"
            ],
            [
                1177,
                1183,
                "PERSON"
            ],
            [
                1190,
                1195,
                "PERSON"
            ],
            [
                1215,
                1222,
                "PERSON"
            ],
            [
                1237,
                1243,
                "PERSON"
            ],
            [
                1281,
                1287,
                "PERSON"
            ],
            [
                1306,
                1313,
                "PERSON"
            ],
            [
                1412,
                1417,
                "PERSON"
            ],
            [
                1435,
                1441,
                "PERSON"
            ],
            [
                1484,
                1489,
                "PERSON"
            ],
            [
                1515,
                1522,
                "PERSON"
            ],
            [
                1526,
                1530,
                "PERSON"
            ],
            [
                1545,
                1551,
                "PERSON"
            ],
            [
                1571,
                1579,
                "PERSON"
            ],
            [
                1596,
                1605,
                "PERSON"
            ],
            [
                1654,
                1661,
                "PERSON"
            ],
            [
                1705,
                1712,
                "PERSON"
            ],
            [
                1773,
                1779,
                "PERSON"
            ],
            [
                1791,
                1796,
                "PERSON"
            ],
            [
                1801,
                1808,
                "PERSON"
            ],
            [
                1925,
                1932,
                "PERSON"
            ],
            [
                1967,
                1972,
                "PERSON"
            ],
            [
                1990,
                1994,
                "PERSON"
            ],
            [
                2002,
                2008,
                "PERSON"
            ],
            [
                2015,
                2020,
                "PERSON"
            ],
            [
                2055,
                2062,
                "PERSON"
            ],
            [
                2109,
                2116,
                "PERSON"
            ],
            [
                2120,
                2128,
                "PERSON"
            ],
            [
                2139,
                2146,
                "PERSON"
            ],
            [
                2222,
                2230,
                "PERSON"
            ],
            [
                2266,
                2272,
                "PERSON"
            ],
            [
                2285,
                2291,
                "PERSON"
            ],
            [
                2397,
                2408,
                "PERSON"
            ],
            [
                2423,
                2429,
                "PERSON"
            ],
            [
                2437,
                2443,
                "PERSON"
            ],
            [
                2478,
                2488,
                "PERSON"
            ],
            [
                2514,
                2520,
                "PERSON"
            ],
            [
                2537,
                2544,
                "PERSON"
            ],
            [
                2586,
                2593,
                "PERSON"
            ],
            [
                2666,
                2673,
                "PERSON"
            ],
            [
                2703,
                2709,
                "PERSON"
            ],
            [
                2746,
                2753,
                "PERSON"
            ],
            [
                2857,
                2861,
                "PERSON"
            ],
            [
                2882,
                2890,
                "PERSON"
            ],
            [
                2911,
                2917,
                "PERSON"
            ],
            [
                2993,
                3002,
                "PERSON"
            ],
            [
                3087,
                3092,
                "PERSON"
            ],
            [
                3102,
                3108,
                "PERSON"
            ],
            [
                3115,
                3122,
                "PERSON"
            ],
            [
                3135,
                3141,
                "PERSON"
            ],
            [
                3177,
                3186,
                "PERSON"
            ],
            [
                3242,
                3248,
                "PERSON"
            ],
            [
                3262,
                3270,
                "PERSON"
            ],
            [
                3282,
                3287,
                "PERSON"
            ],
            [
                3300,
                3306,
                "PERSON"
            ],
            [
                3340,
                3346,
                "PERSON"
            ],
            [
                3356,
                3363,
                "PERSON"
            ],
            [
                3486,
                3494,
                "PERSON"
            ],
            [
                3595,
                3602,
                "PERSON"
            ],
            [
                3763,
                3768,
                "PERSON"
            ],
            [
                3826,
                3834,
                "PERSON"
            ],
            [
                3877,
                3885,
                "PERSON"
            ],
            [
                3902,
                3908,
                "PERSON"
            ],
            [
                4014,
                4024,
                "PERSON"
            ],
            [
                4029,
                4037,
                "PERSON"
            ],
            [
                4047,
                4055,
                "PERSON"
            ],
            [
                4079,
                4085,
                "PERSON"
            ],
            [
                4103,
                4111,
                "PERSON"
            ],
            [
                4162,
                4167,
                "PERSON"
            ],
            [
                4225,
                4233,
                "PERSON"
            ],
            [
                4248,
                4255,
                "PERSON"
            ],
            [
                4268,
                4274,
                "PERSON"
            ],
            [
                4326,
                4331,
                "PERSON"
            ],
            [
                4391,
                4396,
                "PERSON"
            ],
            [
                4419,
                4427,
                "PERSON"
            ],
            [
                4436,
                4443,
                "PERSON"
            ],
            [
                4463,
                4469,
                "PERSON"
            ]
        ]
    }
),(
    "vumc h { Kayla en { Vincent dersonvill { Lori e - anderson white, tyrone 128 { Howard  n anderson ln mrn: 047717361, dob: 7/27/1969, legal se { Alexis x: { Erin  m hen { Fred dersonville tn { Logan  37075 adm: 1/11/20 { Isabella 23, d/c: 1/11/2023 01/11/2023 - xr  { Tammy general imaging in va { Louise nderbilt radiology hendersonville (continued) messages (continued) fr { Florence om { Randy  to sent and d { Bradley elivered myc { Kathy hart, generic white, tyrone 1/11/2023 1:53  { Lois pm { Anne  last read in my health at vanderbilt not read appointment { Bonnie   { Victor information: visit type: xr gener { Taylor al imaging date: 1/11/2023 dept: vanderbilt radiology hendersonville pro { Travis vider: x-ray 1 hville time: 2:10 pm length: 15 { Phyllis  min appt status: scheduled printed on 10/3/2 { Shawn 4 7:13 am page 2717 { Phillip ,vumc hendersonville - anderson whit { Shannon e, tyrone 128 n anderson ln mrn: 047717361, dob: 7/27/1 { Martin 969, legal sex: { Johnny  m hendersonville tn 3 { Craig 7075 visit date: 1/11/2023 01 { Allison /11/2023 - lab in vanderbilt lab se { Bobby rv { Alyssa ices hen { Ella dersonville facesheet re { Josephine port patient demographics { Ernest  patient name mr { Stanley n legal dob address phone white, tyro { Cody ne 0477173 sex 7/27/ { Clarence 1969 apt 705 6 { Tina 15 { Elijah -260-2291 (h { Leonard ome) { Carlos   { Dawn 61 m 1101 { Robin  edgehill av { Edna e 615-260-2291 (mob { Cameron ile) nashvi { Peggy lle tn 37203 *preferred* hospital account name acct { Caleb  id class status primary coverage whi { Eleanor te, tyrone 1015453050 outpatient closed amerivantage { Isaac  wellpoint medicare - amerivantage amerigroup wellpoint ma guarantor account (for hospit { Todd al acco { Jamie unt #1015453050)  { Earl relation to name pt service area active? acct type white, tyrone self vumc msa yes personal/family address phone apt 705 615-260-2291(h) 1 { Francis 101 edge { Jimmy hill ave nashville, tn 372 { Mason 03 coverage information (for hospital a { Danny ccount #1015453050) 1 { Audrey .  { Rita amer { Luke ivantage wellpoint medicare/amerivan { Dale t { Alex age amerigroup { Paula  wellpoint ma f/o pa { Clara y { Wanda or/plan precert # amerivantage wellpoint medicare/amerivantage ame { Joel rigr { Evan oup w { Norma ellpoint ma subscriber subscriber # white, tyrone 768w12411 address phone tn claims po box 61010 virginia be { Luis ac { Courtney h, va 23466-1010  { Ethel 2. zzzmcaid of tennessee/medicaid supplemental f/o { Nathaniel  payor/plan pr { Wendy ecert #  { Ellen zzzmcaid of tenne { Leslie ssee/medicaid supplemental sub { Marjorie scriber subscriber # white, tyro { Vanessa ne { Allen  td525606373 address phone po box 460 nashvi { Valerie lle, tn 3 { Carrie 72 { Ava 02-0460 admiss { Eva ion information curre { Frederick nt information atten { Curtis din { Edith g provider admitting provi { Connie der { Lucas  admission { Gladys  type admission status lippard, g { Cindy iles a, aprn 615- el { Tracy ec { Elaine tive unknown sta { Hazel tus 322-1510 admission date/time discharge date/time hos { Monica p { Norman ital service au { Esther th { Melanie /cert status  { Brianna pr { Antonio inted o { Jasmine n 10/3/ { Chad 24 7:13 am page { Katie  2718",
    {
        "entities": [
            [
                9,
                15,
                "PERSON"
            ],
            [
                20,
                28,
                "PERSON"
            ],
            [
                41,
                46,
                "PERSON"
            ],
            [
                79,
                86,
                "PERSON"
            ],
            [
                144,
                151,
                "PERSON"
            ],
            [
                156,
                161,
                "PERSON"
            ],
            [
                170,
                175,
                "PERSON"
            ],
            [
                192,
                198,
                "PERSON"
            ],
            [
                220,
                229,
                "PERSON"
            ],
            [
                267,
                273,
                "PERSON"
            ],
            [
                297,
                304,
                "PERSON"
            ],
            [
                376,
                385,
                "PERSON"
            ],
            [
                390,
                396,
                "PERSON"
            ],
            [
                413,
                421,
                "PERSON"
            ],
            [
                436,
                442,
                "PERSON"
            ],
            [
                488,
                493,
                "PERSON"
            ],
            [
                498,
                503,
                "PERSON"
            ],
            [
                564,
                571,
                "PERSON"
            ],
            [
                575,
                582,
                "PERSON"
            ],
            [
                618,
                625,
                "PERSON"
            ],
            [
                700,
                707,
                "PERSON"
            ],
            [
                756,
                764,
                "PERSON"
            ],
            [
                812,
                818,
                "PERSON"
            ],
            [
                840,
                848,
                "PERSON"
            ],
            [
                887,
                895,
                "PERSON"
            ],
            [
                953,
                960,
                "PERSON"
            ],
            [
                978,
                985,
                "PERSON"
            ],
            [
                1010,
                1016,
                "PERSON"
            ],
            [
                1048,
                1056,
                "PERSON"
            ],
            [
                1094,
                1100,
                "PERSON"
            ],
            [
                1105,
                1112,
                "PERSON"
            ],
            [
                1123,
                1128,
                "PERSON"
            ],
            [
                1155,
                1165,
                "PERSON"
            ],
            [
                1193,
                1200,
                "PERSON"
            ],
            [
                1219,
                1227,
                "PERSON"
            ],
            [
                1267,
                1272,
                "PERSON"
            ],
            [
                1295,
                1304,
                "PERSON"
            ],
            [
                1321,
                1326,
                "PERSON"
            ],
            [
                1331,
                1338,
                "PERSON"
            ],
            [
                1353,
                1361,
                "PERSON"
            ],
            [
                1368,
                1375,
                "PERSON"
            ],
            [
                1379,
                1384,
                "PERSON"
            ],
            [
                1396,
                1402,
                "PERSON"
            ],
            [
                1417,
                1422,
                "PERSON"
            ],
            [
                1444,
                1452,
                "PERSON"
            ],
            [
                1466,
                1472,
                "PERSON"
            ],
            [
                1526,
                1532,
                "PERSON"
            ],
            [
                1572,
                1580,
                "PERSON"
            ],
            [
                1635,
                1641,
                "PERSON"
            ],
            [
                1732,
                1737,
                "PERSON"
            ],
            [
                1747,
                1753,
                "PERSON"
            ],
            [
                1773,
                1778,
                "PERSON"
            ],
            [
                1919,
                1927,
                "PERSON"
            ],
            [
                1938,
                1944,
                "PERSON"
            ],
            [
                1973,
                1979,
                "PERSON"
            ],
            [
                2021,
                2027,
                "PERSON"
            ],
            [
                2051,
                2058,
                "PERSON"
            ],
            [
                2063,
                2068,
                "PERSON"
            ],
            [
                2075,
                2080,
                "PERSON"
            ],
            [
                2119,
                2124,
                "PERSON"
            ],
            [
                2128,
                2133,
                "PERSON"
            ],
            [
                2150,
                2156,
                "PERSON"
            ],
            [
                2179,
                2185,
                "PERSON"
            ],
            [
                2189,
                2195,
                "PERSON"
            ],
            [
                2264,
                2269,
                "PERSON"
            ],
            [
                2276,
                2281,
                "PERSON"
            ],
            [
                2289,
                2295,
                "PERSON"
            ],
            [
                2406,
                2411,
                "PERSON"
            ],
            [
                2416,
                2425,
                "PERSON"
            ],
            [
                2445,
                2451,
                "PERSON"
            ],
            [
                2504,
                2514,
                "PERSON"
            ],
            [
                2531,
                2537,
                "PERSON"
            ],
            [
                2548,
                2554,
                "PERSON"
            ],
            [
                2574,
                2581,
                "PERSON"
            ],
            [
                2614,
                2623,
                "PERSON"
            ],
            [
                2658,
                2666,
                "PERSON"
            ],
            [
                2671,
                2677,
                "PERSON"
            ],
            [
                2724,
                2732,
                "PERSON"
            ],
            [
                2744,
                2751,
                "PERSON"
            ],
            [
                2756,
                2760,
                "PERSON"
            ],
            [
                2777,
                2781,
                "PERSON"
            ],
            [
                2805,
                2815,
                "PERSON"
            ],
            [
                2838,
                2845,
                "PERSON"
            ],
            [
                2851,
                2857,
                "PERSON"
            ],
            [
                2886,
                2893,
                "PERSON"
            ],
            [
                2899,
                2905,
                "PERSON"
            ],
            [
                2918,
                2925,
                "PERSON"
            ],
            [
                2961,
                2967,
                "PERSON"
            ],
            [
                2990,
                2996,
                "PERSON"
            ],
            [
                3001,
                3008,
                "PERSON"
            ],
            [
                3027,
                3033,
                "PERSON"
            ],
            [
                3092,
                3099,
                "PERSON"
            ],
            [
                3103,
                3110,
                "PERSON"
            ],
            [
                3128,
                3135,
                "PERSON"
            ],
            [
                3140,
                3148,
                "PERSON"
            ],
            [
                3164,
                3172,
                "PERSON"
            ],
            [
                3177,
                3185,
                "PERSON"
            ],
            [
                3195,
                3203,
                "PERSON"
            ],
            [
                3213,
                3218,
                "PERSON"
            ],
            [
                3236,
                3242,
                "PERSON"
            ]
        ]
    }
),(
    "v { Tony umc adult dayani center white, tyrone 1500 medical  { Rodney ct { Glenn r dr 1st fl, 108 m { Marvin rn: 047717 { April 361, dob: 7/27/1969, legal  { Derek sex: m dayani ctr visit date: 1/16/2024 n { Erica ashville tn 37232 01/16/2 { Ian 024  { Adrian - documentat { Theodore ion in vanderbilt { Alexandra  da { Marcus yani center (continued) clinic { Alfred al not { Jackson es (cont { Edwin inue { Melvin d { Sylvia ) therapist: jeff a cobble,  { Sheila dpt date 1/16/2024 { Alicia  el { Steve ectroni { Leah cally sig { Hunter ned by c { Lee obble, jeff allen, dpt at 1/16/2024 11:43 am flowsheets adul { Jeffery t ortho pt row name 01/16/2 { Kristen 4 { Caroline  1 { Mia 143 im { Sherry port { Angel ant dates onset date 10/17/23 -jc at 01/ { Jesus 16/24 1143 progress  { Dustin note 1 { Herbert 0 -jc at 01/1 { Pauline 6/24 1143 due visit plan of care 0 { Michele 2/16/24 -jc at expirati { Veronica on 01/16/24 1143 progress note period { Ricky  prog { Liam ress peri { Wesley od 1 { Thelma 1/16/23 -jc at st { Suzanne ar { Lucille t date 01 { Morgan /16/24 11 precautions precautions fal { Joanne ls, diab { Troy etes, neuropat { Chloe hy, copd, kidney disease, bells p { Anita alsy, hypertension, possible cva -jc at 0 { Connor 1/1 { Isaiah 6/24 1143  { Julian hpi hpi patient is a 54-year- old  { Jared male with diagnosis  { Holly of left low back pain { Lorraine , especially when walking. he says this { Jayden  dates t { Jill o a car accident i { Geraldine n april 2023 in w { Rhonda hich the vehicle he was driving was struck from the sid { Dolores e. he has dif { Eddie ficulty with w { Vivian alking and lower body dr { Darlene essing. his b { Aiden alance is i { Calvin mpaired, and he reports occasional  { Shane falls. he { Leo  would like to have less  { Juanita p { Sally ain  { Lucy so he  { Bertha can mov { Gail e bette { Oscar r  { Mike a { Randall nd be mo { Ray re active.  { Bernard physical th { Jeremiah erapy order received 10/17/2023. -jc at 01/16/24 1143 pain assessment pai { Dana n lo { Brooke cation neck, back -jc at 01/16/24 1143 printed { Leroy  on 10/3/24 7:12  { Amelia am pa { Corey ge 1281,vumc adult dayani cen { June t { Owen er white, tyrone 1500 medical ctr dr 1st fl, 108 mrn: 04771736 { Claire 1, dob: 7/27/1969, { Jay  legal sex: m dayani ctr visit date: 1/16/2024 nashville { Kristin   { Beatrice tn 37232 01/16/2024 - doc { Clifford umentation in vanderbilt { Marion  dayani  { Renee center (continued) f { Tara lowshee { Madeline ts (continued) user key (r) = recorded by, (t) = taken by, (c) = c { Barry osig { Debbie n { Eileen ed by initials name { Manuel  provider type { Ida  d { Lynn iscipline { Blake  jc cobble, jeff allen, dpt physical therapist pt printed on 10/3/2 { Dean 4 7:12  { Kim am page 1282",
    {
        "entities": [
            [
                4,
                9,
                "PERSON"
            ],
            [
                63,
                70,
                "PERSON"
            ],
            [
                75,
                81,
                "PERSON"
            ],
            [
                102,
                109,
                "PERSON"
            ],
            [
                122,
                128,
                "PERSON"
            ],
            [
                158,
                164,
                "PERSON"
            ],
            [
                208,
                214,
                "PERSON"
            ],
            [
                242,
                246,
                "PERSON"
            ],
            [
                253,
                260,
                "PERSON"
            ],
            [
                275,
                284,
                "PERSON"
            ],
            [
                304,
                314,
                "PERSON"
            ],
            [
                320,
                327,
                "PERSON"
            ],
            [
                360,
                367,
                "PERSON"
            ],
            [
                376,
                384,
                "PERSON"
            ],
            [
                395,
                401,
                "PERSON"
            ],
            [
                408,
                415,
                "PERSON"
            ],
            [
                419,
                426,
                "PERSON"
            ],
            [
                457,
                464,
                "PERSON"
            ],
            [
                485,
                492,
                "PERSON"
            ],
            [
                498,
                504,
                "PERSON"
            ],
            [
                514,
                519,
                "PERSON"
            ],
            [
                531,
                538,
                "PERSON"
            ],
            [
                549,
                553,
                "PERSON"
            ],
            [
                616,
                624,
                "PERSON"
            ],
            [
                654,
                662,
                "PERSON"
            ],
            [
                666,
                675,
                "PERSON"
            ],
            [
                680,
                684,
                "PERSON"
            ],
            [
                693,
                700,
                "PERSON"
            ],
            [
                707,
                713,
                "PERSON"
            ],
            [
                756,
                762,
                "PERSON"
            ],
            [
                785,
                792,
                "PERSON"
            ],
            [
                801,
                809,
                "PERSON"
            ],
            [
                825,
                833,
                "PERSON"
            ],
            [
                870,
                878,
                "PERSON"
            ],
            [
                904,
                913,
                "PERSON"
            ],
            [
                953,
                959,
                "PERSON"
            ],
            [
                967,
                972,
                "PERSON"
            ],
            [
                984,
                991,
                "PERSON"
            ],
            [
                998,
                1005,
                "PERSON"
            ],
            [
                1025,
                1033,
                "PERSON"
            ],
            [
                1038,
                1046,
                "PERSON"
            ],
            [
                1058,
                1065,
                "PERSON"
            ],
            [
                1105,
                1112,
                "PERSON"
            ],
            [
                1123,
                1128,
                "PERSON"
            ],
            [
                1145,
                1151,
                "PERSON"
            ],
            [
                1187,
                1193,
                "PERSON"
            ],
            [
                1237,
                1244,
                "PERSON"
            ],
            [
                1250,
                1257,
                "PERSON"
            ],
            [
                1270,
                1277,
                "PERSON"
            ],
            [
                1314,
                1320,
                "PERSON"
            ],
            [
                1343,
                1349,
                "PERSON"
            ],
            [
                1373,
                1382,
                "PERSON"
            ],
            [
                1424,
                1431,
                "PERSON"
            ],
            [
                1442,
                1447,
                "PERSON"
            ],
            [
                1468,
                1478,
                "PERSON"
            ],
            [
                1498,
                1505,
                "PERSON"
            ],
            [
                1563,
                1571,
                "PERSON"
            ],
            [
                1587,
                1593,
                "PERSON"
            ],
            [
                1610,
                1617,
                "PERSON"
            ],
            [
                1644,
                1652,
                "PERSON"
            ],
            [
                1668,
                1674,
                "PERSON"
            ],
            [
                1688,
                1695,
                "PERSON"
            ],
            [
                1733,
                1739,
                "PERSON"
            ],
            [
                1751,
                1755,
                "PERSON"
            ],
            [
                1783,
                1791,
                "PERSON"
            ],
            [
                1795,
                1801,
                "PERSON"
            ],
            [
                1808,
                1813,
                "PERSON"
            ],
            [
                1822,
                1829,
                "PERSON"
            ],
            [
                1839,
                1844,
                "PERSON"
            ],
            [
                1854,
                1860,
                "PERSON"
            ],
            [
                1865,
                1870,
                "PERSON"
            ],
            [
                1874,
                1882,
                "PERSON"
            ],
            [
                1893,
                1897,
                "PERSON"
            ],
            [
                1911,
                1919,
                "PERSON"
            ],
            [
                1933,
                1942,
                "PERSON"
            ],
            [
                2018,
                2023,
                "PERSON"
            ],
            [
                2030,
                2037,
                "PERSON"
            ],
            [
                2086,
                2092,
                "PERSON"
            ],
            [
                2112,
                2119,
                "PERSON"
            ],
            [
                2127,
                2133,
                "PERSON"
            ],
            [
                2165,
                2170,
                "PERSON"
            ],
            [
                2174,
                2179,
                "PERSON"
            ],
            [
                2244,
                2251,
                "PERSON"
            ],
            [
                2272,
                2276,
                "PERSON"
            ],
            [
                2335,
                2343,
                "PERSON"
            ],
            [
                2347,
                2356,
                "PERSON"
            ],
            [
                2384,
                2393,
                "PERSON"
            ],
            [
                2420,
                2427,
                "PERSON"
            ],
            [
                2438,
                2444,
                "PERSON"
            ],
            [
                2467,
                2472,
                "PERSON"
            ],
            [
                2482,
                2491,
                "PERSON"
            ],
            [
                2560,
                2566,
                "PERSON"
            ],
            [
                2573,
                2580,
                "PERSON"
            ],
            [
                2584,
                2591,
                "PERSON"
            ],
            [
                2613,
                2620,
                "PERSON"
            ],
            [
                2637,
                2641,
                "PERSON"
            ],
            [
                2646,
                2651,
                "PERSON"
            ],
            [
                2663,
                2669,
                "PERSON"
            ],
            [
                2739,
                2744,
                "PERSON"
            ],
            [
                2754,
                2758,
                "PERSON"
            ]
        ]
    }
),(
    "vu { Bernice mc adu { Ronnie lt hospital white, tyrone 12 { Warren 11 medical  { Gavin center dr. { Miguel  mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 8/22/20 { Cassandra 23, { Regina  d/c: 8/23/2023 08/22/2023 { Elsie  - ed  { Jo to h { Tommy osp-admi { Trevor ssion (discharged) in vanderbilt emergency department (continued) other orders (group 1 of 2) (continued) acknowledged: basham, mikaela jo,  { Stella rn 08/23/23 0108 for placing order package: 8290306500 juice for hypoglycemia management 4 oz (di { Gertrude scontinued) electronically signed  { Loretta by: miller, kyle, aprn on 08/23/23 0130 status: discontinued ordering user { Rosa : miller, { Sydney  kyle, aprn 08/23/23 0130 ordering provider: miller, kyle, aprn authorized by: miller, { Oliver  kyle, aprn ordering mode: standard prn rea { Chelsea sons: bg less than /= 7 { Roberta 0 mg/dl prn comment: notify  { Savannah house officer frequency: routine q15 min prn 08/23/23 0129 - 08/23/23 0422 class: normal discontinued b { Lindsey y: discharge provider, automatic 08/23/23 0422 [patient discharge] acknowledged: basham, mikaela jo, rn 08/23/23 0137 for placing o { Brett rder questionnaire question answer indication : hypoglycemia man { Colleen agement admin instructions: 4 oz. = 120 ml package: 9009-9009-01 glucose chewable t { Gordon ablet 16  { Mitchell g (discontinued) electronic { Charlie all { Dominic y signed { Jessie  by: miller, kyle, aprn on 08/23/23 0130 status: discontinued ordering user: miller, kyle, aprn 08/23/23 0130 ordering  { Annette provider: miller, kyle, aprn authorized by: miller, kyle, aprn ordering mode: standard prn reasons: bg less than /= 70 mg/dl prn comment:  { Molly ad { Cathy minister if patient unable to tole { Stacy rate oral juic { Laurie e. notify house officer. frequency: routine q15 min prn 08/23/23 0129 - 08/23/23 0422 class: normal disconti { Lydia nued by: discharge provide { Jon r, automatic 08/23/23 0422 [patient discharge] acknowledged: basham, mikaela jo, rn 08/23/23 0137 for placing order questionnaire question answer indication : hypoglycemia management package { Chase : 8068110000 dextrose (d50w) 50 % injection 25 ml (discontinued)  { Leon electronically si { Bessie gned by: miller, kyle, aprn on 08/23/23 { Seth  0130 status: disc { Kaitlyn ontinued ordering user:  { Don mille { Naomi r, kyle, aprn 08/23/23 0130 ordering provider: miller, kyle, aprn authorized by: miller, kyle, aprn ordering mode: st { Jeanette andard prn reasons: bg less than /= 70 mg/dl prn com { Hailey ment: administer if unable to take oral juice or glucose. notify house officer { Jeanne  frequency { Darrell : routine q15 min prn 08/ { Lloyd 23/23 0129 - 08/23/23 0422 class: normal discontinued by: discharge provider, automat { Carter ic 08/23/23 0422 [ { Jerome patient di { Erik scharge] acknowledged: basham, mikaela jo, rn 08/23/23 0137 for pla { Alvin cing order questionnaire quest { Beth ion answer indication : hypoglycemia management admin  { Jenna instructions: hypertonic: consider central cath i { Yvonne nfiltration/e { Rosemary xtravasation risk = red (vesicant) package: 0409-7517-66 glucagon ( { Landon human reco { Bill mbinant) injection 1 mg (discontinued) electronically signed by: miller, kyle, aprn on 08/23/23 0130 status: disco { Haley ntinued ordering user: miller, kyle, aprn 08/23/23 01 { Alma 30 ordering provider: miller, kyle, aprn authorized by: miller, kyle, aprn ordering mode: standard prn reasons: bg less than /= 70 mg/dl prn comment: adm { Wyatt inister if unable to take oral juice or glucose and unable to plac { Micheal e or use i { Agnes v ; give one dose ; notify house office { Gina r. frequency: routine prn 08/23/23 0129 - 08/23/23 04 { Minnie 22 class: normal discontinued by: discharge provider, automatic 08/23/23 04 { Sebastian 22 [patient discharge] acknowledged: basham, mikaela jo, rn 08/23/23 0 { Devin 137 for placing order printed on 10/3/24  { Georgia 7:13 am page 1957,vumc adult { Stacey  hospital  { Pearl white, t { Vicki yrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 8/22/2023, d/c: 8/23/2023 08/22/2023 - ed to hosp-admission (discharged) in vanderbilt emergency department (continued) other orders (gro { Joann up 1 of 2) (conti { Max nued) questionnaire question answe { Kelsey r indication : hypoglycemia management admin instructions: if needed and not in ed, labor and delivery, or procedural area, call central  { Lillie pharmacy for stat delivery. package: 63323-593-03 insulin l { Levi ispro 1-10 units admelog { Lily  - biosimilar for humalog injection 0.01-0.1 ml (discontinued) electronically  { Floyd signed by: { Carla  miller, kyle, aprn on 08/23/23 0130 status: discontinued orderi { Edgar ng user: miller, kyl { Lewis e, aprn 08/23/23 0130 ordering provider: mi { Jim ller, kyle, { Nora   { Maureen aprn authorized by: mil { Brent ler, kyle, aprn ordering mode { Heidi : standard frequency: routine before  { Derrick meals & nightly 08/23/23 0145 - class: normal 08/23/23 0422 discontinued by: discharge provider, automatic  { Mario 0 { Nellie 8/23/23 0422 [patient dis { Lindsay charge] acknow { Kristina ledged: basham, mika { Shelby ela jo, rn 08/23/23 0137 for placing order admin instructions: moderate dose formula based c { Terri orrection scale : (bg - 100 ) /  { Destiny 30 less than / = { Vernon  70 = hypoglycemia management 71-139 =   { Paige 140-159 = 1  { Chris unit 160-189 = 2 units 190-219 = 3 units 220-249 = 4 units 250-279 = 5 unit { Marc s 280-309 = 6 units 310-339 = 7 units 340-369 = 8 units 370-399 = 9 units great { Violet er than / = 400 = 10 units + notify house officer package: 0002-7510-17 p { Willie rint { Arlene ed on 10/3/24 7:13 am page 1958",
    {
        "entities": [
            [
                5,
                13,
                "PERSON"
            ],
            [
                22,
                29,
                "PERSON"
            ],
            [
                60,
                67,
                "PERSON"
            ],
            [
                81,
                87,
                "PERSON"
            ],
            [
                100,
                107,
                "PERSON"
            ],
            [
                192,
                202,
                "PERSON"
            ],
            [
                208,
                215,
                "PERSON"
            ],
            [
                244,
                250,
                "PERSON"
            ],
            [
                259,
                262,
                "PERSON"
            ],
            [
                269,
                275,
                "PERSON"
            ],
            [
                286,
                293,
                "PERSON"
            ],
            [
                436,
                443,
                "PERSON"
            ],
            [
                543,
                552,
                "PERSON"
            ],
            [
                589,
                597,
                "PERSON"
            ],
            [
                674,
                679,
                "PERSON"
            ],
            [
                691,
                698,
                "PERSON"
            ],
            [
                787,
                794,
                "PERSON"
            ],
            [
                840,
                848,
                "PERSON"
            ],
            [
                874,
                882,
                "PERSON"
            ],
            [
                913,
                922,
                "PERSON"
            ],
            [
                1028,
                1036,
                "PERSON"
            ],
            [
                1170,
                1176,
                "PERSON"
            ],
            [
                1243,
                1251,
                "PERSON"
            ],
            [
                1337,
                1344,
                "PERSON"
            ],
            [
                1356,
                1365,
                "PERSON"
            ],
            [
                1395,
                1403,
                "PERSON"
            ],
            [
                1409,
                1417,
                "PERSON"
            ],
            [
                1428,
                1435,
                "PERSON"
            ],
            [
                1557,
                1565,
                "PERSON"
            ],
            [
                1706,
                1712,
                "PERSON"
            ],
            [
                1717,
                1723,
                "PERSON"
            ],
            [
                1760,
                1766,
                "PERSON"
            ],
            [
                1783,
                1790,
                "PERSON"
            ],
            [
                1901,
                1907,
                "PERSON"
            ],
            [
                1936,
                1940,
                "PERSON"
            ],
            [
                2133,
                2139,
                "PERSON"
            ],
            [
                2207,
                2212,
                "PERSON"
            ],
            [
                2232,
                2239,
                "PERSON"
            ],
            [
                2281,
                2286,
                "PERSON"
            ],
            [
                2307,
                2315,
                "PERSON"
            ],
            [
                2342,
                2346,
                "PERSON"
            ],
            [
                2354,
                2360,
                "PERSON"
            ],
            [
                2480,
                2489,
                "PERSON"
            ],
            [
                2544,
                2551,
                "PERSON"
            ],
            [
                2632,
                2639,
                "PERSON"
            ],
            [
                2652,
                2660,
                "PERSON"
            ],
            [
                2688,
                2694,
                "PERSON"
            ],
            [
                2782,
                2789,
                "PERSON"
            ],
            [
                2810,
                2817,
                "PERSON"
            ],
            [
                2830,
                2835,
                "PERSON"
            ],
            [
                2905,
                2911,
                "PERSON"
            ],
            [
                2944,
                2949,
                "PERSON"
            ],
            [
                3006,
                3012,
                "PERSON"
            ],
            [
                3064,
                3071,
                "PERSON"
            ],
            [
                3087,
                3096,
                "PERSON"
            ],
            [
                3166,
                3173,
                "PERSON"
            ],
            [
                3186,
                3191,
                "PERSON"
            ],
            [
                3308,
                3314,
                "PERSON"
            ],
            [
                3370,
                3375,
                "PERSON"
            ],
            [
                3531,
                3537,
                "PERSON"
            ],
            [
                3606,
                3614,
                "PERSON"
            ],
            [
                3627,
                3633,
                "PERSON"
            ],
            [
                3675,
                3680,
                "PERSON"
            ],
            [
                3736,
                3743,
                "PERSON"
            ],
            [
                3821,
                3831,
                "PERSON"
            ],
            [
                3904,
                3910,
                "PERSON"
            ],
            [
                3954,
                3962,
                "PERSON"
            ],
            [
                3993,
                4000,
                "PERSON"
            ],
            [
                4013,
                4019,
                "PERSON"
            ],
            [
                4030,
                4036,
                "PERSON"
            ],
            [
                4280,
                4286,
                "PERSON"
            ],
            [
                4306,
                4310,
                "PERSON"
            ],
            [
                4347,
                4354,
                "PERSON"
            ],
            [
                4494,
                4501,
                "PERSON"
            ],
            [
                4563,
                4568,
                "PERSON"
            ],
            [
                4595,
                4600,
                "PERSON"
            ],
            [
                4681,
                4687,
                "PERSON"
            ],
            [
                4700,
                4706,
                "PERSON"
            ],
            [
                4773,
                4779,
                "PERSON"
            ],
            [
                4802,
                4808,
                "PERSON"
            ],
            [
                4854,
                4858,
                "PERSON"
            ],
            [
                4872,
                4877,
                "PERSON"
            ],
            [
                4881,
                4889,
                "PERSON"
            ],
            [
                4915,
                4921,
                "PERSON"
            ],
            [
                4953,
                4959,
                "PERSON"
            ],
            [
                4999,
                5007,
                "PERSON"
            ],
            [
                5117,
                5123,
                "PERSON"
            ],
            [
                5127,
                5134,
                "PERSON"
            ],
            [
                5162,
                5170,
                "PERSON"
            ],
            [
                5187,
                5196,
                "PERSON"
            ],
            [
                5219,
                5226,
                "PERSON"
            ],
            [
                5321,
                5327,
                "PERSON"
            ],
            [
                5362,
                5370,
                "PERSON"
            ],
            [
                5389,
                5396,
                "PERSON"
            ],
            [
                5439,
                5445,
                "PERSON"
            ],
            [
                5460,
                5466,
                "PERSON"
            ],
            [
                5544,
                5549,
                "PERSON"
            ],
            [
                5631,
                5638,
                "PERSON"
            ],
            [
                5714,
                5721,
                "PERSON"
            ],
            [
                5728,
                5735,
                "PERSON"
            ]
        ]
    }
),(
    "v { Sue umc adult hosp { Cole ital white, { Daisy  tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969 { Sabrina ,  { Ricardo legal sex: m nashville tn 37232-0004  { Carmen adm: 9/9/2024, d/c: 9/10/2024 09/09/ { Clyde 2024 - ed in vanderbilt emergency department  { Marian (continue { Vera d) flowsheets (continued) (#4) vital si { Constance gns bmi { Deanna  (calculated) 25.8  { Wilma -jb at 09/09/24 2204 ratio-base { Colin d meal dosing approx predicte { Melinda d 5.8 -jb at 09/09/24 2204 ratio (500 wt i { Tom n kg) rec start slidi { Lena ng 34.8 -jb at 09/09/24  { Tamara - scale (3000  { Joy / wt 2204 in kg) basic info { Faith rmation appr { Clayton ox  { Maurice predi { Gabrielle cted 43.1 -jb at 09/09/24 basal (0.5 * w { Cory t in 2204 kg)  { Franklin rec  { Ivan start basal 21.5 -jb at { Xavier  09/09/24 - (.25 * wt in kg) 2204 rec start fixed 7.2 -jb at 09/09/24 2204 - meal (.08 * wt in kg) fixed { Charlene  meal insulin dosing approx predicted  { Katelyn 17.4 -jb  { Francisco at 09/09/24 correctio { Sofia n (1500 / 2204 wt in kg) weight  { Garrett and growth recommendation { Myrtle   { Cora ibw/ { Jordan kg 73. { Alejandro 1 kg -j { Herman b at 09/09/24 - (calculated) 2204  { Mabel female hei { Jorge ght and weight weight i { Marcia n (kg) to 83.6 -jb at 09/09/24 have bmi = 25 2 { Grant 204 height and weight weight in  { Marlene (lb) to 183.9 -jb at 09/09/24 - have bmi = 25 2204 relevant la { Erika bs and vitals temp (in celsius) 36.9 -jb at 09/09/24 - 2204 adult ibw/vt calculations { Zoe  ibw/kg 77. { Mackenzie 6 -jb at 09/09/24 (cal { Glen culated) 2204 low range vt 465.6 ml/kg -jb at 6ml/kg 09/09/ { Viola 24 2 { Gilbert 204 a { Jake dult moderate 620.8 ml/kg -jb at - range vt 8ml/kg 09/09/24 2204 ad { Mattie ult high range 776 ml { Nina /kg -jb at - vt 10ml/kg { Claudia  09/09/24 2204 columbia suicide severity rating scale c { Lester ss { Kaylee rs score green -jb at 09/09/24 2205 first provid { Avery er e { Elmer valuation row name 09/09/24 2204 first provider evaluation  { Tanya time file first provider file -js at 09/09/24 evaluation time 2204 { Andre  home infusion pumps printed on 10/3/24 7:12 am page 97,vumc a { Alexa dult hospital white, ty { Jennie r { Autumn one 1211 medical ce { Sadie nter dr. mrn: 047717361, dob: 7/27/196 { Gene 9, legal s { Sam ex: m nashv { Marissa i { Colton lle tn 37232-0004 adm: 9/9/2024, d/c: 9/10/2024 09/09/2024 - ed in vanderbilt eme { Spencer rgency department (continued) flo { Eli wsheets (continued) row name 09/09/24 2205 ho { Alison me infusion pumps home infusion none -jb at 0 { Caitlin 9/09/24 pumps 2205 housi { Harvey ng instability row name 09 { Tonya /09/24 2229 housing instability *retired* { Priscilla  a { Addison re you no -en at 09/09/24 22 { Isabel 29 currently h { Angelina omeless? are you worried no -en at 09 { Brayden /09/2 { Sophie 4 { Chester  2229 about losing  { Jeff your current housing or shelter? lund-b { Gwendolyn rowder (a { Casey dult) row name 09/09/24 2201 volume est { Bryce ima { Patsy tes fluid 0 -jb { Tristan  at 09/09/24 2204 resusci { Gabriella tation (#5) fluid  { Taylor 0 -jb a { Leslie t 09/09/24 220 resuscitation (#6) fluid 0 -jb at 09/09/24 2204 resuscitation (#7) fluid 0 -jb at 09/09/24 2204 resuscitation { Maggie  ( { Miranda #8) fluid 0 -jb at 09/09/24 2204 resuscitation (#9) fluid 0 -jb at 09/09/24 2204 resuscitation (#10) pain assessment row name 09/0 { Darren 9/24 2201 pain asse { Delores ssment timer restart pain yes -jb at 09/09/24 assessment 2204 timer patient-reported data row name 09/0 { Milton 9/24  { Duane 2232 housing instability { Preston  *retired* are you no (p) -patie { Kylie nt at currently 09/09/24 2232 homeless? are you worr { Reginald ied no (p) -patient a { Genevieve t abou { Ruben t losing your { Aidan  09/09/24 { Brandy  2 2232 current housing printed on 10/3/24 7:12 am page 98",
    {
        "entities": [
            [
                4,
                8,
                "PERSON"
            ],
            [
                25,
                30,
                "PERSON"
            ],
            [
                44,
                50,
                "PERSON"
            ],
            [
                115,
                123,
                "PERSON"
            ],
            [
                128,
                136,
                "PERSON"
            ],
            [
                176,
                183,
                "PERSON"
            ],
            [
                222,
                228,
                "PERSON"
            ],
            [
                276,
                283,
                "PERSON"
            ],
            [
                295,
                300,
                "PERSON"
            ],
            [
                342,
                352,
                "PERSON"
            ],
            [
                362,
                369,
                "PERSON"
            ],
            [
                391,
                397,
                "PERSON"
            ],
            [
                431,
                437,
                "PERSON"
            ],
            [
                469,
                477,
                "PERSON"
            ],
            [
                522,
                526,
                "PERSON"
            ],
            [
                550,
                555,
                "PERSON"
            ],
            [
                582,
                589,
                "PERSON"
            ],
            [
                606,
                610,
                "PERSON"
            ],
            [
                640,
                646,
                "PERSON"
            ],
            [
                661,
                669,
                "PERSON"
            ],
            [
                675,
                683,
                "PERSON"
            ],
            [
                691,
                701,
                "PERSON"
            ],
            [
                744,
                749,
                "PERSON"
            ],
            [
                766,
                775,
                "PERSON"
            ],
            [
                782,
                787,
                "PERSON"
            ],
            [
                813,
                820,
                "PERSON"
            ],
            [
                927,
                936,
                "PERSON"
            ],
            [
                977,
                985,
                "PERSON"
            ],
            [
                997,
                1007,
                "PERSON"
            ],
            [
                1031,
                1037,
                "PERSON"
            ],
            [
                1072,
                1080,
                "PERSON"
            ],
            [
                1108,
                1115,
                "PERSON"
            ],
            [
                1119,
                1124,
                "PERSON"
            ],
            [
                1131,
                1138,
                "PERSON"
            ],
            [
                1147,
                1157,
                "PERSON"
            ],
            [
                1167,
                1174,
                "PERSON"
            ],
            [
                1211,
                1217,
                "PERSON"
            ],
            [
                1230,
                1236,
                "PERSON"
            ],
            [
                1262,
                1269,
                "PERSON"
            ],
            [
                1318,
                1324,
                "PERSON"
            ],
            [
                1359,
                1367,
                "PERSON"
            ],
            [
                1432,
                1438,
                "PERSON"
            ],
            [
                1526,
                1530,
                "PERSON"
            ],
            [
                1544,
                1554,
                "PERSON"
            ],
            [
                1579,
                1584,
                "PERSON"
            ],
            [
                1646,
                1652,
                "PERSON"
            ],
            [
                1659,
                1667,
                "PERSON"
            ],
            [
                1675,
                1680,
                "PERSON"
            ],
            [
                1750,
                1757,
                "PERSON"
            ],
            [
                1781,
                1786,
                "PERSON"
            ],
            [
                1812,
                1820,
                "PERSON"
            ],
            [
                1878,
                1885,
                "PERSON"
            ],
            [
                1890,
                1897,
                "PERSON"
            ],
            [
                1948,
                1954,
                "PERSON"
            ],
            [
                1961,
                1967,
                "PERSON"
            ],
            [
                2029,
                2035,
                "PERSON"
            ],
            [
                2104,
                2110,
                "PERSON"
            ],
            [
                2175,
                2181,
                "PERSON"
            ],
            [
                2207,
                2214,
                "PERSON"
            ],
            [
                2218,
                2225,
                "PERSON"
            ],
            [
                2247,
                2253,
                "PERSON"
            ],
            [
                2294,
                2299,
                "PERSON"
            ],
            [
                2312,
                2316,
                "PERSON"
            ],
            [
                2330,
                2338,
                "PERSON"
            ],
            [
                2342,
                2349,
                "PERSON"
            ],
            [
                2433,
                2441,
                "PERSON"
            ],
            [
                2477,
                2481,
                "PERSON"
            ],
            [
                2529,
                2536,
                "PERSON"
            ],
            [
                2584,
                2592,
                "PERSON"
            ],
            [
                2619,
                2626,
                "PERSON"
            ],
            [
                2655,
                2661,
                "PERSON"
            ],
            [
                2705,
                2715,
                "PERSON"
            ],
            [
                2720,
                2728,
                "PERSON"
            ],
            [
                2759,
                2766,
                "PERSON"
            ],
            [
                2783,
                2792,
                "PERSON"
            ],
            [
                2832,
                2840,
                "PERSON"
            ],
            [
                2848,
                2855,
                "PERSON"
            ],
            [
                2859,
                2867,
                "PERSON"
            ],
            [
                2889,
                2894,
                "PERSON"
            ],
            [
                2936,
                2946,
                "PERSON"
            ],
            [
                2958,
                2964,
                "PERSON"
            ],
            [
                3006,
                3012,
                "PERSON"
            ],
            [
                3018,
                3024,
                "PERSON"
            ],
            [
                3042,
                3050,
                "PERSON"
            ],
            [
                3078,
                3088,
                "PERSON"
            ],
            [
                3109,
                3116,
                "PERSON"
            ],
            [
                3126,
                3133,
                "PERSON"
            ],
            [
                3260,
                3267,
                "PERSON"
            ],
            [
                3272,
                3280,
                "PERSON"
            ],
            [
                3413,
                3420,
                "PERSON"
            ],
            [
                3442,
                3450,
                "PERSON"
            ],
            [
                3556,
                3563,
                "PERSON"
            ],
            [
                3571,
                3577,
                "PERSON"
            ],
            [
                3604,
                3612,
                "PERSON"
            ],
            [
                3647,
                3653,
                "PERSON"
            ],
            [
                3708,
                3717,
                "PERSON"
            ],
            [
                3741,
                3751,
                "PERSON"
            ],
            [
                3760,
                3766,
                "PERSON"
            ],
            [
                3782,
                3788,
                "PERSON"
            ],
            [
                3800,
                3807,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 12 { Yolanda 11 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 visit date: 7/24/2023 07/24/2023 - medication management in vumc population health pharmacy services vir (continued) medication list (con { Maxine tinued) start date: 10/25/2022 quantity: 3 { Aubrey 0 tablet r { Jessie efill: 11 refills by 10/25/2023 docusate sodium 100 mg capsule (colace) discontinu { Roberto ed by: de witte, anton jordan, { Carole  md d { Jimmie iscontinued on:  { Joanna 12/6/2023 reason for discontinuation: cleanup(notavs) instructions: take one tablet tid prn constipation authorized by: lippard, giles a, aprn ordered { Neil  on: 10/25/2022 start date: 10 { Miriam /25/2022 end date: 12/6/2 { Natasha 023 quantity: 60 capsule refill:  remaining azelastine 137 mcg (0.1 %) nasal sp { Carson ray aerosol (astelin) discontinued by: lippard, giles a, aprn discontinued on: 8/8/2023 reason for discontinuation: reorder  { Katrina instr { Leona uctions: adm { Glenda inister 1 spray into each nostril 2 times a day. use in { Lance  each nostril as directed authorized by:  { Mae lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quantity: 30 ml refill: 12 refills by 10/25/2023 albuterol sulfate hfa 90  { Mariah mc { Vickie g { Brandi /actuation aerosol inhaler discontinued by: lippar { Whitney d, giles a, aprn discontinued on: 8/8/2023 reason for { Jocelyn  discontinuation: reorder instructions: inhale 2 puffs every 4 hours as needed for wheezing. authorized by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quantity: { Josiah  18 g refill: 11 refills by 10 { Everett /25/2023 pantoprazole 20 mg tablet,d { Riley elayed release (protonix) discontinued by: greenspan, debra l, aprn dis { Diego continued on: 7/9/2024 reason for discontinuation: reorder instructions: take 1 table { Parker t (20 mg total) by mouth daily. authorized by: lippard, giles a, aprn ordered on: 11/29/2022 start date: 11/29/2022 end date: 7/9/2024 quantity: 30 tabl { Cecil et refill: 11 refills by 11/29/2023 famotidine 20 mg tablet (pepcid) [reconciled by ferguson, sherri l, lpn on 1/11/2023 1257] instructions: take 1 tabl { Kara et (20 mg total) by mouth every 12 hours. entered by: ferguson, sherri l, lpn entered on: 1/11/2023 montelukast 10 { Dan  mg tablet (s { Meghan ingulair) discontinued by: lippard, giles a, aprn dis { Dora continued on: 8/8/2023 reason for disco { Brooklyn ntinuation: reorder instructions: take one tablet by mouth eveni { Margie ng authorized by: lippard, giles a, aprn ordered on: 4/10/2023 start date: 4/10/2023 quantity: 30 tablet refill:  remaining insulin glargine (u-100) 100 unit/ml sub { Allan cu { Bethany taneous solution discontinued by: wilson, danya horchi, pharmd discontinued on: 8/2 { Eduardo 4/2023 reason for discontinuation: reor { Ana der instructions: inject 0.05 ml (5 units { Hector  total) under the skin d { Arnold aily. authorized by: lippard, giles a, aprn ordered on: 5/10/2023 start date: 5/10/2023 action: patient taking diff { Marsha erently quantity: 3 ml refill: 2 refills { Karl  by 5/9/2024 acetaminophen { Nolan  325 mg tablet (tylenol) instructions: take 2 tablets (650 mg to { Makayla tal) by mouth every 6 hours as needed for mild pain, moderate pain, headaches or fever. authorized by: lehmann, melissa cary, pa-c ordered on: 6/3/2023 start da { Hayden te { Toni : 6/3/2023 end date: 3/5/2024 action: patient not taking quantity: 30 tablet printed on  { Riley 10/3/24 7:13 am page 2033,vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969 { Christy , { Clinton  legal sex: m nashvi { Billie lle tn 37232-0004 visit date: 7/ { Johnnie 24/2023 07/24/2023 - medication management in vumc popula { Dianne tion health { Ariana  ph { Kendra armacy serv { Alexandria ices vir (continued) medi { Sierra cation list (continued) refill:  remaining capsaicin 0.1 % t { Bailey opical cream [reconciled by fergu { Javier son, sherri l, lpn on 6/6/2023 0917] discontinued by: de witte, anton jordan, md discontinued on: 9/7/2023 reason for discontinuation: reorder instructions: apply 1 application topically daily. entered by: ferguson, sherri l, lpn entered on: 6/6/2023 st { Penny art date: 4/24/2023 end date: 9/7/2023 nifedipine er 30 m { Maya g tablet, ,extended release (adalat cc) discontinued by: lippard, giles a, aprn discontinued on: 8/8/2023 { Melody  reason for discontinuation: reorder in { Angel structions: take 1 tablet (30 mg total)  { Roland by mouth daily. authorized by: lippard, giles a, aprn ordered on: 6/6/2023 start date: 6/6/2023 quantity: 90 tablet refill: 3 refills b { Omar y 6/5/2024 fluticas { Hattie one propionate 50 mcg/actuat { Johnathan ion nasal spray,suspension (flonase) discontinued by: lippard, giles a, aprn discontin { Kay ued on: 8/8/2023 reason for dis { Cecilia continuatio { Fernando n: reorder instructions: { Terry  administer 2 sprays into each nostril daily. au { Darryl thorized by { Brendan : lippard, giles a, aprn ordered on: 6/6 { Felicia /2023 start date: 6/6/2023 quantity: 16 g { Maxwell  refill:  { Jenny 1 { Micah 1 refi { Isabelle lls by 6/5/2024 gabapentin 30 { Ada 0 mg capsule (neurontin) discont { Briana inued by: lippard, giles a, aprn discontinued on: 8/1/2023 reason for discontinuation: reorder { Tanner  instructions: take 1 capsule (300 mg total { Marguerite ) by mouth  { Jamie every night. dose reduced given  { Lonnie worsening renal failure authorized by { Angelica : lippard, giles a, aprn ordered on: 6/7/2023 start date: 6/7/2023 quantity: 90 capsule refill:  remaining dulaglutide 1.5 mg/0.5 ml s { Marshall ubcutaneous pen i { Bridget njector (trulicity) discontinued by: wi { Tracey lson, danya horchi, pha { Bob rmd discontinued on: 8/7/2023 reason for { Abraham  discontinuati { Arianna on: dose adjustment  { Madelyn (cancelrx) instructions: inject 1.5 mg und { Adriana er { Andy  the skin weekly. authorized by: lippard, giles a, aprn ordere { Brady d on: 7/10/20 { Guy 23 start date: 7/10/2023 end date: 8/7/2023 quantity: 2 ml refill: 5 refills by 7/9/2024 stopped in visit medications last reviewed by lippard, giles a, aprn on 6/9/2023 1130 freestyl { Nicolas e libre 2 sensor kit (flash glucose sensor) discontinued by: smith, rosa maria, rn discontinued on: 7/31/2023 reason for discontinuation: other (cancelrx) clinical notes addendum note smith, rosa maria, rn at 7/24/2023 1329 author: smith, rosa maria, rn  { Jackie service: author ty { Candace pe: registered nurse filed: 7/31/2023 3:26 pm encounter date: 7/24/2023 status: signed editor:  { Hope smith, rosa maria, rn (registered nurse) addended by: smith, rosa on: 7/31/2023 03:26 pm printed on 10/3/24 7:13 am page 2 { Karla 034",
    {
        "entities": [
            [
                39,
                47,
                "PERSON"
            ],
            [
                278,
                285,
                "PERSON"
            ],
            [
                330,
                337,
                "PERSON"
            ],
            [
                350,
                357,
                "PERSON"
            ],
            [
                442,
                450,
                "PERSON"
            ],
            [
                483,
                490,
                "PERSON"
            ],
            [
                498,
                505,
                "PERSON"
            ],
            [
                524,
                531,
                "PERSON"
            ],
            [
                684,
                689,
                "PERSON"
            ],
            [
                722,
                729,
                "PERSON"
            ],
            [
                757,
                765,
                "PERSON"
            ],
            [
                847,
                854,
                "PERSON"
            ],
            [
                981,
                989,
                "PERSON"
            ],
            [
                997,
                1003,
                "PERSON"
            ],
            [
                1018,
                1025,
                "PERSON"
            ],
            [
                1083,
                1089,
                "PERSON"
            ],
            [
                1133,
                1137,
                "PERSON"
            ],
            [
                1283,
                1290,
                "PERSON"
            ],
            [
                1295,
                1302,
                "PERSON"
            ],
            [
                1306,
                1313,
                "PERSON"
            ],
            [
                1366,
                1374,
                "PERSON"
            ],
            [
                1430,
                1438,
                "PERSON"
            ],
            [
                1627,
                1634,
                "PERSON"
            ],
            [
                1667,
                1675,
                "PERSON"
            ],
            [
                1714,
                1720,
                "PERSON"
            ],
            [
                1794,
                1800,
                "PERSON"
            ],
            [
                1888,
                1895,
                "PERSON"
            ],
            [
                2050,
                2056,
                "PERSON"
            ],
            [
                2211,
                2216,
                "PERSON"
            ],
            [
                2333,
                2337,
                "PERSON"
            ],
            [
                2353,
                2360,
                "PERSON"
            ],
            [
                2416,
                2421,
                "PERSON"
            ],
            [
                2463,
                2472,
                "PERSON"
            ],
            [
                2539,
                2546,
                "PERSON"
            ],
            [
                2713,
                2719,
                "PERSON"
            ],
            [
                2724,
                2732,
                "PERSON"
            ],
            [
                2818,
                2826,
                "PERSON"
            ],
            [
                2868,
                2872,
                "PERSON"
            ],
            [
                2916,
                2923,
                "PERSON"
            ],
            [
                2950,
                2957,
                "PERSON"
            ],
            [
                3075,
                3082,
                "PERSON"
            ],
            [
                3125,
                3130,
                "PERSON"
            ],
            [
                3159,
                3165,
                "PERSON"
            ],
            [
                3232,
                3240,
                "PERSON"
            ],
            [
                3403,
                3410,
                "PERSON"
            ],
            [
                3415,
                3420,
                "PERSON"
            ],
            [
                3511,
                3517,
                "PERSON"
            ],
            [
                3634,
                3642,
                "PERSON"
            ],
            [
                3646,
                3654,
                "PERSON"
            ],
            [
                3677,
                3684,
                "PERSON"
            ],
            [
                3719,
                3727,
                "PERSON"
            ],
            [
                3787,
                3794,
                "PERSON"
            ],
            [
                3808,
                3815,
                "PERSON"
            ],
            [
                3821,
                3828,
                "PERSON"
            ],
            [
                3842,
                3853,
                "PERSON"
            ],
            [
                3881,
                3888,
                "PERSON"
            ],
            [
                3951,
                3958,
                "PERSON"
            ],
            [
                3994,
                4001,
                "PERSON"
            ],
            [
                4257,
                4263,
                "PERSON"
            ],
            [
                4323,
                4328,
                "PERSON"
            ],
            [
                4436,
                4443,
                "PERSON"
            ],
            [
                4485,
                4491,
                "PERSON"
            ],
            [
                4534,
                4541,
                "PERSON"
            ],
            [
                4679,
                4684,
                "PERSON"
            ],
            [
                4706,
                4713,
                "PERSON"
            ],
            [
                4744,
                4754,
                "PERSON"
            ],
            [
                4843,
                4847,
                "PERSON"
            ],
            [
                4881,
                4889,
                "PERSON"
            ],
            [
                4903,
                4912,
                "PERSON"
            ],
            [
                4939,
                4945,
                "PERSON"
            ],
            [
                4996,
                5003,
                "PERSON"
            ],
            [
                5017,
                5025,
                "PERSON"
            ],
            [
                5068,
                5076,
                "PERSON"
            ],
            [
                5120,
                5128,
                "PERSON"
            ],
            [
                5140,
                5146,
                "PERSON"
            ],
            [
                5150,
                5156,
                "PERSON"
            ],
            [
                5165,
                5174,
                "PERSON"
            ],
            [
                5206,
                5210,
                "PERSON"
            ],
            [
                5245,
                5252,
                "PERSON"
            ],
            [
                5349,
                5356,
                "PERSON"
            ],
            [
                5402,
                5413,
                "PERSON"
            ],
            [
                5427,
                5433,
                "PERSON"
            ],
            [
                5468,
                5475,
                "PERSON"
            ],
            [
                5515,
                5524,
                "PERSON"
            ],
            [
                5661,
                5670,
                "PERSON"
            ],
            [
                5690,
                5698,
                "PERSON"
            ],
            [
                5740,
                5747,
                "PERSON"
            ],
            [
                5773,
                5777,
                "PERSON"
            ],
            [
                5820,
                5828,
                "PERSON"
            ],
            [
                5845,
                5853,
                "PERSON"
            ],
            [
                5876,
                5884,
                "PERSON"
            ],
            [
                5929,
                5937,
                "PERSON"
            ],
            [
                5942,
                5947,
                "PERSON"
            ],
            [
                6012,
                6018,
                "PERSON"
            ],
            [
                6034,
                6038,
                "PERSON"
            ],
            [
                6224,
                6232,
                "PERSON"
            ],
            [
                6489,
                6496,
                "PERSON"
            ],
            [
                6517,
                6525,
                "PERSON"
            ],
            [
                6623,
                6628,
                "PERSON"
            ],
            [
                6753,
                6759,
                "PERSON"
            ]
        ]
    }
),(
    "vumc hendersonv { Claude ille - anderson white, { Pedro  tyrone 128 n anderson ln mrn: 047717361, dob: 7/27/1969, { Jillian  legal sex: m hende { Brittney rsonville tn 37075 visit date: 3/6/2023 03/06 { Rachael /2023 - office visit in vanderbilt  { Damian primary care  { Lola hendersonville (continued) flowsheets (continued) perc { Ross e { Kate nt of ibw - 3745.36 percent -sl { Layla  at  { Jade 03/06/23 1404 ebw (kg) 3021.71 kg -sl at 03/06/23 1404  { Harriet ebw (lbs) 3012.88 lbs -sl at  { Bobbie 03/06/23 1404 weight change - 85.71 kg -sl at since p { Misty re { Cooper op 03/06/2 { Kurt 3 1404 i { Kristine n { Aaliyah itial excess - -80.74 kg -sl at weight 03/06/23 1404 percent of ibw - 106.18 percent -sl at 03/06/23 1 { Geneva 404 ebw (kg) - 4.97 kg -sl at 03/06/23 1404 ebw (lb) - 11 lb -sl at 03/06/23 1404 volume estimates f { Desiree luid 0 -sl at 03/06/23 1404 resuscitation (#4) vital signs bmi (calcul { Dakota ate { Shelly d) - 25.6 -sl at 03/06/23 1404 ratio-based meal dosing app { Becky rox predicted - 5.8 -sl  { Jacquelyn at 03/06/23 1404 ratio (500 / wt in kg) rec s { Kelly tart sliding 35 -sl { Blanche  at 03/06/23 1404 sc { Velma ale (3000 / wt in kg) b { Krystal asic  { Raul information approx pred { Brad icted - 42.9 -sl at 03/06/23 basal  { Ben (0.5 wt in 1404 kg) rec st { Sherri art basal - 21.4 -sl at 03/06/23 (.25 wt in kg) 1404 rec start fixed - 7.1 -sl at 03/06/23 1404 me { Andres al (.08 wt in kg) fixed meal  { Susie in { Fannie sulin dosin { Sidney g approx predicted - 17.5 -sl at 03/06/23 cor { Zoey rection (1500 / 1404 wt in kg) weight and grow { Miles th recommendation ibw/kg 73.1 kg -sl at 03/06/23 (calculated) 1404 female he { Breanna ight  { Monique and weight wei { Hugh ght in (k { Lula g) t { Iris o 83.6 -sl at 03/06/23 have bmi = 25 { Mathew  1404 height and weight weight in (lb) to 1 { Byron 83.9  { Jodi -sl at { Casey  03/0 { Tim 6/23 have bmi = 25 1404 relevant la { Elias bs and vitals temp (in celsius) - 36.8 -sl at 03 { Rick / { Wallace 06/23 1404 adult ibw/ { Harper vt calculations ibw/kg - 77.6 -sl at 03/06/2 { Summer 3 (calculated) 1404  { Grayson low ran { Dalton ge vt - 465.6 ml/kg -sl at 03/06/23 1404 printed on 10/3 { Lynda /24 7:13 am p { Harrison age 2549,vumc hendersonville -  { Krista anderson white, tyron { Hilda e 128 n anders { Tyrone on ln mrn: 047717361, dob: 7/27/1969, legal sex: m { Jaxon  hendersonville tn 37075 visit da { Rafael te: 3/6/2023 03/06/2023 { Gabriela  - off { Nevaeh ice visit in vanderbilt primary care hendersonv { Sheryl ille (cont { Drew inue { Kristi d) flowsheets (continue { Rosie d) { Elena  6ml/kg adult moderate 620.8 ml/kg -sl at range v { Angelo t 8ml/kg 03/06/23 1404 adult high  { Eunice range - 7 { Shelley 76 ml/kg -sl at vt 10ml/kg 03/0 { Nelson 6/23 1404 encounter vitals row name  { Antoinette 03/06/23 { Dwayne  1402 encounter vi { Dwight tals bp 137/90 -sl at 03/06/23 1404 pulse 84 -sl at 03/06/23 1404 temp 36.8 °c (98.2 °f { Kelli ) - sl at 03/06/23 1404 spo2 99 % -sl at 03/06/23 1404 weight 85.7 k { Julius g (189 lb) -sl at 03/06/23 1404 height 182 { Jackie .9 cm (72\" { Trinity ) -sl at 03/06/23 1404 lund-browder (adult) row na { Shaun me 03/06/23 1402 volume est { Meredith im { Kaleb ates fluid 0 -sl at 03/06/23 1404 resuscitation (#5) fluid 0 -sl at 03/06/23 1404 { Sergio  resuscitation (#6) fluid 0 -sl  { Collin at  { Rebekah 03 { Greg /06/23 1404 resuscitation (#7) fluid 0 -sl at 03/ { Emmanuel 06/ { Mamie 23 1 { Willard 404 resuscitation (#8) fluid 0 -sl at 03/06/23  { Ramon 1404 resuscitation (#9 { Perry ) fluid 0 -sl at 03/0 { Bianca 6/23 1404 resuscitation (#10) pa { Kristy in ques { Candice tions  { Jaden row name 03/06/23 1356 pain assessment is the p { Belinda atient no -sl at 03/06/23 1356 having pain today? personal  { Gracie safety row name office visit from 3/6/2023 in vanderbilt primary care hen { Devon dersonville printed on 10/3/24 7:13 am page 255 { Ted 0",
    {
        "entities": [
            [
                18,
                25,
                "PERSON"
            ],
            [
                50,
                56,
                "PERSON"
            ],
            [
                116,
                124,
                "PERSON"
            ],
            [
                146,
                155,
                "PERSON"
            ],
            [
                203,
                211,
                "PERSON"
            ],
            [
                249,
                256,
                "PERSON"
            ],
            [
                272,
                277,
                "PERSON"
            ],
            [
                334,
                339,
                "PERSON"
            ],
            [
                343,
                348,
                "PERSON"
            ],
            [
                382,
                388,
                "PERSON"
            ],
            [
                395,
                400,
                "PERSON"
            ],
            [
                458,
                466,
                "PERSON"
            ],
            [
                498,
                505,
                "PERSON"
            ],
            [
                561,
                567,
                "PERSON"
            ],
            [
                572,
                579,
                "PERSON"
            ],
            [
                592,
                597,
                "PERSON"
            ],
            [
                608,
                617,
                "PERSON"
            ],
            [
                621,
                629,
                "PERSON"
            ],
            [
                734,
                741,
                "PERSON"
            ],
            [
                844,
                852,
                "PERSON"
            ],
            [
                925,
                932,
                "PERSON"
            ],
            [
                938,
                945,
                "PERSON"
            ],
            [
                1006,
                1012,
                "PERSON"
            ],
            [
                1039,
                1049,
                "PERSON"
            ],
            [
                1097,
                1103,
                "PERSON"
            ],
            [
                1125,
                1133,
                "PERSON"
            ],
            [
                1156,
                1162,
                "PERSON"
            ],
            [
                1188,
                1196,
                "PERSON"
            ],
            [
                1204,
                1209,
                "PERSON"
            ],
            [
                1235,
                1240,
                "PERSON"
            ],
            [
                1278,
                1282,
                "PERSON"
            ],
            [
                1311,
                1318,
                "PERSON"
            ],
            [
                1419,
                1426,
                "PERSON"
            ],
            [
                1458,
                1464,
                "PERSON"
            ],
            [
                1469,
                1476,
                "PERSON"
            ],
            [
                1490,
                1497,
                "PERSON"
            ],
            [
                1545,
                1550,
                "PERSON"
            ],
            [
                1599,
                1605,
                "PERSON"
            ],
            [
                1684,
                1692,
                "PERSON"
            ],
            [
                1700,
                1708,
                "PERSON"
            ],
            [
                1725,
                1730,
                "PERSON"
            ],
            [
                1742,
                1747,
                "PERSON"
            ],
            [
                1754,
                1759,
                "PERSON"
            ],
            [
                1798,
                1805,
                "PERSON"
            ],
            [
                1851,
                1857,
                "PERSON"
            ],
            [
                1865,
                1870,
                "PERSON"
            ],
            [
                1879,
                1885,
                "PERSON"
            ],
            [
                1893,
                1897,
                "PERSON"
            ],
            [
                1935,
                1941,
                "PERSON"
            ],
            [
                1992,
                1997,
                "PERSON"
            ],
            [
                2001,
                2009,
                "PERSON"
            ],
            [
                2033,
                2040,
                "PERSON"
            ],
            [
                2087,
                2094,
                "PERSON"
            ],
            [
                2117,
                2125,
                "PERSON"
            ],
            [
                2135,
                2142,
                "PERSON"
            ],
            [
                2201,
                2207,
                "PERSON"
            ],
            [
                2223,
                2232,
                "PERSON"
            ],
            [
                2266,
                2273,
                "PERSON"
            ],
            [
                2297,
                2303,
                "PERSON"
            ],
            [
                2320,
                2327,
                "PERSON"
            ],
            [
                2380,
                2386,
                "PERSON"
            ],
            [
                2422,
                2429,
                "PERSON"
            ],
            [
                2455,
                2464,
                "PERSON"
            ],
            [
                2473,
                2480,
                "PERSON"
            ],
            [
                2530,
                2537,
                "PERSON"
            ],
            [
                2550,
                2555,
                "PERSON"
            ],
            [
                2562,
                2569,
                "PERSON"
            ],
            [
                2595,
                2601,
                "PERSON"
            ],
            [
                2606,
                2612,
                "PERSON"
            ],
            [
                2664,
                2671,
                "PERSON"
            ],
            [
                2708,
                2715,
                "PERSON"
            ],
            [
                2727,
                2735,
                "PERSON"
            ],
            [
                2769,
                2776,
                "PERSON"
            ],
            [
                2815,
                2826,
                "PERSON"
            ],
            [
                2837,
                2844,
                "PERSON"
            ],
            [
                2865,
                2872,
                "PERSON"
            ],
            [
                2962,
                2968,
                "PERSON"
            ],
            [
                3039,
                3046,
                "PERSON"
            ],
            [
                3091,
                3098,
                "PERSON"
            ],
            [
                3111,
                3119,
                "PERSON"
            ],
            [
                3172,
                3178,
                "PERSON"
            ],
            [
                3208,
                3217,
                "PERSON"
            ],
            [
                3222,
                3228,
                "PERSON"
            ],
            [
                3312,
                3319,
                "PERSON"
            ],
            [
                3354,
                3361,
                "PERSON"
            ],
            [
                3367,
                3375,
                "PERSON"
            ],
            [
                3380,
                3385,
                "PERSON"
            ],
            [
                3437,
                3446,
                "PERSON"
            ],
            [
                3452,
                3458,
                "PERSON"
            ],
            [
                3465,
                3473,
                "PERSON"
            ],
            [
                3523,
                3529,
                "PERSON"
            ],
            [
                3554,
                3560,
                "PERSON"
            ],
            [
                3584,
                3591,
                "PERSON"
            ],
            [
                3626,
                3633,
                "PERSON"
            ],
            [
                3643,
                3651,
                "PERSON"
            ],
            [
                3660,
                3666,
                "PERSON"
            ],
            [
                3716,
                3724,
                "PERSON"
            ],
            [
                3786,
                3793,
                "PERSON"
            ],
            [
                3869,
                3875,
                "PERSON"
            ],
            [
                3925,
                3929,
                "PERSON"
            ]
        ]
    }
),(
    "vumc hendersonv { Claude ille - anderson white, { Pedro  tyrone 128 n anderson ln mrn: 047717361, dob: 7/27/1969, { Jillian  legal sex: m hende { Brittney rsonville tn 37075 visit date: 3/6/2023 03/06 { Rachael /2023 - office visit in vanderbilt  { Damian primary care  { Lola hendersonville (continued) flowsheets (continued) perc { Ross e { Kate nt of ibw - 3745.36 percent -sl { Layla  at  { Jade 03/06/23 1404 ebw (kg) 3021.71 kg -sl at 03/06/23 1404  { Harriet ebw (lbs) 3012.88 lbs -sl at  { Bobbie 03/06/23 1404 weight change - 85.71 kg -sl at since p { Misty re { Cooper op 03/06/2 { Kurt 3 1404 i { Kristine n { Aaliyah itial excess - -80.74 kg -sl at weight 03/06/23 1404 percent of ibw - 106.18 percent -sl at 03/06/23 1 { Geneva 404 ebw (kg) - 4.97 kg -sl at 03/06/23 1404 ebw (lb) - 11 lb -sl at 03/06/23 1404 volume estimates f { Desiree luid 0 -sl at 03/06/23 1404 resuscitation (#4) vital signs bmi (calcul { Dakota ate { Shelly d) - 25.6 -sl at 03/06/23 1404 ratio-based meal dosing app { Becky rox predicted - 5.8 -sl  { Jacquelyn at 03/06/23 1404 ratio (500 / wt in kg) rec s { Kelly tart sliding 35 -sl { Blanche  at 03/06/23 1404 sc { Velma ale (3000 / wt in kg) b { Krystal asic  { Raul information approx pred { Brad icted - 42.9 -sl at 03/06/23 basal  { Ben (0.5 wt in 1404 kg) rec st { Sherri art basal - 21.4 -sl at 03/06/23 (.25 wt in kg) 1404 rec start fixed - 7.1 -sl at 03/06/23 1404 me { Andres al (.08 wt in kg) fixed meal  { Susie in { Fannie sulin dosin { Sidney g approx predicted - 17.5 -sl at 03/06/23 cor { Zoey rection (1500 / 1404 wt in kg) weight and grow { Miles th recommendation ibw/kg 73.1 kg -sl at 03/06/23 (calculated) 1404 female he { Breanna ight  { Monique and weight wei { Hugh ght in (k { Lula g) t { Iris o 83.6 -sl at 03/06/23 have bmi = 25 { Mathew  1404 height and weight weight in (lb) to 1 { Byron 83.9  { Jodi -sl at { Casey  03/0 { Tim 6/23 have bmi = 25 1404 relevant la { Elias bs and vitals temp (in celsius) - 36.8 -sl at 03 { Rick / { Wallace 06/23 1404 adult ibw/ { Harper vt calculations ibw/kg - 77.6 -sl at 03/06/2 { Summer 3 (calculated) 1404  { Grayson low ran { Dalton ge vt - 465.6 ml/kg -sl at 03/06/23 1404 printed on 10/3 { Lynda /24 7:13 am p { Harrison age 2549,vumc hendersonville -  { Krista anderson white, tyron { Hilda e 128 n anders { Tyrone on ln mrn: 047717361, dob: 7/27/1969, legal sex: m { Jaxon  hendersonville tn 37075 visit da { Rafael te: 3/6/2023 03/06/2023 { Gabriela  - off { Nevaeh ice visit in vanderbilt primary care hendersonv { Sheryl ille (cont { Drew inue { Kristi d) flowsheets (continue { Rosie d) { Elena  6ml/kg adult moderate 620.8 ml/kg -sl at range v { Angelo t 8ml/kg 03/06/23 1404 adult high  { Eunice range - 7 { Shelley 76 ml/kg -sl at vt 10ml/kg 03/0 { Nelson 6/23 1404 encounter vitals row name  { Antoinette 03/06/23 { Dwayne  1402 encounter vi { Dwight tals bp 137/90 -sl at 03/06/23 1404 pulse 84 -sl at 03/06/23 1404 temp 36.8 °c (98.2 °f { Kelli ) - sl at 03/06/23 1404 spo2 99 % -sl at 03/06/23 1404 weight 85.7 k { Julius g (189 lb) -sl at 03/06/23 1404 height 182 { Jackie .9 cm (72\" { Trinity ) -sl at 03/06/23 1404 lund-browder (adult) row na { Shaun me 03/06/23 1402 volume est { Meredith im { Kaleb ates fluid 0 -sl at 03/06/23 1404 resuscitation (#5) fluid 0 -sl at 03/06/23 1404 { Sergio  resuscitation (#6) fluid 0 -sl  { Collin at  { Rebekah 03 { Greg /06/23 1404 resuscitation (#7) fluid 0 -sl at 03/ { Emmanuel 06/ { Mamie 23 1 { Willard 404 resuscitation (#8) fluid 0 -sl at 03/06/23  { Ramon 1404 resuscitation (#9 { Perry ) fluid 0 -sl at 03/0 { Bianca 6/23 1404 resuscitation (#10) pa { Kristy in ques { Candice tions  { Jaden row name 03/06/23 1356 pain assessment is the p { Belinda atient no -sl at 03/06/23 1356 having pain today? personal  { Gracie safety row name office visit from 3/6/2023 in vanderbilt primary care hen { Devon dersonville printed on 10/3/24 7:13 am page 255 { Ted 0",
    {
        "entities": [
            [
                18,
                25,
                "PERSON"
            ],
            [
                50,
                56,
                "PERSON"
            ],
            [
                116,
                124,
                "PERSON"
            ],
            [
                146,
                155,
                "PERSON"
            ],
            [
                203,
                211,
                "PERSON"
            ],
            [
                249,
                256,
                "PERSON"
            ],
            [
                272,
                277,
                "PERSON"
            ],
            [
                334,
                339,
                "PERSON"
            ],
            [
                343,
                348,
                "PERSON"
            ],
            [
                382,
                388,
                "PERSON"
            ],
            [
                395,
                400,
                "PERSON"
            ],
            [
                458,
                466,
                "PERSON"
            ],
            [
                498,
                505,
                "PERSON"
            ],
            [
                561,
                567,
                "PERSON"
            ],
            [
                572,
                579,
                "PERSON"
            ],
            [
                592,
                597,
                "PERSON"
            ],
            [
                608,
                617,
                "PERSON"
            ],
            [
                621,
                629,
                "PERSON"
            ],
            [
                734,
                741,
                "PERSON"
            ],
            [
                844,
                852,
                "PERSON"
            ],
            [
                925,
                932,
                "PERSON"
            ],
            [
                938,
                945,
                "PERSON"
            ],
            [
                1006,
                1012,
                "PERSON"
            ],
            [
                1039,
                1049,
                "PERSON"
            ],
            [
                1097,
                1103,
                "PERSON"
            ],
            [
                1125,
                1133,
                "PERSON"
            ],
            [
                1156,
                1162,
                "PERSON"
            ],
            [
                1188,
                1196,
                "PERSON"
            ],
            [
                1204,
                1209,
                "PERSON"
            ],
            [
                1235,
                1240,
                "PERSON"
            ],
            [
                1278,
                1282,
                "PERSON"
            ],
            [
                1311,
                1318,
                "PERSON"
            ],
            [
                1419,
                1426,
                "PERSON"
            ],
            [
                1458,
                1464,
                "PERSON"
            ],
            [
                1469,
                1476,
                "PERSON"
            ],
            [
                1490,
                1497,
                "PERSON"
            ],
            [
                1545,
                1550,
                "PERSON"
            ],
            [
                1599,
                1605,
                "PERSON"
            ],
            [
                1684,
                1692,
                "PERSON"
            ],
            [
                1700,
                1708,
                "PERSON"
            ],
            [
                1725,
                1730,
                "PERSON"
            ],
            [
                1742,
                1747,
                "PERSON"
            ],
            [
                1754,
                1759,
                "PERSON"
            ],
            [
                1798,
                1805,
                "PERSON"
            ],
            [
                1851,
                1857,
                "PERSON"
            ],
            [
                1865,
                1870,
                "PERSON"
            ],
            [
                1879,
                1885,
                "PERSON"
            ],
            [
                1893,
                1897,
                "PERSON"
            ],
            [
                1935,
                1941,
                "PERSON"
            ],
            [
                1992,
                1997,
                "PERSON"
            ],
            [
                2001,
                2009,
                "PERSON"
            ],
            [
                2033,
                2040,
                "PERSON"
            ],
            [
                2087,
                2094,
                "PERSON"
            ],
            [
                2117,
                2125,
                "PERSON"
            ],
            [
                2135,
                2142,
                "PERSON"
            ],
            [
                2201,
                2207,
                "PERSON"
            ],
            [
                2223,
                2232,
                "PERSON"
            ],
            [
                2266,
                2273,
                "PERSON"
            ],
            [
                2297,
                2303,
                "PERSON"
            ],
            [
                2320,
                2327,
                "PERSON"
            ],
            [
                2380,
                2386,
                "PERSON"
            ],
            [
                2422,
                2429,
                "PERSON"
            ],
            [
                2455,
                2464,
                "PERSON"
            ],
            [
                2473,
                2480,
                "PERSON"
            ],
            [
                2530,
                2537,
                "PERSON"
            ],
            [
                2550,
                2555,
                "PERSON"
            ],
            [
                2562,
                2569,
                "PERSON"
            ],
            [
                2595,
                2601,
                "PERSON"
            ],
            [
                2606,
                2612,
                "PERSON"
            ],
            [
                2664,
                2671,
                "PERSON"
            ],
            [
                2708,
                2715,
                "PERSON"
            ],
            [
                2727,
                2735,
                "PERSON"
            ],
            [
                2769,
                2776,
                "PERSON"
            ],
            [
                2815,
                2826,
                "PERSON"
            ],
            [
                2837,
                2844,
                "PERSON"
            ],
            [
                2865,
                2872,
                "PERSON"
            ],
            [
                2962,
                2968,
                "PERSON"
            ],
            [
                3039,
                3046,
                "PERSON"
            ],
            [
                3091,
                3098,
                "PERSON"
            ],
            [
                3111,
                3119,
                "PERSON"
            ],
            [
                3172,
                3178,
                "PERSON"
            ],
            [
                3208,
                3217,
                "PERSON"
            ],
            [
                3222,
                3228,
                "PERSON"
            ],
            [
                3312,
                3319,
                "PERSON"
            ],
            [
                3354,
                3361,
                "PERSON"
            ],
            [
                3367,
                3375,
                "PERSON"
            ],
            [
                3380,
                3385,
                "PERSON"
            ],
            [
                3437,
                3446,
                "PERSON"
            ],
            [
                3452,
                3458,
                "PERSON"
            ],
            [
                3465,
                3473,
                "PERSON"
            ],
            [
                3523,
                3529,
                "PERSON"
            ],
            [
                3554,
                3560,
                "PERSON"
            ],
            [
                3584,
                3591,
                "PERSON"
            ],
            [
                3626,
                3633,
                "PERSON"
            ],
            [
                3643,
                3651,
                "PERSON"
            ],
            [
                3660,
                3666,
                "PERSON"
            ],
            [
                3716,
                3724,
                "PERSON"
            ],
            [
                3786,
                3793,
                "PERSON"
            ],
            [
                3869,
                3875,
                "PERSON"
            ],
            [
                3925,
                3929,
                "PERSON"
            ]
        ]
    }
),(
    "vumc { Santiago  adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 visit date: 6/14/2024 06/14/2024 - medication management in vumc popul { Tracy ation health pharmacy services { Julio  vir (continued) medication list (continued) authorized by: valen { Kaden zuela, daniel ale { Abby jand { Alana ro, md or { Daniela dered on: 7/22/2024 start date: 7/22/2024 quantity: 3 ml refill: 1 refill { Kaitlin  by 7/22/2025 ketorolac 0.4 % eye drops (acular) discontinued by: gingrow, barbara, lpn discontinued on: { Myra  9/16/2024 reason for discontinuation: duplicate orde { Roxanne r instructions: after surgery, use 1 dro { Caden p to the right eye every  { Damien 2 hours while a { Annabelle wake until bedtime.  { Karina beginning the next day, decrease to 1 drop to the right eye 4 times a day fo { Donnie r 2 weeks then stop authorized by: valenzuela, daniel alejandro,  { Kendall md ordered on: 7/22/2024 start date: 7/22/2024 end date: 9/16/2024 quantity:  { Homer 5 ml refill: 1 refill by 7/22/2025 clotrimazole 1 % topical cream (lot { Lyle rimin) instructions: apply 1 appli { Mallory cation topically 2 times a day for 30 { Callie  days. authorized by: pauw, { Juliana  emily kathryn, md ordered on: 7/26/2024 start date: 7/26/2024 quantity: 28 g refill:  remaining lidocaine hcl 2 % mucosal jelly (xylocaine  { Josie jelly) inst { Celeste ructions: apply topically as neede { Mckenzie d for mild pain (apply to toe). authori { Adeline zed by: pauw, emily kathryn, md ordered on: 7/26/2024 st { Cassidy art date: 7/26/2024 en { Gerard d date: 8/25/2024 quanti { Aurora ty: 30 ml refill:  remaining stopped in visit none clin { Rex ical notes progress notes wi { Enrique lson, danya horchi, pharmd at 6/1 { Otis 4/2024 0835 author: wilson, danya horchi, pharmd service: author type: pharmacist  { Lilly filed: 7/26/2024 12:21 pm encounter date: 6/14/2024  { Gage status: signed editor: wilson, danya horchi, pharmd (pharmacist) missed call an { Jody d vm from mr. white this am. edited transcript { Earnest ion b { Donovan elow \"this is mr. white. i am cal { Janelle ling to have a conversation  { Serenity with yo { Trenton u. i might have to go in and check myself into the  { Aimee hospital to { Kirsten  do dialy { Asher si { Hubert s. i am  { Dianna losing we { Avery ight so i don't wanna die. so if you could call me back and tell me who to call. thank you, i { Cecelia 'll b { Dominique e in the house all day so you can reach me on the phone at 260-2291\" electronically signed by wilson, danya horchi, pharmd at 7/26/2024  { Mya 12:21 pm wilson, danya horchi, pharmd at 6/14/2024 0835 author: w { Emmett ilson, danya horchi, pharmd service: author type: pharmacist filed: 7/26/2024 12:21 pm { Olive  encounter date: 6/14/2024 status: signed editor: wils { Wilbur on, danya horchi, pharm { Lacey d (pharma { Liliana ci { Mateo st) good morning,  { Salvatore i had a missed call from mr. white this mo { Cara rning. h { Muriel e i { Alfredo s fairly concerned { Neal  about weight lo { Mikayla ss he is experiencing and { Eliza  he want { Hayley s to discuss urgency of dialysis w { Nadine i { Bernadette th someone. he was last seen by nephrology clinic 12/2023. p { Celia rinted on 10/3/24 7:12 am page 787,vumc adult hospital white, tyrone 1211 medical center  { Arturo dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashv { Cristina ille tn 37232-0004 visit date: 6/14/2024 06/14/2024 - medication management  { Jan in vumc pop { Jordyn ulation health ph { Lana armacy services vir (conti { Erma nued) clinical notes (continued) thank you, danya h. wilson, pharmd po { Dana pul { Jaclyn ation health clinical pharmacist 615-322- { Vicky 4663 ele { Caitlyn ctronically signed by wilson, danya horchi, pharmd at 7/26/2024 12:21 pm neitz, rose a, { Alec  rn at 6/14/2024 0835 auth { Gretchen or: neitz, rose a, rn service: author type: registered nurse filed: 7/2 { Leland 6/2024 12:21 pm encounter da { Johanna te: 6/14/2024 status: signed editor: neitz, rose a, rn (registered nurse) nephrol { Horace o { Shannon gy new patient note name: tyrone white mrn: 047717361 dob: 7/27/19 { Jaime 69  { Colby refe { Joey rring provider: . provider found chief complaint: subjective interval { Abel  history/history of present illness: tyrone white is a 54 y.o.  { Traci year old { Kellie  male who presents with past medical history: diag { Estelle nosis date an { Kerry emia bell's palsy chronic renal failure, stage 3b (cms/hcc) hyperka { Ira lemia 11/ { Conner 09/2022 hypertension hypoalbuminemia 11/09/2022 type 2 diabetes mellitus wi { Payton th complications (cms/hcc) 2001 { Wendell  approxima { Ezra te date of  { Trent ons { Eloise et urine test positive for { Rudolph  microalbuminuria 11/09/2022  medical history pertinent negatives. past surgical history: procedure laterality date ankle surgery right exploratory laparotomy gunshot wound social history  { Aria printed on 10/3/24  { Ivy 7:12 am page 78 { Graham 8",
    {
        "entities": [
            [
                7,
                16,
                "PERSON"
            ],
            [
                212,
                218,
                "PERSON"
            ],
            [
                251,
                257,
                "PERSON"
            ],
            [
                325,
                331,
                "PERSON"
            ],
            [
                351,
                356,
                "PERSON"
            ],
            [
                363,
                369,
                "PERSON"
            ],
            [
                381,
                389,
                "PERSON"
            ],
            [
                465,
                473,
                "PERSON"
            ],
            [
                580,
                585,
                "PERSON"
            ],
            [
                641,
                649,
                "PERSON"
            ],
            [
                692,
                698,
                "PERSON"
            ],
            [
                726,
                733,
                "PERSON"
            ],
            [
                751,
                761,
                "PERSON"
            ],
            [
                784,
                791,
                "PERSON"
            ],
            [
                870,
                877,
                "PERSON"
            ],
            [
                945,
                953,
                "PERSON"
            ],
            [
                1033,
                1039,
                "PERSON"
            ],
            [
                1112,
                1117,
                "PERSON"
            ],
            [
                1154,
                1162,
                "PERSON"
            ],
            [
                1202,
                1209,
                "PERSON"
            ],
            [
                1239,
                1247,
                "PERSON"
            ],
            [
                1390,
                1396,
                "PERSON"
            ],
            [
                1410,
                1418,
                "PERSON"
            ],
            [
                1455,
                1464,
                "PERSON"
            ],
            [
                1506,
                1514,
                "PERSON"
            ],
            [
                1573,
                1581,
                "PERSON"
            ],
            [
                1606,
                1613,
                "PERSON"
            ],
            [
                1640,
                1647,
                "PERSON"
            ],
            [
                1705,
                1709,
                "PERSON"
            ],
            [
                1740,
                1748,
                "PERSON"
            ],
            [
                1784,
                1789,
                "PERSON"
            ],
            [
                1874,
                1880,
                "PERSON"
            ],
            [
                1935,
                1940,
                "PERSON"
            ],
            [
                2022,
                2027,
                "PERSON"
            ],
            [
                2076,
                2084,
                "PERSON"
            ],
            [
                2092,
                2100,
                "PERSON"
            ],
            [
                2136,
                2144,
                "PERSON"
            ],
            [
                2175,
                2184,
                "PERSON"
            ],
            [
                2194,
                2202,
                "PERSON"
            ],
            [
                2256,
                2262,
                "PERSON"
            ],
            [
                2276,
                2284,
                "PERSON"
            ],
            [
                2296,
                2302,
                "PERSON"
            ],
            [
                2307,
                2314,
                "PERSON"
            ],
            [
                2325,
                2332,
                "PERSON"
            ],
            [
                2344,
                2350,
                "PERSON"
            ],
            [
                2446,
                2454,
                "PERSON"
            ],
            [
                2462,
                2472,
                "PERSON"
            ],
            [
                2611,
                2615,
                "PERSON"
            ],
            [
                2683,
                2690,
                "PERSON"
            ],
            [
                2779,
                2785,
                "PERSON"
            ],
            [
                2842,
                2849,
                "PERSON"
            ],
            [
                2875,
                2881,
                "PERSON"
            ],
            [
                2893,
                2901,
                "PERSON"
            ],
            [
                2906,
                2912,
                "PERSON"
            ],
            [
                2933,
                2943,
                "PERSON"
            ],
            [
                2988,
                2993,
                "PERSON"
            ],
            [
                3004,
                3011,
                "PERSON"
            ],
            [
                3017,
                3025,
                "PERSON"
            ],
            [
                3046,
                3051,
                "PERSON"
            ],
            [
                3070,
                3078,
                "PERSON"
            ],
            [
                3106,
                3112,
                "PERSON"
            ],
            [
                3123,
                3130,
                "PERSON"
            ],
            [
                3167,
                3174,
                "PERSON"
            ],
            [
                3178,
                3189,
                "PERSON"
            ],
            [
                3252,
                3258,
                "PERSON"
            ],
            [
                3350,
                3357,
                "PERSON"
            ],
            [
                3414,
                3423,
                "PERSON"
            ],
            [
                3502,
                3506,
                "PERSON"
            ],
            [
                3520,
                3527,
                "PERSON"
            ],
            [
                3547,
                3552,
                "PERSON"
            ],
            [
                3581,
                3586,
                "PERSON"
            ],
            [
                3659,
                3664,
                "PERSON"
            ],
            [
                3670,
                3677,
                "PERSON"
            ],
            [
                3721,
                3727,
                "PERSON"
            ],
            [
                3738,
                3746,
                "PERSON"
            ],
            [
                3836,
                3841,
                "PERSON"
            ],
            [
                3870,
                3879,
                "PERSON"
            ],
            [
                3953,
                3960,
                "PERSON"
            ],
            [
                3991,
                3999,
                "PERSON"
            ],
            [
                4083,
                4090,
                "PERSON"
            ],
            [
                4094,
                4102,
                "PERSON"
            ],
            [
                4171,
                4177,
                "PERSON"
            ],
            [
                4183,
                4189,
                "PERSON"
            ],
            [
                4196,
                4201,
                "PERSON"
            ],
            [
                4273,
                4278,
                "PERSON"
            ],
            [
                4344,
                4350,
                "PERSON"
            ],
            [
                4361,
                4368,
                "PERSON"
            ],
            [
                4421,
                4429,
                "PERSON"
            ],
            [
                4445,
                4451,
                "PERSON"
            ],
            [
                4521,
                4525,
                "PERSON"
            ],
            [
                4537,
                4544,
                "PERSON"
            ],
            [
                4622,
                4629,
                "PERSON"
            ],
            [
                4663,
                4671,
                "PERSON"
            ],
            [
                4684,
                4689,
                "PERSON"
            ],
            [
                4703,
                4709,
                "PERSON"
            ],
            [
                4715,
                4722,
                "PERSON"
            ],
            [
                4751,
                4759,
                "PERSON"
            ],
            [
                4950,
                4955,
                "PERSON"
            ],
            [
                4977,
                4981,
                "PERSON"
            ],
            [
                4999,
                5006,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 visit date: 8/15/2024 08/15/2024 - communication in virtual vumc nephrology fa { Allyson cesheet report patient demographics  { Darius patient name mrn legal dob address phone white, tyrone  { Michaela 0477 { Gerardo 173 sex 7/27/1 { Lora 969 apt 705 615-260-2291 (home) 61 m 1101 edgehill ave 615-260 { Mable -2291 (mobile) nashville tn 37203 *preferred* hospital account not on fil { Kerry e admission information current informatio { Selena n attending pr { Kelvin ovider ad { Bennie mitting  { Garry provider admission type admission status unknown status admission date/tim { Lynn e  { Rylee discharge date/time hospital service auth/cert status hospital area unit room/bed referring provider 08/15/2024 - communication in virtual vumc nephrology (continued) visit information provider i { Willis nformation encounter provider ahmad, noaman, mbbs department name address virtual vumc nephrology tn medication list medication list 1 this report is  { Kenny for documentation purposes o { Reagan nly { Tessa . the patient should not follow medication instruc { Easton tion { Kayden s within. for accurate in { Edmund struction { Rosemarie s regarding medications, the patient should instead c { Archie onsult their physician or after visit summary. active at the end of visit medications last reviewed by perry, charles r, rn on 8/14/2024 2027 famotidine 20 mg tablet (pepcid) [reconciled by ferguson, sherri l, lpn on 1/11/2023 1257] instructions: take 1 tablet (20  { Bryant mg total) by mo { Marisa uth every 12 hours. ente { Latoya red by: ferguson, sherri l, lpn entered on: 1/11/2023 atorvastatin 80 mg tablet (lipitor) discontinu { Rene ed by: mickey, lisa, lpn discontinued on: 10/2/2024 instructions: take 1 tablet (80 mg total) by { Emanuel  mouth daily. authorized by:  { Alejandra lippard, giles a, aprn ordered on: 8/ { Raquel 8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 re { Tyson fills by 8/7/2024 cyclobenzaprine 5 mg tablet (flexeril) [reconciled by maples, chantis on 9/5/2023 1522] instructions: take  { Randolph 1 tablet (5 mg total) by mouth { Henrietta  ever { Corinne y 8 hours as needed. printed on 10/3/24  { Marcos 7:12 am page 251,vumc adult hosp { Doreen ital white, tyrone 1 { Nick 211 medical center dr. mr { Kiara n: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 { Josue  visit date: 8/15/2024 08/15/2024 - communication in vi { Roderick rtual vumc nephrology (continued) medication list (continued) ente { Robin r { Janis ed by: maples, chantis entered on: 9/5/2023 start date: 7/22 { Sallie /2023 triamcinolone acetonide 55 mcg nasal spray aerosol (nasacort) discontinued by: gingrow, b { Shawna arbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate order instructions: administer 2 sprays { May  (110 mc { Chance g total) i { Cassie nto each nostril 2 times a day. authorized by: greenspan, debra l, aprn ordere { Forrest d on: 9/5/2023 start date: { Nettie  9/5/2023 end date: 9/16/2024 quantity: 16.5 g r { Braxton efill: 11 r { Carlton efills by 9/4/2024 capsaicin 0.1 % topical  { Rudy cream discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuati { Allie on: duplicate order instructions: apply 1 applica { Noel tion topically daily for 90 days. a { Betsy uthorized by: de witte, anton jordan, md ordered on: 9 { Malachi /7/2023 start date: 9/7/2023 end date: 9/16/2024 acti { Christie on: pa { Johnnie tient { Zane  not taking quantity: { Orlando  42.5 g refill { Peyton : { Rochelle   remaining aspirin 81 mg tablet,delayed release instructions: take 1 tablet (81 mg total)  { Silas by mouth daily. authorized by: de witte, a { Ashlee nton jordan, md ordered on: 9/7/2023 start d { Elisabeth ate: 9/7/2023 qu { Elliott antity: 90 ta { Kylee blet refill: 3 refills by 9/6/2024 cetirizine 10 mg tablet (zyrtec) instruc { Salvador tions: take 1 tablet (10 mg total) by mouth  { Shelia once a day as needed for { Ryder  allergie { Kelley s. authoriz { Weston ed by: de witte, anton jordan, md ordered on: 10/4/202 { Leila 3 start date: 10/4/2023 q { Grady uantity: 30 tablet refill: 9 refills by 10/3/2024 lidocaine 5 % topical p { Mollie atch (lidoderm)  { Lynette instructions: apply 1 pa { Benny tch topically dail { Laurel y. apply to painful area 12 hours per day, remove for 12 hours. authorized by: de witte, anton jordan, md ordere { Wilson d on: 10/17/2023 start date: 10/17/2023 end date: 10/16/2024 quantity: 30 patch refill: 11 refills by 10/16/2024 nifedipine er 30 mg tablet,extended release (adalat cc) instructions: take 2 tablets (60 mg total) by mouth daily. authorized by: de  { Meagan witte, ant { Ty on jordan, md or { Delbert dered on: 4/23/2024 start date: 4/23/2024 quantity: 180 tablet refill:  { Eula 3 refil { Alton ls by 4/23/2025 lantus solostar u-100 insulin 100 unit/ml (3 ml) subcutaneous pen (ins { Piper ulin glargine) instructions: inject 10 units under the skin 2 times a day. authorized by: greenspan, deb { Anastasia ra l, aprn ordered on: { Jana  7/9/2024 start dat { Loren e: 7 { Jaxson /9/2024 quantity { Braden : 18 ml refill: 3 refills by 7/9/2025 pantoprazole 20 mg tablet,delayed release (protonix) instructions: take 1 tablet (20 mg total) by mouth daily. authorized by: greenspan, debra l, aprn ordered on: 7/9/2024 sta { Ginger rt date: 7/9/2024 end date: 7/9/2025 quantity: 30 tablet refill: 11 refills by 7/9/2025 predniso { Esmeralda lone acetate 1 % eye drops,suspension (pred forte) instructions: after surg { Paulette ery, use 1 drop to the right eye every 2 hours while  { Julianna awak { Clark e until bedtime. beginning the next day, { Jasmin  decrease to  { Ashlyn 1 drop to the right eye 4 time { Mona s a d { Lottie ay for 1 week, then 3 times a day for 1 week, then 2 times a day for 1 w { Mercedes eek, then daily for 1 week, then stop printed on 10 { Adrianna /3/24 7:12 am page 252",
    {
        "entities": [
            [
                208,
                216,
                "PERSON"
            ],
            [
                255,
                262,
                "PERSON"
            ],
            [
                320,
                329,
                "PERSON"
            ],
            [
                336,
                344,
                "PERSON"
            ],
            [
                361,
                366,
                "PERSON"
            ],
            [
                431,
                437,
                "PERSON"
            ],
            [
                513,
                519,
                "PERSON"
            ],
            [
                564,
                571,
                "PERSON"
            ],
            [
                588,
                595,
                "PERSON"
            ],
            [
                607,
                614,
                "PERSON"
            ],
            [
                625,
                631,
                "PERSON"
            ],
            [
                708,
                713,
                "PERSON"
            ],
            [
                718,
                724,
                "PERSON"
            ],
            [
                922,
                929,
                "PERSON"
            ],
            [
                1082,
                1088,
                "PERSON"
            ],
            [
                1119,
                1126,
                "PERSON"
            ],
            [
                1132,
                1138,
                "PERSON"
            ],
            [
                1191,
                1198,
                "PERSON"
            ],
            [
                1205,
                1212,
                "PERSON"
            ],
            [
                1240,
                1247,
                "PERSON"
            ],
            [
                1259,
                1269,
                "PERSON"
            ],
            [
                1325,
                1332,
                "PERSON"
            ],
            [
                1600,
                1607,
                "PERSON"
            ],
            [
                1625,
                1632,
                "PERSON"
            ],
            [
                1659,
                1666,
                "PERSON"
            ],
            [
                1769,
                1774,
                "PERSON"
            ],
            [
                1873,
                1881,
                "PERSON"
            ],
            [
                1913,
                1923,
                "PERSON"
            ],
            [
                1963,
                1970,
                "PERSON"
            ],
            [
                2033,
                2039,
                "PERSON"
            ],
            [
                2167,
                2176,
                "PERSON"
            ],
            [
                2209,
                2219,
                "PERSON"
            ],
            [
                2227,
                2235,
                "PERSON"
            ],
            [
                2278,
                2285,
                "PERSON"
            ],
            [
                2320,
                2327,
                "PERSON"
            ],
            [
                2350,
                2355,
                "PERSON"
            ],
            [
                2383,
                2389,
                "PERSON"
            ],
            [
                2458,
                2464,
                "PERSON"
            ],
            [
                2522,
                2531,
                "PERSON"
            ],
            [
                2600,
                2606,
                "PERSON"
            ],
            [
                2610,
                2616,
                "PERSON"
            ],
            [
                2679,
                2686,
                "PERSON"
            ],
            [
                2784,
                2791,
                "PERSON"
            ],
            [
                2910,
                2914,
                "PERSON"
            ],
            [
                2925,
                2932,
                "PERSON"
            ],
            [
                2945,
                2952,
                "PERSON"
            ],
            [
                3033,
                3041,
                "PERSON"
            ],
            [
                3070,
                3077,
                "PERSON"
            ],
            [
                3128,
                3136,
                "PERSON"
            ],
            [
                3150,
                3158,
                "PERSON"
            ],
            [
                3204,
                3209,
                "PERSON"
            ],
            [
                3308,
                3314,
                "PERSON"
            ],
            [
                3366,
                3371,
                "PERSON"
            ],
            [
                3409,
                3415,
                "PERSON"
            ],
            [
                3472,
                3480,
                "PERSON"
            ],
            [
                3536,
                3545,
                "PERSON"
            ],
            [
                3554,
                3562,
                "PERSON"
            ],
            [
                3570,
                3575,
                "PERSON"
            ],
            [
                3599,
                3607,
                "PERSON"
            ],
            [
                3624,
                3631,
                "PERSON"
            ],
            [
                3635,
                3644,
                "PERSON"
            ],
            [
                3738,
                3744,
                "PERSON"
            ],
            [
                3789,
                3796,
                "PERSON"
            ],
            [
                3843,
                3853,
                "PERSON"
            ],
            [
                3872,
                3880,
                "PERSON"
            ],
            [
                3896,
                3902,
                "PERSON"
            ],
            [
                3980,
                3989,
                "PERSON"
            ],
            [
                4036,
                4043,
                "PERSON"
            ],
            [
                4070,
                4076,
                "PERSON"
            ],
            [
                4088,
                4095,
                "PERSON"
            ],
            [
                4109,
                4116,
                "PERSON"
            ],
            [
                4173,
                4179,
                "PERSON"
            ],
            [
                4207,
                4213,
                "PERSON"
            ],
            [
                4289,
                4296,
                "PERSON"
            ],
            [
                4315,
                4323,
                "PERSON"
            ],
            [
                4350,
                4356,
                "PERSON"
            ],
            [
                4377,
                4384,
                "PERSON"
            ],
            [
                4499,
                4506,
                "PERSON"
            ],
            [
                4755,
                4762,
                "PERSON"
            ],
            [
                4775,
                4778,
                "PERSON"
            ],
            [
                4797,
                4805,
                "PERSON"
            ],
            [
                4879,
                4884,
                "PERSON"
            ],
            [
                4894,
                4900,
                "PERSON"
            ],
            [
                4989,
                4995,
                "PERSON"
            ],
            [
                5102,
                5112,
                "PERSON"
            ],
            [
                5137,
                5142,
                "PERSON"
            ],
            [
                5164,
                5170,
                "PERSON"
            ],
            [
                5177,
                5184,
                "PERSON"
            ],
            [
                5203,
                5210,
                "PERSON"
            ],
            [
                5426,
                5433,
                "PERSON"
            ],
            [
                5532,
                5542,
                "PERSON"
            ],
            [
                5620,
                5629,
                "PERSON"
            ],
            [
                5685,
                5694,
                "PERSON"
            ],
            [
                5701,
                5707,
                "PERSON"
            ],
            [
                5750,
                5757,
                "PERSON"
            ],
            [
                5773,
                5780,
                "PERSON"
            ],
            [
                5813,
                5818,
                "PERSON"
            ],
            [
                5826,
                5833,
                "PERSON"
            ],
            [
                5908,
                5917,
                "PERSON"
            ],
            [
                5971,
                5980,
                "PERSON"
            ]
        ]
    }
),(
    "vumc  { Ezekiel adult h { Morgan ospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, l { August egal sex: m nashville tn 37232-0004 visit date: 4/3/2023 04/03/2023 - medication management in { Pablo  vumc population health pharmacy  { Mandy services vir (continued) medication list (continued) au { Sawyer thorized by: lippard, giles a, aprn ordered on: 1/11/202 { Dante 3 start date: 1/11 { Maude /2023 end date: 6/3/2023 action:  { Lorena p { Will atient not taking quantity: 20 tablet refill:   { Myron remaining ergocalciferol (vitamin d2) 1,250 mcg (50,000 unit) capsule (vitamin d2) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 re { Lucia ason for di { Teri scontinuati { Bentley on: stop taking at discharge (cancelrx) instruction { Patti s: take 1 capsule (50,000 units total) by mouth weekly. authorized by: wang, zhijian, aprn ordered on: 2/7/2023  { Serena start date: 2/7/2023 end date: 6/3/2023  { Jasper quantity: 12 capsule refill:  remaining montelukas { Nadia t 10 mg tablet (si { Camden ngulair) discontinued by: lippard, giles a, aprn discontinued on: 8/8/2023 reason for discontinuation: reorder instructions: take one tablet by mouth evening authorized by: lippard, gile { Elisa s a, aprn ordered on: 4/10/2023 st { Alexia art date: 4/10/2023 quantity: 30 tab { Elliot let refill:  remaining dulaglutide 1.5  { Giselle mg/0.5 ml subcutaneous pen injector  { Brock (trulicity) discont { Malik inued by: parker, robin paige, pharmd discontinued on: 7/10/2023 reason for discon { Mindy tinuatio { Addie n: reorder instructions: inject 1.5 mg under the skin weekly. authorized by: lip { Myles pard, g { Ernestine iles a, aprn ordered on: 4/10/2023 start date: 4/10/2023 end date: 7/10/2023 quantity: 2 ml refill: 2 refills by 4/9/2024 gabapentin 300 mg capsule (neurontin) discontinued by: lehmann, melissa cary, p { Ernesto a-c discontinued on: 6/3/2 { Carolina 023 reason  { Irving for discontinuation: re { Effie order instructions: take one capsule by mouth three times a day authorize { Katharine d by: lippard, giles a, aprn ordered on: 4/24/2023 start date: 4/24/2023 quantity: 90 capsule refill:  remaining insulin { Winifred  glargine (u-100) 100 unit/ml subcutaneous solution discontinued by: wilson, danya horchi, pharmd d { Leticia iscontinued on: 8/24/2023 re { Mila ason for discontinuati { Pete on: reorder instructions: inject 0.05 ml  { Dixie (5 units t { Kimberley otal) under the skin daily. author { Trisha ized by: lippard, giles a, { Kai  aprn ordered on: 5/10/2023 start date: 5/10/2023 action: patient taking differently quantity: 3 ml refill: 2 refills by 5/9/2024 stopped in visit none clinical notes progre { Kyla ss no { Nikki tes wilson, danya  { Sammy horchi, pharmd at 4/3/2023 1429 author: wilson, danya horchi, pharmd service: - auth { Sylvester o { Madeleine r type: pharmacist filed: 5/12/2023 2:49 pm encounter date: 4/3/2023 status: signed editor: wilson, da { Ebony nya horchi, pharmd (phar { Jermaine macist) incomi { Laverne ng phone call with tyrone white for dm follow-up pr { Darla inted on 10/3/24 7:13 am page 2467,vumc adult hospital white, tyrone 1211 medical center d { Ellis r. mrn: 047717361, dob: 7/27/1969, legal sex: m na { Freda shville tn 37232-0004 visit date: 4/ { Bennett 3/2023 04/03/2 { Quentin 023 - medication management in vumc population health pharmacy services vir (continued) clinical notes (continued { Tricia ) interval history: * pt is i { Lela n ca trying to find  { Shari housing with a friend, it did not work out, car troubles and stuc { Alonzo k in c { Cedric a * food continues to be incon { Axel sistence * li { Jazmin ving in car, states he has been in contact { Justine  with soc { Tucker ial work and he is { Leigh  on an affordable housing list * pt wants to go back on trulicity bc he did { Alaina  not have hypogly { Griffin cemia and his sugars were better contr { Frankie oled. stat { Etta es he was not eating much while on it bc his old  { Lowell roommates would e { Sonja at his food * some swelling in his lower legs, but he is unsure if it is related to sl { Adele e { Emilio eping in his car * wag on 15316 nordhoff st, north hills, ca 91343 current dm meds: rybelsus 14mg once daily lantus 10 units once { Ollie  daily smbg: per libre 2 as reported by patient: 7-day average: 211 14-day ave { Stacie rage: 207 4/1: 78 (1030am), 54  { Laurence (130pm { Stefanie ), 180 (7pm) 4/2: 302 (9pm) 4/3: 263 (230am), 181 (745am), 243 (825am), 179 (245pm) lab  { Junior results  { Raven component value d { Alfonso ate hemoglobin a1c level 8.9 03/06/2023 creatinine level { Delilah  3.33 (h) { Harley  03/06/2023 egfrcr { Aubree  21 (l) 03/06/2023 triglyceride lvl 151 (h) 08/11/2022 assessment/plan: pts bg is uncontr { Makenzie olled,  { Nickolas above a1c goal <7%. bg fluct { Kristie uates tremendously based on meal type/time for pt. meals are very inconsistent for mr. white due to food insecurity so it can be hard to gauge his control. pt having hyper and hypoglycemia bc of this especially when he takes insulin. pt wanting to switch from rybelsus/insulin to { Skyler  trulicity due to him thinking it helped him more and did  { Jewel not cause as { Saul  muc { Fabian h hypoglycemia. endo prev { Clarissa iously had concerns about trulicity due to potential decreased appetite/wt loss. when { Margarita  discussed with pt he states he w { Beau as not eating only because his old living s { Pat ituation had { Wilbert  people stealing his food. he reports he still had an { Jayla  appetite while on trulicity 3mg. will message endo. follow-up: danya h. wil { Alissa son, pharmd population health clinica { Kerri l pharmacist 615-322-4663 pri { Yesenia nted on 10/3/24 7:13 am page 24 { Moses 68",
    {
        "entities": [
            [
                8,
                16,
                "PERSON"
            ],
            [
                26,
                33,
                "PERSON"
            ],
            [
                115,
                122,
                "PERSON"
            ],
            [
                219,
                225,
                "PERSON"
            ],
            [
                261,
                267,
                "PERSON"
            ],
            [
                325,
                332,
                "PERSON"
            ],
            [
                391,
                397,
                "PERSON"
            ],
            [
                418,
                424,
                "PERSON"
            ],
            [
                460,
                467,
                "PERSON"
            ],
            [
                471,
                476,
                "PERSON"
            ],
            [
                526,
                532,
                "PERSON"
            ],
            [
                691,
                697,
                "PERSON"
            ],
            [
                711,
                716,
                "PERSON"
            ],
            [
                730,
                738,
                "PERSON"
            ],
            [
                792,
                798,
                "PERSON"
            ],
            [
                913,
                920,
                "PERSON"
            ],
            [
                963,
                970,
                "PERSON"
            ],
            [
                1023,
                1029,
                "PERSON"
            ],
            [
                1050,
                1057,
                "PERSON"
            ],
            [
                1246,
                1252,
                "PERSON"
            ],
            [
                1289,
                1296,
                "PERSON"
            ],
            [
                1335,
                1342,
                "PERSON"
            ],
            [
                1384,
                1392,
                "PERSON"
            ],
            [
                1431,
                1437,
                "PERSON"
            ],
            [
                1459,
                1465,
                "PERSON"
            ],
            [
                1550,
                1556,
                "PERSON"
            ],
            [
                1567,
                1573,
                "PERSON"
            ],
            [
                1656,
                1662,
                "PERSON"
            ],
            [
                1672,
                1682,
                "PERSON"
            ],
            [
                1886,
                1894,
                "PERSON"
            ],
            [
                1923,
                1932,
                "PERSON"
            ],
            [
                1946,
                1953,
                "PERSON"
            ],
            [
                1979,
                1985,
                "PERSON"
            ],
            [
                2061,
                2071,
                "PERSON"
            ],
            [
                2194,
                2203,
                "PERSON"
            ],
            [
                2305,
                2313,
                "PERSON"
            ],
            [
                2344,
                2349,
                "PERSON"
            ],
            [
                2374,
                2379,
                "PERSON"
            ],
            [
                2423,
                2429,
                "PERSON"
            ],
            [
                2442,
                2452,
                "PERSON"
            ],
            [
                2489,
                2496,
                "PERSON"
            ],
            [
                2525,
                2529,
                "PERSON"
            ],
            [
                2705,
                2710,
                "PERSON"
            ],
            [
                2718,
                2724,
                "PERSON"
            ],
            [
                2745,
                2751,
                "PERSON"
            ],
            [
                2838,
                2848,
                "PERSON"
            ],
            [
                2852,
                2862,
                "PERSON"
            ],
            [
                2967,
                2973,
                "PERSON"
            ],
            [
                3000,
                3009,
                "PERSON"
            ],
            [
                3026,
                3034,
                "PERSON"
            ],
            [
                3088,
                3094,
                "PERSON"
            ],
            [
                3187,
                3193,
                "PERSON"
            ],
            [
                3246,
                3252,
                "PERSON"
            ],
            [
                3291,
                3299,
                "PERSON"
            ],
            [
                3316,
                3324,
                "PERSON"
            ],
            [
                3440,
                3447,
                "PERSON"
            ],
            [
                3479,
                3484,
                "PERSON"
            ],
            [
                3507,
                3513,
                "PERSON"
            ],
            [
                3581,
                3588,
                "PERSON"
            ],
            [
                3597,
                3604,
                "PERSON"
            ],
            [
                3637,
                3642,
                "PERSON"
            ],
            [
                3658,
                3665,
                "PERSON"
            ],
            [
                3710,
                3718,
                "PERSON"
            ],
            [
                3730,
                3737,
                "PERSON"
            ],
            [
                3758,
                3764,
                "PERSON"
            ],
            [
                3842,
                3849,
                "PERSON"
            ],
            [
                3869,
                3877,
                "PERSON"
            ],
            [
                3918,
                3926,
                "PERSON"
            ],
            [
                3939,
                3944,
                "PERSON"
            ],
            [
                3996,
                4003,
                "PERSON"
            ],
            [
                4023,
                4029,
                "PERSON"
            ],
            [
                4118,
                4124,
                "PERSON"
            ],
            [
                4128,
                4135,
                "PERSON"
            ],
            [
                4267,
                4273,
                "PERSON"
            ],
            [
                4354,
                4361,
                "PERSON"
            ],
            [
                4395,
                4404,
                "PERSON"
            ],
            [
                4413,
                4422,
                "PERSON"
            ],
            [
                4513,
                4520,
                "PERSON"
            ],
            [
                4531,
                4537,
                "PERSON"
            ],
            [
                4557,
                4565,
                "PERSON"
            ],
            [
                4624,
                4632,
                "PERSON"
            ],
            [
                4644,
                4651,
                "PERSON"
            ],
            [
                4672,
                4679,
                "PERSON"
            ],
            [
                4771,
                4780,
                "PERSON"
            ],
            [
                4790,
                4799,
                "PERSON"
            ],
            [
                4830,
                4838,
                "PERSON"
            ],
            [
                5120,
                5127,
                "PERSON"
            ],
            [
                5188,
                5194,
                "PERSON"
            ],
            [
                5209,
                5214,
                "PERSON"
            ],
            [
                5221,
                5228,
                "PERSON"
            ],
            [
                5256,
                5265,
                "PERSON"
            ],
            [
                5353,
                5363,
                "PERSON"
            ],
            [
                5399,
                5404,
                "PERSON"
            ],
            [
                5450,
                5454,
                "PERSON"
            ],
            [
                5469,
                5477,
                "PERSON"
            ],
            [
                5533,
                5539,
                "PERSON"
            ],
            [
                5618,
                5625,
                "PERSON"
            ],
            [
                5665,
                5671,
                "PERSON"
            ],
            [
                5703,
                5711,
                "PERSON"
            ],
            [
                5745,
                5751,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white,  { Aliyah tyrone 1211 medical center dr. mrn: 047 { Hanna 717361, dob: 7/27/1969, legal sex: m  { Eliana nashville tn 37232-0004 visit da { Estella te: - 06/02/ { Irvin 2023 - pro { Lorene cedure pass  { Sherman in vanderbilt university adult hospital facesheet report patient demographics patient nam { Essie e m { Dawson rn legal  { Khloe d { Rena ob address phone white, tyrone 0477173 sex 7/27/1969 apt 705 615-260 { Marla -2 { Trey 291 (home) 61 m 1101 edgehil { Tasha l ave 61 { Marina 5-260-2291 (mobile) { Sheldon  nash { Tami ville tn 37203 *preferred* h { Ora ospita { Bonita l account not { Lane  on file adm { Mack ission inf { Lizzie ormat { Woodrow ion current information attending provider admitting provider admission type admission st { Sasha atus unknown  { Ervin status admission date/time  { Alondra discharge date/time h { Jude ospital service auth/cert status hospi { Kaiden tal area unit room/ { Lucinda bed referring provider  { Bettie 06/02/2023 - proced { Declan ure pass in vanderbilt university adult hospi { Paisley tal (continued) vi { Gustavo si { Clay t information admission info { Mckenna rmati { Kira on arrival d { Daphne ate/time: admit date/time: ip a { Terence dm. date/time: { Lesley  admission type: point of origin: admit category: means of arrival: primary service: secon { Roosevelt dary service: n/a t { Iva ransfer source: service area: unit: admit provider: attending provider: referring provider: { Mariana  discharge info { Britney rmation date/time: - disposition: - destinati { Delaney on: - provider: - unit: - printed on 10/3 { Valentina /24 7:13 am page 2335, { Emilia vumc adult ho { Terrell spita { Helena l white, tyrone 1211 med { Desmond ical c { Kyra enter dr. mrn: 047717361, dob: 7/27/1969, legal sex: m  { Cheri nashville tn 3 { Ismael 7232-0004 visit date: - 06/ { Rachelle 02/2023 - procedure pass in v { Gregg anderbilt university { Leanne   { Fern adult ho { Tori spital facesheet report patient demograp { Janine hics patient name mrn leg { Delia al dob addr { Jeannie ess phone white, { Charity  tyr { Jalen o { London ne 0477173 sex 7/ { Jazmine 27/1969 apt 705 615-260-2291 (h { Ron ome) 61 m 1101 edgehill ave 615-260-2291 (mobile) nashville tn 37203 *p { Cornelius referred* hos { Keegan pital account not on file admissi { Corbin on informatio { Kim n curr { Francine ent informati { Conrad on attending  { Laila provider admit { Ashleigh ting provider admission typ { Jameson e  { Angeline admission status unknown status admission date/time d { Goldie ischar { Rufus ge date/time hospital service auth/cert status hospital area unit room/bed referring { Tammie  provider 06/02/20 { Lukas 23 - procedure pass i { Therese n vanderbilt university adult hospit { Sherrie al (continued)  { Tia visit information adm { Demetrius ission informa { Noelle tion  { Eden arrival date/time: a { Aileen dmit  { Ciara d { Christa ate/time: i { Greyson p adm. date/time: admission type: point of origin: admit category: means of arrival { Athena : { Carroll  primary service: s { Reba econd { Reid ary se { Patrice rvice: n/a transfe { Lenora r source: service ar { Toby ea: unit: admit provider: at { Alisa tending provider: referring provider: dis { Kendall charge information date/time: - disposition: - destination: - provider: - { Dexter  unit: - printed { Tatiana  on 10/3/24 7:13 am page 2336",
    {
        "entities": [
            [
                30,
                37,
                "PERSON"
            ],
            [
                79,
                85,
                "PERSON"
            ],
            [
                125,
                132,
                "PERSON"
            ],
            [
                167,
                175,
                "PERSON"
            ],
            [
                190,
                196,
                "PERSON"
            ],
            [
                209,
                216,
                "PERSON"
            ],
            [
                231,
                239,
                "PERSON"
            ],
            [
                331,
                337,
                "PERSON"
            ],
            [
                343,
                350,
                "PERSON"
            ],
            [
                362,
                368,
                "PERSON"
            ],
            [
                372,
                377,
                "PERSON"
            ],
            [
                448,
                454,
                "PERSON"
            ],
            [
                459,
                464,
                "PERSON"
            ],
            [
                495,
                501,
                "PERSON"
            ],
            [
                512,
                519,
                "PERSON"
            ],
            [
                541,
                549,
                "PERSON"
            ],
            [
                557,
                562,
                "PERSON"
            ],
            [
                593,
                597,
                "PERSON"
            ],
            [
                606,
                613,
                "PERSON"
            ],
            [
                629,
                634,
                "PERSON"
            ],
            [
                649,
                654,
                "PERSON"
            ],
            [
                667,
                674,
                "PERSON"
            ],
            [
                682,
                690,
                "PERSON"
            ],
            [
                782,
                788,
                "PERSON"
            ],
            [
                804,
                810,
                "PERSON"
            ],
            [
                840,
                848,
                "PERSON"
            ],
            [
                872,
                877,
                "PERSON"
            ],
            [
                918,
                925,
                "PERSON"
            ],
            [
                947,
                955,
                "PERSON"
            ],
            [
                981,
                988,
                "PERSON"
            ],
            [
                1010,
                1017,
                "PERSON"
            ],
            [
                1065,
                1073,
                "PERSON"
            ],
            [
                1094,
                1102,
                "PERSON"
            ],
            [
                1107,
                1112,
                "PERSON"
            ],
            [
                1143,
                1151,
                "PERSON"
            ],
            [
                1159,
                1164,
                "PERSON"
            ],
            [
                1179,
                1186,
                "PERSON"
            ],
            [
                1220,
                1228,
                "PERSON"
            ],
            [
                1245,
                1252,
                "PERSON"
            ],
            [
                1345,
                1355,
                "PERSON"
            ],
            [
                1377,
                1381,
                "PERSON"
            ],
            [
                1475,
                1483,
                "PERSON"
            ],
            [
                1501,
                1509,
                "PERSON"
            ],
            [
                1557,
                1565,
                "PERSON"
            ],
            [
                1609,
                1619,
                "PERSON"
            ],
            [
                1644,
                1651,
                "PERSON"
            ],
            [
                1667,
                1675,
                "PERSON"
            ],
            [
                1683,
                1690,
                "PERSON"
            ],
            [
                1717,
                1725,
                "PERSON"
            ],
            [
                1734,
                1739,
                "PERSON"
            ],
            [
                1797,
                1803,
                "PERSON"
            ],
            [
                1820,
                1827,
                "PERSON"
            ],
            [
                1857,
                1866,
                "PERSON"
            ],
            [
                1898,
                1904,
                "PERSON"
            ],
            [
                1927,
                1934,
                "PERSON"
            ],
            [
                1938,
                1943,
                "PERSON"
            ],
            [
                1954,
                1959,
                "PERSON"
            ],
            [
                2002,
                2009,
                "PERSON"
            ],
            [
                2037,
                2043,
                "PERSON"
            ],
            [
                2057,
                2065,
                "PERSON"
            ],
            [
                2084,
                2092,
                "PERSON"
            ],
            [
                2099,
                2105,
                "PERSON"
            ],
            [
                2109,
                2116,
                "PERSON"
            ],
            [
                2136,
                2144,
                "PERSON"
            ],
            [
                2178,
                2182,
                "PERSON"
            ],
            [
                2256,
                2266,
                "PERSON"
            ],
            [
                2282,
                2289,
                "PERSON"
            ],
            [
                2325,
                2332,
                "PERSON"
            ],
            [
                2348,
                2352,
                "PERSON"
            ],
            [
                2361,
                2370,
                "PERSON"
            ],
            [
                2386,
                2393,
                "PERSON"
            ],
            [
                2409,
                2415,
                "PERSON"
            ],
            [
                2432,
                2441,
                "PERSON"
            ],
            [
                2471,
                2479,
                "PERSON"
            ],
            [
                2484,
                2493,
                "PERSON"
            ],
            [
                2549,
                2556,
                "PERSON"
            ],
            [
                2565,
                2571,
                "PERSON"
            ],
            [
                2658,
                2665,
                "PERSON"
            ],
            [
                2686,
                2692,
                "PERSON"
            ],
            [
                2716,
                2724,
                "PERSON"
            ],
            [
                2763,
                2771,
                "PERSON"
            ],
            [
                2789,
                2793,
                "PERSON"
            ],
            [
                2817,
                2827,
                "PERSON"
            ],
            [
                2844,
                2851,
                "PERSON"
            ],
            [
                2859,
                2864,
                "PERSON"
            ],
            [
                2887,
                2894,
                "PERSON"
            ],
            [
                2902,
                2908,
                "PERSON"
            ],
            [
                2912,
                2920,
                "PERSON"
            ],
            [
                2934,
                2942,
                "PERSON"
            ],
            [
                3028,
                3035,
                "PERSON"
            ],
            [
                3039,
                3047,
                "PERSON"
            ],
            [
                3069,
                3074,
                "PERSON"
            ],
            [
                3082,
                3087,
                "PERSON"
            ],
            [
                3096,
                3104,
                "PERSON"
            ],
            [
                3125,
                3132,
                "PERSON"
            ],
            [
                3155,
                3160,
                "PERSON"
            ],
            [
                3191,
                3197,
                "PERSON"
            ],
            [
                3241,
                3249,
                "PERSON"
            ],
            [
                3325,
                3332,
                "PERSON"
            ],
            [
                3351,
                3359,
                "PERSON"
            ]
        ]
    }
),(
    " { Orville vumc hendersonville - anderson white, tyr { Phoebe one 128 n anderson ln { Mayra  mrn: 04771736 { Bette 1, dob: 7/27/1969, legal sex: m hendersonvil { Ronda le tn 37075 visit date: 9/7/2023 09/07/ { Melba 2023 - communication in vand { Dane erbilt primary care hendersonville (continued) medicat { Deloris ion list (continued) cetirizine 10 mg tablet ( { Quinn zyrtec) discontinued by: mickey, lisa, lpn discontin { Kasey ued on: 10/4/2 { Lucile 023 reason for discontinua { Tommie t { Brielle ion: reorder instructions: take 1 tablet (10 mg total) by mouth once a day as needed for allergies. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 30 tablet refill: 11 refills by 8/7/2024 montelukast 10 mg tab { Drake let (singulair) instructions: take  { Dakota 1 tablet (10 mg total) by mouth every evening. authorized by: lippard { Trina , giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quan { Luna tity: 90 tablet refill: 3 refil { Dewey ls by { Keri  8/7/2024 nifedipine er 30 mg tablet,extend { Chelsey ed release (adalat cc) discontinued by: de witte, anton jordan, md discontinued on: 4/23/2024 reason for discontinuation:  { Jayson reo { Amos rder instructions: take 1 tablet (30 mg total)  { Dorothea by mouth daily. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quanti { Jolene ty { Clint : 90 tablet refill: 3 refills by 8/7/2024 i { Lamar nsulin g { Frankie largine (u-100) { Cayden  100 unit/ml subcutaneous solution disc { Hugo ontinued by: de witte, anton jordan, md discontinued { Ina  on: 10/20/2023 reason for discontinuation: reorder instructi { Darin ons: inject 0.05 ml (5 units total) under the skin { Chandler  daily. authorized by: lippard, giles a, aprn ordered on: 8/24/2023 start date: 8/24/2023 quantity: 4.5 ml { Rodolfo  refill:  remaining cyclobenzaprine 5 mg tablet (flexe { Shawn ril) [r { Antonia econciled by maples, chantis on 9/5/ { Solomon 2023 15 { Marisol 22] ins { Bert tructions: take 1 tablet (5 mg total) by mouth every 8 hours as needed. en { Lou tered by: maples,  { Stewart chantis entered on: 9/5/2023 start date: 7/22/2023 triam { Kassandra cinolone aceto { Matilda nide 55 mcg nasal spray aerosol (nasacort) discontinued by { Tracie : gingrow, barbara,  { Jaylen lpn discontinued on: 9/16/2024 reason for discontinuat { Elbert ion: duplicate order instructions: administer 2 sprays (110 mcg total) into each nost { Janette ril 2 times a day. authoriz { Willow ed b { Maddox y: greenspan, debra l, aprn ordered on: 9/5/2023 start date: 9/5/2023 end date: 9/16/2024 quantity: 16.5 g refill: 11 refills by 9/4/2024 losartan 25 mg tablet (cozaar) discontinued by: kovtun, roman, md discontinued on: 8/1/2024 reason for discontinuation: stop (cancelrx, on avs) instructions: take 1 tablet (25 mg tota { Helene l) by mout { Fatima h daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 end d { Ken ate: 8/1/2024 quantity: 90 table { Quinn t refill: 3 refills by { Brennan  9/6/202 { Jamal 4 capsaicin 0.1 % topical cream discontinued by: gingrow, barbara, lpn dis { Amir conti { Dena nued on: 9 { Leilani /16/2024 reason for discontinuation: duplicate order  { Pam instruction { Reese s: apply 1 application to { Blaine pically daily for 90 days. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/202 { Diamond 3 end date: 9/1 { Kameron 6/2024 action: patient not takin { Otto g quantity: 42.5 g refill:  remaining printed on 10/3/24 7:13 am page 1767,vumc hen { Bradford de { Jody rsonville - anderson white, t { Myrna yrone 128 n anderson ln mrn: 047717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 visit date: 9/7/2023 09/07/2023 - communication  { Sterling in vanderbilt prima { Latasha ry care hendersonville (continued) medicat { Teddy ion list (continued) gabapentin 300 mg capsule (neurontin) discontinued by: hock, richard { Asia  lloyd { Lea , md discontinued on: 11/9/2023  { Alina reason for disc { Brooklynn ontinuation: reorder instructions: take 1 capsule (300 mg total) by mouth daily. authorized by: de witte, anton jo { Fay rdan, md ordered on: 9/7/2023 start date: 9/7/2023 quantity: 90 capsu { Shayla le refill:  remaining aspirin 81 mg tablet,delayed { Marty  release instructions: take 1 tablet (81 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: { Shana  9/7/2023 quantity: 90 tablet refill: 3 refills by 9/6/2024 stopped in visit none clinical notes telephone encounter earls, clarice a at 9/7/2023 0931 author: earls, clarice a { Kayleigh  service: author type: patient access filed: 9/7/2023 9:35 am encounter date: 9/7/2023 s { Jakob tatus: signed editor: earls, clarice a (patient access) i { Emil nstructions for message sender: co { Tania nfirm routing pool to use in redcap decision support tool messa { Percy ge template: nurse general: messag { Fiona e reason: asks to  { Zackary speak with nurse (not about an urgent or emergent symptom in the decision tree) message details: patient w { Randi as discharged from vumc on 08/23/ 2023 and se { Cherie veral admissions to er for acute care. ju { Billie st for your i { Keira nformation and c { Julianne an call  { Lionel if have any questions. name of caller: mitzi with { Dina  amerigroup tennesse best callback number: 757-277-0321 relationship to patient: payer best callback timeframe: 8am to 8 { Reed pm est mon wed  { Savanna and thursdays provider: lippard, giles a, aprn department: primary { Elva  care hendersonville 9/7/2023, 9:32 am cdt mes { Reuben sage from patient access services please do not reply / do not reply all reason for  { Esteban communication: clinician communication instructions for message sende { Francesca r: p { Lyla rint { Lorna ed  { Hallie on 10/3/24 7:13 am page 1768",
    {
        "entities": [
            [
                3,
                11,
                "PERSON"
            ],
            [
                55,
                62,
                "PERSON"
            ],
            [
                86,
                92,
                "PERSON"
            ],
            [
                109,
                115,
                "PERSON"
            ],
            [
                162,
                168,
                "PERSON"
            ],
            [
                210,
                216,
                "PERSON"
            ],
            [
                247,
                252,
                "PERSON"
            ],
            [
                309,
                317,
                "PERSON"
            ],
            [
                366,
                372,
                "PERSON"
            ],
            [
                427,
                433,
                "PERSON"
            ],
            [
                450,
                457,
                "PERSON"
            ],
            [
                486,
                493,
                "PERSON"
            ],
            [
                497,
                505,
                "PERSON"
            ],
            [
                760,
                766,
                "PERSON"
            ],
            [
                804,
                811,
                "PERSON"
            ],
            [
                883,
                889,
                "PERSON"
            ],
            [
                954,
                959,
                "PERSON"
            ],
            [
                993,
                999,
                "PERSON"
            ],
            [
                1007,
                1012,
                "PERSON"
            ],
            [
                1058,
                1066,
                "PERSON"
            ],
            [
                1191,
                1198,
                "PERSON"
            ],
            [
                1204,
                1209,
                "PERSON"
            ],
            [
                1259,
                1268,
                "PERSON"
            ],
            [
                1373,
                1380,
                "PERSON"
            ],
            [
                1385,
                1391,
                "PERSON"
            ],
            [
                1437,
                1443,
                "PERSON"
            ],
            [
                1454,
                1462,
                "PERSON"
            ],
            [
                1480,
                1487,
                "PERSON"
            ],
            [
                1529,
                1534,
                "PERSON"
            ],
            [
                1589,
                1593,
                "PERSON"
            ],
            [
                1657,
                1663,
                "PERSON"
            ],
            [
                1716,
                1725,
                "PERSON"
            ],
            [
                1834,
                1842,
                "PERSON"
            ],
            [
                1899,
                1905,
                "PERSON"
            ],
            [
                1915,
                1923,
                "PERSON"
            ],
            [
                1962,
                1970,
                "PERSON"
            ],
            [
                1980,
                1988,
                "PERSON"
            ],
            [
                1998,
                2003,
                "PERSON"
            ],
            [
                2080,
                2084,
                "PERSON"
            ],
            [
                2105,
                2113,
                "PERSON"
            ],
            [
                2172,
                2182,
                "PERSON"
            ],
            [
                2199,
                2207,
                "PERSON"
            ],
            [
                2268,
                2275,
                "PERSON"
            ],
            [
                2298,
                2305,
                "PERSON"
            ],
            [
                2362,
                2369,
                "PERSON"
            ],
            [
                2457,
                2465,
                "PERSON"
            ],
            [
                2495,
                2502,
                "PERSON"
            ],
            [
                2509,
                2516,
                "PERSON"
            ],
            [
                2840,
                2847,
                "PERSON"
            ],
            [
                2860,
                2867,
                "PERSON"
            ],
            [
                2968,
                2972,
                "PERSON"
            ],
            [
                3007,
                3013,
                "PERSON"
            ],
            [
                3038,
                3046,
                "PERSON"
            ],
            [
                3057,
                3063,
                "PERSON"
            ],
            [
                3140,
                3145,
                "PERSON"
            ],
            [
                3153,
                3158,
                "PERSON"
            ],
            [
                3171,
                3179,
                "PERSON"
            ],
            [
                3235,
                3239,
                "PERSON"
            ],
            [
                3253,
                3259,
                "PERSON"
            ],
            [
                3287,
                3294,
                "PERSON"
            ],
            [
                3406,
                3414,
                "PERSON"
            ],
            [
                3432,
                3440,
                "PERSON"
            ],
            [
                3475,
                3480,
                "PERSON"
            ],
            [
                3566,
                3575,
                "PERSON"
            ],
            [
                3580,
                3585,
                "PERSON"
            ],
            [
                3617,
                3623,
                "PERSON"
            ],
            [
                3767,
                3776,
                "PERSON"
            ],
            [
                3798,
                3806,
                "PERSON"
            ],
            [
                3851,
                3857,
                "PERSON"
            ],
            [
                3949,
                3954,
                "PERSON"
            ],
            [
                3963,
                3967,
                "PERSON"
            ],
            [
                4002,
                4008,
                "PERSON"
            ],
            [
                4026,
                4036,
                "PERSON"
            ],
            [
                4153,
                4157,
                "PERSON"
            ],
            [
                4229,
                4236,
                "PERSON"
            ],
            [
                4289,
                4295,
                "PERSON"
            ],
            [
                4439,
                4445,
                "PERSON"
            ],
            [
                4623,
                4632,
                "PERSON"
            ],
            [
                4723,
                4729,
                "PERSON"
            ],
            [
                4789,
                4794,
                "PERSON"
            ],
            [
                4831,
                4837,
                "PERSON"
            ],
            [
                4903,
                4909,
                "PERSON"
            ],
            [
                4946,
                4952,
                "PERSON"
            ],
            [
                4973,
                4981,
                "PERSON"
            ],
            [
                5090,
                5096,
                "PERSON"
            ],
            [
                5144,
                5151,
                "PERSON"
            ],
            [
                5195,
                5202,
                "PERSON"
            ],
            [
                5218,
                5224,
                "PERSON"
            ],
            [
                5243,
                5252,
                "PERSON"
            ],
            [
                5263,
                5270,
                "PERSON"
            ],
            [
                5322,
                5327,
                "PERSON"
            ],
            [
                5450,
                5455,
                "PERSON"
            ],
            [
                5473,
                5481,
                "PERSON"
            ],
            [
                5550,
                5555,
                "PERSON"
            ],
            [
                5604,
                5611,
                "PERSON"
            ],
            [
                5698,
                5706,
                "PERSON"
            ],
            [
                5778,
                5788,
                "PERSON"
            ],
            [
                5795,
                5800,
                "PERSON"
            ],
            [
                5807,
                5813,
                "PERSON"
            ],
            [
                5819,
                5826,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult dayani center white, tyrone 150 { Shauna 0 medical ctr dr 1st fl, 108  { Wilfred mrn { Amaya : 047 { Zachery 717361, dob: 7/27/1969, legal sex: m dayani ctr visit date: 12 { Quinton /1/2023 nashville tn 37232 { Hillary  12/01/2 { Kyler 023 - appointm { Luca ent in vanderbilt dayani  { Cleo ce { Darrin nter (contin { Juliet ued) referral ( { Guillermo continued) referral notes general  { Gwen by ogles, pam { Bret ela j at 11/17/20 { Felipe 23 0755 { Katelynn  pt { Jodie  approved for 8 visits auth #02k4hyh33 exp 1/14/2024 general by ogles, pa { Aubrey mela j at 10/20/2023 1220 therapy benefits coverage 80 { Millie % deductible $0 out of pocke { Clare t $8,300 co-pay $0 precert after eval telep { Darrel hone number or website availity person spoke with { Luella  na on 10 { Merle /20 call reference number na visit limit pending { Shanna  auth aquatics based on med nec not a guarantee  { Marlon of coverage triage triage information decision: none sche { Rylan dule by date: 11/3/202 { Daniella 3 coverages u { Cathleen hc community dual { Doyle  snp plan: uhc c { Lara ommunity dual  { Randal covered: covered from: 1/1/2024 to: 1/3 { Debora 1/2024 snp member #: 127670950 amerivantage  { Carissa well { Macy point medicare plan:  { Rory amerivanta { Brenna ge covered: covered fr { Karin om: 6/1/2023 to:  { Hadley 2/29/2024 amerigroup { Heath  wellpoint m { Kendrick a member #: 768w12411 auth #: 02k4hyh33 zzzmcai { Paris d of tennessee plan: medicaid supplement { Eugenia al cov { Alyson ered: covered from: 6/1/2022 to: 12/31/2023 membe { Francis r #: td525606373 auth  { Davis #: nar tc { Makenna  tenncare { Elsa  select plan: t { Branden c select  covered { Ola : { Polly  covered fr { Jayce om: 8/30/2024 to: 8 { Jayne /3 { Darnell 0/2024 member #: zedm13004089 messag { Jewell es sorry you missed your appointment. let's reschedule. printed on 10/3/24 7:13 am page 1477,vumc adult dayani center white, tyron { Dick e 1500 medical ctr dr 1st f { Angelique l, 108 mrn: 0477 { Jaiden 17361, dob: 7/27/1969,  { Tomas l { Haylee egal sex: m dayani ctr visit date: 1 { Joaquin 2/1/2023 nash { Joni ville tn 37232 12/01/2023 - appointment in vanderbilt da { Candy yani center (continued) messages (continued) from to sent and d { Alta elivered mychart, { Sienna  generic w { Zion hite, tyrone 12/5/2023 6:09 pm la { Zachariah st read in { Deandre  my heal { Royce th at vanderbilt no { Louie t read decem { Talia ber 5, 2023 tyrone white apt 705 1101 edg { Juliette ehill ave nashville, tn 37203 dea { Taryn r tyrone white,  { Cierra our records show you recently missed t { Tiana he app { Cameron ointment below. cobble, jeff  { Gilberto allen, dpt in vanderbilt dayani center on 12/1/2023 a { Ali t 8:00 am please call { Alivia  615-322-4751 to reschedule { Susanne  this appointment if you haven't done so already, or you may be able to reschedule  { Winston with the schedul { Brooks ing fea { Rocco ture on  { Stephan the my health at vanderbilt portal. we want to make sure you get the bes { Van t, safest care and treatment possible. be { Alayna cause you're an i { Antoine mportant part of y { Jarrod our health care team, it's important you keep all scheduled appo { Rosetta intments. { Sheena  if you've received this letter b { Moises y mistake, or i { Leola f you need  { Kingston any help, call us at 615-322-4751. sincerely, cobble, jeff allen { Simone , dpt a { Xander ppointment scheduled from to sent and { Ingrid  del { Adelaide ivered mychart, generic white, tyrone 11/20/ { Cade 2023 9:2 { Rocky 7 am last read in my health at vanderbilt not read appointment infor { Evangeline mation: visit type: pt therapy date: 12/1/2023 dept: vanderbilt dayani center provider: jeff a cobble time: 8:00 am length { Imogene : 50 min printed on 10/3/24 7:13 am page 1478",
    {
        "entities": [
            [
                45,
                52,
                "PERSON"
            ],
            [
                84,
                92,
                "PERSON"
            ],
            [
                98,
                104,
                "PERSON"
            ],
            [
                112,
                120,
                "PERSON"
            ],
            [
                185,
                193,
                "PERSON"
            ],
            [
                222,
                230,
                "PERSON"
            ],
            [
                241,
                247,
                "PERSON"
            ],
            [
                264,
                269,
                "PERSON"
            ],
            [
                297,
                302,
                "PERSON"
            ],
            [
                307,
                314,
                "PERSON"
            ],
            [
                329,
                336,
                "PERSON"
            ],
            [
                354,
                364,
                "PERSON"
            ],
            [
                401,
                406,
                "PERSON"
            ],
            [
                422,
                427,
                "PERSON"
            ],
            [
                447,
                454,
                "PERSON"
            ],
            [
                464,
                473,
                "PERSON"
            ],
            [
                479,
                485,
                "PERSON"
            ],
            [
                561,
                568,
                "PERSON"
            ],
            [
                625,
                632,
                "PERSON"
            ],
            [
                663,
                669,
                "PERSON"
            ],
            [
                715,
                722,
                "PERSON"
            ],
            [
                774,
                781,
                "PERSON"
            ],
            [
                793,
                799,
                "PERSON"
            ],
            [
                850,
                857,
                "PERSON"
            ],
            [
                908,
                915,
                "PERSON"
            ],
            [
                975,
                981,
                "PERSON"
            ],
            [
                1006,
                1015,
                "PERSON"
            ],
            [
                1031,
                1040,
                "PERSON"
            ],
            [
                1060,
                1066,
                "PERSON"
            ],
            [
                1085,
                1090,
                "PERSON"
            ],
            [
                1107,
                1114,
                "PERSON"
            ],
            [
                1156,
                1163,
                "PERSON"
            ],
            [
                1210,
                1218,
                "PERSON"
            ],
            [
                1225,
                1230,
                "PERSON"
            ],
            [
                1254,
                1259,
                "PERSON"
            ],
            [
                1272,
                1279,
                "PERSON"
            ],
            [
                1304,
                1310,
                "PERSON"
            ],
            [
                1330,
                1337,
                "PERSON"
            ],
            [
                1360,
                1366,
                "PERSON"
            ],
            [
                1381,
                1390,
                "PERSON"
            ],
            [
                1440,
                1446,
                "PERSON"
            ],
            [
                1489,
                1497,
                "PERSON"
            ],
            [
                1506,
                1513,
                "PERSON"
            ],
            [
                1565,
                1573,
                "PERSON"
            ],
            [
                1598,
                1604,
                "PERSON"
            ],
            [
                1616,
                1624,
                "PERSON"
            ],
            [
                1636,
                1641,
                "PERSON"
            ],
            [
                1659,
                1667,
                "PERSON"
            ],
            [
                1687,
                1691,
                "PERSON"
            ],
            [
                1695,
                1701,
                "PERSON"
            ],
            [
                1715,
                1721,
                "PERSON"
            ],
            [
                1743,
                1749,
                "PERSON"
            ],
            [
                1754,
                1762,
                "PERSON"
            ],
            [
                1801,
                1808,
                "PERSON"
            ],
            [
                1941,
                1946,
                "PERSON"
            ],
            [
                1976,
                1986,
                "PERSON"
            ],
            [
                2005,
                2012,
                "PERSON"
            ],
            [
                2038,
                2044,
                "PERSON"
            ],
            [
                2048,
                2055,
                "PERSON"
            ],
            [
                2094,
                2102,
                "PERSON"
            ],
            [
                2118,
                2123,
                "PERSON"
            ],
            [
                2182,
                2188,
                "PERSON"
            ],
            [
                2254,
                2259,
                "PERSON"
            ],
            [
                2279,
                2286,
                "PERSON"
            ],
            [
                2299,
                2304,
                "PERSON"
            ],
            [
                2340,
                2350,
                "PERSON"
            ],
            [
                2363,
                2371,
                "PERSON"
            ],
            [
                2382,
                2388,
                "PERSON"
            ],
            [
                2410,
                2416,
                "PERSON"
            ],
            [
                2431,
                2437,
                "PERSON"
            ],
            [
                2481,
                2490,
                "PERSON"
            ],
            [
                2526,
                2532,
                "PERSON"
            ],
            [
                2551,
                2558,
                "PERSON"
            ],
            [
                2599,
                2605,
                "PERSON"
            ],
            [
                2614,
                2622,
                "PERSON"
            ],
            [
                2654,
                2663,
                "PERSON"
            ],
            [
                2719,
                2723,
                "PERSON"
            ],
            [
                2747,
                2754,
                "PERSON"
            ],
            [
                2784,
                2792,
                "PERSON"
            ],
            [
                2878,
                2886,
                "PERSON"
            ],
            [
                2905,
                2912,
                "PERSON"
            ],
            [
                2922,
                2928,
                "PERSON"
            ],
            [
                2939,
                2947,
                "PERSON"
            ],
            [
                3022,
                3026,
                "PERSON"
            ],
            [
                3070,
                3077,
                "PERSON"
            ],
            [
                3097,
                3105,
                "PERSON"
            ],
            [
                3126,
                3133,
                "PERSON"
            ],
            [
                3200,
                3208,
                "PERSON"
            ],
            [
                3220,
                3227,
                "PERSON"
            ],
            [
                3263,
                3270,
                "PERSON"
            ],
            [
                3288,
                3294,
                "PERSON"
            ],
            [
                3308,
                3317,
                "PERSON"
            ],
            [
                3384,
                3391,
                "PERSON"
            ],
            [
                3401,
                3408,
                "PERSON"
            ],
            [
                3448,
                3455,
                "PERSON"
            ],
            [
                3462,
                3471,
                "PERSON"
            ],
            [
                3518,
                3523,
                "PERSON"
            ],
            [
                3534,
                3540,
                "PERSON"
            ],
            [
                3611,
                3622,
                "PERSON"
            ],
            [
                3747,
                3755,
                "PERSON"
            ]
        ]
    }
),(
    "vumc vis one hundred { Adan  oaks white, tyrone 719 thomps { Willa on ln m { Sondra rn: 047717361, dob: 7/27/1969, { Anderson  legal sex { Boyd : m  { Mickey vanderbilt imaging of o { Hayden ne hundred oaks visit date: 12/18/2023 nashville tn  { Harmony 37204 12/18/2023 - appointment in vanderbilt i { Grover maging services one hundred oaks  { Trudy (continued) referral (continued) coverages am { Elvira erivantage wellpo { Hilary int medicare plan: amerivantage covered: covered from: 6/1/2023 to: 2/29/2024 amerigroup wellpoint ma member #: 768w12411 auth #: oho #233068678 72148 3/9/2024 zzzmcaid of tennessee plan: medicaid supplementa { Nell l  { Staci covered: covered from: 6/1 { Johnathon /2022 to: 12 { Pat /31 { Emilie /2023 me { Jenifer mber #: td52560637 { Kinsley 3 auth  { Dewayne #: nar tc tenncare select  { Camryn plan: tc select   { Rosalind cov { Arielle ered: covered from: 8/30/2024 to: 8 { Kailey /30/2024 member #: zedm13 { Selma 004089 messa { Brenden ges tyrone white your radiology appointment is coming up. from to sent and delivered mychart, generic white, tyrone 12/17/2023  { Maximus 6:08  { Luz pm last read in my health at vanderbi { Eve lt not read hello tyrone, this is a { Freddy  reminder of your upcom { King ing visit with vanderbilt imagi { Jerald ng service { Ed s  { Ximena one hundred oak { Izabella s on 12/18/23 at 3:45 pm cst. vanderbilt imaging services one hundred oaks 719 thompson ln suite 23300 nashville tn 372 { Charlie 04 thank you for choosing { Maverick  vanderbilt for { Ryker  your upcoming appointment! w { Logan e want you to get the care and attention you need while avoidi { Timmy ng infecti { Rhoda on during your visit to vanderbilt university medical center. saf { Vance ety is trul { Greta y import { Glenna ant to us. while we  require patient { Kali s or staff to wear a mask in clinical areas { Heaven , we encourage e { Eldon veryone t { Audra o mask if th { Kiana ey ar { Blanca e exp { Walker eriencing respiratory symptoms. you are welcome to bring y { Jonas our own bag { Nannie  for y { Jess our clothes and belongings. this small { Marley  step makes a big difference for the env { Quincy i { Clarice ronment. by bringing your own bag, you are helping us reduce waste  { Guadalupe and pro { Chasity tect our planet. it's an easy { Leanna  way to take care of your health an { Frieda d the earth at the same time. you can have up to 2 visitors  { Emerson accompany you to your imaging app { Isla ointment. for safety reasons, family and guests of adult patie { Rogelio nts are to remain in the waiting area while you are being imaged. pediatric patients may have an adult gua { Ryleigh rdian accompany them. we ki { Ramiro ndly request that all visitors under the  { Darwin age 12 be su { Ryan pervised by a { Murray n { Aniyah  adult, over the age of  { Cary 18, at all times. this allows us to maintain a safer environment for our { Brynn  patients, guests, and your he { Leann althcare team. vand { Maribel erbilt university medical center printed on 10/3/24 7:12  { Conor am page 1371, { Edmond vumc vis one hundred oaks white, tyrone 719 thompson ln mrn: 047717361, dob: 7/27/1969, legal sex: m vanderbilt imaging of { Kenya  one hundred oa { Bobbi ks visit date: 12/18/2023 nashville tn 3720 { Larissa 4 12/18 { Jean /2023 - appo { Margo intment in vanderbilt  { Lizbeth imaging  { Landen se { Tiara rvices one hundre { Gael d oaks (continued) messages (continued) appointment scheduled from to sent and delivered { Marquis  mychart, generic white, tyron { Rowan e 11/17/ { Thaddeus 2023 1:53 pm last rea { Harley d in my health { Elaina  at vanderbilt not read appointment information: visit type: mri lumbar sp { Malia ine wo cont { Bridgette rast date: 12/18/2023 dept: vanderbilt imaging services one hundred oaks provider: { Jeannine  vis oho mri 2-1.5t time: 3:45 pm length: 30  { Tamika min  { Sidney appt status: scheduled appt instructions: adult instructions: if you have any impl { Jeanine an { Matt ted medical devi { Ericka ces, please bring your identification card with you. the card should include the serial number so that our team can verify that it is safe to enter the mri. printed { Courtney  on 10/3/24 7:12 am page 1 { Milo 372",
    {
        "entities": [
            [
                23,
                28,
                "PERSON"
            ],
            [
                61,
                67,
                "PERSON"
            ],
            [
                77,
                84,
                "PERSON"
            ],
            [
                117,
                126,
                "PERSON"
            ],
            [
                139,
                144,
                "PERSON"
            ],
            [
                151,
                158,
                "PERSON"
            ],
            [
                184,
                191,
                "PERSON"
            ],
            [
                246,
                254,
                "PERSON"
            ],
            [
                303,
                310,
                "PERSON"
            ],
            [
                346,
                352,
                "PERSON"
            ],
            [
                400,
                407,
                "PERSON"
            ],
            [
                427,
                434,
                "PERSON"
            ],
            [
                645,
                650,
                "PERSON"
            ],
            [
                655,
                661,
                "PERSON"
            ],
            [
                690,
                700,
                "PERSON"
            ],
            [
                715,
                719,
                "PERSON"
            ],
            [
                725,
                732,
                "PERSON"
            ],
            [
                743,
                751,
                "PERSON"
            ],
            [
                772,
                780,
                "PERSON"
            ],
            [
                790,
                798,
                "PERSON"
            ],
            [
                827,
                834,
                "PERSON"
            ],
            [
                854,
                863,
                "PERSON"
            ],
            [
                869,
                877,
                "PERSON"
            ],
            [
                915,
                922,
                "PERSON"
            ],
            [
                950,
                956,
                "PERSON"
            ],
            [
                971,
                979,
                "PERSON"
            ],
            [
                1109,
                1117,
                "PERSON"
            ],
            [
                1125,
                1129,
                "PERSON"
            ],
            [
                1169,
                1173,
                "PERSON"
            ],
            [
                1211,
                1218,
                "PERSON"
            ],
            [
                1244,
                1249,
                "PERSON"
            ],
            [
                1283,
                1290,
                "PERSON"
            ],
            [
                1303,
                1306,
                "PERSON"
            ],
            [
                1311,
                1318,
                "PERSON"
            ],
            [
                1336,
                1345,
                "PERSON"
            ],
            [
                1467,
                1475,
                "PERSON"
            ],
            [
                1503,
                1512,
                "PERSON"
            ],
            [
                1530,
                1536,
                "PERSON"
            ],
            [
                1568,
                1574,
                "PERSON"
            ],
            [
                1639,
                1645,
                "PERSON"
            ],
            [
                1658,
                1664,
                "PERSON"
            ],
            [
                1732,
                1738,
                "PERSON"
            ],
            [
                1752,
                1758,
                "PERSON"
            ],
            [
                1769,
                1776,
                "PERSON"
            ],
            [
                1815,
                1820,
                "PERSON"
            ],
            [
                1866,
                1873,
                "PERSON"
            ],
            [
                1892,
                1898,
                "PERSON"
            ],
            [
                1910,
                1916,
                "PERSON"
            ],
            [
                1931,
                1937,
                "PERSON"
            ],
            [
                1945,
                1952,
                "PERSON"
            ],
            [
                1960,
                1967,
                "PERSON"
            ],
            [
                2028,
                2034,
                "PERSON"
            ],
            [
                2048,
                2055,
                "PERSON"
            ],
            [
                2064,
                2069,
                "PERSON"
            ],
            [
                2110,
                2117,
                "PERSON"
            ],
            [
                2160,
                2167,
                "PERSON"
            ],
            [
                2171,
                2179,
                "PERSON"
            ],
            [
                2249,
                2259,
                "PERSON"
            ],
            [
                2269,
                2277,
                "PERSON"
            ],
            [
                2309,
                2316,
                "PERSON"
            ],
            [
                2354,
                2361,
                "PERSON"
            ],
            [
                2424,
                2432,
                "PERSON"
            ],
            [
                2468,
                2473,
                "PERSON"
            ],
            [
                2538,
                2546,
                "PERSON"
            ],
            [
                2655,
                2663,
                "PERSON"
            ],
            [
                2693,
                2700,
                "PERSON"
            ],
            [
                2744,
                2751,
                "PERSON"
            ],
            [
                2766,
                2771,
                "PERSON"
            ],
            [
                2787,
                2794,
                "PERSON"
            ],
            [
                2798,
                2805,
                "PERSON"
            ],
            [
                2832,
                2837,
                "PERSON"
            ],
            [
                2912,
                2918,
                "PERSON"
            ],
            [
                2951,
                2957,
                "PERSON"
            ],
            [
                2979,
                2987,
                "PERSON"
            ],
            [
                3047,
                3053,
                "PERSON"
            ],
            [
                3069,
                3076,
                "PERSON"
            ],
            [
                3201,
                3207,
                "PERSON"
            ],
            [
                3225,
                3231,
                "PERSON"
            ],
            [
                3277,
                3285,
                "PERSON"
            ],
            [
                3295,
                3300,
                "PERSON"
            ],
            [
                3315,
                3321,
                "PERSON"
            ],
            [
                3346,
                3354,
                "PERSON"
            ],
            [
                3365,
                3372,
                "PERSON"
            ],
            [
                3377,
                3383,
                "PERSON"
            ],
            [
                3403,
                3408,
                "PERSON"
            ],
            [
                3499,
                3507,
                "PERSON"
            ],
            [
                3540,
                3546,
                "PERSON"
            ],
            [
                3557,
                3566,
                "PERSON"
            ],
            [
                3590,
                3597,
                "PERSON"
            ],
            [
                3614,
                3621,
                "PERSON"
            ],
            [
                3698,
                3704,
                "PERSON"
            ],
            [
                3718,
                3728,
                "PERSON"
            ],
            [
                3813,
                3822,
                "PERSON"
            ],
            [
                3870,
                3877,
                "PERSON"
            ],
            [
                3884,
                3891,
                "PERSON"
            ],
            [
                3976,
                3984,
                "PERSON"
            ],
            [
                3989,
                3994,
                "PERSON"
            ],
            [
                4013,
                4020,
                "PERSON"
            ],
            [
                4187,
                4196,
                "PERSON"
            ],
            [
                4225,
                4230,
                "PERSON"
            ]
        ]
    }
),(
    "vumc henderson { Winnie ville - anderson white, tyrone 128 n an { Rodrigo derson ln { Deana  mrn: { Lamont  047717361, { Payton  d { Rolando ob:  { Emiliano 7/ { Skyler 27/ { Cruz 1969, le { Julissa gal sex: m hendersonville tn 37075 { India  visit date: 3/6/2 { Isiah 023 03/06/2023 - { Anton  office visit i { Brantley n vanderbilt pr { Harlan imary care hendersonville (con { James tinued) fl { Alanna ows { Lexi heets  { Garland (continued) wit { Colette h lipp { Waylon ard,  { Carmela giles a, aprn personal { Nola  safety does anyone no  { Imani -sl at 03/06/23 1357 neglect, hurt, or threat { Holden en t { Jarrett he patient? phq-2 to 9 depression scale  { Ashlynn row name 03/06/23 13 { Norah 57 over the past 2 weeks, ho { Keisha w  { Camilla oft { Kody e { Charley n have you been bothered by  { Dominique any of th { Emilee e followi { Paola ng problems? little interest or sever { Maritza al d { Aisha ays -sl a { Darcy t p { Chris leasure in do { Jimmie ing 03/06/23 1357 things feeling down { Brianne ,  { Bettye several days -sl at depres { Jeri sed, or 03/06/23 135 { Jan 7 hopeless phq 2 sco { Amie re 2 -sl at 03/06/23 1357  { Michael vital signs row na { Lenore m { Cyrus e 03 { Emery /0 { Dale 6/23 14 { Jami 02 other hr/pulse 84 -sl  { Duncan at 03/06/23 1404  { Finn us { Latonya er key { Stacy  (r) = recorded by, (t) = taken  { Tonia b { Buddy y, { Doug  (c) = cosigned by initials name p { Monte rovid { Loyd er type disci { Angelia pline { Presley  sl lee, samantha lee licensed nurse nurse messages appo { Ila intment s { Bailey chedule { Issac d from to sent and delivered mychart, generic white, tyro { Burton ne 12/6/2022 11:29 am last read in my health at { Nikolas  vander { Robbie bilt not read appointment in { Stefan formation: visit type: retu { Jefferson rn visit date: 3/6/202 { Rae 3 { Teagan  de { Abbie pt: vanderbilt pr { Lorie imary care hendersonvill { Nia e provider: giles a lipp { Rene ard t { John im { Cornelia e: 2:20 pm length: 20 min appt status: scheduled prin { Marcy ted on 10/3/ { Bernadine 24 7:1 { Denis 3 am pa { Cleveland ge 2551,vumc hendersonv { Viviana ille -  { Rodger anderson { Katy  white, tyrone { Devon  128 n an { Zander derson ln mrn: 047717361,  { Virgie d { Flossie ob: 7/27/1969, legal sex: m hende { Roscoe rsonville tn 37075 visit date: 3/6/2023 03/06/2023 - office visit in va { Ladonna nderbilt primary care hendersonville (continued) me { Tobias ssag { Robbie es  { Rosalyn (continu { Mauricio ed) prin { Noe t { Jayden ed on 10/3 { Ayla / { Harriett 24 7:13 am page 25 { Christi 52",
    {
        "entities": [
            [
                17,
                24,
                "PERSON"
            ],
            [
                66,
                74,
                "PERSON"
            ],
            [
                86,
                92,
                "PERSON"
            ],
            [
                100,
                107,
                "PERSON"
            ],
            [
                121,
                128,
                "PERSON"
            ],
            [
                133,
                141,
                "PERSON"
            ],
            [
                148,
                157,
                "PERSON"
            ],
            [
                162,
                169,
                "PERSON"
            ],
            [
                175,
                180,
                "PERSON"
            ],
            [
                191,
                199,
                "PERSON"
            ],
            [
                236,
                242,
                "PERSON"
            ],
            [
                263,
                269,
                "PERSON"
            ],
            [
                288,
                294,
                "PERSON"
            ],
            [
                312,
                321,
                "PERSON"
            ],
            [
                339,
                346,
                "PERSON"
            ],
            [
                379,
                385,
                "PERSON"
            ],
            [
                398,
                405,
                "PERSON"
            ],
            [
                411,
                416,
                "PERSON"
            ],
            [
                425,
                433,
                "PERSON"
            ],
            [
                451,
                459,
                "PERSON"
            ],
            [
                468,
                475,
                "PERSON"
            ],
            [
                483,
                491,
                "PERSON"
            ],
            [
                516,
                521,
                "PERSON"
            ],
            [
                547,
                553,
                "PERSON"
            ],
            [
                601,
                608,
                "PERSON"
            ],
            [
                615,
                623,
                "PERSON"
            ],
            [
                666,
                674,
                "PERSON"
            ],
            [
                697,
                703,
                "PERSON"
            ],
            [
                734,
                741,
                "PERSON"
            ],
            [
                746,
                754,
                "PERSON"
            ],
            [
                760,
                765,
                "PERSON"
            ],
            [
                769,
                777,
                "PERSON"
            ],
            [
                808,
                818,
                "PERSON"
            ],
            [
                830,
                837,
                "PERSON"
            ],
            [
                849,
                855,
                "PERSON"
            ],
            [
                895,
                903,
                "PERSON"
            ],
            [
                910,
                916,
                "PERSON"
            ],
            [
                928,
                934,
                "PERSON"
            ],
            [
                940,
                946,
                "PERSON"
            ],
            [
                962,
                969,
                "PERSON"
            ],
            [
                1009,
                1017,
                "PERSON"
            ],
            [
                1022,
                1029,
                "PERSON"
            ],
            [
                1058,
                1063,
                "PERSON"
            ],
            [
                1086,
                1090,
                "PERSON"
            ],
            [
                1113,
                1118,
                "PERSON"
            ],
            [
                1147,
                1155,
                "PERSON"
            ],
            [
                1176,
                1183,
                "PERSON"
            ],
            [
                1187,
                1193,
                "PERSON"
            ],
            [
                1200,
                1206,
                "PERSON"
            ],
            [
                1211,
                1216,
                "PERSON"
            ],
            [
                1226,
                1231,
                "PERSON"
            ],
            [
                1259,
                1266,
                "PERSON"
            ],
            [
                1286,
                1291,
                "PERSON"
            ],
            [
                1296,
                1304,
                "PERSON"
            ],
            [
                1313,
                1319,
                "PERSON"
            ],
            [
                1354,
                1360,
                "PERSON"
            ],
            [
                1364,
                1370,
                "PERSON"
            ],
            [
                1375,
                1380,
                "PERSON"
            ],
            [
                1417,
                1423,
                "PERSON"
            ],
            [
                1431,
                1436,
                "PERSON"
            ],
            [
                1452,
                1460,
                "PERSON"
            ],
            [
                1468,
                1476,
                "PERSON"
            ],
            [
                1535,
                1539,
                "PERSON"
            ],
            [
                1551,
                1558,
                "PERSON"
            ],
            [
                1568,
                1574,
                "PERSON"
            ],
            [
                1634,
                1641,
                "PERSON"
            ],
            [
                1691,
                1699,
                "PERSON"
            ],
            [
                1709,
                1716,
                "PERSON"
            ],
            [
                1747,
                1754,
                "PERSON"
            ],
            [
                1784,
                1794,
                "PERSON"
            ],
            [
                1819,
                1823,
                "PERSON"
            ],
            [
                1827,
                1834,
                "PERSON"
            ],
            [
                1840,
                1846,
                "PERSON"
            ],
            [
                1866,
                1872,
                "PERSON"
            ],
            [
                1899,
                1903,
                "PERSON"
            ],
            [
                1930,
                1935,
                "PERSON"
            ],
            [
                1943,
                1948,
                "PERSON"
            ],
            [
                1953,
                1962,
                "PERSON"
            ],
            [
                2018,
                2024,
                "PERSON"
            ],
            [
                2039,
                2049,
                "PERSON"
            ],
            [
                2058,
                2064,
                "PERSON"
            ],
            [
                2074,
                2084,
                "PERSON"
            ],
            [
                2110,
                2118,
                "PERSON"
            ],
            [
                2128,
                2135,
                "PERSON"
            ],
            [
                2146,
                2151,
                "PERSON"
            ],
            [
                2168,
                2174,
                "PERSON"
            ],
            [
                2186,
                2193,
                "PERSON"
            ],
            [
                2222,
                2229,
                "PERSON"
            ],
            [
                2233,
                2241,
                "PERSON"
            ],
            [
                2277,
                2284,
                "PERSON"
            ],
            [
                2358,
                2366,
                "PERSON"
            ],
            [
                2420,
                2427,
                "PERSON"
            ],
            [
                2434,
                2441,
                "PERSON"
            ],
            [
                2447,
                2455,
                "PERSON"
            ],
            [
                2466,
                2475,
                "PERSON"
            ],
            [
                2486,
                2490,
                "PERSON"
            ],
            [
                2494,
                2501,
                "PERSON"
            ],
            [
                2514,
                2519,
                "PERSON"
            ],
            [
                2523,
                2532,
                "PERSON"
            ],
            [
                2553,
                2561,
                "PERSON"
            ]
        ]
    }
),(
    " { Elma vumc adult medical center east white, tyrone 1211 medical cent { Lacy er dr mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232 visit date: 7/9/2024 07/09/2024 - lab in vanderb { Judah ilt diabetes (continued) laboratory reports (conti { Kiera nued) component value reference range flag { Malinda  lab he { Maddison moglobin 11.9 14.0 - 18.1 gm/dl cerner comment: this test was perf { Corrine ormed at: vanderbilt hospital la { Madalyn boratory,clia #44d0659066,adam se { Emerson egmiller md, phd, 1301 medical center drive, 4605 tvc, nashville, tn, 37232, testing performed by lab - abbreviation name director address val { Mara id date range 123 cerner { Elnora  vumc cerner lab adam seegmiller; 4605 tvc vumc 11/2 { Dorian 2/21 1014 - present jennifer b. 1301 medical cent { Susana er  { Kaylie gordetsky drive nashville tn 37232- 5310 indications chronic renal disease, stage iv (cms/hcc) [n18.4 (icd-10-cm)] all reviewers list freeman, genevieve clayton, aprn on 7/10/2 { Erwin 024 10:27 bmp (final r { Brendon esult) electronically signed by: freeman, genevieve clayton, aprn on 06/27/24 1527 status: completed ordering user: freeman, { Rhett  genevieve clayton, aprn 06 { Skye /27/24 1527 ordering provider: freeman, geneviev { Tabatha e clayton, aprn authorized by: freeman, genevieve clayton, aprn o { Tanisha rdering mode:  { Elton standard frequency: routine 06/27/24 - class: lab colle { Johnie ct quantity: 1  { Ruthie lab status: final result  { Silvia instance released by: fair- { Elyse myers, shon { Neva da michelle 7/9/2024 3:41 pm diagnoses chronic renal disease, stage iv ( { Nathanael cms/hcc) [n18.4] specimen information id type source collect { Noreen ed by 24 { Kamryn -191-0 { Sammie 12066 blood - fair-myers, shonda michelle 07/09/24 1553 bmp (abnormal) resulted: 07/09/24 1751, result status: fi { Marlin nal result ordering provide { Mavis r: freeman,  { Keaton genevieve clayton, aprn 07/09/24 order status: completed 1541 filed by: interface, lab results in 07/09/ { Titus 24 1751 collected by: fair-myers, { Karissa  shonda mi { Tristen chelle 07/09/24 1553 resulting lab: vumc cerner lab acknowledged by: freeman, genevieve clayton, aprn on 07/10/24 1027 components component value reference range flag lab sodium level 141 136 - 145 mmol/l - cerner potassium level 5. { Cadence 3 3.3 - 4.8 mmol/l h a cerner comm { Christian ent: plasma refer { Lauryn ence ranges are shown, please note that serum reference ranges are higher than plasma. chloride level 10 { Elinor 9 98 107 mmol/l h cerner carbon dioxide 23 22 29 mmol/l - cerner glucose lev { Noemi el 294 70 99 mg/dl h^ cerne { Raphael r blood urea nitrog { Vaughn en 34 8 26 mg/dl h^ cerner creatini { Robert ne le { Leonel vel 3.58 0.72 - 1.25 mg/dl h cerner calcium level total 8.5 8.4 10.5 mg/dl - cerner anion gap  { Kurtis 9 m { Nova mol/l - cerner comment: this test was performed at: vanderbilt hospital laboratory, clia #44d0659066, adam seeg { Jase miller md, phd, 1301 medical printed on 10/3/2 { Yasmin 4 7:12 am page 683,vumc adult medical center east white, tyron { Tatum e 1211  { Kristian medical center { Barrett  dr mrn: 047717361,  { Elwood dob: 7/27/1969, lega { Cecile l sex: m nashville tn  { Elvin 37232 visit date: 7/9/2024 07/09/2024 - lab  { Norbert in vanderbilt diabetes (continued) laborato { Dorthy ry reports (continued) ce { Gale nter drive, 4605 tvc, ,nashville, { Lia  tn,37232, testing  { Reynaldo performed by lab - abbrev { Nelda i { Phil ation name director address valid date range 123 cerner vumc cerner lab ada { Ashton m seegmiller; 4605 tvc vumc 11/22/21 1014 - present jennifer b. 1301 medical center go { Dolly rdetsky drive nashville tn 3 { Romeo 7232- 5310 indications chronic renal disease, stage iv (cms/hcc { Louisa ) [n18.4 (icd-10-cm)] all reviewers list freem { Aden an, genevieve clayton, aprn on 7/10/2024 10:27  { Terrie egfrcr (f { Latisha inal result) status: completed order placed  { Catalina as a reflex t { Perla o bmp ordered on 06/27/24 at 1527 ordering user: interface, lab results in 07/09/24 15 { Derick 53 ordering  { Mikaela p { Adalyn rovider: freeman, genev { Maura ieve clayton, aprn authorized by: freeman, genevieve clayton, aprn o { Precious rdering mode: standard frequency { Alessandra : routine 07/09/24 1553 - class: lab collect quantity: { Nathalie  1 lab status: final result specimen information id type source collected by 24-191-012066  { Carmella blood - sde 07 { Susanna /09/24 { Melisa  1553 egfrcr  { Baylee (abnormal) resulted: 07/09/24 1751, result status: final result ordering provider: freeman, genevieve  { Vicente clayton, aprn 07/09/24 order status: completed 1553 filed by: interface, lab results in 07/09/24 1751 collected by: sde 07/09/24 1553 resulting lab: vumc cer { Aron ner lab acknowledg { Sybil ed { Paxton  by: freeman, genevieve clayt { Cash on, aprn on 07/10/24 1027 components component value reference range flag lab egfrcr 19 >=60 ml/min/1.73 cerner m2 comment: the egfrcr was calcul { Selina ated using t { Adalynn he 2021 ckd-epi egfr creatinine equation, which does not include race as a factor. this equation is validated in individu { Jayda a { Danna ls 18 years of age and older. these changes went into effect on 12/7/22 and due to the new equation, will not be trended with older egfr. values should b { Kelsie e interpreted in the context of the  { Kassidy patient's full { Esperanza  clinical presentatio { Sage n. reference: delgado, cynthia, et al. \"a unifying approach for gfr estimation: recommendations of the nkf-asn task force on reassessing the inclusion of race in diagnosing kidney disease.\" american journal of kidney diseases (2021 { Emely ) gfr categories in chroni { Reece c kidney disease (ckd) gfr gfr (ml/min/1.73 category: square meters) interpretation: g1 90 or greater normal or high* g2 60-89 mild decrease* g3a 45-59 mild to moderate decrease g3b 30-44 m { Hollie oderate to severe decrease g4 15-29 severe decrease g5 14 or less { Dollie  kidney failure *in the absence of evidence of kidney damage, neither gfr cat egory g1 nor g2 fulfill the criteria for ckd (kidney int suppl 2013;3:1-150) printed on 10/3/24 7:12 am { Letha  page 684",
    {
        "entities": [
            [
                3,
                8,
                "PERSON"
            ],
            [
                73,
                78,
                "PERSON"
            ],
            [
                199,
                205,
                "PERSON"
            ],
            [
                258,
                264,
                "PERSON"
            ],
            [
                309,
                317,
                "PERSON"
            ],
            [
                327,
                336,
                "PERSON"
            ],
            [
                405,
                413,
                "PERSON"
            ],
            [
                448,
                456,
                "PERSON"
            ],
            [
                492,
                500,
                "PERSON"
            ],
            [
                645,
                650,
                "PERSON"
            ],
            [
                677,
                684,
                "PERSON"
            ],
            [
                739,
                746,
                "PERSON"
            ],
            [
                798,
                805,
                "PERSON"
            ],
            [
                811,
                818,
                "PERSON"
            ],
            [
                997,
                1003,
                "PERSON"
            ],
            [
                1028,
                1036,
                "PERSON"
            ],
            [
                1163,
                1169,
                "PERSON"
            ],
            [
                1199,
                1204,
                "PERSON"
            ],
            [
                1255,
                1263,
                "PERSON"
            ],
            [
                1331,
                1339,
                "PERSON"
            ],
            [
                1356,
                1362,
                "PERSON"
            ],
            [
                1420,
                1427,
                "PERSON"
            ],
            [
                1445,
                1452,
                "PERSON"
            ],
            [
                1480,
                1487,
                "PERSON"
            ],
            [
                1517,
                1523,
                "PERSON"
            ],
            [
                1537,
                1542,
                "PERSON"
            ],
            [
                1617,
                1627,
                "PERSON"
            ],
            [
                1690,
                1697,
                "PERSON"
            ],
            [
                1708,
                1715,
                "PERSON"
            ],
            [
                1724,
                1731,
                "PERSON"
            ],
            [
                1847,
                1854,
                "PERSON"
            ],
            [
                1884,
                1890,
                "PERSON"
            ],
            [
                1905,
                1912,
                "PERSON"
            ],
            [
                2019,
                2025,
                "PERSON"
            ],
            [
                2061,
                2069,
                "PERSON"
            ],
            [
                2082,
                2090,
                "PERSON"
            ],
            [
                2325,
                2333,
                "PERSON"
            ],
            [
                2370,
                2380,
                "PERSON"
            ],
            [
                2400,
                2407,
                "PERSON"
            ],
            [
                2514,
                2521,
                "PERSON"
            ],
            [
                2600,
                2606,
                "PERSON"
            ],
            [
                2636,
                2644,
                "PERSON"
            ],
            [
                2666,
                2673,
                "PERSON"
            ],
            [
                2711,
                2718,
                "PERSON"
            ],
            [
                2726,
                2733,
                "PERSON"
            ],
            [
                2830,
                2837,
                "PERSON"
            ],
            [
                2843,
                2848,
                "PERSON"
            ],
            [
                2962,
                2967,
                "PERSON"
            ],
            [
                3016,
                3023,
                "PERSON"
            ],
            [
                3088,
                3094,
                "PERSON"
            ],
            [
                3104,
                3113,
                "PERSON"
            ],
            [
                3130,
                3138,
                "PERSON"
            ],
            [
                3161,
                3168,
                "PERSON"
            ],
            [
                3191,
                3198,
                "PERSON"
            ],
            [
                3223,
                3229,
                "PERSON"
            ],
            [
                3276,
                3284,
                "PERSON"
            ],
            [
                3330,
                3337,
                "PERSON"
            ],
            [
                3365,
                3370,
                "PERSON"
            ],
            [
                3406,
                3410,
                "PERSON"
            ],
            [
                3432,
                3441,
                "PERSON"
            ],
            [
                3469,
                3475,
                "PERSON"
            ],
            [
                3479,
                3484,
                "PERSON"
            ],
            [
                3562,
                3569,
                "PERSON"
            ],
            [
                3658,
                3664,
                "PERSON"
            ],
            [
                3695,
                3701,
                "PERSON"
            ],
            [
                3767,
                3774,
                "PERSON"
            ],
            [
                3823,
                3828,
                "PERSON"
            ],
            [
                3878,
                3885,
                "PERSON"
            ],
            [
                3897,
                3905,
                "PERSON"
            ],
            [
                3952,
                3961,
                "PERSON"
            ],
            [
                3977,
                3983,
                "PERSON"
            ],
            [
                4072,
                4079,
                "PERSON"
            ],
            [
                4094,
                4102,
                "PERSON"
            ],
            [
                4106,
                4113,
                "PERSON"
            ],
            [
                4139,
                4145,
                "PERSON"
            ],
            [
                4216,
                4225,
                "PERSON"
            ],
            [
                4260,
                4271,
                "PERSON"
            ],
            [
                4328,
                4337,
                "PERSON"
            ],
            [
                4431,
                4440,
                "PERSON"
            ],
            [
                4457,
                4465,
                "PERSON"
            ],
            [
                4474,
                4481,
                "PERSON"
            ],
            [
                4497,
                4504,
                "PERSON"
            ],
            [
                4609,
                4617,
                "PERSON"
            ],
            [
                4777,
                4782,
                "PERSON"
            ],
            [
                4803,
                4809,
                "PERSON"
            ],
            [
                4814,
                4821,
                "PERSON"
            ],
            [
                4853,
                4858,
                "PERSON"
            ],
            [
                5006,
                5013,
                "PERSON"
            ],
            [
                5028,
                5036,
                "PERSON"
            ],
            [
                5160,
                5166,
                "PERSON"
            ],
            [
                5170,
                5176,
                "PERSON"
            ],
            [
                5332,
                5339,
                "PERSON"
            ],
            [
                5378,
                5386,
                "PERSON"
            ],
            [
                5403,
                5413,
                "PERSON"
            ],
            [
                5437,
                5442,
                "PERSON"
            ],
            [
                5676,
                5682,
                "PERSON"
            ],
            [
                5711,
                5717,
                "PERSON"
            ],
            [
                5909,
                5916,
                "PERSON"
            ],
            [
                5984,
                5991,
                "PERSON"
            ],
            [
                6175,
                6181,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical  { Macie center dr. mrn: 04771736 { Georgina 1, dob: 7/2 { Brenton 7/1969, legal sex: m na { Scotty sh { Gus ville tn 37232-0004 adm:  { Bryanna 6/2/ { Dona 2023, d/c { Zayden : 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt university adult hospital (continued) { Marcel  procedure reporting (cont { Annika i { Nyla nued) time ecg completed 4:45 pm date ecg completed 6/3/2023 ecg interpretive report - scan on 6/3/2023 4:45 pm: ekg electrocardiogram (below)  { Annmarie 047717361 white, tyrone 03-jun-2023 16:45:30 dob: 27-jul-1969 53 yea { Kiley rs male b  { Pierre vumc-adult (1) main cam { Dion pus inpatient (15) vuh adul { Valarie t ed (03) hr 85 sinus rhythm room: c55 rr 704 * consid { Claudette er anterior infarct, acute oper: hs41 pr 185  { Ashley  change icd10: 149.9 qrsd 86 qt 355 qtcb 423 qtcf 399 -- axis account #: 1970256447037 p 71 orde { Ariel r # 373040246 ors 43 t 47 - abnorm { Katlyn al ecg enc id: 1970256447037 reas { Adolph on:  { River 149.9 edited p { Caiden revious:02-jun { Abbey -2023 14:04:31 - abnormal  { Beverley confirmed requested by: lehmann melissa cary 12 lead; standard placement electronically signed by: crossley, george h 03-ju { Araceli n-2023 19: { Chanel 17:41 avr v1 v4 avl v2 v5 ii { Ophelia i avf v3 v6 v1  { Lupe device: { Ignacio  us72134796 speed: 25 mm/sec limb: 10 mm/mv chest: 10 mm/mv 60 0.5-150 hz w ph110c p? sp { Keenan ecimen information id { Raegan  type source collected by - 06/03/ { Gillian 23 1645 ekg 12 lead resul { Bobbie ted: 06/03/23 1917, result status: final re { Alexus sult ordering provider: lehmann, melissa cary, pa-c 06/03/23 1017 order s { Tate tatus: completed resulted by: crossley, george hinton { Rylie  iii, md filed by { Matteo : edicardin  { Martina 06/03/23 1917 perf { Cortney ormed: 06/ { Kyleigh 03/23  { Nayeli 1017 - 06/03/23 1645 collected by: 06/03/23 1645 resulting la { Emery b: vumc intellispace ec { Zackery g acknowledged by: schwall, allison leigh, md on 06/07/23 1634 compone { Kaylin nts compone { Arabella nt value refere { Chandra nce range flag  { Kimberlee lab heart ra { Roslyn te 85 bpm - ecg rr interval 704 ms -  { Roxie ecg atrial rate 85 m { Rusty s ecg pr inte { Sarai rval 185 ms ecg p duration 99 ms e { Gay cg p horizontal axi { Jarvis s { Pearlie  9 deg ecg frontal axis p 71 degrees ecg printed on 10/3/24 7:13 am page 2269,vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-000 { Londyn 4 adm: 6/2/2023, d/c: 6/3/2023 06/02/2023 -  { Efrain ed to hosp-admission (discharged) in v { Ahmad anderbilt university adult hospital (continued) procedure reporting (continued) q onset 499 ms - ecg q { Magdalena rs duration 86 m { Beatriz sec - ecg qt interval 355 msec - ecg qtcb 423 msec - ecg qtcf 399 ms - ecg qrs horizontal axis -40 deg - ecg frontal axis mean qrs 43 d { Saundra egrees - ecg horizontal axis:  { Dulce initial 40 ms -15 deg - ecg frontal axis initial 40 ms 59 degrees - ecg horizontal axis: ter { Colt minal 40 ms -60 deg - ecg frontal axis terminal 40 ms 38 degrees - ecg t horizontal axis 7 { Justice 8 deg - ecg t wave axis 47 deg - ecg  { Skylar s-t horizon { Paulina tal axis 86 deg - ecg front { Cali al axis st 60 degrees - ecg ecg seve { Amari rity t - - - ecg abnormal ecg - ecg imp { Khalil ression sinus rhythm - - ecg ecg impression consider - ecg anterior infarct, acute ecg  { Chuck impre { Eleanore ssion  - - ecg change tes { Elissa ting performed by lab - abbrevi { Jamison ation nam { Juana e director address valid date range 129 ecg vumc unknown nashville tn 37232- 07/21/16 0911 - present inte { Ilene llispace e { Kenzie cg 5310 signed electronically signed by crossley, george hinton iii, md on 6/3/23 at 1917 cdt all reviewers list schwall, allison leig { Maci h, md on 6/7/2 { Kade 023 16:34 { Devin  other orders ( { Josh group 1 of 3) admission initiate observation status (completed) electronically signed by: pauw, emily kathryn, md o { Odessa n 06/02/23 1948 status: completed ordering user: pauw, { Sonny  emily kat { Ulysses hryn, md 06/02/23 1948 ordering provider: pauw, emily kathryn, md authorized by: jo { Krystle rdan, mary kate, md ordering mode { Lorrie : standard cosigning events electronically cosigned by jordan, mary kate, md 0 { Arline 6/02/23 2130 for ordering frequency: routine once 06/02/23 1949 - 1 occurr { Loraine ence clas { Sharron s: ho { Deirdre spital performed quantity: 1 instance release { Lilian d by: { Liza  pauw, emily kathryn, md (auto-released) 6/2/2023 7:48 pm questionnai { Madilyn re question answer service  { Gunner g { Carey eneral { Belle  internal medicine general medicine teams riven obs a are they the primary team? yes diagno { Tommie sis ams (altered mental { Ned  st { Alyce atus) futur { Prince e attending provider schwall, allison leigh  { Everly printed on 10/3/24 7:13 am page 2270",
    {
        "entities": [
            [
                50,
                56,
                "PERSON"
            ],
            [
                83,
                92,
                "PERSON"
            ],
            [
                106,
                114,
                "PERSON"
            ],
            [
                140,
                147,
                "PERSON"
            ],
            [
                152,
                156,
                "PERSON"
            ],
            [
                184,
                192,
                "PERSON"
            ],
            [
                199,
                204,
                "PERSON"
            ],
            [
                216,
                223,
                "PERSON"
            ],
            [
                335,
                342,
                "PERSON"
            ],
            [
                371,
                378,
                "PERSON"
            ],
            [
                382,
                387,
                "PERSON"
            ],
            [
                533,
                542,
                "PERSON"
            ],
            [
                613,
                619,
                "PERSON"
            ],
            [
                632,
                639,
                "PERSON"
            ],
            [
                665,
                670,
                "PERSON"
            ],
            [
                700,
                708,
                "PERSON"
            ],
            [
                765,
                775,
                "PERSON"
            ],
            [
                823,
                830,
                "PERSON"
            ],
            [
                929,
                935,
                "PERSON"
            ],
            [
                972,
                979,
                "PERSON"
            ],
            [
                1015,
                1022,
                "PERSON"
            ],
            [
                1029,
                1035,
                "PERSON"
            ],
            [
                1052,
                1059,
                "PERSON"
            ],
            [
                1076,
                1082,
                "PERSON"
            ],
            [
                1111,
                1120,
                "PERSON"
            ],
            [
                1246,
                1254,
                "PERSON"
            ],
            [
                1267,
                1274,
                "PERSON"
            ],
            [
                1305,
                1313,
                "PERSON"
            ],
            [
                1331,
                1336,
                "PERSON"
            ],
            [
                1346,
                1354,
                "PERSON"
            ],
            [
                1445,
                1452,
                "PERSON"
            ],
            [
                1476,
                1483,
                "PERSON"
            ],
            [
                1520,
                1528,
                "PERSON"
            ],
            [
                1556,
                1563,
                "PERSON"
            ],
            [
                1609,
                1616,
                "PERSON"
            ],
            [
                1692,
                1697,
                "PERSON"
            ],
            [
                1753,
                1759,
                "PERSON"
            ],
            [
                1779,
                1786,
                "PERSON"
            ],
            [
                1801,
                1809,
                "PERSON"
            ],
            [
                1830,
                1838,
                "PERSON"
            ],
            [
                1851,
                1859,
                "PERSON"
            ],
            [
                1868,
                1875,
                "PERSON"
            ],
            [
                1939,
                1945,
                "PERSON"
            ],
            [
                1971,
                1979,
                "PERSON"
            ],
            [
                2052,
                2059,
                "PERSON"
            ],
            [
                2073,
                2082,
                "PERSON"
            ],
            [
                2100,
                2108,
                "PERSON"
            ],
            [
                2126,
                2136,
                "PERSON"
            ],
            [
                2151,
                2158,
                "PERSON"
            ],
            [
                2198,
                2204,
                "PERSON"
            ],
            [
                2227,
                2233,
                "PERSON"
            ],
            [
                2249,
                2255,
                "PERSON"
            ],
            [
                2292,
                2296,
                "PERSON"
            ],
            [
                2318,
                2325,
                "PERSON"
            ],
            [
                2329,
                2337,
                "PERSON"
            ],
            [
                2543,
                2550,
                "PERSON"
            ],
            [
                2597,
                2604,
                "PERSON"
            ],
            [
                2645,
                2651,
                "PERSON"
            ],
            [
                2756,
                2766,
                "PERSON"
            ],
            [
                2785,
                2793,
                "PERSON"
            ],
            [
                2931,
                2939,
                "PERSON"
            ],
            [
                2972,
                2978,
                "PERSON"
            ],
            [
                3073,
                3078,
                "PERSON"
            ],
            [
                3171,
                3179,
                "PERSON"
            ],
            [
                3219,
                3226,
                "PERSON"
            ],
            [
                3240,
                3248,
                "PERSON"
            ],
            [
                3278,
                3283,
                "PERSON"
            ],
            [
                3322,
                3328,
                "PERSON"
            ],
            [
                3370,
                3377,
                "PERSON"
            ],
            [
                3467,
                3473,
                "PERSON"
            ],
            [
                3481,
                3490,
                "PERSON"
            ],
            [
                3518,
                3525,
                "PERSON"
            ],
            [
                3559,
                3567,
                "PERSON"
            ],
            [
                3579,
                3585,
                "PERSON"
            ],
            [
                3693,
                3699,
                "PERSON"
            ],
            [
                3712,
                3719,
                "PERSON"
            ],
            [
                3856,
                3861,
                "PERSON"
            ],
            [
                3878,
                3883,
                "PERSON"
            ],
            [
                3895,
                3901,
                "PERSON"
            ],
            [
                3919,
                3924,
                "PERSON"
            ],
            [
                4042,
                4049,
                "PERSON"
            ],
            [
                4106,
                4112,
                "PERSON"
            ],
            [
                4125,
                4133,
                "PERSON"
            ],
            [
                4219,
                4227,
                "PERSON"
            ],
            [
                4263,
                4270,
                "PERSON"
            ],
            [
                4351,
                4358,
                "PERSON"
            ],
            [
                4435,
                4443,
                "PERSON"
            ],
            [
                4455,
                4463,
                "PERSON"
            ],
            [
                4471,
                4479,
                "PERSON"
            ],
            [
                4527,
                4534,
                "PERSON"
            ],
            [
                4542,
                4547,
                "PERSON"
            ],
            [
                4619,
                4627,
                "PERSON"
            ],
            [
                4657,
                4664,
                "PERSON"
            ],
            [
                4668,
                4674,
                "PERSON"
            ],
            [
                4683,
                4689,
                "PERSON"
            ],
            [
                4783,
                4790,
                "PERSON"
            ],
            [
                4816,
                4820,
                "PERSON"
            ],
            [
                4826,
                4832,
                "PERSON"
            ],
            [
                4846,
                4853,
                "PERSON"
            ],
            [
                4900,
                4907,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232- { Abram 0004 adm { Marta : 8/14/2024, d/c: 8/15/2024 08/14/2024 - ed in vanderbilt emergency department (continued) ed  { Jerry care timeline (continued) 22:53:29 orders new - bmp; adult d { Melina iet other; carbohydrate controlled (diabetic); sodium mcquitty, karrah, acknowledged zirconium cyclosilicate (lokelma) oral powder in packet  { Danica 10 g rn discontinued  { Janae - sodium zirconi { Jacklyn um cyclosilicate (lokelma) oral powder in packet 10 g 22:53:54 orde { Elvis rs completed poc lab: glucose; source: ca { Octavia pillary mcquit { Jeanie ty, karrah, rn 22:53:54 poc lab: glucose; poc lab:  { Wiley glucose; source: ca { Carley pil { Avis lary mcquitty, karrah, sourc { Tyler e: capillary  { Santos rn completed 22:56 ed { Shayna  reassessment ed reas { Reese sessment szymanski, elliot iv site check: done lynn, rn cardiac reassessment: done vascular/perfusion reasse { Rhea ssment: done respiratory reassessment: done psychosocial/behavioral r { Al eassessment: done safet { Brandie y precauti { Cathryn on: fall p { Russel re { Asa cautions fall/injury pr { Augustus ecaution: environmental asses { Emory sment completed 22:58 medication given sodium { Lilliana  zirconium cyclosilicate (lokelma) oral powder in packet 10 g - szymanski, elliot dose: 10 g ; route: oral ; scheduled time: 2300 l { Consuelo y { Nanette nn, rn 22:58 remove nurse maibuecher, jake walke { Suzette r, rn remo { Chrystal ved as registere { Curt d nurse maibuecher, jake walker { Kobe , rn 22:59:15 assign attending pauw, emily kathryn, md assigned a { Raelynn s attending pauw, emily kathryn, md 22:59:15 assign physician pauw, emily kathryn, md 23:00:15 complete urinalysis w/micro&rfx cu { Althea lture sz { Joselyn ymanski, ellio { Nicolette t collection lynn, rn information for urinalysis w/micro&rfx culture completed 23:00:25 print label for bmp bmp - type: blood szymanski, ell { Charmaine i { Major ot completed lynn, rn 23:00:25 print label for urinalysis w/micro&rfx culture - type: urine szymans { Chantel ki, elliot urinalysis lynn, { Parker  rn w/micro&rfx culture completed 23:04 psofa psofa scoring (most recent) epic, user overall score: 0 respira { Anya tory: () coagulation: 0 liver (bilirubin): () 23:05 psofa { Blair  psofa scoring (most recent) epic,  { Karter user cardi { Lina ovascular: () central nervous s { Bruno yste { Porter m: 0 renal: () { Tyrell  other flowsheet entries overall score: 0 sofa { Carmen  respiratory: 0 coagulation: 0 liver: 0 cardiovascular: 0 central nervous system: 0 renal: 0 23:08 collect bmp { Uriel  bmp - type: blood szymanski, elliot completed lynn, rn printed on 10/3/24  { Humberto 7:12 am page 283,vumc  { Zelma adult  { Millard hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashvi { Pierce lle tn 37232-0004 adm: 8/14/2024, d/c: 8/15/2024 08/14/2024 - ed in vanderbilt emergency department (continued) ed care timeline (continued) 23:08 c { Coy ollect urinalysis urinalysis w/micro&rfx culture - type: uri { Javon ne szymanski, elliot w/micro&rfx lynn, rn culture completed 23:08 specimens urinalysis w/micro&rfx culture - id: 24-227-015075 type: u { Hal rine bmp - szymanski, elliot collected id: 24-227-015074 type:  { Coleman b { Libby lood lyn { Ainsley n, rn 23:08 specimens egfrcr - id: 24-227-015074 { Liana  type: blood urinalysis microscopic - id: collected 24-227-015 { Gemma 075 type: urine 23:08:41 lab ordered egfrcr interface, l { Ernie ab result { Samara s in 23:14 intake/output output  { Galen (ml) szymanski, elliot urine: 425 ml lynn, rn u { Nona rine assessment urinar { Casandra y incontinence:   { Bart color: yellow/straw urine character: pale 23:27:15 { Corina   { Damion ed note filed by ed prov note filed by morro { Nehemiah w, seyjil shantha turpin, md morrow, seyjil resident/app shantha tur { Rickie pin, md 23:29:26 lab ordered urinalysis microscopic interface, lab results in  { Shelby 23:33:12 bmp resulted abnormal result collecte { Justice d: 8/14/ { Janna 2024 23:08 last updated: 8/14/2024 23 { Jax :3 { Earline 3 interface, lab status: final result sodium level: 135 mmol/ { Itzel l [ref  { Ollie range: 136 - 145] results in potassium level: 6.4 mmol/l [ref range: 3.3 - 4.8] (mod { Lavonne erate hemolysis - result i { Kermit ncreased. plasma reference ran { Norris ges are shown, please note that serum reference ranges are higher than plasma. ) chlorid { Marianna e level: 107 mmol/l [ref range: 98 - 107] carbon dioxi { Kaitlynn de: 19 mmol/l [ { Thurman ref range: 22 - 29] (moderate hemolysis - result { Orion  decreased.) glucose level: 373 mg/dl a [ref range: 70 - 99] blood urea nitrogen: 38 mg/dl a [ref range: 8 - 26] creatinine level: 3.91 mg/dl a [ref range: 0.72 - 1.25] calcium level total: 8.8 mg/d { Cristal l [re { Lyric f range: 8.4  { Augusta - 10.5] anion gap: 9 (this test was perfo { Destinee rmed at: vanderbilt hospital laborato { William ry, clia #44d0659066,a { Kathrine dam seegmiller md, phd, 13 { Miracle 01 medical center drive,  { Kaleigh 4605 tvc,nashville,tn,37232,) 23:33:13 lab resulted (final result) bmp interface, lab resu { Aldo lts in 23:33:13 lab resulted (final result) egfrcr interface, lab results in 23:33:13 collect bmp bmp interface, lab  { Mina discontinued results in 2 { Aniya 3:33:13 print { Barney  label fo { Monroe r bmp bmp interface, lab discontinued results in prin { Adelyn ted on 10/3/24 { Jamar  7:12 am page 284",
    {
        "entities": [
            [
                125,
                131,
                "PERSON"
            ],
            [
                142,
                148,
                "PERSON"
            ],
            [
                245,
                251,
                "PERSON"
            ],
            [
                314,
                321,
                "PERSON"
            ],
            [
                465,
                472,
                "PERSON"
            ],
            [
                496,
                502,
                "PERSON"
            ],
            [
                521,
                529,
                "PERSON"
            ],
            [
                599,
                605,
                "PERSON"
            ],
            [
                649,
                657,
                "PERSON"
            ],
            [
                674,
                681,
                "PERSON"
            ],
            [
                735,
                741,
                "PERSON"
            ],
            [
                763,
                770,
                "PERSON"
            ],
            [
                776,
                781,
                "PERSON"
            ],
            [
                812,
                818,
                "PERSON"
            ],
            [
                834,
                841,
                "PERSON"
            ],
            [
                865,
                872,
                "PERSON"
            ],
            [
                896,
                902,
                "PERSON"
            ],
            [
                1013,
                1018,
                "PERSON"
            ],
            [
                1090,
                1093,
                "PERSON"
            ],
            [
                1119,
                1127,
                "PERSON"
            ],
            [
                1140,
                1148,
                "PERSON"
            ],
            [
                1161,
                1168,
                "PERSON"
            ],
            [
                1173,
                1177,
                "PERSON"
            ],
            [
                1203,
                1212,
                "PERSON"
            ],
            [
                1244,
                1250,
                "PERSON"
            ],
            [
                1298,
                1307,
                "PERSON"
            ],
            [
                1441,
                1450,
                "PERSON"
            ],
            [
                1454,
                1462,
                "PERSON"
            ],
            [
                1513,
                1521,
                "PERSON"
            ],
            [
                1534,
                1543,
                "PERSON"
            ],
            [
                1562,
                1567,
                "PERSON"
            ],
            [
                1601,
                1606,
                "PERSON"
            ],
            [
                1674,
                1682,
                "PERSON"
            ],
            [
                1814,
                1821,
                "PERSON"
            ],
            [
                1832,
                1840,
                "PERSON"
            ],
            [
                1857,
                1867,
                "PERSON"
            ],
            [
                2010,
                2020,
                "PERSON"
            ],
            [
                2024,
                2030,
                "PERSON"
            ],
            [
                2132,
                2140,
                "PERSON"
            ],
            [
                2170,
                2177,
                "PERSON"
            ],
            [
                2289,
                2294,
                "PERSON"
            ],
            [
                2354,
                2360,
                "PERSON"
            ],
            [
                2398,
                2405,
                "PERSON"
            ],
            [
                2418,
                2423,
                "PERSON"
            ],
            [
                2457,
                2463,
                "PERSON"
            ],
            [
                2470,
                2477,
                "PERSON"
            ],
            [
                2494,
                2501,
                "PERSON"
            ],
            [
                2550,
                2557,
                "PERSON"
            ],
            [
                2670,
                2676,
                "PERSON"
            ],
            [
                2754,
                2763,
                "PERSON"
            ],
            [
                2788,
                2794,
                "PERSON"
            ],
            [
                2803,
                2811,
                "PERSON"
            ],
            [
                2912,
                2919,
                "PERSON"
            ],
            [
                3070,
                3074,
                "PERSON"
            ],
            [
                3137,
                3143,
                "PERSON"
            ],
            [
                3280,
                3284,
                "PERSON"
            ],
            [
                3350,
                3358,
                "PERSON"
            ],
            [
                3362,
                3368,
                "PERSON"
            ],
            [
                3379,
                3387,
                "PERSON"
            ],
            [
                3438,
                3444,
                "PERSON"
            ],
            [
                3509,
                3515,
                "PERSON"
            ],
            [
                3574,
                3580,
                "PERSON"
            ],
            [
                3592,
                3599,
                "PERSON"
            ],
            [
                3634,
                3640,
                "PERSON"
            ],
            [
                3690,
                3695,
                "PERSON"
            ],
            [
                3720,
                3729,
                "PERSON"
            ],
            [
                3749,
                3754,
                "PERSON"
            ],
            [
                3807,
                3814,
                "PERSON"
            ],
            [
                3818,
                3825,
                "PERSON"
            ],
            [
                3872,
                3881,
                "PERSON"
            ],
            [
                3952,
                3959,
                "PERSON"
            ],
            [
                4040,
                4047,
                "PERSON"
            ],
            [
                4096,
                4104,
                "PERSON"
            ],
            [
                4115,
                4121,
                "PERSON"
            ],
            [
                4161,
                4165,
                "PERSON"
            ],
            [
                4170,
                4178,
                "PERSON"
            ],
            [
                4242,
                4248,
                "PERSON"
            ],
            [
                4258,
                4264,
                "PERSON"
            ],
            [
                4351,
                4359,
                "PERSON"
            ],
            [
                4388,
                4395,
                "PERSON"
            ],
            [
                4428,
                4435,
                "PERSON"
            ],
            [
                4526,
                4535,
                "PERSON"
            ],
            [
                4592,
                4601,
                "PERSON"
            ],
            [
                4619,
                4627,
                "PERSON"
            ],
            [
                4678,
                4684,
                "PERSON"
            ],
            [
                4885,
                4893,
                "PERSON"
            ],
            [
                4901,
                4907,
                "PERSON"
            ],
            [
                4923,
                4931,
                "PERSON"
            ],
            [
                4975,
                4984,
                "PERSON"
            ],
            [
                5024,
                5032,
                "PERSON"
            ],
            [
                5057,
                5066,
                "PERSON"
            ],
            [
                5095,
                5103,
                "PERSON"
            ],
            [
                5131,
                5139,
                "PERSON"
            ],
            [
                5232,
                5237,
                "PERSON"
            ],
            [
                5357,
                5362,
                "PERSON"
            ],
            [
                5390,
                5396,
                "PERSON"
            ],
            [
                5412,
                5419,
                "PERSON"
            ],
            [
                5431,
                5438,
                "PERSON"
            ],
            [
                5494,
                5501,
                "PERSON"
            ],
            [
                5518,
                5524,
                "PERSON"
            ]
        ]
    }
),(
    "vumc vis midtown white, tyrone 337 22nd ave n mrn: 04771736 { Phoenix 1, dob: 7/27/ { Kinley 1969, { Meaghan  legal sex: m  { Carina nashville tn 37203 adm: 4/16/2024, d/c: 4/16/2024  { Darian 04/16/2024 - fl vid { Terra eo swallow w spe { Earnestine ech in vanderbilt imaging services midtown (continued) imag { Janell ing reports (contin { Blair ued) co { Kellen mparison: radiograph on 8/22/2023. technique: patient position: later { Fernanda al with ap esophageal sweep consistency used: barium contrast was utilized. please refer to speech  { Mari pathology documentatio { Roseann n of consistencies administered. a barium pill was a { Regan lso administered. ex { Maximilian am  { Rashad performed with speech pathologist. fluoro time: 2.15 min air k { Adrian erma: 27.61 mgy dap: { Joesph  5958.4 mgy-cm2 supervision stateme { Dee nt: i, samuel ostrum, md, was prese { Mohamed nt for  { Irwin the entire procedure. findings: scout images show degenerative  { Graciela changes of the cervical spine. fluoroscop { Margery ic observations: oral phase: within normal limits. pharyngeal p { Kareem hase: penetration without aspiration. limited esop { Tiffani hageal sweep: wi { Shania thin normal limits. unchanged punctate rounded metallic density overlying the right upper { Marcie  abd { Adolfo omen. impression: 1. penetra { Deja tion without aspiration. 2. please see the report by the speech pathologist for additional details and recommendations. electronically signed by samuel ostrum, md on 4/16/2024 4:12 pm acknowledged by: rebula, emily rose kueser, pa { Reva -c on 04/17/24 1258 testing performed by lab - abbreviation name director address valid date range 622 - powerscribe vumc ps360 unk { Deidre nown nas { Alvaro hville { Anika  t { Dayna n 05/18/2 { Brice 0 1026 - present 37232-5310 indications speech disturbance, unsp { Celina ecified type [r47.9 (icd-10-cm { Deanne )] anosmia [r43.0 (icd-10-cm)] { Donte  dysphagia, oral pha { Kaydence se [r13.11 (icd-10-cm)] si { Tierra gned electronically signed by ostrum, samuel gilbert, md on 4/16/24 at 1612 cdt all reviewers { Lyndsey  list rebula, emily rose { Gia  kueser, pa- { Eddie c on 4/17/2024 12:58 other orders medications printed on 10/3/24 7:12 am page 1069,vumc { Kierra  vis midtown white, tyrone 337 22nd ave n mrn: { Lynnette  047717361, dob: 7/27/1969, legal sex: m nashville t { Dayton n { Rhiannon  37203 adm: 4/16/202 { Forest 4, d/c: 4/16/2024 04/16/2024 - fl video swallow w speech in vanderbilt imagin { Denny g services midtown (continued) { Edwina  other orders (continued) barium  { Jerri sulfate (varibar thin liquid) 81 % { Houston  (w/w) oral powder 70 ml (completed) electronically signed { Markus  by: henry, shel { Braylon ly monique, r.t.(r)(arrt) on 04/16/24 1602 status: compl { Aspen et { Hans ed ordering user: henry, shelly monique, r.t.(r)(arrt) { Carlene  04/16 { Beckett /24 ordering provider: rebu { Anahi la, emily rose { Jeffry  kueser, pa-c 1602 authorized by: rebula, emily rose kueser, pa-c ordering mode per activated protocol frequency: rout { Rosella ine once 04/ { Lelia 16/24 1615 - 1 occurrence class: normal p { Deshawn ackage: 32909-105-10 barium sulfate (varibar pudding) 40 % (w/v), 30% (w/w) oral paste 15 ml (completed) electronically signed by: henry, shelly monique, r.t.(r)(arrt) on 04/16/ { Madisyn 24 1602 status: com { Harris pleted ordering user: henry, shelly monique, { Luann  t.(r)(arrt) 04/16/24 ordering provider: rebula,  { Jett emily rose kueser, pa-c 1602 authorized by: rebula, em { Alden ily { Ester  rose kueser, pa-c ordering mode: per activated protocol frequenc { Reyna y: routine once 04/16/24 1615 - 1 occurrence class: normal package: 32909-125-22 barium sulfate (e-z-disk) tablet 700 mg (completed { Denver ) electronically { Cohen  signed by: henry, shelly monique, r.t.(r)(arrt) on 04/16/24 1602 status: completed ordering us { Minerva er: henry, shelly monique, r.t.(r)(arrt) 04/16 { Vivienne /24 ordering { Hailee  provider: rebula, emily  { Quintin rose kueser, pa-c 1602 author { Elisha ized { Jaqueline  by: rebula, emily rose kueser, pa-c ordering mode: per activated protocol frequency: routine once 04/16/24 1615 - 1 occurrence class: normal package: 103 { Wilmer 61-778-31 ref { Lexie erral diagnostic imaging #17565740 [last  { Rosalinda edited by epic, user on 8/26/2024 0923] reason: specialty services required priority: routine class: internal status: closed - system closed - visit(s) completed status updated on: 3/21/2024 valid dates: from 2/ { Maximiliano 28/20 { Cheyanne 24 to 4/30/2025 referred from location: vumc adult medical center east department: otolaryngology facial plastics mce 7 dep { Alphonso artment phone: 615-322-6180 provider: rebula, emily rose kueser, pa-c provider phone: 615-322-6180 provider ad { Rosanna dress: 1215 21st avenue south medical  { Rosanne center east, south tower, suite 7209 nashville tn 37232-8605 referred to specialty: radiology visits requested: 1 authorized: 1 co { Mary mplete { Kieran d: 1 schedu { Jayme led: 0 procedures img743 - fluoroscopy vi { Bryon deo swa { Armand llow with speech number requeste { Carlo d: 1 n { Addyson umber approved: 1 dia { Katheryn gnoses r47.9 (icd-10-cm) - spee { Gideon ch disturbance, unspecified type r { Kory 43. { Stephany 0 (icd-10-cm) - anosmia r1 { Mariam 3.11  { Truman (icd-10-cm) - dysphagia, oral phase order fluoroscopy video swallow with speech [421649521] electronically signed by: rebula, emily rose kueser, pa-c on 02/28/24 1 { Kathie 458 status: completed printed on 10/3/24 7:12 am page 1070",
    {
        "entities": [
            [
                62,
                70,
                "PERSON"
            ],
            [
                86,
                93,
                "PERSON"
            ],
            [
                101,
                109,
                "PERSON"
            ],
            [
                126,
                133,
                "PERSON"
            ],
            [
                186,
                193,
                "PERSON"
            ],
            [
                215,
                221,
                "PERSON"
            ],
            [
                240,
                251,
                "PERSON"
            ],
            [
                313,
                320,
                "PERSON"
            ],
            [
                342,
                348,
                "PERSON"
            ],
            [
                358,
                365,
                "PERSON"
            ],
            [
                437,
                446,
                "PERSON"
            ],
            [
                548,
                553,
                "PERSON"
            ],
            [
                578,
                586,
                "PERSON"
            ],
            [
                641,
                647,
                "PERSON"
            ],
            [
                670,
                681,
                "PERSON"
            ],
            [
                687,
                694,
                "PERSON"
            ],
            [
                759,
                766,
                "PERSON"
            ],
            [
                789,
                796,
                "PERSON"
            ],
            [
                834,
                838,
                "PERSON"
            ],
            [
                876,
                884,
                "PERSON"
            ],
            [
                894,
                900,
                "PERSON"
            ],
            [
                966,
                975,
                "PERSON"
            ],
            [
                1019,
                1027,
                "PERSON"
            ],
            [
                1093,
                1100,
                "PERSON"
            ],
            [
                1153,
                1161,
                "PERSON"
            ],
            [
                1180,
                1187,
                "PERSON"
            ],
            [
                1279,
                1286,
                "PERSON"
            ],
            [
                1293,
                1300,
                "PERSON"
            ],
            [
                1331,
                1336,
                "PERSON"
            ],
            [
                1569,
                1574,
                "PERSON"
            ],
            [
                1708,
                1715,
                "PERSON"
            ],
            [
                1726,
                1733,
                "PERSON"
            ],
            [
                1742,
                1748,
                "PERSON"
            ],
            [
                1753,
                1759,
                "PERSON"
            ],
            [
                1771,
                1777,
                "PERSON"
            ],
            [
                1844,
                1851,
                "PERSON"
            ],
            [
                1884,
                1891,
                "PERSON"
            ],
            [
                1924,
                1930,
                "PERSON"
            ],
            [
                1953,
                1962,
                "PERSON"
            ],
            [
                1991,
                1998,
                "PERSON"
            ],
            [
                2094,
                2102,
                "PERSON"
            ],
            [
                2129,
                2133,
                "PERSON"
            ],
            [
                2148,
                2154,
                "PERSON"
            ],
            [
                2244,
                2251,
                "PERSON"
            ],
            [
                2300,
                2309,
                "PERSON"
            ],
            [
                2364,
                2371,
                "PERSON"
            ],
            [
                2375,
                2384,
                "PERSON"
            ],
            [
                2407,
                2414,
                "PERSON"
            ],
            [
                2494,
                2500,
                "PERSON"
            ],
            [
                2533,
                2540,
                "PERSON"
            ],
            [
                2576,
                2582,
                "PERSON"
            ],
            [
                2619,
                2627,
                "PERSON"
            ],
            [
                2688,
                2695,
                "PERSON"
            ],
            [
                2714,
                2722,
                "PERSON"
            ],
            [
                2781,
                2787,
                "PERSON"
            ],
            [
                2792,
                2797,
                "PERSON"
            ],
            [
                2854,
                2862,
                "PERSON"
            ],
            [
                2871,
                2879,
                "PERSON"
            ],
            [
                2909,
                2915,
                "PERSON"
            ],
            [
                2932,
                2939,
                "PERSON"
            ],
            [
                3060,
                3068,
                "PERSON"
            ],
            [
                3083,
                3089,
                "PERSON"
            ],
            [
                3133,
                3141,
                "PERSON"
            ],
            [
                3321,
                3329,
                "PERSON"
            ],
            [
                3351,
                3358,
                "PERSON"
            ],
            [
                3405,
                3411,
                "PERSON"
            ],
            [
                3463,
                3468,
                "PERSON"
            ],
            [
                3525,
                3531,
                "PERSON"
            ],
            [
                3537,
                3543,
                "PERSON"
            ],
            [
                3611,
                3617,
                "PERSON"
            ],
            [
                3751,
                3758,
                "PERSON"
            ],
            [
                3777,
                3783,
                "PERSON"
            ],
            [
                3881,
                3889,
                "PERSON"
            ],
            [
                3938,
                3947,
                "PERSON"
            ],
            [
                3962,
                3969,
                "PERSON"
            ],
            [
                3997,
                4005,
                "PERSON"
            ],
            [
                4037,
                4044,
                "PERSON"
            ],
            [
                4051,
                4061,
                "PERSON"
            ],
            [
                4218,
                4225,
                "PERSON"
            ],
            [
                4241,
                4247,
                "PERSON"
            ],
            [
                4291,
                4301,
                "PERSON"
            ],
            [
                4515,
                4527,
                "PERSON"
            ],
            [
                4535,
                4544,
                "PERSON"
            ],
            [
                4670,
                4679,
                "PERSON"
            ],
            [
                4792,
                4800,
                "PERSON"
            ],
            [
                4841,
                4849,
                "PERSON"
            ],
            [
                4982,
                4987,
                "PERSON"
            ],
            [
                4996,
                5003,
                "PERSON"
            ],
            [
                5017,
                5023,
                "PERSON"
            ],
            [
                5067,
                5073,
                "PERSON"
            ],
            [
                5083,
                5090,
                "PERSON"
            ],
            [
                5125,
                5131,
                "PERSON"
            ],
            [
                5140,
                5148,
                "PERSON"
            ],
            [
                5172,
                5181,
                "PERSON"
            ],
            [
                5215,
                5222,
                "PERSON"
            ],
            [
                5259,
                5264,
                "PERSON"
            ],
            [
                5270,
                5279,
                "PERSON"
            ],
            [
                5308,
                5315,
                "PERSON"
            ],
            [
                5323,
                5330,
                "PERSON"
            ],
            [
                5496,
                5503,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hundred  { Ari oaks white, tyrone 719 thompson lane, nashville mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date: 10/17/2023 10/17/ { Lilah 2023 - office visit  { Evie in vanderbilt one hundred oaks primary care north (continued) referral (continued) t { Ayanna riage information decision: none schedule by date: 11/ { Monty 1/2023  { Karyn coverages amerivantage wellpoint medicare plan: amerivantage covered: cov { Agustin ered from: 6/1/2023 to: 2/29/2024 amerigroup wellpoint ma member #: 768w12411  { Ursula zzzmcaid of tennessee p { Tyra lan: m { Kane edicaid supplemental cov { Sanford ered: covered from: 6 { Ward /1/20 { Colten 22 to: 12/31/2023 memb { Cullen er # { Carlie : td5 { Queen 25606373 flowsheets c { Marietta ustom formula data row na { Ahmed me 10/17/23 142 { Bria 4 10/17/23 { Maynard  1426 other depression score 0 -dd at 10/17/23 1424 - (if g { Tess reater than or = 3, fill out phq- 9) phq-8 score 0 -dd a { Jannie t 10/17/ { Gerry 23 1424 - total score 0 -dd at 10/17/23 1424 - calculated firefly negative { Merle  -dd at - depr { Renae essi { Wendi on 10/17/23 1424 s { Royal creening result: phq-2 to 9 or epds must be  { Fletcher completed (at this encounter) for this to populate. if the calculate { Odell d result is missing, make sure the phq- { Gunnar 2 to  { Kaelyn 9 or epds has been documented. bmi (calc { Jordon ulated)  { Kiersten -  { Deon 25.1 -dd at 10/17/23 1427 percent excess - -1 { Annabel 03.74 percent -dd weight loss at 10/17/23 1427 ibw in lbs - 178.05 -dd at 10/17/23 (bariatric) 1427 { Joelle  weight change - 83.78 kg -dd at since last visit { Tillie  10/17/23 1427 ibw in kg - 80.76 -dd at 10/17/23 (bariatric) 1427 bmi (calculated) - 25.1 -dd at 10/17/23 1427 bsa (calculated - { Merlin  - 2.06 sq meters -dd sq m) at 10/ { Chelsie 17/23 1427 { Leeann  mosteller - 2.063 -dd at 10/17/2 { Lourdes 3 1427 dubois - 2.06 -dd at 10/17/23 1427 haycock  { Amira - 2.07 -dd at 10/17/ { Kirby 23 { Noel  printed on 10/3/24 7:13 am page 161 { Mohammad 1 { Corine ,vumc a { Isabell dult one hundred oaks white, tyrone 719 thompson lane, nash { Hollis ville mrn: 04771 { Madelynn 7361, dob: 7/27/1969, le { Finley gal sex: m nashville tn 37204 visit date: 10/17/2023 10/17/2023 - office visit in vanderbilt o { Vito ne hundred oaks primary care north (continued) flowsheets (continued) 1 { Kaila 427 gehan & george 2.072 -dd at 10/17/23 1427 bmi (calculated) - 25.1  { Princess -dd at 10/17/ { Halle 23 1427 ibw (lb) 178.05 -dd at 10/17/23 1427 weight change - 83.78 kg -dd at from preop 10/17/23 { Felicity  1427 weight change - 83.78 kg -dd at since last visit 10/17/23 1427 weight chang { Donnell e - 83. { Kris 78 -dd at 10/17/23 from preop (kg) 1427 ibw in lbs 184.38 -dd at 10/17/23  { Jarred (bariatric) 1 { Brayan 427 percent { Aline  weight 83.78 lb { Amari s -dd at c { Karson hange since 10/17/23 1427 preop percent weight - 2956.93 percent { Ronny  -dd ch { Sheree ange s { Muhammad ince at 10/1 { Tamera 7/23 1427 last vis { Maia it current ebw - 0.37 lb -dd at 1 { Frederic 0/17/23 1427 current bmi -  { Tyree 25.1 -dd at 10/17/23 (calcu { Braeden lated) 1427 percent weight  { Remington - 78724.8 percent -dd change since at { Margot  10/17/23 1427 last { Ezequiel  visit mifflin- { Jena st je { Nita or - 1716.13 -dd at rmr (kcal) 1 { Stacey 0/17/23 1427 { Amara  weight cha { Rowan nge - 2.07 percent -dd at since last visit { Elle  10/17/23 { Tamra  1427 (%) bmi last visit - 24.5 -dd at 10/17/23 (calculated) 1427 bmi change - 0.6 -dd at 10/17/23 since last  { Marci visit 1427 bmi change - 2.45 -dd at 10/17/23 since last v { Camron isit 1 { Bailee 427 (percent) bmi change - 2.45 percent -dd at since last visit 10/17/23 1427 (%) bsa (calculated - - 2.06 sq meters -dd sq m at 10/17/23 1427 weig { Concetta ht change - 184.71 lbs -dd at since preop (lbs) 10/17/2 { Hester 3 { Mohammed  1427 initial excess - -80.76 lbs -dd at weight 10 { Melva /17/2 { Tammi 3 1427 weight chang { Nigel e - 3.75 lbs -dd at since last visit 10/17/23 1427 (lbs) current weight - 83.8 kg at 10/17/20 { Earlene 23 2:26 pm -dd at 10/17/23 1427 ibw/kg - { Isaias  77.62 kg -dd at ( { Delmar calculated) male 10/1 { Haven 7/23 1427 bmi (calculated) - 25.1 -dd  { Arya at 10/17/23 1427  { Lesly percent excess - -103.74  { Janessa percent -dd weight loss  { Lessie at 10/17/23 1427 ibw  { Micaela in kg - 80 { Kourtney .76 kg -dd at p { Thea rinted on 10/3/24 7:13 am page 1612",
    {
        "entities": [
            [
                26,
                30,
                "PERSON"
            ],
            [
                174,
                180,
                "PERSON"
            ],
            [
                203,
                208,
                "PERSON"
            ],
            [
                295,
                302,
                "PERSON"
            ],
            [
                359,
                365,
                "PERSON"
            ],
            [
                375,
                381,
                "PERSON"
            ],
            [
                457,
                465,
                "PERSON"
            ],
            [
                546,
                553,
                "PERSON"
            ],
            [
                579,
                584,
                "PERSON"
            ],
            [
                593,
                598,
                "PERSON"
            ],
            [
                625,
                633,
                "PERSON"
            ],
            [
                657,
                662,
                "PERSON"
            ],
            [
                670,
                677,
                "PERSON"
            ],
            [
                702,
                709,
                "PERSON"
            ],
            [
                716,
                723,
                "PERSON"
            ],
            [
                731,
                737,
                "PERSON"
            ],
            [
                761,
                770,
                "PERSON"
            ],
            [
                798,
                804,
                "PERSON"
            ],
            [
                822,
                827,
                "PERSON"
            ],
            [
                840,
                848,
                "PERSON"
            ],
            [
                910,
                915,
                "PERSON"
            ],
            [
                974,
                981,
                "PERSON"
            ],
            [
                992,
                998,
                "PERSON"
            ],
            [
                1075,
                1081,
                "PERSON"
            ],
            [
                1098,
                1104,
                "PERSON"
            ],
            [
                1111,
                1117,
                "PERSON"
            ],
            [
                1138,
                1144,
                "PERSON"
            ],
            [
                1191,
                1200,
                "PERSON"
            ],
            [
                1271,
                1277,
                "PERSON"
            ],
            [
                1319,
                1326,
                "PERSON"
            ],
            [
                1334,
                1341,
                "PERSON"
            ],
            [
                1384,
                1391,
                "PERSON"
            ],
            [
                1402,
                1411,
                "PERSON"
            ],
            [
                1416,
                1421,
                "PERSON"
            ],
            [
                1469,
                1477,
                "PERSON"
            ],
            [
                1579,
                1586,
                "PERSON"
            ],
            [
                1638,
                1645,
                "PERSON"
            ],
            [
                1776,
                1783,
                "PERSON"
            ],
            [
                1820,
                1828,
                "PERSON"
            ],
            [
                1841,
                1848,
                "PERSON"
            ],
            [
                1884,
                1892,
                "PERSON"
            ],
            [
                1945,
                1951,
                "PERSON"
            ],
            [
                1974,
                1980,
                "PERSON"
            ],
            [
                1985,
                1990,
                "PERSON"
            ],
            [
                2029,
                2038,
                "PERSON"
            ],
            [
                2042,
                2049,
                "PERSON"
            ],
            [
                2059,
                2067,
                "PERSON"
            ],
            [
                2129,
                2136,
                "PERSON"
            ],
            [
                2155,
                2164,
                "PERSON"
            ],
            [
                2191,
                2198,
                "PERSON"
            ],
            [
                2295,
                2300,
                "PERSON"
            ],
            [
                2374,
                2380,
                "PERSON"
            ],
            [
                2453,
                2462,
                "PERSON"
            ],
            [
                2478,
                2484,
                "PERSON"
            ],
            [
                2583,
                2592,
                "PERSON"
            ],
            [
                2676,
                2684,
                "PERSON"
            ],
            [
                2694,
                2699,
                "PERSON"
            ],
            [
                2776,
                2783,
                "PERSON"
            ],
            [
                2799,
                2806,
                "PERSON"
            ],
            [
                2820,
                2826,
                "PERSON"
            ],
            [
                2845,
                2851,
                "PERSON"
            ],
            [
                2864,
                2871,
                "PERSON"
            ],
            [
                2938,
                2944,
                "PERSON"
            ],
            [
                2954,
                2961,
                "PERSON"
            ],
            [
                2970,
                2979,
                "PERSON"
            ],
            [
                2994,
                3001,
                "PERSON"
            ],
            [
                3022,
                3027,
                "PERSON"
            ],
            [
                3063,
                3072,
                "PERSON"
            ],
            [
                3102,
                3108,
                "PERSON"
            ],
            [
                3138,
                3146,
                "PERSON"
            ],
            [
                3176,
                3186,
                "PERSON"
            ],
            [
                3226,
                3233,
                "PERSON"
            ],
            [
                3255,
                3264,
                "PERSON"
            ],
            [
                3282,
                3287,
                "PERSON"
            ],
            [
                3295,
                3300,
                "PERSON"
            ],
            [
                3335,
                3342,
                "PERSON"
            ],
            [
                3357,
                3363,
                "PERSON"
            ],
            [
                3377,
                3383,
                "PERSON"
            ],
            [
                3428,
                3433,
                "PERSON"
            ],
            [
                3445,
                3451,
                "PERSON"
            ],
            [
                3564,
                3570,
                "PERSON"
            ],
            [
                3630,
                3637,
                "PERSON"
            ],
            [
                3646,
                3653,
                "PERSON"
            ],
            [
                3803,
                3812,
                "PERSON"
            ],
            [
                3870,
                3877,
                "PERSON"
            ],
            [
                3881,
                3890,
                "PERSON"
            ],
            [
                3943,
                3949,
                "PERSON"
            ],
            [
                3957,
                3963,
                "PERSON"
            ],
            [
                3985,
                3991,
                "PERSON"
            ],
            [
                4087,
                4095,
                "PERSON"
            ],
            [
                4138,
                4145,
                "PERSON"
            ],
            [
                4166,
                4173,
                "PERSON"
            ],
            [
                4197,
                4203,
                "PERSON"
            ],
            [
                4244,
                4249,
                "PERSON"
            ],
            [
                4269,
                4275,
                "PERSON"
            ],
            [
                4303,
                4311,
                "PERSON"
            ],
            [
                4338,
                4345,
                "PERSON"
            ],
            [
                4369,
                4377,
                "PERSON"
            ],
            [
                4390,
                4399,
                "PERSON"
            ],
            [
                4417,
                4422,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult m { Domingo edica { Weldon l center east white, tyrone 1211 medical center dr mrn: 04 { Theo 7717361, dob { Devan : 7/27/1 { Lorelei 969, legal sex: m nashv { Estrella ille tn 37232 visit d { Braydon ate: 9/16/2024 09/16/2 { Kaye 024 - offic { Haleigh e visit in vanderbilt  { Maryanne orthopaedics (continued) letters (continued) beginning the next day, decrease to 1 d { Tianna rop to the  { Clement right e { Scot ye 4 t { Petra imes a day for 1 w { Lakeisha eek, then stop nifedipine er 30 mg take 2 tablets (60 mg 180 tablet 3 tablet,extended release total) by mouth (adalat cc) daily. pantoprazole 20 m { Benito g take 1 tablet (20 mg 30 tablet 11 tablet, delayed release total) by mouth (protonix) daily. { Garret  prednisolone { Deena  acetate 1 % eye after surgery, use 1 5  { Mitchel ml 1 drops,suspension (pred drop to the righ { Trista t eye { Hiram  forte) every 2 hours while awake until bedtime. beginning the next day, decrea { Moshe se to 1 drop to the right eye 4 t { Aleah imes a day for 1  { Osvaldo w { Benita eek, then 3 times a d { Chadwick ay for 1 week, then 2 times a day for 1 week, then daily for 1 week, then stop   { Adrianne facility-administered { Edythe  m { Elisha edications for this visit. social history occupational history not on file tobacco use smoking sta { Zelda tus: every d { Alena ay current p { Trevon acks/day: 0.50 average packs/day { Melany : 0.5 packs/day for 1 { Nataly 5.0 years ( { Demarcus 7.5 ttl pk-yrs) types: cigars,  { Kaley cigarettes smokeless tobacco: never tobacco comments: black and milds, smoking 5 per day substance and sexual activity alcohol use: n { Bertie ot cur { Jaden rently drug use: not currently types: cocaine, marijuana sexual activity: defer printed on 10/3/24 7 { Journey :12 am page 37,vumc ad { Aurelia ult medical center east whi { Thalia te, t { Kris yrone 1211 medical center dr mrn: 047717361, dob: 7/27/1969, legal sex { Bridgett : m nashville tn 37232 visi { Josefina t date: 9/16/2024 09 { Tameka /16 { Eddy /2024 - o { Rocio ffice visit in vanderbilt orthopaedics  { Annabella (c { David ontinued) letters (continued) family history problem relation age { Adriel  of onset diabetes neg hx lung cancer neg hx  for i { Marlee nput(s): \"hgba1c\" in the last 1080 hours. objective pe: general: alert and orie { Madge n { Jazlyn ted  { Roxanna x 3, with  d { Elliana istress. respirato { Georgette ry: unlabored breathing cardiovascular { Desirae : edema  { Catharine absent, bilateral lower extr { Messiah emities. do { Maricela rsalis pedis: present poster { Addison ior tibi { Marilynn al: present capillary { Rebeca  f { Dallas illing tim { Aubrie e is less than 3 se { Laverne conds bilateral. var { Kailyn icosities are not  { Carey ob { Augustine served. skin: skin tempe { Cari rature, tone, and turgor are { Louella  within normal limits for th { Scarlet e patient's age and health status. toe na { Kolton ils great and fifth bilateral mycotic { Mitzi  nails discolored, elongat { Amiyah ed, crumbly, with  { Gena subungual debr { Maud is and dystrophic changes greater than { Brittani  1 mm. pedal hair growth absent. skin color skin-colored. hyperke { Carmine rato { Maryellen sis noted  { Amalia to 1st a { Seymour nd 5th { Gussie  metatarsal head plantarly bilateral. grade o dfu right. neur { Marylou ological: grossly absent to sensati { Iker o { Jesse n. noted diminished swmf to 10  { Nash tested lo { Kristal cations on the foot. musculoskeletal: gait and standing examination rev { Maryjane eals  deform { Sharlene ity postion. pedal joints were taken through a range of motion and were noted to be adeq { Lakisha uate without crepitus . there is  upon palpation of the feet. bi { Loren lateral ankle dorsiflexion limited, with knee exntended.  blockage noted. deformities: hammertoes. radiology: none iwgdf foot risk categories { Davon : 1-pn, { Archer  ,  assessment diagnosis plan 1. onychomycosis { Merrill  2. benign ne { Dara opl { Dylan asm of skin of lower extremity, urea 20 % topical c { Liberty ream (carmol) { Charlee  unspecified laterality print { Jaelyn ed on 10/3/24 7:12 am page 38",
    {
        "entities": [
            [
                15,
                23,
                "PERSON"
            ],
            [
                31,
                38,
                "PERSON"
            ],
            [
                99,
                104,
                "PERSON"
            ],
            [
                119,
                125,
                "PERSON"
            ],
            [
                136,
                144,
                "PERSON"
            ],
            [
                170,
                179,
                "PERSON"
            ],
            [
                203,
                211,
                "PERSON"
            ],
            [
                236,
                241,
                "PERSON"
            ],
            [
                255,
                263,
                "PERSON"
            ],
            [
                288,
                297,
                "PERSON"
            ],
            [
                384,
                391,
                "PERSON"
            ],
            [
                405,
                413,
                "PERSON"
            ],
            [
                423,
                428,
                "PERSON"
            ],
            [
                437,
                443,
                "PERSON"
            ],
            [
                464,
                473,
                "PERSON"
            ],
            [
                622,
                629,
                "PERSON"
            ],
            [
                725,
                732,
                "PERSON"
            ],
            [
                748,
                754,
                "PERSON"
            ],
            [
                797,
                805,
                "PERSON"
            ],
            [
                852,
                859,
                "PERSON"
            ],
            [
                867,
                873,
                "PERSON"
            ],
            [
                955,
                961,
                "PERSON"
            ],
            [
                997,
                1003,
                "PERSON"
            ],
            [
                1023,
                1031,
                "PERSON"
            ],
            [
                1035,
                1042,
                "PERSON"
            ],
            [
                1066,
                1075,
                "PERSON"
            ],
            [
                1158,
                1167,
                "PERSON"
            ],
            [
                1191,
                1198,
                "PERSON"
            ],
            [
                1203,
                1210,
                "PERSON"
            ],
            [
                1311,
                1317,
                "PERSON"
            ],
            [
                1332,
                1338,
                "PERSON"
            ],
            [
                1353,
                1360,
                "PERSON"
            ],
            [
                1395,
                1402,
                "PERSON"
            ],
            [
                1426,
                1433,
                "PERSON"
            ],
            [
                1447,
                1456,
                "PERSON"
            ],
            [
                1490,
                1496,
                "PERSON"
            ],
            [
                1632,
                1639,
                "PERSON"
            ],
            [
                1648,
                1654,
                "PERSON"
            ],
            [
                1757,
                1765,
                "PERSON"
            ],
            [
                1790,
                1798,
                "PERSON"
            ],
            [
                1828,
                1835,
                "PERSON"
            ],
            [
                1843,
                1848,
                "PERSON"
            ],
            [
                1921,
                1930,
                "PERSON"
            ],
            [
                1960,
                1969,
                "PERSON"
            ],
            [
                1992,
                1999,
                "PERSON"
            ],
            [
                2005,
                2010,
                "PERSON"
            ],
            [
                2022,
                2028,
                "PERSON"
            ],
            [
                2070,
                2080,
                "PERSON"
            ],
            [
                2085,
                2091,
                "PERSON"
            ],
            [
                2159,
                2166,
                "PERSON"
            ],
            [
                2220,
                2227,
                "PERSON"
            ],
            [
                2309,
                2315,
                "PERSON"
            ],
            [
                2319,
                2326,
                "PERSON"
            ],
            [
                2333,
                2341,
                "PERSON"
            ],
            [
                2356,
                2364,
                "PERSON"
            ],
            [
                2385,
                2395,
                "PERSON"
            ],
            [
                2436,
                2444,
                "PERSON"
            ],
            [
                2455,
                2465,
                "PERSON"
            ],
            [
                2496,
                2504,
                "PERSON"
            ],
            [
                2518,
                2527,
                "PERSON"
            ],
            [
                2558,
                2566,
                "PERSON"
            ],
            [
                2577,
                2586,
                "PERSON"
            ],
            [
                2610,
                2617,
                "PERSON"
            ],
            [
                2622,
                2629,
                "PERSON"
            ],
            [
                2642,
                2649,
                "PERSON"
            ],
            [
                2671,
                2679,
                "PERSON"
            ],
            [
                2702,
                2709,
                "PERSON"
            ],
            [
                2730,
                2736,
                "PERSON"
            ],
            [
                2741,
                2751,
                "PERSON"
            ],
            [
                2778,
                2783,
                "PERSON"
            ],
            [
                2814,
                2822,
                "PERSON"
            ],
            [
                2853,
                2861,
                "PERSON"
            ],
            [
                2905,
                2912,
                "PERSON"
            ],
            [
                2952,
                2958,
                "PERSON"
            ],
            [
                2987,
                2994,
                "PERSON"
            ],
            [
                3015,
                3020,
                "PERSON"
            ],
            [
                3037,
                3042,
                "PERSON"
            ],
            [
                3083,
                3092,
                "PERSON"
            ],
            [
                3160,
                3168,
                "PERSON"
            ],
            [
                3175,
                3185,
                "PERSON"
            ],
            [
                3198,
                3205,
                "PERSON"
            ],
            [
                3216,
                3224,
                "PERSON"
            ],
            [
                3233,
                3240,
                "PERSON"
            ],
            [
                3304,
                3312,
                "PERSON"
            ],
            [
                3350,
                3355,
                "PERSON"
            ],
            [
                3359,
                3365,
                "PERSON"
            ],
            [
                3399,
                3404,
                "PERSON"
            ],
            [
                3416,
                3424,
                "PERSON"
            ],
            [
                3498,
                3507,
                "PERSON"
            ],
            [
                3522,
                3531,
                "PERSON"
            ],
            [
                3622,
                3630,
                "PERSON"
            ],
            [
                3697,
                3703,
                "PERSON"
            ],
            [
                3847,
                3853,
                "PERSON"
            ],
            [
                3863,
                3870,
                "PERSON"
            ],
            [
                3919,
                3927,
                "PERSON"
            ],
            [
                3943,
                3948,
                "PERSON"
            ],
            [
                3954,
                3960,
                "PERSON"
            ],
            [
                4014,
                4022,
                "PERSON"
            ],
            [
                4038,
                4046,
                "PERSON"
            ],
            [
                4078,
                4085,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medica { Donny l center dr { Yasmine . mrn: 047717361, dob: 7/27/1969, legal sex:  { Keely m nashville tn 37232-0004 adm: 6/2/2023, d/c: 6/3/2023 06/02/ { Lawson 2023 - ed to hosp-admission (discharged) in van { Kacie derbi { Maeve lt university adult { Yadira  hospital (continued) ed care timeline (continued) 19:20 skin skin haugen, jayda, color/ { Kori conditi { Basil on skin assessme { Luisa nt wnl rn 19:20 circulation cardiac haugen, jayda, cardiac  { Adela assessment : wnl { Madyson  rn vascular { America /perfusion vascular assessment: wnl 19:20 musculoskeletal activity/musculoskeleta { Anaya l { Ibrahim  hauge { Linwood n, jayda, a { Theron ctivity/musculoskeletal assessment: oel (abnormal gait  { Alecia from baseline) rn generalized weakness: noted rue strength: 5/5 movement against gravity { Lilia  with full resistance lue strength: 5/5 movement { Jimena  against gravity with full resistance rle strength: 4/5 movement against gravity with some r { Antwan esistance lle strength: 4/5 movement against gravity with some resistance rue movemen { Aida t:  from baseline lue movement:  from baseline rle movement:  { Wilda  from baseline lle movement:  from baseline 19:20 psychosocial psychosocial haugen, jayda, psychosocial assessment: wnl rn family presence: patie { Karlee nt al { Mckayla one emotional care/interventions: active listening; reassured; encouraged expressi { Amina on; offer info 19:20 safety a { Zara ssessment safety precautions hauge { Enzo n, jayda, safety preca { Myla ution: fall precautions rn fall/injury precaution: environmental assessment completed jhfrat fall risk if patient does not meet ei { Justina ther of the above two criteria, please use tool: jhfrat age: <60 years fall history { Micah :  fall { Charles  within the last 6 months elimination, bowel and urine:   { Chaya problems medications (see row information): on 2 or more high fall risk drugs patient care equipment (see row information): two presen { Shaina t mobility -choose all that apply: requires assistance or supervision for mobility, transfer, or ambulation; unsteady  { Caryn gait cognition - choose all that apply:  impairment total fall risk score: 11 fall risk: moderate fall risk 19:20 custom formula confusion assessment method-icu (cam-icu)  { Arleen haugen, jayda, data feature 3: altered level of consciou { Gino sness: negative rn fall risk scale fall risk calculated score: 11 (jhfrat) 19:20:08 orders placed medications - lactated ringer's  { Coleen bolus 1,000 ml; guaifenesin  { Jarod (rob { Dalia itussin) pauw, emily 100 mg/5 ml oral liquid 10 ml kathryn, md lab ck 19:2 { Francisca 0:10 lab ordered ck pauw, emily kathryn, md 19:21 admission request russell, rebe { Hunter cca in process jean, rn 19:21 admisison request admission request status ( { Brynlee documentation in  { Cyril this section for mac & russell, rebecca cac only) jean, rn status of request:: in process 19:21:56 rem { Gale ove technician kelton e hutsell removed a { Christen s care partner hutse { Carlee ll, kelton e printed on 10/3/24 7:13 am p { Stevie age 2167,vumc adult ho { Zella spital white, tyrone 1211 medical center dr. mrn: 047717 { Nikita 361,  { Annalise dob: 7/27/1969, legal sex: m nash { Birdie ville tn 37232-0004 adm: 6/2/2023 { Lizette , d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt university adult hospital (continued) ed care t { Lulu ime { Morton line (continued) 19 { Roselyn :22:55 orders new - ck; lac { Lawanda tated ringer's bolus 1,000 ml; guaifenesin (ro { Hank bitussin) haugen, jayda, acknowle { Janel dged 100 mg/5 ml oral liq { Ronan uid 10 ml rn 19:24:11 assign technician kelton e hutsell assigned as care partner hutsell, kelton { Nasir  e 19:35 admiss { Celine ion request russell, rebecca complete jean, rn 19:35 admisison request admission reque { Diann st statu { Knox s (documentation { Kailee  in this section for mac & russell, rebecca c { Annemarie ac only) jean, rn status of request:: done (admission acc { Kasey epted) 19:48 ed obs { Wilhelmina ervation pauw, emily p { Judson atient kathryn, md 19:48:43 admit disposition pauw, emi { Giovanna ly selected kathryn, md 19:48:43 team member mary kate j { Brodie ordan, md assigned as admitting pauw, emily assigned kathryn, md 19:48:43 admit disposition ed disposition set to admit pauw, emily se { Mervin lected kathryn, md 19: { Moriah 48:43 disposition pauw, emily selected kathryn, md 19:48:43 orders placed { Tamia  admission - i { Georgie nitiate observation status pauw, emily kathryn, md 19:48:49 team member mary kate jordan, md removed as admitting pauw, emily removed kathryn,  { Cori md 19:48:49 team member mary kate jordan, md assigned as admitting pauw, emily assigned kathryn, md 19:48:50 bed requested requested: general internal medicine pauw, emily kathryn, md 19:48:51 bed  { Ashly request ready re { Anissa ady to plan: general internal medicine pauw { Shayne , emily to plan kathryn, md 19:48:52 ed ip bed initiate observat { Sloane ion status - [372923735] pauw, emily requested kathryn, md 19:58:38 remove resident emily kathryn pauw, md removed { Vince  as resident pauw, emily kathryn, md 20:00:09 note shared ed prov note f { Matias iled by emily kathryn pauw, md pauw, emily kathryn, md 20:00:09 ed note shared by ed prov note filed by emily kathryn pauw, md pauw, emily scribe kathr { Clair yn, m { Damaris d 20:44:50 orders pla { Breanne ced nursing - vita { Erich l signs - high acuity; notify provider (standard adult howard, w { Christin illiam parameters); notify provider (specify parameters); height and weight on timothy, pa admission; poc lab:  { Shelton glucose; source: capillary; poc lab: glucose; { Maranda  source: capillary medications - albuterol hfa 90 mcg/actuation inhaler 2 puff; atorvastatin (lipitor) tablet 80 mg;  { Luciano cetirizine (zyrtec) tablet 10 mg; gabapentin (neurontin) capsule 100 mg; pantoprazole (protonix) ec t { Marquita abl { Ciera et 20 mg; normosol or plasmalyte-a carrier 500 ml; acetaminophen (tylenol) tablet 650 mg; juice for hypoglycemia management 4 oz; glucose chewable tablet 16 g; dextrose (d50w) 50 % injection 25  { Dayana ml; glucagon (human recombinant) injection 1 mg lab - bmp; cbc w/ dif { Skyla ferential code sta { Reggie tus - full co { Jolie de core measures - padua risk score low (less than 4) 20:44:51 orders placed medications - insulin lispro 1-6 units admelog - biosimilar for humal { Finley og howard, william injection 0.01- { Kaia 0.06 ml timothy, pa 20:44:54 lab ordered cbc w/ differential, bmp howard, william timoth { Dudley y, { Ashanti  pa 20:44:58 orders completed padua risk score low (less than 4) howard, william timothy, pa printed on 10/3/24 7:13 am page 2168",
    {
        "entities": [
            [
                48,
                54,
                "PERSON"
            ],
            [
                68,
                76,
                "PERSON"
            ],
            [
                124,
                130,
                "PERSON"
            ],
            [
                194,
                201,
                "PERSON"
            ],
            [
                251,
                257,
                "PERSON"
            ],
            [
                265,
                271,
                "PERSON"
            ],
            [
                293,
                300,
                "PERSON"
            ],
            [
                391,
                396,
                "PERSON"
            ],
            [
                406,
                412,
                "PERSON"
            ],
            [
                431,
                437,
                "PERSON"
            ],
            [
                499,
                505,
                "PERSON"
            ],
            [
                524,
                532,
                "PERSON"
            ],
            [
                547,
                555,
                "PERSON"
            ],
            [
                639,
                645,
                "PERSON"
            ],
            [
                649,
                657,
                "PERSON"
            ],
            [
                666,
                674,
                "PERSON"
            ],
            [
                688,
                695,
                "PERSON"
            ],
            [
                753,
                760,
                "PERSON"
            ],
            [
                851,
                857,
                "PERSON"
            ],
            [
                908,
                915,
                "PERSON"
            ],
            [
                1010,
                1017,
                "PERSON"
            ],
            [
                1105,
                1110,
                "PERSON"
            ],
            [
                1174,
                1180,
                "PERSON"
            ],
            [
                1328,
                1335,
                "PERSON"
            ],
            [
                1343,
                1351,
                "PERSON"
            ],
            [
                1436,
                1442,
                "PERSON"
            ],
            [
                1474,
                1479,
                "PERSON"
            ],
            [
                1516,
                1521,
                "PERSON"
            ],
            [
                1546,
                1551,
                "PERSON"
            ],
            [
                1684,
                1692,
                "PERSON"
            ],
            [
                1778,
                1784,
                "PERSON"
            ],
            [
                1794,
                1802,
                "PERSON"
            ],
            [
                1862,
                1868,
                "PERSON"
            ],
            [
                2005,
                2012,
                "PERSON"
            ],
            [
                2133,
                2139,
                "PERSON"
            ],
            [
                2313,
                2320,
                "PERSON"
            ],
            [
                2379,
                2384,
                "PERSON"
            ],
            [
                2517,
                2524,
                "PERSON"
            ],
            [
                2555,
                2561,
                "PERSON"
            ],
            [
                2568,
                2574,
                "PERSON"
            ],
            [
                2651,
                2661,
                "PERSON"
            ],
            [
                2745,
                2752,
                "PERSON"
            ],
            [
                2829,
                2837,
                "PERSON"
            ],
            [
                2857,
                2863,
                "PERSON"
            ],
            [
                2968,
                2973,
                "PERSON"
            ],
            [
                3017,
                3026,
                "PERSON"
            ],
            [
                3049,
                3056,
                "PERSON"
            ],
            [
                3100,
                3107,
                "PERSON"
            ],
            [
                3132,
                3138,
                "PERSON"
            ],
            [
                3197,
                3204,
                "PERSON"
            ],
            [
                3212,
                3221,
                "PERSON"
            ],
            [
                3257,
                3264,
                "PERSON"
            ],
            [
                3300,
                3308,
                "PERSON"
            ],
            [
                3435,
                3440,
                "PERSON"
            ],
            [
                3446,
                3453,
                "PERSON"
            ],
            [
                3475,
                3483,
                "PERSON"
            ],
            [
                3513,
                3521,
                "PERSON"
            ],
            [
                3570,
                3575,
                "PERSON"
            ],
            [
                3611,
                3617,
                "PERSON"
            ],
            [
                3645,
                3651,
                "PERSON"
            ],
            [
                3751,
                3757,
                "PERSON"
            ],
            [
                3775,
                3782,
                "PERSON"
            ],
            [
                3871,
                3877,
                "PERSON"
            ],
            [
                3888,
                3893,
                "PERSON"
            ],
            [
                3912,
                3919,
                "PERSON"
            ],
            [
                3967,
                3977,
                "PERSON"
            ],
            [
                4037,
                4043,
                "PERSON"
            ],
            [
                4065,
                4076,
                "PERSON"
            ],
            [
                4101,
                4108,
                "PERSON"
            ],
            [
                4166,
                4175,
                "PERSON"
            ],
            [
                4234,
                4241,
                "PERSON"
            ],
            [
                4378,
                4385,
                "PERSON"
            ],
            [
                4410,
                4417,
                "PERSON"
            ],
            [
                4493,
                4499,
                "PERSON"
            ],
            [
                4516,
                4524,
                "PERSON"
            ],
            [
                4670,
                4675,
                "PERSON"
            ],
            [
                4875,
                4881,
                "PERSON"
            ],
            [
                4900,
                4907,
                "PERSON"
            ],
            [
                4953,
                4960,
                "PERSON"
            ],
            [
                5027,
                5034,
                "PERSON"
            ],
            [
                5151,
                5157,
                "PERSON"
            ],
            [
                5232,
                5239,
                "PERSON"
            ],
            [
                5393,
                5399,
                "PERSON"
            ],
            [
                5407,
                5415,
                "PERSON"
            ],
            [
                5439,
                5447,
                "PERSON"
            ],
            [
                5468,
                5474,
                "PERSON"
            ],
            [
                5541,
                5550,
                "PERSON"
            ],
            [
                5664,
                5672,
                "PERSON"
            ],
            [
                5720,
                5728,
                "PERSON"
            ],
            [
                5848,
                5856,
                "PERSON"
            ],
            [
                5960,
                5969,
                "PERSON"
            ],
            [
                5975,
                5981,
                "PERSON"
            ],
            [
                6178,
                6185,
                "PERSON"
            ],
            [
                6257,
                6263,
                "PERSON"
            ],
            [
                6284,
                6291,
                "PERSON"
            ],
            [
                6307,
                6313,
                "PERSON"
            ],
            [
                6462,
                6469,
                "PERSON"
            ],
            [
                6506,
                6511,
                "PERSON"
            ],
            [
                6602,
                6609,
                "PERSON"
            ],
            [
                6614,
                6622,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: { Deangelo  m nashville tn 37232-0004 adm: 8/23/2024, d/c: 8/2 { Kennedi 4/2024 08/23/2024 - ed in vanderbilt emergency department (continued) after visit summary (continued) ed after visit summary (below) after visit summary va { Leta nderbilt health tyro { Miley ne white mrn: 0477 { Abbigail 17361 8/23/2024 vanderbilt emergency d { Iona epartment 615-322-5000 instructions today' { Roxana s visit you were seen at { Annamarie  vanderbilt university medical center for eye irritation you were seen by chr { Layne istine leanne after cataract surgery. prochnow reason for visit additional important informat { Nico ion: eye pain follow-up: please call your primary care doctor to schedule a follow up { Maudie  diagnoses visit as soon as possible. ideally, i would like you to follow up within 2 pain of left eye weeks. please ma { Veda ke the first available appointment. if yo { Davion u do not have history of left cataract surgery a pri { Twila mary doctor, please call the number on your insura { Kaylyn nce card. l { Dahlia ab tests completed pl { Angelita ease take all of the eye drops as prescribed. this i { Jadyn s very auto diff imp { Kitty ortant for your post-operative recovery. bmp reasons to return to hospital: cb { Korey c w/ d { Foster ifferential general discharge: please call your primary d { Letitia octor or return to the egfrcr nearest emergency department immediately if you fail to improve desp { Jamari ite the recommendations discussed today, your symptoms are done toda { Katina y worsening or c { Kamila hanging, { Chana  or a { Earle ny other acute concern.  { Adelina inpatient consult to ophthalmology what left is eye? the patien { Shea t's unable to visual participate acuity in the in please kee { Vern p  { Cherry this paperwork for your reference and for fo { Lesa llow-up visits exam; please specify why patient is with your primary physician and specialists. follow-up care is always unable to participate in exa { Aylin m: in required after a visit to t { Trace he emergency department. your  { Valencia diagnosis progress today is provisional based on the information available at the time of visual  { Danika acuity screening  { Alfreda your visit. your symptoms an { Cleo d diagnosis may change as  { Cornell more information becomes available to your physicia { Dessie ns. thank you for medications given visiting vanderbil { Tayler t { Jamel  emergency department, it was a pleasure to care for oxycodone (roxi { Marlena codone) last you today. f { Claudine or any perceived medical emergency, call 911  { Izaiah or go  { Joe to  { Breana the given at 9:27 pm emergency department. blood temperature pressure your medications have changed (ora { Julien l) 155/79 { Iliana  97.5 °f change how you take: pulse ketorolac (acular) respiratio { Hana n 78 18 moxifloxacin (v { Rigoberto igamo { Brycen x) prednisolone acetat { Milan e (pr { Racheal ed f { Leora orte) oxygen saturation ? ask how to take: 100% acetaminophen 325 mg tablet (tylenol) lancets 33 gauge misc (trueplus  { Cordelia lancets) lokelma 10 gram powder in packet (sodium zirconium cyclosilicate) review { Johan  your updated medication list below. tyrone white (mrn: 047717361 { Farrah ) (7/27/19 { Triston 69) printed at 8/24/2024 12:09 am page 1 of 12 epic printed on 10/3/24 7:12 am page 187,vumc adult hospital white, tyrone 1211 medical { Florine  center dr. mrn: 047717361, dob: 7/27/1969, legal sex:  { Alva m nashville { Booker  tn 37232-0004 adm: 8 { Rhys /23/2024, d/c: 8/24/2024 08/23/ { Millicent 2024 - ed in vanderbilt emergency department (continued) after  { Cristopher visit summ { Maliyah ary { Kacey  (continued) instructions (continued) read  { Alysha the attached information cataract surgery, { Isis  discha { Tevin rge instructions (english) pick  { Ace up these medications at vanderb { Leia ilt university 100 oaks - nashville, tn - 719 thompson ln ketorolac . moxifloxacin prednisolone acetate address: 719 thompson ln suite 241 { Jaron 30, nashville tn 37204 phone: 615-322 { Dusty -2688 what's next aug pod#1 with daniel alejandro valenzuela vanderbilt eye institute 26 monday august 26 12:45 pm (arrive by 12:30 pm) 2311 pierc { Talon e ave 2024 save time. skip the line. vande { Demetria rbilt eye institute nashville tn 37232 you can now use self { Patience   { Bo check-in for your visit. 615-936-20 { Dirk 20 use  { Lauri my hea { Janiyah lth at vanderbilt to check-in for your visit right from your phone. let us know you're here open the mhav app on yo { Winona ur phone when you arrive and click \"i'm here\" once you arrive at your appointment.  { Rayna sep a { Mikel fter surgery with daniel alejandro valenzuela vanderbilt eye institute 4 wednesda { Ashlie y september 4 11:00 am (arrive by 10:45 am) 2311 pierce ave 2024 save time { Joseph . skip the line. vanderbilt eye institute nashville tn 37232 you can now use self check-in for your visit { Kaylynn . 615-936-2020 use my health at vanderbilt to check-in f { Emmitt or your v { Ariella isit right from { Daren  your phone.  { Portia let us { Laney  know you're h { Devonte ere open the mhav app on your phone wh { Karly en you arrive and click \"i'm here\" once you a { Kenton rrive at y { Sade our appoi { Blake ntment. tyrone { Lacie  wh { Karli ite (mrn: 047717361) (7/27/1969) printed at 8/24/2024 12:09 am page 2 of 12 epic printed on 10/3 { Fidel /24 7:12 am page 188",
    {
        "entities": [
            [
                103,
                112,
                "PERSON"
            ],
            [
                166,
                174,
                "PERSON"
            ],
            [
                332,
                337,
                "PERSON"
            ],
            [
                360,
                366,
                "PERSON"
            ],
            [
                387,
                396,
                "PERSON"
            ],
            [
                437,
                442,
                "PERSON"
            ],
            [
                487,
                494,
                "PERSON"
            ],
            [
                521,
                531,
                "PERSON"
            ],
            [
                611,
                617,
                "PERSON"
            ],
            [
                713,
                718,
                "PERSON"
            ],
            [
                806,
                813,
                "PERSON"
            ],
            [
                935,
                940,
                "PERSON"
            ],
            [
                984,
                991,
                "PERSON"
            ],
            [
                1046,
                1052,
                "PERSON"
            ],
            [
                1105,
                1112,
                "PERSON"
            ],
            [
                1126,
                1133,
                "PERSON"
            ],
            [
                1157,
                1166,
                "PERSON"
            ],
            [
                1221,
                1227,
                "PERSON"
            ],
            [
                1250,
                1256,
                "PERSON"
            ],
            [
                1337,
                1343,
                "PERSON"
            ],
            [
                1352,
                1359,
                "PERSON"
            ],
            [
                1419,
                1427,
                "PERSON"
            ],
            [
                1528,
                1535,
                "PERSON"
            ],
            [
                1606,
                1613,
                "PERSON"
            ],
            [
                1632,
                1639,
                "PERSON"
            ],
            [
                1650,
                1656,
                "PERSON"
            ],
            [
                1664,
                1670,
                "PERSON"
            ],
            [
                1697,
                1705,
                "PERSON"
            ],
            [
                1771,
                1776,
                "PERSON"
            ],
            [
                1839,
                1844,
                "PERSON"
            ],
            [
                1849,
                1856,
                "PERSON"
            ],
            [
                1903,
                1908,
                "PERSON"
            ],
            [
                2060,
                2066,
                "PERSON"
            ],
            [
                2102,
                2108,
                "PERSON"
            ],
            [
                2141,
                2150,
                "PERSON"
            ],
            [
                2250,
                2257,
                "PERSON"
            ],
            [
                2277,
                2285,
                "PERSON"
            ],
            [
                2316,
                2321,
                "PERSON"
            ],
            [
                2350,
                2358,
                "PERSON"
            ],
            [
                2412,
                2419,
                "PERSON"
            ],
            [
                2476,
                2483,
                "PERSON"
            ],
            [
                2487,
                2493,
                "PERSON"
            ],
            [
                2564,
                2572,
                "PERSON"
            ],
            [
                2600,
                2609,
                "PERSON"
            ],
            [
                2657,
                2664,
                "PERSON"
            ],
            [
                2673,
                2677,
                "PERSON"
            ],
            [
                2683,
                2690,
                "PERSON"
            ],
            [
                2797,
                2804,
                "PERSON"
            ],
            [
                2816,
                2823,
                "PERSON"
            ],
            [
                2891,
                2896,
                "PERSON"
            ],
            [
                2922,
                2932,
                "PERSON"
            ],
            [
                2940,
                2947,
                "PERSON"
            ],
            [
                2972,
                2978,
                "PERSON"
            ],
            [
                2986,
                2994,
                "PERSON"
            ],
            [
                3001,
                3007,
                "PERSON"
            ],
            [
                3128,
                3137,
                "PERSON"
            ],
            [
                3221,
                3227,
                "PERSON"
            ],
            [
                3295,
                3302,
                "PERSON"
            ],
            [
                3315,
                3323,
                "PERSON"
            ],
            [
                3460,
                3468,
                "PERSON"
            ],
            [
                3526,
                3531,
                "PERSON"
            ],
            [
                3545,
                3552,
                "PERSON"
            ],
            [
                3576,
                3581,
                "PERSON"
            ],
            [
                3615,
                3625,
                "PERSON"
            ],
            [
                3691,
                3702,
                "PERSON"
            ],
            [
                3715,
                3723,
                "PERSON"
            ],
            [
                3729,
                3735,
                "PERSON"
            ],
            [
                3781,
                3788,
                "PERSON"
            ],
            [
                3833,
                3838,
                "PERSON"
            ],
            [
                3848,
                3854,
                "PERSON"
            ],
            [
                3889,
                3893,
                "PERSON"
            ],
            [
                3927,
                3932,
                "PERSON"
            ],
            [
                4073,
                4079,
                "PERSON"
            ],
            [
                4119,
                4125,
                "PERSON"
            ],
            [
                4274,
                4280,
                "PERSON"
            ],
            [
                4325,
                4334,
                "PERSON"
            ],
            [
                4396,
                4405,
                "PERSON"
            ],
            [
                4409,
                4412,
                "PERSON"
            ],
            [
                4450,
                4455,
                "PERSON"
            ],
            [
                4465,
                4471,
                "PERSON"
            ],
            [
                4480,
                4488,
                "PERSON"
            ],
            [
                4606,
                4613,
                "PERSON"
            ],
            [
                4699,
                4705,
                "PERSON"
            ],
            [
                4713,
                4719,
                "PERSON"
            ],
            [
                4803,
                4810,
                "PERSON"
            ],
            [
                4887,
                4894,
                "PERSON"
            ],
            [
                5002,
                5010,
                "PERSON"
            ],
            [
                5069,
                5076,
                "PERSON"
            ],
            [
                5088,
                5096,
                "PERSON"
            ],
            [
                5114,
                5120,
                "PERSON"
            ],
            [
                5136,
                5143,
                "PERSON"
            ],
            [
                5152,
                5158,
                "PERSON"
            ],
            [
                5175,
                5183,
                "PERSON"
            ],
            [
                5224,
                5230,
                "PERSON"
            ],
            [
                5278,
                5285,
                "PERSON"
            ],
            [
                5298,
                5303,
                "PERSON"
            ],
            [
                5315,
                5321,
                "PERSON"
            ],
            [
                5338,
                5344,
                "PERSON"
            ],
            [
                5350,
                5356,
                "PERSON"
            ],
            [
                5455,
                5461,
                "PERSON"
            ]
        ]
    }
),(
    " { Rosalee vumc adult one hundred oaks white, tyrone 719 thompson lane, na { Shantel shville mrn: 0477173 { Sage 61, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date: 9/7/2023 09/07/2023 - lab in vander { Enoch bilt laboratory services north o { Reina ne hundred oaks (continued) laboratory reports  { Pansy (cont { Alia inued) this o { Kolby rder may be acted on in another encounter. orde { Vida ring user: gre { Antony en { Kenna span, debra l, aprn 09/05/23 1714 ordering { Michel  provider: greenspan, debra l, aprn authorized by: greenspan, debra l, aprn ord { Domenic ering mode: standard frequency: routine 09/05/23 - class: lab collect quantity: 1 lab status: final result instance released by: anderson, natashe l 9/7/2023 3:25 pm diagnoses chronic kidney disease (ckd), stage iv ( { Claud severe) (cms/hcc) [n18. { Mayme 4] specimen information id type source collected by 23-250-011348 blood - carter, tewauna 09/07/23 1545 bmp (abnormal) resulted: 09/07/23 1732, resul { Lucian t status: f { Gayla inal result ordering  { Monika provider: greenspan, debra l, aprn 09/07/23 1525 order status: completed filed by: interface, lab results in 09/07/23 1732 collected by: carter, tewauna 09/07/23 1545 resulting l { Buford ab: vumc cerner lab acknowledged by: greenspan, debra l, aprn on  { Nathanial 09/08/ { Rohan 23 0807 components { Freeman  component value reference range flag lab sodium  { Lona leve { Garth l 142 136 145 mmol { Macey /l - cerner p { Mandi otassium level  { Denice 5.2 3.3 - 4.8 mmol/l h a cerner comment: plasma reference range { Cayla s are shown, please note that serum reference ranges are higher than plasma. chloride level 111 98 107 mmo { Raina l/l h cerner carbon dioxide 26 22 29 mmol/l - cer { Coral ner  { Deidra glucose level 99 70 99 mg/dl - cerner blood urea nitrogen 31 8 - 26 mg/dl h^ cerner creatinine level 3.47 0.72 - 1.25 mg/dl h'  { Ivette cerner calcium level total 8.8 8.4 - 10.5 mg/dl - cerner anion gap 5  { Alycia - cerner comment: this test was performed at: rapid response 100 oaks laboratory, clia# 44d1093367,ada c seegmiller, md, phd, medical director, 719 thompson lane, suite 21100, nashville, tn, 37204, usa testing performed by lab - abbreviation name director ad { Marisela dress valid date range 123 cerner vumc cerner lab { Mariela  adam seegmiller; 4605 tvc vumc 11/22/21 1014 - present jen { Carmelo nifer b. { Brook  1301 medical center { Scottie  gordetsky drive nashville tn 3723 { Octavio 2- 5310 indic { Lidia ations chro { Catrina nic kidney disease (ckd), stage i { Sandy v (severe) (cms/hcc) [n18.4 (icd- { Vada 10-cm)] result notes debra l greenspan, aprn 9/8/2023 8:07 am cdt labs reviewed though abnormal relativel { Ayana y stable given histo { Robby ry of stage iv ckd. efgr 20, stable over the past 2 months, potassium elevated at 5.2 ( { Davin sending message to limit intake of potassium rich foods) all reviewers list { Karlie  greenspan, debra l, aprn on 9/8/2023 08:07 printed on 10/3/24 7:13 am page 1713,vumc adult one hu { Gregorio ndred oaks white, tyrone 719 thompson la { Freida ne, nashville mrn: { Kallie  047717361, dob: 7/27/19 { Rosario 69, legal sex: m nashville tn 37204 visit date: 9/7/2023 09/07/2023 - lab in van { Kennith derbilt laboratory services north one { Felecia  hundred oaks (continue { Christiana d) laboratory reports (continued) tibc (final result) electronically { Manuela  signed by: de witte, anton jordan, md on 09/07/23  { Retha 1506 status: { Renata  co { Mekhi mpleted this order may be act { Arlo ed on in an { Cordell other encounter. ordering user: de witte, anton jordan, md 09/07/23 1506 ordering provid { Denzel er: de witte, ant { Stevie on jordan, md authorized by: sullivan, kathleen pollard, md ordering mode: standard frequency: routine 09/07/23 - class: lab collect quantity:  { Jaimie 1 lab status: final result instance released by: anderson, natashe l 9/7/2023 3:25 pm diagnoses normocytic anemia [d64.9] specimen information id type source collected by 23-250-011348 blood carter,  { Lucretia tewauna 09/07/23  { Tera 1545 tibc (abnormal) resu { Beryl lted: 09/07/23 2150, result stat { Mira us: final result ordering provider: { Cherish  de witte, anton jordan, md 09/07/23 1525 order status: completed filed by: interface { Deven , lab results in 09 { Gianni /07/23 2150 collected by: { Kadence  carter, tewaun { Herschel a 09/07/23 1545 resulting lab: vumc cerner lab ack { Ryland nowled { Cortez ged by de witte, anton jordan, md on 09/11/23 1904 adams, danielle on 09 { Danette /12/23 1158 sullivan, kathleen pollard, md  { Rowena on 09/12/23 1828 components component value r { Everette eference range flag lab iron level 69 50 - 175 mcg/dl - cerner iron bin { Ivory d { Bernardo ing capacity total 214 250 - 450 mcg/ml ly cerner iron sat { Joslyn uration 32 % - cerne { Ali r testing performed { Wilford  by lab - abbreviation name director address valid date range 123 cerner vumc cerner lab adam seegmiller; 4605 tvc vumc 11/22/21 1014 - present jennifer b. 1301 medical center gordetsky drive nashville { Ulises  tn 37232- 5310 indications normocytic anemia [d64.9 (icd-10-cm)] result notes anton jordan de witte, md 9/8/2023 8:13 am cdt iron { Ember  studies: normal fola { Alvina te: normal vitamin b12: normal ferritin: normal psa: 1.48 vitamin d: normal p { Leota hosphorus: n { Yahir ormal bmp obtained by endocrinol { Kristofer ogy showed a k of 5.2. will ho { Lenard ld losartan and repeat bmp in 1 week attempt { Sandi ed to call patient however  { Marva num { Gonzalo ber listed in chart is incorrect. will ask patie { Luciana nt for updated telephone number admin team: can we send him a letter asap (to his a { Benson ddres { Ansley s) with the letter i wrote out below (i went ahead an { Marybeth d placed this information in a letter commu { Gracelyn nication a { Dinah ddressed to oho pod north in case this helps out more - thanks! all reviewers list { Vonda  printed on 10/3/24 7:13 am page { Darien  1714",
    {
        "entities": [
            [
                3,
                11,
                "PERSON"
            ],
            [
                77,
                85,
                "PERSON"
            ],
            [
                108,
                113,
                "PERSON"
            ],
            [
                215,
                221,
                "PERSON"
            ],
            [
                256,
                262,
                "PERSON"
            ],
            [
                312,
                318,
                "PERSON"
            ],
            [
                326,
                331,
                "PERSON"
            ],
            [
                347,
                353,
                "PERSON"
            ],
            [
                403,
                408,
                "PERSON"
            ],
            [
                425,
                432,
                "PERSON"
            ],
            [
                437,
                443,
                "PERSON"
            ],
            [
                488,
                495,
                "PERSON"
            ],
            [
                577,
                585,
                "PERSON"
            ],
            [
                804,
                810,
                "PERSON"
            ],
            [
                836,
                842,
                "PERSON"
            ],
            [
                994,
                1001,
                "PERSON"
            ],
            [
                1015,
                1021,
                "PERSON"
            ],
            [
                1045,
                1052,
                "PERSON"
            ],
            [
                1233,
                1240,
                "PERSON"
            ],
            [
                1308,
                1318,
                "PERSON"
            ],
            [
                1327,
                1333,
                "PERSON"
            ],
            [
                1354,
                1362,
                "PERSON"
            ],
            [
                1414,
                1419,
                "PERSON"
            ],
            [
                1426,
                1432,
                "PERSON"
            ],
            [
                1453,
                1459,
                "PERSON"
            ],
            [
                1475,
                1481,
                "PERSON"
            ],
            [
                1499,
                1506,
                "PERSON"
            ],
            [
                1572,
                1578,
                "PERSON"
            ],
            [
                1687,
                1693,
                "PERSON"
            ],
            [
                1745,
                1751,
                "PERSON"
            ],
            [
                1758,
                1765,
                "PERSON"
            ],
            [
                1895,
                1902,
                "PERSON"
            ],
            [
                1974,
                1981,
                "PERSON"
            ],
            [
                2242,
                2251,
                "PERSON"
            ],
            [
                2303,
                2311,
                "PERSON"
            ],
            [
                2373,
                2381,
                "PERSON"
            ],
            [
                2392,
                2398,
                "PERSON"
            ],
            [
                2421,
                2429,
                "PERSON"
            ],
            [
                2466,
                2474,
                "PERSON"
            ],
            [
                2490,
                2496,
                "PERSON"
            ],
            [
                2510,
                2518,
                "PERSON"
            ],
            [
                2554,
                2560,
                "PERSON"
            ],
            [
                2596,
                2601,
                "PERSON"
            ],
            [
                2709,
                2715,
                "PERSON"
            ],
            [
                2738,
                2744,
                "PERSON"
            ],
            [
                2834,
                2840,
                "PERSON"
            ],
            [
                2918,
                2925,
                "PERSON"
            ],
            [
                3026,
                3035,
                "PERSON"
            ],
            [
                3078,
                3085,
                "PERSON"
            ],
            [
                3106,
                3113,
                "PERSON"
            ],
            [
                3140,
                3148,
                "PERSON"
            ],
            [
                3231,
                3239,
                "PERSON"
            ],
            [
                3279,
                3287,
                "PERSON"
            ],
            [
                3313,
                3324,
                "PERSON"
            ],
            [
                3395,
                3403,
                "PERSON"
            ],
            [
                3457,
                3463,
                "PERSON"
            ],
            [
                3478,
                3485,
                "PERSON"
            ],
            [
                3491,
                3497,
                "PERSON"
            ],
            [
                3529,
                3534,
                "PERSON"
            ],
            [
                3548,
                3556,
                "PERSON"
            ],
            [
                3647,
                3654,
                "PERSON"
            ],
            [
                3674,
                3681,
                "PERSON"
            ],
            [
                3827,
                3834,
                "PERSON"
            ],
            [
                4036,
                4045,
                "PERSON"
            ],
            [
                4065,
                4070,
                "PERSON"
            ],
            [
                4098,
                4104,
                "PERSON"
            ],
            [
                4139,
                4144,
                "PERSON"
            ],
            [
                4182,
                4190,
                "PERSON"
            ],
            [
                4278,
                4284,
                "PERSON"
            ],
            [
                4306,
                4313,
                "PERSON"
            ],
            [
                4341,
                4349,
                "PERSON"
            ],
            [
                4367,
                4376,
                "PERSON"
            ],
            [
                4429,
                4436,
                "PERSON"
            ],
            [
                4445,
                4452,
                "PERSON"
            ],
            [
                4527,
                4535,
                "PERSON"
            ],
            [
                4581,
                4588,
                "PERSON"
            ],
            [
                4636,
                4645,
                "PERSON"
            ],
            [
                4719,
                4725,
                "PERSON"
            ],
            [
                4729,
                4738,
                "PERSON"
            ],
            [
                4799,
                4806,
                "PERSON"
            ],
            [
                4829,
                4833,
                "PERSON"
            ],
            [
                4855,
                4863,
                "PERSON"
            ],
            [
                5067,
                5074,
                "PERSON"
            ],
            [
                5207,
                5213,
                "PERSON"
            ],
            [
                5237,
                5244,
                "PERSON"
            ],
            [
                5324,
                5330,
                "PERSON"
            ],
            [
                5345,
                5351,
                "PERSON"
            ],
            [
                5386,
                5396,
                "PERSON"
            ],
            [
                5429,
                5436,
                "PERSON"
            ],
            [
                5483,
                5489,
                "PERSON"
            ],
            [
                5519,
                5525,
                "PERSON"
            ],
            [
                5531,
                5539,
                "PERSON"
            ],
            [
                5590,
                5598,
                "PERSON"
            ],
            [
                5684,
                5691,
                "PERSON"
            ],
            [
                5699,
                5706,
                "PERSON"
            ],
            [
                5762,
                5771,
                "PERSON"
            ],
            [
                5817,
                5826,
                "PERSON"
            ],
            [
                5839,
                5845,
                "PERSON"
            ],
            [
                5930,
                5936,
                "PERSON"
            ],
            [
                5971,
                5978,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical center east white, tyrone 1211 medical center dr mrn { Wilburn : 047717361,  { Merry dob: 7/27 { Pasquale /1969, legal sex: m { Amani  nashville tn 37232 visit date: 12/12/2023 12/12/2023 - office visit in vande { Raelyn rbilt diabetes and endocrinology (continued { Deann ) laboratory reports (continued) frequency: routine 1 { Melodie 2/12/23 - class: lab collect  { Judi quantity: 1 diagnoses high risk heterosexual  { Magnolia behav { Christopher ior [z72.51] spec { Alphonse imen information  { Tatyana id type source collected { Nan  by blood i { Theodora ndications high risk heterosexual behavior [z72.51 (icd { Freddie -10-cm)] flowsheets custom formula data row  { Kason name 12/ { Randell 12/23 1 { Edison 425 measurements total weight 2222 percent -cm at change percent 12/12/23 1426 weight change 8 { Atticus 2.04 kgs -cm at since preop 12 { Shellie /12/23 1 { Johnna 426 weight change 82.04 kg -cm at since preop 12 { Kelsi /12/23 1426 other weight change 82.04 kg -cm at since last visit 12/12/ { Marquise 23 1426 w { Janiya eight  { Margret change 82.04 kg -c { Alonso m at  { Heriberto from  { Delphine preop 12/12/23 1 { Jairo 426 weight change 82.04 kg -cm  { Jovan a { Karrie t since last visit 12/12/23 1426 weigh { Dionne t change 82.04 -cm at 12 { Christal /12/23 from pre { Kathi op (kg) 1426 percent weight 82.04 lbs -cm at change { Jaquan  since 12/12/23 1426 preop percent weight 2895.4 percent -cm change since at 12/12/23 1426 last visit p { Sawyer ercent weight 289340 percent -cm change since at 1 { Winfred 2/12/23 1426 { Latanya  last vi { Hassan sit weight change 0.56 percent -cm at since last visit 12/12/23 1426 (%) weight change 180.86 lbs -cm a { Elmo t since preop (lbs) 1 { Chastity 2/12/2 { Chantal 3 1426 weight change 1 lbs -cm at 12/12/23 s { Savanah ince last visit 1426 (lbs) current { Phoenix  weight { Jayleen   { Emelia 82.056 kg at 12/12/2023 2:25 pm -cm at 12/12/23 { Gilda  1426 weight change 82.04 kg -cm at since last visit 12/12/23 1426 fluid 0 -cm at 12/12/23 1426 resuscitation { Hershel  (# { Lavern 3) volume estimates printed on 10/3/24 7:12 am p { Maegan age 1403,vumc adult medical center e { Bradly ast white, tyrone 1211 medical c { Nestor e { Arjun nter dr { George  mrn { Belen : 047717361, dob: { Tessie  7/27/1969, legal sex: m nashville tn 37232 v { Haylie isit date: 12/12/2023 12/12/2023 - office visit in vanderbilt diabete { Chaz s and endocrinology (conti { Mathias nued) flowsheets ( { Jacoby continued) fluid 0 -cm at 12/12/23 1426 resuscitation { Jerrod  (#4) ratio-based meal dosing approx predicted { Armani  6.1 -cm at 12/12/23 ratio (500/wt i in 1426 kg) rec start sliding 36.6 -cm at 12/12/23 scale (3000 / wt 1426 in kg) bas { Killian ic informa { Fallon tion approx predicted 41 -cm at 1 { Ione 2/12/23 1426  { Berniece basal (0.5 * wt in kg) rec start basal 20.5 -cm at { Isabela   { Lon 12/12/23 ( { Clementine .25 * wt in kg) 1426 rec st { Laci art  { Lashonda fixed 6.8 -cm at 12/12/23 meal (.08 wt in 1426 kg) fixed me { Adonis al insulin dosing approx predicted 18.3 -cm { Giuliana  at 12/12/23 corre { Debby ction (1500 / 14 { Erin 26 wt in kg) encounte { Jadon r vitals { Jacquelin  ro { Jensen w name 12/12 { Latrice /23 1425 encounter vi { Elouise tals bp 144/91 ! -cm at 12/12/23 1426 pulse 96 -cm at 12/12/23 1426 we { Tisha ight 82.1 kg (180 lb 14.4 oz) -cm at 12/12/23 1426 event-driven clindoc/stork row nam { Alysia e 12/12/23 1425 abnormal vit { Richard als blood pressure 144/91 -cm at 12/1 { Mckinley 2/23 ab { Tiffanie normal 1426 lund-browder (adult) { Dario  row n { Rolland ame 12/12/ { Sydney 23 1425 volume  { Madonna es { Yaretzi timates fluid 0 -cm at 12/12/23 1426 resuscitation { Kaci  (#5) fl { Lisette uid 0 -cm at  { Bennie 12/12/23 { Bobby  1426 resuscitatio { Brant n (#6) fluid 0 -cm at 12/12/23 1426 { Caren  resuscitation (#7) fluid 0 -cm { Shanice  at 12/12/23  { Treva 1426 resuscitatio { Zechariah n (#8) fluid 0 - { Danial cm at 12/12/23 1426 printed on 10/3/24 7:12 am page 1404",
    {
        "entities": [
            [
                74,
                82,
                "PERSON"
            ],
            [
                98,
                104,
                "PERSON"
            ],
            [
                116,
                125,
                "PERSON"
            ],
            [
                147,
                153,
                "PERSON"
            ],
            [
                233,
                240,
                "PERSON"
            ],
            [
                286,
                292,
                "PERSON"
            ],
            [
                348,
                356,
                "PERSON"
            ],
            [
                388,
                393,
                "PERSON"
            ],
            [
                441,
                450,
                "PERSON"
            ],
            [
                458,
                470,
                "PERSON"
            ],
            [
                490,
                499,
                "PERSON"
            ],
            [
                519,
                527,
                "PERSON"
            ],
            [
                554,
                558,
                "PERSON"
            ],
            [
                572,
                581,
                "PERSON"
            ],
            [
                639,
                647,
                "PERSON"
            ],
            [
                694,
                700,
                "PERSON"
            ],
            [
                711,
                719,
                "PERSON"
            ],
            [
                729,
                736,
                "PERSON"
            ],
            [
                833,
                841,
                "PERSON"
            ],
            [
                874,
                882,
                "PERSON"
            ],
            [
                893,
                900,
                "PERSON"
            ],
            [
                951,
                957,
                "PERSON"
            ],
            [
                1031,
                1040,
                "PERSON"
            ],
            [
                1052,
                1059,
                "PERSON"
            ],
            [
                1068,
                1076,
                "PERSON"
            ],
            [
                1097,
                1104,
                "PERSON"
            ],
            [
                1112,
                1122,
                "PERSON"
            ],
            [
                1130,
                1139,
                "PERSON"
            ],
            [
                1158,
                1164,
                "PERSON"
            ],
            [
                1198,
                1204,
                "PERSON"
            ],
            [
                1208,
                1215,
                "PERSON"
            ],
            [
                1256,
                1263,
                "PERSON"
            ],
            [
                1290,
                1299,
                "PERSON"
            ],
            [
                1317,
                1323,
                "PERSON"
            ],
            [
                1377,
                1384,
                "PERSON"
            ],
            [
                1490,
                1497,
                "PERSON"
            ],
            [
                1550,
                1558,
                "PERSON"
            ],
            [
                1573,
                1581,
                "PERSON"
            ],
            [
                1592,
                1599,
                "PERSON"
            ],
            [
                1705,
                1710,
                "PERSON"
            ],
            [
                1734,
                1743,
                "PERSON"
            ],
            [
                1752,
                1760,
                "PERSON"
            ],
            [
                1807,
                1815,
                "PERSON"
            ],
            [
                1852,
                1860,
                "PERSON"
            ],
            [
                1870,
                1878,
                "PERSON"
            ],
            [
                1882,
                1889,
                "PERSON"
            ],
            [
                1939,
                1945,
                "PERSON"
            ],
            [
                2057,
                2065,
                "PERSON"
            ],
            [
                2071,
                2078,
                "PERSON"
            ],
            [
                2129,
                2136,
                "PERSON"
            ],
            [
                2175,
                2182,
                "PERSON"
            ],
            [
                2217,
                2224,
                "PERSON"
            ],
            [
                2228,
                2234,
                "PERSON"
            ],
            [
                2244,
                2251,
                "PERSON"
            ],
            [
                2258,
                2264,
                "PERSON"
            ],
            [
                2284,
                2291,
                "PERSON"
            ],
            [
                2339,
                2346,
                "PERSON"
            ],
            [
                2418,
                2423,
                "PERSON"
            ],
            [
                2452,
                2460,
                "PERSON"
            ],
            [
                2481,
                2488,
                "PERSON"
            ],
            [
                2544,
                2551,
                "PERSON"
            ],
            [
                2600,
                2607,
                "PERSON"
            ],
            [
                2730,
                2738,
                "PERSON"
            ],
            [
                2751,
                2758,
                "PERSON"
            ],
            [
                2794,
                2799,
                "PERSON"
            ],
            [
                2815,
                2824,
                "PERSON"
            ],
            [
                2877,
                2885,
                "PERSON"
            ],
            [
                2889,
                2893,
                "PERSON"
            ],
            [
                2906,
                2917,
                "PERSON"
            ],
            [
                2947,
                2952,
                "PERSON"
            ],
            [
                2959,
                2968,
                "PERSON"
            ],
            [
                3030,
                3037,
                "PERSON"
            ],
            [
                3083,
                3092,
                "PERSON"
            ],
            [
                3113,
                3119,
                "PERSON"
            ],
            [
                3138,
                3143,
                "PERSON"
            ],
            [
                3167,
                3173,
                "PERSON"
            ],
            [
                3184,
                3194,
                "PERSON"
            ],
            [
                3200,
                3207,
                "PERSON"
            ],
            [
                3222,
                3230,
                "PERSON"
            ],
            [
                3254,
                3262,
                "PERSON"
            ],
            [
                3335,
                3341,
                "PERSON"
            ],
            [
                3429,
                3436,
                "PERSON"
            ],
            [
                3467,
                3475,
                "PERSON"
            ],
            [
                3515,
                3524,
                "PERSON"
            ],
            [
                3534,
                3543,
                "PERSON"
            ],
            [
                3578,
                3584,
                "PERSON"
            ],
            [
                3593,
                3601,
                "PERSON"
            ],
            [
                3614,
                3621,
                "PERSON"
            ],
            [
                3639,
                3647,
                "PERSON"
            ],
            [
                3652,
                3660,
                "PERSON"
            ],
            [
                3713,
                3718,
                "PERSON"
            ],
            [
                3729,
                3737,
                "PERSON"
            ],
            [
                3753,
                3760,
                "PERSON"
            ],
            [
                3771,
                3777,
                "PERSON"
            ],
            [
                3798,
                3804,
                "PERSON"
            ],
            [
                3842,
                3848,
                "PERSON"
            ],
            [
                3882,
                3890,
                "PERSON"
            ],
            [
                3906,
                3912,
                "PERSON"
            ],
            [
                3932,
                3942,
                "PERSON"
            ],
            [
                3961,
                3968,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adu { Wilfredo lt dayani center white, tyr { Angus one 1500 medical ctr dr 1st fl, 108 mrn: 047717361, d { Lauretta ob: 7/27/1969, legal sex: m dayani ctr  { Anabel visit  { Corey date: 12/7/2023 nash { Zoie ville tn 37232 12/07/2023  { Gisselle - appoint { Chauncey ment in vanderbilt dayani center (continued) referral (continued) gene { Ryann ral by og { Lyndon les, pamela j at 10 { Tristin /2 { German 0/2023  { Kasandra 1220 therapy benefits coverag { Suzanna e 80% deductible $0 out of { Natalee  pocket $8,300 co-pay $0 precert after eval telepho { Celestine ne number or { Landyn  websi { Tana te availity person spoke with na on 10/20 call { Adaline  referen { Braylen ce number na visit limit pending { Iola  auth aquat { Sullivan ics based on med nec { Oma  not a guarantee of coverage triage triage information decision: none schedule by  { Fanny date: 11/3/ { Lanny 20 { Lemuel 23 coverages uhc community { Samson  dual snp plan: uhc commu { Lettie nity dual covered: covered from: 1/1/ { Milagros 2024 to: 1/31/2024 snp member #: 12 { Jacques 767095 { Valentin 0 amerivantage wellpoint medic { Averie are plan: amerivantage { Shirley   { Omari covered: covered fr { Eliseo om: 6 { Lura /1/2023 to: 2/29/2024 amerigro { Aleena up wellpoint ma member #: 768w12411 auth #:  { Mckinley 02k4 { Barton hyh33 zzzmcaid of tennessee plan:  { Pattie medicaid { Bronson  supple { Jaylin me { Bud ntal covered: { Benton  covered from: 6/1 { Unknown /2022 to: 12/3 { Demi 1/2023 member #: td525606 { Alex 373 auth #: nar { Arely  tc tenncare s { Essence elect plan: tc se { Rico lect  { Dino  covered: covered from: 8/30/2024 to: 8/30/2024 memb { Kristyn er #: zedm13004089 m { Winter essages appointment rescheduled from to se { Kirstin nt and deliv { Sarina ered mycha { Antionette rt, generi { Ally c { Katelin  white, tyrone 1 { Alijah 2/5/2023 1 { Maryam 0:02 am last read in my health at vanderbilt not read printed on 10/3/24 7:13 am page 143 { Starr 3,vumc adult dayani center white, tyrone 1500 medical ctr dr 1st { Kerrie  fl, 108 mrn: 047717361, dob: 7/27/1969, l { Zariah egal sex: m dayani ctr visit date: 12/7/2023 nashville tn  { Unknown 37232 12/07/2023 - a { Chaim ppoint { Amya ment in vanderbi { Aliya lt dayani center (co { Marcelo ntinued) message { Jaylon s (cont { Sunny inued) appoi { Olin ntment information: visit type: pt therapy date: 12/7/2023 de { Evangelina pt: vanderbilt  { Stacia dayani c { Karol ent { Linnea er provider: jeff a cob { Philomena ble time: 3:40 pm  { Uriah lengt { Josef h:  { Jayceon 50 min appt status { Berenice : { Astrid  sch { Raymundo ed { Patrica uled appt instructions: save time. skip the  { Layton line. you can now use self check-in for your visit. use my health at vanderbilt to check-in fo { Abe r your visit right from your phone. let us know you're here { Brigitte  ope { Kian n the mhav app on  { Shyanne your phone  { Richelle when you arrive and click \"i'm h { Kyle e { Andria re\" once you arrive at  { Cliff your a { Griselda ppointment. original appointment information: visit type: pt therapy { Chantelle  date: 12/1/2023 d { Jaida ept: vanderb { Matthias il { Fabiola t dayani center provider: jeff a cobble  { Bernie time: 8:00 am length: 50 min printed on 1 { Magdalene 0/3/24 7:13 am page { Liz  1434",
    {
        "entities": [
            [
                11,
                20,
                "PERSON"
            ],
            [
                50,
                56,
                "PERSON"
            ],
            [
                112,
                121,
                "PERSON"
            ],
            [
                163,
                170,
                "PERSON"
            ],
            [
                179,
                185,
                "PERSON"
            ],
            [
                208,
                213,
                "PERSON"
            ],
            [
                242,
                251,
                "PERSON"
            ],
            [
                263,
                272,
                "PERSON"
            ],
            [
                345,
                351,
                "PERSON"
            ],
            [
                363,
                370,
                "PERSON"
            ],
            [
                392,
                400,
                "PERSON"
            ],
            [
                405,
                412,
                "PERSON"
            ],
            [
                422,
                431,
                "PERSON"
            ],
            [
                463,
                471,
                "PERSON"
            ],
            [
                500,
                508,
                "PERSON"
            ],
            [
                562,
                572,
                "PERSON"
            ],
            [
                587,
                594,
                "PERSON"
            ],
            [
                603,
                608,
                "PERSON"
            ],
            [
                657,
                665,
                "PERSON"
            ],
            [
                676,
                684,
                "PERSON"
            ],
            [
                719,
                724,
                "PERSON"
            ],
            [
                738,
                747,
                "PERSON"
            ],
            [
                770,
                774,
                "PERSON"
            ],
            [
                859,
                865,
                "PERSON"
            ],
            [
                879,
                885,
                "PERSON"
            ],
            [
                890,
                897,
                "PERSON"
            ],
            [
                926,
                933,
                "PERSON"
            ],
            [
                961,
                968,
                "PERSON"
            ],
            [
                1008,
                1017,
                "PERSON"
            ],
            [
                1055,
                1063,
                "PERSON"
            ],
            [
                1072,
                1081,
                "PERSON"
            ],
            [
                1114,
                1121,
                "PERSON"
            ],
            [
                1146,
                1154,
                "PERSON"
            ],
            [
                1158,
                1164,
                "PERSON"
            ],
            [
                1186,
                1193,
                "PERSON"
            ],
            [
                1201,
                1206,
                "PERSON"
            ],
            [
                1239,
                1246,
                "PERSON"
            ],
            [
                1293,
                1302,
                "PERSON"
            ],
            [
                1309,
                1316,
                "PERSON"
            ],
            [
                1353,
                1360,
                "PERSON"
            ],
            [
                1371,
                1379,
                "PERSON"
            ],
            [
                1389,
                1396,
                "PERSON"
            ],
            [
                1401,
                1405,
                "PERSON"
            ],
            [
                1421,
                1428,
                "PERSON"
            ],
            [
                1449,
                1457,
                "PERSON"
            ],
            [
                1474,
                1479,
                "PERSON"
            ],
            [
                1507,
                1512,
                "PERSON"
            ],
            [
                1530,
                1536,
                "PERSON"
            ],
            [
                1553,
                1561,
                "PERSON"
            ],
            [
                1581,
                1586,
                "PERSON"
            ],
            [
                1594,
                1599,
                "PERSON"
            ],
            [
                1654,
                1662,
                "PERSON"
            ],
            [
                1685,
                1692,
                "PERSON"
            ],
            [
                1737,
                1745,
                "PERSON"
            ],
            [
                1760,
                1767,
                "PERSON"
            ],
            [
                1780,
                1791,
                "PERSON"
            ],
            [
                1804,
                1809,
                "PERSON"
            ],
            [
                1813,
                1821,
                "PERSON"
            ],
            [
                1840,
                1847,
                "PERSON"
            ],
            [
                1860,
                1867,
                "PERSON"
            ],
            [
                1959,
                1965,
                "PERSON"
            ],
            [
                2032,
                2039,
                "PERSON"
            ],
            [
                2084,
                2091,
                "PERSON"
            ],
            [
                2152,
                2160,
                "PERSON"
            ],
            [
                2183,
                2189,
                "PERSON"
            ],
            [
                2198,
                2203,
                "PERSON"
            ],
            [
                2222,
                2228,
                "PERSON"
            ],
            [
                2251,
                2259,
                "PERSON"
            ],
            [
                2278,
                2285,
                "PERSON"
            ],
            [
                2295,
                2301,
                "PERSON"
            ],
            [
                2316,
                2321,
                "PERSON"
            ],
            [
                2385,
                2396,
                "PERSON"
            ],
            [
                2414,
                2421,
                "PERSON"
            ],
            [
                2432,
                2438,
                "PERSON"
            ],
            [
                2444,
                2451,
                "PERSON"
            ],
            [
                2477,
                2487,
                "PERSON"
            ],
            [
                2508,
                2514,
                "PERSON"
            ],
            [
                2522,
                2528,
                "PERSON"
            ],
            [
                2534,
                2542,
                "PERSON"
            ],
            [
                2563,
                2572,
                "PERSON"
            ],
            [
                2576,
                2583,
                "PERSON"
            ],
            [
                2590,
                2599,
                "PERSON"
            ],
            [
                2604,
                2612,
                "PERSON"
            ],
            [
                2659,
                2666,
                "PERSON"
            ],
            [
                2763,
                2767,
                "PERSON"
            ],
            [
                2829,
                2838,
                "PERSON"
            ],
            [
                2845,
                2850,
                "PERSON"
            ],
            [
                2871,
                2879,
                "PERSON"
            ],
            [
                2893,
                2902,
                "PERSON"
            ],
            [
                2937,
                2942,
                "PERSON"
            ],
            [
                2946,
                2953,
                "PERSON"
            ],
            [
                2979,
                2985,
                "PERSON"
            ],
            [
                2994,
                3003,
                "PERSON"
            ],
            [
                3074,
                3084,
                "PERSON"
            ],
            [
                3105,
                3111,
                "PERSON"
            ],
            [
                3126,
                3135,
                "PERSON"
            ],
            [
                3140,
                3148,
                "PERSON"
            ],
            [
                3191,
                3198,
                "PERSON"
            ],
            [
                3242,
                3252,
                "PERSON"
            ],
            [
                3274,
                3278,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult on { Aliza e hundred oaks white, ty { Dottie rone 719 thompson lane, nashville mrn:  { Evelynn 047717361, do { Myah b: 7/27/1969, legal sex: m nashville tn 37204 vis { Adelynn it date: 11/17/2023 { Georgiana  11/17/2023 - office visit in  { Holli vanderbil { Enid t one hundred oaks pri { Suzan mar { Venus y care n { Malissa orth (continued { Jaycee ) { Bree  clinical notes (continu { Jeanna ed) gunshot wound medications and alle { Lucius rgies: current outpatient medications o { Aracely n file prior to visit  { Brylee medic { Georgianna at { Paris ion sig dispense ref { Roxann ill albuterol sulfate hfa 90 inhale 2 puff { Nakia s ev { Lainey ery 4 18 g 11 mcg/actuation aerosol inhale { Dominik r hou { Benedict rs as needed for  { Erna wheezing. aspirin 8 { Kellan 1 mg tablet,delayed take 1 tablet (81 mg 90 tablet 3 release total) by { Mackenzie  mo { Dannie uth d { Jed aily. atorvastatin 80 mg tablet take 1 tablet (80 mg 90 tablet 3 (lipitor) total) by mouth daily. azelastine 137 mcg (0.1 { London  %) administer 1 spra { Rodrick y 30 ml 0 { Leif  nasal { Katarina  spray aerosol (astelin) into each nostril 2 times a day as needed for rhinitis { Jaliyah . { Carli  use in e { Coby ach n { Gail ostril as directed capsaicin 0.1 % topical cream apply 1 { Maurine  applica { Zaria tion 42.5 g 0 topically daily for 90 days. cetirizine 10 mg tablet (zyrtec) { Thiago  take 1 tablet (10 mg 30 tablet 9 tot { Marlys al) by mouth once a day as needed for allergies. cyclo { Dovie benzaprine 5 mg tablet take 1 tablet (5 mg (flexeril) total) by mouth every 8 hours as needed. diclofenac 1 % topical gel apply 2  { Mac g topica { Brennen lly 4 100 g  { Kaya 0 times a day for 3 { Kendal 0 days. docusate sodium 100 mg take one tablet tid { Kami  60 capsule 0 capsule (colace) prn constipation famot { Maren idine 20 mg tablet take 1 tablet (20 mg  { Mariano (pepcid) total) by mouth every  { Elda 12 hours. freestyle libre 2 sensor kit 1 kit (1 each total) 2 kit  { Jazmyn 3 (flash gluc { Cason ose sensor) every 14 days. gabapentin 300 mg capsule take 1 capsule (300 90 capsule 0 (neurontin) mg  { Paloma total) by mouth daily. insulin glargine (u-100) 100 inject 0.05  { Kamden ml (5 4.5 ml 0 unit/ml subcutaneous solution units total) under the skin daily. lancets 33 gauge (trueplus 1 lancet 2 times a 200 eac { Jamaal h 2 lan { Corrie cets) day. lido { Errol caine  { Raleigh 5 % topical patch apply 1 patch 30 patch 11 (lidoderm) topica { Brinley lly daily. apply to painful area 12  { Keven printed on 10/3/24 7:13 am page 1499,vumc adult one hundred oaks whi { Annetta te, t { Shiloh yrone 719 thompson lane, nashville mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date: 11/17/2023 11/17/2023 - o { Ferdinand ffice visit in vanderbilt one hundred oaks primary care north (continued) clinical notes (continued) hours per day, remove for 12 hours. losartan 25 mg tablet take 1  { Tracey tablet (25 mg 90 tablet 3 (cozaar) total) by mouth daily. montelukast 10 mg tablet take 1 tablet (10 mg 90 tablet 3 (singulair) total) by mouth every evening. nifedipine er 30 mg ta { Duke ke 1 tablet (30 mg 90 tablet 3 tablet, extended relea { Jaylene se total) by mouth daily. { Rob  (adalat  { Rod cc) pantoprazole 20 mg take 1 tablet (20 mg 30 tablet 11 tablet, dela { Asha yed  { Emersyn release total) by  { Kimber mouth daily. (protonix) { Emmy  pen needle { Allene , di { Caylee abetic 31 gauge x use as directed. to 90 each 3 3/16\" inject  { Lindy in { Lorine su { Daniel li { Agatha n once daily triamcinolone acet { Mireya onide 55 mcg administer 2 spr { Hayes ays 16.5 g 11 nasal spray aerosol (110 mcg total) into (nasacort) each nostri { River l  { Kaliyah 2 times a day. trulicity 0.75 mg/0.5 ml injec { Alessandro t 0 { Alva .75 mg under 6 ml 3 subcutaneous pen injector the skin every 7 (dulaglutide) days. acetaminophen 325 mg tablet take 2 tablets (650 30 tablet 0 (tylenol) mg total) by mouth every 6 hours as  { Jules needed  { Noble for { Alene  mild pain, moderate pain, headach { Alexandrea es or fever. (p { Vincenzo atient not taking: reported on 9/7/2023) [ { Misti expired]  { Thomas blood sugar 1 strip 4 t { Harriette imes a day 120 strip 11 diag { Carol nostic strips (before meals and at bedtime). (patient not taking: reported on 6/6/2023)  facility-a { Broderick dministered medications on file prior to visit.  allergies social history: social history socioeconomic { Braelyn  his { Reta tory marita { Estevan l status: single spouse name: not on file number of childre { Federico n: not on file years of education: { Lolita  not on file printed on 10/3/24  { Jaydon 7:13 am page 1500",
    {
        "entities": [
            [
                16,
                22,
                "PERSON"
            ],
            [
                49,
                56,
                "PERSON"
            ],
            [
                98,
                106,
                "PERSON"
            ],
            [
                122,
                127,
                "PERSON"
            ],
            [
                179,
                187,
                "PERSON"
            ],
            [
                209,
                219,
                "PERSON"
            ],
            [
                252,
                258,
                "PERSON"
            ],
            [
                270,
                275,
                "PERSON"
            ],
            [
                300,
                306,
                "PERSON"
            ],
            [
                312,
                318,
                "PERSON"
            ],
            [
                329,
                337,
                "PERSON"
            ],
            [
                355,
                362,
                "PERSON"
            ],
            [
                366,
                371,
                "PERSON"
            ],
            [
                398,
                405,
                "PERSON"
            ],
            [
                446,
                453,
                "PERSON"
            ],
            [
                495,
                503,
                "PERSON"
            ],
            [
                528,
                535,
                "PERSON"
            ],
            [
                543,
                554,
                "PERSON"
            ],
            [
                559,
                565,
                "PERSON"
            ],
            [
                588,
                595,
                "PERSON"
            ],
            [
                640,
                646,
                "PERSON"
            ],
            [
                653,
                660,
                "PERSON"
            ],
            [
                705,
                713,
                "PERSON"
            ],
            [
                721,
                730,
                "PERSON"
            ],
            [
                750,
                755,
                "PERSON"
            ],
            [
                777,
                784,
                "PERSON"
            ],
            [
                857,
                867,
                "PERSON"
            ],
            [
                873,
                880,
                "PERSON"
            ],
            [
                888,
                892,
                "PERSON"
            ],
            [
                1016,
                1023,
                "PERSON"
            ],
            [
                1047,
                1055,
                "PERSON"
            ],
            [
                1067,
                1072,
                "PERSON"
            ],
            [
                1081,
                1090,
                "PERSON"
            ],
            [
                1172,
                1180,
                "PERSON"
            ],
            [
                1184,
                1190,
                "PERSON"
            ],
            [
                1202,
                1207,
                "PERSON"
            ],
            [
                1215,
                1220,
                "PERSON"
            ],
            [
                1279,
                1287,
                "PERSON"
            ],
            [
                1298,
                1304,
                "PERSON"
            ],
            [
                1382,
                1389,
                "PERSON"
            ],
            [
                1429,
                1436,
                "PERSON"
            ],
            [
                1493,
                1499,
                "PERSON"
            ],
            [
                1632,
                1636,
                "PERSON"
            ],
            [
                1647,
                1655,
                "PERSON"
            ],
            [
                1670,
                1675,
                "PERSON"
            ],
            [
                1697,
                1704,
                "PERSON"
            ],
            [
                1757,
                1762,
                "PERSON"
            ],
            [
                1818,
                1824,
                "PERSON"
            ],
            [
                1867,
                1875,
                "PERSON"
            ],
            [
                1909,
                1914,
                "PERSON"
            ],
            [
                1983,
                1990,
                "PERSON"
            ],
            [
                2006,
                2012,
                "PERSON"
            ],
            [
                2116,
                2123,
                "PERSON"
            ],
            [
                2190,
                2197,
                "PERSON"
            ],
            [
                2333,
                2340,
                "PERSON"
            ],
            [
                2350,
                2357,
                "PERSON"
            ],
            [
                2375,
                2381,
                "PERSON"
            ],
            [
                2390,
                2398,
                "PERSON"
            ],
            [
                2462,
                2470,
                "PERSON"
            ],
            [
                2509,
                2515,
                "PERSON"
            ],
            [
                2586,
                2594,
                "PERSON"
            ],
            [
                2602,
                2609,
                "PERSON"
            ],
            [
                2748,
                2758,
                "PERSON"
            ],
            [
                2927,
                2934,
                "PERSON"
            ],
            [
                3118,
                3123,
                "PERSON"
            ],
            [
                3179,
                3187,
                "PERSON"
            ],
            [
                3215,
                3219,
                "PERSON"
            ],
            [
                3231,
                3235,
                "PERSON"
            ],
            [
                3307,
                3312,
                "PERSON"
            ],
            [
                3319,
                3327,
                "PERSON"
            ],
            [
                3348,
                3355,
                "PERSON"
            ],
            [
                3381,
                3386,
                "PERSON"
            ],
            [
                3400,
                3407,
                "PERSON"
            ],
            [
                3414,
                3421,
                "PERSON"
            ],
            [
                3485,
                3491,
                "PERSON"
            ],
            [
                3496,
                3503,
                "PERSON"
            ],
            [
                3508,
                3515,
                "PERSON"
            ],
            [
                3520,
                3527,
                "PERSON"
            ],
            [
                3561,
                3568,
                "PERSON"
            ],
            [
                3600,
                3606,
                "PERSON"
            ],
            [
                3686,
                3692,
                "PERSON"
            ],
            [
                3697,
                3705,
                "PERSON"
            ],
            [
                3753,
                3764,
                "PERSON"
            ],
            [
                3770,
                3775,
                "PERSON"
            ],
            [
                3967,
                3973,
                "PERSON"
            ],
            [
                3983,
                3989,
                "PERSON"
            ],
            [
                3995,
                4001,
                "PERSON"
            ],
            [
                4038,
                4049,
                "PERSON"
            ],
            [
                4067,
                4076,
                "PERSON"
            ],
            [
                4121,
                4127,
                "PERSON"
            ],
            [
                4139,
                4146,
                "PERSON"
            ],
            [
                4172,
                4182,
                "PERSON"
            ],
            [
                4213,
                4219,
                "PERSON"
            ],
            [
                4321,
                4331,
                "PERSON"
            ],
            [
                4437,
                4445,
                "PERSON"
            ],
            [
                4452,
                4457,
                "PERSON"
            ],
            [
                4471,
                4479,
                "PERSON"
            ],
            [
                4541,
                4550,
                "PERSON"
            ],
            [
                4587,
                4594,
                "PERSON"
            ],
            [
                4629,
                4636,
                "PERSON"
            ]
        ]
    }
),(
    "vumc hendersonville - anderson white, tyrone 128 n anderson ln mrn: 047717361, d { Tyrese o { Lakesha b: 7/27 { Ambrose /1969, { Blakely  legal sex: { Oren  m hendersonville tn 37075 vis { Robbin it d { Luanne ate: 8/1/2023 08/ { Jaylah 01/2023  { Shea - patient message in vanderbilt primary care h { Stephon endersonville facesheet report patient demographics patient na { Lucien me mrn  { Cale legal dob address phone  { Turner white, tyrone 0477173 sex 7/27/196 { Wilton 9 apt 705 615 { Briella -260-2291 (home) 61 m 1101 edgehill  { Yazmin ave 615 { Adrien -260-2291 (mobile) nashville t { Doretha n 37203 *prefe { Addisyn rred* hos { Selah pital accoun { Lyman t no { Kole t on fil { Davina e admission i { Francesco nformation current { Cristy  information attending provider admitt { Shakira ing prov { Ofelia ider admissi { Darleen on type admissio { Delois n status unknown status admission d { Nylah ate/time { Dashawn  disc { Lizeth harge dat { Aryanna e/time hospital  { Gertie service auth/cert status hospital area unit room/bed referring provider 08/01/2023 - patient m { Jerrold essage { Stan  in vanderbilt primary care hendersonville { Daria  (conti { Delma nued) vis { Kayley it information { Elana  provider info { Kenia rma { Charley tion encounter pro { Ivory vider acosta, { Santino  li { Estela sa marie, rn { Shanda  departme { Arron nt name addres { Dimitri s va { Finnegan nd { Kianna erbil { Omer t  { Emmalee primary care { Jaylynn  128 n  { Tylor anderson ln hendersonville hendersonv { Aiyana ill { Zack e tn { Salma  37075 message { Anabelle s { Chandler  gabapen { Simeon tin from to sent and del { Fredric ivered lisa { Jabari  marie aco { Marjory sta, rn white,  { Kash tyrone 8/1/2023 3 { Keon : { Lachlan 48 pm last read in m { Marcela y healt { Livia h at vanderbi { Marques lt not read hi tyron { Misael e gile { Montana s has { Thad  sent yo { Andreas ur pre { Darrick scripti { Aja on fo { Mercy r g { Siena abapentin in to the { Samir  pharm { Giana acy. he did give the dose to be taken twice a { Charla  day. please message { Devyn  me with any questions. lisa rn printed on 10 { Milford /3/ { Justus 24 7 { Emile :13 am page 202 { Mazie 5,vumc hendersonville - anderson w { Jaiden hite, tyrone 128 n anderson ln mrn: 0 { Eloy 47717361 { Jacquline , do { Tena b: 7/27/1969, legal { Donavan   { Evalyn sex: m hendersonville tn 37075 visit date: 8/1/2023 08/01/2023 - pati { Malakai ent message in vanderbilt { Starla  primary care hend { Lorri er { Xiomara sonville (con { Odis tinued) messages (continued) { Vesta  printe { Geri d on 1 { Dax 0/3/24 7:13 am pa { Keagan ge 2026",
    {
        "entities": [
            [
                83,
                90,
                "PERSON"
            ],
            [
                94,
                102,
                "PERSON"
            ],
            [
                112,
                120,
                "PERSON"
            ],
            [
                129,
                137,
                "PERSON"
            ],
            [
                151,
                156,
                "PERSON"
            ],
            [
                189,
                196,
                "PERSON"
            ],
            [
                203,
                210,
                "PERSON"
            ],
            [
                230,
                237,
                "PERSON"
            ],
            [
                248,
                253,
                "PERSON"
            ],
            [
                302,
                310,
                "PERSON"
            ],
            [
                375,
                382,
                "PERSON"
            ],
            [
                392,
                397,
                "PERSON"
            ],
            [
                424,
                431,
                "PERSON"
            ],
            [
                468,
                475,
                "PERSON"
            ],
            [
                491,
                499,
                "PERSON"
            ],
            [
                538,
                545,
                "PERSON"
            ],
            [
                555,
                562,
                "PERSON"
            ],
            [
                595,
                603,
                "PERSON"
            ],
            [
                620,
                628,
                "PERSON"
            ],
            [
                640,
                646,
                "PERSON"
            ],
            [
                661,
                667,
                "PERSON"
            ],
            [
                674,
                679,
                "PERSON"
            ],
            [
                690,
                697,
                "PERSON"
            ],
            [
                713,
                723,
                "PERSON"
            ],
            [
                744,
                751,
                "PERSON"
            ],
            [
                792,
                800,
                "PERSON"
            ],
            [
                811,
                818,
                "PERSON"
            ],
            [
                833,
                841,
                "PERSON"
            ],
            [
                860,
                867,
                "PERSON"
            ],
            [
                905,
                911,
                "PERSON"
            ],
            [
                922,
                930,
                "PERSON"
            ],
            [
                938,
                945,
                "PERSON"
            ],
            [
                957,
                965,
                "PERSON"
            ],
            [
                984,
                991,
                "PERSON"
            ],
            [
                1088,
                1096,
                "PERSON"
            ],
            [
                1105,
                1110,
                "PERSON"
            ],
            [
                1155,
                1161,
                "PERSON"
            ],
            [
                1171,
                1177,
                "PERSON"
            ],
            [
                1189,
                1196,
                "PERSON"
            ],
            [
                1213,
                1219,
                "PERSON"
            ],
            [
                1236,
                1242,
                "PERSON"
            ],
            [
                1248,
                1256,
                "PERSON"
            ],
            [
                1277,
                1283,
                "PERSON"
            ],
            [
                1299,
                1307,
                "PERSON"
            ],
            [
                1313,
                1320,
                "PERSON"
            ],
            [
                1335,
                1342,
                "PERSON"
            ],
            [
                1354,
                1360,
                "PERSON"
            ],
            [
                1377,
                1385,
                "PERSON"
            ],
            [
                1392,
                1401,
                "PERSON"
            ],
            [
                1406,
                1413,
                "PERSON"
            ],
            [
                1421,
                1426,
                "PERSON"
            ],
            [
                1431,
                1439,
                "PERSON"
            ],
            [
                1454,
                1462,
                "PERSON"
            ],
            [
                1472,
                1478,
                "PERSON"
            ],
            [
                1518,
                1525,
                "PERSON"
            ],
            [
                1531,
                1536,
                "PERSON"
            ],
            [
                1543,
                1549,
                "PERSON"
            ],
            [
                1566,
                1575,
                "PERSON"
            ],
            [
                1579,
                1588,
                "PERSON"
            ],
            [
                1599,
                1606,
                "PERSON"
            ],
            [
                1633,
                1641,
                "PERSON"
            ],
            [
                1655,
                1662,
                "PERSON"
            ],
            [
                1675,
                1683,
                "PERSON"
            ],
            [
                1701,
                1706,
                "PERSON"
            ],
            [
                1726,
                1731,
                "PERSON"
            ],
            [
                1735,
                1743,
                "PERSON"
            ],
            [
                1766,
                1774,
                "PERSON"
            ],
            [
                1784,
                1790,
                "PERSON"
            ],
            [
                1806,
                1814,
                "PERSON"
            ],
            [
                1837,
                1844,
                "PERSON"
            ],
            [
                1853,
                1861,
                "PERSON"
            ],
            [
                1869,
                1874,
                "PERSON"
            ],
            [
                1885,
                1893,
                "PERSON"
            ],
            [
                1902,
                1910,
                "PERSON"
            ],
            [
                1920,
                1924,
                "PERSON"
            ],
            [
                1932,
                1938,
                "PERSON"
            ],
            [
                1944,
                1950,
                "PERSON"
            ],
            [
                1972,
                1978,
                "PERSON"
            ],
            [
                1987,
                1993,
                "PERSON"
            ],
            [
                2041,
                2048,
                "PERSON"
            ],
            [
                2071,
                2077,
                "PERSON"
            ],
            [
                2125,
                2133,
                "PERSON"
            ],
            [
                2139,
                2146,
                "PERSON"
            ],
            [
                2153,
                2159,
                "PERSON"
            ],
            [
                2177,
                2183,
                "PERSON"
            ],
            [
                2220,
                2227,
                "PERSON"
            ],
            [
                2267,
                2272,
                "PERSON"
            ],
            [
                2283,
                2293,
                "PERSON"
            ],
            [
                2300,
                2305,
                "PERSON"
            ],
            [
                2327,
                2335,
                "PERSON"
            ],
            [
                2339,
                2346,
                "PERSON"
            ],
            [
                2418,
                2426,
                "PERSON"
            ],
            [
                2454,
                2461,
                "PERSON"
            ],
            [
                2482,
                2488,
                "PERSON"
            ],
            [
                2493,
                2501,
                "PERSON"
            ],
            [
                2517,
                2522,
                "PERSON"
            ],
            [
                2553,
                2559,
                "PERSON"
            ],
            [
                2569,
                2574,
                "PERSON"
            ],
            [
                2583,
                2587,
                "PERSON"
            ],
            [
                2607,
                2614,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical center east white, tyrone  { Buster 1211 medical center dr mr { Hoyt n: 047717361, dob: 7/27/1969, legal sex: m  { Journee nashville tn 37232 visit date: 4/17/2024 04/17/2024 -  { Sydnee patient message in vander { Sherlyn bilt diabetes and endocrinology facesheet { Yareli  report patient demo { Kala graphics patient name mrn legal  { Madilynn dob address p { Luka hone white, tyrone 0477173 { Malaysia  sex 7/27/196 { Daxton 9 apt 705 615-260-2291 (home) 61 m 1101 edgehil { Kia l ave 615-260-2291 (mobile)  { Miah nashville tn 37203 *prefe { Verda rred* hospital acco { Newton unt not on file admission information current information attending p { Darby rovider admitting  { Remi p { Kimora rovider admi { Zora ssion { Alesha  type admission status unknown status admission date/time discharge date/time hospital { Dangelo  service auth/cert sta { Bertram tus hospi { Austen tal are { Valentino a unit room/bed referring provider 04/17/2024 - patient { Lindsey  message in  { Sharyn vanderbilt diabete { Cathrine s and endocrinology (continued) visit information provider information encounter provider greenspan, debra { Barbra  l, aprn department name address phone fax vanderbilt diab { Belva etes and endocrinology 1215 21st ave s 615-343-8332 615-343-8346 8th fl, suite 8210 nashville tn 37232 mess { Raymon ages insullin from to sent and { Dorene  delivered d { Deonte ebra l g { Dani reenspan, aprn white, tyrone  { Kip 4/17/2024 3:07 pm last read in my health at vanderbilt not read tyronne: i wi { Antonette ll refill { Germaine  your insulin but remindin { Brisa g you that you missed  { Orval your appoi { Jeana ntment  { Abner with me. please get you { Linnie rself rescheduled. 615 { Beckham -343-8332 deb greenspan printed on 10/3/24 7:12  { Cecily am page 1059,vumc vis midtown white, tyrone 337 22nd av { Efren e n mrn: 047717361, dob: 7/27/1969, legal sex: m nas { Palmer hville tn 37203 adm: 4/16/2024, d/c: 4/16/2024 04/16/2024 - fl vide { Saige o swallow w speech in vand { Alani erbilt imaging se { Boston rvices midtown facesheet { Zuri  report patient demographics patient name mrn legal dob address phone white, tyrone 0477173 sex 7/27/1969 apt 705 615-260-2291 (home) 61 m 1101 edgehill ave 615-260-22 { Devante 91 (mobile) nashville tn  { Jaelynn 37203 *preferred* hospital account not on  { Lorne file admission information current informa { Kale tion attending provider a { Westley dmitting provider admission type admi { Isidro ssion status rebula, emily rose kueser, completed hov pa-c 615-322-6180 admission date/time dis { Madison charge date/time hospital service auth/cert status 04/16/2 { Amelie 4 1506 04/16/24 2359 hos { Deacon pital a { Abril rea unit room/bed referring provider vumc vis midtown fluoroscopy vis rebula, emily r { Toby ose kueser, midtown pa-c 615-32 { Audrina 2-6180 discharg { Niko e d { Madalynn isposition discharg { Tod e { Kenyon  destination home or self care 04/16/2024 { Napoleon  - fl video swallow w speech in vanderbilt imagin { Cindi g services midtown (continued) reason for visit visit diagnoses [last edited by systemgener { Jaylin ated, documentation on 4/16 { Mose /2024 1506] speec { Rosalia h disturbance, unspecified type anosmia { Yosef  dysphagia, ora { Iesha l p { Lilyana hase  { Leighton visit information provider information referring provider rebula, emily rose k { Ona ueser, pa-c department name address phone fax vanderbilt imaging services midtown 337 22nd  { Alyse ave n 615-3 { Lissette 27-1500 615-327-1421 nashville tn 37203 medication list medication list 1 this re { Jamila po { Moira r { Marcellus t  { Garrison is for { Sabina  documen { Una tation purposes only. the patient should not { Quinten  follow medication instructions withi { Rey n. for accurate  { Siobhan instructions { Giancarlo  regarding m { Rubi edications, the patient sho { Nikolai ul { Stanford d instead consult their physician or after visit summar { Armani y. active at the end of visit medications last review { Yusuf ed { Jenelle  by bobo, jennifer, rn on 3/21/2024 1245 panto { Tyron prazole 20 mg tablet,delaye { Joana d release (protonix) pr { Warner i { Carmel nted on 1 { Ronnie 0/3 { Patsy /24 7 { Roma :12 am page 1060",
    {
        "entities": [
            [
                48,
                55,
                "PERSON"
            ],
            [
                83,
                88,
                "PERSON"
            ],
            [
                134,
                142,
                "PERSON"
            ],
            [
                199,
                206,
                "PERSON"
            ],
            [
                234,
                242,
                "PERSON"
            ],
            [
                286,
                293,
                "PERSON"
            ],
            [
                316,
                321,
                "PERSON"
            ],
            [
                356,
                365,
                "PERSON"
            ],
            [
                381,
                386,
                "PERSON"
            ],
            [
                415,
                424,
                "PERSON"
            ],
            [
                440,
                447,
                "PERSON"
            ],
            [
                497,
                501,
                "PERSON"
            ],
            [
                532,
                537,
                "PERSON"
            ],
            [
                565,
                571,
                "PERSON"
            ],
            [
                593,
                600,
                "PERSON"
            ],
            [
                672,
                678,
                "PERSON"
            ],
            [
                699,
                704,
                "PERSON"
            ],
            [
                708,
                715,
                "PERSON"
            ],
            [
                730,
                735,
                "PERSON"
            ],
            [
                743,
                750,
                "PERSON"
            ],
            [
                839,
                847,
                "PERSON"
            ],
            [
                872,
                880,
                "PERSON"
            ],
            [
                892,
                899,
                "PERSON"
            ],
            [
                909,
                919,
                "PERSON"
            ],
            [
                977,
                985,
                "PERSON"
            ],
            [
                1000,
                1007,
                "PERSON"
            ],
            [
                1028,
                1037,
                "PERSON"
            ],
            [
                1146,
                1153,
                "PERSON"
            ],
            [
                1214,
                1220,
                "PERSON"
            ],
            [
                1330,
                1337,
                "PERSON"
            ],
            [
                1370,
                1377,
                "PERSON"
            ],
            [
                1392,
                1399,
                "PERSON"
            ],
            [
                1410,
                1415,
                "PERSON"
            ],
            [
                1447,
                1451,
                "PERSON"
            ],
            [
                1531,
                1541,
                "PERSON"
            ],
            [
                1553,
                1562,
                "PERSON"
            ],
            [
                1591,
                1597,
                "PERSON"
            ],
            [
                1622,
                1628,
                "PERSON"
            ],
            [
                1641,
                1647,
                "PERSON"
            ],
            [
                1657,
                1663,
                "PERSON"
            ],
            [
                1689,
                1696,
                "PERSON"
            ],
            [
                1721,
                1729,
                "PERSON"
            ],
            [
                1780,
                1787,
                "PERSON"
            ],
            [
                1845,
                1851,
                "PERSON"
            ],
            [
                1906,
                1913,
                "PERSON"
            ],
            [
                1983,
                1989,
                "PERSON"
            ],
            [
                2018,
                2024,
                "PERSON"
            ],
            [
                2044,
                2051,
                "PERSON"
            ],
            [
                2078,
                2083,
                "PERSON"
            ],
            [
                2253,
                2261,
                "PERSON"
            ],
            [
                2289,
                2297,
                "PERSON"
            ],
            [
                2342,
                2348,
                "PERSON"
            ],
            [
                2393,
                2398,
                "PERSON"
            ],
            [
                2426,
                2434,
                "PERSON"
            ],
            [
                2474,
                2481,
                "PERSON"
            ],
            [
                2579,
                2587,
                "PERSON"
            ],
            [
                2648,
                2655,
                "PERSON"
            ],
            [
                2682,
                2689,
                "PERSON"
            ],
            [
                2699,
                2705,
                "PERSON"
            ],
            [
                2793,
                2798,
                "PERSON"
            ],
            [
                2832,
                2840,
                "PERSON"
            ],
            [
                2858,
                2863,
                "PERSON"
            ],
            [
                2869,
                2878,
                "PERSON"
            ],
            [
                2900,
                2904,
                "PERSON"
            ],
            [
                2908,
                2915,
                "PERSON"
            ],
            [
                2959,
                2968,
                "PERSON"
            ],
            [
                3020,
                3026,
                "PERSON"
            ],
            [
                3120,
                3127,
                "PERSON"
            ],
            [
                3157,
                3162,
                "PERSON"
            ],
            [
                3182,
                3190,
                "PERSON"
            ],
            [
                3232,
                3238,
                "PERSON"
            ],
            [
                3256,
                3262,
                "PERSON"
            ],
            [
                3268,
                3276,
                "PERSON"
            ],
            [
                3284,
                3293,
                "PERSON"
            ],
            [
                3374,
                3378,
                "PERSON"
            ],
            [
                3472,
                3478,
                "PERSON"
            ],
            [
                3492,
                3501,
                "PERSON"
            ],
            [
                3585,
                3592,
                "PERSON"
            ],
            [
                3597,
                3603,
                "PERSON"
            ],
            [
                3607,
                3617,
                "PERSON"
            ],
            [
                3622,
                3631,
                "PERSON"
            ],
            [
                3640,
                3647,
                "PERSON"
            ],
            [
                3658,
                3662,
                "PERSON"
            ],
            [
                3709,
                3717,
                "PERSON"
            ],
            [
                3757,
                3761,
                "PERSON"
            ],
            [
                3780,
                3788,
                "PERSON"
            ],
            [
                3803,
                3813,
                "PERSON"
            ],
            [
                3828,
                3833,
                "PERSON"
            ],
            [
                3863,
                3871,
                "PERSON"
            ],
            [
                3876,
                3885,
                "PERSON"
            ],
            [
                3943,
                3950,
                "PERSON"
            ],
            [
                4006,
                4012,
                "PERSON"
            ],
            [
                4017,
                4025,
                "PERSON"
            ],
            [
                4074,
                4080,
                "PERSON"
            ],
            [
                4110,
                4116,
                "PERSON"
            ],
            [
                4142,
                4149,
                "PERSON"
            ],
            [
                4153,
                4160,
                "PERSON"
            ],
            [
                4172,
                4179,
                "PERSON"
            ],
            [
                4185,
                4191,
                "PERSON"
            ],
            [
                4199,
                4204,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult { Cannon  m { Freya edical { Kaycee  c { Hellen enter east  { Susannah white, tyro { Ami ne { Daquan  1211 medical center dr { Candi  mrn: 04771736 { Cecil 1, dob: 7/27 { Lyndsay /1969, legal sex: m nashville tn 37232 visit date: 9/ { Keyla 5/2023 09/05/2023 -  { Destini office visi { Sariah t in vanderbilt diab { Sammie etes and en { Shonda doc { Aric rinology  { Dwain (continued) letters (con { Elvia tinued) imm gran automa { Zayne ted 0. { Destiney 2 % absolut { Miller e imm gran automated  { Daron 0.01 { Odin  0.00 - 0.03 x10(3)/mcl patie { Danita nt location poc sodium handheld { Lenny   { Franco 136  { Star - 144 mmol/l potassium handheld 3.3 - 4.8 mmol/l chloride handhe { Delmer ld 98 - 107 mmol/l total co2 handhe { Renita ld 21 - 29 mmol/l c { Elida alcium ionized (ica) handheld 4.5 - { Zola  5.3 mg/ { Shyla dl glucose handheld 70 - 99 mg/dl blood urea nit { Corinna rog { Zaiden en (bun) handheld 8 - 26 mg/dl creatinine handheld  { Jacey 0.6 - 1.3 mg/dl an { Darrius ion gap handheld 10 - 20 mmol/l hematocrit handheld 41 49 % hemoglobin handheld 14.0 -  { Legend 18.1 gm/dl uri { Raheem ne { Drew   { Abdullah c { Kyrie ol { Velda or urine appe { Lashawn arance urine specif { Juliann ic gr { Alesia avity  { Jayde 1.015 - 1.025 urine ph { Laureen  5.0 - 6.5 urine { Maleah  g { Columbus lucose negative mg/dl urine protein negative mg/dl urin { Danelle e ketones negative r mg/dl urine bi { Hezekiah lirubin negati { Valorie ve urine urobilinogen <2 mg/dl urine leuk { Kayden ocyte esterase printe { Korbin d  { Ginny on 10/3/24 7:13 am page 1833,vumc adult  { Charleen medical center east { Nicky  white, tyr { Kayleen one  { Emmie 1211 medical center dr mrn: { Karley  0477 { Dandre 17361, dob: 7 { Niki /27/1969, legal sex: m nashvi { Kalyn lle tn 3723 { Jerold 2 visit date: 9/5/2 { Anthony 023 09/05/2023 - offi { Rivka ce visit in  { Halie vande { Jerrell rbilt diabet { Marilee es and endocrinology (con { Soren tinued) letters (continued) ne { Callum g { Olen ativ { Cletus e urine nitrite negative u { Sky rine blood negative sod { Dania ium level 138 136 - 145 mmol/l potassium level 4.4 3.3 - 4.8 { Genaro  mmol/l  { Jedidiah chloride level 108 (h) 98 - 107 mmol/l carbon di { Pamala oxide 20 (l) 22 - 29 { Imelda  mmol/l glucos { Yessenia e level 143 (h) 7 { Raiden 0 - 99 mg/dl  { Blaise blo { Roseanne od urea nitrogen 42 (h) 8 - 26 mg/d { Collette l creatinine level 4.31 (h) 0.72 - 1.25 - mg/dl calcium l { Hamza evel total 8.8 8.4 - 10.5 mg/dl a { Braiden nion gap 10 biliru { Anson bin direct 0.1 0.0 - 0.5 - mg/dl bilirubin total 0.3 0.2 - 1.2 mg/dl al { Emerald bumin level  { Elroy 3.6 3.5 - 5.2 gm/dl alkaline phosphatase 100 40 - 150 unit/l alan { Rikki ine aminotransferase 30 0 - - 55 unit/l aspartate aminotransfera { Peggie se 41 (h) 5 -  { Katerina 40 u { Dorris nit/l protein { Amaris  total 6.4 6.0 - 8.3 gm/dl abo type rh typ { Ashely e auto ab screen specimen expiration cho { Leyla lest { Sol erol total 0 { Kandice  - 199 m { Ellsworth g/dl triglyceride lvl <=149 mg/dl  { Salina printed on 10/3/24 7:13 am page 1834",
    {
        "entities": [
            [
                13,
                20,
                "PERSON"
            ],
            [
                25,
                31,
                "PERSON"
            ],
            [
                40,
                47,
                "PERSON"
            ],
            [
                52,
                59,
                "PERSON"
            ],
            [
                73,
                82,
                "PERSON"
            ],
            [
                96,
                100,
                "PERSON"
            ],
            [
                105,
                112,
                "PERSON"
            ],
            [
                138,
                144,
                "PERSON"
            ],
            [
                161,
                167,
                "PERSON"
            ],
            [
                182,
                190,
                "PERSON"
            ],
            [
                246,
                252,
                "PERSON"
            ],
            [
                275,
                283,
                "PERSON"
            ],
            [
                297,
                304,
                "PERSON"
            ],
            [
                327,
                334,
                "PERSON"
            ],
            [
                348,
                355,
                "PERSON"
            ],
            [
                361,
                366,
                "PERSON"
            ],
            [
                378,
                384,
                "PERSON"
            ],
            [
                411,
                417,
                "PERSON"
            ],
            [
                443,
                449,
                "PERSON"
            ],
            [
                458,
                467,
                "PERSON"
            ],
            [
                481,
                488,
                "PERSON"
            ],
            [
                512,
                518,
                "PERSON"
            ],
            [
                525,
                530,
                "PERSON"
            ],
            [
                562,
                569,
                "PERSON"
            ],
            [
                603,
                609,
                "PERSON"
            ],
            [
                613,
                620,
                "PERSON"
            ],
            [
                627,
                632,
                "PERSON"
            ],
            [
                699,
                706,
                "PERSON"
            ],
            [
                744,
                751,
                "PERSON"
            ],
            [
                773,
                779,
                "PERSON"
            ],
            [
                817,
                822,
                "PERSON"
            ],
            [
                833,
                839,
                "PERSON"
            ],
            [
                890,
                898,
                "PERSON"
            ],
            [
                904,
                911,
                "PERSON"
            ],
            [
                965,
                971,
                "PERSON"
            ],
            [
                992,
                1000,
                "PERSON"
            ],
            [
                1090,
                1097,
                "PERSON"
            ],
            [
                1114,
                1121,
                "PERSON"
            ],
            [
                1126,
                1131,
                "PERSON"
            ],
            [
                1135,
                1144,
                "PERSON"
            ],
            [
                1148,
                1154,
                "PERSON"
            ],
            [
                1159,
                1165,
                "PERSON"
            ],
            [
                1181,
                1189,
                "PERSON"
            ],
            [
                1211,
                1219,
                "PERSON"
            ],
            [
                1227,
                1234,
                "PERSON"
            ],
            [
                1243,
                1249,
                "PERSON"
            ],
            [
                1274,
                1282,
                "PERSON"
            ],
            [
                1301,
                1308,
                "PERSON"
            ],
            [
                1313,
                1322,
                "PERSON"
            ],
            [
                1380,
                1388,
                "PERSON"
            ],
            [
                1426,
                1435,
                "PERSON"
            ],
            [
                1452,
                1460,
                "PERSON"
            ],
            [
                1504,
                1511,
                "PERSON"
            ],
            [
                1535,
                1542,
                "PERSON"
            ],
            [
                1547,
                1553,
                "PERSON"
            ],
            [
                1596,
                1605,
                "PERSON"
            ],
            [
                1627,
                1633,
                "PERSON"
            ],
            [
                1647,
                1655,
                "PERSON"
            ],
            [
                1662,
                1668,
                "PERSON"
            ],
            [
                1698,
                1705,
                "PERSON"
            ],
            [
                1713,
                1720,
                "PERSON"
            ],
            [
                1736,
                1741,
                "PERSON"
            ],
            [
                1773,
                1779,
                "PERSON"
            ],
            [
                1793,
                1800,
                "PERSON"
            ],
            [
                1822,
                1830,
                "PERSON"
            ],
            [
                1854,
                1860,
                "PERSON"
            ],
            [
                1875,
                1881,
                "PERSON"
            ],
            [
                1889,
                1897,
                "PERSON"
            ],
            [
                1912,
                1920,
                "PERSON"
            ],
            [
                1948,
                1954,
                "PERSON"
            ],
            [
                1987,
                1994,
                "PERSON"
            ],
            [
                1998,
                2003,
                "PERSON"
            ],
            [
                2010,
                2017,
                "PERSON"
            ],
            [
                2046,
                2050,
                "PERSON"
            ],
            [
                2076,
                2082,
                "PERSON"
            ],
            [
                2145,
                2152,
                "PERSON"
            ],
            [
                2163,
                2172,
                "PERSON"
            ],
            [
                2223,
                2230,
                "PERSON"
            ],
            [
                2253,
                2260,
                "PERSON"
            ],
            [
                2277,
                2286,
                "PERSON"
            ],
            [
                2306,
                2313,
                "PERSON"
            ],
            [
                2329,
                2336,
                "PERSON"
            ],
            [
                2342,
                2351,
                "PERSON"
            ],
            [
                2389,
                2398,
                "PERSON"
            ],
            [
                2458,
                2464,
                "PERSON"
            ],
            [
                2500,
                2508,
                "PERSON"
            ],
            [
                2529,
                2535,
                "PERSON"
            ],
            [
                2609,
                2617,
                "PERSON"
            ],
            [
                2632,
                2638,
                "PERSON"
            ],
            [
                2706,
                2712,
                "PERSON"
            ],
            [
                2779,
                2786,
                "PERSON"
            ],
            [
                2803,
                2812,
                "PERSON"
            ],
            [
                2819,
                2826,
                "PERSON"
            ],
            [
                2842,
                2849,
                "PERSON"
            ],
            [
                2894,
                2901,
                "PERSON"
            ],
            [
                2944,
                2950,
                "PERSON"
            ],
            [
                2957,
                2961,
                "PERSON"
            ],
            [
                2976,
                2984,
                "PERSON"
            ],
            [
                2995,
                3005,
                "PERSON"
            ],
            [
                3042,
                3049,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical c { Leandro enter dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashvi { Vernice lle tn 37232-0004 adm: 6/2/2023, d/c: 6/3/2023  { Taliyah 06/02/2023 { Arden  - ed to hosp-admission (discharged) in vanderbil { Tamela t univers { Jamil ity adult hospital (continued) ed care timeline (co { Mathilda ntinued) 14:03:44 hi { Kinsey story reviewed sect { Cristobal ions reviewed: to { Charli bacco greenwood- simpson, { Lyn  el { Dee izabeth, rn 14:03:45 history reviewed sections reviewed: drug use greenwood- { Tosha   { Hettie simpson, elizabeth, rn 14:03:50 history reviewed  { Berta section { Bowen s reviewed:  { Anais medical, surgical, alcohol, { Yaritza  tobacco, drug use, sexual { Stefani  greenwood- activity, social documentation, family simpson, elizabeth, rn 14:03:55 pod destinat { Eryn ion bickett, selected christopher ryan, md 14:04 height and weight height and weight greenwood- height: 182.9 cm (72\") simpson, heigh { Tory t method: st { Elia ated elizabeth, rn weight: 86.2 kg (190 lb) weight method: stated 14:04 domestic vio { Xzavier lence domestic violence screening - as { Kristian k patient the following greenwood- screening in { Kasen  the past year, { Amiya  have yo { Nikole u been physically hurt by someone (hit, slapped, si { Krysta mpson, kicked, or had pressure put on your neck, throat, or other part of y { Rory our body elizabeth, rn making it har { Dewitt d { Bess  for you to  { Gracelynn br { Kassie eathe)?:  you in a relationship with a pers { Burl on who threatens or { Nicola  physically hurts you { Mariann ?:  anyon { Miya e forced you to have sexual activities that made you feel un { Flor comfortable?:  you feel unsafe at home? { Nedra : :04 lund- { Dorcas browder volume estimates greenwood- (adult) fluid re { Daleyza suscitation (#6 { Donnie ):  { Alpha 0 { Shelli  simpson, fluid resuscitation ( { Vernell #5): 0 elizabeth, rn fluid resuscitation (#7): 0 fluid resuscitation (#8): 0 fluid resuscitation (#9):  { Kylan 0 fluid resuscitation (#10): 0 printed on 10/3 { Seamus /24 7:13 am page 2151,vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nash { Elyssa ville { Michell  tn 37232-0004 a { Tressa dm: 6/2/2023, d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt university adult hospital (continued) ed care timeline (continued) 14:04 custom formula measurements g { Ariah reenwood- data tot { Danae al weight change p { Alysa ercent: 2222 percent simpson, weight change since preop: 86.16 { Tad  kgs elizabeth, rn initia { Stormy l excess weight: -80.74 kgs percent of ibw: 3765.17 percent ebw ( { Maryjo kg): 3037.71 kg { Tristian  ebw (lbs): 3028.88 lbs weight change since preop: 86.16 kg initial excess weight: -80.74 kg perce { Leandra nt of ibw: 106.74 percent ebw (kg): 5.42 k { Eliot g ebw (lb): 12 lb volume estimates fluid resuscitation (#4): 0 basic info { Aurelio rmation approx predicted basal  { Rogers (0.5 * wt in kg): 43.1 rec star { Tonja t basal (.25 { Twyla  * wt in k { Selene g): 21.5 rec start fi { Ike xed meal (.08 * wt in kg): 7.2 vital signs bmi (calculated): 25.8 height and weight weight in (lb) to have bmi = 25: 183.9 fixed meal insulin dosing approx predicted correction (1500 / wt in kg): 17.4 ratio-based meal dosing  { Case approx predicted  { Unique ratio (500 / wt in kg): 5.8 rec start sliding scale (3000 { Rubin  / wt in kg): 34.8 height an { Tegan d weight weight in  { Gene (kg) to have bmi = 25: 8 { Bryn 3.6 weight and growth recommendation ibw/kg (calculated) fema { Eboni le:  { Connie 73.1 kg adult ibw/vt calcul { Cory at { Leonora ion { Darion s ibw/kg (calculated) : 77.6 low { Aletha  r { Sherrill ange vt 6ml/kg : 465.6 ml/kg adu { Ruthann lt moderate rang { Carmelita e vt 8ml/kg  { Kaiya : 620.8 ml/kg adult high range vt 10ml/k { Johnson g: 776 ml/kg other flowsheet entries bmi (calculated): 25.8 percent excess weight loss: -106.71 percent ibw in lbs (bariatric): 178 weight  { Reagan change since { Lani  last visit: 86 { Remy .16 kg ibw in kg (bariatric): 80.74 bmi (calculated): 25.8 bsa (calculated - sq m): 2.09 sq meters mosteller: 2.092 dubois: 2.084 haycock: 2.102 gehan & george: 2.102 bmi  { Lonnie (calculated): 25.8 ibw (lb): 178 { Meadow  weight ch { Rubye ange from p { Mimi reop: 86.16 kg weight { Jorden  change since last visit: 86.16 kg { Nelly  weight change from preop (kg): 86.16 ibw in lbs (bariatric): 184.33 percent weight change since preop: 86.16 lbs percent weight cha { Marcelino nge since last  { Latosha visit: 3041 percent current ebw: 5.6 { Chiquita 7 lb current bmi (calculat { Arlen ed): 25.8 percent weight chang { Amirah e since la { Juniper st visit: 303900 per { Socorro cent mifflin-s { Dereck t jeor rmr (kcal): 1744.83 weight change since last visit (%): 0.53 percent bmi last visit (calculated): 25.6 bmi change since last visit: 0.2 bmi change since last visit (percent): 0.78 printed on 1 { Monserrat 0/3/24 7:13 am page 2152",
    {
        "entities": [
            [
                51,
                59,
                "PERSON"
            ],
            [
                123,
                131,
                "PERSON"
            ],
            [
                181,
                189,
                "PERSON"
            ],
            [
                202,
                208,
                "PERSON"
            ],
            [
                260,
                267,
                "PERSON"
            ],
            [
                279,
                285,
                "PERSON"
            ],
            [
                339,
                348,
                "PERSON"
            ],
            [
                371,
                378,
                "PERSON"
            ],
            [
                400,
                410,
                "PERSON"
            ],
            [
                430,
                437,
                "PERSON"
            ],
            [
                465,
                469,
                "PERSON"
            ],
            [
                475,
                479,
                "PERSON"
            ],
            [
                558,
                564,
                "PERSON"
            ],
            [
                568,
                575,
                "PERSON"
            ],
            [
                627,
                633,
                "PERSON"
            ],
            [
                643,
                649,
                "PERSON"
            ],
            [
                664,
                670,
                "PERSON"
            ],
            [
                700,
                708,
                "PERSON"
            ],
            [
                737,
                745,
                "PERSON"
            ],
            [
                843,
                848,
                "PERSON"
            ],
            [
                984,
                989,
                "PERSON"
            ],
            [
                1004,
                1009,
                "PERSON"
            ],
            [
                1096,
                1104,
                "PERSON"
            ],
            [
                1145,
                1154,
                "PERSON"
            ],
            [
                1204,
                1210,
                "PERSON"
            ],
            [
                1228,
                1234,
                "PERSON"
            ],
            [
                1245,
                1252,
                "PERSON"
            ],
            [
                1306,
                1313,
                "PERSON"
            ],
            [
                1391,
                1396,
                "PERSON"
            ],
            [
                1435,
                1442,
                "PERSON"
            ],
            [
                1446,
                1451,
                "PERSON"
            ],
            [
                1466,
                1476,
                "PERSON"
            ],
            [
                1481,
                1488,
                "PERSON"
            ],
            [
                1534,
                1539,
                "PERSON"
            ],
            [
                1561,
                1568,
                "PERSON"
            ],
            [
                1592,
                1600,
                "PERSON"
            ],
            [
                1612,
                1617,
                "PERSON"
            ],
            [
                1680,
                1685,
                "PERSON"
            ],
            [
                1727,
                1733,
                "PERSON"
            ],
            [
                1747,
                1754,
                "PERSON"
            ],
            [
                1809,
                1817,
                "PERSON"
            ],
            [
                1835,
                1842,
                "PERSON"
            ],
            [
                1848,
                1854,
                "PERSON"
            ],
            [
                1858,
                1865,
                "PERSON"
            ],
            [
                1899,
                1907,
                "PERSON"
            ],
            [
                2013,
                2019,
                "PERSON"
            ],
            [
                2068,
                2075,
                "PERSON"
            ],
            [
                2207,
                2214,
                "PERSON"
            ],
            [
                2222,
                2230,
                "PERSON"
            ],
            [
                2249,
                2256,
                "PERSON"
            ],
            [
                2450,
                2456,
                "PERSON"
            ],
            [
                2477,
                2483,
                "PERSON"
            ],
            [
                2504,
                2510,
                "PERSON"
            ],
            [
                2575,
                2579,
                "PERSON"
            ],
            [
                2607,
                2614,
                "PERSON"
            ],
            [
                2682,
                2689,
                "PERSON"
            ],
            [
                2707,
                2716,
                "PERSON"
            ],
            [
                2817,
                2825,
                "PERSON"
            ],
            [
                2870,
                2876,
                "PERSON"
            ],
            [
                2952,
                2960,
                "PERSON"
            ],
            [
                2994,
                3001,
                "PERSON"
            ],
            [
                3035,
                3041,
                "PERSON"
            ],
            [
                3056,
                3062,
                "PERSON"
            ],
            [
                3075,
                3082,
                "PERSON"
            ],
            [
                3106,
                3110,
                "PERSON"
            ],
            [
                3338,
                3343,
                "PERSON"
            ],
            [
                3363,
                3370,
                "PERSON"
            ],
            [
                3430,
                3436,
                "PERSON"
            ],
            [
                3467,
                3473,
                "PERSON"
            ],
            [
                3495,
                3500,
                "PERSON"
            ],
            [
                3527,
                3532,
                "PERSON"
            ],
            [
                3596,
                3602,
                "PERSON"
            ],
            [
                3609,
                3616,
                "PERSON"
            ],
            [
                3646,
                3651,
                "PERSON"
            ],
            [
                3656,
                3664,
                "PERSON"
            ],
            [
                3670,
                3677,
                "PERSON"
            ],
            [
                3712,
                3719,
                "PERSON"
            ],
            [
                3724,
                3733,
                "PERSON"
            ],
            [
                3768,
                3776,
                "PERSON"
            ],
            [
                3795,
                3805,
                "PERSON"
            ],
            [
                3820,
                3826,
                "PERSON"
            ],
            [
                3869,
                3877,
                "PERSON"
            ],
            [
                4019,
                4026,
                "PERSON"
            ],
            [
                4041,
                4046,
                "PERSON"
            ],
            [
                4064,
                4069,
                "PERSON"
            ],
            [
                4243,
                4250,
                "PERSON"
            ],
            [
                4285,
                4292,
                "PERSON"
            ],
            [
                4305,
                4311,
                "PERSON"
            ],
            [
                4325,
                4330,
                "PERSON"
            ],
            [
                4354,
                4361,
                "PERSON"
            ],
            [
                4398,
                4404,
                "PERSON"
            ],
            [
                4539,
                4549,
                "PERSON"
            ],
            [
                4567,
                4575,
                "PERSON"
            ],
            [
                4614,
                4623,
                "PERSON"
            ],
            [
                4652,
                4658,
                "PERSON"
            ],
            [
                4691,
                4698,
                "PERSON"
            ],
            [
                4711,
                4719,
                "PERSON"
            ],
            [
                4742,
                4750,
                "PERSON"
            ],
            [
                4767,
                4774,
                "PERSON"
            ],
            [
                4976,
                4986,
                "PERSON"
            ]
        ]
    }
),(
    "vumc hendersonville - anderson white, tyrone 12 { Bristol 8 n a { Jaunita nder { Evelin son ln mrn: 047717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 visit date: 3/13/202 { Britany 3 03/13/2023 - refill in vanderbilt primary care hendersonville facesheet report patient demographics patient name mrn legal dob addres { Carson s phone white, tyrone 0477173 sex 7/27/1969 apt 705 615-260-2291 (home) 61 m 1101 edgeh { Huey ill ave 615-260-2 { Gabriel 291 (mobile) nashville tn 37203 *preferred* hospital account not on { Kisha  file admission i { Edwardo nformation current information attendi { Priya ng provider admitting p { Lucero rovider admission type admiss { Kayli ion status unknown status admission date/time discharge date/time hospit { Lavern al service auth/cert status hospital area unit room/bed referring provider  { Kayson 03/13/2023 - refill in vanderbil { Garnet t primary care hendersonville (con { Kamari tinued) reason for vis { Giuseppe it chief complaint m { Carleen ed refill visit information nursing assessment  assessment  { Jacky available for  { Hilton this encounter. communication tracking calls/me { Jamey ssages interface (incoming) on 3/13/2 { Jaylee 023 1412 caller name: vanderbilt university phone num { Laquita ber: 615-322-2688 100 oaks - nashville, tn - 719 thompson ln medication list medic { Ford ation list i this report is for documentation purp { Julieta oses only. the patient should not fol { Shaylee low medication instructions { Allyssa  w { Dortha ithin. for accurate instructions regarding medications, the patient should in { Kennedy stead consult their physician or after visit summary. active at the { Keila  end of visit medications last reviewed by { Jamarion  lippard, giles { Marnie  a, aprn on 3/6/2023 151 { Calista 6 carve { Ima dilol 6.25  { Ephraim mg tablet (coreg) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 instructions: { Frida  take 1 tablet (6.25 mg total) by mouth  { Lars daily. authorized by: lippard, giles a, aprn ord { Shay ered on: 10/2 { Murphy 5/2022 printed on 10/3/24 7:13 am page 2517,vumc hendersonville - anderson white, tyrone 128 n anderson ln mrn: 047717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 visit date: 3/13/2023 0 { Dillan 3/13/2023 - refill in vanderbilt primary care hendersonville (continued) medication list (continued) start date: 10/25/2022 end date: { Noelia  6/3/2023 quantity: 90 tablet refill: 1 refill by  { Shameka 10/25/2023 atorvastatin 80 mg tablet (lipitor) discontinued by: lippard, giles a, aprn discontinued on: 8/8/2023 reason for discontinuation: reorder instructions: take 1 tablet (80 mg total) b { Richie y mouth daily. authorize { Leighton d by: li { Marlyn ppard, giles a, aprn ordered on: 10/25/2022  { Sincere start date: { Paislee  10/25/2022 quantity: 90 tablet refill: 3 refills by 1 { Tristan 0/25/2023 cetirizine 10 mg table { Eliezer t (zyrtec) discontinued  { Alexandro by: lippard, giles a, aprn disc { Tawana ontinued on: 8/8/202 { Khadijah 3 reason for di { Nathaly scontinuation: reorder instructions: take { Jaxton  1 tablet (10 mg total) by mouth once a day as needed for allergies. authorized by: lippard, giles a, aprn ordered on: 10/25/2022 start date:  { Mariel 10/25/2022 quantity: 30 tablet refill: 11 refills by 10/25/2023 docusate sodium 100 mg c { Lucie apsule (colace) discontinued by: de witte, anton jordan, md d { Samira iscontinued o { Ivana n: 12/6/2023 reason for discontinuation: cleanup(not { Joleen avs) instructions { Taya : take one  { Branson tablet  { Channing tid prn constipation authorized b { Bodhi y: lippard, giles a, aprn or { Janeen dered on: 10/25/2022 start date { Wilber : 10/25/2022 end date: 12/6/2023 quantity: 60 capsule ref { Joey ill { Bettina :  remaining fluticasone p { Gerry ropionate 50 m { Melvina cg/actuation nasal spray,suspension (flonase) discontinued by { Alannah : lippard, giles a, aprn discont { Kaylen inued on: 6/6/2023 reason for discontinuat { Darci ion: reorder instructions: administer 2 sprays into each nostril daily. authorized by: lippard, giles a, aprn ordered on:  { Marlo 10/25/2022 sta { Stanton rt date: 10/25/2022 quantity: 16 g refill: 11 refills by 10/25/2023 azelastine 137 mcg (0.1 %) nasal spray aerosol (astelin) discontinued by: lippard, gil { Ivanna es a, aprn discontinued on: 8/8/2023 reason for discontinuation: reorder i { Eden nstructions: administer 1 spray into each nostril 2 times a day. use in each nostril as directe { Brittni d autho { Jessi rized by:  { Vikki lippard, giles a, aprn ordered { Milan  on:  { Lilianna 10/25/2022 start date: 10/25/2022 qu { Terese antity: 30 ml refill: 12 refills by 10/25/ { Arlie 2023 lidoca { Kandace ine 5 % topical patch (lidoderm) discontinued by: leh { Sharla mann, melissa cary, pa-c di { Floy scontinued on: 6/3 { Felton /2023 reason for discontinuation: stop taking at discharge (cancelrx) instructions: apply 1 patch topically daily. { Khalid  apply to p { Joycelyn ainful area 12 hours per day, remove  { Vanesa for 12 hours. authorized by: lippard, giles a, aprn ordered on: 10/ { Harlow 25/2 { Lazaro 022 start date: 10/25/2022 end date: 6/3 { Teressa /2 { Burt 023 action: patient not taking quantity: 30 p { Giovani atch refill: { Karmen  11 refills by 10 { Rosita /25/2023 albutero { Elianna l sulfate hfa 90 mcg/actuation aerosol inhaler discontinued by: lippard, giles  { Donn a, aprn discontinued on: 8/8/2023 reason for discontinuation: reorder instructions: inhale 2 puffs every 4 hours as needed for wheezing. authorized by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quan { Annabell tity: 18 g refill: 11 refills by 10/25/2023 pantoprazole 20 mg tablet,delayed release (protonix) printed on 10/3/24 7:13 am page 2518",
    {
        "entities": [
            [
                50,
                58,
                "PERSON"
            ],
            [
                66,
                74,
                "PERSON"
            ],
            [
                81,
                88,
                "PERSON"
            ],
            [
                187,
                195,
                "PERSON"
            ],
            [
                333,
                340,
                "PERSON"
            ],
            [
                430,
                435,
                "PERSON"
            ],
            [
                455,
                463,
                "PERSON"
            ],
            [
                533,
                539,
                "PERSON"
            ],
            [
                559,
                567,
                "PERSON"
            ],
            [
                608,
                614,
                "PERSON"
            ],
            [
                640,
                647,
                "PERSON"
            ],
            [
                679,
                685,
                "PERSON"
            ],
            [
                760,
                767,
                "PERSON"
            ],
            [
                845,
                852,
                "PERSON"
            ],
            [
                887,
                894,
                "PERSON"
            ],
            [
                931,
                938,
                "PERSON"
            ],
            [
                963,
                972,
                "PERSON"
            ],
            [
                995,
                1003,
                "PERSON"
            ],
            [
                1065,
                1071,
                "PERSON"
            ],
            [
                1088,
                1095,
                "PERSON"
            ],
            [
                1145,
                1151,
                "PERSON"
            ],
            [
                1191,
                1198,
                "PERSON"
            ],
            [
                1254,
                1262,
                "PERSON"
            ],
            [
                1347,
                1352,
                "PERSON"
            ],
            [
                1405,
                1413,
                "PERSON"
            ],
            [
                1453,
                1461,
                "PERSON"
            ],
            [
                1491,
                1499,
                "PERSON"
            ],
            [
                1504,
                1511,
                "PERSON"
            ],
            [
                1591,
                1599,
                "PERSON"
            ],
            [
                1669,
                1675,
                "PERSON"
            ],
            [
                1720,
                1729,
                "PERSON"
            ],
            [
                1747,
                1754,
                "PERSON"
            ],
            [
                1781,
                1789,
                "PERSON"
            ],
            [
                1799,
                1803,
                "PERSON"
            ],
            [
                1817,
                1825,
                "PERSON"
            ],
            [
                1930,
                1936,
                "PERSON"
            ],
            [
                1979,
                1984,
                "PERSON"
            ],
            [
                2035,
                2040,
                "PERSON"
            ],
            [
                2056,
                2063,
                "PERSON"
            ],
            [
                2265,
                2272,
                "PERSON"
            ],
            [
                2408,
                2415,
                "PERSON"
            ],
            [
                2468,
                2476,
                "PERSON"
            ],
            [
                2671,
                2678,
                "PERSON"
            ],
            [
                2705,
                2714,
                "PERSON"
            ],
            [
                2725,
                2732,
                "PERSON"
            ],
            [
                2779,
                2787,
                "PERSON"
            ],
            [
                2801,
                2809,
                "PERSON"
            ],
            [
                2866,
                2874,
                "PERSON"
            ],
            [
                2909,
                2917,
                "PERSON"
            ],
            [
                2944,
                2954,
                "PERSON"
            ],
            [
                2988,
                2995,
                "PERSON"
            ],
            [
                3018,
                3027,
                "PERSON"
            ],
            [
                3045,
                3053,
                "PERSON"
            ],
            [
                3097,
                3104,
                "PERSON"
            ],
            [
                3249,
                3256,
                "PERSON"
            ],
            [
                3347,
                3353,
                "PERSON"
            ],
            [
                3417,
                3424,
                "PERSON"
            ],
            [
                3440,
                3446,
                "PERSON"
            ],
            [
                3501,
                3508,
                "PERSON"
            ],
            [
                3528,
                3533,
                "PERSON"
            ],
            [
                3547,
                3555,
                "PERSON"
            ],
            [
                3565,
                3574,
                "PERSON"
            ],
            [
                3610,
                3616,
                "PERSON"
            ],
            [
                3647,
                3654,
                "PERSON"
            ],
            [
                3688,
                3695,
                "PERSON"
            ],
            [
                3755,
                3760,
                "PERSON"
            ],
            [
                3766,
                3774,
                "PERSON"
            ],
            [
                3803,
                3809,
                "PERSON"
            ],
            [
                3826,
                3834,
                "PERSON"
            ],
            [
                3898,
                3906,
                "PERSON"
            ],
            [
                3941,
                3948,
                "PERSON"
            ],
            [
                3993,
                3999,
                "PERSON"
            ],
            [
                4124,
                4130,
                "PERSON"
            ],
            [
                4147,
                4155,
                "PERSON"
            ],
            [
                4312,
                4319,
                "PERSON"
            ],
            [
                4396,
                4401,
                "PERSON"
            ],
            [
                4499,
                4507,
                "PERSON"
            ],
            [
                4517,
                4523,
                "PERSON"
            ],
            [
                4536,
                4542,
                "PERSON"
            ],
            [
                4575,
                4581,
                "PERSON"
            ],
            [
                4589,
                4598,
                "PERSON"
            ],
            [
                4637,
                4644,
                "PERSON"
            ],
            [
                4689,
                4695,
                "PERSON"
            ],
            [
                4709,
                4717,
                "PERSON"
            ],
            [
                4773,
                4780,
                "PERSON"
            ],
            [
                4810,
                4815,
                "PERSON"
            ],
            [
                4836,
                4843,
                "PERSON"
            ],
            [
                4960,
                4967,
                "PERSON"
            ],
            [
                4981,
                4990,
                "PERSON"
            ],
            [
                5030,
                5037,
                "PERSON"
            ],
            [
                5107,
                5114,
                "PERSON"
            ],
            [
                5121,
                5128,
                "PERSON"
            ],
            [
                5171,
                5179,
                "PERSON"
            ],
            [
                5184,
                5189,
                "PERSON"
            ],
            [
                5237,
                5245,
                "PERSON"
            ],
            [
                5260,
                5267,
                "PERSON"
            ],
            [
                5287,
                5294,
                "PERSON"
            ],
            [
                5314,
                5322,
                "PERSON"
            ],
            [
                5404,
                5409,
                "PERSON"
            ],
            [
                5637,
                5646,
                "PERSON"
            ]
        ]
    }
),(
    "vumc a { Emmalyn dult { Lavina  hospital white, tyrone 1211 medic { Valery al center dr. mrn: 047 { Vivien 717361, dob: 7/27/ { Nanci 1969, legal sex: m nashville tn 37 { Travon 232-0004 visit { Annalee  dat { Theda e { Aryan : 9/15 { Alexzander /2023  { Antwon 09/15/2023 - patient message i { Kamron n vumc patient access servi { Bonny ces vir (continue { Lacy d) mess { Natalya ages ( { Zain continu { Aryana ed) we look forwa { Haywood rd to seeing you soon. printed on  { Idella 10/3/24 7:13 am page 1663,vumc adult  { Adina one hundred oaks white, tyrone 719 thompson { Kaela  lane, na { Devyn shville { Demario  mrn: 047717361, dob: 7/27/1969, legal s { Harmon ex: m n { Shante ashville tn 37204 visit date { Lue : 9/14/ { Lonny 2023 09/14/2023 - commu { Mickey nication in vanderbilt one hundred oaks  { Trever primary care north facesheet report patient demographics patient name mrn legal dob address pho { Denisse ne white, tyrone 04 { Eleanora 77173 { Toney   { Aleta sex 7/27/1969 apt 705 6 { Litzy 15-260- { Draven 2291 (home) 61 m 1101 edgehill ave 615-260-2291 (mobile) nashville tn 372 { Lexus 03 *preferred* hospital account not  { Blaze on file admission information current { Karma  informat { Johana i { Sarahi on attending provider admitting provider ad { Otha mission type admission status unknown status ad { Eloisa mission date/time  { Jazlynn discharge d { Anders ate/ti { Adriane me  { Blanch hospital { Oakley   { Juli service auth/cert status hospital area unit room/bed referring provider 09/14/2023  { Akeem - communic { Kai ation in  { Leigh vanderbi { Artie lt one hundred oaks primary care north (continued) visit information provider information encounter provider ford, keynin ro { Carolynn eshelle, rn departme { Fritz nt name address phone vanderbilt one hundred  { Caryl oaks primary 719 thompson ln 61 { Baron 5-936-2187 care north suite  { Jerod 20400 nashvill { Maximo e tn 37204  { Estefania medication list medication list i th { Felisha i { Ardis s repor { Joan t is for document { Whitney ation purposes only. the p { Violeta atient should not follow  { Rashawn medicatio { Galilea n instructions wit { Leigha hin. for  { Nicki accur { Nila ate instructions reg { Addilyn arding medications, the  { Maxim patient should instea { Akira d consult their phys { Kelley ician or after visit summary. active at the e { Buck nd of { Jackeline  visit medications l { Jessika ast reviewed  { Alba by dekorte, davita on { Saniya  9/7/2023 1424 docusate sod { Adria ium 1 { Baby 00 { Cedrick  mg capsule (colace) discontinue { Arnulfo d by: de witte, anton jordan, md disc { Darron ontinued on: 12/6/2023 reas { Nannette on for disconti { Jakayla nuation: cleanup(notavs)  { Darian instructions: take one { Keshia  tablet tid prn constipation authorized by: lippard, giles a, { Valentine  aprn o { Hortense rdered on: 10/25/2022 { Zahra  start d { Keeley ate: 10/ { Chanda 25/2022 end date: 12/6/2023 quantity:  { Mittie 60 capsule refill:  remaining pantoprazole 20 mg tab { Carlotta let,del { Maisie ayed release (proton { Shaniya ix) discontinued by: greens { Leander pan, debra  { Azalea l, ap { Lavon rn discontinued on: 7/9 { Karon /2024 reason for discontinuation: reorder instructions: take 1 tablet (20 mg total) by mouth daily. printed on 10/3/24 7:13 am page 1664",
    {
        "entities": [
            [
                9,
                17,
                "PERSON"
            ],
            [
                24,
                31,
                "PERSON"
            ],
            [
                68,
                75,
                "PERSON"
            ],
            [
                100,
                107,
                "PERSON"
            ],
            [
                128,
                134,
                "PERSON"
            ],
            [
                171,
                178,
                "PERSON"
            ],
            [
                195,
                203,
                "PERSON"
            ],
            [
                210,
                216,
                "PERSON"
            ],
            [
                220,
                226,
                "PERSON"
            ],
            [
                235,
                246,
                "PERSON"
            ],
            [
                255,
                262,
                "PERSON"
            ],
            [
                295,
                302,
                "PERSON"
            ],
            [
                332,
                338,
                "PERSON"
            ],
            [
                358,
                363,
                "PERSON"
            ],
            [
                373,
                381,
                "PERSON"
            ],
            [
                390,
                395,
                "PERSON"
            ],
            [
                405,
                412,
                "PERSON"
            ],
            [
                432,
                440,
                "PERSON"
            ],
            [
                477,
                484,
                "PERSON"
            ],
            [
                524,
                530,
                "PERSON"
            ],
            [
                576,
                582,
                "PERSON"
            ],
            [
                594,
                600,
                "PERSON"
            ],
            [
                610,
                618,
                "PERSON"
            ],
            [
                661,
                668,
                "PERSON"
            ],
            [
                678,
                685,
                "PERSON"
            ],
            [
                716,
                720,
                "PERSON"
            ],
            [
                730,
                736,
                "PERSON"
            ],
            [
                762,
                769,
                "PERSON"
            ],
            [
                812,
                819,
                "PERSON"
            ],
            [
                917,
                925,
                "PERSON"
            ],
            [
                947,
                956,
                "PERSON"
            ],
            [
                964,
                970,
                "PERSON"
            ],
            [
                974,
                980,
                "PERSON"
            ],
            [
                1006,
                1012,
                "PERSON"
            ],
            [
                1022,
                1029,
                "PERSON"
            ],
            [
                1105,
                1111,
                "PERSON"
            ],
            [
                1150,
                1156,
                "PERSON"
            ],
            [
                1196,
                1202,
                "PERSON"
            ],
            [
                1214,
                1221,
                "PERSON"
            ],
            [
                1225,
                1232,
                "PERSON"
            ],
            [
                1278,
                1283,
                "PERSON"
            ],
            [
                1333,
                1340,
                "PERSON"
            ],
            [
                1361,
                1369,
                "PERSON"
            ],
            [
                1383,
                1390,
                "PERSON"
            ],
            [
                1399,
                1407,
                "PERSON"
            ],
            [
                1413,
                1420,
                "PERSON"
            ],
            [
                1431,
                1438,
                "PERSON"
            ],
            [
                1442,
                1447,
                "PERSON"
            ],
            [
                1533,
                1539,
                "PERSON"
            ],
            [
                1552,
                1556,
                "PERSON"
            ],
            [
                1568,
                1574,
                "PERSON"
            ],
            [
                1585,
                1591,
                "PERSON"
            ],
            [
                1718,
                1727,
                "PERSON"
            ],
            [
                1750,
                1756,
                "PERSON"
            ],
            [
                1804,
                1810,
                "PERSON"
            ],
            [
                1844,
                1850,
                "PERSON"
            ],
            [
                1881,
                1887,
                "PERSON"
            ],
            [
                1904,
                1911,
                "PERSON"
            ],
            [
                1925,
                1935,
                "PERSON"
            ],
            [
                1974,
                1982,
                "PERSON"
            ],
            [
                1986,
                1992,
                "PERSON"
            ],
            [
                2002,
                2007,
                "PERSON"
            ],
            [
                2027,
                2035,
                "PERSON"
            ],
            [
                2064,
                2072,
                "PERSON"
            ],
            [
                2100,
                2108,
                "PERSON"
            ],
            [
                2120,
                2128,
                "PERSON"
            ],
            [
                2149,
                2156,
                "PERSON"
            ],
            [
                2168,
                2174,
                "PERSON"
            ],
            [
                2182,
                2187,
                "PERSON"
            ],
            [
                2210,
                2218,
                "PERSON"
            ],
            [
                2245,
                2251,
                "PERSON"
            ],
            [
                2275,
                2281,
                "PERSON"
            ],
            [
                2304,
                2311,
                "PERSON"
            ],
            [
                2359,
                2364,
                "PERSON"
            ],
            [
                2372,
                2382,
                "PERSON"
            ],
            [
                2405,
                2413,
                "PERSON"
            ],
            [
                2429,
                2434,
                "PERSON"
            ],
            [
                2458,
                2465,
                "PERSON"
            ],
            [
                2495,
                2501,
                "PERSON"
            ],
            [
                2509,
                2514,
                "PERSON"
            ],
            [
                2519,
                2527,
                "PERSON"
            ],
            [
                2562,
                2570,
                "PERSON"
            ],
            [
                2610,
                2617,
                "PERSON"
            ],
            [
                2647,
                2656,
                "PERSON"
            ],
            [
                2674,
                2682,
                "PERSON"
            ],
            [
                2710,
                2717,
                "PERSON"
            ],
            [
                2742,
                2749,
                "PERSON"
            ],
            [
                2813,
                2823,
                "PERSON"
            ],
            [
                2833,
                2842,
                "PERSON"
            ],
            [
                2866,
                2872,
                "PERSON"
            ],
            [
                2883,
                2890,
                "PERSON"
            ],
            [
                2901,
                2908,
                "PERSON"
            ],
            [
                2949,
                2956,
                "PERSON"
            ],
            [
                3011,
                3020,
                "PERSON"
            ],
            [
                3030,
                3037,
                "PERSON"
            ],
            [
                3060,
                3068,
                "PERSON"
            ],
            [
                3098,
                3106,
                "PERSON"
            ],
            [
                3120,
                3127,
                "PERSON"
            ],
            [
                3135,
                3141,
                "PERSON"
            ],
            [
                3167,
                3173,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hundred oaks whit { Kelton e, tyrone 719 { Brigid  thompso { Nicolle n lane, nashville mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date: 9/ { Tayla 7/2023 09/07/202 { Maira 3 - office vi { Pearline si { Sharonda t in vanderbilt { Makena  one hund { Kaylah red oaks primary car { Dawna e north (continued) laboratory  { Amberly reports (continued) indications type 2 diabe { Mika tes mellitus with chronic { Andrea  kidney dis { Corrina eas { Charline e, with long-term current use of  { Ashli insulin, u { Hamilton nspecified ckd stage (cms/hcc) [e11.22, z79.4 (icd-10-cm)] p { Treasure rinted { Karis  on 10/3/24 7:13 am p { Dayanara age 1751,vu { Karsyn mc { Nevin  adult one  { Kenley hundred oaks wh { Shyann ite, ty { Jovani rone 719  { Meta thompson lane, { Princeton  nashvi { Adell lle mrn: { Verla  0 { Josette 47717361, dob: 7/2 { Leone 7/1969, l { Constantine egal sex: m nashv { Venessa ille tn 37204 { Kalvin   { Kraig visit date: 9/7 { Darcy /2023 09/07/2023 - office visit in van { Teena derbilt { Lindsay  one hun { Nathen dred oaks primary care n { Cinthia orth (con { Giovanny tinued) i { Cain maging re { Margarette ports ima { Braelynn ging lung cancer  { Maribeth screening { Britton  ( { Isai expire { Tariq d) e { Myrtis lectro { Breann ni { Emmet cally { Camren  signed b { Cassius y: de w { Sommer itte, anton jordan, md on 09/07/23 1506 status: expired ordering user:  { Baby d { Rhianna e w { Shanika itte, anto { Gaylord n jordan, md 09/07/23 1506 ordering provider: de witte, anton jordan, md a { Jagger ut { Russ hor { Matthew ized by: sullivan, kathleen po { Saniyah ll { Brandan ard, md { Semaj  ordering mode: standard frequency: routine 09 { Bryana /07/23 - { Ivonne  class: ancillary perfor { Kathryne m { Elian ed { Carrol   { Abdul quantity: { Stephenie  1 { Danyelle  indica { Golda tions of use: lung scre { Jocelynn en { Noor ing consult diagnoses po { Leatha lysu { Maybelle bst { Mellissa ance abuse (c { Yoselin ms/hcc) [ { Mauro f19.1 { Lupe 0] personal history of nicotin { Darcie e dependence [z87.891] qu { Hadassah estionnaire question answ { Franklyn er pa { Zion tient meets criteria listed in { Kristan  order pro { Ishmael cess ins { Ramsey truc { Alayah tions: { Jasmyn  yes in { Abigayle dications po { Derrell lysu { Evette bstance abuse (cms/hcc) [f { Ayesha 19.10 ( { Paul ic { Lauren d-10-cm)] personal hist { Art ory of nico { Jalisa tine { Antone  dependence [z87.891 (icd-10-cm)] printed on 10/3/24 7:13 { Carroll  am page 1752",
    {
        "entities": [
            [
                35,
                42,
                "PERSON"
            ],
            [
                58,
                65,
                "PERSON"
            ],
            [
                76,
                84,
                "PERSON"
            ],
            [
                183,
                189,
                "PERSON"
            ],
            [
                208,
                214,
                "PERSON"
            ],
            [
                230,
                239,
                "PERSON"
            ],
            [
                244,
                253,
                "PERSON"
            ],
            [
                271,
                278,
                "PERSON"
            ],
            [
                290,
                297,
                "PERSON"
            ],
            [
                320,
                326,
                "PERSON"
            ],
            [
                360,
                368,
                "PERSON"
            ],
            [
                415,
                420,
                "PERSON"
            ],
            [
                448,
                455,
                "PERSON"
            ],
            [
                469,
                477,
                "PERSON"
            ],
            [
                483,
                492,
                "PERSON"
            ],
            [
                528,
                534,
                "PERSON"
            ],
            [
                547,
                556,
                "PERSON"
            ],
            [
                619,
                628,
                "PERSON"
            ],
            [
                637,
                643,
                "PERSON"
            ],
            [
                667,
                676,
                "PERSON"
            ],
            [
                690,
                697,
                "PERSON"
            ],
            [
                702,
                708,
                "PERSON"
            ],
            [
                722,
                729,
                "PERSON"
            ],
            [
                747,
                754,
                "PERSON"
            ],
            [
                764,
                771,
                "PERSON"
            ],
            [
                783,
                788,
                "PERSON"
            ],
            [
                805,
                815,
                "PERSON"
            ],
            [
                825,
                831,
                "PERSON"
            ],
            [
                842,
                848,
                "PERSON"
            ],
            [
                853,
                861,
                "PERSON"
            ],
            [
                882,
                888,
                "PERSON"
            ],
            [
                900,
                912,
                "PERSON"
            ],
            [
                932,
                940,
                "PERSON"
            ],
            [
                956,
                963,
                "PERSON"
            ],
            [
                967,
                973,
                "PERSON"
            ],
            [
                991,
                997,
                "PERSON"
            ],
            [
                1038,
                1044,
                "PERSON"
            ],
            [
                1054,
                1062,
                "PERSON"
            ],
            [
                1073,
                1080,
                "PERSON"
            ],
            [
                1107,
                1115,
                "PERSON"
            ],
            [
                1127,
                1136,
                "PERSON"
            ],
            [
                1148,
                1153,
                "PERSON"
            ],
            [
                1165,
                1176,
                "PERSON"
            ],
            [
                1188,
                1197,
                "PERSON"
            ],
            [
                1217,
                1226,
                "PERSON"
            ],
            [
                1238,
                1246,
                "PERSON"
            ],
            [
                1251,
                1256,
                "PERSON"
            ],
            [
                1265,
                1271,
                "PERSON"
            ],
            [
                1278,
                1285,
                "PERSON"
            ],
            [
                1294,
                1301,
                "PERSON"
            ],
            [
                1306,
                1312,
                "PERSON"
            ],
            [
                1320,
                1327,
                "PERSON"
            ],
            [
                1339,
                1347,
                "PERSON"
            ],
            [
                1357,
                1364,
                "PERSON"
            ],
            [
                1438,
                1443,
                "PERSON"
            ],
            [
                1447,
                1455,
                "PERSON"
            ],
            [
                1461,
                1469,
                "PERSON"
            ],
            [
                1482,
                1490,
                "PERSON"
            ],
            [
                1567,
                1574,
                "PERSON"
            ],
            [
                1579,
                1584,
                "PERSON"
            ],
            [
                1590,
                1598,
                "PERSON"
            ],
            [
                1631,
                1639,
                "PERSON"
            ],
            [
                1644,
                1652,
                "PERSON"
            ],
            [
                1662,
                1668,
                "PERSON"
            ],
            [
                1717,
                1724,
                "PERSON"
            ],
            [
                1735,
                1742,
                "PERSON"
            ],
            [
                1769,
                1778,
                "PERSON"
            ],
            [
                1782,
                1788,
                "PERSON"
            ],
            [
                1793,
                1800,
                "PERSON"
            ],
            [
                1804,
                1810,
                "PERSON"
            ],
            [
                1822,
                1832,
                "PERSON"
            ],
            [
                1837,
                1846,
                "PERSON"
            ],
            [
                1856,
                1862,
                "PERSON"
            ],
            [
                1888,
                1897,
                "PERSON"
            ],
            [
                1902,
                1907,
                "PERSON"
            ],
            [
                1934,
                1941,
                "PERSON"
            ],
            [
                1948,
                1957,
                "PERSON"
            ],
            [
                1963,
                1972,
                "PERSON"
            ],
            [
                1988,
                1996,
                "PERSON"
            ],
            [
                2008,
                2014,
                "PERSON"
            ],
            [
                2022,
                2027,
                "PERSON"
            ],
            [
                2060,
                2067,
                "PERSON"
            ],
            [
                2095,
                2104,
                "PERSON"
            ],
            [
                2132,
                2141,
                "PERSON"
            ],
            [
                2149,
                2154,
                "PERSON"
            ],
            [
                2187,
                2195,
                "PERSON"
            ],
            [
                2208,
                2216,
                "PERSON"
            ],
            [
                2227,
                2234,
                "PERSON"
            ],
            [
                2241,
                2248,
                "PERSON"
            ],
            [
                2257,
                2264,
                "PERSON"
            ],
            [
                2274,
                2283,
                "PERSON"
            ],
            [
                2298,
                2306,
                "PERSON"
            ],
            [
                2313,
                2320,
                "PERSON"
            ],
            [
                2349,
                2356,
                "PERSON"
            ],
            [
                2366,
                2371,
                "PERSON"
            ],
            [
                2376,
                2383,
                "PERSON"
            ],
            [
                2409,
                2413,
                "PERSON"
            ],
            [
                2427,
                2434,
                "PERSON"
            ],
            [
                2441,
                2448,
                "PERSON"
            ],
            [
                2508,
                2516,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 { Brigette  adm: 6/2/ { Prudence 2023, d/c: 6/3/20 { Chet 23 06/02/2023 - ed to hosp-admission (discharged) in  { Orpha vanderbilt university adult hospital { Anabella  (continued) ed care timeline (continued) 10:00 assessment neu { Memphis ro a { Ira ll hamm, shannon { Averi  neuro assess { Harper ment : wel (slurs wor { Karolyn ds while speaking slowly) c neuro problem: speech deficit ment { Santana al status/rass score: alert and calm seizur { Verne e activity:  coma scale eye opening: spontaneous best verbal resp { Edgardo onse: or { Brandt iented best motor res { Kelsea ponse: o { Hailie beys commands glasgow coma scale score: 15 cardiac cardiac assessment : { Leatrice  wel heart sounds: s1s2 heart rate: { Tamar  regular cardiac intervention { Brandee s { Aliana : vs vascular/perfusion all vascular assessment: wel pulse locations: r radial; l radial; r dorsalis pedal; l { Eleni  dorsalis pedal r r { Gaye adial : 2+ - palpable l radial : 2+ - palpable r dorsalis pedal: 2 { Lida + - palpable l dorsalis pedal: 2+ - palpable neurovascular extremity asmt: yes rue neurovascular assessment: done lue neurovascular assessment: done rle neurovascular asses { Ronin sment: numbness lle neuro { Novella vascular assessment: numbness re { Shamar spiratory respiratory assessment: oel re { Domonique spiratory problem: breathing pattern alteration r { Williams e { Arden spi { Leopoldo ratory effort: dyspnea on exert trachea position: mid-l { Javion ine gastrointestinal a { Tenley ll gi assessment : wnl renal/urinary all renal/urinary assessment: wel (aki on ckd 4) renal/urinary problem { Lennox : renal altera { Emmaline tion reproductive all reproductive asse { Roseanna ssment: male skin all skin assessment : wel skin proble { Jair m: skin inte { Hyman gr { Rayne i { Jamir ty impairment (pt ha { Douglass s diabetic ulcer to left big toe) skin breakdown control: absorbent underpad braden scale sensory perceptions:  moisture: rarely m { Demond oist activity: walks occasionally mobility: slightly limited nutrition: adequate friction and  { Latricia shear:  problem braden scale sco { Normand re: 20 braden risk level: standard risk activity/musculoskeletal all a { Jade ctivity/musculoskeletal asse { Leonidas ssment: oel acti { Serina vity/musculoskeletal problem: mo { Carter bility impairment generalize { Vickey d weakness: noted strength alteration?: yes rue strength: 5/5 movement against gravity with full resistance lue strength: 5/5 movement { Coty  aga { Jonnie inst gravity with full resistance rle strength: 4/5 m { Shae oveme { Ines nt against gravity with some resistance lle strength: 4/5 movement ag { Leonor ainst gravity with some resistance positioning intervention: self-repositioning j hopkins highest  { Yuliana mobility lvl evaluation: (6) walk 10+ steps mobility intervention: walk 10+ steps printed on 10/3/24 7:13 am page 2187,vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, le { Pennie gal sex: m nashville tn 37232-0004 adm: 6/2/2023, d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt university adult hospital (continued) ed c { Camilo are timelin { Rayan e (continued) le { Gwyneth vel of mobility assistance ne { Orin eded: standby assist fluid/nutrition all fluids/elect { Dontae rolyte asse { Reanna ssm { Missy ent : wel fluid therapy interventions: hydration nutrition assessment: wnl safety all (alcohol/drug, fall { Fawn , elopement, restraint, self-mutilation, su { Joi icide, violence) falls risk/safety risk assessment: wel broset? does thi { Lesli s patient require a broset assessment?: yes broset violence checklist  { Kandi violence risk:  threat confused: not present irritable: not pre { Lovie sent boisterou { Malaya s: not present verbally: not present physically threatening: not present attacking objects: not present broset s { Atlas um -n { Yehuda ursin { Janay g: i { Harlee f score 2/>, start de-escalation.: 0 jhfrat fall risk if patient d { Lilith oes not meet eith { Rihanna er of the above two criteria, please use tool: jhfrat age: <60 years fall history:  fall within the last 6 months elimination, bowel and uri { Jenni ne:  problems medications (see { Menachem  row information): on 1 high fall risk drug patient care equipment (see row information): one present mobility -choose all that  { Katia apply: requires assistance or supervision for mobility, transfer, or ambulation; unsteady gait cognition - choose all that apply:  impairment total f { Koby all risk score: 8 fall risk: moder { Kelsey ate fall risk medication all medication assess { Deron ment: { Lisbeth  wel self care (adl) all self care assessme { Reyes nt: wel skin breakdown control: absorbent underpad infection/metabolic all infection/metabolic assessment: wel infection/metabolic problem { Nathalia s: endocrine alteration; glucose alteration; infection risk isolation precautions: protective environment psychosocial all ps { Hobert ychosocial assessment: wel family presence: pati { Briley ent alone family/patient engagement interventions: d { Aarav aily goals discussed emotional care/ { Venita interv { Oralia entions: active listening; reassured; offer info communication care/intervent { Alfonzo ions: commun { Hendrix ication board patient goals patient/family goal for admission: to get better & go home shift goals: activity { Graeme ; neurological; safety/fall activity goal: will to { Whitley lerate prescribed activity n { Tanesha eurologic { Berry al goal: will maintain neurologic { Nichelle al stability; will experience  activity safety/fall goal: pa { Areli tie { Sheridan nt will rem { Abagail ain free from falls and harm from falls plan of care discharge plan of care reviewed: done review education: done printed on 10/ { Charleigh 3/24 7:13 am { Mitch  page 2188",
    {
        "entities": [
            [
                129,
                138,
                "PERSON"
            ],
            [
                151,
                160,
                "PERSON"
            ],
            [
                180,
                185,
                "PERSON"
            ],
            [
                241,
                247,
                "PERSON"
            ],
            [
                286,
                295,
                "PERSON"
            ],
            [
                360,
                368,
                "PERSON"
            ],
            [
                375,
                379,
                "PERSON"
            ],
            [
                398,
                404,
                "PERSON"
            ],
            [
                420,
                427,
                "PERSON"
            ],
            [
                451,
                459,
                "PERSON"
            ],
            [
                524,
                532,
                "PERSON"
            ],
            [
                578,
                584,
                "PERSON"
            ],
            [
                652,
                660,
                "PERSON"
            ],
            [
                671,
                678,
                "PERSON"
            ],
            [
                702,
                709,
                "PERSON"
            ],
            [
                720,
                727,
                "PERSON"
            ],
            [
                801,
                810,
                "PERSON"
            ],
            [
                848,
                854,
                "PERSON"
            ],
            [
                886,
                894,
                "PERSON"
            ],
            [
                898,
                905,
                "PERSON"
            ],
            [
                1017,
                1023,
                "PERSON"
            ],
            [
                1045,
                1050,
                "PERSON"
            ],
            [
                1119,
                1124,
                "PERSON"
            ],
            [
                1299,
                1305,
                "PERSON"
            ],
            [
                1333,
                1341,
                "PERSON"
            ],
            [
                1376,
                1383,
                "PERSON"
            ],
            [
                1426,
                1436,
                "PERSON"
            ],
            [
                1488,
                1497,
                "PERSON"
            ],
            [
                1501,
                1507,
                "PERSON"
            ],
            [
                1513,
                1522,
                "PERSON"
            ],
            [
                1580,
                1587,
                "PERSON"
            ],
            [
                1612,
                1619,
                "PERSON"
            ],
            [
                1729,
                1736,
                "PERSON"
            ],
            [
                1753,
                1762,
                "PERSON"
            ],
            [
                1804,
                1813,
                "PERSON"
            ],
            [
                1871,
                1876,
                "PERSON"
            ],
            [
                1891,
                1897,
                "PERSON"
            ],
            [
                1902,
                1908,
                "PERSON"
            ],
            [
                1912,
                1918,
                "PERSON"
            ],
            [
                1941,
                1950,
                "PERSON"
            ],
            [
                2083,
                2090,
                "PERSON"
            ],
            [
                2187,
                2196,
                "PERSON"
            ],
            [
                2231,
                2239,
                "PERSON"
            ],
            [
                2312,
                2317,
                "PERSON"
            ],
            [
                2348,
                2357,
                "PERSON"
            ],
            [
                2376,
                2383,
                "PERSON"
            ],
            [
                2418,
                2425,
                "PERSON"
            ],
            [
                2456,
                2463,
                "PERSON"
            ],
            [
                2600,
                2605,
                "PERSON"
            ],
            [
                2612,
                2619,
                "PERSON"
            ],
            [
                2675,
                2680,
                "PERSON"
            ],
            [
                2688,
                2693,
                "PERSON"
            ],
            [
                2765,
                2772,
                "PERSON"
            ],
            [
                2873,
                2881,
                "PERSON"
            ],
            [
                3095,
                3102,
                "PERSON"
            ],
            [
                3272,
                3279,
                "PERSON"
            ],
            [
                3293,
                3299,
                "PERSON"
            ],
            [
                3318,
                3326,
                "PERSON"
            ],
            [
                3358,
                3363,
                "PERSON"
            ],
            [
                3419,
                3426,
                "PERSON"
            ],
            [
                3440,
                3447,
                "PERSON"
            ],
            [
                3453,
                3459,
                "PERSON"
            ],
            [
                3567,
                3572,
                "PERSON"
            ],
            [
                3618,
                3622,
                "PERSON"
            ],
            [
                3697,
                3703,
                "PERSON"
            ],
            [
                3776,
                3782,
                "PERSON"
            ],
            [
                3848,
                3854,
                "PERSON"
            ],
            [
                3871,
                3878,
                "PERSON"
            ],
            [
                3993,
                3999,
                "PERSON"
            ],
            [
                4007,
                4014,
                "PERSON"
            ],
            [
                4022,
                4028,
                "PERSON"
            ],
            [
                4035,
                4042,
                "PERSON"
            ],
            [
                4111,
                4118,
                "PERSON"
            ],
            [
                4138,
                4146,
                "PERSON"
            ],
            [
                4289,
                4295,
                "PERSON"
            ],
            [
                4328,
                4337,
                "PERSON"
            ],
            [
                4468,
                4474,
                "PERSON"
            ],
            [
                4626,
                4631,
                "PERSON"
            ],
            [
                4668,
                4675,
                "PERSON"
            ],
            [
                4724,
                4730,
                "PERSON"
            ],
            [
                4738,
                4746,
                "PERSON"
            ],
            [
                4792,
                4798,
                "PERSON"
            ],
            [
                4939,
                4948,
                "PERSON"
            ],
            [
                5076,
                5083,
                "PERSON"
            ],
            [
                5134,
                5141,
                "PERSON"
            ],
            [
                5196,
                5202,
                "PERSON"
            ],
            [
                5241,
                5248,
                "PERSON"
            ],
            [
                5257,
                5264,
                "PERSON"
            ],
            [
                5344,
                5352,
                "PERSON"
            ],
            [
                5367,
                5375,
                "PERSON"
            ],
            [
                5486,
                5493,
                "PERSON"
            ],
            [
                5546,
                5554,
                "PERSON"
            ],
            [
                5585,
                5593,
                "PERSON"
            ],
            [
                5605,
                5611,
                "PERSON"
            ],
            [
                5647,
                5656,
                "PERSON"
            ],
            [
                5719,
                5725,
                "PERSON"
            ],
            [
                5731,
                5740,
                "PERSON"
            ],
            [
                5754,
                5762,
                "PERSON"
            ],
            [
                5893,
                5903,
                "PERSON"
            ],
            [
                5918,
                5924,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one h { Shirlee undred oaks white,  { Gannon tyr { Waldo one 719 thompson lane, nashville mrn: 04 { Jessa 7 { Maximillian 717361, dob: 7/2 { Ean 7/1969, legal sex: m nashville tn 37204  { Mustafa visit date { Rollin : 10/5/ { Aretha 2023 { Jonna  10/05/2023 - med { Elden ication { Kattie  managemen { Oswaldo t in vanderbi { Yajaira lt one hundred { Aviana  oaks primar { Carolann y care  { Walton north (con { Zeke tinued) cl { Mai inical not { Ashtyn es (continued)  { Abigale popul { Andra ation health clinical pharmacist 615-322-4663 electronically signed by wilson, danya hor { Lizabeth chi { Cheryle , pharmd at 10/5/2023 3:06 pm printed on 10/3/24 { Lennon  7:13 am page 1621,vumc adult hospi { Orrin tal white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27 { Sherwood /1969, legal sex { Bishop : m nashville t { Keshawn n 37232-0004 visit date: 10/4/2023 10/04/2023 - { Cathie  patient message in vanderbilt pharmacy { Isadore  retail services facesheet report patient demographic { Aloysius s patient name mrn legal dob address phone white, tyrone 0477173 s { Donald ex 7/27/1969 apt 705 615-260-2291 (home) 61 m 1101 edgeh { Annalisa ill ave 615-260-2291 (mobile) nashville tn 37203 *pre { Era ferr { Mariyah ed* hospital account not on f { Derik ile a { Talmadge dmi { Ora ssion i { Jemma n { Kevin for { Rosario ma { Lucio tion current information attending provider { Samual  admitting provider admis { Val sion  { Saylor type admission stat { Cal us unknown  { Candis s { Garfield tatus admission date/time discharge date/time { Johnpaul  hospital service auth/cert st { Nixon atus { Ardith  h { Merritt o { Nelle spit { Meg al { Yasmeen  area unit  { Vladimir room { Shira /bed referr { Jaidyn i { Marcello ng pro { Zakary vid { Linsey er 10 { Cami /04/2023 - patient message { Tatianna  in va { Shamika nderbilt pharm { Sloan acy retail services (continued) visit  { Kortney information prov { Zena ide { Chanelle r informat { Chyna ion e { Stephani n { Shanta counter provider mychart, generic provider, a { Rupert cnp, crna, rn department na { Dejuan me address va { Rosina nderbilt { Garrick  pharmacy reta { Claudio il services tn { Adella  messages medication fill from to sent and delivered la { Zavier uren hope bedwell, cpht white, tyrone 10/4/2023 3:54 pm last read in { Shelbie  my healt { Rayford h at vanderbilt not read dear mr. white: according to our  { Charisse r { Syble ecords, you are due for the foll { Jordy owing refill(s): { Campbell  pl { Easter ease answer the following 3 qu { Tre estions to hav { Arlette e your delivery set up: 1. please i { Lorelai ndicate the address where your order should be sent. 2. would  { Adilene yo { Hardy u like to sign for the package with ups? please note, signature is required for all  { Carleigh prescripti { Ronaldo ons billed to tenncare and schedule 2 me { Ireland dicines (all rx num { Granville bers starting { Shelbi  w { Lupita ith 2). printed on  { Shaquille 10/3/24 7:13 am { Gerri   { Joshua page 1622",
    {
        "entities": [
            [
                19,
                27,
                "PERSON"
            ],
            [
                49,
                56,
                "PERSON"
            ],
            [
                62,
                68,
                "PERSON"
            ],
            [
                111,
                117,
                "PERSON"
            ],
            [
                121,
                133,
                "PERSON"
            ],
            [
                152,
                156,
                "PERSON"
            ],
            [
                199,
                207,
                "PERSON"
            ],
            [
                220,
                227,
                "PERSON"
            ],
            [
                237,
                244,
                "PERSON"
            ],
            [
                251,
                257,
                "PERSON"
            ],
            [
                277,
                283,
                "PERSON"
            ],
            [
                293,
                300,
                "PERSON"
            ],
            [
                313,
                321,
                "PERSON"
            ],
            [
                337,
                345,
                "PERSON"
            ],
            [
                362,
                369,
                "PERSON"
            ],
            [
                384,
                393,
                "PERSON"
            ],
            [
                403,
                410,
                "PERSON"
            ],
            [
                423,
                428,
                "PERSON"
            ],
            [
                441,
                445,
                "PERSON"
            ],
            [
                458,
                465,
                "PERSON"
            ],
            [
                483,
                491,
                "PERSON"
            ],
            [
                499,
                505,
                "PERSON"
            ],
            [
                596,
                605,
                "PERSON"
            ],
            [
                611,
                619,
                "PERSON"
            ],
            [
                670,
                677,
                "PERSON"
            ],
            [
                715,
                721,
                "PERSON"
            ],
            [
                791,
                800,
                "PERSON"
            ],
            [
                819,
                826,
                "PERSON"
            ],
            [
                844,
                852,
                "PERSON"
            ],
            [
                902,
                909,
                "PERSON"
            ],
            [
                951,
                959,
                "PERSON"
            ],
            [
                1015,
                1024,
                "PERSON"
            ],
            [
                1093,
                1100,
                "PERSON"
            ],
            [
                1159,
                1168,
                "PERSON"
            ],
            [
                1224,
                1228,
                "PERSON"
            ],
            [
                1235,
                1243,
                "PERSON"
            ],
            [
                1275,
                1281,
                "PERSON"
            ],
            [
                1289,
                1298,
                "PERSON"
            ],
            [
                1304,
                1308,
                "PERSON"
            ],
            [
                1318,
                1324,
                "PERSON"
            ],
            [
                1328,
                1334,
                "PERSON"
            ],
            [
                1340,
                1348,
                "PERSON"
            ],
            [
                1353,
                1359,
                "PERSON"
            ],
            [
                1405,
                1412,
                "PERSON"
            ],
            [
                1440,
                1444,
                "PERSON"
            ],
            [
                1452,
                1459,
                "PERSON"
            ],
            [
                1481,
                1485,
                "PERSON"
            ],
            [
                1499,
                1506,
                "PERSON"
            ],
            [
                1510,
                1519,
                "PERSON"
            ],
            [
                1567,
                1576,
                "PERSON"
            ],
            [
                1609,
                1615,
                "PERSON"
            ],
            [
                1622,
                1629,
                "PERSON"
            ],
            [
                1634,
                1642,
                "PERSON"
            ],
            [
                1646,
                1652,
                "PERSON"
            ],
            [
                1659,
                1663,
                "PERSON"
            ],
            [
                1668,
                1676,
                "PERSON"
            ],
            [
                1690,
                1699,
                "PERSON"
            ],
            [
                1706,
                1712,
                "PERSON"
            ],
            [
                1726,
                1733,
                "PERSON"
            ],
            [
                1737,
                1746,
                "PERSON"
            ],
            [
                1755,
                1762,
                "PERSON"
            ],
            [
                1768,
                1775,
                "PERSON"
            ],
            [
                1783,
                1788,
                "PERSON"
            ],
            [
                1817,
                1826,
                "PERSON"
            ],
            [
                1835,
                1843,
                "PERSON"
            ],
            [
                1860,
                1866,
                "PERSON"
            ],
            [
                1907,
                1915,
                "PERSON"
            ],
            [
                1934,
                1939,
                "PERSON"
            ],
            [
                1945,
                1954,
                "PERSON"
            ],
            [
                1967,
                1973,
                "PERSON"
            ],
            [
                1981,
                1990,
                "PERSON"
            ],
            [
                1994,
                2001,
                "PERSON"
            ],
            [
                2049,
                2056,
                "PERSON"
            ],
            [
                2086,
                2093,
                "PERSON"
            ],
            [
                2109,
                2116,
                "PERSON"
            ],
            [
                2127,
                2135,
                "PERSON"
            ],
            [
                2152,
                2160,
                "PERSON"
            ],
            [
                2177,
                2184,
                "PERSON"
            ],
            [
                2242,
                2249,
                "PERSON"
            ],
            [
                2320,
                2328,
                "PERSON"
            ],
            [
                2340,
                2348,
                "PERSON"
            ],
            [
                2409,
                2418,
                "PERSON"
            ],
            [
                2422,
                2428,
                "PERSON"
            ],
            [
                2463,
                2469,
                "PERSON"
            ],
            [
                2488,
                2497,
                "PERSON"
            ],
            [
                2503,
                2510,
                "PERSON"
            ],
            [
                2543,
                2547,
                "PERSON"
            ],
            [
                2564,
                2572,
                "PERSON"
            ],
            [
                2610,
                2618,
                "PERSON"
            ],
            [
                2683,
                2691,
                "PERSON"
            ],
            [
                2696,
                2702,
                "PERSON"
            ],
            [
                2789,
                2798,
                "PERSON"
            ],
            [
                2811,
                2819,
                "PERSON"
            ],
            [
                2862,
                2870,
                "PERSON"
            ],
            [
                2892,
                2902,
                "PERSON"
            ],
            [
                2918,
                2925,
                "PERSON"
            ],
            [
                2930,
                2937,
                "PERSON"
            ],
            [
                2959,
                2969,
                "PERSON"
            ],
            [
                2987,
                2993,
                "PERSON"
            ],
            [
                2997,
                3004,
                "PERSON"
            ]
        ]
    }
),(
    "vumc { Deshaun  adult medical center ea { Keyon st whit { Melonie e, tyrone 1211 medical center dr mrn: 047717361, { Judie  dob: 7/27/ { Katlin 1969, legal sex: m nashville t { Cody n 37232 visit dat { Anjali e: 9/25/2023 09/25/2023 - appointment in vanderbil { Bronwyn t diabetes ( { Karie continued) referral (continued) e11.22,z79.4 (icd-10-cm) - type  { Kesha 2 diabetes mellitus with chronic kidney disea { Demarco se, with long-term current us { Annelise e of insul { Kenneth in, unspecified ckd st { Harland age { Boyce  (cms/hcc) l84 (icd-10-cm) - foot callus referral notes provider comme { Krystina nts by de witt { Oleta e, anton jordan, md at 9/ { Kensley 7/2023 1 { Marin 506 summary: { Filomena  prov { Dorsey ider comments t2dm with significant callus burden order ambulatory referral t { Edd o podiatry [39283 { Ray 3 { Edward 799] electronically signed by: de witt { Makai e, anton jordan, md  { Rona on 09/07 { Nya /23 1506 status: active  { Emory o { Charissa rdering user: de witte { Jessenia , anton jordan, md 09/07/23 1506 ordering prov { Joellen ider: de witte, anton { Jasiah  jordan, md authorized by: { Lailah  sullivan, kathleen pollard, md ordered { Dian  during: offi { Cary ce vi { Jamarcus sit { Kyree  on { Billy  09/07/20 { Sanaa 23 d { Davian iagnoses type 2 diabet { Malcom es mellitus w { Elizabeth ith c { Shannan hronic kidney disease, with long-term current use of in { Milena sulin, unspecified ckd stage (cms/hcc) [e11.22, z79.4] foot callu { Sunshine s [l84] order comments: t2dm with significant call { Hulda u { Destin s burden triage triage information decision: n { Marcelle one { Gustave  schedule by date: 10/7/2023 coverages ambetter exchange plan: ambetter select covered: covered from: 5/1/2024 member #: u71 { Zainab 93652401 exchange  { Shantell uhc community dual snp { Dock  plan: uhc community dual covered: cove { Kyndall r { Lynsey ed from: 9/1/2024 mem { Callen ber #: 127 { Brea 670950 snp uhc community  { Charlize dual { Jannette  snp plan: uhc community dual covered: cov { Shaniqua ered fro { Marquez m: 1/1 { Aya /2024 to: 1/31/2024 snp mem { Delano ber #: 12767095 { Farah 0 amerivantage wellpoint medicare plan: amerivantage co { Isela vered: covered from: 6/1/2023 { Joanie  to: 2/29/2024 amerigroup wellpoint ma member #: 768w12411 zzzmcaid of tenn { Keanu essee plan { Junius : medicaid supplemental covered: covered fr { Tressie om: 6/1/2022 to: 12/3 { Kandy 1/2023 member #: td525606373 tc ten { Laylah ncare select plan { Jason : { Caleigh  tc select  covered: covered fro { Kalie m: 8/30/2024 to: 8/ { Kaylene 30/2024 member #: zedm13004089 printed on 10/3/24 7:13 am page 1 { Madden 651,vumc adult medical center east white, tyrone 1211 medical center dr mrn: 047717361, dob: 7/27/1969, legal sex { Riya : m nashville tn 37232 visi { Rosendo t date: 9/25/2023 09/25/2023 - appointment in vanderbilt d { Adelle ia { Kyson betes (continued) r { Ernestina eferral ( { Deion continued) messages ap { Heidy pointment rescheduled from to sent and delivered m { Marshal ychart { Mahlon , gen { Azaria eric white, t { Zaire y { Aydan rone 9/19/2023 9:55 am last read in my healt { Donavon h at vanderbilt { Ilana  not read a { Savana ppo { Mozelle intment inform { Andrew ation: visit type: new patient visit date: 9/25/2023 dept: vanderbilt diabetes p { Amare rovider: dawn m maste { Trena rni { Artis ck time: 8:00 am length { Tobin : 15 min appt status: scheduled origin { Trinidad al appointment information: visit type { Jerilyn : new patient visit date: 9/25/2023 dept: vanderbilt orthopaedics provider: adam bradburn hicks time: 11:40 am length: 10 mi { Bambi n cancel reason: clinic request printed on 10/3/24 7:13 am page { Jerrie  1652",
    {
        "entities": [
            [
                7,
                15,
                "PERSON"
            ],
            [
                42,
                48,
                "PERSON"
            ],
            [
                58,
                66,
                "PERSON"
            ],
            [
                117,
                123,
                "PERSON"
            ],
            [
                137,
                144,
                "PERSON"
            ],
            [
                177,
                182,
                "PERSON"
            ],
            [
                202,
                209,
                "PERSON"
            ],
            [
                262,
                270,
                "PERSON"
            ],
            [
                285,
                291,
                "PERSON"
            ],
            [
                358,
                364,
                "PERSON"
            ],
            [
                412,
                420,
                "PERSON"
            ],
            [
                452,
                461,
                "PERSON"
            ],
            [
                474,
                482,
                "PERSON"
            ],
            [
                507,
                515,
                "PERSON"
            ],
            [
                521,
                527,
                "PERSON"
            ],
            [
                600,
                609,
                "PERSON"
            ],
            [
                626,
                632,
                "PERSON"
            ],
            [
                660,
                668,
                "PERSON"
            ],
            [
                679,
                685,
                "PERSON"
            ],
            [
                700,
                709,
                "PERSON"
            ],
            [
                717,
                724,
                "PERSON"
            ],
            [
                804,
                808,
                "PERSON"
            ],
            [
                828,
                832,
                "PERSON"
            ],
            [
                836,
                843,
                "PERSON"
            ],
            [
                884,
                890,
                "PERSON"
            ],
            [
                913,
                918,
                "PERSON"
            ],
            [
                929,
                933,
                "PERSON"
            ],
            [
                960,
                966,
                "PERSON"
            ],
            [
                970,
                979,
                "PERSON"
            ],
            [
                1004,
                1013,
                "PERSON"
            ],
            [
                1062,
                1070,
                "PERSON"
            ],
            [
                1094,
                1101,
                "PERSON"
            ],
            [
                1130,
                1137,
                "PERSON"
            ],
            [
                1179,
                1184,
                "PERSON"
            ],
            [
                1200,
                1205,
                "PERSON"
            ],
            [
                1213,
                1222,
                "PERSON"
            ],
            [
                1228,
                1234,
                "PERSON"
            ],
            [
                1240,
                1246,
                "PERSON"
            ],
            [
                1258,
                1264,
                "PERSON"
            ],
            [
                1271,
                1278,
                "PERSON"
            ],
            [
                1303,
                1310,
                "PERSON"
            ],
            [
                1326,
                1336,
                "PERSON"
            ],
            [
                1344,
                1352,
                "PERSON"
            ],
            [
                1410,
                1417,
                "PERSON"
            ],
            [
                1485,
                1494,
                "PERSON"
            ],
            [
                1547,
                1553,
                "PERSON"
            ],
            [
                1557,
                1564,
                "PERSON"
            ],
            [
                1613,
                1622,
                "PERSON"
            ],
            [
                1628,
                1636,
                "PERSON"
            ],
            [
                1763,
                1770,
                "PERSON"
            ],
            [
                1791,
                1800,
                "PERSON"
            ],
            [
                1825,
                1830,
                "PERSON"
            ],
            [
                1872,
                1880,
                "PERSON"
            ],
            [
                1884,
                1891,
                "PERSON"
            ],
            [
                1915,
                1922,
                "PERSON"
            ],
            [
                1935,
                1940,
                "PERSON"
            ],
            [
                1968,
                1977,
                "PERSON"
            ],
            [
                1984,
                1993,
                "PERSON"
            ],
            [
                2038,
                2047,
                "PERSON"
            ],
            [
                2058,
                2066,
                "PERSON"
            ],
            [
                2075,
                2079,
                "PERSON"
            ],
            [
                2109,
                2116,
                "PERSON"
            ],
            [
                2134,
                2140,
                "PERSON"
            ],
            [
                2198,
                2204,
                "PERSON"
            ],
            [
                2236,
                2243,
                "PERSON"
            ],
            [
                2321,
                2327,
                "PERSON"
            ],
            [
                2340,
                2347,
                "PERSON"
            ],
            [
                2393,
                2401,
                "PERSON"
            ],
            [
                2425,
                2431,
                "PERSON"
            ],
            [
                2469,
                2476,
                "PERSON"
            ],
            [
                2496,
                2502,
                "PERSON"
            ],
            [
                2506,
                2514,
                "PERSON"
            ],
            [
                2549,
                2555,
                "PERSON"
            ],
            [
                2577,
                2585,
                "PERSON"
            ],
            [
                2652,
                2659,
                "PERSON"
            ],
            [
                2775,
                2780,
                "PERSON"
            ],
            [
                2810,
                2818,
                "PERSON"
            ],
            [
                2879,
                2886,
                "PERSON"
            ],
            [
                2891,
                2897,
                "PERSON"
            ],
            [
                2919,
                2929,
                "PERSON"
            ],
            [
                2941,
                2947,
                "PERSON"
            ],
            [
                2972,
                2978,
                "PERSON"
            ],
            [
                3031,
                3039,
                "PERSON"
            ],
            [
                3048,
                3055,
                "PERSON"
            ],
            [
                3063,
                3070,
                "PERSON"
            ],
            [
                3086,
                3092,
                "PERSON"
            ],
            [
                3096,
                3102,
                "PERSON"
            ],
            [
                3149,
                3157,
                "PERSON"
            ],
            [
                3175,
                3181,
                "PERSON"
            ],
            [
                3195,
                3202,
                "PERSON"
            ],
            [
                3208,
                3216,
                "PERSON"
            ],
            [
                3233,
                3240,
                "PERSON"
            ],
            [
                3323,
                3329,
                "PERSON"
            ],
            [
                3353,
                3359,
                "PERSON"
            ],
            [
                3365,
                3371,
                "PERSON"
            ],
            [
                3397,
                3403,
                "PERSON"
            ],
            [
                3444,
                3453,
                "PERSON"
            ],
            [
                3494,
                3502,
                "PERSON"
            ],
            [
                3629,
                3635,
                "PERSON"
            ],
            [
                3701,
                3708,
                "PERSON"
            ]
        ]
    }
),(
    "vum { Arvin c adult ho { Devan spital  { Tye white, tyrone 1211 medical ce { Greggory nt { Sydni er dr. mrn: 0477173 { Jaylyn 61, dob: 7/27/1 { Wynter 969, legal sex: m n { Alaya ashville tn 37232-0004 adm: 8/22/2023, d/c: 8/2 { Norine 3/2023 08/22/2023 - { Syed  ed to hosp-admission (discharged) in vande { Meryl rbilt emergency de { Jacalyn partment (continued) ed notes (continued) result value sodium level 138 potassium level 4.4 { Donita  chloride level 108 (*) carbon dioxide 20 (*) glucose level { Jovanni  143 (*) blood urea nitrogen 42 (*) creatinine level 4.31 (*) calci { Roxane um level total 8.8 anion gap { Garett  10 c { Shara bc w { Delvin / different { Marlen ia { Pilar l - abn { Judd ormal white b { Jamya lood cells 5.9 red blood cells 4 { Kacy . { Dalila 50 hemoglobin 12.1 (*) h { Stetson ema { Concepcion tocrit 38 (*) mean cell volume  { Lakeshia 85 mean cell hemoglobin 26 { Salena .9 (*) mean cell hemoglobin  { Karim 31.5 conc { Kati entration rdw sd 45.0 rdw cv 14.6 (*) platelet 238  { Kora mean platelet volume 11.3 nucleated rbc 0 nucleated rbc abs 0.00 auto neutrophil 3.20 a { Soraya bsolute troponin-i - abnormal troponin- 0.09 (*) troponin-i - a { Winnifred bnormal troponin-i 0. { Nikhil 08 (*) hep { Lillianna atic function pnl { Wylie  - abnormal bilirubin direct 0.1 { Lavinia  bilirubin total 0.3 albumin level 3.6 alkaline phosp { Caesar hatase 100 alanine  { Jeanetta 30 amin { Alexandre otransfer { Laron ase aspart { Latonia ate 41 (*) aminotransferase protein total 6.4 egfrcr - abnormal  { Vina egfrcr 15 (*) bnp  { Jennifer b-type natr { Regis iuretic <10 peptide lipase lvl lipase level 22 aut { Cambria o diff neutrop { Eldridge hils 54.3 absolute neutrophils 3.20 lymp { Cailyn hs 33.0 printed on  { Jacquelynn 10/3/24 7:13 am  { Deandra page 1913,vumc adult hospital w { Jordyn hite, tyrone 1211 medical center  { Louann dr. mrn: 047717361, dob: { Elvera  7/27/1969, legal sex: m nashville { Sherie  tn 3723 { Casie 2-0004 adm: 8/22/2023, d/c: 8/23/2023 { Shon  08/22/202 { Langston 3 - ed to hosp-admission (di { Jewel scharged) in vanderbilt emergency department (co { Tarah ntinued) ed notes (c { Ileana ontinued) absolute lymph { Kylah ocytes 1.94 monocytes 5.3 abs { Winfield olute monocyt { Romona es 0.31 eosinophils 6.0 absolute eosinophi { Grecia ls 0.35 b { Laken asophils 1.2 a { Jazmyne bsolute basophils 0.07 imm gran automated  { Yisroel 0. { Anneliese 2 absolute imm g { Brittanie ran 0.01 automat { Ayleen ed tsh thyroid stimulating 0.749 horm { Adelia one  { Tawnya free t4 thyr { Alvis oxine f { Hayleigh ree 0.74 ur ed drug scrn  bmp cbc lipid pnl troponin-i imagi { Lianna ng: x-ray chest 2  { Aydin views  { Dasia fina { Torrey l result xr che { Albertha st 2 vw history: ches { Mordechai t pain views: pa and lateral views { Hosea  of the chest. comparison: 1/11/2023. findings { Jayna : mild hy { Domenick poinflation of the lungs.  opac { Kevon i { Johnie ties of the lung.  effusion. . cardiomediastinal silhouette is withi { Von n normal limits. redemonstratio { Hakeem n of two p { Stone unctate metallic op { Kathlyn acities projecting over th { Curtiss e soft tissues of the back in the right back and left upper quadrant pa view. impressio { Braylee n:   { Tatum evidence of acute cardiopulmonar { Marita y disease. ele { Cydney ctronically signed by ad { Elwin rian marce { Chace l jaramillo, md  { Levon on 8/22/2023 1 { Darell 1:10 pm printed on 10/3/24 7: { Sydnie 13 am page 191 { Evonne 4",
    {
        "entities": [
            [
                6,
                12,
                "PERSON"
            ],
            [
                25,
                31,
                "PERSON"
            ],
            [
                41,
                45,
                "PERSON"
            ],
            [
                77,
                86,
                "PERSON"
            ],
            [
                91,
                97,
                "PERSON"
            ],
            [
                119,
                126,
                "PERSON"
            ],
            [
                144,
                151,
                "PERSON"
            ],
            [
                173,
                179,
                "PERSON"
            ],
            [
                229,
                236,
                "PERSON"
            ],
            [
                258,
                263,
                "PERSON"
            ],
            [
                309,
                315,
                "PERSON"
            ],
            [
                336,
                344,
                "PERSON"
            ],
            [
                438,
                445,
                "PERSON"
            ],
            [
                507,
                515,
                "PERSON"
            ],
            [
                585,
                592,
                "PERSON"
            ],
            [
                623,
                630,
                "PERSON"
            ],
            [
                638,
                644,
                "PERSON"
            ],
            [
                651,
                658,
                "PERSON"
            ],
            [
                672,
                679,
                "PERSON"
            ],
            [
                684,
                690,
                "PERSON"
            ],
            [
                700,
                705,
                "PERSON"
            ],
            [
                721,
                727,
                "PERSON"
            ],
            [
                762,
                767,
                "PERSON"
            ],
            [
                771,
                778,
                "PERSON"
            ],
            [
                805,
                813,
                "PERSON"
            ],
            [
                819,
                830,
                "PERSON"
            ],
            [
                864,
                873,
                "PERSON"
            ],
            [
                902,
                909,
                "PERSON"
            ],
            [
                940,
                946,
                "PERSON"
            ],
            [
                958,
                963,
                "PERSON"
            ],
            [
                1017,
                1022,
                "PERSON"
            ],
            [
                1112,
                1119,
                "PERSON"
            ],
            [
                1185,
                1195,
                "PERSON"
            ],
            [
                1219,
                1226,
                "PERSON"
            ],
            [
                1239,
                1249,
                "PERSON"
            ],
            [
                1269,
                1275,
                "PERSON"
            ],
            [
                1310,
                1318,
                "PERSON"
            ],
            [
                1374,
                1381,
                "PERSON"
            ],
            [
                1403,
                1412,
                "PERSON"
            ],
            [
                1422,
                1432,
                "PERSON"
            ],
            [
                1444,
                1450,
                "PERSON"
            ],
            [
                1463,
                1471,
                "PERSON"
            ],
            [
                1538,
                1543,
                "PERSON"
            ],
            [
                1564,
                1573,
                "PERSON"
            ],
            [
                1587,
                1593,
                "PERSON"
            ],
            [
                1646,
                1654,
                "PERSON"
            ],
            [
                1671,
                1680,
                "PERSON"
            ],
            [
                1723,
                1730,
                "PERSON"
            ],
            [
                1752,
                1763,
                "PERSON"
            ],
            [
                1782,
                1790,
                "PERSON"
            ],
            [
                1824,
                1831,
                "PERSON"
            ],
            [
                1867,
                1874,
                "PERSON"
            ],
            [
                1901,
                1908,
                "PERSON"
            ],
            [
                1945,
                1952,
                "PERSON"
            ],
            [
                1963,
                1969,
                "PERSON"
            ],
            [
                2009,
                2014,
                "PERSON"
            ],
            [
                2027,
                2036,
                "PERSON"
            ],
            [
                2067,
                2073,
                "PERSON"
            ],
            [
                2124,
                2130,
                "PERSON"
            ],
            [
                2153,
                2160,
                "PERSON"
            ],
            [
                2187,
                2193,
                "PERSON"
            ],
            [
                2225,
                2234,
                "PERSON"
            ],
            [
                2250,
                2257,
                "PERSON"
            ],
            [
                2302,
                2309,
                "PERSON"
            ],
            [
                2321,
                2327,
                "PERSON"
            ],
            [
                2344,
                2352,
                "PERSON"
            ],
            [
                2397,
                2405,
                "PERSON"
            ],
            [
                2410,
                2420,
                "PERSON"
            ],
            [
                2439,
                2449,
                "PERSON"
            ],
            [
                2468,
                2475,
                "PERSON"
            ],
            [
                2515,
                2522,
                "PERSON"
            ],
            [
                2529,
                2536,
                "PERSON"
            ],
            [
                2551,
                2557,
                "PERSON"
            ],
            [
                2567,
                2576,
                "PERSON"
            ],
            [
                2639,
                2646,
                "PERSON"
            ],
            [
                2667,
                2673,
                "PERSON"
            ],
            [
                2682,
                2688,
                "PERSON"
            ],
            [
                2695,
                2702,
                "PERSON"
            ],
            [
                2720,
                2729,
                "PERSON"
            ],
            [
                2753,
                2763,
                "PERSON"
            ],
            [
                2800,
                2806,
                "PERSON"
            ],
            [
                2855,
                2861,
                "PERSON"
            ],
            [
                2873,
                2882,
                "PERSON"
            ],
            [
                2916,
                2922,
                "PERSON"
            ],
            [
                2926,
                2933,
                "PERSON"
            ],
            [
                3004,
                3008,
                "PERSON"
            ],
            [
                3042,
                3049,
                "PERSON"
            ],
            [
                3062,
                3068,
                "PERSON"
            ],
            [
                3090,
                3098,
                "PERSON"
            ],
            [
                3127,
                3135,
                "PERSON"
            ],
            [
                3225,
                3233,
                "PERSON"
            ],
            [
                3240,
                3246,
                "PERSON"
            ],
            [
                3281,
                3288,
                "PERSON"
            ],
            [
                3305,
                3312,
                "PERSON"
            ],
            [
                3339,
                3345,
                "PERSON"
            ],
            [
                3358,
                3364,
                "PERSON"
            ],
            [
                3383,
                3389,
                "PERSON"
            ],
            [
                3406,
                3413,
                "PERSON"
            ],
            [
                3445,
                3452,
                "PERSON"
            ],
            [
                3469,
                3476,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult h { Anisa ospital white, tyrone 1211 medica { Sarita l  { Jesica center dr. mrn: { Kya  047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 9/9 { Lanette /2024, d/c: 9/10/2 { Darline 024 09/09/2024 - ed in vanderbilt emergency department (continued) medication list (continued) instructions: after surgery, use 1 drop to the r { Reilly igh { Hayward t eye every 2 hours while awake until bedtime. beginning the next day, decrease to 1 drop to the right eye 4 tim { Randy es a day for 1 week, then 3 times a day for 1 week, then 2 times a day for 1 week, then daily for 1 { Breonna  week, then stop authorized by: valenzuela, da { Ayaan niel alejandro, md ordered on: 7/22/2024 start date: 7/22/2024 quantity: 5 ml refill: 1 refill by 7/22/2025 moxi { Dallin floxacin 0.5 %  { Nyasia eye d { Lylah rops (vigamox) instructions: after surgery, use 1 drop to the right eye every 2 hours while awake until bedtime. beginning the next day, decrease to 1 drop to the right eye 4  { Jeromy times a day for 1 week, then stop authorized by: valenzuela, daniel ale { Afton jandro, md ordered on: 7/22/2024 start date: 7/22/20 { Leslee 24 quantity: 3 ml refill: 1 refill by 7/22/2025 ketorolac 0.4 eye drops (acular) discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason f { Patricia or discontinuation: duplicate order in { Kalli structions: after surgery, use 1 d { Yousef rop to the right eye every 2 hours while awake until bedtime. beginni { Leisa ng the next day, d { Vienna ecrease to 1 drop t { Glynn o the right { Chiara  eye 4 times  { Sabra a day for 2 weeks then stop authorized by: valenzuela, daniel alejandro, md ordered on: 7/22/2024 start date: 7/22/2024 end date { Konner : 9/16/2024 quantity: 5 ml refill: 1 refill b { Shanon y 7/22/20 { Wren 25 clotrimazole 1 % topical cream (lotrimin) instru { Santana ction { Eleazar s: apply 1 application topically 2 times a day for 30 days. authorized by: pauw, emily ka { Windy thryn, { Kingsley  md ordered on: 7/26/2024 start date: 7/26/2024 quantity: 28 g refill:  remaining albuterol { Noa  sulfate hfa 90 mcg/actuation { Frances  aerosol inhaler instructions: inhal { Malka e 2 puffs every 4 hours a { Madisen s needed for wheezing. aut { Yahaira horized by: chakravarthy, rohini, md ordered on: 8/6/ { Magaly 2024 start date: 8/6/2024 quantity: 18 g refill: 11 refills by 8/6/2025 fluticasone propionate 50 mcg/actuation nasal spray,suspension (flonase) instructions: administer 2  { Evans sprays into each nostril 2 times a day. authori { Nada zed by: connoll { Avianna y, patr { Immanuel ick james, md ordered on: 8/6/2024 start date: 8/6/2024 quantity: 16 g { Denton  refill: 2 refills by 8/6/2025 gabapentin 300 mg ca { Jaeden psule (neurontin) instructions: take 2 capsules (600 mg total) by mouth daily.  { Nyah authorized by: chakravarthy, rohini, md ordered o { Antione n: 8/6/2024 s { Marvel tart date: 8/6/2024 quantity: 180 cap { Tyshawn sule refill:  remaining ipratropium bromide 42 mcg (0.06 ° nasal spray (atrovent) instructions: administer 1 spray into each nostril 4 time { Arian s a day. start with 1 spray once daily f { Christene or one week, then incre { Paityn as { Zaniyah e to 2 sprays once a day for 2 weeks, { Brandyn  can increase t { Konnor o 3-4 sprays a day authorized by: chakravarthy, rohini, md ordered on: 8/6/2024 start date: 8/6/2024 quantity: 15 ml refill: 12 refills by 8/6/2025 ketorolac 0.5 % eye drops (acular) instructions: administer 1 drop into the left eye 4 times a day for 28 days. authorized by: brown, sarah karee, md ordered on: 8/23/2024 start date: 8/23/2024 end date: 9/20/2024 quantity: 5 ml refill:  remaining printed on 10/3/24 7:12 am page 47,vumc adult hospital white, tyrone 1211 me { Rubie dical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 9/9/2024, d/c: 9/10/2024 09/09/2024 - ed in van { Arther derbilt emergency { Fran  department (continued) medication list (continued) prednisolon { Alda i acetate 1 % eye drops,suspension (pred forte) discontinued by: gingrow, barbara, lpn discontinued on:  { Gayle 9/16/2024 reason for discontinuation: duplicate order instructions: administer 1 drop into the left eye 4 times a day for 7 days, then 1 drop 3 times a day for 7 days, then 1 drop 2 times a day for 7 days, then 1 drop daily for 7 days. authorized by: brown, sarah karee, md ordered on: 8/23/2024 start date: 8/23/2024 end da { Avah te: 9/16/2024 quantity: 10 ml refill:  remaining loperamide 2 mg capsu { Xochitl le (imodium) instructions: take 1 capsu { Taj le (2 { Albina  mg total) by mouth 3 times a day as needed for diarrhea fo { Racquel r up to 10 days. authorized by: chakravarthy, rohini, md ordered on: 9/3/2024 start date: 9/3/2024 quantity:  { Lashanda 30 capsu { Campbell le refill:  remaining lokelma 10 gram oral powder packet (sodium zirconium cyclosilicate) instructions: take 10  { Jerimiah g by mouth daily for 7 days. authorized by: freeman, genevieve clayton, aprn { Jaren  ordered on: 9/9/2024 start date: 9/9/2024 end date: 9/16/2024 quantity: 7 packet refill:  remaining doxycycline hyclate 100 mg capsule (vibramycin) discontinued by: gingrow, barbara, lpn discontinued on: 9/16 { Gisele /2024 reason for discontinuation: duplicate order instructions: take 1 capsule (100 mg total) by mouth every 12 { Jacque  hours for 5 days. authorized by: b { Norberto oaglio, sean michael, do  { Edie ordered on: 9/10/2024 sta { Gavyn rt date: 9/10/202 { Lanie 4 end date: 9/16/2024 quantity: { Dori  10 capsule re { Kadin fill:  remain { Nena ing stopped in visit none treatment team provider se { Ariyah rvice role provider team specialty from to boaglio, se { Kamilah an att { Shalonda ending { Shoshana  emergency 09/10/24  { Vilma 0147 09/10/24 0338 mic { Britni hael, do medicine ed notes ed triage notes by brusch, joan, rn at 9/9/20 { Taniya 24 { Alize  2201  { Caitlynn author: { Lyda  brusch, joan, rn service: - author type: registered nurse filed: 9/9/202 { Lesley 4 10:04 pm date of service: 9/9/2024 10:0 { Margarett 1 pm status: signed editor: brusch, joan, rn (registered nurse) pt a { Shanell mbulatory to triage with c/o \"boil on s { Casimir haft of penis\" for the past couple days with purulent drainage noted. assessment of area deferred { Wally   { Zina in triage. electronically signed by brusch, joan, rn a { Aditya t 9/9/2024 10:04 pm ed triage practitioner note by suszanski, juli { Geralyn an, md at  { Bethel 9/9/2024 2204 { Shirlene  auth { Donell or: suszanski, julian, md service: em { Brain ergency medicine au { Linden thor type: physician filed: 9/9/2024 10:04 pm date of service: 9/9/2024 10:04 pm status: signed editor: suszanski, julian, md (physician) printed on 10/3/24 7:12 am page 48",
    {
        "entities": [
            [
                15,
                21,
                "PERSON"
            ],
            [
                57,
                64,
                "PERSON"
            ],
            [
                69,
                76,
                "PERSON"
            ],
            [
                94,
                98,
                "PERSON"
            ],
            [
                174,
                182,
                "PERSON"
            ],
            [
                203,
                211,
                "PERSON"
            ],
            [
                357,
                364,
                "PERSON"
            ],
            [
                370,
                378,
                "PERSON"
            ],
            [
                493,
                499,
                "PERSON"
            ],
            [
                601,
                609,
                "PERSON"
            ],
            [
                658,
                664,
                "PERSON"
            ],
            [
                779,
                786,
                "PERSON"
            ],
            [
                804,
                811,
                "PERSON"
            ],
            [
                819,
                825,
                "PERSON"
            ],
            [
                1003,
                1010,
                "PERSON"
            ],
            [
                1084,
                1090,
                "PERSON"
            ],
            [
                1145,
                1152,
                "PERSON"
            ],
            [
                1310,
                1319,
                "PERSON"
            ],
            [
                1360,
                1366,
                "PERSON"
            ],
            [
                1403,
                1410,
                "PERSON"
            ],
            [
                1482,
                1488,
                "PERSON"
            ],
            [
                1509,
                1516,
                "PERSON"
            ],
            [
                1538,
                1544,
                "PERSON"
            ],
            [
                1558,
                1565,
                "PERSON"
            ],
            [
                1581,
                1587,
                "PERSON"
            ],
            [
                1718,
                1725,
                "PERSON"
            ],
            [
                1773,
                1780,
                "PERSON"
            ],
            [
                1792,
                1797,
                "PERSON"
            ],
            [
                1851,
                1859,
                "PERSON"
            ],
            [
                1867,
                1875,
                "PERSON"
            ],
            [
                1967,
                1973,
                "PERSON"
            ],
            [
                1982,
                1991,
                "PERSON"
            ],
            [
                2085,
                2089,
                "PERSON"
            ],
            [
                2121,
                2129,
                "PERSON"
            ],
            [
                2168,
                2174,
                "PERSON"
            ],
            [
                2202,
                2210,
                "PERSON"
            ],
            [
                2239,
                2247,
                "PERSON"
            ],
            [
                2303,
                2310,
                "PERSON"
            ],
            [
                2485,
                2491,
                "PERSON"
            ],
            [
                2541,
                2546,
                "PERSON"
            ],
            [
                2564,
                2572,
                "PERSON"
            ],
            [
                2582,
                2591,
                "PERSON"
            ],
            [
                2664,
                2671,
                "PERSON"
            ],
            [
                2725,
                2732,
                "PERSON"
            ],
            [
                2814,
                2819,
                "PERSON"
            ],
            [
                2871,
                2879,
                "PERSON"
            ],
            [
                2895,
                2902,
                "PERSON"
            ],
            [
                2942,
                2950,
                "PERSON"
            ],
            [
                3092,
                3098,
                "PERSON"
            ],
            [
                3141,
                3151,
                "PERSON"
            ],
            [
                3177,
                3184,
                "PERSON"
            ],
            [
                3189,
                3197,
                "PERSON"
            ],
            [
                3237,
                3245,
                "PERSON"
            ],
            [
                3263,
                3270,
                "PERSON"
            ],
            [
                3745,
                3751,
                "PERSON"
            ],
            [
                3892,
                3899,
                "PERSON"
            ],
            [
                3919,
                3924,
                "PERSON"
            ],
            [
                3990,
                3995,
                "PERSON"
            ],
            [
                4102,
                4108,
                "PERSON"
            ],
            [
                4435,
                4440,
                "PERSON"
            ],
            [
                4513,
                4521,
                "PERSON"
            ],
            [
                4563,
                4567,
                "PERSON"
            ],
            [
                4575,
                4582,
                "PERSON"
            ],
            [
                4644,
                4652,
                "PERSON"
            ],
            [
                4764,
                4773,
                "PERSON"
            ],
            [
                4784,
                4793,
                "PERSON"
            ],
            [
                4908,
                4917,
                "PERSON"
            ],
            [
                4996,
                5002,
                "PERSON"
            ],
            [
                5214,
                5221,
                "PERSON"
            ],
            [
                5335,
                5342,
                "PERSON"
            ],
            [
                5380,
                5389,
                "PERSON"
            ],
            [
                5417,
                5422,
                "PERSON"
            ],
            [
                5450,
                5456,
                "PERSON"
            ],
            [
                5476,
                5482,
                "PERSON"
            ],
            [
                5516,
                5521,
                "PERSON"
            ],
            [
                5538,
                5544,
                "PERSON"
            ],
            [
                5560,
                5565,
                "PERSON"
            ],
            [
                5620,
                5627,
                "PERSON"
            ],
            [
                5684,
                5692,
                "PERSON"
            ],
            [
                5701,
                5710,
                "PERSON"
            ],
            [
                5719,
                5728,
                "PERSON"
            ],
            [
                5751,
                5757,
                "PERSON"
            ],
            [
                5782,
                5789,
                "PERSON"
            ],
            [
                5864,
                5871,
                "PERSON"
            ],
            [
                5876,
                5882,
                "PERSON"
            ],
            [
                5891,
                5900,
                "PERSON"
            ],
            [
                5910,
                5915,
                "PERSON"
            ],
            [
                5991,
                5998,
                "PERSON"
            ],
            [
                6042,
                6052,
                "PERSON"
            ],
            [
                6123,
                6131,
                "PERSON"
            ],
            [
                6173,
                6181,
                "PERSON"
            ],
            [
                6281,
                6287,
                "PERSON"
            ],
            [
                6291,
                6296,
                "PERSON"
            ],
            [
                6353,
                6360,
                "PERSON"
            ],
            [
                6429,
                6437,
                "PERSON"
            ],
            [
                6450,
                6457,
                "PERSON"
            ],
            [
                6473,
                6482,
                "PERSON"
            ],
            [
                6490,
                6497,
                "PERSON"
            ],
            [
                6537,
                6543,
                "PERSON"
            ],
            [
                6565,
                6572,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. mrn: 04771 { Desire 7361, d { Ouida ob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 6/2/2023,  { Margarito d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt university adult hospital (continued) { Tresa  flow { Carleton sheets (continued) 06/03/23 1013 06/03/23 1248 fric { Remy tion and   shear problem -eb at problem  { Brionna -sh at 06/03/23  { Tyesha 1013 06/03/23 1248 braden scale 20 -eb at 06/03/23 1013 20 - { Shad sh at 06/03/23 1 { Rylan 248 score braden ri { Sylas sk standard risk -eb at standard risk -sh at level 06/03/23 1013  { Jaimee 06/03/23 1248 activity/musculosk { Mellisa eletal all activity/musculos oel -eb at 06/03/23 oel -s { Teagan h at 06/03/23 keletal 1013 1248 assessment act { Ameer ivity/mus { Gil culos - mob { Lavon ility impairment mobilit { Austin y impairme { Evelina nt keletal problem -sh { Aaden  at 06/03/23 1248 abnormal { Janey  gait -eb at 06/03/23 general { Bilal i { Alyssia zed { Casper  noted -eb at 06/03/23 noted -sh at 06/03/23 weakness 1013 1248 strength - { Lawerence  yes -sh at 06/03/23 alteration? 1248 rue strength 5/5 movement 5/5 movement against gravit { Flynn y with against gravity w { Lafayette ith { Estefany  full r { Jevon esistance -eb at full resis { Rylee tance -sh at 06/03/23 101 { Emogene 3 06/03/23 1248 lue stren { Eugenio gth 5/5 movement 5/5 movement against gravity with against gravity wi { Keara th full resistance -eb at f { Brian ull resistance -sh at 06/03/23 1013 06/03/23 1248 rle strength 4/5 movement 4/5 movement against gravity with against gravity with some resistan { Dorinda ce -eb some resistance -sh at 06/03/23 1013 at 06/03/23 1248 lle strength 4/5 movement 4/5 movement against { Aundrea  gravity with against gravity with some resistance - { Esme eb some resistance -sh { Slade  at 06/03/23 1013 at 06/03/23 1248 positioning self-reposi { Jai t { Madaline ioning self-repositioning inter { Amia vention eb at 06/03/23 1013 sh at 06/03/23 1248 j hopkins (6) walk 10 { Everleigh +  { Wes steps (6) walk 10+ steps highest mo { Tanika bility -eb at 06/03/23 1013 -sh at 06 { Deric /03/23 1248 lvl evaluation mobility walk 10+ steps -eb walk 10+ steps -sh intervention at 06/03/23 1013 at 06/03/23 1248 level of mobility standby assist -eb  { Meyer standby assist -sh ass { Britt istance at 06/03/23 1013 at 06/03/23 1248 needed fluid/nutrition all fluids/electrolyte wel -eb at 06/03/23 wel -sh at 06/03/23 assessment 1013 1248 flui { Jesenia d therapy hydration  { Remington -eb at hydration -sh at inte { Rueben rventions 06/03/23 1013 06/03/2 { Cherise 3 1248 nutrition wnl -eb at 06/03/23  { Alford wnl -sh at 06/03/23 assessment 1013 1248 food intak { Alexander e breakfast -eb at - 06/03/23 1013 safety all (alcohol/drug, fall, elopement, restraint, self { Norene -mutilation, suicide, violence) falls risk/safety wel -eb at 06/03/23 wel -sh  { Lorinda at 06/03/23 risk assessment 1013 1248 broset? does this patient yes  { Melia -eb at 06/03/23 yes -sh at 06/03/23 require a broset 1013 1248 assessment? broset violence checklist printed on 10/3/24 7:13 am page 2317,vumc adult hospital white, tyrone 1211  { Hernan medical center dr. mrn: 047717361,  { Dorotha dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 6/2/2023, d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt university adult hosp { Jeramy ital (continued) flowsheets (continued) violence { Brannon  risk  threat  threat eb at 06/03/23 101 { Anastacia 3 sh at 06/03/23 1248 confused not present -eb at n { Enrico ot present -s { Rasheed h at 06/03/23 1013 06/03/23 1248 irritable not present -eb at not present -sh at { Kendyl  06/03/23 1013 06/03/23 1248 boiste { Austyn rous not present -eb at not present -sh at 06/03/23 1013 06/03/23 1248 verbally not pres { Antonella ent -eb at not present  { Lorin -sh at 06/03/23 1013 06/03/23 1248 physically not present -eb at not present -sh at threatening 06/03/23 1013 06/03/23 1248 attacking objects - not present -eb at not  { Shanelle present -sh at 06/03/23 1013 06/03/23 1248 broset sum - 0 -e { Westin b at 06/03/23 1013 0  { Jaci -sh at 06/03/23 1248 nursing: if score  { Jessy 2/>, start de- escalation. jhfrat fall risk if patient  { Andi does not jh { Alyvia frat -eb at  { Yamileth jhfrat { Emani  -sh at meet either of the 06/03/23 1013 06/03/23 1248 above two criteria, please use tool age <60 years -eb at <60 years -sh at 06/03/23 1013 06/03/23 1248 fall history  fall  fall within the last 6 within the last 6  { Waymon months -eb at 06/03/23 months -sh at { Celena  1013 06/03/23 1248 { Jalyn  elimination,   bowel and urine p { Iyana roblems -eb at problems -sh at 06/03/23 1013 06/03/23 1248 medications (see on 1 high fall risk on  { Anisha 1 high fall risk row information) drug -eb  { Kolten at 06/03/23 drug -sh  { Myranda at 06/03/23 1013 1248 patient care one present -eb at one present -sh at equipment (see 06/03/23 1013 06/03/23 1248 row informati { Steven on) mobility -choose requires assistance re { Fermin quires assistance all that apply or supervision for or supervisio { Alora n for mobility, transfer, or mobility, transfer, or ambulation; unstead ambulation; unstead y gait -eb at 06/0 { Giles 3/23 y gait -sh at 06/03/23 1013 1248 cognition   choose all that impair { Kendell ment -eb at impa { Neha irment -sh at apply 06/03/23 1013 06/03/23 1248 total fall risk 8 -eb at 0 { Fisher 6/03/23 1013 8 -sh at 06/03/23 1248 score fall risk moderate fall risk { Jerrica  moderate fall risk -eb at 06/03/23 1013 -sh at 06/03/23 1248 medic { Santo ation all medication wel -eb at 06/03/23 wel -sh at 06/0 { Mikael 3/23 assessment 1013 { Roni  1248 medication medication risk -eb problem  { Cataleya at 06/03/23 1013 self care (adl { Gareth ) all self care wnl -eb at 0 { Ryne 6/03/23 we { Yaakov l -sh at 06/03/23 assessm { Gearldine ent 1013 1248 infection/metabolic all infection/metaboli - wel -eb at 06/03/23 wel -sh at 06/03/23 c assessment 1013 1248 { Jasen  printed on 10/3/24 { Ferne  7:13 am page 2318",
    {
        "entities": [
            [
                71,
                78,
                "PERSON"
            ],
            [
                88,
                94,
                "PERSON"
            ],
            [
                164,
                174,
                "PERSON"
            ],
            [
                289,
                295,
                "PERSON"
            ],
            [
                303,
                312,
                "PERSON"
            ],
            [
                366,
                371,
                "PERSON"
            ],
            [
                414,
                422,
                "PERSON"
            ],
            [
                441,
                448,
                "PERSON"
            ],
            [
                511,
                516,
                "PERSON"
            ],
            [
                535,
                541,
                "PERSON"
            ],
            [
                563,
                569,
                "PERSON"
            ],
            [
                637,
                644,
                "PERSON"
            ],
            [
                679,
                687,
                "PERSON"
            ],
            [
                745,
                752,
                "PERSON"
            ],
            [
                801,
                807,
                "PERSON"
            ],
            [
                819,
                823,
                "PERSON"
            ],
            [
                837,
                843,
                "PERSON"
            ],
            [
                870,
                877,
                "PERSON"
            ],
            [
                890,
                898,
                "PERSON"
            ],
            [
                923,
                929,
                "PERSON"
            ],
            [
                958,
                964,
                "PERSON"
            ],
            [
                996,
                1002,
                "PERSON"
            ],
            [
                1006,
                1014,
                "PERSON"
            ],
            [
                1020,
                1027,
                "PERSON"
            ],
            [
                1104,
                1114,
                "PERSON"
            ],
            [
                1208,
                1214,
                "PERSON"
            ],
            [
                1241,
                1251,
                "PERSON"
            ],
            [
                1257,
                1266,
                "PERSON"
            ],
            [
                1276,
                1282,
                "PERSON"
            ],
            [
                1312,
                1318,
                "PERSON"
            ],
            [
                1346,
                1354,
                "PERSON"
            ],
            [
                1382,
                1390,
                "PERSON"
            ],
            [
                1462,
                1468,
                "PERSON"
            ],
            [
                1498,
                1504,
                "PERSON"
            ],
            [
                1651,
                1659,
                "PERSON"
            ],
            [
                1769,
                1777,
                "PERSON"
            ],
            [
                1832,
                1837,
                "PERSON"
            ],
            [
                1862,
                1868,
                "PERSON"
            ],
            [
                1929,
                1933,
                "PERSON"
            ],
            [
                1937,
                1946,
                "PERSON"
            ],
            [
                1980,
                1985,
                "PERSON"
            ],
            [
                2057,
                2067,
                "PERSON"
            ],
            [
                2072,
                2076,
                "PERSON"
            ],
            [
                2114,
                2121,
                "PERSON"
            ],
            [
                2161,
                2167,
                "PERSON"
            ],
            [
                2328,
                2334,
                "PERSON"
            ],
            [
                2359,
                2365,
                "PERSON"
            ],
            [
                2521,
                2529,
                "PERSON"
            ],
            [
                2552,
                2562,
                "PERSON"
            ],
            [
                2593,
                2600,
                "PERSON"
            ],
            [
                2634,
                2642,
                "PERSON"
            ],
            [
                2682,
                2689,
                "PERSON"
            ],
            [
                2743,
                2753,
                "PERSON"
            ],
            [
                2849,
                2856,
                "PERSON"
            ],
            [
                2937,
                2945,
                "PERSON"
            ],
            [
                3016,
                3022,
                "PERSON"
            ],
            [
                3202,
                3209,
                "PERSON"
            ],
            [
                3247,
                3255,
                "PERSON"
            ],
            [
                3422,
                3429,
                "PERSON"
            ],
            [
                3480,
                3488,
                "PERSON"
            ],
            [
                3531,
                3541,
                "PERSON"
            ],
            [
                3595,
                3602,
                "PERSON"
            ],
            [
                3618,
                3626,
                "PERSON"
            ],
            [
                3709,
                3716,
                "PERSON"
            ],
            [
                3754,
                3761,
                "PERSON"
            ],
            [
                3852,
                3862,
                "PERSON"
            ],
            [
                3888,
                3894,
                "PERSON"
            ],
            [
                4064,
                4073,
                "PERSON"
            ],
            [
                4136,
                4143,
                "PERSON"
            ],
            [
                4167,
                4172,
                "PERSON"
            ],
            [
                4214,
                4220,
                "PERSON"
            ],
            [
                4278,
                4283,
                "PERSON"
            ],
            [
                4297,
                4304,
                "PERSON"
            ],
            [
                4319,
                4328,
                "PERSON"
            ],
            [
                4337,
                4343,
                "PERSON"
            ],
            [
                4564,
                4571,
                "PERSON"
            ],
            [
                4610,
                4617,
                "PERSON"
            ],
            [
                4639,
                4645,
                "PERSON"
            ],
            [
                4681,
                4687,
                "PERSON"
            ],
            [
                4789,
                4796,
                "PERSON"
            ],
            [
                4842,
                4849,
                "PERSON"
            ],
            [
                4873,
                4881,
                "PERSON"
            ],
            [
                5013,
                5020,
                "PERSON"
            ],
            [
                5066,
                5073,
                "PERSON"
            ],
            [
                5141,
                5147,
                "PERSON"
            ],
            [
                5260,
                5266,
                "PERSON"
            ],
            [
                5341,
                5349,
                "PERSON"
            ],
            [
                5368,
                5373,
                "PERSON"
            ],
            [
                5450,
                5457,
                "PERSON"
            ],
            [
                5530,
                5538,
                "PERSON"
            ],
            [
                5608,
                5614,
                "PERSON"
            ],
            [
                5673,
                5680,
                "PERSON"
            ],
            [
                5703,
                5708,
                "PERSON"
            ],
            [
                5756,
                5765,
                "PERSON"
            ],
            [
                5799,
                5806,
                "PERSON"
            ],
            [
                5837,
                5842,
                "PERSON"
            ],
            [
                5855,
                5862,
                "PERSON"
            ],
            [
                5890,
                5900,
                "PERSON"
            ],
            [
                6024,
                6030,
                "PERSON"
            ],
            [
                6052,
                6058,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hundred oaks white, tyrone 719 thompson  { Maxie lane, nashville mrn: 047717361, do { Brandon b: 7/27/1969, legal sex: m nashville tn 37204 visit date: 9/7/2023 09/07/2023 - office visit in vanderbilt one hundred oaks primary care north (continued) communication tracking (continued) calls/messages text mes { Arnoldo sage on 9/7/2023 1140 phone number: 615-733-5206 message: you have an appointment scheduled at 1:40 pm cdt. when you arrive at the clinic, open your myhealth at vanderbilt app and click \"i am here.\" this will be availab { Zaid le 30 minutes prior to your appointment time. text { Glendon   { Elizebeth message on 9/7/2023 1240 relation: other phone number: 615-430 { Delton -4331 messa { Reynold ge: you hav { Ronna e an appointment scheduled at  { Cassondra 1 { Erlinda :40 pm cdt. when you arrive at the clinic, open your myhealth at vanderbilt  { Frank app and click \"i am here.\" this will be avail { Shani able 30 m { Lucious inutes p { Rian rior to your appointment time. medication list medication list i this report is for documentation purposes on { Smith ly. the patient should not follow medication instructions within. for accura { Mark te instructions regarding medications, the patient should instea { Bridger d consult their physician or after visit summary. active at the  { Jewell end of v { Unnamed isit medications last reviewed by dekorte, davita on 9/7/2023 1424 docusate sodium 100 mg capsule  { Bernita (colace) discontinued by: de witte, ant { Milana on jordan, md discontinued on: 12/6/2023 reason for discontin { Braulio uatio { Tyrus n: cleanup(notavs) instructions: take one tablet tid prn constipation authorized by: lippard, giles a, aprn ordered on: 10 { Kaeden /25/20 { Beverly 22 start date: 10/25/2022 end date: 12/6/2023 quantity: 60 capsule refill:  remaining pantoprazole 20 mg tablet,delayed release (protonix) disco { Carin ntinued by: greenspan, debra l, aprn disc { Tasia ontinued on: 7/ { Abdiel 9/2024 reason for discontinuation: reorder instructions: take 1 tablet (20 mg total) by mouth daily. authorized by: lippard, giles a, aprn ordered on: 11/29/2022 start date: 11/29/2022 end date: 7/9/2024 quantity: 30 tablet refill: 11 refills by 11/29/2023 { China   { Tripp fam { Carnell otidine 20 mg tablet  { Makaila (pepcid) [reconciled by ferguson, sherri l, lpn on 1/11/ { Analia 2023 1257] instructions: take 1 tablet (20 mg total) by mouth every 12 hours. entered by: ferguson, sher { Cinda ri l, lpn entered on: 1/11/2023 acetaminophen 325 mg table { Adalberto t (tylenol) instruct { Allegra ions: take 2 tablets (650 mg total) by mouth every 6 hours as need { Florene ed for mild pain, moderate pain, headaches or fever. authorized by: lehman { Kamille n, melissa cary, { Blythe  pa-c ordered on: 6/3/2023  { Dorian start date: 6/3/2023 end date: 3/5/2024 action: p { Holland atient n { Kendal ot takin { Alethea g quantity: 30 tablet refill:  remain { Violette ing trulicity 0.75 mg/0.5 ml subcutaneous pen injector { Mindi  (dula { Eric glutide) discontinued by: greenspan, de { Karri bra l, aprn discontinued on: 7/10/2024 { Shiela  reason for discontin { Kalen uation: other (cancelrx) instructions: inject 0.75 mg under the skin ever { Daisha y 7 days.  { Natali authorized by: lipp { Shasta ar { Henri d, { Nate  giles a, aprn ordered on: 8/ { Cristin 7/2023 start date: 8/7/2023 end date: 7/10/2024 quantity: 6 ml refill: 3 refills by 8/6/2024 albutero { Linus l sulfate hfa 90 mcg/actuation aerosol inhaler discontinued by: chakravarthy, rohini, md discontinued on: 8/6/2024 printed on 10/3/24 7:13 am page { Christophe   { Ajay 1731,vumc adult one hundred oaks white, tyrone 719 thompson lane, nashville mrn: 047717361, dob { Malika : 7/27/1969, legal sex: m nashville { Iyanna  tn 37204 visit date: 9/7/2023 09/07/2023 - office visit  { Gibson in vanderbilt one hundred oaks primary care north (continued) medication list (continued) reason for discontinuation: reorder instructions: inhale 2 puffs every 4 hours as needed for wheezing. au { Kalia thorized by: lippard, giles a, aprn ordered on: 8/8/2023 start { Deasia  date: 8/8/2023 quantity:  { Jeramiah 18 g refill: 11 refill { Hilario s by 8/7/2024 atorvastatin 80 mg tablet (lipitor) discontinued by: mickey, lisa, lpn disconti { Alaysia nued on: 10/2/2024 instruc { Vergie tions: take 1 tablet (80 mg total) by mouth daily. aut { Aaron horized by: lippard, g { Everardo iles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 refills by 8/7/2024 azelastine 137 mcg ( { Audriana 0.1 %) nasal spray aeroso { Shay l (astelin) discontinued by: de witte, anton jordan, md discontinued on: 10/17/2023 reason for discontinuation: cleanup(notavs) inst { Hali ructions: administer 1 spray i { Keli nto  { Kailani each  { Iman n { Denisha ostril 2 times a day. use in each nostril as directed authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start dat { Pricilla e: 8/8/2023 quantity: 30 ml refill: 12 refills by 8/7/20 { Barron 24 cetirizine 10 mg tablet (zyrtec) discontinued by: mickey { Daryl , lisa, lpn discontinued on: 1 { Lennie 0/4/2023  { Alida reason for discontinuation: reorder instructions: take 1 tablet (10 mg total) by mouth once a day as needed for allergies. author { Samatha ized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 qu { Timothy antity: 30 tablet refill: 11 refills by 8/7/2024 monteluk { Leana ast 10 mg tablet (singulair) instr { Jaxen uctions: take 1 tablet (10 mg total) by mouth every even { Henry ing. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 refills { Marcell  by 8/7/2024 nifedipine er 30 mg tablet,extended release (adalat cc) discontinued by: de witte, anton jordan, md discontinued on: 4/23/2024 reason for discontinuation: reorder instructions: take 1 tab { Jalynn let (30 mg total) by mouth daily. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 refills by 8/7/2024 insulin glargine (u-100) 100 unit/ml subcutaneous solution discontinued by: de witte, anton jordan, md discontinued on: 10/20/2023 reason for di { Dannielle scontinuati { Cleve on: reorder instructions: inject 0.05 ml (5 units  { Elliot total) under the skin daily. authorized by: lippard, giles a, apr { Kalani n ordered on: 8/24/2023 start date: 8/24/2023 quantity: 4.5 ml refill:  re { Jolynn maining cyclobenzaprine 5 mg tablet (f { Abelardo lexeril) [reconciled by maples, chantis on 9/5/2023 1522] instructions: take 1 tablet (5 mg total) by mouth every 8 hours as needed. entered by { Dusty : maples, chantis entered on: 9/5/2023 start date: 7/22/2023 triamcinolone acetonide 55 mcg nasal spray aerosol (nasacort) discontinued { Dedra  by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason fo { Landry r discontinuation: duplicate  { Winford order instructions: administer 2 sprays (110 mcg total) into each nostril 2 times a day. printed on 10/3/24 { Paulo  7:13 am page 1732",
    {
        "entities": [
            [
                58,
                64,
                "PERSON"
            ],
            [
                101,
                109,
                "PERSON"
            ],
            [
                325,
                333,
                "PERSON"
            ],
            [
                555,
                560,
                "PERSON"
            ],
            [
                613,
                621,
                "PERSON"
            ],
            [
                625,
                635,
                "PERSON"
            ],
            [
                700,
                707,
                "PERSON"
            ],
            [
                721,
                729,
                "PERSON"
            ],
            [
                743,
                749,
                "PERSON"
            ],
            [
                782,
                792,
                "PERSON"
            ],
            [
                796,
                804,
                "PERSON"
            ],
            [
                883,
                889,
                "PERSON"
            ],
            [
                937,
                943,
                "PERSON"
            ],
            [
                955,
                963,
                "PERSON"
            ],
            [
                974,
                979,
                "PERSON"
            ],
            [
                1091,
                1097,
                "PERSON"
            ],
            [
                1176,
                1181,
                "PERSON"
            ],
            [
                1248,
                1256,
                "PERSON"
            ],
            [
                1323,
                1330,
                "PERSON"
            ],
            [
                1341,
                1349,
                "PERSON"
            ],
            [
                1450,
                1458,
                "PERSON"
            ],
            [
                1500,
                1507,
                "PERSON"
            ],
            [
                1571,
                1579,
                "PERSON"
            ],
            [
                1587,
                1593,
                "PERSON"
            ],
            [
                1718,
                1725,
                "PERSON"
            ],
            [
                1734,
                1742,
                "PERSON"
            ],
            [
                1889,
                1895,
                "PERSON"
            ],
            [
                1939,
                1945,
                "PERSON"
            ],
            [
                1963,
                1970,
                "PERSON"
            ],
            [
                2229,
                2235,
                "PERSON"
            ],
            [
                2239,
                2245,
                "PERSON"
            ],
            [
                2251,
                2259,
                "PERSON"
            ],
            [
                2283,
                2291,
                "PERSON"
            ],
            [
                2350,
                2357,
                "PERSON"
            ],
            [
                2464,
                2470,
                "PERSON"
            ],
            [
                2531,
                2541,
                "PERSON"
            ],
            [
                2564,
                2572,
                "PERSON"
            ],
            [
                2641,
                2649,
                "PERSON"
            ],
            [
                2726,
                2734,
                "PERSON"
            ],
            [
                2753,
                2760,
                "PERSON"
            ],
            [
                2790,
                2797,
                "PERSON"
            ],
            [
                2849,
                2857,
                "PERSON"
            ],
            [
                2868,
                2875,
                "PERSON"
            ],
            [
                2886,
                2894,
                "PERSON"
            ],
            [
                2934,
                2943,
                "PERSON"
            ],
            [
                3000,
                3006,
                "PERSON"
            ],
            [
                3015,
                3020,
                "PERSON"
            ],
            [
                3062,
                3068,
                "PERSON"
            ],
            [
                3109,
                3116,
                "PERSON"
            ],
            [
                3140,
                3146,
                "PERSON"
            ],
            [
                3222,
                3229,
                "PERSON"
            ],
            [
                3242,
                3249,
                "PERSON"
            ],
            [
                3271,
                3278,
                "PERSON"
            ],
            [
                3283,
                3289,
                "PERSON"
            ],
            [
                3294,
                3299,
                "PERSON"
            ],
            [
                3331,
                3339,
                "PERSON"
            ],
            [
                3443,
                3449,
                "PERSON"
            ],
            [
                3598,
                3609,
                "PERSON"
            ],
            [
                3613,
                3618,
                "PERSON"
            ],
            [
                3716,
                3723,
                "PERSON"
            ],
            [
                3761,
                3768,
                "PERSON"
            ],
            [
                3828,
                3835,
                "PERSON"
            ],
            [
                4033,
                4039,
                "PERSON"
            ],
            [
                4104,
                4111,
                "PERSON"
            ],
            [
                4140,
                4149,
                "PERSON"
            ],
            [
                4174,
                4182,
                "PERSON"
            ],
            [
                4278,
                4286,
                "PERSON"
            ],
            [
                4315,
                4322,
                "PERSON"
            ],
            [
                4379,
                4385,
                "PERSON"
            ],
            [
                4410,
                4419,
                "PERSON"
            ],
            [
                4547,
                4556,
                "PERSON"
            ],
            [
                4584,
                4589,
                "PERSON"
            ],
            [
                4724,
                4729,
                "PERSON"
            ],
            [
                4762,
                4767,
                "PERSON"
            ],
            [
                4774,
                4782,
                "PERSON"
            ],
            [
                4790,
                4795,
                "PERSON"
            ],
            [
                4799,
                4807,
                "PERSON"
            ],
            [
                4932,
                4941,
                "PERSON"
            ],
            [
                5000,
                5007,
                "PERSON"
            ],
            [
                5069,
                5075,
                "PERSON"
            ],
            [
                5108,
                5115,
                "PERSON"
            ],
            [
                5127,
                5133,
                "PERSON"
            ],
            [
                5265,
                5273,
                "PERSON"
            ],
            [
                5352,
                5360,
                "PERSON"
            ],
            [
                5420,
                5426,
                "PERSON"
            ],
            [
                5463,
                5469,
                "PERSON"
            ],
            [
                5528,
                5534,
                "PERSON"
            ],
            [
                5659,
                5667,
                "PERSON"
            ],
            [
                5870,
                5877,
                "PERSON"
            ],
            [
                6188,
                6198,
                "PERSON"
            ],
            [
                6212,
                6218,
                "PERSON"
            ],
            [
                6271,
                6278,
                "PERSON"
            ],
            [
                6346,
                6353,
                "PERSON"
            ],
            [
                6430,
                6437,
                "PERSON"
            ],
            [
                6478,
                6487,
                "PERSON"
            ],
            [
                6633,
                6639,
                "PERSON"
            ],
            [
                6777,
                6783,
                "PERSON"
            ],
            [
                6849,
                6856,
                "PERSON"
            ],
            [
                6888,
                6896,
                "PERSON"
            ],
            [
                7006,
                7012,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 visit date: 10/19/2023 10/19/2023 - communication in vanderbilt pharmacy retail serv { Crista ices (continued) medication list (continued) pantoprazole 20 mg tablet,delayed release (protonix) discontinued by: greenspan, debra l, aprn discontinued on: 7/9/2024 reas { Trevin o { Rina n for discontinuation: reorder instructions: take 1 tablet (20 mg  { Elliott total) by mouth daily. au { Carie thorized by: lippard, giles a, aprn ordered on: { Jeane  11/29/2022 start date: 11/29/2022 end  { Shreya date { Luigi : 7 { Elora /9/2024 quantity: 30 tablet  { Debrah refill: 11 refills by 11/29/2023 famo { Len tidine 20 mg t { Tawny ablet (pepcid) [reconciled by ferguson, sherri l, lpn on 1/11/2023 1257] instructions: take 1 tablet (20 mg total) { Anitra  by mouth every 12 hours. entere { Eulalia d by: ferguson, sherri l { Giavanna , lpn ent { Jaylen ered on: 1/11/2023 acetaminop { Zona hen 325 mg tablet (tylenol { Alessia ) instructions { Marko : take 2 tablets (650 mg total) by mouth every 6 hours as need { Janeth ed for mild pain, moderate pain, headaches or fever. authorized by: lehmann, melissa cary, pa-c ordered on: 6/3/2023 start date: 6/3/2023 end date: 3/5/2024 action: patient not taking quantity: 30 tablet refill:  remaining tr { Boris ulicity 0.75 mg/0.5 ml subcutaneou { Anjelica s pen injector (dulaglutide) discontinued by: greenspan, debr { Thor a l, aprn discontinued on: 7/10/2024 reason for discontinuation: other (cancelrx) instructions: { Zayn  in { Shiloh ject 0.75 mg unde { Kasie r the skin every 7 days. autho { Montserrat rized by: lippard, giles a, aprn ordered on: 8/7/2023 start date { Roel : 8/7/2023 end date: 7/10/2024 quantity: 6 ml refill: 3 refills by 8/6/2024 albuterol sulfate hfa 90 mcg/actuation { Chrissy  aerosol inhaler discontinued by: cha { Jovanny kravarthy, rohini, md discontinued on: 8/6/2024 reason for di { Mechelle scontinuation: reorder instructions: inhale 2 puffs every 4 hours as need { Maddie ed for wheezing. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 18 g refill: 11 refills by 8/7/2024 atorvastatin 80 mg tablet (lipitor) discontinued b { Jeremie y: mickey, lisa, lpn discon { Merrick tinued on: { Carolee  10/2/2024 instructions: take 1 tablet (80 mg total) by mouth daily. authorized  { Lavada by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 refills by 8/7/2024 montelukast 1 { Dominque 0 mg ta { Dickie blet (singulair) instructions: take 1 tablet (1 { Dedrick 0 mg total) by mouth every evening. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 refills by 8/7/2024 nifedipine er 30 mg tablet,extended release (adalat cc) discontinued by: de witte, anton { Stephaine  jorda { Dylon n, md discontinued on: 4/23/2024 reason for discontinuation { Maria : reorder  { Tremaine inst { Isreal ructions: take 1 tablet (30 mg total) by mouth daily. a { Bradyn uthorized b { Shmuel y: lippard, giles a, aprn ordered on: 8/8/2023 start d { Kenyatta ate: 8/8/ { Ethelyn 2023 quanti { Sami ty: 90 { Melony  tablet refill: 3 refills by 8 { Webster /7/2024 cyclobenzaprine 5 mg tablet (flexeril) [reco { Kirstie nciled by maples, chantis on 9/5/2023 1522] instructions: take 1 tablet (5 mg total) by mouth every 8 hours  { Hollis as needed. entered  { Azariah by: maples, chantis entered on: 9/5/2023 start date: 7/22/2023 printed on 10/3/24 7:13 am page 1587,vumc adult hospital white, tyrone 1211 medical center dr. mrn { Kaylan : 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 visit date: 10/19/2023 10/19/2023 - communicatio { Gauge n in v { Reilly and { Diya erbilt pharmacy retail se { Audrianna rvices (continued) medication list (continued) triamcin { Renea olone acetonide 55 mcg nasal spray aerosol (nasacort) discontinued by: gingrow, { Dwaine  barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate orde { Aliah r instructions: administer 2 sprays (110 mcg total) into each nost { Esta ril 2 times  { Franchesca a day. autho { Monet rized by: greenspan, debra l, aprn orde { Hasan red on: 9/5/2023 start date: 9/5/2023 end date: 9/16/2024 quantity: 16.5 g refill: 11 refills by 9/4/2024 losartan 25 mg tablet (cozaar) disc { Zenobia ontinued by: kovtun, roman, md discontinued on: 8/ { Melvyn 1/2024 { Terrill  reason for discontinuation: stop (cancelrx, on avs) instructions: take 1 tablet (25 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 end date { Jovany : 8/1/2024 quantity: 90 tablet refill: 3 refills by 9/6/2024 capsaicin 0.1 % topical cream discontinued by: gingrow, barbara, lpn discontinued on: 9/1 { Cherrie 6/2024 reason for disconti { Almeda nuati { Yamilet on: duplicate order instructions: apply  { Elna 1 application t { Evan opically daily for 90 days. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 end date: 9/16/2024 { Ottis  action: patie { Johann nt  { Krysten not taking quantity: 42.5 g refill:  remaining gabapentin 300 mg capsule (neurontin) discontinue { Kimberly d by: hock, richard lloyd, md discontinued on: 11/9/2023 reason for discontinuation: reorder instructions: take 1 caps { Mallorie ule (300 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 star { Myrtie t date: 9/7/2023 quantity: 90 capsul { Carrol e refill:  remaining aspirin 81 mg tablet,delayed release instructions: take 1 tablet (81 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 9/ { Zariyah 7/2023 start date: 9/7/2023 quantity: 90 tablet refill: 3 refills by 9/6/2024 cetirizine 10 mg tablet (zyrte { Albin c) instructions: take 1 tablet (10 mg total) by mouth once a day as needed for allergies. authorized by: de witte, { Calli  anton jordan, md ordered on: 10/4/20 { Ricki 23 start date: 10/4/ { Elin 2023 quantity: 30 t { Jericho ablet refill: 9 refills by 10/3/2024 fluticasone propionate 50 mcg/actuation nasal spray,suspension (flonase) [reconciled by dekorte, davita on 10/17/2023 1422] disconti { Herlinda n { Holley ued by: de witte, anton jordan, md discontinued on: 10/24/2023 reason for discontinuation: reorder entered by: dekorte, davita entered on: 10/17/2023 st { Ladarius art date: 9/13/2023 lidocaine 5 % topical patch (lidoderm) instructions: apply { Horacio  1 patch topic { Ellis ally daily. apply to painful area 12 hours per day, remove for 12 hours. authorized by: de witte, anton jordan, md ordered on: 10/17/2023 start  { Kynlee date: 10/17/2023 end date: 10/16/2024 quantity: 3 { Artie 0 patch  { Priscila refill: 11 refills by 10/16/2024 diclofenac 1 %  { Davonte topical gel discontinued by: c { Montgomery one, brittany discontinued on: 6/27/2024 prin { Liv ted on 10/3/24 7:13 am { Katalina  page 1588",
    {
        "entities": [
            [
                214,
                221,
                "PERSON"
            ],
            [
                394,
                401,
                "PERSON"
            ],
            [
                405,
                410,
                "PERSON"
            ],
            [
                479,
                487,
                "PERSON"
            ],
            [
                515,
                521,
                "PERSON"
            ],
            [
                571,
                577,
                "PERSON"
            ],
            [
                619,
                626,
                "PERSON"
            ],
            [
                633,
                639,
                "PERSON"
            ],
            [
                645,
                651,
                "PERSON"
            ],
            [
                682,
                689,
                "PERSON"
            ],
            [
                729,
                733,
                "PERSON"
            ],
            [
                750,
                756,
                "PERSON"
            ],
            [
                873,
                880,
                "PERSON"
            ],
            [
                915,
                923,
                "PERSON"
            ],
            [
                950,
                959,
                "PERSON"
            ],
            [
                971,
                978,
                "PERSON"
            ],
            [
                1010,
                1015,
                "PERSON"
            ],
            [
                1044,
                1052,
                "PERSON"
            ],
            [
                1069,
                1075,
                "PERSON"
            ],
            [
                1140,
                1147,
                "PERSON"
            ],
            [
                1375,
                1381,
                "PERSON"
            ],
            [
                1418,
                1427,
                "PERSON"
            ],
            [
                1491,
                1496,
                "PERSON"
            ],
            [
                1594,
                1599,
                "PERSON"
            ],
            [
                1605,
                1612,
                "PERSON"
            ],
            [
                1632,
                1638,
                "PERSON"
            ],
            [
                1671,
                1682,
                "PERSON"
            ],
            [
                1749,
                1754,
                "PERSON"
            ],
            [
                1871,
                1879,
                "PERSON"
            ],
            [
                1919,
                1927,
                "PERSON"
            ],
            [
                1991,
                2000,
                "PERSON"
            ],
            [
                2076,
                2083,
                "PERSON"
            ],
            [
                2279,
                2287,
                "PERSON"
            ],
            [
                2317,
                2325,
                "PERSON"
            ],
            [
                2338,
                2346,
                "PERSON"
            ],
            [
                2429,
                2436,
                "PERSON"
            ],
            [
                2571,
                2580,
                "PERSON"
            ],
            [
                2590,
                2597,
                "PERSON"
            ],
            [
                2647,
                2655,
                "PERSON"
            ],
            [
                2912,
                2922,
                "PERSON"
            ],
            [
                2931,
                2937,
                "PERSON"
            ],
            [
                2999,
                3005,
                "PERSON"
            ],
            [
                3018,
                3027,
                "PERSON"
            ],
            [
                3034,
                3041,
                "PERSON"
            ],
            [
                3099,
                3106,
                "PERSON"
            ],
            [
                3120,
                3127,
                "PERSON"
            ],
            [
                3184,
                3193,
                "PERSON"
            ],
            [
                3205,
                3213,
                "PERSON"
            ],
            [
                3227,
                3232,
                "PERSON"
            ],
            [
                3241,
                3248,
                "PERSON"
            ],
            [
                3281,
                3289,
                "PERSON"
            ],
            [
                3344,
                3352,
                "PERSON"
            ],
            [
                3463,
                3470,
                "PERSON"
            ],
            [
                3492,
                3500,
                "PERSON"
            ],
            [
                3664,
                3671,
                "PERSON"
            ],
            [
                3788,
                3794,
                "PERSON"
            ],
            [
                3803,
                3810,
                "PERSON"
            ],
            [
                3816,
                3821,
                "PERSON"
            ],
            [
                3849,
                3859,
                "PERSON"
            ],
            [
                3917,
                3923,
                "PERSON"
            ],
            [
                4005,
                4012,
                "PERSON"
            ],
            [
                4098,
                4104,
                "PERSON"
            ],
            [
                4173,
                4178,
                "PERSON"
            ],
            [
                4193,
                4204,
                "PERSON"
            ],
            [
                4219,
                4225,
                "PERSON"
            ],
            [
                4267,
                4273,
                "PERSON"
            ],
            [
                4417,
                4425,
                "PERSON"
            ],
            [
                4478,
                4485,
                "PERSON"
            ],
            [
                4494,
                4502,
                "PERSON"
            ],
            [
                4708,
                4715,
                "PERSON"
            ],
            [
                4868,
                4876,
                "PERSON"
            ],
            [
                4905,
                4912,
                "PERSON"
            ],
            [
                4920,
                4928,
                "PERSON"
            ],
            [
                4971,
                4976,
                "PERSON"
            ],
            [
                4994,
                4999,
                "PERSON"
            ],
            [
                5133,
                5139,
                "PERSON"
            ],
            [
                5156,
                5163,
                "PERSON"
            ],
            [
                5169,
                5177,
                "PERSON"
            ],
            [
                5276,
                5285,
                "PERSON"
            ],
            [
                5406,
                5415,
                "PERSON"
            ],
            [
                5520,
                5527,
                "PERSON"
            ],
            [
                5566,
                5573,
                "PERSON"
            ],
            [
                5748,
                5756,
                "PERSON"
            ],
            [
                5867,
                5873,
                "PERSON"
            ],
            [
                5990,
                5996,
                "PERSON"
            ],
            [
                6036,
                6042,
                "PERSON"
            ],
            [
                6065,
                6070,
                "PERSON"
            ],
            [
                6092,
                6100,
                "PERSON"
            ],
            [
                6272,
                6281,
                "PERSON"
            ],
            [
                6285,
                6292,
                "PERSON"
            ],
            [
                6447,
                6456,
                "PERSON"
            ],
            [
                6537,
                6545,
                "PERSON"
            ],
            [
                6562,
                6568,
                "PERSON"
            ],
            [
                6715,
                6722,
                "PERSON"
            ],
            [
                6774,
                6780,
                "PERSON"
            ],
            [
                6791,
                6800,
                "PERSON"
            ],
            [
                6851,
                6859,
                "PERSON"
            ],
            [
                6892,
                6903,
                "PERSON"
            ],
            [
                6951,
                6955,
                "PERSON"
            ],
            [
                6980,
                6989,
                "PERSON"
            ]
        ]
    }
),(
    "vumc { Vita  adult h { Justyn ospi { Oran tal w { Sibyl hite, tyrone 1211 medical center dr. { Adrain  mrn: 047717361, dob: 7/27/1 { Delanie 969, le { Zula gal sex: m nashville  { Jacinda tn 3723 { Jose 2 { Porsha -0004 visit date { Jennings : - 11/16/2023 - procedure pass facesheet report pat { Blane ient demo { Georgene graphics patien { Jaylan t name mrn legal dob  { Jarrell address phon { Lita e white, ty { Farrell rone 047717 { Markell 3 sex 7/27/ { Crosby 196 { Santa 9 apt 705 615-26 { Jailyn 0-2291 (home) 61 m 1101 edgehill ave 615-260-2 { Carlyn 291 (mobile) nashv { Leena ille tn 37203 *preferred* hospital account not o { Karleigh n file admission information cur { Myrtice rent information attending p { Wardell rovider  { Elicia admitting provide { Marti r admiss { Jeraldine ion type admission status unkn { Tiera own s { Migdalia tatus { Annaliese  admission  { Markel dat { Trinidad e/ { Marylin time discharge date/time hospital service auth/ { Clem ce { Crawford rt status  { Jaleel ho { Casen spi { Linette tal area unit { Trystan  r { Sherita oom/bed referring p { Rana rov { Kye ider 11 { Aleigha /16/ { Fredy 2023 - procedure pass (continued { Keren ) visit informatio { Savion n admis { Blaire sion info { Watson rmati { Shlomo on arriva { Halley l { Armond  date/time:  { Elmira admit date/time: i { Meghann p adm. date/time: ad { Jere m { Jordynn ission  { Julisa type: point { Meredith  of o { Damarion rigin: admit category: means of arrival: primary service: secondary servic { Melani e: n/a transfer source: service area: unit: admit p { Canaan rovider: atte { Marley nding pr { Oakley ovider: referring pr { Gearld ovider: discharge in { Tabetha forma { Benjiman tion { Grayce  date/time:  { Bayleigh - d { Schuyler isposition: - destination:  { Torin - provider: unit: - printed  { Tristen on  { Kristoffer 10/3/24 7:13 am page 1533 { Jordin ,vumc  { Cristine ad { Velva ult dayani center white, tyrone 1500 medical ctr { Viridiana  dr 1st fl, 108 mrn: 047717361, dob: 7/27/196 { Jammie 9, legal sex: m dayani ctr visit date: 1/16/2024  { Leesa nashville tn 37232 { Melton  11/16/2023 - daya { Sharee n { Arman i pt  { Lilla (therapy) epis { Leeroy ode { Corwin  info type: therapy noted date: 11/16/20 { Kyndal 23 resolved date { Gaige : 1/16 { Del /2024 associated visits 11/16/2023 - { Gaynell  evaluation in  { Aislinn vanderbilt dayani center 01/16/20 { Clair 24 - documentation in vanderbilt day { Yara an { Merton i center associated problems chronic left-sided low back pai { Rich n witho { Demetra ut { Timmothy  sc { Delphia iati { Lennon ca (primary) muscle weakness printed on { Dameon  10 { Elina /3/24 7:13 am page 1534",
    {
        "entities": [
            [
                7,
                12,
                "PERSON"
            ],
            [
                23,
                30,
                "PERSON"
            ],
            [
                37,
                42,
                "PERSON"
            ],
            [
                50,
                56,
                "PERSON"
            ],
            [
                95,
                102,
                "PERSON"
            ],
            [
                133,
                141,
                "PERSON"
            ],
            [
                151,
                156,
                "PERSON"
            ],
            [
                180,
                188,
                "PERSON"
            ],
            [
                198,
                203,
                "PERSON"
            ],
            [
                207,
                214,
                "PERSON"
            ],
            [
                233,
                242,
                "PERSON"
            ],
            [
                297,
                303,
                "PERSON"
            ],
            [
                315,
                324,
                "PERSON"
            ],
            [
                342,
                349,
                "PERSON"
            ],
            [
                373,
                381,
                "PERSON"
            ],
            [
                396,
                401,
                "PERSON"
            ],
            [
                415,
                423,
                "PERSON"
            ],
            [
                437,
                445,
                "PERSON"
            ],
            [
                459,
                466,
                "PERSON"
            ],
            [
                472,
                478,
                "PERSON"
            ],
            [
                497,
                504,
                "PERSON"
            ],
            [
                553,
                560,
                "PERSON"
            ],
            [
                581,
                587,
                "PERSON"
            ],
            [
                638,
                647,
                "PERSON"
            ],
            [
                682,
                690,
                "PERSON"
            ],
            [
                721,
                729,
                "PERSON"
            ],
            [
                740,
                747,
                "PERSON"
            ],
            [
                767,
                773,
                "PERSON"
            ],
            [
                784,
                794,
                "PERSON"
            ],
            [
                827,
                833,
                "PERSON"
            ],
            [
                841,
                850,
                "PERSON"
            ],
            [
                858,
                868,
                "PERSON"
            ],
            [
                882,
                889,
                "PERSON"
            ],
            [
                895,
                904,
                "PERSON"
            ],
            [
                909,
                917,
                "PERSON"
            ],
            [
                967,
                972,
                "PERSON"
            ],
            [
                977,
                986,
                "PERSON"
            ],
            [
                999,
                1006,
                "PERSON"
            ],
            [
                1011,
                1017,
                "PERSON"
            ],
            [
                1023,
                1031,
                "PERSON"
            ],
            [
                1047,
                1055,
                "PERSON"
            ],
            [
                1060,
                1068,
                "PERSON"
            ],
            [
                1090,
                1095,
                "PERSON"
            ],
            [
                1101,
                1105,
                "PERSON"
            ],
            [
                1115,
                1123,
                "PERSON"
            ],
            [
                1130,
                1136,
                "PERSON"
            ],
            [
                1171,
                1177,
                "PERSON"
            ],
            [
                1198,
                1205,
                "PERSON"
            ],
            [
                1215,
                1222,
                "PERSON"
            ],
            [
                1234,
                1241,
                "PERSON"
            ],
            [
                1249,
                1256,
                "PERSON"
            ],
            [
                1268,
                1275,
                "PERSON"
            ],
            [
                1279,
                1286,
                "PERSON"
            ],
            [
                1301,
                1308,
                "PERSON"
            ],
            [
                1329,
                1337,
                "PERSON"
            ],
            [
                1360,
                1365,
                "PERSON"
            ],
            [
                1369,
                1377,
                "PERSON"
            ],
            [
                1387,
                1394,
                "PERSON"
            ],
            [
                1408,
                1417,
                "PERSON"
            ],
            [
                1425,
                1434,
                "PERSON"
            ],
            [
                1511,
                1518,
                "PERSON"
            ],
            [
                1572,
                1579,
                "PERSON"
            ],
            [
                1595,
                1602,
                "PERSON"
            ],
            [
                1613,
                1620,
                "PERSON"
            ],
            [
                1643,
                1650,
                "PERSON"
            ],
            [
                1673,
                1681,
                "PERSON"
            ],
            [
                1689,
                1698,
                "PERSON"
            ],
            [
                1705,
                1712,
                "PERSON"
            ],
            [
                1727,
                1736,
                "PERSON"
            ],
            [
                1742,
                1751,
                "PERSON"
            ],
            [
                1781,
                1787,
                "PERSON"
            ],
            [
                1818,
                1826,
                "PERSON"
            ],
            [
                1832,
                1843,
                "PERSON"
            ],
            [
                1871,
                1878,
                "PERSON"
            ],
            [
                1887,
                1896,
                "PERSON"
            ],
            [
                1901,
                1907,
                "PERSON"
            ],
            [
                1958,
                1968,
                "PERSON"
            ],
            [
                2016,
                2023,
                "PERSON"
            ],
            [
                2075,
                2081,
                "PERSON"
            ],
            [
                2102,
                2109,
                "PERSON"
            ],
            [
                2130,
                2137,
                "PERSON"
            ],
            [
                2141,
                2147,
                "PERSON"
            ],
            [
                2155,
                2161,
                "PERSON"
            ],
            [
                2178,
                2185,
                "PERSON"
            ],
            [
                2191,
                2198,
                "PERSON"
            ],
            [
                2241,
                2248,
                "PERSON"
            ],
            [
                2267,
                2273,
                "PERSON"
            ],
            [
                2282,
                2286,
                "PERSON"
            ],
            [
                2325,
                2333,
                "PERSON"
            ],
            [
                2351,
                2359,
                "PERSON"
            ],
            [
                2395,
                2401,
                "PERSON"
            ],
            [
                2440,
                2445,
                "PERSON"
            ],
            [
                2450,
                2457,
                "PERSON"
            ],
            [
                2520,
                2525,
                "PERSON"
            ],
            [
                2535,
                2543,
                "PERSON"
            ],
            [
                2548,
                2557,
                "PERSON"
            ],
            [
                2563,
                2571,
                "PERSON"
            ],
            [
                2578,
                2585,
                "PERSON"
            ],
            [
                2627,
                2634,
                "PERSON"
            ],
            [
                2640,
                2646,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical center east white,  { Glory tyrone 1211 medical center  { Chelsi dr mrn: 047717361, dob: 7/27/1969,  { Alisson legal sex: m nashville tn 37232 { Richmond  visit date: 9/5/2023  { Kaelynn 09/05/2023 -  { Felice of { Patricio fice visit in vanderbilt diabetes a { Naya n { Darold d endocrinology  { Aubri (cont { Estell inued) lmr encounter level s { Magnus cans (con { Cherri ti { Presley nued) daily log freestyle libre 9 august 2023 - 5 september 2023 (28 days) 00: { Colter 00 02:00 04:00  { Darrion 06:00 08:00 10:00 12 { Ridge :00 14:0 { Catina 0 16:00 18:00 { Justin  20: { Chante 00 22:00 00:00 { Woody  350 sat 2 { Ransom 6 au { Jamia g mg/dl 180 70 g { Suzy lucose 0 mg/dl 107  { August 18 { Shavon 2 75 { Echo   { Luana 7 { Reymundo 5 176 1 { Stevan 84  { Karlene 145 35 { Kenisha 0 mg/dl sun 27 aug 180 o 70 glucose 0 mg/dl 222  { Poppy 138  { Raylene 132 229 275 162 carbs grams 350 mg/dl mon 28 { Ryley  aug  { Ely 180 70 glucose 0 mg/dl 73 115 72 74 { Justen  85 { Lynwood  11 { Rayden 7 168 187 232 { Laurene   { Niya legend high glucose (>240) low gl { Rowland ucose (<70) { Samiyah  str { Tawanda ip tes { Kathaleen t o sensor scan q { Isha  logged post-meal peak new sensor { Sofie  time change printed on 10/3/24 7:13 am pag { Verona e 1865,vumc ad { Bartholomew ult { Estefani  medi { Kameron cal center  { Rowen east white, tyr { Infant one { Barbara  1211 medical  { Kairi center dr mrn: 0 { Brennon 4771736 { Jocelyne 1, dob: { Audie  7/27/ { Braedon 1969, l { Emmeline egal sex: m nashville tn 3723 { Eamon 2 visit da { Sherryl te: 9/5/20 { Risa 23 09/05/2023 - office visit in  { Jerad vanderb { Acacia ilt d { Quiana iabetes and endo { Nalani crinology (c { Odell ontinued) lmr encounter { Davida  level scans (continued) daily log freestyle libre 9 augus { Marleen t 2023 - 5 september { Adolphus  2023 (28 days) 00:00 02:00 { Niles  04:00 0 { Fonda 6:00 08:00 10:00 12:00 { Janiah  14:00 16: { Simran 00 18 { Gray :00 20:00 22:00 00:00 350 tue 29 a { Raylan ug mg/d { Armaan l 180 70 glucose 0 mg/dl 120 126 { Joseluis  222 201 350 mg/dl wed 30 { Toya  aug 180 { Loy  70 glucose 0 mg/dl 124 221 91 74 96 { Callan  350 mg/dl thu 31 aug 180 70 glucose 0 mg/d { Cruz l 160 225 169 247 118 164 { Amar  99 350 mg/dl fri 1 sep 180 70 glucose 0 mg/dl { Spencer  284 121 142 109 72 legend high gluco { Fay se (>240) low glucos { Aubrianna e  { Gisela (< { Symone 70) strip test o  { Alexys senso { Charis r { Jefferey  scan q logged  { Domenico po { Carma st-meal p { Kirby e { Stefany ak new { Yair  sens { Colby or time change printed  { Mirna on 10/3/24 7:13 am page { Clarance  1866",
    {
        "entities": [
            [
                41,
                47,
                "PERSON"
            ],
            [
                77,
                84,
                "PERSON"
            ],
            [
                122,
                130,
                "PERSON"
            ],
            [
                164,
                173,
                "PERSON"
            ],
            [
                198,
                206,
                "PERSON"
            ],
            [
                222,
                229,
                "PERSON"
            ],
            [
                234,
                243,
                "PERSON"
            ],
            [
                281,
                286,
                "PERSON"
            ],
            [
                290,
                297,
                "PERSON"
            ],
            [
                316,
                322,
                "PERSON"
            ],
            [
                330,
                337,
                "PERSON"
            ],
            [
                368,
                375,
                "PERSON"
            ],
            [
                387,
                394,
                "PERSON"
            ],
            [
                399,
                407,
                "PERSON"
            ],
            [
                488,
                495,
                "PERSON"
            ],
            [
                513,
                521,
                "PERSON"
            ],
            [
                544,
                550,
                "PERSON"
            ],
            [
                561,
                568,
                "PERSON"
            ],
            [
                584,
                591,
                "PERSON"
            ],
            [
                598,
                605,
                "PERSON"
            ],
            [
                622,
                628,
                "PERSON"
            ],
            [
                641,
                648,
                "PERSON"
            ],
            [
                655,
                661,
                "PERSON"
            ],
            [
                680,
                685,
                "PERSON"
            ],
            [
                707,
                714,
                "PERSON"
            ],
            [
                719,
                726,
                "PERSON"
            ],
            [
                733,
                738,
                "PERSON"
            ],
            [
                742,
                748,
                "PERSON"
            ],
            [
                752,
                761,
                "PERSON"
            ],
            [
                771,
                778,
                "PERSON"
            ],
            [
                784,
                792,
                "PERSON"
            ],
            [
                801,
                809,
                "PERSON"
            ],
            [
                860,
                866,
                "PERSON"
            ],
            [
                873,
                881,
                "PERSON"
            ],
            [
                928,
                934,
                "PERSON"
            ],
            [
                942,
                946,
                "PERSON"
            ],
            [
                984,
                991,
                "PERSON"
            ],
            [
                997,
                1005,
                "PERSON"
            ],
            [
                1011,
                1018,
                "PERSON"
            ],
            [
                1034,
                1042,
                "PERSON"
            ],
            [
                1046,
                1051,
                "PERSON"
            ],
            [
                1087,
                1095,
                "PERSON"
            ],
            [
                1109,
                1117,
                "PERSON"
            ],
            [
                1124,
                1132,
                "PERSON"
            ],
            [
                1141,
                1151,
                "PERSON"
            ],
            [
                1171,
                1176,
                "PERSON"
            ],
            [
                1212,
                1218,
                "PERSON"
            ],
            [
                1264,
                1271,
                "PERSON"
            ],
            [
                1288,
                1300,
                "PERSON"
            ],
            [
                1306,
                1315,
                "PERSON"
            ],
            [
                1323,
                1331,
                "PERSON"
            ],
            [
                1345,
                1351,
                "PERSON"
            ],
            [
                1369,
                1376,
                "PERSON"
            ],
            [
                1382,
                1390,
                "PERSON"
            ],
            [
                1407,
                1413,
                "PERSON"
            ],
            [
                1432,
                1440,
                "PERSON"
            ],
            [
                1450,
                1459,
                "PERSON"
            ],
            [
                1469,
                1475,
                "PERSON"
            ],
            [
                1484,
                1492,
                "PERSON"
            ],
            [
                1502,
                1511,
                "PERSON"
            ],
            [
                1543,
                1549,
                "PERSON"
            ],
            [
                1562,
                1570,
                "PERSON"
            ],
            [
                1583,
                1588,
                "PERSON"
            ],
            [
                1623,
                1629,
                "PERSON"
            ],
            [
                1639,
                1646,
                "PERSON"
            ],
            [
                1654,
                1661,
                "PERSON"
            ],
            [
                1680,
                1687,
                "PERSON"
            ],
            [
                1702,
                1708,
                "PERSON"
            ],
            [
                1734,
                1741,
                "PERSON"
            ],
            [
                1802,
                1810,
                "PERSON"
            ],
            [
                1833,
                1842,
                "PERSON"
            ],
            [
                1872,
                1878,
                "PERSON"
            ],
            [
                1889,
                1895,
                "PERSON"
            ],
            [
                1920,
                1927,
                "PERSON"
            ],
            [
                1940,
                1947,
                "PERSON"
            ],
            [
                1955,
                1960,
                "PERSON"
            ],
            [
                1997,
                2004,
                "PERSON"
            ],
            [
                2014,
                2021,
                "PERSON"
            ],
            [
                2056,
                2065,
                "PERSON"
            ],
            [
                2093,
                2098,
                "PERSON"
            ],
            [
                2109,
                2113,
                "PERSON"
            ],
            [
                2152,
                2159,
                "PERSON"
            ],
            [
                2205,
                2210,
                "PERSON"
            ],
            [
                2238,
                2243,
                "PERSON"
            ],
            [
                2292,
                2300,
                "PERSON"
            ],
            [
                2340,
                2344,
                "PERSON"
            ],
            [
                2367,
                2377,
                "PERSON"
            ],
            [
                2382,
                2389,
                "PERSON"
            ],
            [
                2394,
                2401,
                "PERSON"
            ],
            [
                2421,
                2428,
                "PERSON"
            ],
            [
                2436,
                2443,
                "PERSON"
            ],
            [
                2447,
                2456,
                "PERSON"
            ],
            [
                2474,
                2483,
                "PERSON"
            ],
            [
                2488,
                2494,
                "PERSON"
            ],
            [
                2506,
                2512,
                "PERSON"
            ],
            [
                2516,
                2524,
                "PERSON"
            ],
            [
                2533,
                2538,
                "PERSON"
            ],
            [
                2546,
                2552,
                "PERSON"
            ],
            [
                2578,
                2584,
                "PERSON"
            ],
            [
                2610,
                2619,
                "PERSON"
            ]
        ]
    }
),(
    "vum { Avi c adult hos { Obie pital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, leg { Sahara al sex: m nashvi { Hampton lle tn 37232-0004 visit date: 7/17/2024 07/17/2024 - medication management in vumc population health pharmacy services vir (continued) medica { Mikala tion list (continued) instructions: take 1 tablet (5 mg to { Jacqulyn tal) by mouth every 8 hours as needed. entered by: mapl { Sana es, chantis entered on: 9/5/2023 start date: 7/22/2023 { Loyal  triamcinolone acetonide 55 mcg  { Jelani nasal spray aerosol (nasacort) discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate order instructions: administer 2 sprays (110 m { Vidal cg total) into each nost { Naima ril 2 times a day. authorized by: greenspan, debra l, aprn ordered { Vernita  on: 9/5/202 { Eda 3 start date: 9/5/2023 end date: { Suzann  9/16/2024 { Elysia  quantity: 16.5 g ref { Inga ill: 11 refills by 9/4/2024 capsaicin 0.1 % topical cream discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason { Gustav  for discontinuation: duplicate order instructions: apply 1 application topicall { Soledad y daily for 90 days. a { Maiya uthorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 end date: 9/16/2024 action: patien { Sherilyn t not taking quantity: 42.5 g refill:  remaining cetirizine 1 { Delana 0 mg tablet (zyrtec) instructions: take 1 tablet { Kindra  (10 mg total) by mouth once a day as needed for allergies. authorized by: de witte, anton jordan, md ordered { Oswald  on: 10/4/2023 start d { Shimon ate: 10/4/2023 quantity: 3 { Eldora 0 tablet refi { Ocie ll: 9 refills by 10/3/2024 lidocaine 5 % topical patch (lidoderm) instructions: apply 1 patch topically daily. apply to painful area 12 hours per day,  { Ammon remove for 12 hours. authorized by: de witte, anton jordan, m { Marty d ordered on: 10/17/2023 start date: 10/17/2023 end date: 10/16/2024 quantity: 30 patc { Abrianna h refill: 11 refills by 10/16/2024 nifedipine er 30 mg ta { Jerica blet,ext { Bonnie ended release (adalat cc) instructions: take 2 tablets (60 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 4/23/2024 start date: 4/23/2024 quantity: 180 tablet ref { Barbie ill: 3 refills by 4/23/2025 lantus solostar u-100 insu { Jasmyne lin 100 unit/ml (3 ml) subcutaneous pen (insulin glargine) instructions: inject 10 units under the skin 2 times a day. authorized by: greenspan, debra l, aprn ordered on: 7/9/2024 star { Sylvie t date: 7/9/2024 quantity: 18 ml refill: 3 refills by 7/9/2025 pantopra { Alvie zole 20 mg { Tawanna  tablet,delayed  { Jered release (protonix) instr { Arianne uctions: take 1 tablet (20 mg total) by mouth daily. authorized by: greens { Liane pan, debra l, aprn ordered on: 7/9/2024 start date: 7/9/2024 end date: 7/9/2025 quantity: 30 tablet refill: 11 refills by 7/9/2025 prednisolone ac { Tarik etate 1 % eye drops,suspension (pred forte) instructions: after surgery, use 1 drop to the  { Edson right eye every 2 hours while awake until bedtime. beginning { Yitzchok  the next day, decrease to 1 drop to the right eye 4 times a day for 1 w { Audry eek, then 3 times a day for 1 week, then 2 times a day for 1 week, then daily for 1 week, then stop auth { Landry orized by: valenzuela, daniel alejandro, md ordered on: 7/22/2024 start date: 7/22/2024 quantity: 5 ml refill: 1 refill by 7/22/2025 moxifloxacin 0.5 % eye drops (vigamox) printed  { Alexi on 10/3/2 { Layne 4 7:12  { Neveah am p { Josefa age 603,vumc adult hos { Jacinta pital white, { Elayna  tyrone 121 { Lissa 1 medical center dr. mrn: 047717 { Martez 361, dob: 7/27/1969, legal sex: m nashv { Reno ille tn 37232-0004 visit date: 7/17/2024 07/17/2024 - { Jessica  medication management in vumc population health pharmacy services vir (continued) medication  { Lilyan list (continued) instruct { Ritchie ions: after surger { Sudie y, use 1 drop to the right eye every 2 hours while awake until bedt { Maliah ime. beginning the next day, { Haven  decrease to 1 drop t { Gigi o the ri { Apollo ght eye 4 times a day for 1 week, then stop authorized by: valenzuela, daniel alejandro, md ordered on: 7/22/2024 start date: 7 { Henley /22/2024 quantity: 3 ml refill: 1 refill by 7/22/2025 ketorolac 0.4 % eye drops (acular) discontinued by: gingrow { Cleta , barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate order instructions: after su { Scarlette rgery, use 1 drop to the right eye every 2 hours while awake until  { Bertrand bedtime. beginning the next day, decrease to 1 drop to the right eye 4 t { Cammie imes a { Willian  day for 2 weeks then stop { Lu  authorized by: valenzuela, daniel alejandro, md ordered on: 7/22/2024 start date: 7/22/2024 end date: 9/16/2024 quantity: 5 ml refill: 1 refill by 7/22/2025 albuterol sulfate hfa 90 mcg/actuation aerosol in { Aminah haler instructions:  { Grey inhale 2 puffs every 4 hours as needed for wheezing. authorized by: cha { Tremayne kravarthy, rohini, md ordered on: 8/6/2024 start d { Dariel ate: 8/6/2024 quantity: 18 g refill: 11 refills by 8/6/2025 flutic { Lane asone propionate 50 mcg/actuation nasal spray,suspension (flonase) instructions: administer 2 sprays into each nostril 2 times a day. authorized by: connolly, patrick james, md o { Dell rdered o { Tavon n: 8/6/2024 start date: 8/6/2024 quantity: 16 g refill: 2 refills by  { Rachell 8/6/2025 gabap { Ishaan entin 300 mg capsule (neurontin) instructions: take 2 capsules (600 mg total) by mouth daily. authorized by: chakravarthy, rohini, md ordered on: 8/6/2024 start date: 8/6/2024 quantity: 180 capsule refill:  remai { Refugio ning ipratropium bromide 42 mcg (0.06 nasal spray (atrovent) instructions: administer 1 spray into each nostril 4 times a day. start with 1 spray  { Alisia once daily for one week, then increase to 2 sprays once a day for 2 weeks, can increase to 3-4 sprays a day authorized by: chakravarthy, rohini, md ordered on: 8/6/2024 start d { Malorie ate: 8/6/2024 quantity: 15 { Marielle  ml refill: 12 refil { Promise ls by 8/6/2025 ketorolac 0.5 % eye drops (acu { Camryn lar) instructions: administer 1 drop { Donta  into the left { Kyron  eye 4 times a day for 28 days. authorized by: brown, sarah karee, md ordered on { Reinaldo : 8/23/2024 start date: 8/23/2024 end { General  date: 9/20/2024 qua { Theadore ntity: 5 ml refill:  remaining moxifloxacin 0.5 % eye drops (vigamox) discontinued by: gingrow, ba { Analise rbara, lpn discontinued on: 9/16/2024 reason for d { Lilyanna iscontinuation: duplicate order instructions: administer 1 drop into the left eye 4 time { Lyra s a day for 7 days. authorized by: brown, sarah karee,  { Kelby md ordered on: { Izabelle  8/23/2024 start date: 8/23/2024 end date: 9/16/2024 quanti { Kailynn ty: 3 ml refill:  remaining prednisolone acetate 1 % eye ops,suspension (pred forte) discontinu { Early ed by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate order inst { Nikia ructions: administer 1 drop into the left { Katlynn  eye 4 times a day for 7 days, then 1 drop 3 times a day for 7 days, then 1 drop 2 times a day for 7 days, then 1 drop daily for 7 days. authorized by: brown, sarah k { Alek aree, md ordered on: 8/23/2024 printed on 10/3/ { Jayme 24 7:12 a { Joselin m page 604",
    {
        "entities": [
            [
                6,
                10,
                "PERSON"
            ],
            [
                24,
                29,
                "PERSON"
            ],
            [
                111,
                118,
                "PERSON"
            ],
            [
                137,
                145,
                "PERSON"
            ],
            [
                289,
                296,
                "PERSON"
            ],
            [
                357,
                366,
                "PERSON"
            ],
            [
                424,
                429,
                "PERSON"
            ],
            [
                486,
                492,
                "PERSON"
            ],
            [
                527,
                534,
                "PERSON"
            ],
            [
                718,
                724,
                "PERSON"
            ],
            [
                751,
                757,
                "PERSON"
            ],
            [
                826,
                834,
                "PERSON"
            ],
            [
                849,
                853,
                "PERSON"
            ],
            [
                888,
                895,
                "PERSON"
            ],
            [
                908,
                915,
                "PERSON"
            ],
            [
                939,
                944,
                "PERSON"
            ],
            [
                1077,
                1084,
                "PERSON"
            ],
            [
                1167,
                1175,
                "PERSON"
            ],
            [
                1200,
                1206,
                "PERSON"
            ],
            [
                1326,
                1335,
                "PERSON"
            ],
            [
                1399,
                1406,
                "PERSON"
            ],
            [
                1457,
                1464,
                "PERSON"
            ],
            [
                1576,
                1583,
                "PERSON"
            ],
            [
                1608,
                1615,
                "PERSON"
            ],
            [
                1644,
                1651,
                "PERSON"
            ],
            [
                1667,
                1672,
                "PERSON"
            ],
            [
                1826,
                1832,
                "PERSON"
            ],
            [
                1896,
                1902,
                "PERSON"
            ],
            [
                1991,
                2000,
                "PERSON"
            ],
            [
                2060,
                2067,
                "PERSON"
            ],
            [
                2078,
                2085,
                "PERSON"
            ],
            [
                2283,
                2290,
                "PERSON"
            ],
            [
                2347,
                2355,
                "PERSON"
            ],
            [
                2542,
                2549,
                "PERSON"
            ],
            [
                2623,
                2629,
                "PERSON"
            ],
            [
                2642,
                2650,
                "PERSON"
            ],
            [
                2669,
                2675,
                "PERSON"
            ],
            [
                2702,
                2710,
                "PERSON"
            ],
            [
                2787,
                2793,
                "PERSON"
            ],
            [
                2942,
                2948,
                "PERSON"
            ],
            [
                3042,
                3048,
                "PERSON"
            ],
            [
                3111,
                3120,
                "PERSON"
            ],
            [
                3195,
                3201,
                "PERSON"
            ],
            [
                3308,
                3315,
                "PERSON"
            ],
            [
                3498,
                3504,
                "PERSON"
            ],
            [
                3516,
                3522,
                "PERSON"
            ],
            [
                3532,
                3539,
                "PERSON"
            ],
            [
                3546,
                3553,
                "PERSON"
            ],
            [
                3578,
                3586,
                "PERSON"
            ],
            [
                3601,
                3608,
                "PERSON"
            ],
            [
                3622,
                3628,
                "PERSON"
            ],
            [
                3663,
                3670,
                "PERSON"
            ],
            [
                3712,
                3717,
                "PERSON"
            ],
            [
                3773,
                3781,
                "PERSON"
            ],
            [
                3878,
                3885,
                "PERSON"
            ],
            [
                3913,
                3921,
                "PERSON"
            ],
            [
                3942,
                3948,
                "PERSON"
            ],
            [
                4018,
                4025,
                "PERSON"
            ],
            [
                4056,
                4062,
                "PERSON"
            ],
            [
                4086,
                4091,
                "PERSON"
            ],
            [
                4102,
                4109,
                "PERSON"
            ],
            [
                4239,
                4246,
                "PERSON"
            ],
            [
                4362,
                4368,
                "PERSON"
            ],
            [
                4479,
                4489,
                "PERSON"
            ],
            [
                4559,
                4568,
                "PERSON"
            ],
            [
                4643,
                4650,
                "PERSON"
            ],
            [
                4659,
                4667,
                "PERSON"
            ],
            [
                4696,
                4699,
                "PERSON"
            ],
            [
                4909,
                4916,
                "PERSON"
            ],
            [
                4939,
                4944,
                "PERSON"
            ],
            [
                5018,
                5027,
                "PERSON"
            ],
            [
                5080,
                5087,
                "PERSON"
            ],
            [
                5156,
                5161,
                "PERSON"
            ],
            [
                5342,
                5347,
                "PERSON"
            ],
            [
                5358,
                5364,
                "PERSON"
            ],
            [
                5436,
                5444,
                "PERSON"
            ],
            [
                5461,
                5468,
                "PERSON"
            ],
            [
                5683,
                5691,
                "PERSON"
            ],
            [
                5840,
                5847,
                "PERSON"
            ],
            [
                6026,
                6034,
                "PERSON"
            ],
            [
                6063,
                6072,
                "PERSON"
            ],
            [
                6095,
                6103,
                "PERSON"
            ],
            [
                6151,
                6158,
                "PERSON"
            ],
            [
                6197,
                6203,
                "PERSON"
            ],
            [
                6220,
                6226,
                "PERSON"
            ],
            [
                6309,
                6318,
                "PERSON"
            ],
            [
                6358,
                6366,
                "PERSON"
            ],
            [
                6389,
                6398,
                "PERSON"
            ],
            [
                6499,
                6507,
                "PERSON"
            ],
            [
                6560,
                6569,
                "PERSON"
            ],
            [
                6660,
                6665,
                "PERSON"
            ],
            [
                6723,
                6729,
                "PERSON"
            ],
            [
                6746,
                6755,
                "PERSON"
            ],
            [
                6817,
                6825,
                "PERSON"
            ],
            [
                6923,
                6929,
                "PERSON"
            ],
            [
                7036,
                7042,
                "PERSON"
            ],
            [
                7086,
                7094,
                "PERSON"
            ],
            [
                7263,
                7268,
                "PERSON"
            ],
            [
                7318,
                7324,
                "PERSON"
            ],
            [
                7336,
                7344,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult village at vanderbilt white, tyrone 1500 21st ave s 2nd fl, 2500 mrn: 047717361, dob: 7/27/ { Kayle 1969, legal sex: m village at vanderbilt visit date: 12/27/2023 nashville tn 37212-3160  { Hortencia 12/27/2023 - care  { Page coordination in vanderbilt renal transplant clinic (continued { Charlette ) clinical notes (continued) pt ha { Shawnee s been re { Evelyne scheduled to 5/29/24 with dr. lango { Tory ne { Hermon . pt has also been scheduled for a social work phone visit on 5/22/24. i explained to pt reschedule { Brittny  policy and pt sta { Jaycob ted that they understood. new appt letter to be mailed to pt address. pt is aware that these appts are on central standard time. electronically signed by campbell, mary-alyx at 4/11/2024 10:03 am biggs, joy noele, rn at 5/23/2024 0838 author: biggs, joy noele, rn service: author type: registered nurse filed: 5/23/2024  { Loni 8:38 am encounter date: 12/27/2023 status: signed editor: biggs, joy noele, rn (registered nurse) kidney transplant eval pro { Maxie tocol order entered for signature. thanks electronically si { Elmore gned by biggs, joy noel { Delfina e, r { Kimberli n a { Gaston t 5/23/2024 8:38 am bruns, cassandra, aprn at 5/23/2024 0916 author: brun { Vernie s, cassandra, aprn service: - author type: nurse pra { Tyrel ctitioner filed: 5/23/2024 { Corene  9:16 { Kalee  am enc { Rishi ounter date: 12/27/2 { Aiyanna 023 status: signed editor: bruns, cassandra, aprn (nurse practitioner) signed electronically signed by bruns, cassandra, aprn at 5/23/2024 9:16 am  { Milani campbell, mary-alyx at 5/28/2024 1328 author: cam { Luvenia pbell, mary { Tangela -alyx service: - author type: clerk filed: 5/28/2024 1 { Yolonda :29 pm encounter date: 12/27/2023 status: signed editor: campbell, mary-alyx (clerk) called pt to confirm evaluation appts, lvm electronically signed by campb { Gretta ell, mary-alyx at 5/28/ { Raeann 2024 1:29 pm laboratory reports quantiferon-tb (comple { Heavenly ted) electronically signed by: biggs, joy  { Edsel noele, rn on 05/23/24 1342 status { Henderson : completed ordering user: biggs, joy noele, rn 05/23/24 1 { Prentice 34 { Adyson 2 ordering provider: langone, anthony j, md authorized by: langone, an { Clyde thony j, md ordering mode: p { Elgin er activated protocol frequency: routine 05/23/24 - class: lab c { Wayland ollect quantity: 1 diagnoses pre-transplant evaluation for kidney { Sharen  transplant [z01. { Larue 818] specimen information id type source collec { Ronald ted by - blood { Samaria  - - indications pre-transplant evaluati { Lino on for kidney transplant [z01.818 { Dilan  (icd-10-cm)] printed on 10 { Pearl /3/24 7:12 am page 1299,vumc adult v { Isobel illage at vanderbilt white, tyrone 1 { Elayne 500 21st ave s 2nd fl, 2500 m { Jackelyn rn: 0477 { Michal 17361, dob: 7/27/1969, legal sex: m village at vanderbilt visit d { Nella ate: 12/27/2023 { Dempsey  nashville tn 37212-3160 12/27/2023 - care coordination in vanderbilt renal transplant clinic (c { Erasmo ontinued) laboratory reports (continued) cbc w/ different { Devorah ial  { Ema (compl { Leeanna eted) e { Desean lectronically signed by: biggs, joy noele, rn on 0 { Dayne 5/23/24 1342 status: completed or { Jonathan dering user: biggs, joy noele, rn 05/23/24 13 { Reece 42 ordering provider: langone, anthony j, md authorized by: langone, an { Kamryn thony j, md ordering mode: per activated protocol frequency: routine 05/23/24 - cl { Tierney ass: lab collect quantity: 1 diagnoses pre-transplant { Kenji  evaluati { Elly on for kidney transplant [z01.818] specimen  { Kellee information id type source collected by - blood - - indications pr { Rayshawn e-trans { Jacie plant evaluation for kidney transplant [z01.818 { Amado  (icd-10-cm)] platelet count (discontinued) electronically signed by: biggs, joy noele, rn on 05/2 { Abrielle 3/24 1342 stat { Pinkie u { Deondre s: discont { Isaak inued ordering user: biggs, joy noele, rn 05/23/24 1342 order { Shianne ing provider: { Jaslyn  langone, anthony j, md authorized by: langone, a { Gabe nthony j, md or { Dominga dering mode: per activated protocol frequency: routine 05/23/24 { Regan  - class: lab collect quantity: 1 discontinued by: interface, lab results in 05/29/24 1119 [system duplicate { Romina  cancel] diagnoses pre- { Allison transplant evaluation for  { Elzie kidney t { Kay ransplant [z01.818] specimen information id type source collected by - blood - - indications pre-transplant evaluation for kidney { Garnett  transplant [z01.818 (ic { Howell d-10-cm)] cmp (completed) electronically si { Trae gned by: biggs, joy noele,  { Joella rn on 05/23/24 1342  { Yessica status:  { Montrell completed ordering user: biggs, joy noele, rn  { Paolo 05/23/24 1342 ordering provider: langone, anthony j, md authorized by: langone, anthony j, md ordering mode: per a { Baxter ctivated protocol frequenc { Angelic y: routine 05/23 { Emmit /24 - class: lab collect quantity: 1 diagnoses pre-transplant evaluation for kidney transplant [z0 { Malina 1.818] specimen information id t { Thomasina ype source collected by - blood - - indications pre-transplant  { Audrie evaluation { Barb  for kidney transplant [z01.818 (icd-10-cm)] type/scrn (abo/rh/ab scrn) (comp { Suzie leted) elec { Lonzo tronically signed by: biggs, joy n { Sebastien oele, rn on 05/23/24 1342 status: completed ordering user: biggs, joy noele, rn 05/23/24 1342 ordering provider: { Dionna  langone, anthony j,  { Misha md  { Debi authorized by: langone, anthony j, md ordering mode: per activated protocol frequency: routine 05/23/24 - class: lab collect quantity: 1 diagnoses pre-transplant evaluatio { Ellison n for kidn { Khadija ey transplant [z01.818] printed on 10/3/24 7:12 am page 1300",
    {
        "entities": [
            [
                105,
                111,
                "PERSON"
            ],
            [
                202,
                212,
                "PERSON"
            ],
            [
                233,
                238,
                "PERSON"
            ],
            [
                302,
                312,
                "PERSON"
            ],
            [
                349,
                357,
                "PERSON"
            ],
            [
                369,
                377,
                "PERSON"
            ],
            [
                415,
                420,
                "PERSON"
            ],
            [
                425,
                432,
                "PERSON"
            ],
            [
                534,
                542,
                "PERSON"
            ],
            [
                563,
                570,
                "PERSON"
            ],
            [
                893,
                898,
                "PERSON"
            ],
            [
                1025,
                1031,
                "PERSON"
            ],
            [
                1093,
                1100,
                "PERSON"
            ],
            [
                1126,
                1134,
                "PERSON"
            ],
            [
                1141,
                1150,
                "PERSON"
            ],
            [
                1156,
                1163,
                "PERSON"
            ],
            [
                1239,
                1246,
                "PERSON"
            ],
            [
                1301,
                1307,
                "PERSON"
            ],
            [
                1336,
                1343,
                "PERSON"
            ],
            [
                1351,
                1357,
                "PERSON"
            ],
            [
                1367,
                1373,
                "PERSON"
            ],
            [
                1396,
                1404,
                "PERSON"
            ],
            [
                1554,
                1561,
                "PERSON"
            ],
            [
                1613,
                1621,
                "PERSON"
            ],
            [
                1635,
                1643,
                "PERSON"
            ],
            [
                1700,
                1708,
                "PERSON"
            ],
            [
                1869,
                1876,
                "PERSON"
            ],
            [
                1902,
                1909,
                "PERSON"
            ],
            [
                1966,
                1975,
                "PERSON"
            ],
            [
                2020,
                2026,
                "PERSON"
            ],
            [
                2062,
                2072,
                "PERSON"
            ],
            [
                2133,
                2142,
                "PERSON"
            ],
            [
                2147,
                2154,
                "PERSON"
            ],
            [
                2227,
                2233,
                "PERSON"
            ],
            [
                2264,
                2270,
                "PERSON"
            ],
            [
                2337,
                2345,
                "PERSON"
            ],
            [
                2413,
                2420,
                "PERSON"
            ],
            [
                2440,
                2446,
                "PERSON"
            ],
            [
                2496,
                2503,
                "PERSON"
            ],
            [
                2520,
                2528,
                "PERSON"
            ],
            [
                2571,
                2576,
                "PERSON"
            ],
            [
                2612,
                2618,
                "PERSON"
            ],
            [
                2648,
                2654,
                "PERSON"
            ],
            [
                2693,
                2700,
                "PERSON"
            ],
            [
                2739,
                2746,
                "PERSON"
            ],
            [
                2778,
                2787,
                "PERSON"
            ],
            [
                2798,
                2805,
                "PERSON"
            ],
            [
                2873,
                2879,
                "PERSON"
            ],
            [
                2897,
                2905,
                "PERSON"
            ],
            [
                3004,
                3011,
                "PERSON"
            ],
            [
                3071,
                3079,
                "PERSON"
            ],
            [
                3086,
                3090,
                "PERSON"
            ],
            [
                3099,
                3107,
                "PERSON"
            ],
            [
                3117,
                3124,
                "PERSON"
            ],
            [
                3177,
                3183,
                "PERSON"
            ],
            [
                3219,
                3228,
                "PERSON"
            ],
            [
                3276,
                3282,
                "PERSON"
            ],
            [
                3356,
                3363,
                "PERSON"
            ],
            [
                3448,
                3456,
                "PERSON"
            ],
            [
                3512,
                3518,
                "PERSON"
            ],
            [
                3530,
                3535,
                "PERSON"
            ],
            [
                3582,
                3589,
                "PERSON"
            ],
            [
                3658,
                3667,
                "PERSON"
            ],
            [
                3677,
                3683,
                "PERSON"
            ],
            [
                3733,
                3739,
                "PERSON"
            ],
            [
                3840,
                3849,
                "PERSON"
            ],
            [
                3866,
                3873,
                "PERSON"
            ],
            [
                3877,
                3885,
                "PERSON"
            ],
            [
                3898,
                3904,
                "PERSON"
            ],
            [
                3968,
                3976,
                "PERSON"
            ],
            [
                3992,
                3999,
                "PERSON"
            ],
            [
                4051,
                4056,
                "PERSON"
            ],
            [
                4074,
                4082,
                "PERSON"
            ],
            [
                4148,
                4154,
                "PERSON"
            ],
            [
                4265,
                4272,
                "PERSON"
            ],
            [
                4298,
                4306,
                "PERSON"
            ],
            [
                4335,
                4341,
                "PERSON"
            ],
            [
                4352,
                4356,
                "PERSON"
            ],
            [
                4488,
                4496,
                "PERSON"
            ],
            [
                4523,
                4530,
                "PERSON"
            ],
            [
                4576,
                4581,
                "PERSON"
            ],
            [
                4611,
                4618,
                "PERSON"
            ],
            [
                4641,
                4649,
                "PERSON"
            ],
            [
                4660,
                4669,
                "PERSON"
            ],
            [
                4718,
                4724,
                "PERSON"
            ],
            [
                4841,
                4848,
                "PERSON"
            ],
            [
                4877,
                4885,
                "PERSON"
            ],
            [
                4904,
                4910,
                "PERSON"
            ],
            [
                5011,
                5018,
                "PERSON"
            ],
            [
                5053,
                5063,
                "PERSON"
            ],
            [
                5129,
                5136,
                "PERSON"
            ],
            [
                5149,
                5154,
                "PERSON"
            ],
            [
                5234,
                5240,
                "PERSON"
            ],
            [
                5254,
                5260,
                "PERSON"
            ],
            [
                5297,
                5307,
                "PERSON"
            ],
            [
                5422,
                5429,
                "PERSON"
            ],
            [
                5453,
                5459,
                "PERSON"
            ],
            [
                5465,
                5470,
                "PERSON"
            ],
            [
                5644,
                5652,
                "PERSON"
            ],
            [
                5665,
                5673,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hundred oaks white, tyrone 719 thompson lane, nashville mrn: 047 { Michal 717361, dob: 7/27/1969, lega { Kevan l sex: m nashville tn 37204 visit date: 11/9/2023 11 { Bethanie /09/2023 - communication in vanderbilt one hundred oaks primary care north ( { Alain continued) medication list (continued) instructions: administer 2 sprays (110 mcg total) into eac { Keyana h nostril 2 times a da { Coraline y. authorized by: greenspan, debra l, aprn ordered on: { Jeniffer   { Cayson 9/5/2023 start date: 9/5/2023 end date: 9/16/2024 quantity: 16.5 g refil { Derwin l: 11 refills by 9/4/2024 losartan 25 mg ta { Verlin blet (cozaar) discontinued by: kovtun, roman, md discontinued on: 8/1/2024 reason for discontinuation: stop (cancelrx, on avs) instructions: take 1 tablet  { Ariane (25 mg total) by mouth daily. authorized b { Taniyah y: de witte, anton jorda { Linda n, md o { Kamari rdered  { Briar on: 9/7/2023 st { Raya art date: 9/7/2023 end date: 8/1/2024 quantity: 90 tablet refill: 3 refills by 9/6/2024 capsaicin 0.1 % topical cream discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate order instructions: apply 1 application topically daily for 90 days. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 end date: 9/16/2024 action: patient not taking quantity: 42.5 g refill:  remaining gabapentin 300 mg capsule (neurontin) discontinued by: hock, richard lloyd, md discontinued on: 11/9/2023 reason for discontinuation: reorder instructions: take 1 capsule (300 mg total)  { Arlin by mouth daily. au { Nailah thorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 { Willy  quantity: 90 capsule { Ellyn  refill:  r { Krish emaining aspirin 81 mg tablet,delayed release instructions { Christ : take 1 tablet (81 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 quantity: 90 tablet refill: 3 refills by 9/6/2024 cetirizine 10 mg t { Corie ablet (zyrtec) instructions: take 1 tablet (10 mg { Infant  total) by mouth once a day as needed for allergies. authorized by: de witte, anton jordan, md ordered on: 10/4/2023 start date: 10/4/2023 quantity: 30 tablet re { Gaetano fill: 9 refills by 10/3/2024 lidocaine 5 % topical patch (lidoderm) ins { Blayne tructions: apply 1 patch topically daily. apply to painful area 12 hours per day, remo { Rianna ve for 12 hour { Margaret s. authorized by: de witte,  { Hobart anton jordan, md ordered on { Christofer : 10/17/2023 start date: 10/17/2023 end  { Jadiel date:  { Queenie 10/16/2024 quantity: 30 patch refill: 11 refills { Brogan  by 10/1 { Marleigh 6/2024 diclofenac 1 % topical gel discontinued by: cone, brittany discontinued on: 6/2 { Harleigh 7/2024 reason for discontinuation: therapy completed (cancelrx) instructions: apply 2 g topically 4 times a day for 30 days. authorized by: de witte, anton jordan, md ordered on: 10/17/2023 start date: 10/17/2023 end date: 6/27/2024 action: patient not taking quantity: 100 g refill:  remaining azelastine 137 mcg (0.1 %) nasal spray aerosol (astelin)  { Tenisha instructions: admini { Keyshawn ster 1 spray in { Quincy to each nostril 2 times a  { Shanae day as needed for rhinitis. us { Anibal e in each nostril as directed authorized by: de witte, anton  { Nydia jordan, md ordered on: 10/17/2023 printed on 10/3/24 7:13 am page 1551,vumc adult one hun { Benjamen dred oaks white { Juancarlos , tyrone 719 thompson lane, nashville mrn: 0 { Yetta 47717361, dob: 7/27/1969, lega { Kamren l  { Lathan sex: m nashville tn 37204 visit date: 11/9/2023 11/09/2023 - communi { Chantell ca { Sherwin tion in vanderbilt one hundred oaks primary care north (continu { Jaymie ed) medication list (continued) start date: 10/17/2023 quan { Denzil tity: 30 ml re { Analisa fill:  remaining insulin glargin { Manny e (u-100) 100 unit/ml subcutaneous s { Randa olution disco { Harlie ntinued by: prasad, sonika h, rn discontinued on: 2/2 { Loriann 2/2024 reason for discontinuation: cleanu { Marge p(notavs) instructions: inject 0.05 ml (5 units total) un { Burke der the skin daily. authorized by: de witte, anton jordan, { Sunny  md ordered on: 10/20/2023 start date: 10/20/2023 quantity: 4.5  { Cristofer ml refill:  remaining flutic { Kegan asone propionate 50 mcg/actuation nasal spray, ,suspension (flonase)  { Honesty discontinued by: virk, z { Ria ain m, md discontinued on: 11/17/2023 reason for discontinuation: reorder instructions: administe { Herminia r 2 sprays in { Johnny to each nostril 2 times a day. authorized b { Christel y: de witte,  { Vinson anton jordan,  { Lanita md ordered on: 10/24/2023 start date: 10/24/2023 quantity: 16 g refill: 2 refills by 10/23/ { Mariella 2024 stopped in visit none clinical notes telephone encounter ford, keynin roeshelle, rn at 11/9/202 { Nereida 3 0829 auth { Shanita or: ford, { Eldred  keynin r { Daniele oeshelle, rn service: author type: registered nurse filed: 11/9/2023 8:41 am encounter date: 1 { Baylor 1/9/2023 status: signed editor: ford, keynin roeshelle, { Gaven  rn (registered nurse) patient stopped by o { Magen ffice today. states his bell's palsy is not improving. has lasted 4+ mo { Dennie nths. he says s { Kimberlie ometimes he can speak clearly,  { Tahlia but most of the time it is slurred. reports he has not taken steroids since the onset of these sx. he says he  { Amiah is still having sinus congestion and that the flonase is more helpful than the astelin. h { Jaya e states he { Britta  has o { Alea ne more month on his gabapentin then it will need a refill. he had appt with pm& { Pranav r at dayani, but got there { Faustino  late because he got on the wrong bus. reports t { Keanna hat he is having fecal inc { Simona ontinence at times. states he ca { Damari n't tell when he needs to go. i could not get a clear time frame on when this started. he den { Tomeka ies hx of s { Rosalina pine injury, bu { Kahlil t there is the mva and chronic back pain. advised to reschedule pm&r. please advise. spine clinic referral? electronically signed by ford, keynin roeshelle, rn at 11/9/2023 8:41 am opthof, jessica  { Demetri t  { Nels at 11/10/2023 1610 { Calla  au { Debbra thor: opthof { Evon , jessica t  { Claudie service: author type: patient access filed: 11/1 { Carlyle 0/2023 4:11 pm en { Kiarra counter date: 11/9/2023 status: { Citlali  signed editor: opthof, jessica t (patient access) left message electronically signed by opthof, jessica t at 11/10/2023 4: { Jaquelin 11 pm opthof, jessica t at 11/13/2023 1443 author: opthof, jessica t service: author type: patient access filed: 11/13/2023 2:43 pm encounter date: 11/9/2023 status: signed editor: opthof, jessica t (patient access) printed on 10/3/24 7:13 am page 1552",
    {
        "entities": [
            [
                82,
                89,
                "PERSON"
            ],
            [
                120,
                126,
                "PERSON"
            ],
            [
                181,
                190,
                "PERSON"
            ],
            [
                269,
                275,
                "PERSON"
            ],
            [
                375,
                382,
                "PERSON"
            ],
            [
                407,
                416,
                "PERSON"
            ],
            [
                473,
                482,
                "PERSON"
            ],
            [
                486,
                493,
                "PERSON"
            ],
            [
                568,
                575,
                "PERSON"
            ],
            [
                621,
                628,
                "PERSON"
            ],
            [
                786,
                793,
                "PERSON"
            ],
            [
                838,
                846,
                "PERSON"
            ],
            [
                873,
                879,
                "PERSON"
            ],
            [
                889,
                896,
                "PERSON"
            ],
            [
                906,
                912,
                "PERSON"
            ],
            [
                930,
                935,
                "PERSON"
            ],
            [
                1582,
                1588,
                "PERSON"
            ],
            [
                1609,
                1616,
                "PERSON"
            ],
            [
                1700,
                1706,
                "PERSON"
            ],
            [
                1730,
                1736,
                "PERSON"
            ],
            [
                1750,
                1756,
                "PERSON"
            ],
            [
                1817,
                1824,
                "PERSON"
            ],
            [
                2025,
                2031,
                "PERSON"
            ],
            [
                2083,
                2090,
                "PERSON"
            ],
            [
                2254,
                2262,
                "PERSON"
            ],
            [
                2336,
                2343,
                "PERSON"
            ],
            [
                2432,
                2439,
                "PERSON"
            ],
            [
                2456,
                2465,
                "PERSON"
            ],
            [
                2496,
                2503,
                "PERSON"
            ],
            [
                2533,
                2544,
                "PERSON"
            ],
            [
                2587,
                2594,
                "PERSON"
            ],
            [
                2603,
                2611,
                "PERSON"
            ],
            [
                2662,
                2669,
                "PERSON"
            ],
            [
                2680,
                2689,
                "PERSON"
            ],
            [
                2778,
                2787,
                "PERSON"
            ],
            [
                3142,
                3150,
                "PERSON"
            ],
            [
                3173,
                3182,
                "PERSON"
            ],
            [
                3200,
                3207,
                "PERSON"
            ],
            [
                3236,
                3243,
                "PERSON"
            ],
            [
                3276,
                3283,
                "PERSON"
            ],
            [
                3347,
                3353,
                "PERSON"
            ],
            [
                3445,
                3454,
                "PERSON"
            ],
            [
                3472,
                3483,
                "PERSON"
            ],
            [
                3530,
                3536,
                "PERSON"
            ],
            [
                3569,
                3576,
                "PERSON"
            ],
            [
                3581,
                3588,
                "PERSON"
            ],
            [
                3659,
                3668,
                "PERSON"
            ],
            [
                3673,
                3681,
                "PERSON"
            ],
            [
                3747,
                3754,
                "PERSON"
            ],
            [
                3816,
                3823,
                "PERSON"
            ],
            [
                3840,
                3848,
                "PERSON"
            ],
            [
                3883,
                3889,
                "PERSON"
            ],
            [
                3928,
                3934,
                "PERSON"
            ],
            [
                3950,
                3957,
                "PERSON"
            ],
            [
                4013,
                4021,
                "PERSON"
            ],
            [
                4065,
                4071,
                "PERSON"
            ],
            [
                4131,
                4137,
                "PERSON"
            ],
            [
                4198,
                4204,
                "PERSON"
            ],
            [
                4271,
                4281,
                "PERSON"
            ],
            [
                4312,
                4318,
                "PERSON"
            ],
            [
                4390,
                4398,
                "PERSON"
            ],
            [
                4425,
                4429,
                "PERSON"
            ],
            [
                4529,
                4538,
                "PERSON"
            ],
            [
                4554,
                4561,
                "PERSON"
            ],
            [
                4607,
                4616,
                "PERSON"
            ],
            [
                4632,
                4639,
                "PERSON"
            ],
            [
                4656,
                4663,
                "PERSON"
            ],
            [
                4757,
                4766,
                "PERSON"
            ],
            [
                4869,
                4877,
                "PERSON"
            ],
            [
                4891,
                4899,
                "PERSON"
            ],
            [
                4911,
                4918,
                "PERSON"
            ],
            [
                4930,
                4938,
                "PERSON"
            ],
            [
                5035,
                5042,
                "PERSON"
            ],
            [
                5100,
                5106,
                "PERSON"
            ],
            [
                5152,
                5158,
                "PERSON"
            ],
            [
                5232,
                5239,
                "PERSON"
            ],
            [
                5257,
                5267,
                "PERSON"
            ],
            [
                5301,
                5308,
                "PERSON"
            ],
            [
                5421,
                5427,
                "PERSON"
            ],
            [
                5519,
                5524,
                "PERSON"
            ],
            [
                5538,
                5545,
                "PERSON"
            ],
            [
                5554,
                5559,
                "PERSON"
            ],
            [
                5642,
                5649,
                "PERSON"
            ],
            [
                5678,
                5687,
                "PERSON"
            ],
            [
                5738,
                5745,
                "PERSON"
            ],
            [
                5774,
                5781,
                "PERSON"
            ],
            [
                5816,
                5823,
                "PERSON"
            ],
            [
                5919,
                5926,
                "PERSON"
            ],
            [
                5940,
                5949,
                "PERSON"
            ],
            [
                5967,
                5974,
                "PERSON"
            ],
            [
                6174,
                6182,
                "PERSON"
            ],
            [
                6187,
                6192,
                "PERSON"
            ],
            [
                6213,
                6219,
                "PERSON"
            ],
            [
                6225,
                6232,
                "PERSON"
            ],
            [
                6247,
                6252,
                "PERSON"
            ],
            [
                6267,
                6275,
                "PERSON"
            ],
            [
                6326,
                6334,
                "PERSON"
            ],
            [
                6354,
                6361,
                "PERSON"
            ],
            [
                6395,
                6403,
                "PERSON"
            ],
            [
                6529,
                6538,
                "PERSON"
            ]
        ]
    }
),(
    " { Deonna vumc franklin edward curd white, tyron { Mylee e 2105 edward cu { Talan rd lane mrn: 047717361, dob: 7/27/1969, legal sex: m suite b100 visit date: 11/27/202 { Laverna 3 franklin tn 37067 11/27/2023 - appointment in vanderbilt { Dillard  neurology franklin (continued) medication list (continued) instructions: take 1 tablet (20 mg total) by mouth daily. authorized by: greenspan, debra l, aprn ordered on: 7/9/2024 start date: 7/9/2024 end date: 7/9/2025 quantity: 30 tablet refill: { Augustin  11 re { Loran fills by 7/9/2025 pre { Geovanni dnisolone acetate 1 % eye dro { Jerel p { Delta s,suspension (pred forte) instructions: after surgery,  { Dequan use 1 drop to the right eye  { Ansel every 2  { Malaki hours while awake until bedtime.  { Kyrie beginning the next day, decrease to 1 dr { Konrad op to the right ey { Randel e 4 times a day for 1 week, then 3 times a day for 1 week, then 2 times a day fo { Lorenza r 1 w { Donal eek, then daily for 1 { Kiya  week, then stop autho { Annamaria rized by: valenzuela, daniel alejandro, md ordered on: 7/22/ { Kwame 2024 start date: 7/22/2024 quantity: 5 { Courtland  ml refill: 1 refill  { Jeremey by 7/2 { Avraham 2/2025 moxifloxacin 0.5 % eye drops (vigamox) instruct { Gerson ions: after sur { Yulissa gery, use 1 drop to the right eye  { Zaira every 2 hour { Miguelangel s while awake until bedtime. beginning the next day, decrease to 1 dro { Ariya p to the { Kohen  right eye 4 times  { Cyndi a day for { Homero  1 week, then stop authorized by: valenzuela, da { Rahul niel alejandro, m { Karolina d ordered on: 7/22/2024 start date: 7/22/20 { Clarke 24 qu { Tiarra antity: 3 ml refill: 1 refill by 7/22/2025 clotrimazole 1 % topical cream (lotrimin) instructions: apply 1 application topically 2 times a day for 30 days. authorized by: pauw, emily kathryn, md { Ethen  o { Shane rd { Jeffrey ered on: 7/26/2024 start date: 7/26/2024 quantity: 28 g refill:  remaining albuterol sulfate hfa 90 mcg/actuation aerosol inhaler instructions: inhale 2 puffs ev { Lavonda ery 4 hours as needed for wheezing. a { Haden uthorize { Phillis d by: chak { Tyanna ravarthy, rohini { Grayson , md ordered on: 8/6/2024 sta { Conley rt date: 8/6/2024 quantity: 18 g refill: 11 refills by 8/6/2025 fluticasone propionate 50 mcg/actuation nasal spray,suspension (flon { Loreen ase) i { Enos nstructions: administer 2 sprays into { Sampson  each nostril 2 times a day. authorized by: connolly, patrick james, md ordered on: 8/6/2024 start date: 8/6/2024 quantity: 16 g refill: 2 refills by 8/6/2025 gabapentin 300 m { Jones g ca { Kirt psule (neurontin) instructions: take 2 capsules (600 mg total) by mouth daily. authorized by: chakravarthy, rohini, md ordered on: 8/6/2024 s { Cailin tart date: 8/6/2024 quantity: 180 capsule { Brien  refill:  remaining ipratropium bromide 42 mcg (0.06 ° nasal spray (atrovent) instructions: a { Alanah dminister  { Lenna 1 spray into each nostril 4 times a day. start with 1 spray once daily for one week, then increase to 2 sprays once a day for 2 weeks, can  { Dorothy increase to 3-4 sprays { Mayson  a day authorized by: chakravarthy, rohini, md ordered on: 8/6/2024 start date: 8/6/2024 quantity: 15 ml refill: 12 refills { Cassidy  by 8/6/2025 loperamide 2 mg capsule (imodium) instructions: take 1 capsule (2 mg total) by mouth 3 tim { Vonnie e { Deneen s a day as needed for diarrhea for up to 10 days. authorized by: chakravarthy, rohini, md ordered on: 9/3/2024 start date: 9/3/2024 quantity: 30 capsule refill:  remaining losartan 25 mg tablet (cozaar) [ { Andie reconcil { Candida ed by gingrow, barbara, lpn on 9/16/2024 0800 { Sheyla ] entered by: gingrow, barbara, lp { Clemente n entered on: 9/16/2024 pri { Rowdy nted on 10/3/24 7:13 am page 1487,vumc  { Kimberlyn franklin edward curd w { Jaheim hite, tyrone 2105 edward curd lane mrn: 047717361 { Mel , dob: 7/27/1969, legal sex: m suite b100 visit date: 11/27/2023 franklin tn 37067 11/27/2023 - appointment in van { Letty derbilt neurology franklin (continued) medication list (continued) start date: 8/27/2024 urea 20 % topical cream (carmol) instructions: apply 1 application topically as needed for dry skin (apply 1 gram to affected area) for u { Janene p to 365 days. authorized by: hicks, adam br { Haskell adburn, dpm orde { Lou red on: 9/16/2024 start date:  { Walter 9/16/2024  { Cailey quantity: 85 g refill: 1 refi { Jodie ll by 9/16/20 { Emalee 25 atorvastatin 80 mg tablet (lipitor) instructio { Augustine ns: take one tablet by mouth every day authorized by: ch { Duwayne akravarthy, rohini, md o { Dalilah rdered on: 10/2/2024  { Aidyn start date: 10/2/2024 quantity: 90 tablet refill: 1 refill by 10/2/2025 stopped in visit none referral office visit - consult or new patient #12328268 [last edited by epic,  { Juwan user on 12/6/2023 0208]  { Obed reason: specialty services required priority: routine class: int { Alli ernal status: canceled status updated on: 12/6/2023 valid dates: from 11/29/2022 to 1/3/2024 referred from location: vumc henders { Jarett onvill { Dasha e - anderson department: primary care hendersonville provider: lippard, giles a, { Sal  aprn prov { Cloe ider phon { Louvenia e: 615-322-1510 provider address: 128 north anderson lane hendersonville tn 37075 vis { Jennette its request { Zulema ed: 1 authorized: 1 completed: 0 scheduled: 0 diagnoses r26.9 (icd-10-cm) - abnormal gait order ambulatory referral to neurology [334939281] electronicall { Merissa y signed by: lippard, giles a, aprn on 11/29/22 1609 status: discontinued ordering user: lippard, giles a, aprn 11/29/22 1609 ordering provider: lippard, giles a, aprn authorized by: lippar { Bodie d, giles a, aprn ordered  { Sutton during: orders only on 11/29/2022 discontinued by: epic, user 12/06/23 0208 [order expired] diagnoses abnormal gait [r26.9] screening form ge { Lucca neral information patient na { Janaya me { Vincenza : white, tyrone mrn: 047717361 date of birth: 7/27/1969 home phone: 615-260-2291 legal sex: male mobile: 615-260-2291 procedure ordering provider a { Jacki uthorizing provider app { Angella ointment informa { Mikhail tion { Mahogany  amb referral to lippard, giles a, aprn lippard, giles a, aprn neurology 615-322-1510 615-322-1510 screening form questions printed on 10/3/24 7:13 am page  { Betty 1488",
    {
        "entities": [
            [
                3,
                10,
                "PERSON"
            ],
            [
                51,
                57,
                "PERSON"
            ],
            [
                76,
                82,
                "PERSON"
            ],
            [
                170,
                178,
                "PERSON"
            ],
            [
                239,
                247,
                "PERSON"
            ],
            [
                496,
                505,
                "PERSON"
            ],
            [
                514,
                520,
                "PERSON"
            ],
            [
                544,
                553,
                "PERSON"
            ],
            [
                585,
                591,
                "PERSON"
            ],
            [
                595,
                601,
                "PERSON"
            ],
            [
                659,
                666,
                "PERSON"
            ],
            [
                697,
                703,
                "PERSON"
            ],
            [
                714,
                721,
                "PERSON"
            ],
            [
                757,
                763,
                "PERSON"
            ],
            [
                806,
                813,
                "PERSON"
            ],
            [
                834,
                841,
                "PERSON"
            ],
            [
                924,
                932,
                "PERSON"
            ],
            [
                940,
                946,
                "PERSON"
            ],
            [
                970,
                975,
                "PERSON"
            ],
            [
                1000,
                1010,
                "PERSON"
            ],
            [
                1073,
                1079,
                "PERSON"
            ],
            [
                1120,
                1130,
                "PERSON"
            ],
            [
                1154,
                1162,
                "PERSON"
            ],
            [
                1171,
                1179,
                "PERSON"
            ],
            [
                1236,
                1243,
                "PERSON"
            ],
            [
                1261,
                1269,
                "PERSON"
            ],
            [
                1306,
                1312,
                "PERSON"
            ],
            [
                1327,
                1339,
                "PERSON"
            ],
            [
                1412,
                1418,
                "PERSON"
            ],
            [
                1429,
                1435,
                "PERSON"
            ],
            [
                1457,
                1463,
                "PERSON"
            ],
            [
                1475,
                1482,
                "PERSON"
            ],
            [
                1533,
                1539,
                "PERSON"
            ],
            [
                1559,
                1568,
                "PERSON"
            ],
            [
                1614,
                1621,
                "PERSON"
            ],
            [
                1629,
                1636,
                "PERSON"
            ],
            [
                1833,
                1839,
                "PERSON"
            ],
            [
                1844,
                1850,
                "PERSON"
            ],
            [
                1855,
                1863,
                "PERSON"
            ],
            [
                2027,
                2035,
                "PERSON"
            ],
            [
                2075,
                2081,
                "PERSON"
            ],
            [
                2092,
                2100,
                "PERSON"
            ],
            [
                2113,
                2120,
                "PERSON"
            ],
            [
                2139,
                2147,
                "PERSON"
            ],
            [
                2179,
                2186,
                "PERSON"
            ],
            [
                2321,
                2328,
                "PERSON"
            ],
            [
                2337,
                2342,
                "PERSON"
            ],
            [
                2382,
                2390,
                "PERSON"
            ],
            [
                2568,
                2574,
                "PERSON"
            ],
            [
                2581,
                2586,
                "PERSON"
            ],
            [
                2730,
                2737,
                "PERSON"
            ],
            [
                2781,
                2787,
                "PERSON"
            ],
            [
                2883,
                2890,
                "PERSON"
            ],
            [
                2903,
                2909,
                "PERSON"
            ],
            [
                3051,
                3059,
                "PERSON"
            ],
            [
                3084,
                3091,
                "PERSON"
            ],
            [
                3217,
                3225,
                "PERSON"
            ],
            [
                3331,
                3338,
                "PERSON"
            ],
            [
                3342,
                3349,
                "PERSON"
            ],
            [
                3556,
                3562,
                "PERSON"
            ],
            [
                3573,
                3581,
                "PERSON"
            ],
            [
                3629,
                3636,
                "PERSON"
            ],
            [
                3673,
                3682,
                "PERSON"
            ],
            [
                3712,
                3718,
                "PERSON"
            ],
            [
                3760,
                3770,
                "PERSON"
            ],
            [
                3795,
                3802,
                "PERSON"
            ],
            [
                3854,
                3858,
                "PERSON"
            ],
            [
                3975,
                3981,
                "PERSON"
            ],
            [
                4210,
                4217,
                "PERSON"
            ],
            [
                4264,
                4272,
                "PERSON"
            ],
            [
                4291,
                4295,
                "PERSON"
            ],
            [
                4328,
                4335,
                "PERSON"
            ],
            [
                4348,
                4355,
                "PERSON"
            ],
            [
                4387,
                4393,
                "PERSON"
            ],
            [
                4409,
                4416,
                "PERSON"
            ],
            [
                4468,
                4478,
                "PERSON"
            ],
            [
                4537,
                4545,
                "PERSON"
            ],
            [
                4572,
                4580,
                "PERSON"
            ],
            [
                4604,
                4610,
                "PERSON"
            ],
            [
                4786,
                4792,
                "PERSON"
            ],
            [
                4819,
                4824,
                "PERSON"
            ],
            [
                4891,
                4896,
                "PERSON"
            ],
            [
                5028,
                5035,
                "PERSON"
            ],
            [
                5044,
                5050,
                "PERSON"
            ],
            [
                5133,
                5137,
                "PERSON"
            ],
            [
                5150,
                5155,
                "PERSON"
            ],
            [
                5167,
                5176,
                "PERSON"
            ],
            [
                5264,
                5273,
                "PERSON"
            ],
            [
                5287,
                5294,
                "PERSON"
            ],
            [
                5451,
                5459,
                "PERSON"
            ],
            [
                5651,
                5657,
                "PERSON"
            ],
            [
                5685,
                5692,
                "PERSON"
            ],
            [
                5836,
                5842,
                "PERSON"
            ],
            [
                5873,
                5880,
                "PERSON"
            ],
            [
                5885,
                5894,
                "PERSON"
            ],
            [
                6044,
                6050,
                "PERSON"
            ],
            [
                6076,
                6084,
                "PERSON"
            ],
            [
                6103,
                6111,
                "PERSON"
            ],
            [
                6118,
                6127,
                "PERSON"
            ],
            [
                6286,
                6292,
                "PERSON"
            ]
        ]
    }
),(
    "v { Brittaney umc hendersonville - anderson white, tyrone 128 { Carolyne  n ander { Jettie son ln mrn:  { Jarret 047717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 visit  { Yael date: 6/6/2023 06/06/2 { Jeramie 023 -  { Bryleigh office { Jael  visit in vander { Ashlea bilt primary care hendersonville  { Brett (continued) clinical notes (continued) chronic k { Erinn idney disease due to type 2 diabetes mellitus (cms/hcc) relevant medications pioglitazone 30 mg tablet (actos) { Omarion  endocrine/metabolic mixed hyperlipidem { Salome ia c { Zandra urrent assessment & plan current regimen { Charisma : -l { Kaylani ovastatin 2 { Marceline 0 mg daily will repeat { Sky  lipid panel today reinforced lifestyle changes continue treatment plan we will continue to monitor other tobacco abuse current asse { Alix ssment & pla { Keyonna n today's screening for tobacc { Giada o use is posit { Larry ive, to { Marni bacco use form includes cigars, tobacco cessation inte { Kenan rvention to { Javonte day is: tobacco cessation counseling gi { Treyvon ve { Janett n. polysubstance abuse { Price  (cms/hcc) current a { Azucena ssessment & plan strongly reinforced to { Latrell  avoid marijua { Kera na and cocai { Jordana ne use. med changes today: (blank if none) start taking nifedipin { Malena e er 30 mg take 1  { Luc tablet (30 mg total) by mouth daily. tablet,extended release (adalat cc) stop taking  { Michel carvedilol 6.25 mg tablet take 1 tablet (6.25 mg tota { Maile l) by mouth { Vihaan  2 times a day (with (coreg) breakfast and dinner). i h { Creighton ave kin { Kalynn dly requested that the patient return to our internal medicine { Karena  cl { Maxx inic as not { Isadora ed: return in about 3 months (around 9/6/2023) for recheck. printed on  { Jakobe 10/3/24 7:13 am page 2121,vumc hendersonville - anderson white, tyrone 128 n { Darvin  anderso { Joseline n ln mrn: 047717361, dob: 7/27/1969, legal sex:  { Myesha m hendersonville tn 3707 { Raymond 5 visit date: 6/6/2023 06/06/2023 - office visit in vander { Joetta bi { Charlton lt primary care hendersonville (continued) clinical notes (continued) giles a lippard, aprn electronically signed by lippa { Rosaline rd, giles a, aprn at 6/9/20 { Georgetta 23 11:31 am electronically signed by craig, kaylin smith { Palma , md at 6/27/2023 2:21 pm laboratory reports hemoglobin a1c (completed) electronically signed by: lippard, giles a, aprn on 06/06/23 1443 status: completed ordering us { Crew er: lippard, giles a, aprn 06/06/23 1443 ordering provider: lippard, giles a, aprn authorized by: lippard, giles a, aprn ordering mode: standard frequency: routine 0 { Sarah 6/0 { Kathlene 6/23  { Sedrick - class: lab col { Torrance lect qua { Leighann ntity: 1 diagnoses type 2 diabe { Tillman tes mellitus with stage 4 chronic kidney  { Chase disease, with long-term current use of insulin  { Drucilla (cms/hcc) [e11. { Lili 22, n18.4, z79.4] annual physical exam [z00.00] s { Sariyah pecimen information id type source collected by - blood  { Raylee - - indications type 2 diabetes mellitus with stage 4 chroni { Dillion c kidney dis { Alise ease, with long-t { Kalea erm c { Chassidy urrent use { Nala  of insulin (cms/hcc) [e11.22, n18.4, z79.4 (icd-10-cm)] annual physical exam [z00.00 (icd-10-cm) { Cy ] lipid pnl (comp { Carri leted) electronically { Jacobi   { Neville signed by: li { Sanjuanita ppard, { Sequoia  giles a,  { Aedan aprn on 06/06/23 1443 status { Jessalyn : completed ordering user: lippard, giles a, aprn 06/06/23 1443 ordering provider: lippard, gile { Judge s a, aprn author { Marlie ized by: lippard, giles a, aprn  { Tracee ordering mode { Rachele : standard frequ { Leeanne ency: routine 06/06/23 - cl { Raelene ass: lab collect quanti { Johny ty: 1 diagnoses annual physical exam [z00.00] speci { Jaycie me { Wynona n information id type source collected { Tyquan  by - blood - - in { Laraine dications annual physical exam [z00.0 { Codi 0 (icd-10-cm) { Mickie ] bmp (completed) electronically signed b { Florida y: lippard, giles a, aprn on 06/06/23 { Annamae  1443 status: completed ordering us { Deedee er: lippard, giles a, aprn 06/06/23 1443 { Porfirio  ordering provider: li { Gracyn ppard, giles a, aprn authorized by: lippard, giles a, aprn orderi { Willam ng mode: s { Rileigh tandard frequency: routine 06/06/23 - class: lab collect quantit { Roxy y: 1 diagnoses annual physical exam [z00.00] specimen information id type source collected by - { Florian  blood - - printed on 10/3/24 7:13 am p { Viva age 2122",
    {
        "entities": [
            [
                4,
                14,
                "PERSON"
            ],
            [
                64,
                73,
                "PERSON"
            ],
            [
                84,
                91,
                "PERSON"
            ],
            [
                106,
                113,
                "PERSON"
            ],
            [
                186,
                191,
                "PERSON"
            ],
            [
                216,
                224,
                "PERSON"
            ],
            [
                233,
                242,
                "PERSON"
            ],
            [
                251,
                256,
                "PERSON"
            ],
            [
                275,
                282,
                "PERSON"
            ],
            [
                318,
                324,
                "PERSON"
            ],
            [
                375,
                381,
                "PERSON"
            ],
            [
                494,
                502,
                "PERSON"
            ],
            [
                544,
                551,
                "PERSON"
            ],
            [
                558,
                565,
                "PERSON"
            ],
            [
                608,
                617,
                "PERSON"
            ],
            [
                624,
                632,
                "PERSON"
            ],
            [
                646,
                656,
                "PERSON"
            ],
            [
                681,
                685,
                "PERSON"
            ],
            [
                820,
                825,
                "PERSON"
            ],
            [
                840,
                848,
                "PERSON"
            ],
            [
                881,
                887,
                "PERSON"
            ],
            [
                904,
                910,
                "PERSON"
            ],
            [
                920,
                926,
                "PERSON"
            ],
            [
                983,
                989,
                "PERSON"
            ],
            [
                1003,
                1011,
                "PERSON"
            ],
            [
                1053,
                1061,
                "PERSON"
            ],
            [
                1066,
                1073,
                "PERSON"
            ],
            [
                1098,
                1104,
                "PERSON"
            ],
            [
                1127,
                1135,
                "PERSON"
            ],
            [
                1177,
                1185,
                "PERSON"
            ],
            [
                1202,
                1207,
                "PERSON"
            ],
            [
                1222,
                1230,
                "PERSON"
            ],
            [
                1298,
                1305,
                "PERSON"
            ],
            [
                1326,
                1330,
                "PERSON"
            ],
            [
                1418,
                1425,
                "PERSON"
            ],
            [
                1481,
                1487,
                "PERSON"
            ],
            [
                1501,
                1508,
                "PERSON"
            ],
            [
                1566,
                1576,
                "PERSON"
            ],
            [
                1586,
                1593,
                "PERSON"
            ],
            [
                1658,
                1665,
                "PERSON"
            ],
            [
                1671,
                1676,
                "PERSON"
            ],
            [
                1690,
                1698,
                "PERSON"
            ],
            [
                1772,
                1779,
                "PERSON"
            ],
            [
                1858,
                1865,
                "PERSON"
            ],
            [
                1876,
                1885,
                "PERSON"
            ],
            [
                1936,
                1943,
                "PERSON"
            ],
            [
                1970,
                1978,
                "PERSON"
            ],
            [
                2039,
                2046,
                "PERSON"
            ],
            [
                2051,
                2060,
                "PERSON"
            ],
            [
                2185,
                2194,
                "PERSON"
            ],
            [
                2224,
                2234,
                "PERSON"
            ],
            [
                2293,
                2299,
                "PERSON"
            ],
            [
                2469,
                2474,
                "PERSON"
            ],
            [
                2642,
                2648,
                "PERSON"
            ],
            [
                2654,
                2663,
                "PERSON"
            ],
            [
                2671,
                2679,
                "PERSON"
            ],
            [
                2698,
                2707,
                "PERSON"
            ],
            [
                2718,
                2727,
                "PERSON"
            ],
            [
                2761,
                2769,
                "PERSON"
            ],
            [
                2813,
                2819,
                "PERSON"
            ],
            [
                2869,
                2878,
                "PERSON"
            ],
            [
                2896,
                2901,
                "PERSON"
            ],
            [
                2953,
                2961,
                "PERSON"
            ],
            [
                3020,
                3027,
                "PERSON"
            ],
            [
                3090,
                3098,
                "PERSON"
            ],
            [
                3113,
                3119,
                "PERSON"
            ],
            [
                3139,
                3145,
                "PERSON"
            ],
            [
                3153,
                3162,
                "PERSON"
            ],
            [
                3175,
                3180,
                "PERSON"
            ],
            [
                3280,
                3283,
                "PERSON"
            ],
            [
                3303,
                3309,
                "PERSON"
            ],
            [
                3333,
                3340,
                "PERSON"
            ],
            [
                3344,
                3352,
                "PERSON"
            ],
            [
                3368,
                3379,
                "PERSON"
            ],
            [
                3388,
                3396,
                "PERSON"
            ],
            [
                3409,
                3415,
                "PERSON"
            ],
            [
                3446,
                3455,
                "PERSON"
            ],
            [
                3554,
                3560,
                "PERSON"
            ],
            [
                3579,
                3586,
                "PERSON"
            ],
            [
                3621,
                3628,
                "PERSON"
            ],
            [
                3644,
                3652,
                "PERSON"
            ],
            [
                3671,
                3679,
                "PERSON"
            ],
            [
                3709,
                3717,
                "PERSON"
            ],
            [
                3743,
                3749,
                "PERSON"
            ],
            [
                3803,
                3810,
                "PERSON"
            ],
            [
                3815,
                3822,
                "PERSON"
            ],
            [
                3863,
                3870,
                "PERSON"
            ],
            [
                3891,
                3899,
                "PERSON"
            ],
            [
                3939,
                3944,
                "PERSON"
            ],
            [
                3960,
                3967,
                "PERSON"
            ],
            [
                4011,
                4019,
                "PERSON"
            ],
            [
                4059,
                4067,
                "PERSON"
            ],
            [
                4105,
                4112,
                "PERSON"
            ],
            [
                4155,
                4164,
                "PERSON"
            ],
            [
                4189,
                4196,
                "PERSON"
            ],
            [
                4264,
                4271,
                "PERSON"
            ],
            [
                4284,
                4292,
                "PERSON"
            ],
            [
                4359,
                4364,
                "PERSON"
            ],
            [
                4462,
                4470,
                "PERSON"
            ],
            [
                4512,
                4517,
                "PERSON"
            ]
        ]
    }
),(
    "vumc the vanderbilt clin { Ceasar ic white, tyrone 1301 medical center dr mrn: 047 { Gaylon 717361, dob: 7/27/1969 { Gian , legal sex: m the vanderbilt cl { Haiden inic visit date: 1/5/2023 nashville tn 37232-0028 01/05/2023  { Titan - appointment in vanderbilt nephrology/renal transplant clinic (continued) { Rosia  medication list (continued) discontinued by: wilson, danya horchi, { Danilo  pharmd discontinued on: 3/1/2023 reason for discontinuation: other (cancelrx) instructions: i { Domenica nject  { Rollie 3 mg under the sk { Channing in every 7 { Dajuan  days. authorized by: lippard, giles a, aprn ordered on: 1/4/2023 start dat { Izayah e: 1/4/2023 end date: 3/1/2023 qu { Keily antity: 2 ml r { Loyce efill: 2 refills by 1/4/2024 cyclobenzaprine 5 mg tablet (flexeril) [reco { Shaelyn nciled by woehler, kristina, rn on 1/11/2023  { Ignatius 0756] discontinued  { Honey by: lehmann, melissa cary, pa-c dis { Madelaine continued on: 6/3/2023 reason for discontinuation: stop taking at disc { Amarion harge (canc { Larisa elrx) instruc { Briggs tions: tak { Lex e 1 tablet (5 mg total) by m { Brook outh every 8  { Karan hours { Loree  as n { Dennise eeded. entered by: woe { Danya hler, kristina, rn entered on: 1/11/2023 st { Graydon art date: 12/7/2022 end da { Zev te: 6/3/2023 { Kentrell  famotidine 20 mg tablet (pepcid) [reconciled by ferguson, sherri l, { Kael  lpn on 1/11/2023  { Koen 1257] instructions: take 1 tablet (20 { Lambert  mg total) by mo { Javan uth every 12 hours. entered b { Nicklaus y: ferguson, sherri l, lpn entered on: 1/11/2023 benzonatate 200 mg capsule (tessalon) discontinued  { Jame by: lehmann { Rolanda , melissa cary, pa-c discon { Thurston tinued on: 6/3/2023 reas { Ciarra on for discontinuation: stop taking at discharge (cancelrx) { Salomon  instructions: take 1 capsule (200 mg to { Coen tal) by mouth 3 times a day as needed for cough. aut { Krystin horized by: lippard, gi { Spring les a, aprn ordered on: 1/11/2 { Jayvon 023 start date: 1/11/2023 end date: 6/3/2023 action: patient not taking quantity:  { Auston 42 capsule refil { Michelle l:  rem { Joye aining ondansetron hcl 4 mg tablet (zofran) discontinued by: lehmann { Ariadne , melis { Azul sa cary, p { Jenae a-c d { Ardella iscontinued o { Ari n:  { Stormie 6/3/ { Magali 2023 reason fo { Manda r discontinuation: stop taking at discharge (cancelrx) instruc { Les tions: take 1 { Werner  tablet (4 mg total) by mo { Abbygail uth 3 t { Adah imes a day as needed for nausea or  { Keegan vomiting. authorized by: lippard, gi { Jonatan les a, aprn ord { Caprice ered on: 1/11/2023 start date: 1/11/2023 end date: 6/3/2023 actio { Sapphire n: patient not taking quantity: 20 tablet refill:  remaining ergocalciferol (vitamin { Deontae  d2) 1,250 mcg (50,000 unit) capsule (vitamin d2)  { Katherin discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for discontinuation: stop taking at { Rebecka  discharge (cancelrx { Zaida ) instructions: take 1 capsule (50,000 units total) by mouth weekly. au { Gentry thorized by: wang, zhijian, aprn or { Jadyn dered { Kymani  on: 2/7/2023 start date: 2/7/2023 end date: 6/3/2023 quantity: 12 capsule refill:  remaining gabapentin 30 { Nicholaus 0 mg capsule (neurontin) discontinued by: darks { Ronni , tina m, aprn discontinued on: 3/13/2023 inst { Sheron ructions: tak { Butch e one ca { Amit psule by mouth three times a day { Leopold  authorized by: lippard, giles a, aprn ordered on { Idell : 2/9/2023  { Vivianna start date:  { Capri 2/9/2023 quantity: 90 capsule refill:  remaining stopped in visit none messages printed on 10/3/24 7:13 am page 2763,vumc the vanderbilt clinic white, tyrone 1301 medical center dr mrn: 047717361, dob: 7/27/1969, legal sex: m the vanderbilt c { Samiya linic v { Keyanna isi { Nicholas t date: 1/5/2023 nashville tn 37232-0028 01/05/2023 - appointment in vanderbilt nephrology/renal transplant c { Lyric linic (continued) messages (continued) appointment rescheduled from to sent a { Trudi nd delivered mychart, ge { Yandel neric white, tyrone 1/4/2023 4:11 pm la { Makala st read in my health at vanderbilt not read appoint { Iain ment information: visit type: retu { Natosha rn visit dat { Cael e: 1/5/2023 dept: vanderbilt nephrology/renal t { Michale ransplant clinic provider: zhijian wang tim { Destany e: 2:30 pm length: 10 min appt status: scheduled original  { Ellery appointment information: { Kalina  visit type: return visit date: 12/21/2022 dept:  { Bernice v { Brecken anderbilt nephrology/renal transplant { Geno  clinic provi { Avalon der: zhijian wang time: 11:00 am length: 10 min printed on 10 { Elba /3/24 7:13 am page { Ivey  276 { Llewellyn 4",
    {
        "entities": [
            [
                27,
                34,
                "PERSON"
            ],
            [
                85,
                92,
                "PERSON"
            ],
            [
                117,
                122,
                "PERSON"
            ],
            [
                157,
                164,
                "PERSON"
            ],
            [
                228,
                234,
                "PERSON"
            ],
            [
                311,
                317,
                "PERSON"
            ],
            [
                387,
                394,
                "PERSON"
            ],
            [
                491,
                500,
                "PERSON"
            ],
            [
                509,
                516,
                "PERSON"
            ],
            [
                536,
                545,
                "PERSON"
            ],
            [
                558,
                565,
                "PERSON"
            ],
            [
                643,
                650,
                "PERSON"
            ],
            [
                686,
                692,
                "PERSON"
            ],
            [
                709,
                715,
                "PERSON"
            ],
            [
                791,
                799,
                "PERSON"
            ],
            [
                847,
                856,
                "PERSON"
            ],
            [
                878,
                884,
                "PERSON"
            ],
            [
                922,
                932,
                "PERSON"
            ],
            [
                1005,
                1013,
                "PERSON"
            ],
            [
                1027,
                1034,
                "PERSON"
            ],
            [
                1050,
                1057,
                "PERSON"
            ],
            [
                1070,
                1074,
                "PERSON"
            ],
            [
                1105,
                1111,
                "PERSON"
            ],
            [
                1127,
                1133,
                "PERSON"
            ],
            [
                1141,
                1147,
                "PERSON"
            ],
            [
                1155,
                1163,
                "PERSON"
            ],
            [
                1188,
                1194,
                "PERSON"
            ],
            [
                1240,
                1248,
                "PERSON"
            ],
            [
                1277,
                1281,
                "PERSON"
            ],
            [
                1296,
                1305,
                "PERSON"
            ],
            [
                1376,
                1381,
                "PERSON"
            ],
            [
                1402,
                1407,
                "PERSON"
            ],
            [
                1447,
                1455,
                "PERSON"
            ],
            [
                1474,
                1480,
                "PERSON"
            ],
            [
                1512,
                1521,
                "PERSON"
            ],
            [
                1624,
                1629,
                "PERSON"
            ],
            [
                1643,
                1651,
                "PERSON"
            ],
            [
                1681,
                1690,
                "PERSON"
            ],
            [
                1717,
                1724,
                "PERSON"
            ],
            [
                1786,
                1794,
                "PERSON"
            ],
            [
                1837,
                1842,
                "PERSON"
            ],
            [
                1897,
                1905,
                "PERSON"
            ],
            [
                1931,
                1938,
                "PERSON"
            ],
            [
                1971,
                1978,
                "PERSON"
            ],
            [
                2063,
                2070,
                "PERSON"
            ],
            [
                2089,
                2098,
                "PERSON"
            ],
            [
                2108,
                2113,
                "PERSON"
            ],
            [
                2184,
                2192,
                "PERSON"
            ],
            [
                2202,
                2207,
                "PERSON"
            ],
            [
                2220,
                2226,
                "PERSON"
            ],
            [
                2234,
                2242,
                "PERSON"
            ],
            [
                2258,
                2262,
                "PERSON"
            ],
            [
                2268,
                2276,
                "PERSON"
            ],
            [
                2283,
                2290,
                "PERSON"
            ],
            [
                2307,
                2313,
                "PERSON"
            ],
            [
                2378,
                2382,
                "PERSON"
            ],
            [
                2398,
                2405,
                "PERSON"
            ],
            [
                2434,
                2443,
                "PERSON"
            ],
            [
                2453,
                2458,
                "PERSON"
            ],
            [
                2496,
                2503,
                "PERSON"
            ],
            [
                2542,
                2550,
                "PERSON"
            ],
            [
                2568,
                2576,
                "PERSON"
            ],
            [
                2644,
                2653,
                "PERSON"
            ],
            [
                2740,
                2748,
                "PERSON"
            ],
            [
                2801,
                2810,
                "PERSON"
            ],
            [
                2926,
                2934,
                "PERSON"
            ],
            [
                2957,
                2963,
                "PERSON"
            ],
            [
                3037,
                3044,
                "PERSON"
            ],
            [
                3082,
                3088,
                "PERSON"
            ],
            [
                3096,
                3103,
                "PERSON"
            ],
            [
                3213,
                3223,
                "PERSON"
            ],
            [
                3273,
                3279,
                "PERSON"
            ],
            [
                3328,
                3335,
                "PERSON"
            ],
            [
                3351,
                3357,
                "PERSON"
            ],
            [
                3368,
                3373,
                "PERSON"
            ],
            [
                3408,
                3416,
                "PERSON"
            ],
            [
                3468,
                3474,
                "PERSON"
            ],
            [
                3488,
                3497,
                "PERSON"
            ],
            [
                3512,
                3518,
                "PERSON"
            ],
            [
                3763,
                3770,
                "PERSON"
            ],
            [
                3780,
                3788,
                "PERSON"
            ],
            [
                3794,
                3803,
                "PERSON"
            ],
            [
                3915,
                3921,
                "PERSON"
            ],
            [
                4001,
                4007,
                "PERSON"
            ],
            [
                4034,
                4041,
                "PERSON"
            ],
            [
                4083,
                4090,
                "PERSON"
            ],
            [
                4144,
                4149,
                "PERSON"
            ],
            [
                4186,
                4194,
                "PERSON"
            ],
            [
                4209,
                4214,
                "PERSON"
            ],
            [
                4264,
                4272,
                "PERSON"
            ],
            [
                4318,
                4326,
                "PERSON"
            ],
            [
                4387,
                4394,
                "PERSON"
            ],
            [
                4421,
                4428,
                "PERSON"
            ],
            [
                4480,
                4488,
                "PERSON"
            ],
            [
                4492,
                4500,
                "PERSON"
            ],
            [
                4540,
                4545,
                "PERSON"
            ],
            [
                4561,
                4568,
                "PERSON"
            ],
            [
                4632,
                4637,
                "PERSON"
            ],
            [
                4658,
                4663,
                "PERSON"
            ],
            [
                4670,
                4680,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital w { Cortney hite, tyrone 1211 medical center dr. mrn: 047717361, dob: { Elenora   { Soleil 7/27/1969, leg { Larae al sex: m nas { Eugenie hvil { Eben le tn 37232-0004 adm: { Leela  8/22/2023, d/c: 8/23/2023 08/2 { Vianey 2/2023 - ed to hosp-admission (discha { Phylis rged) in vanderbilt emergency department (continued) messages (continued) not read hi mr.  { Guido white, thank you for letting us care  { Yael for you during your rece { Aleksander nt vi { Arnav sit { Janetta  on 8/22/2023. don't forget that you can revi { Bradlee ew your after-visit summary by navigating { Vernell  to the epichttp://recentappts { Chelsy [vi { Leda sit { Suellen  summaries] page. s { Merlyn incerely, your car { Collins e team printed on 10/3/24 7:13 am page 1979,vumc adult hospi { Montana tal white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 8/22/2023, d/c: 8/2 { Retta 3/2023 08/22/2023 - ed to ho { Keturah sp-admissio { Vannessa n (discharged) in vanderbilt emergency  { Jamin d { Kyan epartment ( { Emmalynn continued)  { Billye lmr encounter level scans consent for { Jonelle  routine trea { Marylee tment (hospital) -  { Darrian electronic  { Janee signature on 8/22/2023 10:30 pm (eff { Latesha ect { Ivy ive from 8/22/2023) - { Makenzi  e-signed vand { Kannon erbilt { Lamonte  university medical center consent for healthcare se { Altha rvices vanderbilt uni { Nakita versity medical center { Christoper  (vumc) is an academic { Alonza  medical center. our mission is t { Kathern o: provide high-quality care train healt { Austyn h care  { Levy professionals perform resea { Gary rch to improve health care for you and other patient { Alianna s. i understand th { Aura at during my care, vumc may: treat me as my doctors and health care team direct (including giv { Teresita ing medications, drawing blood and performing other procedur { Arie es { Lonna ) sedate me { Sari  mildly with drugs, if needed for medical reason { Nikko s test my blood for infections if someon { Sahil e is exposed to my blood { Sabastian  or body flu { Marylyn ids collect { Ananya , { Catarina  dispose o { Jacy f, or use information,  { Yoel fluid, or tissue taken from me during tests, treatment, { Jenessa  or  { Buffy surgery to better understand my health and to improv { Delisa e health care for others test my { Codey  genes or dna, if { Otho  needed, to help with my medical issues and find the best ways to treat them take photos or videos of me, if needed, to treat  { Jurnee or identify me us { Tanna e treatments adminis { Ailyn tered to me as an opportunity t { Baylie o study and learn how to improve { Charolette  the care for me and others contact me about research st { Aleisha udies for which i might qualify unless i contact (615) 322-7343  { Makhi to opt-out.  { Talitha i am aware that i have the  { Amparo right to: refuse tests or treatment (as far as the law allows) and { Mirian  to be told  { Boone what might happen if i do ask for { Tito  a copy of \"your rights and responsib { Ena ilities as { Haydee  a patient,\" available in all vanderbilt facili { Nancie ties, in h { Willia andbooks for i { Tamie npatients, and online at vanderbilthealth.com i under { Veronika stand that  are m { Lovell ade about the results of my { Maxton  trea { Archibald tment. vumc may need t { Nickie o share health infor { Brentley mat { Rashaad ion and leftover body fluids o { Nilda r tissue with organizations other than vumc. vumc must follow applicab { Bjorn le privacy { Rochel  laws when doing so, as described in vumc's notice of privacy practices. i will not be paid for any di { Jovita scove { Evert ries or inventions { Berkley   { Autum that might resul { Monroe t from the use of my  { Karah hea { Brieanna lth information and leftover fluid or tissue. printed on 10/3/24  { Jazzlyn 7:13 am page 1980",
    {
        "entities": [
            [
                24,
                32,
                "PERSON"
            ],
            [
                92,
                100,
                "PERSON"
            ],
            [
                104,
                111,
                "PERSON"
            ],
            [
                128,
                134,
                "PERSON"
            ],
            [
                150,
                158,
                "PERSON"
            ],
            [
                165,
                170,
                "PERSON"
            ],
            [
                194,
                200,
                "PERSON"
            ],
            [
                234,
                241,
                "PERSON"
            ],
            [
                281,
                288,
                "PERSON"
            ],
            [
                381,
                387,
                "PERSON"
            ],
            [
                427,
                432,
                "PERSON"
            ],
            [
                459,
                470,
                "PERSON"
            ],
            [
                478,
                484,
                "PERSON"
            ],
            [
                490,
                498,
                "PERSON"
            ],
            [
                546,
                554,
                "PERSON"
            ],
            [
                598,
                606,
                "PERSON"
            ],
            [
                639,
                646,
                "PERSON"
            ],
            [
                652,
                657,
                "PERSON"
            ],
            [
                663,
                671,
                "PERSON"
            ],
            [
                693,
                700,
                "PERSON"
            ],
            [
                721,
                729,
                "PERSON"
            ],
            [
                792,
                800,
                "PERSON"
            ],
            [
                938,
                944,
                "PERSON"
            ],
            [
                975,
                983,
                "PERSON"
            ],
            [
                997,
                1006,
                "PERSON"
            ],
            [
                1048,
                1054,
                "PERSON"
            ],
            [
                1058,
                1063,
                "PERSON"
            ],
            [
                1077,
                1086,
                "PERSON"
            ],
            [
                1100,
                1107,
                "PERSON"
            ],
            [
                1147,
                1155,
                "PERSON"
            ],
            [
                1171,
                1179,
                "PERSON"
            ],
            [
                1201,
                1209,
                "PERSON"
            ],
            [
                1223,
                1229,
                "PERSON"
            ],
            [
                1268,
                1276,
                "PERSON"
            ],
            [
                1282,
                1286,
                "PERSON"
            ],
            [
                1310,
                1318,
                "PERSON"
            ],
            [
                1335,
                1342,
                "PERSON"
            ],
            [
                1351,
                1359,
                "PERSON"
            ],
            [
                1414,
                1420,
                "PERSON"
            ],
            [
                1444,
                1451,
                "PERSON"
            ],
            [
                1476,
                1487,
                "PERSON"
            ],
            [
                1512,
                1519,
                "PERSON"
            ],
            [
                1555,
                1563,
                "PERSON"
            ],
            [
                1606,
                1613,
                "PERSON"
            ],
            [
                1623,
                1628,
                "PERSON"
            ],
            [
                1658,
                1663,
                "PERSON"
            ],
            [
                1718,
                1726,
                "PERSON"
            ],
            [
                1747,
                1752,
                "PERSON"
            ],
            [
                1849,
                1858,
                "PERSON"
            ],
            [
                1921,
                1926,
                "PERSON"
            ],
            [
                1931,
                1937,
                "PERSON"
            ],
            [
                1951,
                1956,
                "PERSON"
            ],
            [
                2007,
                2013,
                "PERSON"
            ],
            [
                2056,
                2062,
                "PERSON"
            ],
            [
                2089,
                2099,
                "PERSON"
            ],
            [
                2114,
                2122,
                "PERSON"
            ],
            [
                2136,
                2143,
                "PERSON"
            ],
            [
                2147,
                2156,
                "PERSON"
            ],
            [
                2169,
                2174,
                "PERSON"
            ],
            [
                2200,
                2205,
                "PERSON"
            ],
            [
                2263,
                2271,
                "PERSON"
            ],
            [
                2278,
                2284,
                "PERSON"
            ],
            [
                2339,
                2346,
                "PERSON"
            ],
            [
                2381,
                2387,
                "PERSON"
            ],
            [
                2407,
                2412,
                "PERSON"
            ],
            [
                2541,
                2548,
                "PERSON"
            ],
            [
                2568,
                2574,
                "PERSON"
            ],
            [
                2597,
                2603,
                "PERSON"
            ],
            [
                2637,
                2644,
                "PERSON"
            ],
            [
                2679,
                2690,
                "PERSON"
            ],
            [
                2749,
                2757,
                "PERSON"
            ],
            [
                2824,
                2830,
                "PERSON"
            ],
            [
                2845,
                2853,
                "PERSON"
            ],
            [
                2883,
                2890,
                "PERSON"
            ],
            [
                2959,
                2966,
                "PERSON"
            ],
            [
                2981,
                2987,
                "PERSON"
            ],
            [
                3023,
                3028,
                "PERSON"
            ],
            [
                3068,
                3072,
                "PERSON"
            ],
            [
                3085,
                3092,
                "PERSON"
            ],
            [
                3142,
                3149,
                "PERSON"
            ],
            [
                3162,
                3169,
                "PERSON"
            ],
            [
                3186,
                3192,
                "PERSON"
            ],
            [
                3248,
                3257,
                "PERSON"
            ],
            [
                3277,
                3284,
                "PERSON"
            ],
            [
                3314,
                3321,
                "PERSON"
            ],
            [
                3329,
                3339,
                "PERSON"
            ],
            [
                3364,
                3371,
                "PERSON"
            ],
            [
                3394,
                3403,
                "PERSON"
            ],
            [
                3409,
                3417,
                "PERSON"
            ],
            [
                3450,
                3456,
                "PERSON"
            ],
            [
                3529,
                3535,
                "PERSON"
            ],
            [
                3548,
                3555,
                "PERSON"
            ],
            [
                3660,
                3667,
                "PERSON"
            ],
            [
                3675,
                3681,
                "PERSON"
            ],
            [
                3702,
                3710,
                "PERSON"
            ],
            [
                3714,
                3720,
                "PERSON"
            ],
            [
                3739,
                3746,
                "PERSON"
            ],
            [
                3770,
                3776,
                "PERSON"
            ],
            [
                3782,
                3791,
                "PERSON"
            ],
            [
                3859,
                3867,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital whi { Arleth te, tyrone 1211 me { Keziah dical center dr. mrn: 047717361, dob: 7/27/1969, leg { Creed al sex: m nashville tn 37232-0004 adm: 6/2/2023, d/c: 6/3/ { Delila 2023 06/02/2023 - ed to hosp-admission (discharged) in va { Amador nderbilt  { Armon universi { Chas ty adult hospital (continued) laboratory reports (continued) ordering user: { Kaytlin  interface, lab results  { Jory in 06/02/23 1414 ordering provider: bickett, christopher rya { Babette n, md authorized by: bickett, christopher ryan, md ord { Jalissa ering mode: s { Meggan ta { June ndard frequency:  { Carsen stat once 06/02/23 1414 - 1 occurrence class: unit coll { Cristi ect quanti { Kierstin ty: 1 lab status: final result specimen informati { Harmoni on id type source collected by 23-153-008765 blood - sde 06/02/23 1414  { Maison auto diff resulted: 06/02/23 1427, result status: final result or { Timmie derin { Mohamad g provider: bickett, chri { Emerie stopher ryan, md 06/02/23 1414 order status: completed filed by: interface, la { Thatcher b results in 06/02/23  { Lianne 1427 collected by: sde 06/02/23 1414 resulting lab: vumc cerner lab c { Rosamond omponents component value reference range flag lab neutr { Julieann ophils 56.8 % cerner comment: per cap, reference ranges should not be reported  { Johnetta for percent cell c { Arnetta ounts when reporting reference ra { Laine nges for absolute number counts to prevent misinterpreta { Stephania tion of wbc d { Edmundo ifferential data. ab { Corban solute ne { Lexis utrophils 3.50 1.60 8.10 - cerner x10(3)/mcl l { Jeremias ymphs  { Pia 30.4 % - cerner comment: per cap, reference rang { Dyllan es should not be reported for percent cell  { Joziah counts when reporting reference range { Chip s for absolute number counts to prevent misinterpretation of wbc diffe { Ilona rential data. absolute lymphocytes 1.87 1.10 3.50 - cerner x10(3)/mcl monocytes 5.2 % - cerner comment: { Jolee  per cap, reference ranges should not be reported for percent cell counts when re { Robb porting reference ranges { Amayah  for absolute number  { Kelcie counts to prevent misinterpretation of wbc differential data. absolute monocytes 0.32 0.30 1.10 - cerner x10(3 { Ever )/mcl eo { Genna sinophils 6.2 % - cerner comment: per cap, reference ranges should not be repor { Keona ted for percent cell counts when reporting reference ranges for absolute number counts to prevent misinterpretation of wbc differential data. absolute eosinophils 0.38 0.00 0.50 - cerner x10(3)/mcl basophils 1.1 % - cerner comment: per cap, reference ranges should not be reported for percent cell counts when reporting reference ranges for absolute number counts to prevent misinterpret { Jacquelyne ation of wbc differential data. absolute basophils 0.07 0.01 0.08 - cerner x10(3)/mcl imm gran automated 0.3 % - cerner absolute imm gran automated 0.02 0.00 0.03 - cerner x10 { Izabel (3)/mcl printed on 10/3/24 7:13 am page 2243,vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 372 { Luiz 32-0004 adm: 6/2/2023, d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt univer { Sanai sity adult hos { Audree pital (continued) laboratory reports (continued) testing performed by lab - abbrevi { Carl ation name  { Allyn director address valid date range 123 cerner vumc cerner lab adam seegmiller; 4605 tvc vumc 1 { Aubrielle 1/22/21 101 { Isidore 4 - present jennifer b. 1301 medical center gordetsky drive nashvill { Rolf e tn 37232- 5310 type/scrn (abo/rh/ab s { Yanira crn) (final result) electronically signed by: bickett, christopher ryan, md on 06/02/23 1408 status: completed ordering user: bickett, chri { Amethyst stopher ry { Nisha an, md 06/02/23 1408 ord { Ezell erin { Mikal g provider: bickett, christopher ryan, md aut { Karoline horized by:  { Godfrey bickett, christopher rya { Doris n, { Octavius  md ordering { Davie  mode { Kelsy : standard frequency: stat stat 06/02/23 1405 - 1 occurrence class: unit collect quantity: 1 la { Nechama b status: final result instance released by: bic { Shonna kett, christopher ryan, md  { Newell (auto-released) 6/2/2023 2:08 pm questionnaire question answer was the patient scanned electronically? ye { Breeanna s specimen information id type source col { Julian lected by 79385909 blood - greenwood-simp { Odalys son,  { Kecia elizabeth, rn 06/02/23 1419 type/scrn (abo/rh/ab scrn) resulted: 06/02/23 1517 { Kathyrn , result status: final result ordering provider: bickett, christopher ryan, md 06/02/23 1408 order status: completed filed by: product tests 06/0 { Lashawn 2/ { Caron 23 1518 { Sharon  collected by: gr { Arthur eenwood-simpson, elizabeth, rn 06/02/23 1419 resulting lab: vumc blood ban { Janiece k  { Machelle components component value reference range flag lab abo type o - - vumo blood bank rh type auto pos - - vumc blood bank ab screen neg - - vumc blood bank specimen expiration 06/05/2023 - - vum { Mackenna c blood bank 23:59 testing performed by lab - abbreviation name director address valid date range 127 - vumc blood vumc blood bank adam seegmiller 1301 medical center dr, 03/11/20 1 { Shavonne 443 - present ba { Maite nk 4605 { Neta -tvc nashville tn 37232 poc lytes-ica-glu-b { Marilou un { Shaun -creat-hct-hgb-handheld (final resul { Theo t) elec { Clive tronically signed by: interface, lab results in on 06/02/23 1428  { Aditi s { Jethro tatus: completed ordering user: in { Brookelyn terface, lab results in 06/02/23 1428 orde { Pamella ring provider: jordan, mary kate, md authorized by: jordan, mary kate, md ordering mode: standard frequency: routine once 06/02/23 1429 - 1 occurr { Angelena ence c { Trudie lass: normal quantity: 1 lab  { Lucky status: final res { Lilli ult specimen information id type sou { Jonny rce { Kaleah  collected by 23-153-011324 blood -  { Rafaela 06/02/23 1428 poc lytes-ica-glu-bun-creat-hct-hgb-handheld (abnor { Ara mal) resulted: 06/02/23 1849, result status: final result printed on 10/3/24 7:13 am page 2244",
    {
        "entities": [
            [
                26,
                33,
                "PERSON"
            ],
            [
                54,
                61,
                "PERSON"
            ],
            [
                116,
                122,
                "PERSON"
            ],
            [
                183,
                190,
                "PERSON"
            ],
            [
                250,
                257,
                "PERSON"
            ],
            [
                269,
                275,
                "PERSON"
            ],
            [
                286,
                291,
                "PERSON"
            ],
            [
                369,
                377,
                "PERSON"
            ],
            [
                404,
                409,
                "PERSON"
            ],
            [
                472,
                480,
                "PERSON"
            ],
            [
                537,
                545,
                "PERSON"
            ],
            [
                561,
                568,
                "PERSON"
            ],
            [
                573,
                578,
                "PERSON"
            ],
            [
                598,
                605,
                "PERSON"
            ],
            [
                663,
                670,
                "PERSON"
            ],
            [
                683,
                692,
                "PERSON"
            ],
            [
                744,
                752,
                "PERSON"
            ],
            [
                826,
                833,
                "PERSON"
            ],
            [
                901,
                908,
                "PERSON"
            ],
            [
                916,
                924,
                "PERSON"
            ],
            [
                952,
                959,
                "PERSON"
            ],
            [
                1040,
                1049,
                "PERSON"
            ],
            [
                1074,
                1081,
                "PERSON"
            ],
            [
                1153,
                1162,
                "PERSON"
            ],
            [
                1221,
                1230,
                "PERSON"
            ],
            [
                1312,
                1321,
                "PERSON"
            ],
            [
                1342,
                1350,
                "PERSON"
            ],
            [
                1386,
                1392,
                "PERSON"
            ],
            [
                1451,
                1461,
                "PERSON"
            ],
            [
                1477,
                1485,
                "PERSON"
            ],
            [
                1508,
                1515,
                "PERSON"
            ],
            [
                1527,
                1533,
                "PERSON"
            ],
            [
                1582,
                1591,
                "PERSON"
            ],
            [
                1600,
                1604,
                "PERSON"
            ],
            [
                1655,
                1662,
                "PERSON"
            ],
            [
                1708,
                1715,
                "PERSON"
            ],
            [
                1755,
                1760,
                "PERSON"
            ],
            [
                1833,
                1839,
                "PERSON"
            ],
            [
                1945,
                1951,
                "PERSON"
            ],
            [
                2035,
                2040,
                "PERSON"
            ],
            [
                2067,
                2074,
                "PERSON"
            ],
            [
                2098,
                2105,
                "PERSON"
            ],
            [
                2218,
                2223,
                "PERSON"
            ],
            [
                2234,
                2240,
                "PERSON"
            ],
            [
                2322,
                2328,
                "PERSON"
            ],
            [
                2718,
                2729,
                "PERSON"
            ],
            [
                2907,
                2914,
                "PERSON"
            ],
            [
                3081,
                3086,
                "PERSON"
            ],
            [
                3193,
                3199,
                "PERSON"
            ],
            [
                3216,
                3223,
                "PERSON"
            ],
            [
                3309,
                3314,
                "PERSON"
            ],
            [
                3328,
                3334,
                "PERSON"
            ],
            [
                3430,
                3440,
                "PERSON"
            ],
            [
                3454,
                3462,
                "PERSON"
            ],
            [
                3533,
                3538,
                "PERSON"
            ],
            [
                3580,
                3587,
                "PERSON"
            ],
            [
                3729,
                3738,
                "PERSON"
            ],
            [
                3751,
                3757,
                "PERSON"
            ],
            [
                3784,
                3790,
                "PERSON"
            ],
            [
                3797,
                3803,
                "PERSON"
            ],
            [
                3851,
                3860,
                "PERSON"
            ],
            [
                3875,
                3883,
                "PERSON"
            ],
            [
                3910,
                3916,
                "PERSON"
            ],
            [
                3921,
                3930,
                "PERSON"
            ],
            [
                3945,
                3951,
                "PERSON"
            ],
            [
                3959,
                3965,
                "PERSON"
            ],
            [
                4063,
                4071,
                "PERSON"
            ],
            [
                4122,
                4129,
                "PERSON"
            ],
            [
                4159,
                4166,
                "PERSON"
            ],
            [
                4274,
                4283,
                "PERSON"
            ],
            [
                4327,
                4334,
                "PERSON"
            ],
            [
                4378,
                4385,
                "PERSON"
            ],
            [
                4393,
                4399,
                "PERSON"
            ],
            [
                4480,
                4488,
                "PERSON"
            ],
            [
                4636,
                4644,
                "PERSON"
            ],
            [
                4649,
                4655,
                "PERSON"
            ],
            [
                4665,
                4672,
                "PERSON"
            ],
            [
                4692,
                4699,
                "PERSON"
            ],
            [
                4776,
                4784,
                "PERSON"
            ],
            [
                4789,
                4798,
                "PERSON"
            ],
            [
                4993,
                5002,
                "PERSON"
            ],
            [
                5186,
                5195,
                "PERSON"
            ],
            [
                5214,
                5220,
                "PERSON"
            ],
            [
                5230,
                5235,
                "PERSON"
            ],
            [
                5281,
                5289,
                "PERSON"
            ],
            [
                5294,
                5300,
                "PERSON"
            ],
            [
                5339,
                5344,
                "PERSON"
            ],
            [
                5354,
                5360,
                "PERSON"
            ],
            [
                5428,
                5434,
                "PERSON"
            ],
            [
                5438,
                5445,
                "PERSON"
            ],
            [
                5482,
                5492,
                "PERSON"
            ],
            [
                5537,
                5545,
                "PERSON"
            ],
            [
                5694,
                5703,
                "PERSON"
            ],
            [
                5712,
                5719,
                "PERSON"
            ],
            [
                5751,
                5757,
                "PERSON"
            ],
            [
                5777,
                5783,
                "PERSON"
            ],
            [
                5822,
                5828,
                "PERSON"
            ],
            [
                5834,
                5841,
                "PERSON"
            ],
            [
                5880,
                5888,
                "PERSON"
            ],
            [
                5956,
                5960,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital wh { Imran ite, tyrone 1211 medical center dr. mr { Sylvan n: 047717361, do { Oriana b: 7/27/1969, legal sex: m nashvil { Berneice le tn 37232-0004 adm: 8/14/2024, d/c: 8/15/2024 08/14/2024 - ed in vanderbilt emergency departmen { Azariah t (continue { Kenyatta d) ed care timeline (continued) 20:55: { Merri 35 lab ordered auto di { Dwane ff interface { Torrence , lab results in 2 { Jonnie 0:57:09 registration daniel, completed rebekah a 20:58:11 auto diff resulted collected: 8/14/2024 20:46 last updated: 8/14/2024 20:58 status: final interface, lab result neutrophils: 74.6 % (per  { Deanthony cap, reference ranges shoul { Ellianna d not be reported r { Lesia esults in for percent cell counts when reporting reference ranges for absolute number counts to prevent misinterpretation of wbc differential data. ) absolute neutrophils: 6.57 x10(3)/mc { Musa l [ref rang { Trenten e: 1.60 - 8.10] lym { Jorja phs: 15.6 % (per cap, reference ranges should not be reported for percent cell counts when reporting reference ranges for absolute number counts to prevent misinterpretation of wbc differential data. ) absolute lymphocytes: 1.37 x10(3)/mcl [r { Caydence ef range: 1.10 - 3.50] monocytes: 4.7 % (per cap, reference { Jenesis  ranges should not be reported for percent ce { Shandra ll counts when reporting reference ranges for absolute number counts to prevent misinterpretation of wbc differential data. ) absolute monocytes: 0.41 x10(3)/mcl [ref range: 0.30 - 1.10] eosinophils: 4.7% (per cap, ref { Davey erence r { Candyce anges should not be reported for percent cell counts wh { Magdalen en reporting reference ranges for absolute number counts to prevent misinterpretation of wbc differential data. ) absolute eosinophils: 0.41 x10(3)/mcl [ref range: 0.03 - 0.51] b { Jodee asophils: 0.3 % (per cap, reference ranges should not be reported for percent { Yarely  cell counts when reporting ref { Demarion erence ranges for absolute number counts to prevent misinterpretation of wbc differential data. ) absolute b { Apryl asophils: 0.03 x10(3)/mcl [ref range: 0.01 - 0.08] imm gran automated: 0.1% absolute imm gran automated: 0.01 x10 { Devlin (3)/mcl [ref range: 0.00 - 0.03] (this te { Kymberly st was perfor { Olympia med at: vanderbilt hospital laboratory, clia #44d { Bobbye 0659066,a adam seegmiller md, p { Leilah hd, 1301 medical center drive, 4605 tvc,nashville,tn,37232, { Quenton ) 20:58:12 cbc w/ differential abnormal result  { Jaleesa collected: 8/14/2024 20:46 last updated: 8/14/2024 20:58 interface, lab resulted status: final result white blood cells: 8.8 x10(3)/mcl [ref range: 3.9 results in 1 { Renato 0.7] red { Jenell  blood cells: 4.17 x10(6)/mcl [ref range: 4.50 - 6.00] hemoglobin: 11.6 gm/dl [ref range: 14.0 - 18.1] he { Trish matocrit: 36 % [ref range: 41 - 49] mean cell volume: 86 fl [ref range: 81 - 98] mean cell hemoglobin: 27.8 pg [ref range: 27.0 - 32.0] mean cell hemo { Damarcus gl { Neoma obin concentration: 32.2 gm/dl [ref range: 31 { Keelan .0 - 35.0] rdw sd: 46.5 fl [ { Madie ref range: 37.4 { Darrien  52.4] rdw cv: 14.6 [ref range: 11.1 - 14.3] platelet: 188 x10(3)/mcl [ref range: 135 - 371] mean platelet volume: 11.5 fl [ref range: 9.3 - 12.8] nucleated rbc: /100 wbc [ref range: 0 0] nucl { Brookelynn eated rbc abs: 0.00 x10(3)/mcl [r { Javen ef range: 0.00 - 0.00] auto neutrophil absolute: 6.57 x10(3)/mcl [ref range: { Mirella  1.60 - 8.10] (this test w { Arlan as perfor { Emeline med at: vanderbilt hospital laboratory,clia #44d065906 { Lluvia 6,adam seegmiller md, phd,1301 medical center dr { Elwyn ive, 4605 tvc, nashville, { Vallie tn,37232,) 20:58:13 lab resulted (final result) auto diff interface, lab results in 20:58:13 lab resulted (fin { Arley al result) cbc w/ differential interface, lab results in 20:58:13 collect cbc w/ cbc w/ differential interface, lab differential results in disconti { Saanvi nued 20:58:13 { Ennis  print label for cbc c { Trayvon bc w/ differential in { Tomasa terface, lab w/ differential results in discontinued printed on 10/3/24  { Angeles 7:12 am page 279,vumc adult hospital white, tyrone 1211 medical center dr. m { Rashida r { Helen n: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 8/14/2 { Urijah 024, d/c: 8/15/2024 08/14/2024 - ed in vanderbilt emergency department (continued) ed care timeline (continued) 21:05 psofa psofa scoring { Romaine  (most recent) epic, user overall score: 0 respiratory: () coagulation: 0 liver  { Derrek (bilirubin): () cardiovascular: ()  { Ravi central nervous system: () renal: () other flowsheet entr { Teodoro ies overall score: 0 sofa respiratory: 0 coagulation: 0 liver: 0 cardiovascular: 0 21:06 psof { Eveline a sofa epic, user central nervous system: 0 renal: 0 21:18 ed { Zakiya  quick updates quick updates mcquitty, karrah, updates: patient is resting comfortably; family at bedside rn 21:18 ed plan of care ed plan of care mcquitty, karrah, ed plan of  { Jamiyah care: { Jovanna  md evalua { Clarisa tion; me { Sean dications; pain management; comfort rn measures; iv assessment; observation; teaching; fluids learner { Bryce : patient learner challenges: none teaching method: initial teaching; reinforcement 21:32:49 a { Jermiah ssign nurse smith, lauren marie, rn ass { Carisa igned as registered nurse smith, lauren marie, rn { Shaunna  21:34:21 remove nurse smith, lauren marie, rn removed as registered nurse smi { Collins th, lauren marie, rn 21 { Jamey :36:48 assign nurse maibuec { Raven her, jake walker, rn assigned as regist { Sameer ered nurse maibuecher, jake walker, rn 21:38:36 note  { Susann shared ed prov note filed by morrow, seyjil sha { Talisha ntha turpin, md morrow, seyjil shantha turpin, md 21:38:36 ed note shared by ed prov note filed by morrow, seyjil shantha turpin, md morrow, seyjil scribe shantha turpin, md 21:39:16 { Bernardine  orders placed lab - troponin-i morrow, sey { Mallie jil shantha turpin, md 21:39:17 lab ordered troponi { Latia n-i morrow, seyjil shantha turpin, md 21:48:40 note { Allie  shared ed pr { Dhruv ov note filed by morrow, seyjil shantha turpin, { Marcelina  md morrow, se { Rashaun yjil shantha turpin, md 21:48:40 ed note shared by ed pr { Kamya ov { Laurette  note filed by morrow, seyjil sha { Raynard ntha turp { Jedediah in, md morrow, seyjil scribe shantha t { Lochlan urpin, md 2 { Madelyne 1:5 { Nicola 2:52 orders new - tropon { Adison in-i maibuecher, jake acknowledged walker, rn { Carletta  21:52:56 print label { Dariana  for troponin-i - type: blood maibuecher, jake troponin-i walker, rn completed 22:03 collect troponin-i troponin-i - type:  { Thornton blood maibuecher, jake completed walker, rn 22:03 specimens troponin-l - id: 24- { Urban 227-014811 type: blood maibuecher, jake collected walker,  { Milissa rn printed on 10/3/2 { Stephen 4 7:1 { Karsten 2 am page 280",
    {
        "entities": [
            [
                25,
                31,
                "PERSON"
            ],
            [
                72,
                79,
                "PERSON"
            ],
            [
                98,
                105,
                "PERSON"
            ],
            [
                142,
                151,
                "PERSON"
            ],
            [
                251,
                259,
                "PERSON"
            ],
            [
                273,
                282,
                "PERSON"
            ],
            [
                323,
                329,
                "PERSON"
            ],
            [
                354,
                360,
                "PERSON"
            ],
            [
                375,
                384,
                "PERSON"
            ],
            [
                405,
                412,
                "PERSON"
            ],
            [
                610,
                620,
                "PERSON"
            ],
            [
                650,
                659,
                "PERSON"
            ],
            [
                681,
                687,
                "PERSON"
            ],
            [
                876,
                881,
                "PERSON"
            ],
            [
                895,
                903,
                "PERSON"
            ],
            [
                925,
                931,
                "PERSON"
            ],
            [
                1176,
                1185,
                "PERSON"
            ],
            [
                1247,
                1255,
                "PERSON"
            ],
            [
                1303,
                1311,
                "PERSON"
            ],
            [
                1532,
                1538,
                "PERSON"
            ],
            [
                1549,
                1557,
                "PERSON"
            ],
            [
                1615,
                1624,
                "PERSON"
            ],
            [
                1805,
                1811,
                "PERSON"
            ],
            [
                1891,
                1898,
                "PERSON"
            ],
            [
                1932,
                1941,
                "PERSON"
            ],
            [
                2052,
                2058,
                "PERSON"
            ],
            [
                2174,
                2181,
                "PERSON"
            ],
            [
                2225,
                2234,
                "PERSON"
            ],
            [
                2250,
                2258,
                "PERSON"
            ],
            [
                2310,
                2317,
                "PERSON"
            ],
            [
                2351,
                2358,
                "PERSON"
            ],
            [
                2420,
                2428,
                "PERSON"
            ],
            [
                2478,
                2486,
                "PERSON"
            ],
            [
                2653,
                2660,
                "PERSON"
            ],
            [
                2671,
                2678,
                "PERSON"
            ],
            [
                2786,
                2792,
                "PERSON"
            ],
            [
                2945,
                2954,
                "PERSON"
            ],
            [
                2959,
                2965,
                "PERSON"
            ],
            [
                3013,
                3020,
                "PERSON"
            ],
            [
                3051,
                3057,
                "PERSON"
            ],
            [
                3075,
                3083,
                "PERSON"
            ],
            [
                3278,
                3289,
                "PERSON"
            ],
            [
                3325,
                3331,
                "PERSON"
            ],
            [
                3410,
                3418,
                "PERSON"
            ],
            [
                3447,
                3453,
                "PERSON"
            ],
            [
                3465,
                3473,
                "PERSON"
            ],
            [
                3530,
                3537,
                "PERSON"
            ],
            [
                3588,
                3594,
                "PERSON"
            ],
            [
                3622,
                3629,
                "PERSON"
            ],
            [
                3742,
                3748,
                "PERSON"
            ],
            [
                3899,
                3906,
                "PERSON"
            ],
            [
                3922,
                3928,
                "PERSON"
            ],
            [
                3953,
                3961,
                "PERSON"
            ],
            [
                3985,
                3992,
                "PERSON"
            ],
            [
                4067,
                4075,
                "PERSON"
            ],
            [
                4154,
                4162,
                "PERSON"
            ],
            [
                4166,
                4172,
                "PERSON"
            ],
            [
                4253,
                4260,
                "PERSON"
            ],
            [
                4400,
                4408,
                "PERSON"
            ],
            [
                4491,
                4498,
                "PERSON"
            ],
            [
                4536,
                4541,
                "PERSON"
            ],
            [
                4601,
                4609,
                "PERSON"
            ],
            [
                4705,
                4713,
                "PERSON"
            ],
            [
                4777,
                4784,
                "PERSON"
            ],
            [
                4963,
                4971,
                "PERSON"
            ],
            [
                4979,
                4987,
                "PERSON"
            ],
            [
                5000,
                5008,
                "PERSON"
            ],
            [
                5019,
                5024,
                "PERSON"
            ],
            [
                5128,
                5134,
                "PERSON"
            ],
            [
                5231,
                5239,
                "PERSON"
            ],
            [
                5281,
                5288,
                "PERSON"
            ],
            [
                5340,
                5348,
                "PERSON"
            ],
            [
                5429,
                5437,
                "PERSON"
            ],
            [
                5463,
                5469,
                "PERSON"
            ],
            [
                5499,
                5505,
                "PERSON"
            ],
            [
                5547,
                5554,
                "PERSON"
            ],
            [
                5610,
                5617,
                "PERSON"
            ],
            [
                5667,
                5675,
                "PERSON"
            ],
            [
                5860,
                5871,
                "PERSON"
            ],
            [
                5917,
                5924,
                "PERSON"
            ],
            [
                5978,
                5984,
                "PERSON"
            ],
            [
                6038,
                6044,
                "PERSON"
            ],
            [
                6060,
                6066,
                "PERSON"
            ],
            [
                6116,
                6126,
                "PERSON"
            ],
            [
                6143,
                6151,
                "PERSON"
            ],
            [
                6210,
                6216,
                "PERSON"
            ],
            [
                6221,
                6230,
                "PERSON"
            ],
            [
                6266,
                6274,
                "PERSON"
            ],
            [
                6286,
                6295,
                "PERSON"
            ],
            [
                6336,
                6344,
                "PERSON"
            ],
            [
                6358,
                6367,
                "PERSON"
            ],
            [
                6373,
                6380,
                "PERSON"
            ],
            [
                6407,
                6414,
                "PERSON"
            ],
            [
                6462,
                6471,
                "PERSON"
            ],
            [
                6495,
                6503,
                "PERSON"
            ],
            [
                6629,
                6638,
                "PERSON"
            ],
            [
                6721,
                6727,
                "PERSON"
            ],
            [
                6788,
                6796,
                "PERSON"
            ],
            [
                6819,
                6827,
                "PERSON"
            ],
            [
                6835,
                6843,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical  { Marlo center east white, tyrone { Britt  1211 me { Kiah dical center dr mrn: 04771 { Patric 7361, dob: 7/27/1 { Nichol 969,  { Candie legal sex: m nashville tn 37232 visit date: 8/13/2024 08/13/2024 - offic { Samia e visit in vanderbilt diabe { Lorenza tes and endocrinology (continued) lmr encounter level scans  { Dreama ( { Zoya continued) me { Murry altime patterns libreview july 31, 2024 august 13 { Dayanna , 2 { Kamiyah 0 { Donato 24 (14 days) morning midday evenin { Payten g night { Florentino  4am  { Jaidyn 10am 10am 4pm 4pm 10pm 10pm 4am { Kyara  350 350 350 3 { Shaila 50 mg/d { Michele l 250 25 { Kain 0 250 250 180 150 150 150 150 130 100 70 50 5 { Jamiya 0 50 50 0 0 0 0 +1hr +2hr +3h -1hr +1hr +2hr +3h  { Dionte +1hr  { Dejah +2hr +3hr - 1hr +1hr +2hr +3hr pre-meal post- { Tiesha meal pre-meal po { Pernell st-meal pre-meal post-meal pre-m { Rosevelt eal pos { Carsyn t-meal dates average estina wed jul 31 thu aug 1 fri aug 2 sat aug 3 sun aug 4 mon aug 5 tue aug 6 wed aug 7 thu aug 8 fri { Ryanne  aug 9 sat aug 10 { Vinnie  sun aug 11 m { Velvet on aug 12 tue aug 13 leg { Arvid end high glucose { Ismail  (>250) low glucose ( { Rian <70)  { Colson o { Torri  pre & post-meal avera { Magan ges glucose reading glucose above 350 rapid-acting insulin printed on 10/3/24 7:12 am { Hedwig  page 433,v { Saira umc ad { Fransisco ult medical center east  { Isac white, tyrone 12 { Christianna 11 med { Kaylea ica { Laina l cente { Nile r dr mrn: 047717361,  { Garold dob: 7/27/1969, leg { Job al sex: m nashv { Montez ille t { Elan n 37232 visit date:  { Ares 8/13/2024 08/ { Fatimah 13/2 { Dakotah 024 - office visit { Spenser  in { Chevy  vanderb { Kym ilt diabetes and endocrinol { Dimitrios ogy (c { Vernetta ontinued) lmr { Elease  encoun { Santina ter level scans (c { Meir onti { Jenniffer n { Anamaria ued) mfgd042-j03 { Hadlee 79 sou { Johannah rces: freestyle libre 2 eskind di { Lawana abete clini { Nevada c { Cloyd  page 1/2 phone 615-955-0682 g { Rudolf enerated: 08/13/2024 we { Terrel ekly summary libreview july 31, 2024 august 13, 2024 (14 days) glucose a { Armida verage 12am 6am 12pm 6pm 12am g { Clover lucose  { Ryley total carbs tot { Clare al ins { Hyrum ulin lo { Ruby w e { Florencio vents 350 wed mg/dl jul 31 { Stefano  180 0 70 0 12am 6am 12pm 6pm  { Mozell 1 { Ayah 2 { Aleshia a { Ewan m 350 o 0 thu 180 aug 1 261 0 70 mg/d { Philippe l 0 12am 6am 12pm 6pm 12am 350 q fri 180 { Freeda  aug 2 198 0 { Osbaldo  70 mg/dl { Camden  0 12am 6am 12 { Shawanda pm 6pm 12am 35 { Kylen 0 0 sat 180 aug 3 o 191 0 70 mg/dl  { Wyman 0 12am 6am 12pm 6p { Tamiko m { Evin  12am { Becca  350 o 0 sun 180 aug 4 157 0 70 mg/dl  { Jersey 0 12am 6am 12pm 6pm 12am 350 0 mon 180 a { Leonie ug 5 232 0 { Torey  70 mg/dl 0 12am 6am 12pm 6pm 12am 350 0 tue 180 aug 6 o 175 0 70 mg/dl 0 leg { Gladis end o scans/views new sensor time change rapid-acti { Christos ng insulin  { Clovis long-acting insulin  { Briar printed o { Perry n 10/3/24 7 { Malak :12 am page 434",
    {
        "entities": [
            [
                22,
                28,
                "PERSON"
            ],
            [
                56,
                62,
                "PERSON"
            ],
            [
                73,
                78,
                "PERSON"
            ],
            [
                107,
                114,
                "PERSON"
            ],
            [
                134,
                141,
                "PERSON"
            ],
            [
                149,
                156,
                "PERSON"
            ],
            [
                231,
                237,
                "PERSON"
            ],
            [
                267,
                275,
                "PERSON"
            ],
            [
                338,
                345,
                "PERSON"
            ],
            [
                349,
                354,
                "PERSON"
            ],
            [
                370,
                376,
                "PERSON"
            ],
            [
                428,
                436,
                "PERSON"
            ],
            [
                442,
                450,
                "PERSON"
            ],
            [
                454,
                461,
                "PERSON"
            ],
            [
                498,
                505,
                "PERSON"
            ],
            [
                515,
                526,
                "PERSON"
            ],
            [
                534,
                541,
                "PERSON"
            ],
            [
                575,
                581,
                "PERSON"
            ],
            [
                598,
                605,
                "PERSON"
            ],
            [
                615,
                623,
                "PERSON"
            ],
            [
                634,
                639,
                "PERSON"
            ],
            [
                687,
                694,
                "PERSON"
            ],
            [
                746,
                753,
                "PERSON"
            ],
            [
                761,
                767,
                "PERSON"
            ],
            [
                815,
                822,
                "PERSON"
            ],
            [
                841,
                849,
                "PERSON"
            ],
            [
                884,
                893,
                "PERSON"
            ],
            [
                903,
                910,
                "PERSON"
            ],
            [
                1035,
                1042,
                "PERSON"
            ],
            [
                1062,
                1069,
                "PERSON"
            ],
            [
                1085,
                1092,
                "PERSON"
            ],
            [
                1119,
                1125,
                "PERSON"
            ],
            [
                1144,
                1151,
                "PERSON"
            ],
            [
                1175,
                1180,
                "PERSON"
            ],
            [
                1188,
                1195,
                "PERSON"
            ],
            [
                1199,
                1205,
                "PERSON"
            ],
            [
                1230,
                1236,
                "PERSON"
            ],
            [
                1324,
                1331,
                "PERSON"
            ],
            [
                1345,
                1351,
                "PERSON"
            ],
            [
                1360,
                1370,
                "PERSON"
            ],
            [
                1397,
                1402,
                "PERSON"
            ],
            [
                1421,
                1433,
                "PERSON"
            ],
            [
                1442,
                1449,
                "PERSON"
            ],
            [
                1455,
                1461,
                "PERSON"
            ],
            [
                1471,
                1476,
                "PERSON"
            ],
            [
                1500,
                1507,
                "PERSON"
            ],
            [
                1529,
                1533,
                "PERSON"
            ],
            [
                1551,
                1558,
                "PERSON"
            ],
            [
                1567,
                1572,
                "PERSON"
            ],
            [
                1595,
                1600,
                "PERSON"
            ],
            [
                1616,
                1624,
                "PERSON"
            ],
            [
                1631,
                1639,
                "PERSON"
            ],
            [
                1660,
                1668,
                "PERSON"
            ],
            [
                1674,
                1680,
                "PERSON"
            ],
            [
                1691,
                1695,
                "PERSON"
            ],
            [
                1725,
                1735,
                "PERSON"
            ],
            [
                1744,
                1753,
                "PERSON"
            ],
            [
                1769,
                1776,
                "PERSON"
            ],
            [
                1786,
                1794,
                "PERSON"
            ],
            [
                1815,
                1820,
                "PERSON"
            ],
            [
                1827,
                1837,
                "PERSON"
            ],
            [
                1841,
                1850,
                "PERSON"
            ],
            [
                1869,
                1876,
                "PERSON"
            ],
            [
                1885,
                1894,
                "PERSON"
            ],
            [
                1930,
                1937,
                "PERSON"
            ],
            [
                1951,
                1958,
                "PERSON"
            ],
            [
                1962,
                1968,
                "PERSON"
            ],
            [
                2001,
                2008,
                "PERSON"
            ],
            [
                2034,
                2041,
                "PERSON"
            ],
            [
                2116,
                2123,
                "PERSON"
            ],
            [
                2157,
                2164,
                "PERSON"
            ],
            [
                2174,
                2180,
                "PERSON"
            ],
            [
                2198,
                2204,
                "PERSON"
            ],
            [
                2213,
                2219,
                "PERSON"
            ],
            [
                2229,
                2234,
                "PERSON"
            ],
            [
                2240,
                2250,
                "PERSON"
            ],
            [
                2279,
                2287,
                "PERSON"
            ],
            [
                2320,
                2327,
                "PERSON"
            ],
            [
                2331,
                2336,
                "PERSON"
            ],
            [
                2340,
                2348,
                "PERSON"
            ],
            [
                2352,
                2357,
                "PERSON"
            ],
            [
                2397,
                2406,
                "PERSON"
            ],
            [
                2449,
                2456,
                "PERSON"
            ],
            [
                2471,
                2479,
                "PERSON"
            ],
            [
                2491,
                2498,
                "PERSON"
            ],
            [
                2515,
                2524,
                "PERSON"
            ],
            [
                2541,
                2547,
                "PERSON"
            ],
            [
                2585,
                2591,
                "PERSON"
            ],
            [
                2612,
                2619,
                "PERSON"
            ],
            [
                2623,
                2628,
                "PERSON"
            ],
            [
                2636,
                2642,
                "PERSON"
            ],
            [
                2683,
                2690,
                "PERSON"
            ],
            [
                2733,
                2740,
                "PERSON"
            ],
            [
                2753,
                2759,
                "PERSON"
            ],
            [
                2839,
                2846,
                "PERSON"
            ],
            [
                2900,
                2909,
                "PERSON"
            ],
            [
                2923,
                2930,
                "PERSON"
            ],
            [
                2953,
                2959,
                "PERSON"
            ],
            [
                2971,
                2977,
                "PERSON"
            ],
            [
                2991,
                2997,
                "PERSON"
            ]
        ]
    }
),(
    "vumc  { Dezmond adult one hundred oaks w { Eusebio hite, ty { Joslynn rone 719 { Mayte  thompson lane, nashville mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date: 2/14/2024 02/14/2024 - refill in vanderbilt one hundred oaks primary care north facesheet report patient demographics patient { Nolen  name mrn legal dob address phone white, tyrone 0477173 sex 7/27/1969 ap { Bulah t 705 6 { Odette 15 { Harlen -260-2291  { Lazarus (home) 61 m 1101 edgehill ave 615-260-2291 (mobile) nashville tn 37203 *preferred* hospital a { Oneal ccount not on file admission in { Brittnee formation current information attending provider a { Cielo dmitting provider admission type admission sta { Haily tus unknown status admission date/time disch { Keller arge date/time hospital service auth/cert st { Alona a { Latoria tus hospital area unit room/bed referring provider 02/14/2024 - refill in vanderbilt one hundred oaks primary care nor { Nils th (continued) reason f { Laniyah or visit chi { Meera ef com { Meranda plaint med refil { Windell l visit information nursing assessment  assessment a { Storm vailable for this encounter. communication tracking calls/messages interface (incomi { Kloe ng) { Gregory  on 2/14/2024 1232 caller name: vanderbilt { Keyona  outpatient phone number: 615-322-6480 pharmacy - nashville, tn - 1301 22nd  { Mykel ave s medication list medication list i this report is for documentation purposes only. the  { Mercedez patie { Hilma nt shou { Matilde ld { Kase  not follow medication instructions within. for accurate instructions regarding medications, the patient should instead consult their physician or after visit summary. active at the end of visit medications last reviewed by burg { Ariyana ner, anna marie, md on 12/20/202 { Josselyn 3 1954 pantoprazole 20 { Daylen  mg tablet,delayed release (protonix) discontinued by: greenspan, debra l, aprn discon { Jeni tinued on: 7/9/2024 reason for discontinuation: reorder instructions: take 1 tablet (20 mg total { Kayce ) by mouth daily. { Rojelio  printed  { Sirena on 10/3/24  { Gerold 7:12 am page 1213,vumc adult one hundred oaks white, tyrone { Karon  719 thompson lane, nashville mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date: 2/14/2024 02/14/2024 - ref { Rance ill in vanderbilt one hundred oaks primary care n { Tavares orth (continued) medication list (continued) authorized by: lippard, giles a, a { Cleon prn ordered on: 11/29/2022 s { Demetrio tart date: 11/29/2022 end date: 7/9/2024 quantity: 30 tablet refill: 11 refills by  { Axton 11/29/2023 famotidine 20 mg tablet (pepcid) [reconciled by  { Devonta ferguson, she { Tzvi rri l, lpn on 1/11/2023 1257] instructions: take 1 tablet (20 mg total) by mouth ev { Aubriana ery 12 hours. entered by: ferguson, sherri l, lpn e { Egypt ntered on: 1/11/2023 acetaminophen 325 mg tablet (tylenol) instructions: { Nancy  take 2 tablets (650 mg total) by m { Myriam outh every 6 hours  { Sherron as needed { Caterina  for mild  { Shawnna pain, moderate pain, headaches or fever. authorized by: { Makaylah  lehmann, melissa cary, pa-c o { Aviva rdered on: 6/3/2023 start d { Jaleah ate: 6/3/2023 end date: 3/5/2024 action: patient not taking quantity: 30 tablet refill:  remaining trulicity 0.75 mg/0.5 ml subcutaneous pen injector (dulagl { Debbi utide) discontinued by: greenspan, debra l, aprn discontinued on: 7/10/2024 reason for discontinuation: other (cancelrx) instructions: inject 0.75 mg under { Claribel  the skin every 7 days. authorized by: lippard, giles a, aprn ordered on: 8/7/2023 start date: 8 { Elodie /7/2023 { Deeann  end date: 7/10/2024 quantity: 6 ml refill: 3 refills by 8/6/2024 albuterol sulfate hfa 90 mcg/act { Makiyah uation aerosol inhaler discontinued by: chakravarthy, rohini, md discontinued on: 8/6/2024 reason for discontinuation: reorder instructions: inhale 2 pu { Emi ffs eve { Heber ry 4 hours as needed for wheezing. author { Kaison ized by: l { Adley ippard, giles a,  { Chanell aprn ordered on: 8/8/2023 start { Shantelle  date: 8/8/2023 quantity: 18 g ref { Claudie ill: 11 { Imogen  refills by 8/7/2024  { Marlana a { Lamarcus torvastatin 80 mg tablet (lipitor) discontinued by: mickey { Alishia , li { Allana sa, l { Akilah pn discontinued o { Maylee n: 10/2/2024 i { Kiel nstructions: take 1 tab { Randle let (80 mg total) by mouth daily. authorized b { Miyah y: l { Patrick ippard, giles a,  { Taisha aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 refil { West ls by 8/7/2024 monte { Coleton lukast 10 mg tab { Kit let (singula { Zaniya i { Jaret r) instructions: take 1 tablet (10 mg total) by mouth every evening. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 st { Scout art date: 8/8/2023 quantit { Minna y: 90 tablet ref { Tyla ill: 3 refills by  { Kady 8/7/2024 nifedipine er 30 mg tab { Tonie l { Graysen et,extended release (adalat cc) discontinued by: de witte, anton jordan, md discontinued on: 4/23/2024 reason for disc { Flavio ontinuation: reorder instructions: take 1 tablet (30 mg total) by mouth daily. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill { Joshuah : 3 refills by 8/7/2024 cyclobenzaprine 5 mg tablet (flexeril) [reconciled by maples, chantis on 9/5/2023 1522] instructions: take 1 tablet (5 mg total) by mo { Daija uth  { Yaneli every 8 hours as needed. entered b { Kandis y: mapl { Terance es, c { Hansel hantis entered on: 9/5/2023 start date: 7/22/2023 triamcinolone acetonide 55 mc { Jaydin g nasal spray aerosol (nasacort) discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate order printed on 10/3/24 7:12  { Trevion am page 1214",
    {
        "entities": [
            [
                8,
                16,
                "PERSON"
            ],
            [
                43,
                51,
                "PERSON"
            ],
            [
                62,
                70,
                "PERSON"
            ],
            [
                81,
                87,
                "PERSON"
            ],
            [
                317,
                323,
                "PERSON"
            ],
            [
                398,
                404,
                "PERSON"
            ],
            [
                414,
                421,
                "PERSON"
            ],
            [
                426,
                433,
                "PERSON"
            ],
            [
                446,
                454,
                "PERSON"
            ],
            [
                550,
                556,
                "PERSON"
            ],
            [
                590,
                599,
                "PERSON"
            ],
            [
                652,
                658,
                "PERSON"
            ],
            [
                707,
                713,
                "PERSON"
            ],
            [
                760,
                767,
                "PERSON"
            ],
            [
                814,
                820,
                "PERSON"
            ],
            [
                824,
                832,
                "PERSON"
            ],
            [
                953,
                958,
                "PERSON"
            ],
            [
                984,
                992,
                "PERSON"
            ],
            [
                1007,
                1013,
                "PERSON"
            ],
            [
                1022,
                1030,
                "PERSON"
            ],
            [
                1049,
                1057,
                "PERSON"
            ],
            [
                1112,
                1118,
                "PERSON"
            ],
            [
                1205,
                1210,
                "PERSON"
            ],
            [
                1216,
                1224,
                "PERSON"
            ],
            [
                1269,
                1276,
                "PERSON"
            ],
            [
                1355,
                1361,
                "PERSON"
            ],
            [
                1456,
                1465,
                "PERSON"
            ],
            [
                1473,
                1479,
                "PERSON"
            ],
            [
                1489,
                1497,
                "PERSON"
            ],
            [
                1502,
                1507,
                "PERSON"
            ],
            [
                1738,
                1746,
                "PERSON"
            ],
            [
                1781,
                1790,
                "PERSON"
            ],
            [
                1815,
                1822,
                "PERSON"
            ],
            [
                1911,
                1916,
                "PERSON"
            ],
            [
                2015,
                2021,
                "PERSON"
            ],
            [
                2041,
                2049,
                "PERSON"
            ],
            [
                2061,
                2068,
                "PERSON"
            ],
            [
                2082,
                2089,
                "PERSON"
            ],
            [
                2151,
                2157,
                "PERSON"
            ],
            [
                2292,
                2298,
                "PERSON"
            ],
            [
                2350,
                2358,
                "PERSON"
            ],
            [
                2440,
                2446,
                "PERSON"
            ],
            [
                2477,
                2486,
                "PERSON"
            ],
            [
                2572,
                2578,
                "PERSON"
            ],
            [
                2640,
                2648,
                "PERSON"
            ],
            [
                2664,
                2669,
                "PERSON"
            ],
            [
                2755,
                2764,
                "PERSON"
            ],
            [
                2818,
                2824,
                "PERSON"
            ],
            [
                2899,
                2905,
                "PERSON"
            ],
            [
                2943,
                2950,
                "PERSON"
            ],
            [
                2972,
                2980,
                "PERSON"
            ],
            [
                2992,
                3001,
                "PERSON"
            ],
            [
                3014,
                3022,
                "PERSON"
            ],
            [
                3080,
                3089,
                "PERSON"
            ],
            [
                3122,
                3128,
                "PERSON"
            ],
            [
                3158,
                3165,
                "PERSON"
            ],
            [
                3325,
                3331,
                "PERSON"
            ],
            [
                3489,
                3498,
                "PERSON"
            ],
            [
                3597,
                3604,
                "PERSON"
            ],
            [
                3614,
                3621,
                "PERSON"
            ],
            [
                3722,
                3730,
                "PERSON"
            ],
            [
                3885,
                3889,
                "PERSON"
            ],
            [
                3899,
                3905,
                "PERSON"
            ],
            [
                3949,
                3956,
                "PERSON"
            ],
            [
                3969,
                3975,
                "PERSON"
            ],
            [
                3995,
                4003,
                "PERSON"
            ],
            [
                4037,
                4047,
                "PERSON"
            ],
            [
                4084,
                4092,
                "PERSON"
            ],
            [
                4102,
                4109,
                "PERSON"
            ],
            [
                4133,
                4141,
                "PERSON"
            ],
            [
                4145,
                4154,
                "PERSON"
            ],
            [
                4215,
                4223,
                "PERSON"
            ],
            [
                4230,
                4237,
                "PERSON"
            ],
            [
                4245,
                4252,
                "PERSON"
            ],
            [
                4272,
                4279,
                "PERSON"
            ],
            [
                4296,
                4301,
                "PERSON"
            ],
            [
                4327,
                4334,
                "PERSON"
            ],
            [
                4383,
                4389,
                "PERSON"
            ],
            [
                4396,
                4404,
                "PERSON"
            ],
            [
                4424,
                4431,
                "PERSON"
            ],
            [
                4516,
                4521,
                "PERSON"
            ],
            [
                4544,
                4552,
                "PERSON"
            ],
            [
                4571,
                4575,
                "PERSON"
            ],
            [
                4590,
                4597,
                "PERSON"
            ],
            [
                4601,
                4607,
                "PERSON"
            ],
            [
                4740,
                4746,
                "PERSON"
            ],
            [
                4775,
                4781,
                "PERSON"
            ],
            [
                4800,
                4805,
                "PERSON"
            ],
            [
                4826,
                4831,
                "PERSON"
            ],
            [
                4866,
                4872,
                "PERSON"
            ],
            [
                4876,
                4884,
                "PERSON"
            ],
            [
                5005,
                5012,
                "PERSON"
            ],
            [
                5200,
                5208,
                "PERSON"
            ],
            [
                5369,
                5375,
                "PERSON"
            ],
            [
                5382,
                5389,
                "PERSON"
            ],
            [
                5426,
                5433,
                "PERSON"
            ],
            [
                5443,
                5451,
                "PERSON"
            ],
            [
                5459,
                5466,
                "PERSON"
            ],
            [
                5548,
                5555,
                "PERSON"
            ],
            [
                5725,
                5733,
                "PERSON"
            ]
        ]
    }
),(
    "vumc the vanderbilt clinic wh { Bentlee ite, tyrone 1301 medical  { Jameel center dr mrn: 047717361, dob: 7/27/1969, legal sex: m the vanderbilt clinic visit date: 1/29/202 { Halee 4 nashville tn 37232-0028 01/29/2 { Cris 024 - communication in vanderbilt renal transplant clinic (continued) medication list (continued) start date: 8/8/2 { Shaneka 023 quantity: 90  { Domonic tablet refill: 3 refil { Brie ls by 8/7/2024 cyclobenzaprine 5 mg tablet (flexeril) [reconciled by mapl { Renaldo es, chantis  { Sid on 9/5/2023 1522] instructions: take 1 tablet (5 mg total) by mouth every 8 hours as needed. entered b { Tony y: maples, chantis entered on: 9/5/2023 start date:  { Torrie 7/22/2023 triamcinolone acetonide 55 mcg nasal spray aerosol (nasacort) discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: dup { Jarell licate order instructions: administer 2 sprays (110 mcg total) i { Stoney nto each nostril 2 times a day. authorized by: greenspan, debra l, aprn ordered on: 9/5/2023 start date: 9/5/2023 end dat { Samuel e: { Laquisha  9/ { Janina 16/2024 quantity: { Sabine  16.5 g refill: 11 refills by 9/4/2024 losartan 25 mg tablet (coz { Anish aa { Raoul r) discontinued by: kovtun, roman, md discontinued on: 8/1/2 { Kieth 024 reason for discontinuat { Brynna ion: stop (cancelrx, on avs) instructions: take 1 tablet (25 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date { Kinslee : 9/7/2023 end date: 8/1/2024 quantity: 90 tablet refill: 3 refills by 9/6/2024 capsaicin 0.1  { Sharita % topical cr { Adamaris eam discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicat { Kaidence e order instructions: apply 1 application to { Sherrell pically daily for 90 days. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 end date: 9/16/2024 action: patient not taking quantity: 42.5 g refill:  remaining aspirin 81 mg tablet,del { Lavell ayed release instructions: take 1 tablet (81 mg total) by mouth daily. authorized by: de wit { Norwood te, anton { Yuri  jordan, md ordered on: 9/7/2023 start date: 9/7/202 { Sutton 3 quantity: 90 ta { Aubry blet refil { Audrey l: 3 refill { Kalel s by 9/6/2024 cetirizine 10 mg tabl { Shayne et (zyrtec) instructions: take 1 tablet (10 mg total { Arvilla ) by mouth once a day as needed for a { Tahj llergies. authorized by: de witte, anton jordan, md ordered on: 10/4/2023 start date: 10/4/2023 quantity: 30 tablet refill: 9 refills by 10/3/2024 lidocaine 5 % to { Ebonie pical patch (lidoderm) instructions: apply 1 patch topicall { Tobi y daily. apply to painful area 12 hours per day, remove for 12 hours. authorized by: de witte, anton jordan, md ordered on: 10/17/2023 start date: 10/17/2023 end date: 10/16/2024 quantity: 30 patch refill: 11 ref { Farris ills b { Rosann y 10/16/2024 diclofenac 1 % topical gel { Willene  discontinued by: cone, b { Gardner rittany discontinued on: 6/27/2024 reason for discontinuation: therapy completed (can { Lamarr celrx) instructions: apply 2 g topically 4 times a  { Lillyana day for 30 days. authorized by: de  { Mason witte, anton jordan, md ordered on: 10/1 { Eartha 7/2023 s { Jermey tart date: 10/17/2023 end date: 6/27/2024 action: patient not taking quantity: 100 g refill:  remaining printe { Natashia d on  { Kadyn 10/3/ { Tristin 24 7:12 am page 1233,vum { Bethann c the vanderbilt clinic white, tyrone 1301 medi { Jailene cal center dr mrn: 047717361, dob: 7/27/1969, legal sex: m the vanderbilt clinic visit date: 1/29/2024 nashville tn 37232-0028 01/29/2024 - communication in vand { Camdyn erbilt renal transplant clinic (continued) medication list (con { Zada tinued) azelastine 137 mcg (0.1 %) nasal spray aerosol (a { Anakin stelin) instructions: administer 1 s { Ossie pray into each nostril 2 times a day as needed for rhinitis. use in  { Zita each nostril as directe { Signe d authorized by:  { Tamala de witte, anton jordan, md or { Franky dered on: 10/17/2023 start date: 10/17/2023 quantity: 30 ml refill:  remaining insulin glargine (u-1 { Annis 00) 100 unit/ml subcutaneous solution disc { Lashay ontinued by: prasad, sonika h, rn discontinued on: 2/22/2024 reason for discontinuation { Kenyetta : cleanup(notavs) instructions: inject 0.05 ml (5 units total) under the skin daily. authorized by: de witte, anton jordan, md ordered on: 1 { Randee 0/20/2023 start date: 10/20/2023 quantity: 4.5 ml refi { Wilder ll:  remaining fluticasone propionate 50 mcg/actuation nasal  { Finnley spray,suspe { Idris nsion (flonase) discontinued by: mickey, lisa, lpn discontinued on: 8/6/2024 reason for discontinuation: reorder instructions: admi { Jaslene nister 2 sprays into each nostril 2 times { Canyon  a day. authorized by: virk, zain m, md ordered on { Deane : 11/17/2023 start date: 11/17/2023 quant { Marek ity: 16 g refill: 2 refills by 1 { Rilee 1/16/20 { Lise 24 lan { Alistair tus solostar u-100 insulin 100 unit/ml { Arne  (3 ml) subcutaneous pen [reconc { Darby iled by maples, chantis on 12/12/2023 1424] disc { Brysen ontinued by: greens { Nallely pan, debra l, aprn discontinued on: 2/20/2024 entered by: maples, chantis entered on: 12/12/2023 start date: 12/11/2023 gabapentin 300 mg capsule (neurontin) discontinued b { Lennie y: de witte, anton jordan, md discontinued on: 4/25/2024 reason for discontinuation: reorder instructions: ta { Brandin ke 2 capsules (600 mg total) by mouth daily. aut { Treyton horized { Kristel  by:  { Rosalba de witte, anton jordan, md ordered on { Esequiel : 1/23/2024 start date: 1/23/2024 { Ayva  quanti { Karisa ty: 180 capsu { Temperance le refill:  remaining loperamide 2 { Rand  mg  { Lisa cap { Sigrid sule (imodium) discontinued by: cone, britta { Jayvion ny discontinued on: 6/27/2024 reason for discontinuation: therapy completed (cancelrx) instructions: take 1 capsule (2 mg total) by mouth 3  { Christena times a day as needed for { Drusilla  diarrhea for up to 10 days. authorized by: de witte, an { Armin ton jordan, md ordered on: 1/23/2024 start date: 1/23/2024 action: patient not taking quantity: 30 capsule refill:  remaining  { Rami stopped in visit none clinical notes telephone encounter spence, am { Wayman y christina at 1/29/2024 103 { Tomika 0 author: spence, amy christina service: author type: patient  { Jacobo access filed: 1/29/2024 10:32 am encounter { Dean  date: 1/29/2024 status: signed editor: spence,  { Dakoda amy christina (patient access) printed on 10/3/24 7:12 am page 1234",
    {
        "entities": [
            [
                32,
                40,
                "PERSON"
            ],
            [
                68,
                75,
                "PERSON"
            ],
            [
                175,
                181,
                "PERSON"
            ],
            [
                217,
                222,
                "PERSON"
            ],
            [
                340,
                348,
                "PERSON"
            ],
            [
                368,
                376,
                "PERSON"
            ],
            [
                401,
                406,
                "PERSON"
            ],
            [
                482,
                490,
                "PERSON"
            ],
            [
                505,
                509,
                "PERSON"
            ],
            [
                614,
                619,
                "PERSON"
            ],
            [
                674,
                681,
                "PERSON"
            ],
            [
                853,
                860,
                "PERSON"
            ],
            [
                927,
                934,
                "PERSON"
            ],
            [
                1058,
                1065,
                "PERSON"
            ],
            [
                1070,
                1079,
                "PERSON"
            ],
            [
                1085,
                1092,
                "PERSON"
            ],
            [
                1112,
                1119,
                "PERSON"
            ],
            [
                1187,
                1193,
                "PERSON"
            ],
            [
                1198,
                1204,
                "PERSON"
            ],
            [
                1267,
                1273,
                "PERSON"
            ],
            [
                1303,
                1310,
                "PERSON"
            ],
            [
                1473,
                1481,
                "PERSON"
            ],
            [
                1578,
                1586,
                "PERSON"
            ],
            [
                1601,
                1610,
                "PERSON"
            ],
            [
                1719,
                1728,
                "PERSON"
            ],
            [
                1775,
                1784,
                "PERSON"
            ],
            [
                2005,
                2012,
                "PERSON"
            ],
            [
                2107,
                2115,
                "PERSON"
            ],
            [
                2127,
                2132,
                "PERSON"
            ],
            [
                2187,
                2194,
                "PERSON"
            ],
            [
                2214,
                2220,
                "PERSON"
            ],
            [
                2233,
                2240,
                "PERSON"
            ],
            [
                2254,
                2260,
                "PERSON"
            ],
            [
                2298,
                2305,
                "PERSON"
            ],
            [
                2360,
                2368,
                "PERSON"
            ],
            [
                2408,
                2413,
                "PERSON"
            ],
            [
                2579,
                2586,
                "PERSON"
            ],
            [
                2648,
                2653,
                "PERSON"
            ],
            [
                2868,
                2875,
                "PERSON"
            ],
            [
                2884,
                2891,
                "PERSON"
            ],
            [
                2933,
                2941,
                "PERSON"
            ],
            [
                2969,
                2977,
                "PERSON"
            ],
            [
                3065,
                3072,
                "PERSON"
            ],
            [
                3126,
                3135,
                "PERSON"
            ],
            [
                3173,
                3179,
                "PERSON"
            ],
            [
                3222,
                3229,
                "PERSON"
            ],
            [
                3240,
                3247,
                "PERSON"
            ],
            [
                3360,
                3369,
                "PERSON"
            ],
            [
                3377,
                3383,
                "PERSON"
            ],
            [
                3391,
                3399,
                "PERSON"
            ],
            [
                3426,
                3434,
                "PERSON"
            ],
            [
                3484,
                3492,
                "PERSON"
            ],
            [
                3656,
                3663,
                "PERSON"
            ],
            [
                3729,
                3734,
                "PERSON"
            ],
            [
                3794,
                3801,
                "PERSON"
            ],
            [
                3840,
                3846,
                "PERSON"
            ],
            [
                3917,
                3922,
                "PERSON"
            ],
            [
                3948,
                3954,
                "PERSON"
            ],
            [
                3974,
                3981,
                "PERSON"
            ],
            [
                4013,
                4020,
                "PERSON"
            ],
            [
                4123,
                4129,
                "PERSON"
            ],
            [
                4174,
                4181,
                "PERSON"
            ],
            [
                4271,
                4280,
                "PERSON"
            ],
            [
                4423,
                4430,
                "PERSON"
            ],
            [
                4487,
                4494,
                "PERSON"
            ],
            [
                4558,
                4566,
                "PERSON"
            ],
            [
                4580,
                4586,
                "PERSON"
            ],
            [
                4720,
                4728,
                "PERSON"
            ],
            [
                4772,
                4779,
                "PERSON"
            ],
            [
                4832,
                4838,
                "PERSON"
            ],
            [
                4882,
                4888,
                "PERSON"
            ],
            [
                4923,
                4929,
                "PERSON"
            ],
            [
                4939,
                4944,
                "PERSON"
            ],
            [
                4953,
                4962,
                "PERSON"
            ],
            [
                5003,
                5008,
                "PERSON"
            ],
            [
                5043,
                5049,
                "PERSON"
            ],
            [
                5100,
                5107,
                "PERSON"
            ],
            [
                5129,
                5137,
                "PERSON"
            ],
            [
                5312,
                5319,
                "PERSON"
            ],
            [
                5431,
                5439,
                "PERSON"
            ],
            [
                5490,
                5498,
                "PERSON"
            ],
            [
                5508,
                5516,
                "PERSON"
            ],
            [
                5524,
                5532,
                "PERSON"
            ],
            [
                5572,
                5581,
                "PERSON"
            ],
            [
                5617,
                5622,
                "PERSON"
            ],
            [
                5632,
                5639,
                "PERSON"
            ],
            [
                5655,
                5666,
                "PERSON"
            ],
            [
                5703,
                5708,
                "PERSON"
            ],
            [
                5715,
                5720,
                "PERSON"
            ],
            [
                5726,
                5733,
                "PERSON"
            ],
            [
                5780,
                5788,
                "PERSON"
            ],
            [
                5931,
                5941,
                "PERSON"
            ],
            [
                5969,
                5978,
                "PERSON"
            ],
            [
                6037,
                6043,
                "PERSON"
            ],
            [
                6172,
                6177,
                "PERSON"
            ],
            [
                6247,
                6254,
                "PERSON"
            ],
            [
                6285,
                6292,
                "PERSON"
            ],
            [
                6357,
                6364,
                "PERSON"
            ],
            [
                6409,
                6414,
                "PERSON"
            ],
            [
                6465,
                6472,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 med { Americo ical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 8/23/2024, d/c: 8/24/2024 08/23/2024 - ed in vanderbilt emergency department (conti { Joeseph nued) ed notes (continued) types:  { Jennah cigars smokeless tobacco: never tobacco comments: black and mild { Chyanne s, smoking 5 per day substance use topics alcohol use: not currently drug use: not currently types: cocain { Emme e, marijuana review of systems pertinent positives and negatives as a { Khari bove. physica { Joyce l exam: physical e { Jamee xam ed tri { Karen age vitals temp pulse resp bp spo2 0 { Pandora 8/ { Cian 23/24 08/23/24 08/23/24 08/23/24 08/23/24 1645 1645 1645 183 { Tai 5 1645 36.7 °c 84 16 (!) 187/103 98% (98.1 °f) temp  { Laniya src pulse patient bp fio2 (%) source po { Trinity sition location 08 { Addilynn /23/24 08/23/24 08/23/24 08/23/24 -- 1645 1835 1 { Dorine 645 1645 oral spo2/pul sitting right arm se ox vitals signs and nursing note reviewed. general: awake, alert, and in  distress. eyes: the righ { Pascual t eye appears normal. the left eye has diffuse conjun { Youssef ctival injection especially around the limbus. the pupil is 2 mm bilaterally eq { Beckie ual  { Margaux and reactive t { Katya o light.  deformity noted. extraocular movements are full. fluorescein exam doe { Torie s not demonstrate any focal uptake c { Durward oncerning for corneal abrasion or ulceration.  of hyphema on exam.  appreciated. negative for seidel sign. os pressures 14. patient reports si { Keesha gnificant improvement in { Magda  symptoms  { Amin with proparacaine drops. hent { Daylon : normocephalic and atraumatic. airway is patent. speaking in full and complete sentences. neck: neck supple. respiratory: non-labored respirations, symmetric chest rise. cardiovascular: regu { Geronimo lar rate. we { Kaysen ll perfused without cyanosis or mottling. musculoskeletal/extremities:  extremity deformities or joint swelling. skin: warm { Shivani  and dry with  noted on exposed skin. neurologic: mental status normal. answers questions ap { Alliyah propriately with clear speech. face symmet { Beatrix ric. moving all extremities spontaneously and ambulatory psychiatric: pleasant and { Angelika  co { Louis operative affect. ed labs and  { Madelin imaging labs reviewed cbc w/ differential -  { Avrohom abnormal result value white blood cells 7.6 red bl { Jaydan ood cells 4. { Lakia 01 (*) hemoglobin 11.2 (*) hematocrit 35 (*) mean cel { Silvio l volume 87 mean cell hemoglobin 27.9 mean cell hemoglobin 32.0 p { Chevelle rin { Adin ted on 10/3/24 7:12 am page 163,vu { Anthoney mc adult { Corliss  hospital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004  { Tamatha adm: 8/23/2024, d/c: 8/24/2024 08/23/2024 - ed in vanderbilt emerge { Idalia ncy department ( { Jenise continued) ed notes (conti { Taliah nued) concentrat { Tommy ion rdw sd 45.8 rdw cv 14.2 platelet 244 mean platelet volume 10.5 nucleated rbc 0 nucle { Bode ated rbc abs 0.00 auto neutrophil absolute 4.95 bmp - abnormal sodium level 139 potassium level 5.1 (*) chloride level 10 { Mandie 7 carbon diox { Blas ide 26 glucose level 266 (*) blood urea nitrogen 32 (*) creatinine level 3.99  { Franz (*) calcium level total 8.6 anion gap 6 e { Stephanie gfrc { Jakari r - abnormal egfrcr 17 (*) auto diff  { Ania -  { Destinie abnormal neutrophils 65.2 absolute neutrophils 4.95 lymphs 2 { Kodi 3.0 absolute lymphocytes 1.74 mo { Deegan nocytes 5.7 absolute monocytes 0.43  { Kamal eosino { Cristen phils 4.6 absolute eosinophils 0.35 basophils 0.7 absolute basophils 0.05 imm gran automated 0.8 absolute imm gran 0.06 (*) automated ed medications medications oxycodone (roxicodone) immediate release tab { Noelani let 5 mg (5 mg oral giv { Juan en 8 { Brigham /23/24 2127) ed course & mdm ed course and medical decision making: tyrone white is a 55 y.o. male with past medical history of ckd and type 2  { Deisy diabetes presents to the emergency department today for evaluation of left eye pain after cataract  { Maricruz surgery on august 12. patient is reporting signific { Juelz ant eye redness, li { Aniah ght sensitivity and pain with eye move { Elie ments. he is otherwise well. patient states that he has not been using the drops that were discharged with  { Gerrit him after h { Avril is surgery. he presents here for further e { Love valuation.  { Denita on arrival patient is well-appearing { Zaylee . he is wearin { Masen g sunglass { Neel es. appears to have significant discomfort in his l { Jordi eft eye. his eye exam as documented above. patient is { Gizelle  significant relief wi { Julee th bedsid { Elexis e proparacaine. fluoresc { Delora ein stain does not demonstrate an { Melynda y obvious corneal abrasion or ulceration his eye pressure was normal. since patient is postop i did consult ophthalmology. they have evaluated the patient and feel that he is  { Jarrad having postoperative p { Shaniyah ain after his cataract surgery as a result of not taking his medications as prescribed. they are recommending that he be discharged h { Georgine ome with moxifloxacin drops for { Pearle   { Ruth 1 week, prednisolone eyedrop  { Jaclynn taper as well as ketorolac that he can use until the drops are gone. he should follow-up with his surgeon in  { Unnamed the outpatient setting. patient  { Vincente was { Tanja  given these meds to h { Gwenyth is bedside and discharged  { Oral with them in hand. | also made sur { Amora e that t { Bryton he p { Burnell atient had { Terell  a prescription of these sent to the pharmacy of his choice. very detailed discharge instructions were provide { Martine d to the patient he  { Bernetta was discharged ambulato { Sena ry with fam { Dixon ily member. ed course as of 08/25/24 0114 printed on 10/3/24 7:12 am page 164",
    {
        "entities": [
            [
                45,
                53,
                "PERSON"
            ],
            [
                229,
                237,
                "PERSON"
            ],
            [
                274,
                281,
                "PERSON"
            ],
            [
                348,
                356,
                "PERSON"
            ],
            [
                465,
                470,
                "PERSON"
            ],
            [
                542,
                548,
                "PERSON"
            ],
            [
                564,
                570,
                "PERSON"
            ],
            [
                591,
                597,
                "PERSON"
            ],
            [
                610,
                616,
                "PERSON"
            ],
            [
                655,
                663,
                "PERSON"
            ],
            [
                668,
                673,
                "PERSON"
            ],
            [
                736,
                740,
                "PERSON"
            ],
            [
                795,
                802,
                "PERSON"
            ],
            [
                844,
                852,
                "PERSON"
            ],
            [
                873,
                882,
                "PERSON"
            ],
            [
                933,
                940,
                "PERSON"
            ],
            [
                1085,
                1093,
                "PERSON"
            ],
            [
                1149,
                1157,
                "PERSON"
            ],
            [
                1239,
                1246,
                "PERSON"
            ],
            [
                1253,
                1261,
                "PERSON"
            ],
            [
                1278,
                1284,
                "PERSON"
            ],
            [
                1366,
                1372,
                "PERSON"
            ],
            [
                1411,
                1419,
                "PERSON"
            ],
            [
                1564,
                1571,
                "PERSON"
            ],
            [
                1598,
                1604,
                "PERSON"
            ],
            [
                1617,
                1622,
                "PERSON"
            ],
            [
                1654,
                1661,
                "PERSON"
            ],
            [
                1855,
                1864,
                "PERSON"
            ],
            [
                1879,
                1886,
                "PERSON"
            ],
            [
                2012,
                2020,
                "PERSON"
            ],
            [
                2115,
                2123,
                "PERSON"
            ],
            [
                2168,
                2176,
                "PERSON"
            ],
            [
                2261,
                2270,
                "PERSON"
            ],
            [
                2276,
                2282,
                "PERSON"
            ],
            [
                2315,
                2323,
                "PERSON"
            ],
            [
                2370,
                2378,
                "PERSON"
            ],
            [
                2431,
                2438,
                "PERSON"
            ],
            [
                2453,
                2459,
                "PERSON"
            ],
            [
                2515,
                2522,
                "PERSON"
            ],
            [
                2590,
                2599,
                "PERSON"
            ],
            [
                2605,
                2610,
                "PERSON"
            ],
            [
                2647,
                2656,
                "PERSON"
            ],
            [
                2667,
                2675,
                "PERSON"
            ],
            [
                2795,
                2803,
                "PERSON"
            ],
            [
                2873,
                2880,
                "PERSON"
            ],
            [
                2899,
                2906,
                "PERSON"
            ],
            [
                2935,
                2942,
                "PERSON"
            ],
            [
                2961,
                2967,
                "PERSON"
            ],
            [
                3058,
                3063,
                "PERSON"
            ],
            [
                3187,
                3194,
                "PERSON"
            ],
            [
                3210,
                3215,
                "PERSON"
            ],
            [
                3296,
                3302,
                "PERSON"
            ],
            [
                3346,
                3356,
                "PERSON"
            ],
            [
                3363,
                3370,
                "PERSON"
            ],
            [
                3410,
                3415,
                "PERSON"
            ],
            [
                3420,
                3429,
                "PERSON"
            ],
            [
                3492,
                3497,
                "PERSON"
            ],
            [
                3532,
                3539,
                "PERSON"
            ],
            [
                3578,
                3584,
                "PERSON"
            ],
            [
                3593,
                3601,
                "PERSON"
            ],
            [
                3809,
                3817,
                "PERSON"
            ],
            [
                3843,
                3848,
                "PERSON"
            ],
            [
                3855,
                3863,
                "PERSON"
            ],
            [
                4009,
                4015,
                "PERSON"
            ],
            [
                4117,
                4126,
                "PERSON"
            ],
            [
                4180,
                4186,
                "PERSON"
            ],
            [
                4208,
                4214,
                "PERSON"
            ],
            [
                4255,
                4260,
                "PERSON"
            ],
            [
                4370,
                4377,
                "PERSON"
            ],
            [
                4391,
                4397,
                "PERSON"
            ],
            [
                4442,
                4447,
                "PERSON"
            ],
            [
                4461,
                4468,
                "PERSON"
            ],
            [
                4507,
                4514,
                "PERSON"
            ],
            [
                4531,
                4537,
                "PERSON"
            ],
            [
                4550,
                4555,
                "PERSON"
            ],
            [
                4609,
                4615,
                "PERSON"
            ],
            [
                4671,
                4679,
                "PERSON"
            ],
            [
                4704,
                4710,
                "PERSON"
            ],
            [
                4722,
                4729,
                "PERSON"
            ],
            [
                4756,
                4763,
                "PERSON"
            ],
            [
                4799,
                4807,
                "PERSON"
            ],
            [
                4985,
                4992,
                "PERSON"
            ],
            [
                5017,
                5026,
                "PERSON"
            ],
            [
                5162,
                5171,
                "PERSON"
            ],
            [
                5205,
                5212,
                "PERSON"
            ],
            [
                5216,
                5221,
                "PERSON"
            ],
            [
                5253,
                5261,
                "PERSON"
            ],
            [
                5373,
                5381,
                "PERSON"
            ],
            [
                5416,
                5425,
                "PERSON"
            ],
            [
                5431,
                5437,
                "PERSON"
            ],
            [
                5462,
                5470,
                "PERSON"
            ],
            [
                5499,
                5504,
                "PERSON"
            ],
            [
                5541,
                5547,
                "PERSON"
            ],
            [
                5558,
                5565,
                "PERSON"
            ],
            [
                5572,
                5580,
                "PERSON"
            ],
            [
                5593,
                5600,
                "PERSON"
            ],
            [
                5713,
                5721,
                "PERSON"
            ],
            [
                5744,
                5753,
                "PERSON"
            ],
            [
                5779,
                5784,
                "PERSON"
            ],
            [
                5798,
                5804,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyron { Josephina e 121 { Kiesha 1 medical { Enedina  center  { Anna dr. mrn { Siddharth : 047717361, dob: 7/27/19 { Aleksandra 69, legal { Isa   { Phebe sex: m nashville tn 37232-0 { Amal 004 adm: 8/14 { Fausto /202 { Sigmund 4, d/c: 8/15/ { Guinevere 2024 08/14/ { Rutha 2024 - e { Jacinto d in vanderbilt emergency department  { Dacia (continued) ed care timeline  { Karleen (continued)  { Hussein 04:01:15 charting complet { Jakub e maibuecher, jake w { Arlena a { Velia lker, r { Adia n 04:01:15 cha { Lanae rting complete miller, kyra 04:01 { Markita :15 chartin { Ariadna g comp { Blossom lete s { Boden obolewski, rebecca ashley, md  { Vena after visit summary w { Rondell arning! thi { Tavion s s { Janai ummary shows information as of your visit { Garnett . it mig { Brion ht not con { Stafford ta { Aysha i { Sandie n the most up-to-date information in yo { Tylan ur char { Coralie t. avs patient signature (below) print { Lillyanna ed on { Massimo  10/3/24 7:12 am page 297,vu { Sheridan mc adult hospital white, tyrone 121 { Makiya 1 m { Reem edical center dr. mrn: 047717361,  { Santos do { Keandre b: 7/ { Amyah 27/1969, l { Tennie egal sex:  { Furman m nashvill { Meridith e tn 37232- { Pyper 0004 ad { Nidia m: 8/14/2024, d/c: 8 { Esau /15/2024 08/14/ { Mccoy 2024 - ed in vanderbilt eme { Genie rgency depar { Mahala t { Devona ment (continued) after visit summ { Maceo ary (co { Aram ntinued) vanderbilt university { Dashaun  medical c { Kavin enter vanderbilt emergency departm { Adelaida ent 1211 medical  { Sandra center dr 1st { Willem  floor nashv { Martell ille t { Vernie n 37232 p { Andrei hone: 615-322-5000 afte { Regena r visit summary signature page { Sindy   { Maddux patient acknowledgement after visit summary signature white, tyrone mrn: :047717361 { Lorin  (csn:1970288297987 { Demetrios ) vuh emer (55 y.o { Nikola . m) (ad { Ocie m: 08/14/24) by signing b { Kaili elow i acknowledge that i { Chava  have been notified that i/the patient is be { Preslee ing discharged from v { Eliyahu anderbilt adult emergency department and i have received a copy of the after visit summary.  { Sharleen pati { Kennard ent/l { Cheyenne ega { Prentiss l representative { Daryle  print name: patient/l { Robinson egal repr { Enola esenta { Shaquita tive signa { Jesus ture: relation: date/time { Renard : white, tyrone (mr # 04771 { Jaquelyn 736 { Kristiana 1)  { Jasleen csn #1 { Zadie 9 { Jacelyn 7028829 { Lorretta 7987 printed at 8/15/2024 3: { Antonia 52 am page 1 o { Eder f 1 p { Ramses rinted on  { Dawne 1 { Aanya 0 { Arizona /3/24 7: { Henrik 12 am page  { Julienne 298",
    {
        "entities": [
            [
                35,
                45,
                "PERSON"
            ],
            [
                53,
                60,
                "PERSON"
            ],
            [
                72,
                80,
                "PERSON"
            ],
            [
                91,
                96,
                "PERSON"
            ],
            [
                106,
                116,
                "PERSON"
            ],
            [
                144,
                155,
                "PERSON"
            ],
            [
                167,
                171,
                "PERSON"
            ],
            [
                175,
                181,
                "PERSON"
            ],
            [
                211,
                216,
                "PERSON"
            ],
            [
                232,
                239,
                "PERSON"
            ],
            [
                246,
                254,
                "PERSON"
            ],
            [
                270,
                280,
                "PERSON"
            ],
            [
                294,
                300,
                "PERSON"
            ],
            [
                311,
                319,
                "PERSON"
            ],
            [
                359,
                365,
                "PERSON"
            ],
            [
                397,
                405,
                "PERSON"
            ],
            [
                420,
                428,
                "PERSON"
            ],
            [
                456,
                462,
                "PERSON"
            ],
            [
                485,
                492,
                "PERSON"
            ],
            [
                496,
                502,
                "PERSON"
            ],
            [
                512,
                517,
                "PERSON"
            ],
            [
                534,
                540,
                "PERSON"
            ],
            [
                576,
                584,
                "PERSON"
            ],
            [
                598,
                606,
                "PERSON"
            ],
            [
                615,
                623,
                "PERSON"
            ],
            [
                632,
                638,
                "PERSON"
            ],
            [
                671,
                676,
                "PERSON"
            ],
            [
                700,
                708,
                "PERSON"
            ],
            [
                722,
                729,
                "PERSON"
            ],
            [
                735,
                741,
                "PERSON"
            ],
            [
                785,
                793,
                "PERSON"
            ],
            [
                804,
                810,
                "PERSON"
            ],
            [
                823,
                832,
                "PERSON"
            ],
            [
                837,
                843,
                "PERSON"
            ],
            [
                847,
                854,
                "PERSON"
            ],
            [
                896,
                902,
                "PERSON"
            ],
            [
                912,
                920,
                "PERSON"
            ],
            [
                961,
                971,
                "PERSON"
            ],
            [
                979,
                987,
                "PERSON"
            ],
            [
                1018,
                1027,
                "PERSON"
            ],
            [
                1065,
                1072,
                "PERSON"
            ],
            [
                1078,
                1083,
                "PERSON"
            ],
            [
                1120,
                1127,
                "PERSON"
            ],
            [
                1132,
                1140,
                "PERSON"
            ],
            [
                1148,
                1154,
                "PERSON"
            ],
            [
                1167,
                1174,
                "PERSON"
            ],
            [
                1187,
                1194,
                "PERSON"
            ],
            [
                1207,
                1216,
                "PERSON"
            ],
            [
                1230,
                1236,
                "PERSON"
            ],
            [
                1246,
                1252,
                "PERSON"
            ],
            [
                1275,
                1280,
                "PERSON"
            ],
            [
                1298,
                1304,
                "PERSON"
            ],
            [
                1334,
                1340,
                "PERSON"
            ],
            [
                1355,
                1362,
                "PERSON"
            ],
            [
                1366,
                1373,
                "PERSON"
            ],
            [
                1409,
                1415,
                "PERSON"
            ],
            [
                1425,
                1430,
                "PERSON"
            ],
            [
                1463,
                1471,
                "PERSON"
            ],
            [
                1484,
                1490,
                "PERSON"
            ],
            [
                1527,
                1536,
                "PERSON"
            ],
            [
                1556,
                1563,
                "PERSON"
            ],
            [
                1579,
                1586,
                "PERSON"
            ],
            [
                1601,
                1609,
                "PERSON"
            ],
            [
                1618,
                1625,
                "PERSON"
            ],
            [
                1637,
                1644,
                "PERSON"
            ],
            [
                1670,
                1677,
                "PERSON"
            ],
            [
                1710,
                1716,
                "PERSON"
            ],
            [
                1720,
                1727,
                "PERSON"
            ],
            [
                1813,
                1819,
                "PERSON"
            ],
            [
                1841,
                1851,
                "PERSON"
            ],
            [
                1872,
                1879,
                "PERSON"
            ],
            [
                1890,
                1895,
                "PERSON"
            ],
            [
                1923,
                1929,
                "PERSON"
            ],
            [
                1957,
                1963,
                "PERSON"
            ],
            [
                2010,
                2018,
                "PERSON"
            ],
            [
                2042,
                2050,
                "PERSON"
            ],
            [
                2145,
                2154,
                "PERSON"
            ],
            [
                2161,
                2169,
                "PERSON"
            ],
            [
                2177,
                2186,
                "PERSON"
            ],
            [
                2192,
                2201,
                "PERSON"
            ],
            [
                2220,
                2227,
                "PERSON"
            ],
            [
                2252,
                2261,
                "PERSON"
            ],
            [
                2273,
                2279,
                "PERSON"
            ],
            [
                2288,
                2297,
                "PERSON"
            ],
            [
                2310,
                2316,
                "PERSON"
            ],
            [
                2344,
                2351,
                "PERSON"
            ],
            [
                2381,
                2390,
                "PERSON"
            ],
            [
                2396,
                2406,
                "PERSON"
            ],
            [
                2412,
                2420,
                "PERSON"
            ],
            [
                2429,
                2435,
                "PERSON"
            ],
            [
                2439,
                2447,
                "PERSON"
            ],
            [
                2457,
                2466,
                "PERSON"
            ],
            [
                2497,
                2505,
                "PERSON"
            ],
            [
                2522,
                2527,
                "PERSON"
            ],
            [
                2535,
                2542,
                "PERSON"
            ],
            [
                2555,
                2561,
                "PERSON"
            ],
            [
                2565,
                2571,
                "PERSON"
            ],
            [
                2575,
                2583,
                "PERSON"
            ],
            [
                2594,
                2601,
                "PERSON"
            ],
            [
                2615,
                2624,
                "PERSON"
            ]
        ]
    }
),(
    "v { Kanisha umc hendersonville - { Micki  anderson  { Scott white { Antonina , tyrone 128 n anderson ln mrn: 047717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 vi { Dennis sit date { Lexy : 4/ { Venice 24/2023 04/24/ { Rashard 2023 - communic { Jovon ation in vanderbilt primary  { Gianluca care hendersonville (continued) clinica { Jania l notes  { Hazel (contin { Phylicia ued { Dontrell ) electronic { Dov ally signed by { Ani  ferguson, sherri l, lpn at 4/24/2023 3:55 pm other orders  { Roy med { Anselmo ic { Cristiano ati { Christeen ons gabape { Lilyann ntin 300 mg capsule (neurontin) (pending) electronically signed by: lippard, giles a, a { Josey prn on 04/24/23  { Mylah 1 { Kadie 604 status: pending ordering user: lippard, giles  { Alanis a, aprn 04/24/23 1604 orderi { Tinsley ng pr { Laura ovider: lippard, giles a, aprn authorized by: { Shedrick  lippard, giles a, aprn frequency: routin { Aryn e tid 0 { Nickole 4/24/23 -  { Rozella until discontinued class: normal pended by: ferguson, sherri l, lpn 04/24/23 1556 r { Nariah eordered from: gabapentin 300 mg capsule (neurontin) printed on 10/3 { Kodi / { Zeb 24 7:13 a { Klarissa m page 2393,vumc hen { Tomi dersonville - anderson whit { Avani e, tyrone 128 n ande { Steffanie rson ln mrn: 0 { Wm 4771 { Camelia 7361, dob: 7/ { Hadleigh 27/1969, legal sex: m hendersonville tn 37075 visit date: 4/24/20 { Gorge 23 04/24/2023 -  { Jesiah refill in vanderbilt primary care hendersonville facesheet report patient demographi { Taina cs patient name mrn legal dob  { Blessing address phone white, { Shona  tyro { Devontae ne 04771 { Dustyn 73 sex 7 { Zahir /27/1969 { Maida  apt { Mica  705 615-2 { Amariah 60 { Gaylene -2291 ( { Durwood home) 61  { Aidan m  { Danyell 1101 edgehil { Milania l ave 615 { Gitty - { Kaelin 260-2291 (mobile) nashville tn { Deante  37203 *p { Danyel referr { Melissa ed* hospital account not { Favian  on file admissi { Jamon on information current information attending provider admitting provider admission type adm { Crissy ission status  { Ludie unkn { Tamya own s { Juston tatus admission date/time d { Marna isch { Yusef arge date/time hospital service auth/cert status hospital area unit room/bed referring pro { Codie vider 04/24/2023 - refill in v { Leamon a { Madelene nderbilt primary care hendersonville (continued) reason for visit chief complaint med refill visit informatio { Troy n nursing assessmen { Tyana t  assessment available fo { Anessa r  { Talya this encou { Jaysen nter. communication  { Rebeka tracking calls/messages interface (incoming) on 4/24/2023 1523 caller name: va { Jalil nderbilt { Keana  university phone num { Malvin be { Jack r: 615- { Cormac 322-26 { Parrish 88  { Wellington 100 oaks - nashville, tn - 719 thompson ln medication list medication list i this report is for documen { Lacee tation purposes only. the patient should not follow medication instructions within. for accurate instructions { Sharif  regarding medications, the patient should instead consult their physician or afte { Harold r visit summary. active at the end of visit medications last reviewed by lippard, giles a, aprn on 3/6/2023 1516 carvedilol 6. { Denae 2 { Viktor 5 mg tablet (coreg) di { Harrell scont { Yahya inued by: leh { Cheryll mann, melissa cary { Coreen , { Daina  pa-c dis { Qiana continued on: 6/3/2023 instructions: take 1 tablet (6.25 mg total) by mouth daily. authorized by: lippard, giles a, ap { Lawrance r { Miesha n or { Shawnda dered on: 10/25/2022 printed on 10/3/24 7:13 am page 2394",
    {
        "entities": [
            [
                4,
                12,
                "PERSON"
            ],
            [
                35,
                41,
                "PERSON"
            ],
            [
                54,
                60,
                "PERSON"
            ],
            [
                68,
                77,
                "PERSON"
            ],
            [
                178,
                185,
                "PERSON"
            ],
            [
                196,
                201,
                "PERSON"
            ],
            [
                208,
                215,
                "PERSON"
            ],
            [
                232,
                240,
                "PERSON"
            ],
            [
                258,
                264,
                "PERSON"
            ],
            [
                295,
                304,
                "PERSON"
            ],
            [
                346,
                352,
                "PERSON"
            ],
            [
                363,
                369,
                "PERSON"
            ],
            [
                379,
                388,
                "PERSON"
            ],
            [
                394,
                403,
                "PERSON"
            ],
            [
                418,
                422,
                "PERSON"
            ],
            [
                439,
                443,
                "PERSON"
            ],
            [
                505,
                509,
                "PERSON"
            ],
            [
                515,
                523,
                "PERSON"
            ],
            [
                528,
                538,
                "PERSON"
            ],
            [
                544,
                554,
                "PERSON"
            ],
            [
                567,
                575,
                "PERSON"
            ],
            [
                665,
                671,
                "PERSON"
            ],
            [
                690,
                696,
                "PERSON"
            ],
            [
                700,
                706,
                "PERSON"
            ],
            [
                759,
                766,
                "PERSON"
            ],
            [
                797,
                805,
                "PERSON"
            ],
            [
                813,
                819,
                "PERSON"
            ],
            [
                867,
                876,
                "PERSON"
            ],
            [
                920,
                925,
                "PERSON"
            ],
            [
                935,
                943,
                "PERSON"
            ],
            [
                956,
                964,
                "PERSON"
            ],
            [
                1050,
                1057,
                "PERSON"
            ],
            [
                1128,
                1133,
                "PERSON"
            ],
            [
                1137,
                1141,
                "PERSON"
            ],
            [
                1153,
                1162,
                "PERSON"
            ],
            [
                1185,
                1190,
                "PERSON"
            ],
            [
                1220,
                1226,
                "PERSON"
            ],
            [
                1249,
                1259,
                "PERSON"
            ],
            [
                1276,
                1279,
                "PERSON"
            ],
            [
                1286,
                1294,
                "PERSON"
            ],
            [
                1310,
                1319,
                "PERSON"
            ],
            [
                1387,
                1393,
                "PERSON"
            ],
            [
                1412,
                1419,
                "PERSON"
            ],
            [
                1506,
                1512,
                "PERSON"
            ],
            [
                1545,
                1554,
                "PERSON"
            ],
            [
                1577,
                1583,
                "PERSON"
            ],
            [
                1591,
                1600,
                "PERSON"
            ],
            [
                1611,
                1618,
                "PERSON"
            ],
            [
                1629,
                1635,
                "PERSON"
            ],
            [
                1646,
                1652,
                "PERSON"
            ],
            [
                1659,
                1664,
                "PERSON"
            ],
            [
                1677,
                1685,
                "PERSON"
            ],
            [
                1690,
                1698,
                "PERSON"
            ],
            [
                1708,
                1716,
                "PERSON"
            ],
            [
                1728,
                1734,
                "PERSON"
            ],
            [
                1739,
                1747,
                "PERSON"
            ],
            [
                1762,
                1770,
                "PERSON"
            ],
            [
                1782,
                1788,
                "PERSON"
            ],
            [
                1792,
                1799,
                "PERSON"
            ],
            [
                1832,
                1839,
                "PERSON"
            ],
            [
                1851,
                1858,
                "PERSON"
            ],
            [
                1867,
                1875,
                "PERSON"
            ],
            [
                1902,
                1909,
                "PERSON"
            ],
            [
                1928,
                1934,
                "PERSON"
            ],
            [
                2028,
                2035,
                "PERSON"
            ],
            [
                2052,
                2058,
                "PERSON"
            ],
            [
                2065,
                2071,
                "PERSON"
            ],
            [
                2079,
                2086,
                "PERSON"
            ],
            [
                2116,
                2122,
                "PERSON"
            ],
            [
                2129,
                2135,
                "PERSON"
            ],
            [
                2228,
                2234,
                "PERSON"
            ],
            [
                2267,
                2274,
                "PERSON"
            ],
            [
                2278,
                2287,
                "PERSON"
            ],
            [
                2399,
                2404,
                "PERSON"
            ],
            [
                2426,
                2432,
                "PERSON"
            ],
            [
                2461,
                2468,
                "PERSON"
            ],
            [
                2473,
                2479,
                "PERSON"
            ],
            [
                2492,
                2499,
                "PERSON"
            ],
            [
                2522,
                2529,
                "PERSON"
            ],
            [
                2610,
                2616,
                "PERSON"
            ],
            [
                2627,
                2633,
                "PERSON"
            ],
            [
                2657,
                2664,
                "PERSON"
            ],
            [
                2669,
                2674,
                "PERSON"
            ],
            [
                2684,
                2691,
                "PERSON"
            ],
            [
                2700,
                2708,
                "PERSON"
            ],
            [
                2714,
                2725,
                "PERSON"
            ],
            [
                2831,
                2837,
                "PERSON"
            ],
            [
                2949,
                2956,
                "PERSON"
            ],
            [
                3041,
                3048,
                "PERSON"
            ],
            [
                3177,
                3183,
                "PERSON"
            ],
            [
                3187,
                3194,
                "PERSON"
            ],
            [
                3219,
                3227,
                "PERSON"
            ],
            [
                3235,
                3241,
                "PERSON"
            ],
            [
                3257,
                3265,
                "PERSON"
            ],
            [
                3286,
                3293,
                "PERSON"
            ],
            [
                3297,
                3303,
                "PERSON"
            ],
            [
                3315,
                3321,
                "PERSON"
            ],
            [
                3442,
                3451,
                "PERSON"
            ],
            [
                3455,
                3462,
                "PERSON"
            ],
            [
                3469,
                3477,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. { Sanjay  mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 visit date: 9/12/2023 09/12/2023 - teleme { Kaytlyn dicine in vanderbilt imaging servi { Demetris c { Kamora es (continued) after visit summary (contin { Rustin ued)  { Khalia be problem { Vania  list (continued { Huston ) as of 9/12/202 { Susan 3 type 2 diabetes mellitus wit { Leobardo h ch { Joel ronic kidne { Sharyl y disease, with long-term current use of insulin (cms/hcc) s { Reginal lurred speech abnormal gait dizziness tobacco abuse diabetic polyn { Launa europathy associated with type 2 diabetes mellitus (cms/hcc) a { Maurice llergies kidney disease, chronic, stage iv (gfr 15-29 ml/min) (cms/hcc) constipation lumbar back pain home { Merrie lessness hypoalbuminem { Ludwig ia urine test positive for microa { Eugene lbuminuria hype { Milly rkalemia cva (cerebral vascular ac { Royalty cident) (cms/hcc) copd (chronic obstructive pulmonary disease) (cms/hcc) small bowel obstr { Siera uction (cms/hcc) gunshot wound cerebrovascular disease depression asthma nose pain nausea and vomiting persistent proteinuria hypertensive renal disease nicotine { Andra  dependen { Zavion ce secondary hyperparath { Koda yroidism (cms/hcc) vitamin d deficiency chronic kidney di { Yadiel sease due to type 2 diabetes mellitus (cms/hcc) up { Arla per respiratory tract in { Larhonda fection acute cough corns and callosities neuropathy due to typ { Ariela e 2 diabetes mellitus ( { Abygail cms/hc { Makaela c) polysubstan { Elois ce abuse (cms/hcc) chest pain my hea { Rayburn lth at vanderbilt view your after visit summary and more onlin { Vivian e at https://myhealthatvanderbilt.com/ if you have questions, please call 615-343-4357 to speak with our my health at vanderbilt staff. tyrone white { Zyaire  (mrn: 047717361) (7/27/1969) printed at 9/12/2023 1:27 pm { Madysen  page 2 of 4 epic printed on 10/3/24 7:13 am page 1 { Rania 691,vumc a { Aries dult hospital white, tyrone 121 { Lois 1 medical center dr. mrn: 047717361, dob: 7/27/ { Zyon 1969, legal sex: m nashvi { Zenaida lle tn 37232-0004 visit date: 9/12/2023 09/12/2023  { Doran - telemedicine in va { Chadd nde { Kaine rbilt imaging services (continue { Ardell d) after visit summary (continued) your medication list as of september 12 { Faigy , 2023 1:27 pm if you have any questions, a { Rayven sk your nurse or doctor. acetaminophen 32 { Delicia 5 mg tablet take  { Wesley 2 tablets (650 mg total) by mouth every 6 hours co { Rush mmonly known as: tylenol as neede { Clemmie d for mil { Fabio d pain, moderate pain, headaches or fever. albuterol hfa 90 mcg/actuation inhaler inhale 2 { Anette  puff { Triniti s every 4 hours as needed fo { Deonta r wheezing. aspirin 81 mg enteric  { Lavera coated tablet  { Golden take 1 tablet (81 mg total) by mouth { Citlaly  daily. atorvastatin 80 mg tablet take 1 tablet (80 mg total) by mo { Regenia uth daily. commonly known as { Tammara : lipitor azelastine 137 mcg (0.1 %) nasal spray adminis { Jullian ter 1 spray  { Karsen into  { Reatha each nostril 2 times a day. use commonly known as: astelin in each nostril as directed capsaicin 0.1% cream apply 1 application topically { Aryeh  daily for 90 days. cetiriz { Itzayana ine { Samya  10 mg tablet take 1 tablet  { Tinley (10 mg total) by mouth once  { Broc a day as commonly known as: zy { Welton rte { Shena c needed for allergies. cyclobenzaprine 5 mg tablet take 1 { Allysa  tablet (5 mg total { Jhon ) by mouth every 8 hours as commonly known as: flexeril needed. docusate sodium 100 mg capsule take  { Curley one tablet tid prn constipation commonly known as: colace famotidin { Jad e 20 mg tablet take 1 tablet (20 mg total) by { Macario  mouth every  { Justine 12 hours. commonly known as: pepcid f { Ammie reestyle libre 2 sensor kit 1 kit (1 e { Verdie ach total) ev { Marquel ery 14 days. generic drug: flash glucose sensor gabapentin 30 { Suri 0 mg capsule take 1 capsule (300 mg total) by mouth daily. { Bentley  commonly know { Ruthanne n as: neurontin insuli { Rio n glargine 100 u { Lizet nit/ml (lantus) injection inje { Angelyn ct 0.05 ml (5 units total) un { Penni der the skin daily. lancets 33 gauge misc 1  { Kerstin lancet 2 times a day. commonly known as: trueplus lancets losartan 25 mg tablet { Carlin   { Marely take 1 tablet (25 mg tota { Kooper l) by mouth daily. commonly known as: cozaar montelukast 10 mg tablet t { Lawton ake { Xenia  1 tablet (10 mg tota { Washington l) { Kyran  by mout { Hagen h every evening. commonly known as: singulair { Izaak  nifedipine cc 30 mg 24 { Nikita  hr tablet take 1 tablet (30 m { Rosana g total) by mouth daily. commonly known as: adalat cc pantoprazole 20 mg ec tablet take 1 tablet (20 mg total) by mouth daily. comm { Jaziel only known as: protonix tyrone white (mrn: 047717 { Oskar 361) (7/27/1969) printed at 9/12/2023 1:27 pm page 3  { Rosio of 4 epic printed on 10/3/24 7:13 am page 1692",
    {
        "entities": [
            [
                60,
                67,
                "PERSON"
            ],
            [
                181,
                189,
                "PERSON"
            ],
            [
                226,
                235,
                "PERSON"
            ],
            [
                239,
                246,
                "PERSON"
            ],
            [
                291,
                298,
                "PERSON"
            ],
            [
                306,
                313,
                "PERSON"
            ],
            [
                326,
                332,
                "PERSON"
            ],
            [
                351,
                358,
                "PERSON"
            ],
            [
                377,
                383,
                "PERSON"
            ],
            [
                416,
                425,
                "PERSON"
            ],
            [
                432,
                437,
                "PERSON"
            ],
            [
                451,
                458,
                "PERSON"
            ],
            [
                521,
                529,
                "PERSON"
            ],
            [
                598,
                604,
                "PERSON"
            ],
            [
                669,
                677,
                "PERSON"
            ],
            [
                786,
                793,
                "PERSON"
            ],
            [
                818,
                825,
                "PERSON"
            ],
            [
                861,
                868,
                "PERSON"
            ],
            [
                886,
                892,
                "PERSON"
            ],
            [
                929,
                937,
                "PERSON"
            ],
            [
                1030,
                1036,
                "PERSON"
            ],
            [
                1200,
                1206,
                "PERSON"
            ],
            [
                1218,
                1225,
                "PERSON"
            ],
            [
                1252,
                1257,
                "PERSON"
            ],
            [
                1317,
                1324,
                "PERSON"
            ],
            [
                1377,
                1382,
                "PERSON"
            ],
            [
                1409,
                1418,
                "PERSON"
            ],
            [
                1484,
                1491,
                "PERSON"
            ],
            [
                1517,
                1525,
                "PERSON"
            ],
            [
                1534,
                1542,
                "PERSON"
            ],
            [
                1559,
                1565,
                "PERSON"
            ],
            [
                1604,
                1612,
                "PERSON"
            ],
            [
                1677,
                1684,
                "PERSON"
            ],
            [
                1835,
                1842,
                "PERSON"
            ],
            [
                1903,
                1911,
                "PERSON"
            ],
            [
                1965,
                1971,
                "PERSON"
            ],
            [
                1984,
                1990,
                "PERSON"
            ],
            [
                2024,
                2029,
                "PERSON"
            ],
            [
                2079,
                2084,
                "PERSON"
            ],
            [
                2112,
                2120,
                "PERSON"
            ],
            [
                2174,
                2180,
                "PERSON"
            ],
            [
                2203,
                2209,
                "PERSON"
            ],
            [
                2215,
                2221,
                "PERSON"
            ],
            [
                2256,
                2263,
                "PERSON"
            ],
            [
                2340,
                2346,
                "PERSON"
            ],
            [
                2392,
                2399,
                "PERSON"
            ],
            [
                2443,
                2451,
                "PERSON"
            ],
            [
                2471,
                2478,
                "PERSON"
            ],
            [
                2531,
                2536,
                "PERSON"
            ],
            [
                2572,
                2580,
                "PERSON"
            ],
            [
                2592,
                2598,
                "PERSON"
            ],
            [
                2691,
                2698,
                "PERSON"
            ],
            [
                2706,
                2714,
                "PERSON"
            ],
            [
                2745,
                2752,
                "PERSON"
            ],
            [
                2789,
                2796,
                "PERSON"
            ],
            [
                2813,
                2820,
                "PERSON"
            ],
            [
                2859,
                2867,
                "PERSON"
            ],
            [
                2937,
                2945,
                "PERSON"
            ],
            [
                2976,
                2984,
                "PERSON"
            ],
            [
                3043,
                3051,
                "PERSON"
            ],
            [
                3066,
                3073,
                "PERSON"
            ],
            [
                3081,
                3088,
                "PERSON"
            ],
            [
                3228,
                3234,
                "PERSON"
            ],
            [
                3264,
                3273,
                "PERSON"
            ],
            [
                3279,
                3285,
                "PERSON"
            ],
            [
                3316,
                3323,
                "PERSON"
            ],
            [
                3354,
                3359,
                "PERSON"
            ],
            [
                3392,
                3399,
                "PERSON"
            ],
            [
                3405,
                3411,
                "PERSON"
            ],
            [
                3472,
                3479,
                "PERSON"
            ],
            [
                3501,
                3506,
                "PERSON"
            ],
            [
                3609,
                3616,
                "PERSON"
            ],
            [
                3686,
                3690,
                "PERSON"
            ],
            [
                3738,
                3746,
                "PERSON"
            ],
            [
                3762,
                3770,
                "PERSON"
            ],
            [
                3810,
                3816,
                "PERSON"
            ],
            [
                3857,
                3864,
                "PERSON"
            ],
            [
                3880,
                3888,
                "PERSON"
            ],
            [
                3952,
                3957,
                "PERSON"
            ],
            [
                4018,
                4026,
                "PERSON"
            ],
            [
                4043,
                4052,
                "PERSON"
            ],
            [
                4077,
                4081,
                "PERSON"
            ],
            [
                4100,
                4106,
                "PERSON"
            ],
            [
                4139,
                4147,
                "PERSON"
            ],
            [
                4179,
                4185,
                "PERSON"
            ],
            [
                4232,
                4240,
                "PERSON"
            ],
            [
                4322,
                4329,
                "PERSON"
            ],
            [
                4333,
                4340,
                "PERSON"
            ],
            [
                4368,
                4375,
                "PERSON"
            ],
            [
                4449,
                4456,
                "PERSON"
            ],
            [
                4462,
                4468,
                "PERSON"
            ],
            [
                4492,
                4503,
                "PERSON"
            ],
            [
                4508,
                4514,
                "PERSON"
            ],
            [
                4525,
                4531,
                "PERSON"
            ],
            [
                4579,
                4585,
                "PERSON"
            ],
            [
                4611,
                4618,
                "PERSON"
            ],
            [
                4651,
                4658,
                "PERSON"
            ],
            [
                4792,
                4799,
                "PERSON"
            ],
            [
                4851,
                4857,
                "PERSON"
            ],
            [
                4913,
                4919,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medic { Anja al center east white, tyrone  { Emir 1211 medical { Shanel  center dr mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232 visit date: 8/13/2024 08/13/2024 - office visit in vanderbilt diabetes and endocrino { Keoni logy (continued) lmr encoun { Anjanette ter le { Versie vel sca { Anwar ns (continued) da { Giulia ily log libreview july 31, 2 { Kenzi 024 august 13, 2024  { Tiffiny (1 { Nikolaus 4 days) 12am  { Benjamin 2am 4am 6am 8am 10am 1 { Mykayla 2pm 2pm 4pm 6pm { Olan  8pm 10pm 12am 350 mon aug 12 1 { Nakisha 80 { Achilles  70 { Arvil  glucose mg/dl 0 max 285  { Pierson 268 317 371  { Albert 374 { Doretta  27 { Rock 4 289 318 303  { Sammuel 346 372 368 318 292 258 259 250  { Tayler 225 234 244 249 2 { Korina 55 296 263 min 246 { Melodee  226 234 331 306 239 195 231 233 237 256 240 12am 2am 4am 6a { Rilla m 8am { Tionna  10am 12pm 2pm 4pm { Earlean  6pm 8pm 10pm 12am 350 tue { Camellia  aug 1 { Rosalynn 3 o 180 70 glucose mg/dl 0 max 287 257 286 279 260 243 231 248 289 329 min 262 252 271 2 { Merl 64 245 234 210 217 246 { Skye  309 legend high { Zach  glucose  { Kaileigh (>250) low glucose { Araya  (<70) o scans/ { Jocelin views logged post-meal peak  { Nazir new sensor time change 17 7.0u-2.0+0.0 1 { Ferris 5.0u meal correct { Vester ion user change total * strip te { Caley st { Geovanny   { Brynne printed on 1 { Kamaria 0/3/24 7:12 am page 431,vumc adult medical center east white, tyrone 1211  { Becki medical center dr mrn: 047717361, dob: 7/27/1969, legal s { Jeb ex: m nashville tn 37232 visit date: 8/13/2024 08/13/2024 - office v { Celestino isit in vanderbilt diabetes an { Odie d endocrinology (co { Avalyn ntinued) lmr encou { Odelia nter level { Micayla  scans (continued) snapshot libreview page { Keion  july 31, 2 { Rosaura 024 august 13, 2024 (14 da { Shakia ys)  { Jaeda glucose gmi 7.9% or 63  { Tashia mmol/mol carbs average glucose daily carbs 350 gra { Micky ms/day  { Kyah average mg/d { Gennie l glucose 190 mg/dl { Glinda  insulin % ab { Jay ove tar { Elvina get 50 %  { Sterling rapid-act { Fox ing % targ { Davy et 49 % median 18 { Osman 0 insulin units/day % { Cassi  be { Baltazar low target 1 % meal 5th 95th perc { Khaleesi entiles 70 correction user change estina phone 0 12am 6am 12p { Marika m 6pm 12a { Cathi m manual  { Najee low glucose events lon { Nayely g { Serene -acting 100 insulin units/day low glucose e { Genoveva vents 2 90 total daily insulin units/day aver { Jaymes a { Lera ge d { Dell ur { Beck ation 51 min 80 comments 70 . gap { Cameryn s found in the insulin { Kollin  data. 14 d { Said ays 60 in this repo { Cassaundra rting period have   { Odalis 50 insulin events. 4 { Joy 0 12a { Mikah m 6am 12pm 6pm 1 { Rexford 2am gaps f { Margaretta ound in food data. 14 days in this reporting  { Axl period have  fo { Tavaris od  { Kavon sensor usage { Modesto  event { Yuridia s % time sensor is active 100% % time sensor is active 74 % { Hurley  ave { Roddy rage scans/views 6 / da { Mabelle y { Ambar  50% 0% 12a { Keilani m 6am 12pm 6pm { Laurine  12am printed o { Merilyn n 10/3/24 7:12 am  { Pauletta page 432",
    {
        "entities": [
            [
                19,
                24,
                "PERSON"
            ],
            [
                56,
                61,
                "PERSON"
            ],
            [
                76,
                83,
                "PERSON"
            ],
            [
                245,
                251,
                "PERSON"
            ],
            [
                281,
                291,
                "PERSON"
            ],
            [
                300,
                307,
                "PERSON"
            ],
            [
                317,
                323,
                "PERSON"
            ],
            [
                343,
                350,
                "PERSON"
            ],
            [
                381,
                387,
                "PERSON"
            ],
            [
                410,
                418,
                "PERSON"
            ],
            [
                423,
                432,
                "PERSON"
            ],
            [
                448,
                457,
                "PERSON"
            ],
            [
                482,
                490,
                "PERSON"
            ],
            [
                508,
                513,
                "PERSON"
            ],
            [
                547,
                555,
                "PERSON"
            ],
            [
                560,
                569,
                "PERSON"
            ],
            [
                575,
                581,
                "PERSON"
            ],
            [
                609,
                617,
                "PERSON"
            ],
            [
                632,
                639,
                "PERSON"
            ],
            [
                645,
                653,
                "PERSON"
            ],
            [
                659,
                664,
                "PERSON"
            ],
            [
                681,
                689,
                "PERSON"
            ],
            [
                724,
                731,
                "PERSON"
            ],
            [
                751,
                758,
                "PERSON"
            ],
            [
                779,
                787,
                "PERSON"
            ],
            [
                850,
                856,
                "PERSON"
            ],
            [
                864,
                871,
                "PERSON"
            ],
            [
                892,
                900,
                "PERSON"
            ],
            [
                929,
                938,
                "PERSON"
            ],
            [
                947,
                956,
                "PERSON"
            ],
            [
                1047,
                1052,
                "PERSON"
            ],
            [
                1077,
                1082,
                "PERSON"
            ],
            [
                1101,
                1106,
                "PERSON"
            ],
            [
                1118,
                1127,
                "PERSON"
            ],
            [
                1148,
                1154,
                "PERSON"
            ],
            [
                1172,
                1180,
                "PERSON"
            ],
            [
                1211,
                1217,
                "PERSON"
            ],
            [
                1260,
                1267,
                "PERSON"
            ],
            [
                1287,
                1294,
                "PERSON"
            ],
            [
                1329,
                1335,
                "PERSON"
            ],
            [
                1340,
                1349,
                "PERSON"
            ],
            [
                1353,
                1360,
                "PERSON"
            ],
            [
                1375,
                1383,
                "PERSON"
            ],
            [
                1460,
                1466,
                "PERSON"
            ],
            [
                1526,
                1530,
                "PERSON"
            ],
            [
                1601,
                1611,
                "PERSON"
            ],
            [
                1644,
                1649,
                "PERSON"
            ],
            [
                1671,
                1678,
                "PERSON"
            ],
            [
                1699,
                1706,
                "PERSON"
            ],
            [
                1719,
                1727,
                "PERSON"
            ],
            [
                1772,
                1778,
                "PERSON"
            ],
            [
                1792,
                1800,
                "PERSON"
            ],
            [
                1829,
                1836,
                "PERSON"
            ],
            [
                1843,
                1849,
                "PERSON"
            ],
            [
                1875,
                1882,
                "PERSON"
            ],
            [
                1935,
                1941,
                "PERSON"
            ],
            [
                1951,
                1956,
                "PERSON"
            ],
            [
                1971,
                1978,
                "PERSON"
            ],
            [
                2000,
                2007,
                "PERSON"
            ],
            [
                2023,
                2027,
                "PERSON"
            ],
            [
                2037,
                2044,
                "PERSON"
            ],
            [
                2056,
                2065,
                "PERSON"
            ],
            [
                2077,
                2081,
                "PERSON"
            ],
            [
                2094,
                2099,
                "PERSON"
            ],
            [
                2119,
                2125,
                "PERSON"
            ],
            [
                2149,
                2155,
                "PERSON"
            ],
            [
                2161,
                2170,
                "PERSON"
            ],
            [
                2206,
                2215,
                "PERSON"
            ],
            [
                2279,
                2286,
                "PERSON"
            ],
            [
                2298,
                2304,
                "PERSON"
            ],
            [
                2316,
                2322,
                "PERSON"
            ],
            [
                2347,
                2354,
                "PERSON"
            ],
            [
                2358,
                2365,
                "PERSON"
            ],
            [
                2411,
                2420,
                "PERSON"
            ],
            [
                2468,
                2475,
                "PERSON"
            ],
            [
                2479,
                2484,
                "PERSON"
            ],
            [
                2491,
                2496,
                "PERSON"
            ],
            [
                2501,
                2506,
                "PERSON"
            ],
            [
                2542,
                2550,
                "PERSON"
            ],
            [
                2575,
                2582,
                "PERSON"
            ],
            [
                2596,
                2601,
                "PERSON"
            ],
            [
                2623,
                2634,
                "PERSON"
            ],
            [
                2656,
                2663,
                "PERSON"
            ],
            [
                2686,
                2690,
                "PERSON"
            ],
            [
                2698,
                2704,
                "PERSON"
            ],
            [
                2723,
                2731,
                "PERSON"
            ],
            [
                2744,
                2755,
                "PERSON"
            ],
            [
                2803,
                2807,
                "PERSON"
            ],
            [
                2825,
                2833,
                "PERSON"
            ],
            [
                2839,
                2845,
                "PERSON"
            ],
            [
                2860,
                2868,
                "PERSON"
            ],
            [
                2877,
                2885,
                "PERSON"
            ],
            [
                2947,
                2954,
                "PERSON"
            ],
            [
                2961,
                2967,
                "PERSON"
            ],
            [
                2993,
                3001,
                "PERSON"
            ],
            [
                3005,
                3011,
                "PERSON"
            ],
            [
                3025,
                3033,
                "PERSON"
            ],
            [
                3050,
                3058,
                "PERSON"
            ],
            [
                3076,
                3084,
                "PERSON"
            ],
            [
                3105,
                3114,
                "PERSON"
            ]
        ]
    }
),(
    "vumc co { Wanita ol springs 2001 mallory white, tyrone 2001 { Tuesday  mall { Joie ory ln mrn: 0477 { Divya 17361, dob: 7/27/1969, legal sex: m s { Layna te 100 visit date: 9/30/ { Kenney 202 { Elio 4 franklin tn 37067-8 { Naoma 234 09/30/2024 - refill in vanderbilt { Jaedyn  nephrology cool springs (continued) clinical notes  { Mammie (continued) please send responses to rx { Avalynn  rsp pharmacy t { Shaquana echnicia { Taron ns { Brissa  pool t { Samaya o avoid delay in care elec { Arin tronically signed by cartailler, jacqueline t, cpht at 10/2/2024 9:28 am falcone, michelle kathryn, pharmd at 10/ { Daysha 2/2024 1224 author: falcone, mi { Elvie chelle kathr { Olaf yn, service: author type: ph { Johna armacist pharmd filed: 10/2/2024 12:25 pm encou { Reena nter date: 9/30/2024 sta { Lachelle tus: signed editor: f { Gerhard alcone, m { Cianna ichel { Dejon le kathryn, p { Verl harmd (pharm { Alvera acist { Jeremy )  { Tavia patient requesting r { Jacqualine e { Greer fill on lokel { Cornelious ma { Maryanna . lates { Pooja t 09/10/24 referen { Amon ce 02:45 range & units potassium level 3.3 4.8 5.1 (h) mmol/l (h { Brittnie ):  { Arica data is  { Ashlin a { Kamiya bnorma { Katey lly high last  { Cherokee visit: 06/27/24 next visit: 01/06/2 { Italia 5 pended for review and  { Melaine modification if appropriate, thanks! michelle falcon { Betsey e, pharmd clinical pharmacist vanderb { Latavia ilt nephrology electronically { Jammie  signed by falcone, michelle kathry { Jerrel n, pharmd at 10/2/ { Chadrick 2024 12:25 p { Daveon m falco { Rylen ne, mi { Terron chelle kathryn, pha { Anniston rmd at 10/2/ { Noella 2024 1529  { Geary author: falcone, michelle kathryn, service: - autho { Serafina r typ { Maury e: pharmacist pharmd filed: 10/2/2024 3:30 pm encounter date: 9/30/2024 status: signed editor: falcone, michelle kathryn, pharmd (pharmacist) i apologize i missed that!! will wait for further scheduling of appts and  { Javian will refuse refill at th { Varun is time. electronically signed by falcone, michelle kathryn, pharmd at 10/2/2024 { Yancy  3:30 pm other orde { Jazelle rs medications lokelma 10 gra { Jairus m ora { Landin l  { Jonell powder packet (sodium zirco { Shanaya nium cyclosilicate) (pending) electronically signed by: falcone, michelle kathryn, phar { Dominque md on 10/ { Kellye 02/2 { Nino 4 1531 status: pending ordering user: fa { Laticia l { Kelis cone,  { Izaac michelle kathryn, pharmd 10 { Cairo /02/24 1531 ordering pr { Kadeem ovider: freeman, genevieve clayton, aprn  { Kamdyn authorized by: freeman, genevieve clayton, aprn frequency: 10/02/24 - until discontinued class: nor { Roque mal pended b { Sarena y: falcone, michelle kathr { Leeland yn { Alois , pharm { Filiberto d  { Kirra 10/02/24 1 { Lakendra 531 printed on 10/3/24 7:12 am p { Kerwin age 9,vumc cool springs 2001 mallory white, tyrone { Drema  2001 mallory ln mrn: 047717361, dob: 7/ { Nahla 27/1969, { Taylar  legal sex: m ste 100 visit { Ozzie  date: 9/30 { Rome / { Carrington 2024 { Kittie  fra { Holland nklin tn 3 { Danisha 7067-8234 { Jaela  09/30/2024 - refill { Stefania  in  { Raechel vanderbilt nephrology cool springs (continued) othe { Ranae r orders (continued) reordered from: lo { Karter kel { Ninfa ma 10 gram oral powder packet (sodium zirconium cyclos { Tarra ilicate)  { Roby printed on 10/3 { Alethia /2 { Ashia 4 7:12 am page 10",
    {
        "entities": [
            [
                10,
                17,
                "PERSON"
            ],
            [
                62,
                70,
                "PERSON"
            ],
            [
                78,
                83,
                "PERSON"
            ],
            [
                102,
                108,
                "PERSON"
            ],
            [
                148,
                154,
                "PERSON"
            ],
            [
                181,
                188,
                "PERSON"
            ],
            [
                194,
                199,
                "PERSON"
            ],
            [
                223,
                229,
                "PERSON"
            ],
            [
                269,
                276,
                "PERSON"
            ],
            [
                331,
                338,
                "PERSON"
            ],
            [
                380,
                388,
                "PERSON"
            ],
            [
                406,
                415,
                "PERSON"
            ],
            [
                426,
                432,
                "PERSON"
            ],
            [
                437,
                444,
                "PERSON"
            ],
            [
                454,
                461,
                "PERSON"
            ],
            [
                490,
                495,
                "PERSON"
            ],
            [
                611,
                618,
                "PERSON"
            ],
            [
                652,
                658,
                "PERSON"
            ],
            [
                673,
                678,
                "PERSON"
            ],
            [
                709,
                715,
                "PERSON"
            ],
            [
                765,
                771,
                "PERSON"
            ],
            [
                798,
                807,
                "PERSON"
            ],
            [
                831,
                839,
                "PERSON"
            ],
            [
                851,
                858,
                "PERSON"
            ],
            [
                866,
                872,
                "PERSON"
            ],
            [
                888,
                893,
                "PERSON"
            ],
            [
                908,
                915,
                "PERSON"
            ],
            [
                923,
                930,
                "PERSON"
            ],
            [
                935,
                941,
                "PERSON"
            ],
            [
                964,
                975,
                "PERSON"
            ],
            [
                979,
                985,
                "PERSON"
            ],
            [
                1001,
                1012,
                "PERSON"
            ],
            [
                1017,
                1026,
                "PERSON"
            ],
            [
                1036,
                1042,
                "PERSON"
            ],
            [
                1063,
                1068,
                "PERSON"
            ],
            [
                1135,
                1144,
                "PERSON"
            ],
            [
                1150,
                1156,
                "PERSON"
            ],
            [
                1167,
                1174,
                "PERSON"
            ],
            [
                1178,
                1185,
                "PERSON"
            ],
            [
                1194,
                1200,
                "PERSON"
            ],
            [
                1217,
                1226,
                "PERSON"
            ],
            [
                1264,
                1271,
                "PERSON"
            ],
            [
                1298,
                1306,
                "PERSON"
            ],
            [
                1361,
                1368,
                "PERSON"
            ],
            [
                1408,
                1416,
                "PERSON"
            ],
            [
                1448,
                1455,
                "PERSON"
            ],
            [
                1493,
                1500,
                "PERSON"
            ],
            [
                1521,
                1530,
                "PERSON"
            ],
            [
                1545,
                1552,
                "PERSON"
            ],
            [
                1562,
                1568,
                "PERSON"
            ],
            [
                1577,
                1584,
                "PERSON"
            ],
            [
                1606,
                1615,
                "PERSON"
            ],
            [
                1630,
                1637,
                "PERSON"
            ],
            [
                1650,
                1656,
                "PERSON"
            ],
            [
                1710,
                1719,
                "PERSON"
            ],
            [
                1727,
                1733,
                "PERSON"
            ],
            [
                1952,
                1959,
                "PERSON"
            ],
            [
                1986,
                1992,
                "PERSON"
            ],
            [
                2075,
                2081,
                "PERSON"
            ],
            [
                2103,
                2111,
                "PERSON"
            ],
            [
                2143,
                2150,
                "PERSON"
            ],
            [
                2158,
                2165,
                "PERSON"
            ],
            [
                2170,
                2177,
                "PERSON"
            ],
            [
                2207,
                2215,
                "PERSON"
            ],
            [
                2305,
                2314,
                "PERSON"
            ],
            [
                2326,
                2333,
                "PERSON"
            ],
            [
                2340,
                2345,
                "PERSON"
            ],
            [
                2388,
                2396,
                "PERSON"
            ],
            [
                2400,
                2406,
                "PERSON"
            ],
            [
                2415,
                2421,
                "PERSON"
            ],
            [
                2451,
                2457,
                "PERSON"
            ],
            [
                2483,
                2490,
                "PERSON"
            ],
            [
                2534,
                2541,
                "PERSON"
            ],
            [
                2643,
                2649,
                "PERSON"
            ],
            [
                2664,
                2671,
                "PERSON"
            ],
            [
                2700,
                2708,
                "PERSON"
            ],
            [
                2713,
                2719,
                "PERSON"
            ],
            [
                2729,
                2739,
                "PERSON"
            ],
            [
                2744,
                2750,
                "PERSON"
            ],
            [
                2763,
                2772,
                "PERSON"
            ],
            [
                2807,
                2814,
                "PERSON"
            ],
            [
                2867,
                2873,
                "PERSON"
            ],
            [
                2916,
                2922,
                "PERSON"
            ],
            [
                2933,
                2940,
                "PERSON"
            ],
            [
                2970,
                2976,
                "PERSON"
            ],
            [
                2990,
                2995,
                "PERSON"
            ],
            [
                2999,
                3010,
                "PERSON"
            ],
            [
                3017,
                3024,
                "PERSON"
            ],
            [
                3031,
                3039,
                "PERSON"
            ],
            [
                3052,
                3060,
                "PERSON"
            ],
            [
                3072,
                3078,
                "PERSON"
            ],
            [
                3101,
                3110,
                "PERSON"
            ],
            [
                3117,
                3125,
                "PERSON"
            ],
            [
                3179,
                3185,
                "PERSON"
            ],
            [
                3227,
                3234,
                "PERSON"
            ],
            [
                3240,
                3246,
                "PERSON"
            ],
            [
                3303,
                3309,
                "PERSON"
            ],
            [
                3321,
                3326,
                "PERSON"
            ],
            [
                3344,
                3352,
                "PERSON"
            ],
            [
                3357,
                3363,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hun { Dione dred { Meri  oaks white, tyrone 719 thompson  { Zayd lane, na { Dimple shville m { Baila rn: 04771736 { Kyndra 1, dob: 7/27/1969, l { Melania egal sex: m n { Leilany ashville tn 37204 visit date: 1 { Nadya 1/17/2023 11/17/2023 - office visit in vanderbilt one hundred oaks { Antwain   { Lavelle primary care north (c { Gissel ontinued) flowsh { Rosaria eets { Zephaniah  (continued) rati { Taraji o-based meal dosing approx p { Cherilyn redicted 6. { Amanda 1 -sc at 1 { Cosmo 1/17/23 ratio (500 wt in 1313 kg) rec star { Gennaro t sli { Jaye ding 36.5 -sc at 11/17/23 scale (3000 / wt 1313 in kg { Jensen ) basic information app { Libbie rox predicted  { Nana 41.1 -sc at 11/17/23 basal (0.5 * wt  { Daryn in 1313 kg) r { Devaughn ec { Waneta  start basal 20.6 -sc at 11/17/23 (.25 * wt in kg) 1313 rec start fixed 6.9 -sc at 11/17/23 meal (.08 * wt in 1313  { Jeanene kg) fixed meal insulin dosing approx predicted 18.2 -sc at 11/17/23 correction (1500 / 1313 wt in k { Kirsty g) vital signs bmi (calculated { Bell ) 24.6 -sc at 11/17/23 { Eriberto  1318 we { Gatlin i { Coretta ght { Rhyan  and growth recommendation ibw/k { Faris g 73.1 kg -sc at (calc { Adriene ulate { Kenda d) 11/17/23 1318 fem { Tiffaney ale height and weight { Lenwood  weight in (kg) to 83.6 -sc { Jazzmine  at 11/17/23 have bmi = 25 1318 heigh { Rain t and weight weight in (lb) to 183.9 -sc at 11/17/23 have { Deshon  bmi = 25 1318 adult ibw/vt { Carman  calculations ibw/kg 77.6 -sc at 11/17/23 (calculated) 1318 low r { Lisha ange  { Braylin vt 465.6 ml/kg -sc { Jacque  at 6ml { Lindell /kg 11/17/23 131 { Honor 8 adult moderate 620.8 ml/kg -sc at range vt 8ml/kg 11/17/23 1318 adult high range 776 ml/kg -sc at vt 10ml/kg 11/17/23 1318 encounter vitals row name 11/17/23 1313 encounter vitals bp 151/80 -sc at 11/17/23 1318 pulse 94 { Raylynn  -sc at 11/17/23 1318 spo2 { Tonda  100 % -sc at 11/17/23 1318  { Alaia weight 82.3 kg (18 { Ardell 1 lb 7 oz) -sc at 11/17/23 1313 height 182.9 cm (72\") - { Dorthea sc at 11/17/23 1318 lund-br { Tarsha owder (adult { Garvin ) printed on 10/3/24 7:13 { Nicklas  am page 1511,vumc ad { Kasi ult { Ariyanna  one hundred oaks white, tyrone 719 thompson lane, nashville mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date:  { Yazmine 11/17/202 { Ramone 3 11/17/2023 - office visit in vanderbilt one hundred o { Yakov aks primary care north (continued) flowsheets (continued) row name 11/17 { Allisson /23 1313 volume estima { Alverta tes fluid 0 -sc at 11/17/23 1313 { Alpha  resusci { Geraldo tation (#5) flui { Dru d 0 -sc at 1 { Ernst 1/17/23 13 { Fulton 13 resuscitation (#6) fluid 0 -sc at 11/17/23 1313 resuscitation (#7) fluid { Malayah  0 -sc at 11/17/23 1313 resuscitation (#8) fluid 0 -sc at 11/17/23  { Pietro 1313 resuscitation (#9 { Rashid ) fluid 0 -sc at 11/17/23 1313 resuscitatio { Celestina n (#10)  { Daylan pain que { Rebbecca stions row nam { Taurean e 11/17/23 1313 pain assessment is the patient yes ! -sc at 11/17/23 having pain 1313 today? { Candelaria  pain loc back -sc at 11/17/2 { Luci 3 1313 vital signs row name 11/17/ { Glynda 23 1313 other hr/pulse 94 - { Arch sc at 11/17/23 1318  { Coltin user key (r) = recorded by, (t) { Wendall  = taken by, (c) = c { Leslye osigned by initials nam { Maryalice e provider type discipline sc cedillo, stephani { Linton e m { Abriana edical assistant - messages app { Lawanna ointment scheduled fro { Malisa m to sent and delivered mychart, { Dash  generic white, tyrone 11 { Candance /13/202 { Latrina 3 2:42 pm last r { Lezlie ead i { Tala n my health at vanderbilt not re { Laquan ad appointment information: visit type: acute date: 11/ { Hazle 17/2023 dept: vand { Mayah erbilt one hundred { Synthia  oaks pr { Darnell imary care nort { Lashunda h provider: zain m virk pr { Daisey int { Kiran ed  { Corbyn on 10/3/24 7:13 am page 1512",
    {
        "entities": [
            [
                21,
                27,
                "PERSON"
            ],
            [
                34,
                39,
                "PERSON"
            ],
            [
                75,
                80,
                "PERSON"
            ],
            [
                91,
                98,
                "PERSON"
            ],
            [
                110,
                116,
                "PERSON"
            ],
            [
                131,
                138,
                "PERSON"
            ],
            [
                161,
                169,
                "PERSON"
            ],
            [
                185,
                193,
                "PERSON"
            ],
            [
                227,
                233,
                "PERSON"
            ],
            [
                302,
                310,
                "PERSON"
            ],
            [
                314,
                322,
                "PERSON"
            ],
            [
                346,
                353,
                "PERSON"
            ],
            [
                372,
                380,
                "PERSON"
            ],
            [
                387,
                397,
                "PERSON"
            ],
            [
                417,
                424,
                "PERSON"
            ],
            [
                455,
                464,
                "PERSON"
            ],
            [
                478,
                485,
                "PERSON"
            ],
            [
                498,
                504,
                "PERSON"
            ],
            [
                549,
                557,
                "PERSON"
            ],
            [
                565,
                570,
                "PERSON"
            ],
            [
                626,
                633,
                "PERSON"
            ],
            [
                659,
                666,
                "PERSON"
            ],
            [
                683,
                688,
                "PERSON"
            ],
            [
                728,
                734,
                "PERSON"
            ],
            [
                750,
                759,
                "PERSON"
            ],
            [
                764,
                771,
                "PERSON"
            ],
            [
                889,
                897,
                "PERSON"
            ],
            [
                999,
                1006,
                "PERSON"
            ],
            [
                1039,
                1044,
                "PERSON"
            ],
            [
                1069,
                1078,
                "PERSON"
            ],
            [
                1089,
                1096,
                "PERSON"
            ],
            [
                1100,
                1108,
                "PERSON"
            ],
            [
                1114,
                1120,
                "PERSON"
            ],
            [
                1155,
                1161,
                "PERSON"
            ],
            [
                1186,
                1194,
                "PERSON"
            ],
            [
                1202,
                1208,
                "PERSON"
            ],
            [
                1231,
                1240,
                "PERSON"
            ],
            [
                1264,
                1272,
                "PERSON"
            ],
            [
                1302,
                1311,
                "PERSON"
            ],
            [
                1351,
                1356,
                "PERSON"
            ],
            [
                1416,
                1423,
                "PERSON"
            ],
            [
                1453,
                1460,
                "PERSON"
            ],
            [
                1528,
                1534,
                "PERSON"
            ],
            [
                1542,
                1550,
                "PERSON"
            ],
            [
                1571,
                1578,
                "PERSON"
            ],
            [
                1588,
                1596,
                "PERSON"
            ],
            [
                1615,
                1621,
                "PERSON"
            ],
            [
                1845,
                1853,
                "PERSON"
            ],
            [
                1882,
                1888,
                "PERSON"
            ],
            [
                1919,
                1925,
                "PERSON"
            ],
            [
                1946,
                1953,
                "PERSON"
            ],
            [
                2011,
                2019,
                "PERSON"
            ],
            [
                2049,
                2056,
                "PERSON"
            ],
            [
                2071,
                2078,
                "PERSON"
            ],
            [
                2106,
                2114,
                "PERSON"
            ],
            [
                2138,
                2143,
                "PERSON"
            ],
            [
                2149,
                2158,
                "PERSON"
            ],
            [
                2298,
                2306,
                "PERSON"
            ],
            [
                2318,
                2325,
                "PERSON"
            ],
            [
                2383,
                2389,
                "PERSON"
            ],
            [
                2464,
                2473,
                "PERSON"
            ],
            [
                2498,
                2506,
                "PERSON"
            ],
            [
                2541,
                2547,
                "PERSON"
            ],
            [
                2558,
                2566,
                "PERSON"
            ],
            [
                2585,
                2589,
                "PERSON"
            ],
            [
                2604,
                2610,
                "PERSON"
            ],
            [
                2623,
                2630,
                "PERSON"
            ],
            [
                2708,
                2716,
                "PERSON"
            ],
            [
                2786,
                2793,
                "PERSON"
            ],
            [
                2818,
                2825,
                "PERSON"
            ],
            [
                2871,
                2881,
                "PERSON"
            ],
            [
                2892,
                2899,
                "PERSON"
            ],
            [
                2910,
                2919,
                "PERSON"
            ],
            [
                2936,
                2944,
                "PERSON"
            ],
            [
                3039,
                3050,
                "PERSON"
            ],
            [
                3082,
                3087,
                "PERSON"
            ],
            [
                3124,
                3131,
                "PERSON"
            ],
            [
                3161,
                3166,
                "PERSON"
            ],
            [
                3189,
                3196,
                "PERSON"
            ],
            [
                3230,
                3238,
                "PERSON"
            ],
            [
                3261,
                3268,
                "PERSON"
            ],
            [
                3294,
                3304,
                "PERSON"
            ],
            [
                3354,
                3361,
                "PERSON"
            ],
            [
                3367,
                3375,
                "PERSON"
            ],
            [
                3409,
                3417,
                "PERSON"
            ],
            [
                3442,
                3449,
                "PERSON"
            ],
            [
                3484,
                3489,
                "PERSON"
            ],
            [
                3517,
                3526,
                "PERSON"
            ],
            [
                3536,
                3544,
                "PERSON"
            ],
            [
                3563,
                3570,
                "PERSON"
            ],
            [
                3578,
                3583,
                "PERSON"
            ],
            [
                3618,
                3625,
                "PERSON"
            ],
            [
                3683,
                3689,
                "PERSON"
            ],
            [
                3710,
                3716,
                "PERSON"
            ],
            [
                3737,
                3745,
                "PERSON"
            ],
            [
                3756,
                3764,
                "PERSON"
            ],
            [
                3782,
                3791,
                "PERSON"
            ],
            [
                3820,
                3827,
                "PERSON"
            ],
            [
                3833,
                3839,
                "PERSON"
            ],
            [
                3845,
                3852,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hundred oaks white, tyrone 719 thompson { Reign  lane, nashville mrn: 047717361, d { Korbyn ob: 7/27/1969, legal sex: m  { Niyah nashville tn 37204 vis { Persephone it date: - 08/05/2024 - { Tesla  procedure pass in vande { Alonna rbilt one hundred oaks primary care nor { Teanna th facesheet report patient d { Ellison emographics patient name mrn legal dob address phone white, tyrone 0477173 sex 7/27/1969  { Quinlan apt 705 615-260-2291 (home)  { Devora 61 m 1101 edgehill ave 615-260-2291  { Mariya (mobile) nashville tn 37203 *preferred* hospital account not on f { Tea ile admission information curren { Jamilah t informa { Kelci tion { Danniel  atten { Dontay ding provider admi { Jael tting provider admission typ { Lake e admission status unkn { Alexsandra own status admissi { Coralee on date/time { Gerald  discharge date/time hospital service auth { Jacob /cert status hospital area unit roo { Carys m/bed referring provid { Taleah er 08/05/2024 - procedure pass in  { Faron vander { Mathieu bilt one  { Sloan hundred oaks primary care north (conti { Thompson nued) visit information admission { Jubilee  information arrival date/time: admit date/time: ip { Myrle  adm. da { Phyliss te { Fae /time: admi { Jianna ssion typ { Hildred e: point { Shepherd  of origin: admit category:  { Jereme mean { Merna s of arrival: primary service: secondary service: n/a transfer source: serv { Lashawnda ice are { Maycee a: unit: admit provide { Dovid r: attending provider: ref { Ameera erring provider: { Niesha  discharge information date/time: - disposition: - destin { Jajuan ation: - { Evalynn  provide { Silvestre r:  { Lucila unit: - printed on 10/3/24 7:12 { Arik  am  { Benicio page  { Germaine 485,vumc { Ailani  adult one hundred oaks white, tyrone 719 thompson  { Mikeal lane, nashville mr { Ethyl n: 04 { Anayah 77 { Mignon 17361, dob: 7/27/1969, legal s { Yesica ex: m nashville tn 37204 visit date: { Elam  8/5/2024 08/ { Walt 05/2024 { Cicely  - letter (out) in vander { Daven bilt one hundred oaks prim { Gwenda ary care north f { Merlene acesheet report patient demographics patient name mrn legal dob address phone white, tyrone  { Silvana 0477173 sex 7/27/1969 apt 705 615- { Chancellor 260-2291 (home) 61 m 1101 edgehill ave 615-2 { Briseida 60-2291 { Pepper   { Andrey (mo { Dev bile) nashville tn 37203 *preferred* hospital account not on file admission information current informatio { Klayton n attendi { Emelie ng pro { Damen vider { Royal  admitting provider admission type adm { Ricki ission status unknown status admission { Caylin  d { Gabino ate/time discharg { Arika e date/time hospi { Rilynn tal service auth/cert status hospital area unit room/bed referring provider 08/05/2024 - letter (out { Aston ) in vanderbilt one hundred oaks primary care north (continued) visit information provider infor { Sumner mation encounter provider markin, s { Jourdan hannon r, lpn department name address pho { Ronnell ne vanderbilt one hundred oaks primary 7 { Gracey 1 { Jannet 9 thomp { Mikaila son ln 615-936-2187 care north suite 2040 { Brandy 0 nashville tn 37204 l { Kabir etters letter by markin,  { Fairy shannon r, lpn on { Adelbert  8/5/2024 { Margarete  status: sent letter body: augus { Maris t 5, 20 { Alston 24 dear mr. white, below you will find the results fro { Norval m your rece { Peighton nt clinic visit on 8/5/2024. these t { Jeshua est result { Curtis s ha { Denna ve ret { Laronda urned with  { Corbett abnormal va { Daja lues which  { Arletta need to be addressed. printed on 10/3/24 7:12 am page 48 { Myka 6",
    {
        "entities": [
            [
                57,
                63,
                "PERSON"
            ],
            [
                100,
                107,
                "PERSON"
            ],
            [
                138,
                144,
                "PERSON"
            ],
            [
                169,
                180,
                "PERSON"
            ],
            [
                206,
                212,
                "PERSON"
            ],
            [
                239,
                246,
                "PERSON"
            ],
            [
                288,
                295,
                "PERSON"
            ],
            [
                327,
                335,
                "PERSON"
            ],
            [
                427,
                435,
                "PERSON"
            ],
            [
                466,
                473,
                "PERSON"
            ],
            [
                512,
                519,
                "PERSON"
            ],
            [
                587,
                591,
                "PERSON"
            ],
            [
                626,
                634,
                "PERSON"
            ],
            [
                646,
                652,
                "PERSON"
            ],
            [
                659,
                667,
                "PERSON"
            ],
            [
                676,
                683,
                "PERSON"
            ],
            [
                704,
                709,
                "PERSON"
            ],
            [
                740,
                745,
                "PERSON"
            ],
            [
                771,
                782,
                "PERSON"
            ],
            [
                803,
                811,
                "PERSON"
            ],
            [
                826,
                833,
                "PERSON"
            ],
            [
                878,
                884,
                "PERSON"
            ],
            [
                922,
                928,
                "PERSON"
            ],
            [
                953,
                960,
                "PERSON"
            ],
            [
                997,
                1003,
                "PERSON"
            ],
            [
                1012,
                1020,
                "PERSON"
            ],
            [
                1032,
                1038,
                "PERSON"
            ],
            [
                1079,
                1088,
                "PERSON"
            ],
            [
                1124,
                1132,
                "PERSON"
            ],
            [
                1186,
                1192,
                "PERSON"
            ],
            [
                1203,
                1211,
                "PERSON"
            ],
            [
                1216,
                1220,
                "PERSON"
            ],
            [
                1234,
                1241,
                "PERSON"
            ],
            [
                1253,
                1261,
                "PERSON"
            ],
            [
                1272,
                1281,
                "PERSON"
            ],
            [
                1312,
                1319,
                "PERSON"
            ],
            [
                1326,
                1332,
                "PERSON"
            ],
            [
                1410,
                1420,
                "PERSON"
            ],
            [
                1430,
                1437,
                "PERSON"
            ],
            [
                1462,
                1468,
                "PERSON"
            ],
            [
                1497,
                1504,
                "PERSON"
            ],
            [
                1523,
                1530,
                "PERSON"
            ],
            [
                1590,
                1597,
                "PERSON"
            ],
            [
                1608,
                1616,
                "PERSON"
            ],
            [
                1627,
                1637,
                "PERSON"
            ],
            [
                1643,
                1650,
                "PERSON"
            ],
            [
                1684,
                1689,
                "PERSON"
            ],
            [
                1696,
                1704,
                "PERSON"
            ],
            [
                1712,
                1721,
                "PERSON"
            ],
            [
                1732,
                1739,
                "PERSON"
            ],
            [
                1793,
                1800,
                "PERSON"
            ],
            [
                1821,
                1827,
                "PERSON"
            ],
            [
                1835,
                1842,
                "PERSON"
            ],
            [
                1847,
                1854,
                "PERSON"
            ],
            [
                1887,
                1894,
                "PERSON"
            ],
            [
                1933,
                1938,
                "PERSON"
            ],
            [
                1954,
                1959,
                "PERSON"
            ],
            [
                1969,
                1976,
                "PERSON"
            ],
            [
                2004,
                2010,
                "PERSON"
            ],
            [
                2039,
                2046,
                "PERSON"
            ],
            [
                2065,
                2073,
                "PERSON"
            ],
            [
                2168,
                2176,
                "PERSON"
            ],
            [
                2213,
                2224,
                "PERSON"
            ],
            [
                2271,
                2280,
                "PERSON"
            ],
            [
                2290,
                2297,
                "PERSON"
            ],
            [
                2301,
                2308,
                "PERSON"
            ],
            [
                2314,
                2318,
                "PERSON"
            ],
            [
                2427,
                2435,
                "PERSON"
            ],
            [
                2447,
                2454,
                "PERSON"
            ],
            [
                2463,
                2469,
                "PERSON"
            ],
            [
                2477,
                2483,
                "PERSON"
            ],
            [
                2524,
                2530,
                "PERSON"
            ],
            [
                2571,
                2578,
                "PERSON"
            ],
            [
                2583,
                2590,
                "PERSON"
            ],
            [
                2610,
                2616,
                "PERSON"
            ],
            [
                2636,
                2643,
                "PERSON"
            ],
            [
                2746,
                2752,
                "PERSON"
            ],
            [
                2851,
                2858,
                "PERSON"
            ],
            [
                2896,
                2904,
                "PERSON"
            ],
            [
                2948,
                2956,
                "PERSON"
            ],
            [
                2999,
                3006,
                "PERSON"
            ],
            [
                3010,
                3017,
                "PERSON"
            ],
            [
                3027,
                3035,
                "PERSON"
            ],
            [
                3079,
                3086,
                "PERSON"
            ],
            [
                3111,
                3117,
                "PERSON"
            ],
            [
                3145,
                3151,
                "PERSON"
            ],
            [
                3171,
                3180,
                "PERSON"
            ],
            [
                3192,
                3202,
                "PERSON"
            ],
            [
                3237,
                3243,
                "PERSON"
            ],
            [
                3253,
                3260,
                "PERSON"
            ],
            [
                3317,
                3324,
                "PERSON"
            ],
            [
                3338,
                3347,
                "PERSON"
            ],
            [
                3386,
                3393,
                "PERSON"
            ],
            [
                3406,
                3413,
                "PERSON"
            ],
            [
                3420,
                3426,
                "PERSON"
            ],
            [
                3435,
                3443,
                "PERSON"
            ],
            [
                3457,
                3465,
                "PERSON"
            ],
            [
                3479,
                3484,
                "PERSON"
            ],
            [
                3498,
                3506,
                "PERSON"
            ],
            [
                3565,
                3570,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medica { Estel l center east white, tyrone 1211 medical center dr mrn: 047717361, dob: 7/27/19 { Ilan 69, legal sex: m nashville tn 37232 visit date: 8/13/2024 08/13/2024 - o { Jenine ffice visit in vanderbilt diabetes and en { Demonte docrinology (continued) after visit summary (continued) your  { Karlyn medication list { Cardell  as of august 13, 2024 11:23 am if { Mahmoud  you have any questions, ask your nurse o { Thane r doctor. albut { Kalena erol hfa 90 mcg/actuation inhaler { Dominica  inhale 2 puffs ever { Elli y  { Tonisha 4 hours as needed for wheezing. aspirin 81 mg enteric coa { Jens ted t { Kelle ablet take 1 tablet (81 mg total) by mouth daily. atorvastatin 80 mg tablet take 1 tablet (80 mg total) by mouth daily. commonly known as: lipit { Cosette or  { Delaine azelasti { Jeanmarie ne 137 mcg (0.1 %) nasal spray administer 1 spray into each nostril 2 times a day as commonly known as:  { Cecilio astelin needed for rhinitis. use in each nostril as directed bd ultra-fine mini { Tova  pen needle 31 gauge x 3/16\" needle use one new pen needle two times a day for generic drug: pen needle, diabetic lantus for diagnoses: type 2 diabetes mellitus with ch { Akash ronic kidney disease, with long-term current use of insulin,  { Khristian unspecified ckd stage (cms/hcc) capsaicin 0.1% cream apply 1 application topically daily for 90 days. cetirizine 10 mg tablet take 1 tablet (10 mg total) b { Roderic y mouth once a day as { Jaila  commonly known as: zyrtec needed for allergies. clotrimazole 1% cream apply 1 application topically 2 time { Frederica s a day for 30 commonly known as: lotrimin days. cyclobenza { Hillard prin { Kizzy e 5 mg tablet take { Aman  1 tablet (5 mg { Finnian  total) by mouth every 8 hours as commonly known as: flexeril needed. famotidine 20 mg tablet take 1 tablet (20 mg tot { Wolfgang al) by mouth every 12 hours. commonly known as: pep { Calliope cid fluticasone propionate 50 mc { Jet g/actuation na { Auther sal spray administer  { Eliel 2 sprays into each nostr { Ezra il 2 times a day. commonly known as: flonase freestyle libre 2 sensor kit 1 kit  { Diamond (1 e { Kimberely ach total) every 14 d { Pascal ays. generic drug: flash glucose sensor gabapentin 300 mg capsule take 2 capsules (600 mg total) by mouth daily. commonly known as: neurontin ipratropium 42 mcg (0.06 %) nasal spray administer 1 spray int { Deandrea o e { Mistie ach nostril 4 times a day. { Alaric  commonly known as: atrovent start with 1 spray once daily for one week, then for diagnoses: vasomotor rhinitis increase to 2 sprays once a da { Yash y for { Lindsy  2 weeks, can increase to 3-4 sprays a day ketorolac 0.4% drops after surgery, use 1 drop to the right eye e { Cady very 2 commonly known as: acular hours  { Glendora while awake until bedtime. beginning the next day, decrease to 1 drop to the right eye 4 times a day for 2 weeks then s { Zahara top tyrone white (mrn:  { Brighton 047717361) (7/27/1969) printed at 8/13/2024 11:23 am page 7  { Kateri of 8 ep { Katrice ic printed on 10/3/24 7:12 am page 403,vumc adult med { Aubriella ical cent { Laisha er east white, tyrone 1211 medical cente { Chasidy r d { Vashti r mrn: 047717361, dob: 7/27/1969, leg { Tamisha al sex: m  { Marlowe nashville tn 37232 visit date: 8/13/2024 08/13/2024 - office visit in vanderbilt diabetes and endocrinology  { Jalon ( { Jorden continued) after visit summary (continued) your medication list (continued) as of august 13, 202 { Artemio 4 11:23 am lancets 33 gauge misc 1 lancet 2 times a day. commonly known as: trueplus lancets lantus solostar u-100 insu { Skylynn lin 100 unit/ml (3 ml) insulin inject 10 units under the skin 2 ti { Adalee mes a day. pen generic drug: insulin glargine lidocaine %  { Siri mucosal jelly apply topically as needed for mild pain (apply to toe). co { Thresa mmonly known as: xylocaine jelly lidocaine 5 % patch apply 1 patch topically daily. apply to painful area 12 commonly known as: lidod { Coletta erm hours per day, remove { Beryl   { Cienna for 12 hours. l { Jeron okelma 10 gram powder in pack { Mendel et  { Candelario tak { Jamesha e 10 g by mouth daily. generic drug: sodium zirconium cyclosilicate for diagnoses: hyperkalemia { Caelan  montelukast 10 mg tablet take 1 tablet (10 mg total) by mouth  { Chassity every eveni { Melville ng. commonly known as: singulair moxifloxacin 0.5 % ophthalmic solution after surgery, use { Adnan  1 { Ciro  dro { Erlene p to the right eye every 2 commonl { Violetta y known as: vi { Tavis g { Aitana amox hours while awake until bedt { Ammar ime. beginning the next day, decrease to  { Georgeann 1 drop to the right eye 4  { Kirstyn times a day for 1 week, then stop nifedi { Neymar pine cc 30 mg 24 hr tablet take 2 tablets (60 mg total) by mouth daily. c { Hildegarde ommonly known as: adalat cc pantoprazole 20 mg ec tablet take 1 tablet (20 mg  { Maryn total) by mouth daily. com { Brayson monly known as: protonix prednisolone acetate % ophthalmic suspension after sur { Johannes gery, use { Marius  1 drop to the right { Breon  eye every 2 commonly known as: pred forte hours while awake until bedtime. beginning the n { Delfino ext day, decrease to 1 drop to the right eye  { Nayla 4 times a day for 1 week, then 3 times a day for 1 week { Porsche , then 2 times a  { Somer day for 1 { Jordin  week, then daily for 1 week, then stop  { Klara triamcinolone 55 mcg nasal inhaler administer 2 sprays (110 mcg total) into each nostril 2 commonly known as: nasacort times a day. tyrone { Zamir  white (mrn: 047717361) (7/2 { Bianka 7/1969) printed at 8/13/2024 11:2 { Sanjuana 3 am page 8 of 8 epic laboratory reports potassium ivl (completed) electronica { Lennox lly signed by: greenspan, debra l, aprn on 08/13/24 1107 status: completed ordering user: greenspan, debra l, aprn 08/13/2 { Abdulrahman 4 1107 ordering provider: greenspan,  { Adriano debra l, aprn  { Calen authorized by: greenspan, debra l, aprn ordering mode: standard printed on 10/3/24 7:12 am page 404",
    {
        "entities": [
            [
                20,
                26,
                "PERSON"
            ],
            [
                108,
                113,
                "PERSON"
            ],
            [
                188,
                195,
                "PERSON"
            ],
            [
                239,
                247,
                "PERSON"
            ],
            [
                311,
                318,
                "PERSON"
            ],
            [
                336,
                344,
                "PERSON"
            ],
            [
                381,
                389,
                "PERSON"
            ],
            [
                433,
                439,
                "PERSON"
            ],
            [
                457,
                464,
                "PERSON"
            ],
            [
                500,
                509,
                "PERSON"
            ],
            [
                532,
                537,
                "PERSON"
            ],
            [
                542,
                550,
                "PERSON"
            ],
            [
                610,
                615,
                "PERSON"
            ],
            [
                623,
                629,
                "PERSON"
            ],
            [
                776,
                784,
                "PERSON"
            ],
            [
                790,
                798,
                "PERSON"
            ],
            [
                809,
                819,
                "PERSON"
            ],
            [
                926,
                934,
                "PERSON"
            ],
            [
                1016,
                1021,
                "PERSON"
            ],
            [
                1192,
                1198,
                "PERSON"
            ],
            [
                1262,
                1272,
                "PERSON"
            ],
            [
                1430,
                1438,
                "PERSON"
            ],
            [
                1462,
                1468,
                "PERSON"
            ],
            [
                1578,
                1588,
                "PERSON"
            ],
            [
                1650,
                1658,
                "PERSON"
            ],
            [
                1665,
                1671,
                "PERSON"
            ],
            [
                1692,
                1697,
                "PERSON"
            ],
            [
                1715,
                1723,
                "PERSON"
            ],
            [
                1844,
                1853,
                "PERSON"
            ],
            [
                1907,
                1916,
                "PERSON"
            ],
            [
                1951,
                1955,
                "PERSON"
            ],
            [
                1972,
                1979,
                "PERSON"
            ],
            [
                2003,
                2009,
                "PERSON"
            ],
            [
                2036,
                2041,
                "PERSON"
            ],
            [
                2124,
                2132,
                "PERSON"
            ],
            [
                2139,
                2149,
                "PERSON"
            ],
            [
                2173,
                2180,
                "PERSON"
            ],
            [
                2387,
                2396,
                "PERSON"
            ],
            [
                2402,
                2409,
                "PERSON"
            ],
            [
                2438,
                2445,
                "PERSON"
            ],
            [
                2590,
                2595,
                "PERSON"
            ],
            [
                2603,
                2610,
                "PERSON"
            ],
            [
                2721,
                2726,
                "PERSON"
            ],
            [
                2768,
                2777,
                "PERSON"
            ],
            [
                2899,
                2906,
                "PERSON"
            ],
            [
                2932,
                2941,
                "PERSON"
            ],
            [
                3004,
                3011,
                "PERSON"
            ],
            [
                3021,
                3029,
                "PERSON"
            ],
            [
                3085,
                3095,
                "PERSON"
            ],
            [
                3107,
                3114,
                "PERSON"
            ],
            [
                3157,
                3165,
                "PERSON"
            ],
            [
                3171,
                3178,
                "PERSON"
            ],
            [
                3218,
                3226,
                "PERSON"
            ],
            [
                3239,
                3247,
                "PERSON"
            ],
            [
                3358,
                3364,
                "PERSON"
            ],
            [
                3368,
                3375,
                "PERSON"
            ],
            [
                3474,
                3482,
                "PERSON"
            ],
            [
                3604,
                3612,
                "PERSON"
            ],
            [
                3681,
                3688,
                "PERSON"
            ],
            [
                3749,
                3754,
                "PERSON"
            ],
            [
                3829,
                3836,
                "PERSON"
            ],
            [
                3972,
                3980,
                "PERSON"
            ],
            [
                4008,
                4014,
                "PERSON"
            ],
            [
                4018,
                4025,
                "PERSON"
            ],
            [
                4043,
                4049,
                "PERSON"
            ],
            [
                4081,
                4088,
                "PERSON"
            ],
            [
                4094,
                4105,
                "PERSON"
            ],
            [
                4111,
                4119,
                "PERSON"
            ],
            [
                4217,
                4224,
                "PERSON"
            ],
            [
                4290,
                4299,
                "PERSON"
            ],
            [
                4313,
                4322,
                "PERSON"
            ],
            [
                4415,
                4421,
                "PERSON"
            ],
            [
                4426,
                4431,
                "PERSON"
            ],
            [
                4438,
                4445,
                "PERSON"
            ],
            [
                4482,
                4491,
                "PERSON"
            ],
            [
                4508,
                4514,
                "PERSON"
            ],
            [
                4518,
                4525,
                "PERSON"
            ],
            [
                4561,
                4567,
                "PERSON"
            ],
            [
                4611,
                4621,
                "PERSON"
            ],
            [
                4650,
                4658,
                "PERSON"
            ],
            [
                4701,
                4708,
                "PERSON"
            ],
            [
                4784,
                4795,
                "PERSON"
            ],
            [
                4876,
                4882,
                "PERSON"
            ],
            [
                4911,
                4919,
                "PERSON"
            ],
            [
                5001,
                5010,
                "PERSON"
            ],
            [
                5022,
                5029,
                "PERSON"
            ],
            [
                5052,
                5058,
                "PERSON"
            ],
            [
                5152,
                5160,
                "PERSON"
            ],
            [
                5208,
                5214,
                "PERSON"
            ],
            [
                5272,
                5280,
                "PERSON"
            ],
            [
                5300,
                5306,
                "PERSON"
            ],
            [
                5318,
                5325,
                "PERSON"
            ],
            [
                5368,
                5374,
                "PERSON"
            ],
            [
                5515,
                5521,
                "PERSON"
            ],
            [
                5552,
                5559,
                "PERSON"
            ],
            [
                5595,
                5604,
                "PERSON"
            ],
            [
                5685,
                5692,
                "PERSON"
            ],
            [
                5817,
                5829,
                "PERSON"
            ],
            [
                5869,
                5877,
                "PERSON"
            ],
            [
                5894,
                5900,
                "PERSON"
            ]
        ]
    }
),(
    "vumc the vanderbilt clinic white, tyrone 1301 medical center dr mrn: 047717361, dob: 7/27/1969, l { Kristen egal sex: m the vanderbilt clinic visit date: 1/5/2023 nashville tn 37232- { Sanders 0028 01/05/2023 - appointment in vanderbilt nephrology/renal transplant clinic (continued) medic { Arwen ation list (continued) quantity: 60 capsule refill:  remaining fluticasone propionate 50 mcg/ { Travion actuation nasal spray,suspension (flonase) discontinued by: lippard, giles a, aprn discontinued on: 6/6/2023 reason for discontinuation: reorder instructions: administer 2 sp { Madlyn rays in { Christiane to each nostril daily. authorized by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quantity: 1 { Keena 6 { Sheilah  g refill: 11 refi { Conway lls by 10/25/2023 azelastine 1 { Norton 37 mcg (0.1 %) nasal spray aerosol (astelin) discontinued by: lippard, giles a, aprn dis { Rudolfo continued on: 8/8/2023 reason for discontinuation: reorder instructions: administer 1 spray into each nostr { Donya il 2 times a day. use in each nostril as directed authorized by: lippard, giles a,  { Carver aprn ordered on: 10/25/2022 start date: 10/25/2022 quantity: 30 ml refill: 12 refills by 10/2 { Kierstyn 5/2023 lidocaine 5 % topical patch (lidoderm) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6 { Burley /3/2023 re { Marshawn ason for discontinuation: stop taking at discharge (cancelrx) instructions: apply 1 patch topically daily { Andre . apply to painful area 12 hours per day, remove for 12 hours. authoriz { Fredia ed by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 en { Marlyn d date: 6/3/2023 action: patient { Arissa  not taking quan { Kaydee tity: 30 patch ref { Raine ill: 11 refills by 10/25/2023 albut { Anastacio erol sulfate hfa 90 mcg/actuation aerosol inhaler discontinued by: lippard, giles a, aprn discontinued o { Mcarthur n: 8/8/2023 reason { Micaiah  for discontinuation: reorder instruc { Romel tions: inhale 2 puffs every 4 hours as ne { Dafne eded for wheezing. authorized by: lippard, giles  { Indira a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quant { Lillia ity: 18 g r { Blain efill: 11 refills by 10/25/2023 pantoprazole 20 mg tablet,delayed release (protonix) discontinued by: greenspan, debra l, aprn discontinued on: 7/9/2024 reason f { Mckenzie or discontinuation: reorder instructions: take 1 tablet (20 mg total) by mouth daily. authorized by: lippard, giles  { Dior a, aprn ordered on: 11/29/2022 start date: 11/29/2022 end date: 7/9/2024 quanti { Keith ty: 30 tablet refill: 11 refills by 11/29/2023 sodium { Olyvia  chloride 0.65 % nasal spray aerosol (ocean { Dashiell  nasal) discontinued by: lehmann, melissa c { Jarad ary, pa-c discontinu { Kashton ed on: 6/3/2023 reason for discontinuati { Jazzmin on: stop taking at discharge (cancelrx) instructions: administer 1 spray into each nostril as needed for rhinitis. authorized by: l { Chesley ippard, giles a,  { Aleen aprn ordered on: 12/6/2022 start date: 12/6/2022 end date: 6/3/2023 quantity: 15 ml refill: 12 refills by 12/6/2023 acetaminophen 325 mg tablet (tylenol) discontinued by: { Verlie  lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for di { Mattias scontinuation { Beverlee : reorder instructions: take 2 tablets (650 { Danni  mg total) by mouth every 6 hours as needed for mild pain. authorized by: lippard, giles a, aprn ordered on: 12/9/2 { Tennille 022 start date: 12/9/2022 end date: 6/3/2023 action: patient not taking quantity: 30 tablet refill:  remaining printed on 10/3/24 7:13 am page 2767,vumc the vanderbilt clinic w { Loma hite, tyrone 1301 medical center dr mrn: 047717361, dob: 7/27/1969, legal sex: m the vanderbilt clinic visit date: 1/5/2023 nashville tn 37232-0028 01/05/2023 - appointment { Kendric   { Kyren in vanderb { Camdyn ilt nephrology/renal tra { Peri nsplant clinic (continued) medicat { Tequila ion list (continued) trulicity 3 mg/0.5 ml sub { Catelyn cutaneou { Lauralee s pen injector (dulaglutide) discontinued by: wilson, danya horchi, pharmd discontinued on: 3/1/2 { Lynelle 023 reason for d { Tari iscontinuation: other (cancelrx) instructions: inject 3 mg under the skin e { Wells very 7 days. authorized by: lippard,  { Malea giles a, aprn ordered on: 1/4/2023 start date: 1/4/2023 end date: 3/1/2023 quantity: 2 ml refill: 2 refills by 1/4/2024 cyclobenzaprine 5 mg tablet (flexeril) [reconciled by woehler, kristina, rn on 1/11/2023 0756] discontinued by: lehmann, melissa cary, pa- { Kensington c discontinued on: 6/3/2023 reason for discontin { Marquetta uation: stop taking at  { Tamyra discharge (cancelrx) instru { Audie ctions: take 1 tablet ( { Leonore 5 mg total) by mouth every 8 hours as needed. entered  { Amada by: { Ozella  woehler, kristina, rn entered on: 1/11/2023 start date: 12/7/2022 end date: 6/3/2023 famotidine 20 { Young  mg tablet (pepcid) [reconciled by ferguson, sherri  { Ayan l, lpn on 1/11/2023 1257] instructions:  { Earlie take 1 tablet (20 mg total) by mouth every 12 hours. entered by: ferguson, sherri l, lpn entered on: 1/11/2023 benzonatate 200 mg capsule (tessalon) discont { Charise inued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for discont { Donna inuatio { Kaniya n: stop taking at discharge (cancelrx) instructions: ta { Ashlei ke 1 capsule (200 mg total) by mouth 3 times a day as needed for  { Lory cough. au { Maylin thorized by: lippard, giles a, aprn ordered on: 1/11/2023 start date: 1/11/2023 end date: 6/3/2023 action: patient not taking quantity: 42 capsule  { Khristopher refill:  rem { Emoni aining o { Oneida ndansetron hcl 4 mg tablet (zofr { Arionna an) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for disconti { Beaulah nuation: stop taking at discharge (cancelrx) instructions: take 1 tablet (4 mg tot { Hedy al) by mouth  { Taneisha 3 times a day as needed for nausea or vomiting. authorize { Callista d by: lippard, giles a, aprn ordered on: 1/11/20 { Glenwood 23 start date: 1/11/ { Theodis 2023 end date: 6/3/2023 action: patient not taking quantity: 20 tablet refill:  remaining ergocalciferol  { Viktoria (vit { Yaretzy amin d2) 1,250 mcg (50,000 unit) capsule (vitamin d2) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for discontinuation: stop taking at discharge (cancelrx) instructions: take 1 capsule (50,000 units total) by mouth wee { Delmas kl { Emmaleigh y. authorized by: wang, zhijian, aprn o { Gema rdered on: 2/7/2023 start { Jeanelle  date: 2/7/2023 end date: 6/3/2023 quantity: 12 c { Donavin apsule  { Umar refill:  remaining gabapentin 300 mg capsule (neurontin { Kaycie ) discontinued by: darks, tina m, apr { Laurie n discontinued on: 3/13/2023 instructio { Helga ns: take one capsule by mouth three times a day a { Kaily uthorized by: lippard, giles a, aprn ord { Juliane ered on: 2/9/2023  { Lakyn start date: 2/9/2023 quantity: { Elinore  90 capsule refill:  remaining stopped in visi { Jimmy t none printed on 10/3/24 7:13 a { Tamica m page 2768",
    {
        "entities": [
            [
                100,
                108,
                "PERSON"
            ],
            [
                185,
                193,
                "PERSON"
            ],
            [
                292,
                298,
                "PERSON"
            ],
            [
                394,
                402,
                "PERSON"
            ],
            [
                579,
                586,
                "PERSON"
            ],
            [
                596,
                607,
                "PERSON"
            ],
            [
                728,
                734,
                "PERSON"
            ],
            [
                738,
                746,
                "PERSON"
            ],
            [
                767,
                774,
                "PERSON"
            ],
            [
                807,
                814,
                "PERSON"
            ],
            [
                905,
                913,
                "PERSON"
            ],
            [
                1023,
                1029,
                "PERSON"
            ],
            [
                1115,
                1122,
                "PERSON"
            ],
            [
                1218,
                1227,
                "PERSON"
            ],
            [
                1339,
                1346,
                "PERSON"
            ],
            [
                1359,
                1368,
                "PERSON"
            ],
            [
                1476,
                1482,
                "PERSON"
            ],
            [
                1556,
                1563,
                "PERSON"
            ],
            [
                1644,
                1651,
                "PERSON"
            ],
            [
                1686,
                1693,
                "PERSON"
            ],
            [
                1712,
                1719,
                "PERSON"
            ],
            [
                1740,
                1746,
                "PERSON"
            ],
            [
                1784,
                1794,
                "PERSON"
            ],
            [
                1901,
                1910,
                "PERSON"
            ],
            [
                1931,
                1939,
                "PERSON"
            ],
            [
                1979,
                1985,
                "PERSON"
            ],
            [
                2029,
                2035,
                "PERSON"
            ],
            [
                2087,
                2094,
                "PERSON"
            ],
            [
                2156,
                2163,
                "PERSON"
            ],
            [
                2177,
                2183,
                "PERSON"
            ],
            [
                2347,
                2356,
                "PERSON"
            ],
            [
                2475,
                2480,
                "PERSON"
            ],
            [
                2562,
                2568,
                "PERSON"
            ],
            [
                2624,
                2631,
                "PERSON"
            ],
            [
                2677,
                2686,
                "PERSON"
            ],
            [
                2732,
                2738,
                "PERSON"
            ],
            [
                2761,
                2769,
                "PERSON"
            ],
            [
                2812,
                2820,
                "PERSON"
            ],
            [
                2954,
                2962,
                "PERSON"
            ],
            [
                2982,
                2988,
                "PERSON"
            ],
            [
                3161,
                3168,
                "PERSON"
            ],
            [
                3239,
                3247,
                "PERSON"
            ],
            [
                3263,
                3272,
                "PERSON"
            ],
            [
                3318,
                3324,
                "PERSON"
            ],
            [
                3442,
                3451,
                "PERSON"
            ],
            [
                3630,
                3635,
                "PERSON"
            ],
            [
                3810,
                3818,
                "PERSON"
            ],
            [
                3822,
                3828,
                "PERSON"
            ],
            [
                3841,
                3848,
                "PERSON"
            ],
            [
                3875,
                3880,
                "PERSON"
            ],
            [
                3917,
                3925,
                "PERSON"
            ],
            [
                3974,
                3982,
                "PERSON"
            ],
            [
                3993,
                4002,
                "PERSON"
            ],
            [
                4102,
                4110,
                "PERSON"
            ],
            [
                4129,
                4134,
                "PERSON"
            ],
            [
                4212,
                4218,
                "PERSON"
            ],
            [
                4258,
                4264,
                "PERSON"
            ],
            [
                4525,
                4536,
                "PERSON"
            ],
            [
                4587,
                4597,
                "PERSON"
            ],
            [
                4623,
                4630,
                "PERSON"
            ],
            [
                4660,
                4666,
                "PERSON"
            ],
            [
                4692,
                4700,
                "PERSON"
            ],
            [
                4757,
                4763,
                "PERSON"
            ],
            [
                4769,
                4776,
                "PERSON"
            ],
            [
                4878,
                4884,
                "PERSON"
            ],
            [
                4939,
                4944,
                "PERSON"
            ],
            [
                4987,
                4994,
                "PERSON"
            ],
            [
                5153,
                5161,
                "PERSON"
            ],
            [
                5246,
                5252,
                "PERSON"
            ],
            [
                5262,
                5269,
                "PERSON"
            ],
            [
                5327,
                5334,
                "PERSON"
            ],
            [
                5402,
                5407,
                "PERSON"
            ],
            [
                5419,
                5426,
                "PERSON"
            ],
            [
                5576,
                5588,
                "PERSON"
            ],
            [
                5603,
                5609,
                "PERSON"
            ],
            [
                5620,
                5627,
                "PERSON"
            ],
            [
                5662,
                5670,
                "PERSON"
            ],
            [
                5767,
                5775,
                "PERSON"
            ],
            [
                5860,
                5865,
                "PERSON"
            ],
            [
                5881,
                5890,
                "PERSON"
            ],
            [
                5950,
                5959,
                "PERSON"
            ],
            [
                6010,
                6019,
                "PERSON"
            ],
            [
                6042,
                6050,
                "PERSON"
            ],
            [
                6158,
                6167,
                "PERSON"
            ],
            [
                6174,
                6182,
                "PERSON"
            ],
            [
                6436,
                6443,
                "PERSON"
            ],
            [
                6448,
                6458,
                "PERSON"
            ],
            [
                6500,
                6505,
                "PERSON"
            ],
            [
                6533,
                6542,
                "PERSON"
            ],
            [
                6594,
                6602,
                "PERSON"
            ],
            [
                6612,
                6617,
                "PERSON"
            ],
            [
                6675,
                6682,
                "PERSON"
            ],
            [
                6722,
                6729,
                "PERSON"
            ],
            [
                6771,
                6777,
                "PERSON"
            ],
            [
                6829,
                6835,
                "PERSON"
            ],
            [
                6878,
                6886,
                "PERSON"
            ],
            [
                6907,
                6913,
                "PERSON"
            ],
            [
                6946,
                6954,
                "PERSON"
            ],
            [
                7003,
                7009,
                "PERSON"
            ],
            [
                7044,
                7051,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hundred oaks whit { Imari e, tyrone 719 thompson lane, na { Makyla shville mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37204 visit date: 8/5/2024 08/05/2024 - orders  { Perri only in vanderbilt one hundr { Mick ed oaks primary care north (continued) medication list (continued) instructions: take 1 tablet (10 mg total) by  { Lillyan mouth every evening. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/2023 quantity: 90  { Karime tablet refill: 3 refills by 8/7/2024 cyclo { Kiran benzaprin { Travis e 5 mg tablet (flexeril) [reconciled b { Aarush y maples, chantis on 9/5/2023 1522] instructions: take 1 tablet (5 mg total) by mouth every 8 hours as needed. entered by: maples, chantis entered on: 9/5/2023 start date: 7/22/2023 as { Angela pirin 81 mg tablet,delayed release instructions: take 1 tablet (81 mg total) by mo { Adilynn uth daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 quantity: 90 t { Rebecca ablet refill: 3 refills by 9/6/2024 cetirizine 10 mg tablet (zyrtec) instructions: take 1 table { Nika t (10 mg total) by mouth once a day as needed for allergies. authorized by: de witte, anton jordan, md ordered on: 10/ { Waverly 4/2023 start date: 10/4/2023 quantity: 30 tablet refill: 9 refills by  { Brayton 10/3/2024 lidocaine 5 % topical patch (lidoderm) instructions:  { Earl apply 1 patch topically daily. apply to painful area 12 hours per day, remove for 12 hours. authorized by: de  { Marilu witte, anton jordan, md ordered on: 10/17/ { Corry 2023 start date: 10/17/2023 end date: 10/16/2024 quantity: 30 patch refill: 11 refills by 10/16/2024 azelastine 137 { Kipp  mcg (0.1 %) n { Felipa asal spray aerosol (astelin) instructions: administer 1 sp { Tallulah ray into each nostril 2 times a day as needed for rhinitis { Passion . use in each nostril as directed authorized by: de witte, anton jordan, { Twana  md ordered on: 10/17/ { Janya 2023 start dat { Ciana e: 10/17/2023 quantity: 30 ml refill:  remaining ni { Marie fedipine e { Alyna r 30 mg tablet,extended release (adalat cc { Celso ) instructions: take 2 tablets (60 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 4/23 { Chynna /2024 sta { Shemar rt date: 4/23/2024 quantity: 180 tablet refill: 3 refills by 4/23/20 { Kacee 25 lantus solostar u-100 insulin 100 unit/ml (3 ml) subcutaneous pen (insulin glargine) instructions: inject 10 units under the skin 2 times a day. authorized by: greenspan, debra l, aprn ordered on: 7/9/2024 start date: 7/9/2024 qua { Thomasine ntity: 18 ml refill: 3 refills by 7/9/2025 pantoprazole 20 mg tablet,delayed release (protonix) instructions: take 1 tablet (20 mg total { Almeta ) by mouth daily. au { Rori thorized by: greenspan, debra l, aprn ordered on: 7/9/2024 start date: 7/9/2024 end date: 7/9/2025 quantity: 30 tablet  { Adam refill: 11 refills by  { Indigo 7/9/2025 prednisolone acet { Korie ate 1 % eye drops,suspension (pred forte) instructions: after surgery { Maisy , use 1 drop to the right eye every 2 hours wh { Sahar ile awake until bedtime. beginning { Tanner   { Sven the next day, decrease to 1 drop to the righ { Talmage t eye 4 times a day for 1 week, then { Arlie  3 time { Shan s a day for { Maryrose  1 week, then 2 time { Ahmir s a day for 1 week, then daily for 1 week, then stop authorized by: valenzuela, daniel alejandro, md ordered on: 7/22/2024 start date: 7/22/ { Hilary 2024 quantity: 5 ml refill: 1 refil { Edyth l by 7/22/2025 printed on 10/3/24 7:12 am page 475,vumc adult one hundred { Stasia  oaks white, { Dayle  tyrone 719 thompson  { Karren lane, nashville mrn: 047717361, dob: 7/27/1969, legal sex: m nashvi { Kassi lle tn 37204 visit date: 8/5/2024 08/05/2024 - orders only in vanderbilt one hundred  { Courtnie oaks primar { Deedra y care north (continued) medication list (continued) moxifloxacin 0.5 % eye drops (vigamox) instructions: after  { Shaye surgery, { Milburn  use 1 drop to the right eye every 2 hours while awake u { Daijah ntil bedtime. beginning the next day, decrease to 1 drop to the right eye 4 times a day for 1 week, th { Delinda en stop authorized by: valenzuela, daniel alejandro, md ordered on: 7/22/2024 start date: 7/22/2024 quant { Felica ity: 3 ml refill: 1 refill by 7/22/2025 clotrimazole 1 % topical cream  { Pamelia (lotrimin) instructions: app { Kaira ly 1 application topically 2  { Karey times a day for 30 days. authorize { Seanna d by: pauw, emily kathryn, md ordered on: 7/26/2024 start date: 7/26/2024 quantity: 28 g refill:  remaining albuterol sulfate hfa 90 mcg/actuation aerosol inhaler instructions: inhale 2 puffs every 4 hours as needed for wheezing. authorized by: chakravarthy, rohini, md { Baldemar  ordered on: 8/6/2024 start date: 8/6/2024 quantity: 18 g refill { Daniell : 11 refills by 8/6/2025 fluticasone propionate 50 mcg/actuat { Wendie ion { Larkin  nasal spray { Jesslyn ,s { Jude uspension { Feliciano  (flonase) instructions: administer 2 sprays into each nostril 2 times a da { Javin y. authorized by: c { Royce onnolly, patrick james, md ordered on: 8/6/2024 start date: 8/6/2024 quanti { Manley ty: 16 g ref { Louise ill: 2 refills by 8/6/ { Aranza 2025 gabapentin 300 mg ca { Laith psule (neurontin) instructions: take 2 capsules (600 mg total) by mouth dai { Malek ly. authorized by: chakravarthy, rohini, md ordered on: 8/6/2024 start date: 8/6/2024 quantity: 180 capsule refill:  remaining ipratropium bromide 42 mcg (0.06%) nasal spray (atrovent) instructions: administer 1 spray into each nostril 4 times a day. start with 1 spray once daily for one week, then in { Ellery crease to 2 sprays once a day for 2 weeks, can increase to { Ayden  3-4 sprays a day auth { Cleopatra ori { Charleston ze { Gregoria d by: chakravarthy, rohini, { Myia  md ordered on: 8/6/2024 sta { Citlalli rt date: 8/6/2024 quan { Derrik tity: 15 ml refill: { Malaika  12 refills by 8/6/2025 loperamide 2 mg capsule (imo { Breck dium) instructions: take 1 capsule (2 mg total) by mouth { Krissy  3 times a day as needed for diarrhea for up to 10 days. authorized by: chakravarthy, rohini, md ordered on: 9/3/2024 start date: 9/3/2024 quantity: 30 capsule refill:  remaining losartan 25 mg tablet  { Chesney (cozaar) [reconciled by gingrow, barbara, l { Fidencio pn on 9/16/2024 0800] e { Keshaun ntered by: gingrow,  { Osiel barbara, lpn entered on: 9/16/2024 start date: 8/27/2024 u { Claire rea 20 % topical cream (carmol) instructions: a { Evander pply  { Cathey 1 application topically as needed for dry skin (apply 1 gram to affected area) for up to 365 days. authorized  { Cheree by: hick { Michela s, adam bradburn, dpm ordered on: 9/16/2024 start date: 9/16/2024 quantity: 85 g refill: 1 refill by 9/16/2025 atorvastatin 80 mg tablet (lipitor) instructions: take one tablet by m { Ishan outh every day authorized by: chakravarthy, rohini, md ordered on: 10/2/2024 start date: 10/2/2024 quantit { Derald y: 90 tablet refill: 1 refill by 1 { Yonatan 0/2/2025 printed on 10/3/24 7:12 am page 476",
    {
        "entities": [
            [
                35,
                41,
                "PERSON"
            ],
            [
                75,
                82,
                "PERSON"
            ],
            [
                198,
                204,
                "PERSON"
            ],
            [
                235,
                240,
                "PERSON"
            ],
            [
                355,
                363,
                "PERSON"
            ],
            [
                480,
                487,
                "PERSON"
            ],
            [
                532,
                538,
                "PERSON"
            ],
            [
                550,
                557,
                "PERSON"
            ],
            [
                598,
                605,
                "PERSON"
            ],
            [
                792,
                799,
                "PERSON"
            ],
            [
                884,
                892,
                "PERSON"
            ],
            [
                1004,
                1012,
                "PERSON"
            ],
            [
                1110,
                1115,
                "PERSON"
            ],
            [
                1236,
                1244,
                "PERSON"
            ],
            [
                1317,
                1325,
                "PERSON"
            ],
            [
                1391,
                1396,
                "PERSON"
            ],
            [
                1509,
                1516,
                "PERSON"
            ],
            [
                1561,
                1567,
                "PERSON"
            ],
            [
                1685,
                1690,
                "PERSON"
            ],
            [
                1707,
                1714,
                "PERSON"
            ],
            [
                1775,
                1784,
                "PERSON"
            ],
            [
                1845,
                1853,
                "PERSON"
            ],
            [
                1928,
                1934,
                "PERSON"
            ],
            [
                1959,
                1965,
                "PERSON"
            ],
            [
                1982,
                1988,
                "PERSON"
            ],
            [
                2042,
                2048,
                "PERSON"
            ],
            [
                2061,
                2067,
                "PERSON"
            ],
            [
                2112,
                2118,
                "PERSON"
            ],
            [
                2240,
                2247,
                "PERSON"
            ],
            [
                2259,
                2266,
                "PERSON"
            ],
            [
                2337,
                2343,
                "PERSON"
            ],
            [
                2579,
                2589,
                "PERSON"
            ],
            [
                2728,
                2735,
                "PERSON"
            ],
            [
                2758,
                2763,
                "PERSON"
            ],
            [
                2885,
                2890,
                "PERSON"
            ],
            [
                2915,
                2922,
                "PERSON"
            ],
            [
                2951,
                2957,
                "PERSON"
            ],
            [
                3029,
                3035,
                "PERSON"
            ],
            [
                3084,
                3090,
                "PERSON"
            ],
            [
                3127,
                3134,
                "PERSON"
            ],
            [
                3138,
                3143,
                "PERSON"
            ],
            [
                3190,
                3198,
                "PERSON"
            ],
            [
                3237,
                3243,
                "PERSON"
            ],
            [
                3253,
                3258,
                "PERSON"
            ],
            [
                3272,
                3281,
                "PERSON"
            ],
            [
                3304,
                3310,
                "PERSON"
            ],
            [
                3453,
                3460,
                "PERSON"
            ],
            [
                3498,
                3504,
                "PERSON"
            ],
            [
                3580,
                3587,
                "PERSON"
            ],
            [
                3602,
                3608,
                "PERSON"
            ],
            [
                3632,
                3639,
                "PERSON"
            ],
            [
                3709,
                3715,
                "PERSON"
            ],
            [
                3803,
                3812,
                "PERSON"
            ],
            [
                3826,
                3833,
                "PERSON"
            ],
            [
                3948,
                3954,
                "PERSON"
            ],
            [
                3965,
                3973,
                "PERSON"
            ],
            [
                4032,
                4039,
                "PERSON"
            ],
            [
                4144,
                4152,
                "PERSON"
            ],
            [
                4260,
                4267,
                "PERSON"
            ],
            [
                4341,
                4349,
                "PERSON"
            ],
            [
                4380,
                4386,
                "PERSON"
            ],
            [
                4418,
                4424,
                "PERSON"
            ],
            [
                4461,
                4468,
                "PERSON"
            ],
            [
                4740,
                4749,
                "PERSON"
            ],
            [
                4816,
                4824,
                "PERSON"
            ],
            [
                4888,
                4895,
                "PERSON"
            ],
            [
                4901,
                4908,
                "PERSON"
            ],
            [
                4923,
                4931,
                "PERSON"
            ],
            [
                4936,
                4941,
                "PERSON"
            ],
            [
                4953,
                4963,
                "PERSON"
            ],
            [
                5041,
                5047,
                "PERSON"
            ],
            [
                5069,
                5075,
                "PERSON"
            ],
            [
                5153,
                5160,
                "PERSON"
            ],
            [
                5175,
                5182,
                "PERSON"
            ],
            [
                5207,
                5214,
                "PERSON"
            ],
            [
                5242,
                5248,
                "PERSON"
            ],
            [
                5326,
                5332,
                "PERSON"
            ],
            [
                5637,
                5644,
                "PERSON"
            ],
            [
                5705,
                5711,
                "PERSON"
            ],
            [
                5736,
                5746,
                "PERSON"
            ],
            [
                5752,
                5763,
                "PERSON"
            ],
            [
                5768,
                5777,
                "PERSON"
            ],
            [
                5807,
                5812,
                "PERSON"
            ],
            [
                5843,
                5852,
                "PERSON"
            ],
            [
                5877,
                5884,
                "PERSON"
            ],
            [
                5906,
                5914,
                "PERSON"
            ],
            [
                5969,
                5975,
                "PERSON"
            ],
            [
                6034,
                6041,
                "PERSON"
            ],
            [
                6245,
                6253,
                "PERSON"
            ],
            [
                6299,
                6308,
                "PERSON"
            ],
            [
                6334,
                6342,
                "PERSON"
            ],
            [
                6365,
                6371,
                "PERSON"
            ],
            [
                6432,
                6439,
                "PERSON"
            ],
            [
                6489,
                6497,
                "PERSON"
            ],
            [
                6505,
                6512,
                "PERSON"
            ],
            [
                6625,
                6632,
                "PERSON"
            ],
            [
                6643,
                6651,
                "PERSON"
            ],
            [
                6835,
                6841,
                "PERSON"
            ],
            [
                6950,
                6957,
                "PERSON"
            ],
            [
                6994,
                7002,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical cente { Marah r east white, tyrone 1211  { Mendy medical cente { Authur r d { Colbie r mrn: 047717361,  { Brenda dob: 7/27/1969, legal sex: m nashville tn 37232 v { Vianney isit  { Tyran date: 12/12 { Louie /2023 12/12/2023 - o { Marli ffice visit in { Jacklynn  vanderb { Landis ilt diabetes and endo { Alma crino { Shelly logy (continued) clinical notes (continued) 70 - 99 mg/dl blood urea nitrogen 8 - 26 mg/dl 36 (h) 28 (h) creatinine level  { Landan 0.72 - 1.2 { Johnmichael 5 mg/dl 3.60 { Ravyn  ( { Dannette h) 3.38 (h) calcium  { Raye level total 8.6 8.8 8.4 - 10.5 mg/dl anio { Emelyn n gap 9 7 patient location p { Shala oc j071 adult hemoglobin a1c level poc 7.9 % egfrcr >=60 ml/min/1.73 m2 19 (l) 21 (l) legend: (h) high (l) low assessment and plan problem list ite { Taelor ms addr { Vivianne essed  { Arvel this visit genitourinary type 2 { Desi  diabetes mellitus with c { Genesis hronic kidney disease, with long-term current use { Virgle  of in { Brynley sulin (cms/hcc) - primary { Petrina  - glycemia not well control { Kedrick led but likely as good as we can get { Albertine  with this gentleman. i don't think he needs less  { Theressa insulin but rather he needs to eat some breakfast every morning. we discussed how to do this. he does not required prescription f { Steele or ensure or for a wheelcha { Dannie ir an { Zipporah d i explained why. i suggested he buy { Sherika  some glucerna or diabet { Finis ic boost and use this for breakfast. the only caveat will be if his phosphorous is too high, he ma { Maksim y n { Rayyan eed a more renal friendly supple { Cassy ment. i urged him to keep his followup with nephrology as scheduled. his foot shows  of swelling but i am concerned { Cristie  about { Rosalva  distal pad. { Rafe  righ { Ubaldo t now he ha { Kaeli s  loss. continue to monitor. u { Sailor rged h { Thora im { Skylah  to walk. relevant medications lantus solostar  { Marino u-100 insulin 100  { Amani unit/ml (3 ml) subcuta { Kalista neous pen other relevant ord { Sol ers pcr chlam { Lew ydia trachomatis/neisseria gonorrhoeae endocrine/metabolic mixed hyperlipidemia - controlled on statin other slurred speech - r/t bells palsy { Michaella  other visit diagnose { Samone s high r { Markeith isk heterose { Cipriano xual behavior - discussed  { Marivel condom { Maybell  use { Stormi . he  { Isamar is aware of h { Raleigh is risk for stis and apparently his partner removed the condom without his consent. will  { Gisel check routine sti screens and  { Jamiah hiv toda { Sueann y. should { Audrea  have repeat hi { Mikki v in another month. relevant orders printed o { Christina n 10/3/24 7:12 am page 1399,vumc adult medical center east white, { Author  tyrone 1211 medical center dr mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn { Conrado  37232 visit date: 12/12/2023 12/12/2023 - office visit in vanderbilt diabetes and endocrinology (continued) clinical notes (cont { Amaia in { Ilse ued) hiv p24 ag and hiv 1/2 ab (completed) pcr chlamydia trachomatis/neisseria gonorrhoea { Christen e (completed) treponemal lgg (completed) hepat { Nour itis c lgg w/rfx pcr (completed) hypertension: cont { Andrae rolled on multidrug reg { Symphony im { Jamell en ckd stageiv: not volume overloaded and does not appear to have  { Tyreek symptomatic uremia. - followup with nephrology orders placed t { Jann his encounter procedures hiv p24 ag and hiv 1/2 ab standing status: futur { Nat e number of occur { Tegan rences: 1 standing expiration date:  { Baileigh 12/12/2024 pcr chlamydia { Caelyn  trachomatis/neisseria gonorrhoeae standing s { Glenn tatus: future num { Novalee be { Salem r of occurrences: 1  { Towanda standing expiratio { Baker n date: 12/12/2024 treponemal igg standing statu { Tayden s: future number of occurrences: 1 standing expiration date: 12/12/2024 hepat { Harleen itis c  { Shaylynn igg w/rfx pcr standing sta { Zaina tus: fut { Shaan ure number of occurrences: 1 standing expiration date: 12/12/2024 pcr chlamydia trachom { Mardell atis/neisse { Sahana ria gonorrhoeae standi { Yanet ng status: future standing expiration date: 12/12/2024 follow up: return in about 4 months (around 4/12/2024) for recheck. patient's medi { Tamira cations, allergies, past medical, surgical, social and f { Kirkland amily histories were reviewed and updated as appropriate. established patient i have spend 27 min in face to face time with the patien { Raekwon t and >50% of this time was spent { Ellena  in counseling on the above stated issues. debra l g { Jiselle reenspan, aprn electronically signed by greenspan, debra l, aprn at 12/13/2023 2:50 pm printed on 10/3/24 7:12 am page 1400",
    {
        "entities": [
            [
                27,
                33,
                "PERSON"
            ],
            [
                62,
                68,
                "PERSON"
            ],
            [
                84,
                91,
                "PERSON"
            ],
            [
                97,
                104,
                "PERSON"
            ],
            [
                125,
                132,
                "PERSON"
            ],
            [
                184,
                192,
                "PERSON"
            ],
            [
                200,
                206,
                "PERSON"
            ],
            [
                220,
                226,
                "PERSON"
            ],
            [
                249,
                255,
                "PERSON"
            ],
            [
                272,
                281,
                "PERSON"
            ],
            [
                292,
                299,
                "PERSON"
            ],
            [
                323,
                328,
                "PERSON"
            ],
            [
                336,
                343,
                "PERSON"
            ],
            [
                468,
                475,
                "PERSON"
            ],
            [
                488,
                500,
                "PERSON"
            ],
            [
                515,
                521,
                "PERSON"
            ],
            [
                526,
                535,
                "PERSON"
            ],
            [
                558,
                563,
                "PERSON"
            ],
            [
                607,
                614,
                "PERSON"
            ],
            [
                645,
                651,
                "PERSON"
            ],
            [
                801,
                808,
                "PERSON"
            ],
            [
                818,
                827,
                "PERSON"
            ],
            [
                836,
                842,
                "PERSON"
            ],
            [
                876,
                881,
                "PERSON"
            ],
            [
                909,
                917,
                "PERSON"
            ],
            [
                969,
                976,
                "PERSON"
            ],
            [
                985,
                993,
                "PERSON"
            ],
            [
                1021,
                1029,
                "PERSON"
            ],
            [
                1060,
                1068,
                "PERSON"
            ],
            [
                1107,
                1117,
                "PERSON"
            ],
            [
                1170,
                1179,
                "PERSON"
            ],
            [
                1311,
                1318,
                "PERSON"
            ],
            [
                1348,
                1355,
                "PERSON"
            ],
            [
                1363,
                1372,
                "PERSON"
            ],
            [
                1412,
                1420,
                "PERSON"
            ],
            [
                1447,
                1453,
                "PERSON"
            ],
            [
                1554,
                1561,
                "PERSON"
            ],
            [
                1567,
                1574,
                "PERSON"
            ],
            [
                1609,
                1615,
                "PERSON"
            ],
            [
                1733,
                1741,
                "PERSON"
            ],
            [
                1750,
                1758,
                "PERSON"
            ],
            [
                1773,
                1778,
                "PERSON"
            ],
            [
                1786,
                1793,
                "PERSON"
            ],
            [
                1807,
                1813,
                "PERSON"
            ],
            [
                1847,
                1854,
                "PERSON"
            ],
            [
                1863,
                1869,
                "PERSON"
            ],
            [
                1874,
                1881,
                "PERSON"
            ],
            [
                1931,
                1938,
                "PERSON"
            ],
            [
                1959,
                1965,
                "PERSON"
            ],
            [
                1990,
                1998,
                "PERSON"
            ],
            [
                2029,
                2033,
                "PERSON"
            ],
            [
                2049,
                2053,
                "PERSON"
            ],
            [
                2197,
                2207,
                "PERSON"
            ],
            [
                2231,
                2238,
                "PERSON"
            ],
            [
                2249,
                2258,
                "PERSON"
            ],
            [
                2273,
                2282,
                "PERSON"
            ],
            [
                2311,
                2319,
                "PERSON"
            ],
            [
                2328,
                2336,
                "PERSON"
            ],
            [
                2343,
                2350,
                "PERSON"
            ],
            [
                2358,
                2365,
                "PERSON"
            ],
            [
                2381,
                2389,
                "PERSON"
            ],
            [
                2481,
                2487,
                "PERSON"
            ],
            [
                2520,
                2527,
                "PERSON"
            ],
            [
                2538,
                2545,
                "PERSON"
            ],
            [
                2557,
                2564,
                "PERSON"
            ],
            [
                2582,
                2588,
                "PERSON"
            ],
            [
                2636,
                2646,
                "PERSON"
            ],
            [
                2714,
                2721,
                "PERSON"
            ],
            [
                2812,
                2820,
                "PERSON"
            ],
            [
                2952,
                2958,
                "PERSON"
            ],
            [
                2963,
                2968,
                "PERSON"
            ],
            [
                3060,
                3069,
                "PERSON"
            ],
            [
                3118,
                3123,
                "PERSON"
            ],
            [
                3177,
                3184,
                "PERSON"
            ],
            [
                3210,
                3219,
                "PERSON"
            ],
            [
                3224,
                3231,
                "PERSON"
            ],
            [
                3300,
                3307,
                "PERSON"
            ],
            [
                3372,
                3377,
                "PERSON"
            ],
            [
                3453,
                3457,
                "PERSON"
            ],
            [
                3477,
                3483,
                "PERSON"
            ],
            [
                3522,
                3531,
                "PERSON"
            ],
            [
                3558,
                3565,
                "PERSON"
            ],
            [
                3613,
                3619,
                "PERSON"
            ],
            [
                3639,
                3647,
                "PERSON"
            ],
            [
                3652,
                3658,
                "PERSON"
            ],
            [
                3681,
                3689,
                "PERSON"
            ],
            [
                3710,
                3716,
                "PERSON"
            ],
            [
                3767,
                3774,
                "PERSON"
            ],
            [
                3854,
                3862,
                "PERSON"
            ],
            [
                3872,
                3881,
                "PERSON"
            ],
            [
                3910,
                3916,
                "PERSON"
            ],
            [
                3927,
                3933,
                "PERSON"
            ],
            [
                4023,
                4031,
                "PERSON"
            ],
            [
                4045,
                4052,
                "PERSON"
            ],
            [
                4077,
                4083,
                "PERSON"
            ],
            [
                4223,
                4230,
                "PERSON"
            ],
            [
                4289,
                4298,
                "PERSON"
            ],
            [
                4435,
                4443,
                "PERSON"
            ],
            [
                4479,
                4486,
                "PERSON"
            ],
            [
                4541,
                4549,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone { Tarek  1211  { Jetta me { Marya dical center dr. mrn: 0 { Nyomi 47717361, dob: 7/27/196 { Zana 9, legal s { Hakim ex: m nashville tn 37232-0004 adm: 6/2/2023, d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vanderbilt university adult hospital (continued) laboratory reports (continued) 23-154-000589  { Yaseen urine - mitchell, jordan, rn 06/03/23 0435 ur sodium lvl resulted: 06/03/23 0543, result status: final result ordering provider: howard, william timothy, pa 06/02/23 2320 order status { Ceola : completed filed by: interface, lab results in 06/03/23 0543 collected { Sunday  by: mitchell, jordan, rn 06/03/23 0435 resulting lab: vumc cerner lab components component value refere { Lovina nce range flag lab urine sodium 88 mmol/l cerner comment: reference values have not been established for random urine collections. testing performed by lab - { Takisha  abbreviation name director address valid date range 123 cerner vumc cerner lab adam seegmiller; 4605 tvc vumc 11/22/21 1014 - present jennifer b. 1301 medi { Garnet cal center gordetsky drive nashville tn 37232- 5310 ur { Kenya  creatinine lvl (final result) electronically signed { Jaymee  by: howa { Garner rd, william timothy, pa on 06/02/23 2320 status: completed ordering user: howard, william ti { Lawrence mothy, pa 06/02/23 2320 ordering provider: howard, wil { Melvin liam { Bently  timothy, { Latina  pa authorized  { Corrinne by: howard, willi { Arie am timothy, pa ordering mo { Abbigale de: standard frequency: routine once 06/02/23 2319 - 1 { Augustina  occurrence class: unit collect quantity: 1 lab status: final result instance  { Mattison released by: howard, william timothy, pa (auto-rele { Kerrigan ased) 6/2/2023 1 { Neda 1:20 pm specimen infor { Christine mation { Emmer  id type source collected by 23-154-000589 urine - mitchell, jordan, rn { Damani  06/03/23 0435 ur creatinine lvl resulted: 06/03/23 0543, result status: final r { Josias esult ordering provider: h { Richardo oward, william timothy, pa 06/02/23 2320 order status: completed filed by: interface, lab results in 06/03/23 0543 collected by: mitchell, jordan, rn 06/03/23 0435 resulting lab: vumc c { Delena erner lab components component value reference range flag lab uri { Nicol ne creatini { Sebrina ne 51 40 200 mg/dl - cerner testing performed by lab - abbreviation name director address valid date range 123 cern { Corissa er vumc cerner l { Vic ab { Angely  adam seegmiller; 4605 tvc  { Alen vumc 11/2 { Minh 2/21 1014 - present jennifer b. 1301 medical  { Lanell center go { Naila rdet { Rebekka sky drive nashvil { Dynasty le tn 37232- 5310 ur urea nitrogen { Ailene  (final result) electronica { Harmonie lly signed by: howard, william timothy, pa on 06/02/23 2320 status: completed ordering user: howard, william timothy, pa 06/02/23 2320 ordering provider: howard, william timoth { Michelina y, pa printed o { Salem n 10/3/24 7:13 am page 2255,vumc adult hospital white, tyrone 1211 medical c { Inell enter dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nash { Joline ville tn 37232-0004 adm { Ever : 6/2/2023, d/c: 6/3/2023 06/02/2023 - { Omega  ed t { Antwone o hosp-admission (discharged) i { Johathan n vanderbilt universit { Kelcey y adult hospi { Nautica tal (continued) laboratory reports (cont { Talisa inued) authorized by: howard, william timothy, pa ordering mode: standard fre { Demetric quency: routine once 06/02/23 2319 - 1 occurrence class: unit collect quanti { Flint ty: 1 lab status: final result instance relea { Jacen sed by: howard, william timothy, pa (auto-released) 6/2/2023 11:20  { Saad pm specimen in { Heide formation id type source collected by 23-154-00058 { Lanora 9  { Barrie ur { Truett ine - mitchell, jordan, rn 06/03/23 0435 ur u { Nubia rea nitrogen (abnormal) resulted: 06/03/23 0543, result status: final result orderi { Noam ng provider: howard, will { Tyce i { Servando am timothy, pa 06/02/23 2320 order status: completed filed by: interface, lab results in 06/03/23 0543 collected by: mitchell, jordan, rn 06/03/23 0435 resulting  { Willette lab: vumc cerner  { Macayla lab components component value reference range flag lab urine bun 286 300 - 1,800 mg/dl cerner testing per { Kayleb formed by lab - abbreviation name director a { Nicole ddress valid date range 123 cerner vumc { Wendel  cerner { Ileen  lab adam { Zakaria  seegmiller; 4605 tvc vumc 11/22/21 1014 - present jennifer b. 13 { Brinda 01 medical center { Patrina  gordetsky drive nashville tn 37232- 5310  { Jarren urinalysis w/micro&rfx culture (final result) electronically signed by: howard, william timothy, pa on 06/02/23 2344 status: completed ordering user: howard, william { Shelba  timothy, pa 06/02/23 2344 ordering provider: howard, william timothy, pa auth { Maple orized b { Noah y: ho { Diandra ward, william timothy, pa ordering mode: standard frequency: routine once 06/02/23 2345 - 1  { Kacey occurrence class: unit collect quantity: 1 lab status: final result instanc { Rose e released by: howar { January d, william timot { Worth hy, pa (auto-released) 6/2/2023 11:44 pm questionna { Riana ire question answer catheterized  informa { Jashawn tion i { Lauro d type source collected by 2 { Tatiyana 3-154-000589 urine - mit { Raj chell, jordan, rn 06/03/23 0435 urinalysis w/micro&rfx culture (abnormal) resulted: 06/03/23 0537, result status: final result ordering provider { Sanjana : howa { Durrell rd, william timothy, pa 06/02/23 2344 order status: completed filed by: interface, lab resu { Wynn lts in 06/03/23 0537 collected by: mitchell, jordan, rn 06/03/23 0435 resulting lab: vumc cerner lab components component value reference range flag lab urine color colorless -  { Lavar - cerner urine appearance clear - - cerner urine specific gravity 1.009 1.015 - 1.025 lv cerner urine ph 6.5 5.0 6 { Abbi .5 - { Douglas  cerner urine glucose negative negative mg/dl - cerner urine prote { Anai in 70 negati { Henery ve mg/dl  { Karyl a ! cerner printed on 10/3/24 7:13 am page 2256",
    {
        "entities": [
            [
                36,
                42,
                "PERSON"
            ],
            [
                51,
                57,
                "PERSON"
            ],
            [
                62,
                68,
                "PERSON"
            ],
            [
                94,
                100,
                "PERSON"
            ],
            [
                126,
                131,
                "PERSON"
            ],
            [
                144,
                150,
                "PERSON"
            ],
            [
                356,
                363,
                "PERSON"
            ],
            [
                549,
                555,
                "PERSON"
            ],
            [
                629,
                636,
                "PERSON"
            ],
            [
                743,
                750,
                "PERSON"
            ],
            [
                910,
                918,
                "PERSON"
            ],
            [
                1077,
                1084,
                "PERSON"
            ],
            [
                1141,
                1147,
                "PERSON"
            ],
            [
                1202,
                1209,
                "PERSON"
            ],
            [
                1221,
                1228,
                "PERSON"
            ],
            [
                1323,
                1332,
                "PERSON"
            ],
            [
                1389,
                1396,
                "PERSON"
            ],
            [
                1403,
                1410,
                "PERSON"
            ],
            [
                1422,
                1429,
                "PERSON"
            ],
            [
                1447,
                1456,
                "PERSON"
            ],
            [
                1476,
                1481,
                "PERSON"
            ],
            [
                1510,
                1519,
                "PERSON"
            ],
            [
                1576,
                1586,
                "PERSON"
            ],
            [
                1667,
                1676,
                "PERSON"
            ],
            [
                1730,
                1739,
                "PERSON"
            ],
            [
                1758,
                1763,
                "PERSON"
            ],
            [
                1788,
                1798,
                "PERSON"
            ],
            [
                1807,
                1813,
                "PERSON"
            ],
            [
                1887,
                1894,
                "PERSON"
            ],
            [
                1977,
                1984,
                "PERSON"
            ],
            [
                2013,
                2022,
                "PERSON"
            ],
            [
                2210,
                2217,
                "PERSON"
            ],
            [
                2285,
                2291,
                "PERSON"
            ],
            [
                2305,
                2313,
                "PERSON"
            ],
            [
                2431,
                2439,
                "PERSON"
            ],
            [
                2458,
                2462,
                "PERSON"
            ],
            [
                2467,
                2474,
                "PERSON"
            ],
            [
                2504,
                2509,
                "PERSON"
            ],
            [
                2521,
                2526,
                "PERSON"
            ],
            [
                2574,
                2581,
                "PERSON"
            ],
            [
                2593,
                2599,
                "PERSON"
            ],
            [
                2606,
                2614,
                "PERSON"
            ],
            [
                2634,
                2642,
                "PERSON"
            ],
            [
                2679,
                2686,
                "PERSON"
            ],
            [
                2716,
                2725,
                "PERSON"
            ],
            [
                2904,
                2914,
                "PERSON"
            ],
            [
                2932,
                2938,
                "PERSON"
            ],
            [
                3017,
                3023,
                "PERSON"
            ],
            [
                3085,
                3092,
                "PERSON"
            ],
            [
                3118,
                3123,
                "PERSON"
            ],
            [
                3164,
                3170,
                "PERSON"
            ],
            [
                3178,
                3186,
                "PERSON"
            ],
            [
                3220,
                3229,
                "PERSON"
            ],
            [
                3254,
                3261,
                "PERSON"
            ],
            [
                3277,
                3285,
                "PERSON"
            ],
            [
                3328,
                3335,
                "PERSON"
            ],
            [
                3415,
                3424,
                "PERSON"
            ],
            [
                3503,
                3509,
                "PERSON"
            ],
            [
                3557,
                3563,
                "PERSON"
            ],
            [
                3633,
                3638,
                "PERSON"
            ],
            [
                3655,
                3661,
                "PERSON"
            ],
            [
                3714,
                3721,
                "PERSON"
            ],
            [
                3726,
                3733,
                "PERSON"
            ],
            [
                3738,
                3745,
                "PERSON"
            ],
            [
                3793,
                3799,
                "PERSON"
            ],
            [
                3885,
                3890,
                "PERSON"
            ],
            [
                3918,
                3923,
                "PERSON"
            ],
            [
                3927,
                3936,
                "PERSON"
            ],
            [
                4101,
                4110,
                "PERSON"
            ],
            [
                4130,
                4138,
                "PERSON"
            ],
            [
                4247,
                4254,
                "PERSON"
            ],
            [
                4301,
                4308,
                "PERSON"
            ],
            [
                4350,
                4357,
                "PERSON"
            ],
            [
                4367,
                4373,
                "PERSON"
            ],
            [
                4385,
                4393,
                "PERSON"
            ],
            [
                4461,
                4468,
                "PERSON"
            ],
            [
                4488,
                4496,
                "PERSON"
            ],
            [
                4541,
                4548,
                "PERSON"
            ],
            [
                4716,
                4723,
                "PERSON"
            ],
            [
                4804,
                4810,
                "PERSON"
            ],
            [
                4821,
                4826,
                "PERSON"
            ],
            [
                4834,
                4842,
                "PERSON"
            ],
            [
                4937,
                4943,
                "PERSON"
            ],
            [
                5021,
                5026,
                "PERSON"
            ],
            [
                5049,
                5057,
                "PERSON"
            ],
            [
                5076,
                5082,
                "PERSON"
            ],
            [
                5136,
                5142,
                "PERSON"
            ],
            [
                5186,
                5194,
                "PERSON"
            ],
            [
                5203,
                5209,
                "PERSON"
            ],
            [
                5240,
                5249,
                "PERSON"
            ],
            [
                5276,
                5280,
                "PERSON"
            ],
            [
                5427,
                5435,
                "PERSON"
            ],
            [
                5444,
                5452,
                "PERSON"
            ],
            [
                5546,
                5551,
                "PERSON"
            ],
            [
                5731,
                5737,
                "PERSON"
            ],
            [
                5854,
                5859,
                "PERSON"
            ],
            [
                5866,
                5874,
                "PERSON"
            ],
            [
                5943,
                5948,
                "PERSON"
            ],
            [
                5963,
                5970,
                "PERSON"
            ],
            [
                5982,
                5988,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adu { Arsenio lt hospital white, tyrone 1211 medical center dr. mrn: 04771 { Asma 7361, dob: 7/2 { Jenee 7 { Shamya /1969, lega { Carsten l { Addalyn  s { Contessa ex: m nashville tn 37232-0004 visit date: 10/19/2023 10/19/2023 - communication in vanderbilt pharmacy retail services (continued) clini { Vella cal notes { Foy  (continued) fer { Aislynn guson { Brennan , sherri l, lpn at 10/20/20 { Carlos 23 1419 author: ferguson, sherri l, lpn service: au { Kaleena thor type: licensed nurse filed: 10/20/2023 2 { Shaylyn :19 pm encounter date: 10/19/2023 status: sig { Braedyn ned editor: ferguson,  { Khaled sherri l, { Tanaya  l { Vernard pn (licensed nurse) refill  { Celesta forwarded to dr de witte basket electronically signed by ferguson, sherri l, lpn at 10/20/2023 2:19 pm de witte, anton jordan, md at 10/20/2023 1603 author: de witte, anton jordan, md service: author type: resident physician filed: 10/ { Marylynn 20/2023 4:03 pm encounter date: 1 { Shanya 0/19/2023 sta { Damond tus: signed editor: de witte, anton jordan, md (resident physician) order signed. thanks electronically signed by de witte, a { Taylin nton jordan, md at 10/20/2023 4:03 pm other orders medications insu { Amaria lin g { Euna largine (u-100) 100 unit/ { Takia ml subcutaneous so { Elza lution (discontinued) electronically signed by: de witte, anton jordan, md on 10/20/2 { Jamaya 3 1603 status: discontinued ordering user: { Jenica  de witte, anton jordan, { Kyli  md 10/20/23 160 { Christain 3 ordering { Gorden  provid { Lelah er: de witte, anton jor { Allissa dan, md authorized by: de witte, anton jordan, md ordering mode: st { Irena andard f { Salman requency: routine dail { Carlota y 10/20/23 - 90 d { Markayla ays class: fax released by: de witte, anton jordan, md 10/20/23 1603 discontinued by: prasad, sonika h, r { Xitlali n 02/22/24 1323 [cleanup(notav { Charly s)] medication comments: plea { Mellie se only fill for 30 days at  { September a time. okay to substitute lantus, { Gabriele  semglee, or biosimilar insul { Yana in glarg { Babyboy ine pr { Vanita oduct based on patient's insurance coverage reordered f { Adler rom: insulin glargine (u-100) 100 unit/ml subcutaneous solution ordering & authorizing prov { Theophilus ider audit trail date/time ordering provider authorizing { Yasir  provider user 10/20/23 1603  { Katharina de witte, anto { Arnaldo n jorda { Danielle n, md de witte, anton jordan, md de witte, anton j { Tyasia ordan, md 10/20/23 1231 lipp { Mychal ard, giles a, aprn lippard, giles a, { Kezia  aprn maynard, megan r, lpn printed on 10/3/24 7: { Sacha 13 am page 1585,vumc adult hospital whit { Seraphina e, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m { Theola  n { Aris ashville tn 3 { Che 7232-0004 visit date: 10/19/2023 10/19/ { Muhammed 2023 - communication in vanderbilt  { Emberly pharmacy retail services face { Janise shee { Joscelyn t report patient demographics patient name mrn legal dob address phone white, tyrone 0477173 sex 7/27/1969 apt 705 615-260-2291 (home) 61 m 1101 edgehill ave 615-260-2291 (mobile) n { Jacorey ashville tn 37203 *preferred* hospital account not on file admission  { Miquel information current i { Glynis nformation attending provider admitting provider a { Katheryne dmis { Bradon sion type admission st { Calleigh atus unknown status adm { Donielle ission date/time discharge date/time hospital s { Cyle ervice auth/cert status hospital  { Dakari are { Mahalia a unit room/bed referring { Arlyn  provider 10/19/2023 - communication in vanderbilt pharmac { Candido y retail services (continued)  { Justyce reaso { Frazier n  { Kevyn for visit chief complaint [last edited by harper, byron  { Narciso e { Tyrek , cpht on 10/19/2023 0957] prior author { Aleesha ization, onset date 10/19/2023 visit information provider information encounter provider de witte, ant { Alessa on jordan, md department name address vanderbilt { Flo  pharmacy retail services tn medication list medication list 1 this report  { Izzabella is  { Melyssa for documentation purposes only. the patient should no { Jermain t follow  { Anayeli medication instructio { Falon ns within. for a { Tiny ccurate instructions regarding medications, th { Brooke e patient { Arlyn  should instead consult their physician or after visit summary. active at the end of visit medications last reviewed by dekorte, da { Dmitri vita on 10/17/2023 142 { Jaedon 3 docusa { Mckay te sodium 100 mg { Saoirse  capsule (colace) discontinued by: de witte, anton jordan, m { Verdell d  { Zebulon discont { Amerie inued on: 12/6/2023 reason for discontinuation: cleanup(notavs) instructions: take one tablet tid prn constipation authorized by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 end date: 12/6/2023 quantity: 60 capsule refill:  remaining printed on 10/3/ { Glennis 24 7:13 am page 1586",
    {
        "entities": [
            [
                11,
                19,
                "PERSON"
            ],
            [
                82,
                87,
                "PERSON"
            ],
            [
                104,
                110,
                "PERSON"
            ],
            [
                114,
                121,
                "PERSON"
            ],
            [
                135,
                143,
                "PERSON"
            ],
            [
                147,
                155,
                "PERSON"
            ],
            [
                160,
                169,
                "PERSON"
            ],
            [
                308,
                314,
                "PERSON"
            ],
            [
                326,
                330,
                "PERSON"
            ],
            [
                349,
                357,
                "PERSON"
            ],
            [
                365,
                373,
                "PERSON"
            ],
            [
                403,
                410,
                "PERSON"
            ],
            [
                464,
                472,
                "PERSON"
            ],
            [
                520,
                528,
                "PERSON"
            ],
            [
                576,
                584,
                "PERSON"
            ],
            [
                609,
                616,
                "PERSON"
            ],
            [
                628,
                635,
                "PERSON"
            ],
            [
                640,
                648,
                "PERSON"
            ],
            [
                678,
                686,
                "PERSON"
            ],
            [
                924,
                933,
                "PERSON"
            ],
            [
                969,
                976,
                "PERSON"
            ],
            [
                992,
                999,
                "PERSON"
            ],
            [
                1127,
                1134,
                "PERSON"
            ],
            [
                1204,
                1211,
                "PERSON"
            ],
            [
                1219,
                1224,
                "PERSON"
            ],
            [
                1252,
                1258,
                "PERSON"
            ],
            [
                1279,
                1284,
                "PERSON"
            ],
            [
                1372,
                1379,
                "PERSON"
            ],
            [
                1424,
                1431,
                "PERSON"
            ],
            [
                1458,
                1463,
                "PERSON"
            ],
            [
                1482,
                1492,
                "PERSON"
            ],
            [
                1505,
                1512,
                "PERSON"
            ],
            [
                1522,
                1528,
                "PERSON"
            ],
            [
                1554,
                1562,
                "PERSON"
            ],
            [
                1632,
                1638,
                "PERSON"
            ],
            [
                1649,
                1656,
                "PERSON"
            ],
            [
                1681,
                1689,
                "PERSON"
            ],
            [
                1709,
                1718,
                "PERSON"
            ],
            [
                1826,
                1834,
                "PERSON"
            ],
            [
                1867,
                1874,
                "PERSON"
            ],
            [
                1906,
                1913,
                "PERSON"
            ],
            [
                1944,
                1954,
                "PERSON"
            ],
            [
                1991,
                2000,
                "PERSON"
            ],
            [
                2032,
                2037,
                "PERSON"
            ],
            [
                2048,
                2056,
                "PERSON"
            ],
            [
                2065,
                2072,
                "PERSON"
            ],
            [
                2130,
                2136,
                "PERSON"
            ],
            [
                2230,
                2241,
                "PERSON"
            ],
            [
                2300,
                2306,
                "PERSON"
            ],
            [
                2338,
                2348,
                "PERSON"
            ],
            [
                2365,
                2373,
                "PERSON"
            ],
            [
                2383,
                2392,
                "PERSON"
            ],
            [
                2445,
                2452,
                "PERSON"
            ],
            [
                2483,
                2490,
                "PERSON"
            ],
            [
                2529,
                2535,
                "PERSON"
            ],
            [
                2587,
                2593,
                "PERSON"
            ],
            [
                2636,
                2646,
                "PERSON"
            ],
            [
                2727,
                2734,
                "PERSON"
            ],
            [
                2739,
                2744,
                "PERSON"
            ],
            [
                2760,
                2764,
                "PERSON"
            ],
            [
                2806,
                2815,
                "PERSON"
            ],
            [
                2853,
                2861,
                "PERSON"
            ],
            [
                2893,
                2900,
                "PERSON"
            ],
            [
                2907,
                2916,
                "PERSON"
            ],
            [
                3100,
                3108,
                "PERSON"
            ],
            [
                3180,
                3187,
                "PERSON"
            ],
            [
                3211,
                3218,
                "PERSON"
            ],
            [
                3271,
                3281,
                "PERSON"
            ],
            [
                3288,
                3295,
                "PERSON"
            ],
            [
                3320,
                3329,
                "PERSON"
            ],
            [
                3355,
                3364,
                "PERSON"
            ],
            [
                3414,
                3419,
                "PERSON"
            ],
            [
                3455,
                3462,
                "PERSON"
            ],
            [
                3468,
                3476,
                "PERSON"
            ],
            [
                3504,
                3510,
                "PERSON"
            ],
            [
                3571,
                3579,
                "PERSON"
            ],
            [
                3612,
                3620,
                "PERSON"
            ],
            [
                3628,
                3636,
                "PERSON"
            ],
            [
                3641,
                3647,
                "PERSON"
            ],
            [
                3706,
                3714,
                "PERSON"
            ],
            [
                3718,
                3724,
                "PERSON"
            ],
            [
                3766,
                3774,
                "PERSON"
            ],
            [
                3879,
                3886,
                "PERSON"
            ],
            [
                3937,
                3941,
                "PERSON"
            ],
            [
                4019,
                4029,
                "PERSON"
            ],
            [
                4035,
                4043,
                "PERSON"
            ],
            [
                4100,
                4108,
                "PERSON"
            ],
            [
                4120,
                4128,
                "PERSON"
            ],
            [
                4152,
                4158,
                "PERSON"
            ],
            [
                4177,
                4182,
                "PERSON"
            ],
            [
                4231,
                4238,
                "PERSON"
            ],
            [
                4250,
                4256,
                "PERSON"
            ],
            [
                4390,
                4397,
                "PERSON"
            ],
            [
                4422,
                4429,
                "PERSON"
            ],
            [
                4440,
                4446,
                "PERSON"
            ],
            [
                4465,
                4473,
                "PERSON"
            ],
            [
                4536,
                4544,
                "PERSON"
            ],
            [
                4549,
                4557,
                "PERSON"
            ],
            [
                4567,
                4574,
                "PERSON"
            ],
            [
                4852,
                4860,
                "PERSON"
            ]
        ]
    }
),(
    "vu { Sharie mc adult { Zoraida  villag { Neftali e at vanderbilt white, tyrone 1500 21st ave s 2nd fl, 2500 mrn: 047717361, dob: 7/27/1969, legal se { Antonio x: m village at vanderbilt adm: 5/29/2024, d/c: 5/29/2024 nashville tn 37212-3160 05/29/2024 - xr general imaging in vumc x-ray village at vanderbilt ( { Detra continued) medi { Faviola cation list (continued) montelukast 10 mg tablet (singulair) instructions: take 1 tablet (10 mg total) by mouth every evening. authorized by: lipp { Antonino ard, giles a, aprn ordered on: 8/8/2023 s { Skip tar { Kinlee t date: 8/8/2023 quantity: 90 tablet refill: 3 refills by { Canon  8/7/2024 cyclobenzaprine  { Charlsie 5 mg tablet (flexeril) [r { Neriah econciled by maples, chantis on 9/5/2023 1522] instructions: take 1 tablet (5 mg total) by mouth every 8 hours as needed. entered by: maples, chantis entered on: 9/5/2023 start date: 7/22/2023 triamcinolone acetonide 55 mcg nasal spray aerosol (nasacort) discontinue { Darel d by: gingrow, barbara, lpn  { Shaquan discontinued on: 9/16/2024 reason for discontinuation: duplicate order in { Tiago structions: administer 2 sprays (110 mcg total) into each nostril 2 times a day. authorized by: greenspan, debra l,  { Cinnamon aprn ord { Orland ered on: 9/5/2023 start date: 9/5/2023 end date: 9/16/2024 quantity: 16.5 g refill: 11 refills by 9/4/2024 losartan 25 mg tablet (coz { Neely aa { Shalyn r) discon { Kyana tinued by: kovtun, rom { Monae an,  { Saray md discontinued on: 8/1/2024 reason for discontinuation: stop (cancelrx, on { Jahir  avs) instructions: take 1 tablet (2 { Naim 5 mg total) by mouth daily. author { Anaiah ized by: de witte, anton jordan, md ordered on: 9/7/2023 star { Lillyann t date: 9/7/2023 end date: 8/1/2024 quantity: 90 tablet r { Zia efill: 3 refills by 9/6/2024 capsaicin 0.1 % topica { Filip l cream discontinued by: gingrow, barbara { Elenor , lpn disco { Eudora ntinued o { Aysia n: 9/16/2024 reason for disco { Floretta ntinuation: duplicate order instructions: apply 1 applicat { Xena ion topically daily for 90 days. authorized by: de witte, anton jord { Wayde an, md ordered on: 9/7/2023 start date: 9/7/2023 end date: 9/16/2024 action: patient not taking quantity:  { Maryelizabeth 42.5 g refill:  remaining aspirin 81 mg ablet,delayed release instructions: take 1 tablet (81 mg t { Palmer otal) by mouth da { Benjaman ily. autho { Jerrick rized by: de witte, anton jordan, md orde { Tyre red on: 9/7/2023 start date: 9/ { Tristyn 7/2023 quantity: 90 tablet refill: 3 ref { Arnita ills by 9/ { Merrill 6/2024 cetirizine 10 mg tablet (zyrtec) instructions: t { Dream ake 1 tablet (10 mg total) by mouth once a day as needed for allergies. authorized by: de witte, anton jordan, { Nerissa  md ordered on: 10/4/2023 start date: 10/4/2023 quantity: 3 { Ambria 0 tablet refill: 9  { Darcey refills by 10/3/2024 lidocai { Shaniece ne 5 % topical patch (lidoderm) instructions: apply 1 patch  { Kiefer topically daily. apply to painful area 12 ho { Cordie urs per day, remove for 12 hours. authorized by: de witte, anton jordan, md ordered on: 10/17/2023 start date: 10/17/2023 end date: 10/16/2024 quantity: 30 patch refill: 11 refills by 10/16/2024 diclofenac 1 % topical gel discontinued by: cone, brittany discontinued on: 6/27/2 { Leisha 024 re { Kainoa ason for disconti { Leyton nua { Marykate tion: therapy completed ( { Acie cancelrx) instructions: apply 2 g topically 4 times a day for 30 days. authorized by: de witte, anton jordan, md ordered on: 1 { Eilene 0/17/2023 printed on 10/3/24 7:12  { Exie am page 897,vumc adult village at vanderbilt w { Gwyn hite, tyrone 1500 21st ave s 2nd fl, 2500 mrn: 047717361, dob: 7/27/1969, legal  { Keosha sex: m village at vanderbilt adm: 5/29/2024, d/c: 5/29/2024 nashville tn 37212-3160 05/29/2024 - xr general imaging in vumc x-ray village at vanderbilt (continued) medication list (contin { Milla ued) start date: 10/17/2023 end  { Thais date:  { Rohit 6/27/2024 action: patient { Annaleigh  not taking quantity: 100 g refill:  remaining azelastine 137  { Letisha mcg (0.1 %) nasal spray aerosol (astelin) instructions: administer 1 spray into each nostril 2 times a day as needed for rhinitis. use in each nostril as directed  { Luetta authorized by: de witte, { Antoni  anton jordan, md ordered on: 10/17/2023 start date: 10/17/2023 quantity: 30 ml refill:  remaining fluticason { Arnie e propionate 50 mcg/actuation nasal spray,suspension (flo { Krishna nase) discontinued by: m { Rayanna ickey, lisa, lpn discontin { Sunni ued on: 8/6/2024 reaso { Durell n for { Gentry  discontinuation: reorder instructions: administer 2 sprays into each nostril 2 times a day. authorized by:  { Syreeta virk, zain m, md ordered on: 11/17/2023 start date: 11/17/2023 quantity: 16 g refill: 2 refills by  { Abran 11/16/2024 loperamide 2 mg cap { Erie sule (imodium) discontinued by: cone, britt { Felisa any discontinued on: 6/27/2024 reason for discontinuation: therapy completed (cancelrx) instructions: take 1 c { Latarsha apsule (2 mg total) by mouth 3 times a day as nee { Cleophus ded for diarrhea for up to { Ewell  10 days. autho { Consuela rized by: de witte, anton jordan, md ordered on: 1/23/2024 start date: 1/23/2024 action: patient not t { Gianni aking quantity: 30 capsule refill:  remaining ipratropium bromide 42 mcg (0.06 nasal spray (atrovent) discontinued by: chakravarthy, rohini, md discontinued o { Kailah n { Lakiesha : 8/6/2024 { Jori  reason for discontinuation: reorder instr { Josalyn uctions: administer 1 spray into each nostril 4 times a day. start with 1 spray once daily for one week, then increase to 2 sprays once a da { Kellyn y for 2 weeks, can increase to 3-4 sprays a day authorized by: rebula, emily rose kueser, pa-c ordered on: 2/28/2024 start date: 2/28/2024 quantity: 15 ml refill: 12 refills by 2/27/2025 lantus solostar u-100 insulin 100 unit/ml (3 ml) subcutaneous pen (insulin glargine) discontinued by: greenspan, debra l, aprn discontinued on: 7/2/2024 instructions: inject 8 units under the skin 2 times a day. authorized by: greenspan, d { Aharon ebra l, aprn ordered on: 4/17/2024 start date: 4/17/2024 quantity: 15 ml refill:  { Dionicio 3 refills by 4/17/ { Remi 2025 nifedipine e { Corrin r 30 mg tablet,ex { Levar tended release (adalat cc) instructions: take 2 tablets (60 mg total) by mouth { Jina  daily. authorized by: de witte, anton jordan, md ordered on: 4/23/2024 start date: 4/23/2024 quantity: 180 tablet refill: 3 { Darris  refills by 4/23/2025 gabapentin 300 mg capsule (neurontin) discontinued by: { Janea  chakravarthy, rohini, md discontinued on: 8/6/2024 reason for discontinuation: reorder { Trula  instructions: take 2 capsules (600 mg total) by mouth daily. authorized by: de witte, an { Alphonzo ton jord { Jovie an, md ordered on: 4/ { Krisha 25/2024 start date: 4/25/202 { Konstantinos 4 quantity: 180 capsule refill:  remaining stopped in visit none printed { Aniston  on { Zayla  10/3/24 7:12 am page 898",
    {
        "entities": [
            [
                5,
                12,
                "PERSON"
            ],
            [
                23,
                31,
                "PERSON"
            ],
            [
                41,
                49,
                "PERSON"
            ],
            [
                151,
                159,
                "PERSON"
            ],
            [
                313,
                319,
                "PERSON"
            ],
            [
                337,
                345,
                "PERSON"
            ],
            [
                494,
                503,
                "PERSON"
            ],
            [
                547,
                552,
                "PERSON"
            ],
            [
                558,
                565,
                "PERSON"
            ],
            [
                625,
                631,
                "PERSON"
            ],
            [
                660,
                669,
                "PERSON"
            ],
            [
                697,
                704,
                "PERSON"
            ],
            [
                973,
                979,
                "PERSON"
            ],
            [
                1010,
                1018,
                "PERSON"
            ],
            [
                1094,
                1100,
                "PERSON"
            ],
            [
                1219,
                1228,
                "PERSON"
            ],
            [
                1239,
                1246,
                "PERSON"
            ],
            [
                1382,
                1388,
                "PERSON"
            ],
            [
                1393,
                1400,
                "PERSON"
            ],
            [
                1412,
                1418,
                "PERSON"
            ],
            [
                1443,
                1449,
                "PERSON"
            ],
            [
                1456,
                1462,
                "PERSON"
            ],
            [
                1540,
                1546,
                "PERSON"
            ],
            [
                1585,
                1590,
                "PERSON"
            ],
            [
                1627,
                1634,
                "PERSON"
            ],
            [
                1698,
                1707,
                "PERSON"
            ],
            [
                1767,
                1771,
                "PERSON"
            ],
            [
                1825,
                1831,
                "PERSON"
            ],
            [
                1875,
                1882,
                "PERSON"
            ],
            [
                1896,
                1903,
                "PERSON"
            ],
            [
                1915,
                1921,
                "PERSON"
            ],
            [
                1953,
                1962,
                "PERSON"
            ],
            [
                2023,
                2028,
                "PERSON"
            ],
            [
                2099,
                2105,
                "PERSON"
            ],
            [
                2214,
                2228,
                "PERSON"
            ],
            [
                2329,
                2336,
                "PERSON"
            ],
            [
                2356,
                2365,
                "PERSON"
            ],
            [
                2378,
                2386,
                "PERSON"
            ],
            [
                2430,
                2435,
                "PERSON"
            ],
            [
                2469,
                2477,
                "PERSON"
            ],
            [
                2520,
                2527,
                "PERSON"
            ],
            [
                2540,
                2548,
                "PERSON"
            ],
            [
                2606,
                2612,
                "PERSON"
            ],
            [
                2725,
                2733,
                "PERSON"
            ],
            [
                2795,
                2802,
                "PERSON"
            ],
            [
                2824,
                2831,
                "PERSON"
            ],
            [
                2862,
                2871,
                "PERSON"
            ],
            [
                2934,
                2941,
                "PERSON"
            ],
            [
                2988,
                2995,
                "PERSON"
            ],
            [
                3275,
                3282,
                "PERSON"
            ],
            [
                3291,
                3298,
                "PERSON"
            ],
            [
                3318,
                3325,
                "PERSON"
            ],
            [
                3331,
                3340,
                "PERSON"
            ],
            [
                3368,
                3373,
                "PERSON"
            ],
            [
                3502,
                3509,
                "PERSON"
            ],
            [
                3546,
                3551,
                "PERSON"
            ],
            [
                3600,
                3605,
                "PERSON"
            ],
            [
                3688,
                3695,
                "PERSON"
            ],
            [
                3885,
                3891,
                "PERSON"
            ],
            [
                3926,
                3932,
                "PERSON"
            ],
            [
                3941,
                3947,
                "PERSON"
            ],
            [
                3975,
                3985,
                "PERSON"
            ],
            [
                4050,
                4058,
                "PERSON"
            ],
            [
                4224,
                4231,
                "PERSON"
            ],
            [
                4258,
                4265,
                "PERSON"
            ],
            [
                4377,
                4383,
                "PERSON"
            ],
            [
                4443,
                4451,
                "PERSON"
            ],
            [
                4478,
                4486,
                "PERSON"
            ],
            [
                4515,
                4521,
                "PERSON"
            ],
            [
                4546,
                4553,
                "PERSON"
            ],
            [
                4561,
                4568,
                "PERSON"
            ],
            [
                4679,
                4687,
                "PERSON"
            ],
            [
                4789,
                4795,
                "PERSON"
            ],
            [
                4828,
                4833,
                "PERSON"
            ],
            [
                4879,
                4886,
                "PERSON"
            ],
            [
                4999,
                5008,
                "PERSON"
            ],
            [
                5060,
                5069,
                "PERSON"
            ],
            [
                5098,
                5104,
                "PERSON"
            ],
            [
                5122,
                5131,
                "PERSON"
            ],
            [
                5236,
                5243,
                "PERSON"
            ],
            [
                5404,
                5411,
                "PERSON"
            ],
            [
                5415,
                5424,
                "PERSON"
            ],
            [
                5437,
                5442,
                "PERSON"
            ],
            [
                5487,
                5495,
                "PERSON"
            ],
            [
                5638,
                5645,
                "PERSON"
            ],
            [
                6074,
                6081,
                "PERSON"
            ],
            [
                6165,
                6174,
                "PERSON"
            ],
            [
                6195,
                6200,
                "PERSON"
            ],
            [
                6220,
                6227,
                "PERSON"
            ],
            [
                6247,
                6253,
                "PERSON"
            ],
            [
                6334,
                6339,
                "PERSON"
            ],
            [
                6466,
                6473,
                "PERSON"
            ],
            [
                6552,
                6558,
                "PERSON"
            ],
            [
                6648,
                6654,
                "PERSON"
            ],
            [
                6746,
                6755,
                "PERSON"
            ],
            [
                6766,
                6772,
                "PERSON"
            ],
            [
                6796,
                6803,
                "PERSON"
            ],
            [
                6834,
                6847,
                "PERSON"
            ],
            [
                6922,
                6930,
                "PERSON"
            ],
            [
                6936,
                6942,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical center east whi { Merritt te, tyrone 1211 medical center  { Shannen dr mrn: 047717361, dob: 7/2 { Theresia 7/1969, lega { Gaylen l se { Johnette x: m nashville { Skylee  tn 37232 visit da { Green te: 8/13 { Kamran /2024 08/13/2024 - office  { Sammantha visit in v { Arya anderbilt diabetes and endocrinology (continued) lett { Aila ers (continued { Clifford ) eosinop { Kalani hils % absolute eosinophils 0.03-0.51x10(3)/mcl basophils % absolute ba { Shepard sophils  { Tamesha 0.01 - 0.08 x10(3)/mcl imm gran automate { Jahmir d % absolute imm gra { Kennth n automated  { Natisha 0.00 - 0.03 x10(3)/mcl - sodium level 135 ( { Shenika l) 136 - 145 mmol/l potassium level 5.5 (h) 3.3 - 4.8 mmol/l chloride level 108 (h) { Makiah  98 - 107 mmol/l carbon dioxide 22 22 - 29 mmol { Anas /l glucose level 208 (h) 70 - 99 mg/dl b { Amity lo { Shekinah od urea { Chayse  nitrogen 39  { Rosalio (h) 8 - 26 { Annabeth  mg/dl creatini { Heyward ne lev { Hilbert el 4.30 (h) 0.72 - 1.25 mg/ { Virginia dl calcium le { Belia vel t { Everlee otal 8.8 8.4 - 10.5 mg/dl anion gap 5 urine wbc 0 10/hpf urine rbc 0  { Maryah 4/hpf urine  { Tehya bacte { Aj ria none seen /hpf urine squamous epithelial cel { Dain ls <=0 { Fred /hpf urine mucous /hpf urine protein  { Zetta level <=15 mg/dl { Jaquez  urine creatinin { Donetta e 40 - 200 mg/dl urine protein/crea { Nicky tinine ratio  { Blade prin { Tristyn ted on 10/ { Bayley 3/24 7:12 am pag { Harry e  { Sabrena 419,vumc adult medical center east white, tyrone 1211 medic { Delpha al center dr mrn: 047717361, dob: 7/27/1969, legal sex { Herb : m nashville tn 37232 visit date: 8/13/2024 08/13/2024 { Nakia  - office visi { Tj t in va { Jenette nderbilt d { Tamekia iabetes and endocrin { Kayne ology (continued) letters (continue { Lanna d) <=0.20 mg/mg iron level  { Breona 61 - 157 mcg/dl iron binding capacity total { Ela  240 - 45 { Martha 0 mcg/dl iron saturation % vita { Hermelinda min d, total 25 - 80 ng/ml phosphorus l { Aamir eve { Cynthia l 2.3 - 4.7 mg/dl parathyroid hormone 16 -  { Cayleigh 77 pg/ml ferritin level 24 - 336 ng/ml egfrcr 15 (l)  { Joette >=60 ml/min/ { Mecca 1.73 { Tashina  m2 troponin-i <=0.03 ing/ml { Ashwin  hemoglo { Claira bin a1c level { Veta  9.9 % legend: (l) lo { Golden w ! abnormal (h) high panic assessment  { Janelly and plan 1. type 2 diabetes mellitus with stage 4 chronic kidney { Jennica  disease, with long-term current use of in { Peter sulin (cms/hcc) sli { Shiann ght im { Boaz provement in glycemic co { Erskine ntrol compared to a m { Sulema onth ago but still not taking  { Jala insulin as directed. he does { Tanvi  not  { Elon need a higher dose but simply needs to take the insulin. recom { Phineas mended he take the  { Akiva insulin on a schedule e { Coley g. 9a { Osiris /9p 10 units bid. discuss { Tex ed how the h { Hanah igh sugars are co { Kyrsten ntributing to his wors { Cortland eni { Mayer ng kidney { Sherrod  disease and his hyperkalemia - ve { Kyrah dc  { Deeanna diabete { Tyren s self { Asya -management e { Rosamaria ducat { Dakotah ion -- new; standing 2. hype { Ryon r { Cam kalemia - just completed treatme { Koa nt with lokelma. note today he is eating potassium rich diet. counseled  { Long pri { Roselle nted on 10 { Isa /3/24 7:12 am page 4 { Anyssa 20",
    {
        "entities": [
            [
                37,
                45,
                "PERSON"
            ],
            [
                79,
                87,
                "PERSON"
            ],
            [
                117,
                126,
                "PERSON"
            ],
            [
                141,
                148,
                "PERSON"
            ],
            [
                155,
                164,
                "PERSON"
            ],
            [
                181,
                188,
                "PERSON"
            ],
            [
                209,
                215,
                "PERSON"
            ],
            [
                226,
                233,
                "PERSON"
            ],
            [
                262,
                272,
                "PERSON"
            ],
            [
                285,
                290,
                "PERSON"
            ],
            [
                346,
                351,
                "PERSON"
            ],
            [
                368,
                377,
                "PERSON"
            ],
            [
                389,
                396,
                "PERSON"
            ],
            [
                470,
                478,
                "PERSON"
            ],
            [
                489,
                497,
                "PERSON"
            ],
            [
                540,
                547,
                "PERSON"
            ],
            [
                570,
                577,
                "PERSON"
            ],
            [
                592,
                600,
                "PERSON"
            ],
            [
                646,
                654,
                "PERSON"
            ],
            [
                740,
                747,
                "PERSON"
            ],
            [
                797,
                802,
                "PERSON"
            ],
            [
                845,
                851,
                "PERSON"
            ],
            [
                856,
                865,
                "PERSON"
            ],
            [
                875,
                882,
                "PERSON"
            ],
            [
                898,
                906,
                "PERSON"
            ],
            [
                919,
                928,
                "PERSON"
            ],
            [
                946,
                954,
                "PERSON"
            ],
            [
                963,
                971,
                "PERSON"
            ],
            [
                1001,
                1010,
                "PERSON"
            ],
            [
                1026,
                1032,
                "PERSON"
            ],
            [
                1040,
                1048,
                "PERSON"
            ],
            [
                1120,
                1127,
                "PERSON"
            ],
            [
                1142,
                1148,
                "PERSON"
            ],
            [
                1156,
                1159,
                "PERSON"
            ],
            [
                1210,
                1215,
                "PERSON"
            ],
            [
                1224,
                1229,
                "PERSON"
            ],
            [
                1269,
                1275,
                "PERSON"
            ],
            [
                1294,
                1301,
                "PERSON"
            ],
            [
                1320,
                1328,
                "PERSON"
            ],
            [
                1366,
                1372,
                "PERSON"
            ],
            [
                1388,
                1394,
                "PERSON"
            ],
            [
                1401,
                1409,
                "PERSON"
            ],
            [
                1422,
                1429,
                "PERSON"
            ],
            [
                1448,
                1454,
                "PERSON"
            ],
            [
                1459,
                1467,
                "PERSON"
            ],
            [
                1529,
                1536,
                "PERSON"
            ],
            [
                1593,
                1598,
                "PERSON"
            ],
            [
                1656,
                1662,
                "PERSON"
            ],
            [
                1679,
                1682,
                "PERSON"
            ],
            [
                1692,
                1700,
                "PERSON"
            ],
            [
                1713,
                1721,
                "PERSON"
            ],
            [
                1744,
                1750,
                "PERSON"
            ],
            [
                1788,
                1794,
                "PERSON"
            ],
            [
                1824,
                1831,
                "PERSON"
            ],
            [
                1877,
                1881,
                "PERSON"
            ],
            [
                1893,
                1900,
                "PERSON"
            ],
            [
                1934,
                1945,
                "PERSON"
            ],
            [
                1987,
                1993,
                "PERSON"
            ],
            [
                1999,
                2007,
                "PERSON"
            ],
            [
                2053,
                2062,
                "PERSON"
            ],
            [
                2118,
                2125,
                "PERSON"
            ],
            [
                2140,
                2146,
                "PERSON"
            ],
            [
                2153,
                2161,
                "PERSON"
            ],
            [
                2192,
                2199,
                "PERSON"
            ],
            [
                2210,
                2217,
                "PERSON"
            ],
            [
                2233,
                2238,
                "PERSON"
            ],
            [
                2262,
                2269,
                "PERSON"
            ],
            [
                2311,
                2319,
                "PERSON"
            ],
            [
                2386,
                2394,
                "PERSON"
            ],
            [
                2439,
                2445,
                "PERSON"
            ],
            [
                2467,
                2474,
                "PERSON"
            ],
            [
                2483,
                2488,
                "PERSON"
            ],
            [
                2515,
                2523,
                "PERSON"
            ],
            [
                2547,
                2554,
                "PERSON"
            ],
            [
                2587,
                2592,
                "PERSON"
            ],
            [
                2623,
                2629,
                "PERSON"
            ],
            [
                2637,
                2642,
                "PERSON"
            ],
            [
                2707,
                2715,
                "PERSON"
            ],
            [
                2737,
                2743,
                "PERSON"
            ],
            [
                2769,
                2775,
                "PERSON"
            ],
            [
                2783,
                2790,
                "PERSON"
            ],
            [
                2818,
                2822,
                "PERSON"
            ],
            [
                2837,
                2843,
                "PERSON"
            ],
            [
                2863,
                2871,
                "PERSON"
            ],
            [
                2896,
                2905,
                "PERSON"
            ],
            [
                2911,
                2917,
                "PERSON"
            ],
            [
                2929,
                2937,
                "PERSON"
            ],
            [
                2974,
                2980,
                "PERSON"
            ],
            [
                2986,
                2994,
                "PERSON"
            ],
            [
                3004,
                3010,
                "PERSON"
            ],
            [
                3019,
                3024,
                "PERSON"
            ],
            [
                3040,
                3050,
                "PERSON"
            ],
            [
                3058,
                3066,
                "PERSON"
            ],
            [
                3097,
                3102,
                "PERSON"
            ],
            [
                3106,
                3110,
                "PERSON"
            ],
            [
                3145,
                3149,
                "PERSON"
            ],
            [
                3224,
                3229,
                "PERSON"
            ],
            [
                3235,
                3243,
                "PERSON"
            ],
            [
                3256,
                3260,
                "PERSON"
            ],
            [
                3283,
                3290,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, d { Dia ob: 7/27/1969,  { Aramis legal sex: m nashville tn 37232-0 { Jule 00 { Jossie 4 adm: 9/ { Lavonna 9/2024, d/c: 9/10/2024 09/09/2024 - ed in vanderbilt emergency department (continued) other orders (continued) disco { Allena ntinued by: discharge provider, automatic 09/10/24 0539 [patient d { Diedra ischarge] questionnaire question answer test for: glucose source capillary point of car { Jailynn e glucose task b { Waverly aseline order comments: baseline blo { Brando od glucose - draw prior to administering insulin poc lab: glucose; source: capillary (discontinued) electronically  { Conan signed by: koetter, paige eden, md on 09/10/24 0216 status: discontinued  { Maliya ordering user: koetter, paige eden, md 0 { Akil 9/10/2 { Alexavier 4 0216 { Jahlil  ordering provider: koetter, paige eden, md autho { Kaidyn r { Fredericka ized by: boaglio, sean michael, do ordering mode: standard frequ { Lynell ency: routine once 09/10/24 0316 - 1 occurrence cl { Christy ass: hospital perfo { Jenice rmed quantity: 1 instance released by: koetter, paige eden, md (auto-released) 9 { Mariaelena /10/2024 2:16 am discontinued by: discharge provider, automatic 09/10/24 0539 [patient discharge] questionnaire question answer test for: glucose source capillary point of care glucose { Yocheved  task first hour after basel { Eris ine order comments: first hour after baseline poc lab: glucose; source: capillary (discontinued) electronically signed by: koetter, paige eden, md on 09/10/24 0216 status: discontinued ordering user:  { Felicitas koetter, paige eden, md 09/10/24 021 { Karyme 6 ordering provider: koetter, paige eden, md authorized by: boaglio, sean michael, do ordering mode: standard frequency: routine once 09/10/24 0416 - 1 occurrence  { Jessy class: hospital performed quantity: 1 instance relea { Jolyn sed b { Kary y: koetter, paige eden, md (auto-released) 9/10/2024 2:16 am  { Ole discontinued by: discharge provider, automatic 09/10/24 0539 [patient discharge] questionnai { Dajah re question answer test for: glucose source capillary point of care glucose task second hour after baseline order comments: second hour after baseline poc lab: glucose; source: capillary (discontinued) electronically signed by:  { Desirea koet { Myriah ter, paige eden, md on 09/10/24 0216 stat { Renay us: discontinued ordering user: koetter, paige eden, md 09/10/24 02 { Lael 16 ordering provider: koetter, paige eden, md authorized by: boaglio, sean michael, do  { Mackenzi ordering mode: standard { Melissia  frequency: routine once 09/10/24 0516 - 1 occurrence class: hospita { Tawni l performed quantity: 1 instance released by: koetter, paige eden, md (auto-released) 9 { Vernon /10/2024 2:16 am discontinued by: discharge provider, automatic 09/10/24 0539 [patient discharge] questionnaire question answer test for: glucose source cap { Fenton illary point of care glucose t { Analicia a { Takiyah sk third hour after baseline order comments: third hour after  { Shawana baseli { Tyisha ne - nurs { Bartley e may cance { Elder l if both three hours el { Aleyah apsed between insulin administration and most rec { Epifanio ent bg, and all values related to hyperkalemia were greater than 140 mg/dl. poc lab: glucose; so { Rakeem urce: capillary (discontinued) printed on 10/3/24 7:12 am page 91,vumc adult ho { Kimball spi { Kamilla tal wh { Shawnta ite, tyro { Andree n { Jovana e 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, l { Mariajose egal sex: { Xavion  m nashville tn 37232-0004 adm: 9/9/2024, d/c: { Birtha  9/10/2024 09/09 { Orson /20 { Reyansh 24 - ed in van { Britny derbilt emergency  { Eason department (continued) other orders (continued) electronically signed by: koetter, paige eden, md on 09/10/24 0216 status: discontinued ordering user: koetter, paige eden, md 09/10/24 0216 orde { Nicholle ring provider: koetter, paige eden, md authorized by: boaglio, sean michael, do ordering mode: standard frequency: routine once 09/10/24 { Shanequa  0616 - 1 occurrence class: hospi { Ellwood tal performed quantity: 1  { Marcial instance released { Angelle  by: koetter, paige eden, md (auto-released)  { Aerial 9/10/ { Candra 2024 2:16 am discontinued by: discharge provider, automatic 09/10/2 { Emmerson 4 0539 [patient discharge] qu { Lotus estionnaire question answer test for: glu { Valda cose source capillary point of ca { Tatyanna re glucose task fourth hour after baseline order comments: fourth hour after base { Laurance line - nurse may cancel if all previous bg values related to hyperkalemia were greater than 140 mg/dl. poc lab: lytes, ica, glu, bun, creat, hct, hgb; source: venous (discontinued) electronically signed by: boaglio, sean michael, do on 09/10/24 0349 status: discontinued mode: ordering in verbal with rea { Mickel dback mode communicated by: stalbaum, angela n, rn ordering user: stalbaum, angela n, rn 09/10/24 0230 ordering provider: boaglio, sea { Taurus n michael, do authorized by: boaglio, sean michael, do ordering mode: verbal with readback additional signing events electronically si { Amy gned by boaglio, sean michael, do 09/10/24 0349, for d { Lyndell iscontinuing i { Amberlee n verbal with readb { Jihad ack mode, communicator - stalbaum, angela  { Georgiann n, rn frequency: stat once 09/10/24 0230 - 1 occurrence class: hospital performed quantity: 1 instance released by: stalbaum, angela n, rn (a { Shandi uto-released) 9/10/2024 2:30 am discontinued by: stalbaum,  { Berkley angela n, rn 09/10/24 0234 questionnaire question answer test for: lytes, ica, glu, bun, creat, hct, hgb source ve { Gaspar nous specimen information id type source collected by - blood - - medication administration record dextrose (d50w) 50 % injection 25 ml [472530 { Verner 280] ordering provider: koetter, paige eden, md status: discontinued (pas { Yasin t end date/time), reason:  { Dezirae patient discharge  { Isael ordered on: 09/10/24 0216 starts/ends: 09/10/24 021 { Tino 5  { Amilia - 09/10/24 0539 ordered dose (remaining/total): 25 ml (-/-) route: intravenous frequency: every 15 min prn ordered rate/order duration: - / admin instructions: hypertonic: consider c { Aide entral cath infiltration/extr { Codi avasation risk = red (vesicant) question answer comment ind { Hadley i { Elisheva cation ::  { Guillermina hypoglycemi { Latrisha a management - ( schedule { Penney d or recorded for this medication in the specified { Avion  date/time range) dextrose  { Pearlene (d50w) 50 % injection 50 ml [466717531] ordering provider: koetter, paige eden, md statu { Alice s: discontinued (past end date/time), reason: patient discharg { Angelene e print { Angelie ed on 10/3/24 7:12 am page 92",
    {
        "entities": [
            [
                78,
                82,
                "PERSON"
            ],
            [
                100,
                107,
                "PERSON"
            ],
            [
                143,
                148,
                "PERSON"
            ],
            [
                153,
                160,
                "PERSON"
            ],
            [
                172,
                180,
                "PERSON"
            ],
            [
                299,
                306,
                "PERSON"
            ],
            [
                375,
                382,
                "PERSON"
            ],
            [
                472,
                480,
                "PERSON"
            ],
            [
                499,
                507,
                "PERSON"
            ],
            [
                546,
                553,
                "PERSON"
            ],
            [
                671,
                677,
                "PERSON"
            ],
            [
                753,
                760,
                "PERSON"
            ],
            [
                803,
                808,
                "PERSON"
            ],
            [
                817,
                827,
                "PERSON"
            ],
            [
                836,
                843,
                "PERSON"
            ],
            [
                895,
                902,
                "PERSON"
            ],
            [
                906,
                917,
                "PERSON"
            ],
            [
                984,
                991,
                "PERSON"
            ],
            [
                1044,
                1052,
                "PERSON"
            ],
            [
                1074,
                1081,
                "PERSON"
            ],
            [
                1164,
                1175,
                "PERSON"
            ],
            [
                1362,
                1371,
                "PERSON"
            ],
            [
                1402,
                1407,
                "PERSON"
            ],
            [
                1610,
                1620,
                "PERSON"
            ],
            [
                1659,
                1666,
                "PERSON"
            ],
            [
                1832,
                1838,
                "PERSON"
            ],
            [
                1893,
                1899,
                "PERSON"
            ],
            [
                1907,
                1912,
                "PERSON"
            ],
            [
                1976,
                1980,
                "PERSON"
            ],
            [
                2075,
                2081,
                "PERSON"
            ],
            [
                2312,
                2320,
                "PERSON"
            ],
            [
                2327,
                2334,
                "PERSON"
            ],
            [
                2378,
                2384,
                "PERSON"
            ],
            [
                2454,
                2459,
                "PERSON"
            ],
            [
                2549,
                2558,
                "PERSON"
            ],
            [
                2584,
                2593,
                "PERSON"
            ],
            [
                2664,
                2670,
                "PERSON"
            ],
            [
                2760,
                2767,
                "PERSON"
            ],
            [
                2926,
                2933,
                "PERSON"
            ],
            [
                2966,
                2975,
                "PERSON"
            ],
            [
                2979,
                2987,
                "PERSON"
            ],
            [
                3052,
                3060,
                "PERSON"
            ],
            [
                3069,
                3076,
                "PERSON"
            ],
            [
                3088,
                3096,
                "PERSON"
            ],
            [
                3110,
                3116,
                "PERSON"
            ],
            [
                3143,
                3150,
                "PERSON"
            ],
            [
                3202,
                3211,
                "PERSON"
            ],
            [
                3310,
                3317,
                "PERSON"
            ],
            [
                3399,
                3407,
                "PERSON"
            ],
            [
                3413,
                3421,
                "PERSON"
            ],
            [
                3430,
                3438,
                "PERSON"
            ],
            [
                3450,
                3457,
                "PERSON"
            ],
            [
                3461,
                3468,
                "PERSON"
            ],
            [
                3530,
                3540,
                "PERSON"
            ],
            [
                3552,
                3559,
                "PERSON"
            ],
            [
                3608,
                3615,
                "PERSON"
            ],
            [
                3634,
                3640,
                "PERSON"
            ],
            [
                3646,
                3654,
                "PERSON"
            ],
            [
                3671,
                3678,
                "PERSON"
            ],
            [
                3699,
                3705,
                "PERSON"
            ],
            [
                3901,
                3910,
                "PERSON"
            ],
            [
                4049,
                4058,
                "PERSON"
            ],
            [
                4094,
                4102,
                "PERSON"
            ],
            [
                4131,
                4139,
                "PERSON"
            ],
            [
                4159,
                4167,
                "PERSON"
            ],
            [
                4215,
                4222,
                "PERSON"
            ],
            [
                4230,
                4237,
                "PERSON"
            ],
            [
                4307,
                4316,
                "PERSON"
            ],
            [
                4348,
                4354,
                "PERSON"
            ],
            [
                4398,
                4404,
                "PERSON"
            ],
            [
                4440,
                4449,
                "PERSON"
            ],
            [
                4533,
                4542,
                "PERSON"
            ],
            [
                4849,
                4856,
                "PERSON"
            ],
            [
                4993,
                5000,
                "PERSON"
            ],
            [
                5137,
                5141,
                "PERSON"
            ],
            [
                5198,
                5206,
                "PERSON"
            ],
            [
                5223,
                5232,
                "PERSON"
            ],
            [
                5254,
                5260,
                "PERSON"
            ],
            [
                5305,
                5315,
                "PERSON"
            ],
            [
                5459,
                5466,
                "PERSON"
            ],
            [
                5528,
                5536,
                "PERSON"
            ],
            [
                5653,
                5660,
                "PERSON"
            ],
            [
                5806,
                5813,
                "PERSON"
            ],
            [
                5889,
                5895,
                "PERSON"
            ],
            [
                5924,
                5932,
                "PERSON"
            ],
            [
                5953,
                5959,
                "PERSON"
            ],
            [
                6013,
                6018,
                "PERSON"
            ],
            [
                6023,
                6030,
                "PERSON"
            ],
            [
                6215,
                6220,
                "PERSON"
            ],
            [
                6252,
                6257,
                "PERSON"
            ],
            [
                6319,
                6326,
                "PERSON"
            ],
            [
                6330,
                6339,
                "PERSON"
            ],
            [
                6352,
                6364,
                "PERSON"
            ],
            [
                6378,
                6387,
                "PERSON"
            ],
            [
                6415,
                6422,
                "PERSON"
            ],
            [
                6475,
                6481,
                "PERSON"
            ],
            [
                6511,
                6520,
                "PERSON"
            ],
            [
                6611,
                6617,
                "PERSON"
            ],
            [
                6682,
                6691,
                "PERSON"
            ],
            [
                6701,
                6709,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hosp { Leo it { Molli al white, tyrone 121 { Raizy 1 medical center dr. mrn: 0477 { Rylynn 17361, dob: { Francheska  7/27/1969, legal s { Armen ex: m { Taran  nashvil { Vishal le tn 37232-0004 adm: 8/23/20 { Bracha 24, d/c: 8/24/2024 08/23/2024 - ed in vanderbilt emergency department ( { Rosy continued) clinical { Ruthe  notes (continued) ant { Latashia er { Rayvon ior vitreous normal normal f { Beyonce undus exam { Ameerah  right left poste { Journi rior vitreous normal norm { Aven al disc flat  { Cherelle and sha { Josslyn rp flat and sharp c/d ratio 0.15  { Nadene 0.15 macula { Seana   { Ferrell normal normal vessel { Lady s attenu { Akira ated attenuated periphe { Murl ry  { Rachel difficult view, gros { Natividad sly flat difficult view, grossly f { Priyanka lat wounds closed, lens clear,  of infection  { Osmar insid { Parth e the eye imagi { Brooklin ng:  found. asses { Gracen sment &  { Nohemi plan eye pain/injection, { Sim  s/p ceoil os -patient reports that he  { Deyanira stop { Tilda ped taking the post-operative drops 3-4 d { Jaedyn ay { Tadeo s after surgery -  of endophthal { Mckenzi mitis. wounds closed { Cliffton , { Hipolito  mi { Kiyah ld 1-2+ cell, 1+ injection vs { Nicoletta  resolving sch. dfe unremarkable. va 20/2 { Annie 0 os - { Orvil  reports let bott { Amarie le fall down sink after or plan - mox { Yuri ifloxacin qi { Tavian d 1 week - prednisolone four t { Kendell imes d { Demitri aily { Jerrad  ( { Dorathy qid) for one week th { Glennie en { Lisamarie  three times daily (tid) for one week then twice da { Ricky ily { Olevia  (bid { Woodie ) for one week then once daily (qd) for one week then stop ketorolac qid { Margarite  until gone - rtc with cataract surg { Demarius eon next week this patient was seen and  { Winton discuss { Kaylei ed  { Jaslynn with senior resident, dr. be { Demitrius rkowitz. p { Jordon lease  { Esha see attending attestation for final { Felicita  plan. jonathan siktberg, md, pgy2 vanderbilt eye i { Levern nstitut { Tramaine e printed on 10/3/24 7:12 am page { Denver  183,vumc adult hospital white, tyrone 1211 medical center dr. mrn: 047717361, d { Haydn o { Lydell b: 7/2 { Adilyn 7/1 { Micheal 969, legal sex: m nashville tn { Katherine  37232-0004 adm: 8/23/2 { Nikolaos 02 { Tyreese 4, d/c { Lavonia : 8/24/2024 08/23/2024 - ed in vanderbilt emergency department (contin { Eston ue { Mortimer d) clinical notes ( { Alline continued) electr { Merrilee o { Caeden nica { Castiel lly signed by s { Zarah ik { Macon tberg, jonath { Parris an, md at 8/23/2024 11:53 pm electronically signed by flemm { Sherrill ons, meghan susan { Jacari ne, { Zaden  md at 8/24/2024 3:23 pm after visit summary warning! { Dyana   { Kameryn this summary shows information { Valeri  as of your visit. it might not contai { Jesusa n the most up-to-date inform { Lemar at { Crysta ion in y { Melita our ch { Faisal a { Cherlyn rt. { Kellyann  excuses (below) printed on 10/3/2 { Jansen 4 7:12 am page 184",
    {
        "entities": [
            [
                18,
                22,
                "PERSON"
            ],
            [
                27,
                33,
                "PERSON"
            ],
            [
                56,
                62,
                "PERSON"
            ],
            [
                95,
                102,
                "PERSON"
            ],
            [
                116,
                127,
                "PERSON"
            ],
            [
                149,
                155,
                "PERSON"
            ],
            [
                163,
                169,
                "PERSON"
            ],
            [
                180,
                187,
                "PERSON"
            ],
            [
                219,
                226,
                "PERSON"
            ],
            [
                300,
                305,
                "PERSON"
            ],
            [
                327,
                333,
                "PERSON"
            ],
            [
                358,
                367,
                "PERSON"
            ],
            [
                372,
                379,
                "PERSON"
            ],
            [
                410,
                418,
                "PERSON"
            ],
            [
                431,
                439,
                "PERSON"
            ],
            [
                459,
                466,
                "PERSON"
            ],
            [
                494,
                499,
                "PERSON"
            ],
            [
                515,
                524,
                "PERSON"
            ],
            [
                534,
                542,
                "PERSON"
            ],
            [
                578,
                585,
                "PERSON"
            ],
            [
                599,
                605,
                "PERSON"
            ],
            [
                609,
                617,
                "PERSON"
            ],
            [
                640,
                645,
                "PERSON"
            ],
            [
                656,
                662,
                "PERSON"
            ],
            [
                688,
                693,
                "PERSON"
            ],
            [
                699,
                706,
                "PERSON"
            ],
            [
                729,
                739,
                "PERSON"
            ],
            [
                776,
                785,
                "PERSON"
            ],
            [
                833,
                839,
                "PERSON"
            ],
            [
                847,
                853,
                "PERSON"
            ],
            [
                871,
                880,
                "PERSON"
            ],
            [
                900,
                907,
                "PERSON"
            ],
            [
                918,
                925,
                "PERSON"
            ],
            [
                952,
                956,
                "PERSON"
            ],
            [
                998,
                1007,
                "PERSON"
            ],
            [
                1014,
                1020,
                "PERSON"
            ],
            [
                1064,
                1071,
                "PERSON"
            ],
            [
                1076,
                1082,
                "PERSON"
            ],
            [
                1117,
                1125,
                "PERSON"
            ],
            [
                1148,
                1157,
                "PERSON"
            ],
            [
                1161,
                1170,
                "PERSON"
            ],
            [
                1176,
                1182,
                "PERSON"
            ],
            [
                1214,
                1224,
                "PERSON"
            ],
            [
                1268,
                1274,
                "PERSON"
            ],
            [
                1283,
                1289,
                "PERSON"
            ],
            [
                1309,
                1316,
                "PERSON"
            ],
            [
                1356,
                1361,
                "PERSON"
            ],
            [
                1376,
                1383,
                "PERSON"
            ],
            [
                1416,
                1424,
                "PERSON"
            ],
            [
                1433,
                1441,
                "PERSON"
            ],
            [
                1448,
                1455,
                "PERSON"
            ],
            [
                1460,
                1468,
                "PERSON"
            ],
            [
                1491,
                1499,
                "PERSON"
            ],
            [
                1504,
                1514,
                "PERSON"
            ],
            [
                1568,
                1574,
                "PERSON"
            ],
            [
                1580,
                1587,
                "PERSON"
            ],
            [
                1595,
                1602,
                "PERSON"
            ],
            [
                1677,
                1687,
                "PERSON"
            ],
            [
                1726,
                1735,
                "PERSON"
            ],
            [
                1778,
                1785,
                "PERSON"
            ],
            [
                1795,
                1802,
                "PERSON"
            ],
            [
                1808,
                1816,
                "PERSON"
            ],
            [
                1847,
                1857,
                "PERSON"
            ],
            [
                1870,
                1877,
                "PERSON"
            ],
            [
                1886,
                1891,
                "PERSON"
            ],
            [
                1929,
                1938,
                "PERSON"
            ],
            [
                1992,
                1999,
                "PERSON"
            ],
            [
                2009,
                2018,
                "PERSON"
            ],
            [
                2054,
                2061,
                "PERSON"
            ],
            [
                2144,
                2150,
                "PERSON"
            ],
            [
                2154,
                2161,
                "PERSON"
            ],
            [
                2170,
                2177,
                "PERSON"
            ],
            [
                2183,
                2191,
                "PERSON"
            ],
            [
                2224,
                2234,
                "PERSON"
            ],
            [
                2260,
                2269,
                "PERSON"
            ],
            [
                2274,
                2282,
                "PERSON"
            ],
            [
                2291,
                2299,
                "PERSON"
            ],
            [
                2372,
                2378,
                "PERSON"
            ],
            [
                2383,
                2392,
                "PERSON"
            ],
            [
                2414,
                2421,
                "PERSON"
            ],
            [
                2441,
                2450,
                "PERSON"
            ],
            [
                2454,
                2461,
                "PERSON"
            ],
            [
                2468,
                2476,
                "PERSON"
            ],
            [
                2494,
                2500,
                "PERSON"
            ],
            [
                2505,
                2511,
                "PERSON"
            ],
            [
                2527,
                2534,
                "PERSON"
            ],
            [
                2596,
                2605,
                "PERSON"
            ],
            [
                2625,
                2632,
                "PERSON"
            ],
            [
                2638,
                2644,
                "PERSON"
            ],
            [
                2700,
                2706,
                "PERSON"
            ],
            [
                2710,
                2718,
                "PERSON"
            ],
            [
                2751,
                2758,
                "PERSON"
            ],
            [
                2799,
                2806,
                "PERSON"
            ],
            [
                2837,
                2843,
                "PERSON"
            ],
            [
                2848,
                2855,
                "PERSON"
            ],
            [
                2866,
                2873,
                "PERSON"
            ],
            [
                2882,
                2889,
                "PERSON"
            ],
            [
                2893,
                2901,
                "PERSON"
            ],
            [
                2907,
                2916,
                "PERSON"
            ],
            [
                2953,
                2960,
                "PERSON"
            ]
        ]
    }
),(
    "vumc vis midtown white, tyrone 337 22nd a { Addelyn ve n  { Brinlee mrn: 047717361, dob: 7/27/196 { Yulisa 9, legal sex: m nashville tn 372 { Yehoshua 03 visit date: 4/12/2024 04/12/2024 - proced { Lashawna ure pass in vanderbilt imaging servi { Nathan ces midtown facesheet report patient demographics patient name mrn legal dob  { Aiden addr { Allen ess phone white, tyrone 0477173 sex 7/27/19 { Allyn 69 apt 7 { Marylouise 05 615-260-2291 (home) 61 m 110 { Safiya 1 { Vann  edgehill ave 615- { Newman 260-2291 (mobile)  { Janella nashville tn 37203 *pref { Kaniyah erred* hospital accou { Dorman nt not on file admission i { Lajuana nf { Vanity ormation current informatio { Kc n attending provider admitting provider admission type admission status unknow { Neill n status admission date/time disc { Desarae harge date/time hospital service auth/ce { Roselynn rt status hospital area unit room/bed referring provider 04/12/2024 - procedure pass in vanderbilt imaging services midtown (continued) visit  { Storm information admission information a { Tesha rrival  { Therman date/time: admit date/time: 04/12/2024 ip adm. date/t { Vikram ime: admission type: point of { Eman  origin: admit categor { Janita y: means of arrival: primary service: sec { Noma onda { Atreyu ry service: n/a transfer source: service area: unit: ad { Casimer mit p { Demari rovider: attending provider: referring provider: discharge information date/time: - dispositio { Maximino n: destination: - provider: u { Karisma nit: flowsheets custom formula data row na { Shanti me 04/12/24 1247 measurements total weight 2222 percent -jn at change percent 04/12/24 1247  { Cordero weight change 84.35 k { Diedre gs -jn at since preop 04/12/24 1247  { Steffany weight change 84.35 kg -jn at since preop 04/12/24 { Rashon  1 { Om 247  { Lurline other weight change 84.35 kg -jn at sin { Maye ce  { Ocean last v { Roan isit 0 { Saverio 4/12/24 1247 weight change 84.35 kg  { Shaunte -jn at from preop 04/12/24 1247 weight change 84.35 kg -jn at since last visit 04/12/24 1247 weight change 84.35 -jn at 04/12/24 1247 printed on 10/3/24 7:12 am page 1101,vumc vis midtown whi { Willetta te, tyrone 337 22nd ave n mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37203 visit  { Evelyn date: 4/12/2024 04/12 { Valentine /2024 - procedur { Nickolaus e pass in vanderb { Lannie ilt imaging s { Deloise ervic { Rayfield es midtown (continued) f { Rondal lowsheets (continued) from pre { Brady op (kg) percent weight 84.35 lbs -jn at change since 04/12/24 1247 preop percent weight 2977 percent - { Marlow jn at change since 04/12/24 1247 last visit weight change -0.11 percent -jn at since las { Bea t visit 04/12/24 (%) weight change 185.96 lbs -jn at since preop ( { Gracia lbs) 04/12/24 1247 weight change -0.2 lbs -jn at 04/12/24 since la { San st visit 1247 (lbs)  { Catherine current weight 84.369 kg at 4/12/2024 12:47 pm -jn at 04 { Domonique /12/24 1247 weight change 84.35 kg -jn  { Evaristo at si { Doloris n { Kaliah ce last visit 04/12/24 1247 fluid 0 -jn at 04/12/24 1247 resuscitation (#3) vo { Laurent lume estimates fluid { Fayth  0 -jn at 04/12 { Kolt /24 1247 res { Londynn uscitation { Meda  (#4) ratio-b { Malani ased meal dos { Zhane ing approx predicted 5.9 -jn at 04/12/24 1247 ratio (500 / wt in kg) rec start sliding 35.6 -jn at 04/12/24 scale (3000 / wt 1247 in { Ruel  kg) basic inform { Fatoumata ati { Kehlani on approx predicted 42.2 -jn at 0 { Laird 4/12 { Keonna /2 { Berton 4 basal (0.5 * wt in 1247 kg) rec s { Mervyn tart basal 21.1 - { Analee jn { Lamya  at 04/12/24 (.25 * wt in kg) 1247 rec start fixed { Prisha  7 -jn at 04/12/24 12 { Dietrich 47 meal (.08 * wt in kg) fixed  { Laurel meal insuli { Arlis n dosing { Calum  approx pr { Gust e { Malky dicted 17.8 -jn  { Marry at 04/12/ { Taja 24 c { Mahdi orrection (1500 / 1247 { Cambree  wt in kg) height and  { Clarence weight row name 04/12/24 1 { Ernest 247 height and weight weight  { Verlene 84.4 kg (186 lb) -jn  { Lisandro at 04/12/24 1247 lund-brow { Anaiya der (adult { Chasen ) row nam { Yovani e 04/12/24 1247 volume estimates printed on 10 { Azalia /3/24 7:12 am pag { Eulah e 1102",
    {
        "entities": [
            [
                44,
                52,
                "PERSON"
            ],
            [
                60,
                68,
                "PERSON"
            ],
            [
                100,
                107,
                "PERSON"
            ],
            [
                142,
                151,
                "PERSON"
            ],
            [
                198,
                207,
                "PERSON"
            ],
            [
                246,
                253,
                "PERSON"
            ],
            [
                333,
                339,
                "PERSON"
            ],
            [
                346,
                352,
                "PERSON"
            ],
            [
                398,
                404,
                "PERSON"
            ],
            [
                415,
                426,
                "PERSON"
            ],
            [
                460,
                467,
                "PERSON"
            ],
            [
                471,
                476,
                "PERSON"
            ],
            [
                497,
                504,
                "PERSON"
            ],
            [
                525,
                533,
                "PERSON"
            ],
            [
                560,
                568,
                "PERSON"
            ],
            [
                592,
                599,
                "PERSON"
            ],
            [
                628,
                636,
                "PERSON"
            ],
            [
                641,
                648,
                "PERSON"
            ],
            [
                678,
                681,
                "PERSON"
            ],
            [
                762,
                768,
                "PERSON"
            ],
            [
                804,
                812,
                "PERSON"
            ],
            [
                855,
                864,
                "PERSON"
            ],
            [
                1009,
                1015,
                "PERSON"
            ],
            [
                1053,
                1059,
                "PERSON"
            ],
            [
                1069,
                1077,
                "PERSON"
            ],
            [
                1133,
                1140,
                "PERSON"
            ],
            [
                1172,
                1177,
                "PERSON"
            ],
            [
                1202,
                1209,
                "PERSON"
            ],
            [
                1253,
                1258,
                "PERSON"
            ],
            [
                1265,
                1272,
                "PERSON"
            ],
            [
                1330,
                1338,
                "PERSON"
            ],
            [
                1346,
                1353,
                "PERSON"
            ],
            [
                1450,
                1459,
                "PERSON"
            ],
            [
                1491,
                1499,
                "PERSON"
            ],
            [
                1544,
                1551,
                "PERSON"
            ],
            [
                1646,
                1654,
                "PERSON"
            ],
            [
                1678,
                1685,
                "PERSON"
            ],
            [
                1724,
                1733,
                "PERSON"
            ],
            [
                1786,
                1793,
                "PERSON"
            ],
            [
                1798,
                1801,
                "PERSON"
            ],
            [
                1808,
                1816,
                "PERSON"
            ],
            [
                1858,
                1863,
                "PERSON"
            ],
            [
                1869,
                1875,
                "PERSON"
            ],
            [
                1884,
                1889,
                "PERSON"
            ],
            [
                1898,
                1906,
                "PERSON"
            ],
            [
                1945,
                1953,
                "PERSON"
            ],
            [
                2147,
                2156,
                "PERSON"
            ],
            [
                2255,
                2262,
                "PERSON"
            ],
            [
                2286,
                2296,
                "PERSON"
            ],
            [
                2315,
                2325,
                "PERSON"
            ],
            [
                2345,
                2352,
                "PERSON"
            ],
            [
                2368,
                2376,
                "PERSON"
            ],
            [
                2384,
                2393,
                "PERSON"
            ],
            [
                2420,
                2427,
                "PERSON"
            ],
            [
                2460,
                2466,
                "PERSON"
            ],
            [
                2571,
                2578,
                "PERSON"
            ],
            [
                2669,
                2673,
                "PERSON"
            ],
            [
                2742,
                2749,
                "PERSON"
            ],
            [
                2818,
                2822,
                "PERSON"
            ],
            [
                2845,
                2855,
                "PERSON"
            ],
            [
                2914,
                2924,
                "PERSON"
            ],
            [
                2966,
                2975,
                "PERSON"
            ],
            [
                2983,
                2991,
                "PERSON"
            ],
            [
                2995,
                3002,
                "PERSON"
            ],
            [
                3083,
                3091,
                "PERSON"
            ],
            [
                3114,
                3120,
                "PERSON"
            ],
            [
                3138,
                3143,
                "PERSON"
            ],
            [
                3158,
                3166,
                "PERSON"
            ],
            [
                3179,
                3184,
                "PERSON"
            ],
            [
                3200,
                3207,
                "PERSON"
            ],
            [
                3223,
                3229,
                "PERSON"
            ],
            [
                3364,
                3369,
                "PERSON"
            ],
            [
                3389,
                3399,
                "PERSON"
            ],
            [
                3405,
                3413,
                "PERSON"
            ],
            [
                3449,
                3455,
                "PERSON"
            ],
            [
                3462,
                3469,
                "PERSON"
            ],
            [
                3474,
                3481,
                "PERSON"
            ],
            [
                3519,
                3526,
                "PERSON"
            ],
            [
                3546,
                3553,
                "PERSON"
            ],
            [
                3558,
                3564,
                "PERSON"
            ],
            [
                3617,
                3624,
                "PERSON"
            ],
            [
                3648,
                3657,
                "PERSON"
            ],
            [
                3691,
                3698,
                "PERSON"
            ],
            [
                3712,
                3718,
                "PERSON"
            ],
            [
                3729,
                3735,
                "PERSON"
            ],
            [
                3748,
                3753,
                "PERSON"
            ],
            [
                3757,
                3763,
                "PERSON"
            ],
            [
                3782,
                3788,
                "PERSON"
            ],
            [
                3800,
                3805,
                "PERSON"
            ],
            [
                3812,
                3818,
                "PERSON"
            ],
            [
                3843,
                3851,
                "PERSON"
            ],
            [
                3876,
                3885,
                "PERSON"
            ],
            [
                3914,
                3921,
                "PERSON"
            ],
            [
                3953,
                3961,
                "PERSON"
            ],
            [
                3985,
                3994,
                "PERSON"
            ],
            [
                4023,
                4030,
                "PERSON"
            ],
            [
                4043,
                4050,
                "PERSON"
            ],
            [
                4062,
                4069,
                "PERSON"
            ],
            [
                4118,
                4125,
                "PERSON"
            ],
            [
                4145,
                4151,
                "PERSON"
            ]
        ]
    }
),(
    "vumc hendersonville - anderson white, tyrone 128 n anderson ln mrn: 0 { Francie 47717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 visit date: 1/ { Romy 10/2023 01/10/2023 - nurse triage in vander { Victor bilt primary care hendersonville (continued) medication list (continued) medications last reviewed by lippard { Benjamine , giles a, aprn on 12/6/2022 1244 insulin glargine (u-100) 100 unit/ml (3 ml { Shaniah ) sub { Vanna cutaneous pe { Vianca n (lantus solostar,basaglar kwikpen) discontinued by: greenspan, { Whittney  debra l, aprn discontinued on: 2/14/2023 reas { Tyshaun on for discontinuation: discontinued by another clinician instructions: inject 25 units under the  { Cassia skin daily. authorized by: lippard, giles a, aprn ordered on: 9/27/2022 start date: 9/27/2022 quantity: 9 ml { Dusti  refill: 11 refills by 9/27/2023 pioglitazone 30 mg tablet (actos) discontinued by: greens { Morgen pan, debra l, aprn discontinued on: 2/14/ { Oliva 2023 reason for discontinuation: discontinued by another cl { Tambra inician instructions: take 1 tablet (30 mg total) by mouth every morning before breakfast. aut { Eliazar horized by: { Zac  lippard, giles a, aprn order { Janyce ed on: 10/13/2022 start date { Viki : 10/13/2022 end date: 2/14/2023 quantity: 90 tablet refill:  remaining carvedilol 6.25 mg tablet (coreg) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 instructions: take 1 tablet (6.25 mg total) by mouth daily. authorized by: lippard, giles a, aprn ordered on: 10/25/2022 { Hilliard  start date: 10/25/2022 end { Aspyn  date: 6/3/2023 quantity: 90 tablet refill: 1 refill by 10/25/2023 atorvastatin 80 mg tablet (lipitor) discontinued by: lippard, giles a, aprn discontinued on: 8/8/2023 reason for discontinuation: reorder instructions: take 1 tablet (80 mg total) by mouth daily. authorized by: lippard, giles a, aprn ordered on: 10/25/2022 { Donella  start date: 10/25/2022 quantity: 90 tablet refill: 3 refills by 10/25/2023 cetirizine 10 { Krishna  mg tabl { Miriah et (zyrtec) discontinued by: lippard, giles a, apr { Jo n discontinued on: 8/8/2023 reason for discontinuation: reorder instructions: take 1 tablet (10 mg total) by mouth once a day as needed for allergies. authorized by: lippard, giles a, aprn ordered on: 10/25 { Charlena /2022 start date: 10/25/2022 quantity: 30 { Ayman  tablet refill: 11 refills by 10/25/2023 sennosides 8.6 mg tablet (senokot) discontinued by: chanthavong, serena discont { Emily inued on: 1/11/2023 r { Charlyn eason for discontinuation: therapy c { Annah ompleted (cancelrx) instructions: take 1 tabl { Monnie et by mouth every night. authorized by: lippard, g { Isabel iles a, aprn ordered { Andreana  on { Florinda : 10/25 { Rani /2022 start date: 10/25 { Brylie /202 { Goldia 2 end date: 1/11/2023 qua { Kyler ntity: 30 tablet refill: 1 refill by 10/25/2023 docusate sodium 100 mg ca { Cloey psule (colace) discontinued by: de witte, anton jordan, md discontin { Ellamae ued on: 12/6/2023 reason for discontinuation: cleanup(notavs) instructions: { Karlos   { Enriqueta take one tablet tid prn constipation authorized by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 end date: 12/6/2023 quantity: 60 capsule refill:  remain { Isacc ing fluticasone propionate 50 mcg/actuation nasal spray,suspension (flonase) discontinued by: lippard, giles a, aprn discontinued on: 6/6/2023 reason for discontin { Taha uation: reorder printed on 10/3/24 7:13 am page 2751,vumc hendersonville - anderson white, tyrone 128 n anderson ln mrn: 047717361, dob { Vergil : 7/27/1969, legal sex: m hendersonville tn 37075 visit date: 1/10/2023 01/10/2023 - nurse triage in vanderbilt primary care hen { Michaele dersonville (continued) medicatio { Ramond n list (continued) instructions: admin { Keysha ister 2 sprays into each nostril daily. authorize { Jadelyn d by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quantity: 16 g refill: 11 refills by 10/25/2023 azelastine 137 mcg (0.1 %) nasal spray aerosol (astelin) discontinued by: { Lorina  lippard, giles a, aprn discont { Dalen inued on: 8/8/20 { Deryl 23 reason for discontinuation: reorder instructions: administer 1 { Rayleigh  spray into ea { Dontavious ch  { Garren nostril 2 times a day. use in each nostril as directed authoriz { Hershell ed by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022  { Romello quantity: 30 ml refill: 12 refills by 10/25/2023 lidocaine 5 % topical patch ( { Dyan lidoderm) discontinued by: lehmann, meliss { Zoee a cary, pa-c discontinued on: 6/3/202 { Hadi 3 { Kolin  reason for discontinuation: stop taking at discharge (cancelrx) instructions: apply 1 patch topically daily. apply { Keaira  to painful area 12 hours per da { Ralph y, remove for 12 hours. authorized by: lippard, giles a, aprn or { Kalin dered on: 10/25/2022 start date: 10/25/2022 end date: 6/3/2023 action: patient not taking quantity: 3 { Anel 0 patch refill: 11 refills by 10/25/2023 albuterol sulfate hfa 90 mcg/actuation aerosol inhaler discontinued by: lippard, giles a, aprn discontinued on: 8/8/2023 reason for disconti { Jenson nuation: reorder in { Adara structions: inhale 2 puffs every 4 hours as needed for wheezing. autho { Azael rized by: lippard, giles a, aprn { Carlin  ordered on: 10/25/2022 start date: 10/25/20 { Darien 22 quantity: 18 g refill: 11 refills by 10/25/2023  { Eian pantoprazole 20 mg tablet,delayed release (protonix)  { Jeronimo disc { Randon ontinued by: greenspan, debra l, aprn discontinued on: 7/9/2024 reason for discontinuation: reorder instructions: take 1 tablet (20 mg total) by { Brianda  mouth daily. authorized by: lippard, giles a, aprn ordered on: 11/29/2022 start date: 11/29/2022 end date: 7/9/2024 quantity: 30 tablet refill: 11 refills  { Safa by 11/29/2023 neosporin (neo-bac-polym) 3.5 mg-400 unit-5,000 unit/gram to { Alease p ointment (neomycin-bacitracn { Brooklynne zn-polymyxnb) discontinued by: chanth { Leyna avong,  { Reda serena di { Dedric sconti { Hildegard nued on: 1/11/2023 reason for discontinuation: therapy completed (cancelrx { Neida ) instructions: apply 1 application topically 2  { Nyssa times a day for 30 days. authorize { Veola d by: lippard, giles a, aprn ordered on: 12/6/2022 start date { Drayton : 12/6/2022 end date: 1/11/2023 quantity: 15 g refill:  remaining sodium chloride 0.65 % nasal spray aerosol (ocean nasal) discontinued by: lehmann,  { Jourdan melissa cary, pa-c disco { Sabino ntinued on: 6/3/2023 reason for discontinuation: stop taking at discharge (cancelrx) instructions: administer 1 spray into each nostril as needed for rhinitis. authorized by: lippard, giles a, aprn ordered  { Fantasia on: 12/6/2022 start date: 12/6/2022 end date: 6/3/2023 quantity: 15 m { Shalanda l refi { Zacharias ll: 12 refills by 12/6/2023 acetaminophen 325 mg tablet (tyl { Cassey enol) discontinued by: le { Anand hmann, melissa cary, pa { Jakai -c discontinued on: 6/3/2023 reason for discontinuation: reorder instructions: take 2 tablets (650 mg  { Shamus total) by mouth every 6 hours as ne { Adamari eded for mild pain. authorized by: lippard, giles a, aprn ordered on: 12/9/2022 start date: 12/9/2022 en { Delwin d date: 6/3/20 { Babygirl 23 action:  { Bryan patient not taking quantity: 30  { Kyanna tablet printe { Esai d on 10/3/24 7:13 am page 2 { Codie 752",
    {
        "entities": [
            [
                72,
                80,
                "PERSON"
            ],
            [
                160,
                165,
                "PERSON"
            ],
            [
                211,
                218,
                "PERSON"
            ],
            [
                330,
                340,
                "PERSON"
            ],
            [
                419,
                427,
                "PERSON"
            ],
            [
                435,
                441,
                "PERSON"
            ],
            [
                456,
                463,
                "PERSON"
            ],
            [
                530,
                539,
                "PERSON"
            ],
            [
                588,
                596,
                "PERSON"
            ],
            [
                697,
                704,
                "PERSON"
            ],
            [
                815,
                821,
                "PERSON"
            ],
            [
                914,
                921,
                "PERSON"
            ],
            [
                965,
                971,
                "PERSON"
            ],
            [
                1033,
                1040,
                "PERSON"
            ],
            [
                1137,
                1145,
                "PERSON"
            ],
            [
                1159,
                1163,
                "PERSON"
            ],
            [
                1195,
                1202,
                "PERSON"
            ],
            [
                1233,
                1238,
                "PERSON"
            ],
            [
                1538,
                1547,
                "PERSON"
            ],
            [
                1577,
                1583,
                "PERSON"
            ],
            [
                1909,
                1917,
                "PERSON"
            ],
            [
                2009,
                2017,
                "PERSON"
            ],
            [
                2028,
                2035,
                "PERSON"
            ],
            [
                2088,
                2091,
                "PERSON"
            ],
            [
                2300,
                2309,
                "PERSON"
            ],
            [
                2353,
                2359,
                "PERSON"
            ],
            [
                2482,
                2488,
                "PERSON"
            ],
            [
                2512,
                2520,
                "PERSON"
            ],
            [
                2559,
                2565,
                "PERSON"
            ],
            [
                2613,
                2620,
                "PERSON"
            ],
            [
                2673,
                2680,
                "PERSON"
            ],
            [
                2703,
                2712,
                "PERSON"
            ],
            [
                2718,
                2727,
                "PERSON"
            ],
            [
                2737,
                2742,
                "PERSON"
            ],
            [
                2768,
                2775,
                "PERSON"
            ],
            [
                2782,
                2789,
                "PERSON"
            ],
            [
                2817,
                2823,
                "PERSON"
            ],
            [
                2899,
                2905,
                "PERSON"
            ],
            [
                2976,
                2984,
                "PERSON"
            ],
            [
                3062,
                3069,
                "PERSON"
            ],
            [
                3073,
                3083,
                "PERSON"
            ],
            [
                3263,
                3269,
                "PERSON"
            ],
            [
                3435,
                3440,
                "PERSON"
            ],
            [
                3578,
                3585,
                "PERSON"
            ],
            [
                3716,
                3725,
                "PERSON"
            ],
            [
                3761,
                3768,
                "PERSON"
            ],
            [
                3809,
                3816,
                "PERSON"
            ],
            [
                3868,
                3876,
                "PERSON"
            ],
            [
                4075,
                4082,
                "PERSON"
            ],
            [
                4116,
                4122,
                "PERSON"
            ],
            [
                4141,
                4147,
                "PERSON"
            ],
            [
                4215,
                4224,
                "PERSON"
            ],
            [
                4241,
                4252,
                "PERSON"
            ],
            [
                4258,
                4265,
                "PERSON"
            ],
            [
                4331,
                4340,
                "PERSON"
            ],
            [
                4419,
                4427,
                "PERSON"
            ],
            [
                4508,
                4513,
                "PERSON"
            ],
            [
                4558,
                4563,
                "PERSON"
            ],
            [
                4603,
                4608,
                "PERSON"
            ],
            [
                4612,
                4618,
                "PERSON"
            ],
            [
                4736,
                4743,
                "PERSON"
            ],
            [
                4778,
                4784,
                "PERSON"
            ],
            [
                4851,
                4857,
                "PERSON"
            ],
            [
                4961,
                4966,
                "PERSON"
            ],
            [
                5150,
                5157,
                "PERSON"
            ],
            [
                5179,
                5185,
                "PERSON"
            ],
            [
                5258,
                5264,
                "PERSON"
            ],
            [
                5299,
                5306,
                "PERSON"
            ],
            [
                5353,
                5360,
                "PERSON"
            ],
            [
                5414,
                5419,
                "PERSON"
            ],
            [
                5475,
                5484,
                "PERSON"
            ],
            [
                5491,
                5498,
                "PERSON"
            ],
            [
                5645,
                5653,
                "PERSON"
            ],
            [
                5812,
                5817,
                "PERSON"
            ],
            [
                5894,
                5901,
                "PERSON"
            ],
            [
                5934,
                5945,
                "PERSON"
            ],
            [
                5985,
                5991,
                "PERSON"
            ],
            [
                6001,
                6006,
                "PERSON"
            ],
            [
                6018,
                6025,
                "PERSON"
            ],
            [
                6034,
                6044,
                "PERSON"
            ],
            [
                6121,
                6127,
                "PERSON"
            ],
            [
                6178,
                6184,
                "PERSON"
            ],
            [
                6221,
                6227,
                "PERSON"
            ],
            [
                6291,
                6299,
                "PERSON"
            ],
            [
                6451,
                6459,
                "PERSON"
            ],
            [
                6486,
                6493,
                "PERSON"
            ],
            [
                6702,
                6711,
                "PERSON"
            ],
            [
                6783,
                6792,
                "PERSON"
            ],
            [
                6801,
                6811,
                "PERSON"
            ],
            [
                6874,
                6881,
                "PERSON"
            ],
            [
                6909,
                6915,
                "PERSON"
            ],
            [
                6941,
                6947,
                "PERSON"
            ],
            [
                7052,
                7059,
                "PERSON"
            ],
            [
                7097,
                7105,
                "PERSON"
            ],
            [
                7212,
                7219,
                "PERSON"
            ],
            [
                7236,
                7245,
                "PERSON"
            ],
            [
                7259,
                7265,
                "PERSON"
            ],
            [
                7300,
                7307,
                "PERSON"
            ],
            [
                7323,
                7328,
                "PERSON"
            ],
            [
                7358,
                7364,
                "PERSON"
            ]
        ]
    }
),(
    "vumc h { Ethelene endersonville { Scottie  - ander { Nahum son white, tyrone 128 n anderson ln mrn: 047717361, dob: 7/27 { Darryn /1969, legal sex: m hendersonville tn 37075 visit date:  { Hayli 3/6/2023 03/06/2023 - office visit in vanderbilt primary care hendersonville (continued) clinical notes (continued) docusate sodium 100 mg capsu { Adora le (colace), take one ta { Julieanne blet tid p { Karely rn constipation, disp: 60 capsule, rfl: 0 easy touch 31 gauge x 1/4\"  { Trevis needle,  { Maja use as directed to inject insulin every day, disp: rfl: { Kanesha  ergocalciferol { Ragan  (vitamin d2) 1,250 mcg (50,000 unit) capsule (vitami { Kena n d2), take 1  { Myrl capsule (50,000 units tot { Semaj al) by mouth weekly., disp: 12 capsule, rfl: 0 famotidine 20 mg tabl { Cort et (pepcid), take 1 tablet (20 mg total) by mouth every 12 hours { Calandra ., disp: rfl: f { Elisia luticasone propionate 50 mcg/actuation nasal spray,suspension  { Jiya (flonase), administer 2 sprays into each nostril daily., dis { Mertie p: 16 g, rfl: 11 freestyle libre 2 sensor kit (flash glucose sensor), 1 kit (1 each total) every 14 days., disp: rfl: gabapentin 300 mg capsule (neurontin), take one  { Mickayla capsule by mouth three times a day, disp: 90 c { Naomie apsule { Shalom , rfl: 0 lancets 33 gauge (trueplus lancets), use as directed., disp: 120 each, rfl: 11 montelukast 10 mg tab { Bari let (singulair), take 1 tablet (10 mg total) by mouth eve { Ebonee ry evening., disp { Jannah : 30 tablet, rfl: 0 pantoprazole 20 mg tablet, delayed release { Dymond  (p { Emiliana rotonix), ta { Markanthony ke 1 tablet (20 mg total) by mouth daily., disp: 30 tablet, rfl: 11 { Brynleigh  rybels { Maisha us 14 mg tablet (semaglutide), t { Emili ake 1 tablet (14 mg t { Jamison otal) by mouth daily., disp: 90 tablet, r { Liesl fl: 3 sodium chloride 0.65 nasal spray aerosol (ocean nasal), { Renzo  administer 1 spray into each nos { Merideth tril  { Rickie as needed for rhinitis., disp: 15 ml, rfl: 12 acetamin { Cooper ophen 325 mg tablet (tylenol), take 2 tablets (650 mg total) by mout { Marbella h every 6 hours as nee { Sharilyn ded for mild pain. (patient not taking: rep { Huxley orted on 3/6/2023), disp: 30 tablet, rfl: 0 lidocaine 5 % topical patch (lidoderm), apply 1 patch top { Vivek ically daily. apply to painful area 12 hours per day, remove for 12 hours. (pa { Demetrice tient not taking: reported on 3/6/2023), disp { Keya : 30 patch, r { Mozella fl: 11 ondansetron hcl 4 mg tablet (zofran), take 1 tablet ( { Ressie 4 mg total) by  { Aren mouth 3 times a day as needed for n { Arion ausea or vomiting. (patient not taking: reported on 3/ { Darl 6/202 { Ivie 3), disp: 20 tablet, rfl: 0 pen needle, diabetic 31 gauge x 3/16\" { Eitan , us { Kallen e as directed. to inject insulin on { Lakin ce d { Rheanna aily (patient not taking: reported on 3/6/2023), disp: 90 each, rfl: 3 pen needle, diabetic 31 gauge x 5/16\", bd { Ayush  ultra-fine short pen nee { Greysen dle 31 gauge x 5/16\" use as  { Kamil directed  { Annice 4 times daily (patient not taking: reported on 3/6/20 { Chelsee 23), disp: rfl: social h { Georgeanna istory social h { Marygrace istory narrative  { Shaylin lives in his home with daughter physical exam bp 137/90 i { Hazen  puls { Bunny e { Demetrius  84 i t { Genevie emp 36.8 °c (98.2 °f) i ht 1 { Zoila 82.9 cm (72\" { Jomar ) i wt 85.7 kg (189 lb) i spo2 99% bmi 25.63 kg/m² wt readings from last 3 encounters: 03/06/23 85.7 kg (189 lb) 02/14/23 86.5 kg (190 lb 11.2 oz) 02/07/23 89.8 kg (198 lb) physical exam vitals reviewe { Vivaan d. constitutional: appearance: normal appearance. hent: nose: rhinorrhea present. cardiovascular: rate and rhythm: normal rate and regular rhythm. printed on 10/3/24 7:1 { Estephanie 3 am page 2543,vumc hendersonville - anderson white, t { Kamarion yrone 128 n anderson ln m { Britnee rn: 047717361, dob: 7/27/1969, legal sex: m hender { Jadah sonville tn 37075 visit date: 3/6/2023 03/06/2023 - office visit in vanderbilt primary care hendersonville (continued) clinical notes (continued) pulses: normal pulses.  { Lovely heart sounds: no { Dustan rmal heart { Robbin  sounds. pulmonary: effort: pulmonary effort is normal. brea { Jaina th sounds { Kennadi : normal breath sounds. musculo { Krystyna skeletal: general: nor { Jayse mal range of motion. right lower leg: . left lower le { Liyah g: . skin: general: skin is warm and dry. capillary refill: capillary refill tak { Caridad es { Desiray  less than 2 seconds.  { Elodia neurological: general:  deficit present. mental status: he is alert and oriented to person, place, and tim { Irie e. psychiatric: mood and affect: mood normal. assessment/plan: white, tyrone is a 53 y.o. male who presents today for chief co { Cally mplain { Marwa t patient p { Sanya resents with return problem list items addressed this visit nervous neuro { Ettie pathy due to type 2 diabetes mellitus (cms/hcc) treatment effective at current dose  side effects  { Lakita cs md checked and aligns with story continue treatmen { Rasheeda t plan will continue to monit { Janely or gen { Tai itourinary type 2 diabet { Harlow es mellit { Huy us with chronic kidney dis { Lyndi ease, with long-term current use of insulin (cms/hcc) continue home meds continue with endocrinology we will continue to mon { Quran itor relevant orders ambulatory referral to ophthalmology kidney di { Annastasia sease, chronic, st { Raeanne age iv (gfr 15-29 ml/min) (cms/hcc) - primary continue with ne { Carmello phrology repeat bmp printed on 10/3/24 7:13 am page 2544",
    {
        "entities": [
            [
                9,
                18,
                "PERSON"
            ],
            [
                34,
                42,
                "PERSON"
            ],
            [
                53,
                59,
                "PERSON"
            ],
            [
                123,
                130,
                "PERSON"
            ],
            [
                189,
                195,
                "PERSON"
            ],
            [
                342,
                348,
                "PERSON"
            ],
            [
                375,
                385,
                "PERSON"
            ],
            [
                398,
                405,
                "PERSON"
            ],
            [
                477,
                484,
                "PERSON"
            ],
            [
                495,
                500,
                "PERSON"
            ],
            [
                558,
                566,
                "PERSON"
            ],
            [
                584,
                590,
                "PERSON"
            ],
            [
                646,
                651,
                "PERSON"
            ],
            [
                668,
                673,
                "PERSON"
            ],
            [
                701,
                707,
                "PERSON"
            ],
            [
                778,
                783,
                "PERSON"
            ],
            [
                850,
                859,
                "PERSON"
            ],
            [
                877,
                884,
                "PERSON"
            ],
            [
                949,
                954,
                "PERSON"
            ],
            [
                1017,
                1024,
                "PERSON"
            ],
            [
                1193,
                1202,
                "PERSON"
            ],
            [
                1251,
                1258,
                "PERSON"
            ],
            [
                1267,
                1274,
                "PERSON"
            ],
            [
                1386,
                1391,
                "PERSON"
            ],
            [
                1451,
                1458,
                "PERSON"
            ],
            [
                1478,
                1485,
                "PERSON"
            ],
            [
                1550,
                1557,
                "PERSON"
            ],
            [
                1563,
                1572,
                "PERSON"
            ],
            [
                1587,
                1599,
                "PERSON"
            ],
            [
                1669,
                1679,
                "PERSON"
            ],
            [
                1689,
                1696,
                "PERSON"
            ],
            [
                1731,
                1737,
                "PERSON"
            ],
            [
                1761,
                1769,
                "PERSON"
            ],
            [
                1813,
                1819,
                "PERSON"
            ],
            [
                1883,
                1889,
                "PERSON"
            ],
            [
                1925,
                1934,
                "PERSON"
            ],
            [
                1942,
                1949,
                "PERSON"
            ],
            [
                2006,
                2013,
                "PERSON"
            ],
            [
                2084,
                2093,
                "PERSON"
            ],
            [
                2118,
                2127,
                "PERSON"
            ],
            [
                2173,
                2180,
                "PERSON"
            ],
            [
                2284,
                2290,
                "PERSON"
            ],
            [
                2371,
                2381,
                "PERSON"
            ],
            [
                2429,
                2434,
                "PERSON"
            ],
            [
                2450,
                2458,
                "PERSON"
            ],
            [
                2521,
                2528,
                "PERSON"
            ],
            [
                2546,
                2551,
                "PERSON"
            ],
            [
                2589,
                2595,
                "PERSON"
            ],
            [
                2652,
                2657,
                "PERSON"
            ],
            [
                2665,
                2670,
                "PERSON"
            ],
            [
                2738,
                2744,
                "PERSON"
            ],
            [
                2751,
                2758,
                "PERSON"
            ],
            [
                2796,
                2802,
                "PERSON"
            ],
            [
                2809,
                2817,
                "PERSON"
            ],
            [
                2932,
                2938,
                "PERSON"
            ],
            [
                2966,
                2974,
                "PERSON"
            ],
            [
                3005,
                3011,
                "PERSON"
            ],
            [
                3023,
                3030,
                "PERSON"
            ],
            [
                3086,
                3094,
                "PERSON"
            ],
            [
                3121,
                3132,
                "PERSON"
            ],
            [
                3150,
                3160,
                "PERSON"
            ],
            [
                3180,
                3188,
                "PERSON"
            ],
            [
                3248,
                3254,
                "PERSON"
            ],
            [
                3262,
                3268,
                "PERSON"
            ],
            [
                3272,
                3282,
                "PERSON"
            ],
            [
                3292,
                3300,
                "PERSON"
            ],
            [
                3331,
                3337,
                "PERSON"
            ],
            [
                3352,
                3358,
                "PERSON"
            ],
            [
                3562,
                3569,
                "PERSON"
            ],
            [
                3741,
                3752,
                "PERSON"
            ],
            [
                3809,
                3818,
                "PERSON"
            ],
            [
                3846,
                3854,
                "PERSON"
            ],
            [
                3907,
                3913,
                "PERSON"
            ],
            [
                4085,
                4092,
                "PERSON"
            ],
            [
                4111,
                4118,
                "PERSON"
            ],
            [
                4131,
                4138,
                "PERSON"
            ],
            [
                4201,
                4207,
                "PERSON"
            ],
            [
                4219,
                4227,
                "PERSON"
            ],
            [
                4261,
                4270,
                "PERSON"
            ],
            [
                4295,
                4301,
                "PERSON"
            ],
            [
                4357,
                4363,
                "PERSON"
            ],
            [
                4446,
                4454,
                "PERSON"
            ],
            [
                4459,
                4467,
                "PERSON"
            ],
            [
                4492,
                4499,
                "PERSON"
            ],
            [
                4608,
                4613,
                "PERSON"
            ],
            [
                4742,
                4748,
                "PERSON"
            ],
            [
                4757,
                4763,
                "PERSON"
            ],
            [
                4777,
                4783,
                "PERSON"
            ],
            [
                4859,
                4865,
                "PERSON"
            ],
            [
                4966,
                4973,
                "PERSON"
            ],
            [
                5029,
                5038,
                "PERSON"
            ],
            [
                5070,
                5077,
                "PERSON"
            ],
            [
                5086,
                5090,
                "PERSON"
            ],
            [
                5117,
                5124,
                "PERSON"
            ],
            [
                5136,
                5140,
                "PERSON"
            ],
            [
                5169,
                5175,
                "PERSON"
            ],
            [
                5302,
                5308,
                "PERSON"
            ],
            [
                5378,
                5389,
                "PERSON"
            ],
            [
                5410,
                5418,
                "PERSON"
            ],
            [
                5483,
                5492,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital white, tyrone 1211  { Lizzette medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 8/14/2024, d/c: 8/15/2024 08/14/2024 - ed in vanderbilt emergency department (continued) ed care timeline (continued) 23:33:13 egfrcr resulted abnormal result collected: 8/14/2024 23:08 last updated: 8/14/2024 23:33 interface, lab status: final result egfrcr: 17 ml/min/1.73 m2 [ref range: >=60] (the results in eg { Luanna frcr was calculated using the  { Shirleen 2021 ckd-epi egfr creatinine equation, which does not inclu { Arcelia de race as a factor. this equation is validated in individuals 18 years of age and older. these changes { Phaedra  went into effect on 12/7/22 and due to the new equation, will not be trended  { Nunzio with older egfr. values should be { Etha  inte { Legacy rpreted { Alvah  in the context of the patie { Korben nt's full clinical presentation. refe { Mattea rence: del { Vernal gado, cynthia, et al. \"a unifying approach for gfr estimation: recommendations of the nkf-asn task force on reassessing the inclusion of race in diagnosing kidney disease.\" ameri { Clora can journal of kidney diseases (2021) gfr categories in chronic kidney disease ( { Julia ckd) gfr gfr (ml/min/1.73 category: square meters) interpretation: g1 90 or greater normal or high* g2 60-89 mild decrease* g3a 45-59 mild  { Quincey to moderate decrease g3b 30-44 moderate t { Hortensia o severe dec { Lavona rease g4 15-29 severe decrease g5 14 or less kid { Loretha ney failure *in  { Stephine the absence of { Verena  evidence of kidney damage, nei { Ivey ther gfr cat egory g1 nor g2 fulfill  { Miki the criteria for ckd (kidney int suppl { Tynisha  2013;3:1-150) this test was performed at: vanderbilt hospital la { Carrington boratory { Aminata , clia #44d0659066,adam seegmiller md, phd, 1301 { Suzannah  medical center drive, 4605 tvc,nashville,tn,37232, ) 23 { Abhinav :33:14 orders placed nursing - poc lab: glucose; source: capillary rupp, jordan  { Adiel douglas, md 23:34:20 orders placed lab - potassium lvl pauw, emily kathryn, md 23:34:22 lab order { Nilsa ed pota { Malakhi ssium lvl pauw, emily kathryn, md 23:34: { Cinthya 43 orders new - poc lab: glucose; source: capillary; potassium lvl  { Madelon mcquitty, karrah, acknowledged rn 23:40 { Harlon  ed qu { Lindy ick updates quic { Xavi k updates mcquitty, karrah, updates: patien { Malique t is r { Alita esting comfortably; family at bedside rn 23:42:30 print la { Gertha bel for potassium lvl - type: blood mcquitty, karrah, potassium lvl rn completed 23:42:55 remove resident morrow, seyjil shantha turpin, md removed as resident morrow, seyjil shantha turpin, md 23:49:52 urinalysis abnormal result { Helaine  collected: 8/14/202 { Grayden 4  { Kaiser 23:08 last updated: 8/14/2024 23:49 interface, lab w/micro&rfx status: final result urine color: colorless urine appearance { Oziel : clear urine results in culture resulted specific gravity: 1.004 [ref range: 1.015 - 1.025] urine ph: 7.0 [ref range: 4.6 - 7.8] urine glucose: 1000 mg/dl ! [ref range: negative] urine protein: 50 mg/dl ! [ref range: negative] urine ketones: negative mg/dl [ref range: negative] urine bilirubin: negative [ref range: negative] urine urobilinogen: <2 mg/dl [ref range: <2] urine leukocyte esterase: negative [ref range: negative] urine nitrite: negative [ref { Zendaya  range: negative] urine blood: small ! { Gisell  [ref range { Tirzah : negative] (this test was performed at: vanderbilt hospital laboratory, clia #44d0659066, adam seegmiller md, phd, 13 { Van 01 me { Nyle dical center drive, 4605 tvc, nashville,  { Haidyn tn, 3723 { Jacquetta 2,) 23:49:55 lab resulted (final result) urin { Onie alysis w/ micro&rfx culture interface, lab results in 23:49:55 critical alert interface, lab triggered results in printed on 10/3/24 7:12 am page 285,vum { Wonda c adult hosp { Jashua ital white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 8/14/2024, d/c: 8/15/2024 08/14/2024 - ed in v { Athan anderbilt em { Amiee ergency department (continued) e { Andriana d care timeline (continued) 23:49:55 collect urinalysis urinalysis w/micro&rfx culture interface, lab w/micro&rfx results in culture discontinued  { Shilo 23:49:55 print label for urinalysis w/micro&rfx culture interface, lab urinalysis results  { Basilio in w/micro&rfx culture discon { Cherryl ti { Shanon nued 2 { Tavin 3:50 coll { Shayan ect potassium potassium lvl - type: blood mcquitty, karrah, ivl completed rn 23:50 specimens potassium lvl - id: 24-227-015216 type: blood mcquitty, karrah, collected rn 23:50:56 critical alert in { Tyrique terface,  { Analy lab triggered re { Louanne sults i { Marinda n 23:51:07 poc lab: glucose; poc lab: glucose; source: capillary mcquitty, karrah, source: capillary rn completed 23:52:13 urinalysis abnormal result co { Deklan llected: 8/14/2024 23:08 last updated: 8/14/2024 23:52 interface, lab microscopic status: final result urine wbc: <1 /hpf [ref range { Obadiah : 0 - 10] urine rbc: 5 results in resulted /hpf [ref range: 0 4] urine bacteria: n { Shelley one seen /hpf [ref range: none seen] (this test was performed at: vanderbilt hospital laboratory, clia #44d0659066,adam seegmiller md, phd,1301 medical center drive, 4605 tvc,nashville,tn,37232,) 23:52:15 lab resulted (final result) urinalys { Gala is microscopic interface, lab results in 23:52:15 critical alert interface, lab trigge { Francois red results in 23:56 specimens poc glucose-bedsid { Najah e meter - id: 24-2 { Gloria 27-0 { Harman 15253 type: blood collected 23:5 { Aime 7:38 poc lab: glucose; poc lab: glucose; source: capillary mcquitty, karrah, source: capillary rn completed 23:58:10 poc glucose- abnormal result collected: 8/14/2024 23:56 last updated: 8/14/2024 23:58 interface, lab bedside meter status: final result patient location poc: w800 glucose poc (glub): 405 results in resul { Amee ted mg/dl [ref range:  { Emiley 70 - { Jennyfer  99] 23:58:12 critical alert interface, lab triggered results  { Shaelynn in 23:59:15 orders placed nursing - poc lab: glu { Britton cose; source: capillary; poc lab: gluco { Loralee se; source: sobolews { Aliyana ki, capillary { Billi  rebecca ashley, medications - juice for hypoglycemia manageme { Rossie nt 4 oz; glucose chew { Elizabet able  { Flavia md tablet 16 g; dextrose (d50w) 50 % in { Brodrick jection 25 ml; glucagon (human recombinant) injection 1 mg; insulin lispro 1-6 units admelog - biosimilar for humalog injection 0.01-0.06 ml  { Marquan 8/15/2024 event details user 00:00 sofa sofa epic,  { Saif user sofa score (do not  { Sharmaine edit): 3 00:00:11 remove attending rupp, jordan douglas, md removed as attending rupp, jordan  { Arin douglas, md 00:01:33 orders poc  { Bryden lab: glucose; source: capillary (08/14/24 2334) poc lab: glucose; sobolewski, discontinued source: capillary (08/15/24  { Brittnay 0000) ; poc lab: glucose; source: capillary ; rebecca ashley, poc lab: glucose; source: capillary { Cord  ; juice for  { Hiroshi hypoglycemia management 4 md { Tyrin  oz ; glu { Cambrie cose chewable ta { Ieshia blet 16 ; dextr { Kilee ose (d50w) 50 % injection 25 ml ; glucagon { Lainie  (human recombinant) injection 1 mg; insul { Laurice in lispro 1-6 units admelog - biosimilar for humalog injection 0.01-0.06 ml printed on 10/3/24 7:12 am page 286",
    {
        "entities": [
            [
                42,
                51,
                "PERSON"
            ],
            [
                461,
                468,
                "PERSON"
            ],
            [
                501,
                510,
                "PERSON"
            ],
            [
                572,
                580,
                "PERSON"
            ],
            [
                686,
                694,
                "PERSON"
            ],
            [
                775,
                782,
                "PERSON"
            ],
            [
                818,
                823,
                "PERSON"
            ],
            [
                831,
                838,
                "PERSON"
            ],
            [
                848,
                854,
                "PERSON"
            ],
            [
                885,
                892,
                "PERSON"
            ],
            [
                932,
                939,
                "PERSON"
            ],
            [
                952,
                959,
                "PERSON"
            ],
            [
                1140,
                1146,
                "PERSON"
            ],
            [
                1229,
                1235,
                "PERSON"
            ],
            [
                1377,
                1385,
                "PERSON"
            ],
            [
                1429,
                1439,
                "PERSON"
            ],
            [
                1454,
                1461,
                "PERSON"
            ],
            [
                1512,
                1520,
                "PERSON"
            ],
            [
                1539,
                1548,
                "PERSON"
            ],
            [
                1565,
                1572,
                "PERSON"
            ],
            [
                1606,
                1611,
                "PERSON"
            ],
            [
                1651,
                1656,
                "PERSON"
            ],
            [
                1697,
                1705,
                "PERSON"
            ],
            [
                1773,
                1784,
                "PERSON"
            ],
            [
                1795,
                1803,
                "PERSON"
            ],
            [
                1854,
                1863,
                "PERSON"
            ],
            [
                1922,
                1930,
                "PERSON"
            ],
            [
                2013,
                2019,
                "PERSON"
            ],
            [
                2119,
                2125,
                "PERSON"
            ],
            [
                2135,
                2143,
                "PERSON"
            ],
            [
                2186,
                2194,
                "PERSON"
            ],
            [
                2264,
                2272,
                "PERSON"
            ],
            [
                2314,
                2321,
                "PERSON"
            ],
            [
                2330,
                2336,
                "PERSON"
            ],
            [
                2355,
                2360,
                "PERSON"
            ],
            [
                2406,
                2414,
                "PERSON"
            ],
            [
                2423,
                2429,
                "PERSON"
            ],
            [
                2490,
                2497,
                "PERSON"
            ],
            [
                2729,
                2737,
                "PERSON"
            ],
            [
                2760,
                2768,
                "PERSON"
            ],
            [
                2773,
                2780,
                "PERSON"
            ],
            [
                2906,
                2912,
                "PERSON"
            ],
            [
                3373,
                3381,
                "PERSON"
            ],
            [
                3422,
                3429,
                "PERSON"
            ],
            [
                3443,
                3450,
                "PERSON"
            ],
            [
                3571,
                3575,
                "PERSON"
            ],
            [
                3583,
                3588,
                "PERSON"
            ],
            [
                3632,
                3639,
                "PERSON"
            ],
            [
                3650,
                3660,
                "PERSON"
            ],
            [
                3708,
                3713,
                "PERSON"
            ],
            [
                3869,
                3875,
                "PERSON"
            ],
            [
                3890,
                3897,
                "PERSON"
            ],
            [
                4063,
                4069,
                "PERSON"
            ],
            [
                4084,
                4090,
                "PERSON"
            ],
            [
                4125,
                4134,
                "PERSON"
            ],
            [
                4283,
                4289,
                "PERSON"
            ],
            [
                4382,
                4390,
                "PERSON"
            ],
            [
                4422,
                4430,
                "PERSON"
            ],
            [
                4435,
                4442,
                "PERSON"
            ],
            [
                4451,
                4457,
                "PERSON"
            ],
            [
                4469,
                4476,
                "PERSON"
            ],
            [
                4675,
                4683,
                "PERSON"
            ],
            [
                4695,
                4701,
                "PERSON"
            ],
            [
                4720,
                4728,
                "PERSON"
            ],
            [
                4738,
                4746,
                "PERSON"
            ],
            [
                4901,
                4908,
                "PERSON"
            ],
            [
                5043,
                5051,
                "PERSON"
            ],
            [
                5136,
                5144,
                "PERSON"
            ],
            [
                5388,
                5393,
                "PERSON"
            ],
            [
                5482,
                5491,
                "PERSON"
            ],
            [
                5543,
                5549,
                "PERSON"
            ],
            [
                5570,
                5577,
                "PERSON"
            ],
            [
                5584,
                5591,
                "PERSON"
            ],
            [
                5626,
                5631,
                "PERSON"
            ],
            [
                5954,
                5959,
                "PERSON"
            ],
            [
                5984,
                5991,
                "PERSON"
            ],
            [
                5998,
                6007,
                "PERSON"
            ],
            [
                6072,
                6081,
                "PERSON"
            ],
            [
                6132,
                6140,
                "PERSON"
            ],
            [
                6182,
                6190,
                "PERSON"
            ],
            [
                6213,
                6221,
                "PERSON"
            ],
            [
                6237,
                6243,
                "PERSON"
            ],
            [
                6308,
                6315,
                "PERSON"
            ],
            [
                6339,
                6348,
                "PERSON"
            ],
            [
                6356,
                6363,
                "PERSON"
            ],
            [
                6405,
                6414,
                "PERSON"
            ],
            [
                6558,
                6566,
                "PERSON"
            ],
            [
                6620,
                6625,
                "PERSON"
            ],
            [
                6652,
                6662,
                "PERSON"
            ],
            [
                6759,
                6764,
                "PERSON"
            ],
            [
                6799,
                6806,
                "PERSON"
            ],
            [
                6928,
                6937,
                "PERSON"
            ],
            [
                7037,
                7042,
                "PERSON"
            ],
            [
                7058,
                7066,
                "PERSON"
            ],
            [
                7097,
                7103,
                "PERSON"
            ],
            [
                7115,
                7123,
                "PERSON"
            ],
            [
                7142,
                7149,
                "PERSON"
            ],
            [
                7167,
                7173,
                "PERSON"
            ],
            [
                7218,
                7225,
                "PERSON"
            ],
            [
                7270,
                7278,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospi { Shaindy tal white, tyrone 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn { Sima  37232-0004 adm: 6/2/2023, d/c: 6/3/2023 06/02/2023 - ed to { Taytum  hosp-admission (discharged) in vanderbilt university adult hospital (contin { Ocean ued) ed note { Syncere s (continued) electronically signed by pauw, emily kathryn, md at 6/2/2023 9:30 pm { Kimiko  electronically signed by jordan, mary kate, md at 6/2/202 { Giorgio 3 10:13 pm ed care timeline patient care timeline (6/2/2023 13:54 to 6/3/2023 11:09) 6/2/2023 event details us { Latoyia er 13:54 patient arrived in olfa { Sunnie ti, saghar ed  { Kizzie 13:54:19 emergency olfati, saghar encounter created 13:54:33 { Shanique  arrival complaint sl { Yuna urred speach 13:57:11 ed triage notes per pt his speech was slurred around 0600 today and yesterday greenwood- he had trouble walking. bs  { Drayden at home was 94. per pcp office this simpson, eliz { Jiovanni abeth, rn m { Darlyn ay have starte { Devion d 3 hs ago while he was at a barber shop. pt is having trouble communicating in triage. md here for evaluation. 14: { Leron 00 travel screening do you have any of the fo { Anali llowi { Lakenya ng new o { Daryll r  { Karan worsening sym { Rony ptoms? none of greenwood- these ; in the last 10 { Valente  days, have you been in contact with  { Abdirahman someone who was simpson, confirmed or suspected to have coronavirus/covid-19? no / unsure ; have elizab { Bernhard eth, rn you had a covid-19 viral test in  { Lucus the  { Devonna last 10 days? no ; have you traveled internationally in the last month?  locati { Indiana ons: travel history not shown for pas { Migel t encounters 14:00:52 trigger for triag { Genelle e greenwood- { Marybelle  sta { Samanta rt simpson, elizabeth, rn  { Bernabe 1 { Aarya 4:00:52 triage { Kylene  started greenwood- simpson, elizabeth, rn 14:00:52 chief complaints speech problem greenwood- updated simpson, elizabeth, rn 1 { Harlem 4:01 vi { Jerell tals greenwood- reassessment timer s { Tamir impson, started elizabeth, rn 14:01 vitals vitals timer greenwood- reassessment restart vitals timer: yes sim { Shera ps { Loreal on, elizabeth, rn 14:01 high risk high risk screening fo { Maha r stroke and acs green { Lenord wood- screening d { Catheryn oes the patient have any signs { Kalei  or symptoms that  { Arrie are suggestive of a stroke simpson, or tia with onset less than 8 hours?: yes ! elizabeth, rn d { Lasonya oes the patient have any signs or symptoms { Neomi  suggestive of acs?: :01 columbia suicide columbia su { Arland icide severity rating scale greenwood- severity rating 1.  { Verle wish to be dead (within the past month): , scale 2. suicidal thou { Lorita ghts (within the past month): , rn 6. { Sandee  suicide behavior question:  on 10/3/24 7:13 am page 2149,vumc adult  { Cass hos { Katilyn pital white, tyrone 1211 medical center d { Raniyah r. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232-0004 adm: 6/2/2023, d/c: 6 { Sinai /3/2023 06/02/2023 - ed to hosp-admis { Fraser sion (discharged) in vanderbilt university adult hospital { Natividad  (continued { Sandro ) ed care timeline (continued) 14:01 vital signs vitals greenwood- temp: 36.9 °c (98.4 °f) simpson, temp src: oral elizabeth, rn pulse: 95 resp: 18  { Elmer bp: 118/79 bp locat { Hanan ion: right arm bp method: automatic patient position: sitting spo2: 98% respiratory support res { Crispin piratory support: none vitals timer restar { Kristin t vitals  { Divine timer: yes restart vitals timer: yes { Mariama   { Shantae other fl { Isom owsheet entries hr/pulse: 95 14:01 custom { Awilda  formula relevan { Margurite t labs an { Girard d { Jarius  vitals greenwood- { Elberta  data temp (in celsius): 36.9 simpson, colu { Kimi mbia suicide severity rating scale elizabeth, rn cssr { Baleigh s score: green 14:02 tetanus status tetanus up to date greenwood- has the patient had a tet { Dorie anus vaccine within the last 5 years?: unknown simpson, elizabeth, rn 14:02:44 orders p { Karson laced ecg - ekg 12 lead - other indication bickett, christopher ryan, md 14:02:46 ecg ordered ekg electrocardiogram greenwood- simpson, elizabeth, rn 14:03 { Silverio  post-triage post-triage destination bickett, destination ed destination: pod christopher ryan, md 1 { Corin 4:03 acuity/destination acuity/destination greenwood- patient acuity: 3 simpson, mobility: non-ambulator { Haileigh y elizabeth, rn primary cond { Laure it { Nakayla ion treate { Gamaliel d: med/surg 14:03 home infus { Jeanpaul i { Breeana on p { Kelsee atient be { Talen longings at bedside greenwood- pumps medical equipment: none simpson, elizabeth, rn 14:03 sepsis screening sepsis screen { Madolyn ing greenwood- are rigors present?: , is t { Tamiya here a suspected  { Demetrious infection?: , rn is the p { Harlin atie { Maleek nt's mental status altered?: :03:06 allergies reviewed greenwood- simpson, elizabeth, rn 14:03:21 home medications green { Oneil wood- reviewed simpson { Creola , { Francene  elizabeth { Sania , rn 14:03:29 history reviewed { Joao  sections reviewed: medical greenwood- simpson, elizabeth, rn 14:03:30 { Kathleen  history reviewed  { Neena sections reviewed: surgical greenwood- simpson, elizabeth, rn 14:03:42 history reviewed section { Stacee s reviewed: alcohol greenwood- simpson, elizabeth,  { Dessa rn printed { Brantlee  on 10/3/24 7:13 am page 2150",
    {
        "entities": [
            [
                19,
                27,
                "PERSON"
            ],
            [
                129,
                134,
                "PERSON"
            ],
            [
                196,
                203,
                "PERSON"
            ],
            [
                282,
                288,
                "PERSON"
            ],
            [
                303,
                311,
                "PERSON"
            ],
            [
                396,
                403,
                "PERSON"
            ],
            [
                464,
                472,
                "PERSON"
            ],
            [
                585,
                593,
                "PERSON"
            ],
            [
                628,
                635,
                "PERSON"
            ],
            [
                652,
                659,
                "PERSON"
            ],
            [
                722,
                731,
                "PERSON"
            ],
            [
                755,
                760,
                "PERSON"
            ],
            [
                901,
                909,
                "PERSON"
            ],
            [
                961,
                970,
                "PERSON"
            ],
            [
                984,
                991,
                "PERSON"
            ],
            [
                1008,
                1015,
                "PERSON"
            ],
            [
                1133,
                1139,
                "PERSON"
            ],
            [
                1187,
                1193,
                "PERSON"
            ],
            [
                1201,
                1209,
                "PERSON"
            ],
            [
                1220,
                1227,
                "PERSON"
            ],
            [
                1232,
                1238,
                "PERSON"
            ],
            [
                1254,
                1259,
                "PERSON"
            ],
            [
                1310,
                1318,
                "PERSON"
            ],
            [
                1358,
                1369,
                "PERSON"
            ],
            [
                1475,
                1484,
                "PERSON"
            ],
            [
                1528,
                1534,
                "PERSON"
            ],
            [
                1541,
                1549,
                "PERSON"
            ],
            [
                1631,
                1639,
                "PERSON"
            ],
            [
                1679,
                1685,
                "PERSON"
            ],
            [
                1727,
                1735,
                "PERSON"
            ],
            [
                1750,
                1760,
                "PERSON"
            ],
            [
                1767,
                1775,
                "PERSON"
            ],
            [
                1804,
                1812,
                "PERSON"
            ],
            [
                1816,
                1822,
                "PERSON"
            ],
            [
                1839,
                1846,
                "PERSON"
            ],
            [
                1976,
                1983,
                "PERSON"
            ],
            [
                1993,
                2000,
                "PERSON"
            ],
            [
                2039,
                2045,
                "PERSON"
            ],
            [
                2157,
                2163,
                "PERSON"
            ],
            [
                2168,
                2175,
                "PERSON"
            ],
            [
                2234,
                2239,
                "PERSON"
            ],
            [
                2264,
                2271,
                "PERSON"
            ],
            [
                2291,
                2300,
                "PERSON"
            ],
            [
                2333,
                2339,
                "PERSON"
            ],
            [
                2360,
                2366,
                "PERSON"
            ],
            [
                2464,
                2472,
                "PERSON"
            ],
            [
                2517,
                2523,
                "PERSON"
            ],
            [
                2579,
                2586,
                "PERSON"
            ],
            [
                2647,
                2653,
                "PERSON"
            ],
            [
                2721,
                2728,
                "PERSON"
            ],
            [
                2768,
                2775,
                "PERSON"
            ],
            [
                2847,
                2852,
                "PERSON"
            ],
            [
                2858,
                2866,
                "PERSON"
            ],
            [
                2910,
                2918,
                "PERSON"
            ],
            [
                3014,
                3020,
                "PERSON"
            ],
            [
                3060,
                3067,
                "PERSON"
            ],
            [
                3127,
                3137,
                "PERSON"
            ],
            [
                3151,
                3158,
                "PERSON"
            ],
            [
                3309,
                3315,
                "PERSON"
            ],
            [
                3337,
                3343,
                "PERSON"
            ],
            [
                3441,
                3449,
                "PERSON"
            ],
            [
                3494,
                3502,
                "PERSON"
            ],
            [
                3514,
                3521,
                "PERSON"
            ],
            [
                3560,
                3568,
                "PERSON"
            ],
            [
                3572,
                3580,
                "PERSON"
            ],
            [
                3591,
                3596,
                "PERSON"
            ],
            [
                3640,
                3647,
                "PERSON"
            ],
            [
                3666,
                3676,
                "PERSON"
            ],
            [
                3688,
                3695,
                "PERSON"
            ],
            [
                3699,
                3706,
                "PERSON"
            ],
            [
                3727,
                3735,
                "PERSON"
            ],
            [
                3781,
                3786,
                "PERSON"
            ],
            [
                3842,
                3850,
                "PERSON"
            ],
            [
                3944,
                3950,
                "PERSON"
            ],
            [
                4040,
                4047,
                "PERSON"
            ],
            [
                4205,
                4214,
                "PERSON"
            ],
            [
                4317,
                4323,
                "PERSON"
            ],
            [
                4430,
                4439,
                "PERSON"
            ],
            [
                4470,
                4476,
                "PERSON"
            ],
            [
                4481,
                4489,
                "PERSON"
            ],
            [
                4502,
                4511,
                "PERSON"
            ],
            [
                4542,
                4551,
                "PERSON"
            ],
            [
                4555,
                4563,
                "PERSON"
            ],
            [
                4570,
                4577,
                "PERSON"
            ],
            [
                4589,
                4595,
                "PERSON"
            ],
            [
                4718,
                4726,
                "PERSON"
            ],
            [
                4771,
                4778,
                "PERSON"
            ],
            [
                4798,
                4809,
                "PERSON"
            ],
            [
                4837,
                4844,
                "PERSON"
            ],
            [
                4851,
                4858,
                "PERSON"
            ],
            [
                4981,
                4987,
                "PERSON"
            ],
            [
                5012,
                5019,
                "PERSON"
            ],
            [
                5023,
                5032,
                "PERSON"
            ],
            [
                5045,
                5051,
                "PERSON"
            ],
            [
                5084,
                5089,
                "PERSON"
            ],
            [
                5162,
                5171,
                "PERSON"
            ],
            [
                5192,
                5198,
                "PERSON"
            ],
            [
                5296,
                5303,
                "PERSON"
            ],
            [
                5357,
                5363,
                "PERSON"
            ],
            [
                5376,
                5385,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical center east white, tyrone 1211 medical cente { Jerron r dr mrn: 047717361, dob: 7/27/1969,  { Martel legal sex: m nashville tn 3723 { Rea 2  { Shardae visit date: 9/16/2024 09/16/20 { Val 24 - office visit in vanderbilt ort { Eligah hopaedics (continued) clini { Naeem cal notes (continued) scribe att { Abigael estation: by signing my name below, i, alex adkins attest that  { Adalia this documentation has been prepared under { Anyla  the direction and { Barrie  in the presence of adam bradburn hicks, dpm. ale { Hafsa x adkins { Shamara , medical scribe. 09/16/24, 9:37 am cdt provider attestation: i, adam brad { Shonta b { Eladio u { Jc rn hicks, dpm, personally performed the services  { Sixto described in this documentation. all medical record { Almira  en { Demian tries made by the scribe  { Ozell were at my direction and in my presence. i have reviewed the chart and discharge instructions (if applicable) and agree { Adena  that t { Julene h { Zak e record reflects my personal performance and is accurate and complete. adam bradburn hicks, dpm, 09/16/ { Chenoa 24, 9:38 am cdt electronically { Dea  signed by hi { Jameka cks, adam bradburn, dpm at 9/16/2024 9:46 am other orders { Nicolina  medications urea 20 % topical cream  { Romana (car { Julious mol) (active) electronically signed { Aadhya  by: hicks, adam bradburn, dpm on 09/16/24 0808 status { Chole : active ordering user: hicks, adam bradburn, dpm { Giacomo  09/16/24 0808 ordering provider: hicks, adam bradburn, dpm authorized by: hicks, adam bradburn, dpm ordering mode: standard prn reaso { Junious ns: dry skin prn comment: apply 1 gram to a { Minor ffected area frequency: routine prn 09/16/24 - 36 { Cher 5 days class: normal { Philippa  diagnoses benign neoplasm of s { Juanpablo kin of lower extremity, unspecified laterality [d23.70] i { Clarabelle ndications benign neoplasm of sk { Kenleigh in of l { Macee ower extremity, unspecified laterality [d23.70 (icd-10-cm)] outpatient referral ambulatory referral to podiat { Shakayla ry (active) electronically signed by: greenspan, debra l, aprn on 08/14 { Ajani /24 1042 status: active this order may be acted on in another encounter. ordering user: greenspan, debra  { Alfonza l, aprn 08/14/24 1042 ordering provider: gre { Dagoberto enspan, debra l, aprn authorized by: greenspan, debra l, apr { Paulino n ordering mode: standard frequency: routine 08/14/24 - cla { Makeda ss: internal referral (vmg) quantity: 1 instanc { Siara e  { Wava released by: williams, cha'keria 9/16/2024 7:43 am diagnoses painful d { Kennon iabetic neuropathy { Ambrosia  (cms/hcc) [e11.40] questionnaire question answer new or e { Kamara stablished patient? new order comments: painful diabetic neur { Siya opathy. unable to increase gabapentin due to ckd> hoping to get patient into new provider specializing in painful diab { Lamonica etic neuropathy. referral details printed on 10/3/2 { Chayce 4 7:12 am page 29,vumc adult medical center ea { Zacharia st white, tyrone  { Georgann 1211 medica { Naida l center dr mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37232 visit date: 9/16/2024 09/16/2024 - office v { Russell isit in vanderbilt o { Jariel rthopaedics (c { Sparkle o { Demetrice ntinued) other orders (continued) referred by referred to type priority { Adelita  gre { Roger enspan, debra l, diagnoses { Albino : painf { Cayden ul diabetic o { Courtenay ffice visit -  { Shemeka routine aprn neuropathy (cms/hcc) consult or 1215 21st avenue order: ambulatory referral to new patient south podi { Justo atry 8210 mce south reason: specialty services tower required nashville tn 37232 phone: 615-343-8332 fax: 615-343-8346 comment: pain { Mathis ful diabetic neuropathy. una { Grisel ble to increase gabapentin due to ckd> hoping to get pa { Kenlee tient into new provider specializing in painful diabetic neuropathy. indications painful diabetic neuropathy (cms/hcc) [e11.40 (icd-10-cm)] referral { Javaris  office visit - co { Medina nsult or ne { Vanda w patient { Burdette  #19497485 [last edited by hicks, adam bradburn, dpm { Jakobi  on 9 { Nabil /16/2024 0947] reason: specialty services required priority: routine class: internal status:  { Cleora closed - system closed - visit(s) completed status updated on:  { Kym 9 { Dimas /16/2024 valid dates: from 8/ { Alizabeth 14/2024 to 10/15/2025 referred from location:  { Zayda vumc adult medical center east department: endocrinology diabetes mce 8 depart { Criselda ment phone: 615-343-8332 provider { Jona : greenspan, deb { Lala ra l,  { Jalin aprn provider phone: 615-3 { Serafin 43-8332 provider address: { Verlon  1215 21st avenue s { Erynn outh 8210 { Joya  mce south tower nashville tn 37232 visits requested: 1 auth { Lynna orized: 1 co { Nakiya mpleted: 1 sc { Wilhelm heduled: 0 procedures ref90 - ambul { Illa atory referral to podiatry number requested: 1 number app { Jaxxon roved: 1 diagnoses e11.40 (icd-10-cm) - painful diabetic neuropathy (cms/hcc) referra { Neo l notes general by brockman, cherith at 9/16/2024 0705 current status:  required this service does not require an authorization with this p { Loris ayor plan according to { Shaunda  confirmation obtained from the paye { Yehudis r. payer plan: ambetter tennessee cpt codes: 99201-99205 dx: e11.40 method of authorization request submission: nar per policy reference number: n/a printed on 10/3/24 7:12 a { Shiv m page 30",
    {
        "entities": [
            [
                66,
                73,
                "PERSON"
            ],
            [
                113,
                120,
                "PERSON"
            ],
            [
                153,
                157,
                "PERSON"
            ],
            [
                162,
                170,
                "PERSON"
            ],
            [
                203,
                207,
                "PERSON"
            ],
            [
                245,
                252,
                "PERSON"
            ],
            [
                282,
                288,
                "PERSON"
            ],
            [
                323,
                331,
                "PERSON"
            ],
            [
                397,
                404,
                "PERSON"
            ],
            [
                449,
                455,
                "PERSON"
            ],
            [
                476,
                483,
                "PERSON"
            ],
            [
                535,
                541,
                "PERSON"
            ],
            [
                552,
                560,
                "PERSON"
            ],
            [
                637,
                644,
                "PERSON"
            ],
            [
                648,
                655,
                "PERSON"
            ],
            [
                659,
                662,
                "PERSON"
            ],
            [
                714,
                720,
                "PERSON"
            ],
            [
                774,
                781,
                "PERSON"
            ],
            [
                787,
                794,
                "PERSON"
            ],
            [
                822,
                828,
                "PERSON"
            ],
            [
                950,
                956,
                "PERSON"
            ],
            [
                966,
                973,
                "PERSON"
            ],
            [
                977,
                981,
                "PERSON"
            ],
            [
                1088,
                1095,
                "PERSON"
            ],
            [
                1128,
                1132,
                "PERSON"
            ],
            [
                1148,
                1155,
                "PERSON"
            ],
            [
                1215,
                1224,
                "PERSON"
            ],
            [
                1264,
                1271,
                "PERSON"
            ],
            [
                1278,
                1286,
                "PERSON"
            ],
            [
                1324,
                1331,
                "PERSON"
            ],
            [
                1388,
                1394,
                "PERSON"
            ],
            [
                1446,
                1454,
                "PERSON"
            ],
            [
                1591,
                1599,
                "PERSON"
            ],
            [
                1645,
                1651,
                "PERSON"
            ],
            [
                1703,
                1708,
                "PERSON"
            ],
            [
                1731,
                1740,
                "PERSON"
            ],
            [
                1774,
                1784,
                "PERSON"
            ],
            [
                1844,
                1855,
                "PERSON"
            ],
            [
                1890,
                1899,
                "PERSON"
            ],
            [
                1909,
                1915,
                "PERSON"
            ],
            [
                2027,
                2036,
                "PERSON"
            ],
            [
                2110,
                2116,
                "PERSON"
            ],
            [
                2224,
                2232,
                "PERSON"
            ],
            [
                2279,
                2289,
                "PERSON"
            ],
            [
                2352,
                2360,
                "PERSON"
            ],
            [
                2422,
                2429,
                "PERSON"
            ],
            [
                2479,
                2485,
                "PERSON"
            ],
            [
                2490,
                2495,
                "PERSON"
            ],
            [
                2568,
                2575,
                "PERSON"
            ],
            [
                2596,
                2605,
                "PERSON"
            ],
            [
                2666,
                2673,
                "PERSON"
            ],
            [
                2737,
                2742,
                "PERSON"
            ],
            [
                2863,
                2872,
                "PERSON"
            ],
            [
                2926,
                2933,
                "PERSON"
            ],
            [
                2982,
                2991,
                "PERSON"
            ],
            [
                3011,
                3020,
                "PERSON"
            ],
            [
                3034,
                3040,
                "PERSON"
            ],
            [
                3162,
                3170,
                "PERSON"
            ],
            [
                3193,
                3200,
                "PERSON"
            ],
            [
                3217,
                3225,
                "PERSON"
            ],
            [
                3229,
                3239,
                "PERSON"
            ],
            [
                3313,
                3321,
                "PERSON"
            ],
            [
                3328,
                3334,
                "PERSON"
            ],
            [
                3363,
                3370,
                "PERSON"
            ],
            [
                3380,
                3387,
                "PERSON"
            ],
            [
                3403,
                3413,
                "PERSON"
            ],
            [
                3430,
                3438,
                "PERSON"
            ],
            [
                3555,
                3561,
                "PERSON"
            ],
            [
                3696,
                3703,
                "PERSON"
            ],
            [
                3734,
                3741,
                "PERSON"
            ],
            [
                3799,
                3806,
                "PERSON"
            ],
            [
                3957,
                3965,
                "PERSON"
            ],
            [
                3986,
                3993,
                "PERSON"
            ],
            [
                4007,
                4013,
                "PERSON"
            ],
            [
                4025,
                4034,
                "PERSON"
            ],
            [
                4089,
                4096,
                "PERSON"
            ],
            [
                4104,
                4110,
                "PERSON"
            ],
            [
                4206,
                4213,
                "PERSON"
            ],
            [
                4279,
                4283,
                "PERSON"
            ],
            [
                4287,
                4293,
                "PERSON"
            ],
            [
                4325,
                4335,
                "PERSON"
            ],
            [
                4384,
                4390,
                "PERSON"
            ],
            [
                4471,
                4480,
                "PERSON"
            ],
            [
                4516,
                4521,
                "PERSON"
            ],
            [
                4540,
                4545,
                "PERSON"
            ],
            [
                4554,
                4560,
                "PERSON"
            ],
            [
                4589,
                4597,
                "PERSON"
            ],
            [
                4625,
                4632,
                "PERSON"
            ],
            [
                4654,
                4660,
                "PERSON"
            ],
            [
                4672,
                4677,
                "PERSON"
            ],
            [
                4740,
                4746,
                "PERSON"
            ],
            [
                4761,
                4768,
                "PERSON"
            ],
            [
                4784,
                4792,
                "PERSON"
            ],
            [
                4830,
                4835,
                "PERSON"
            ],
            [
                4895,
                4902,
                "PERSON"
            ],
            [
                4990,
                4994,
                "PERSON"
            ],
            [
                5136,
                5142,
                "PERSON"
            ],
            [
                5167,
                5175,
                "PERSON"
            ],
            [
                5214,
                5222,
                "PERSON"
            ],
            [
                5399,
                5404,
                "PERSON"
            ]
        ]
    }
),(
    "vumc eye institute nashville whit { Alexzandria e, tyrone 2311 pierce avenue mrn: 047717361, dob: { Manning  7 { Cate /27/196 { Lavenia 9, legal sex: m nashville tn 37232 visit date: 8/24/2024 08/24/2024 - communication in vanderbilt eye inst { Deborah i { Lorenz tute facesheet report patient demographics patient name mrn legal dob address phone white, tyrone 0477173 sex 7/27/1969 apt 705 { Ciaran  615-260-2291 (home) 61 m 1101 edgehill ave 615-260-2291 (mobile) nashville tn 37203 *preferred* hospital account not on file admission information curre { Jamarius nt information attending provider { Kohl  admitting provider admission type admission status unknown status admi { Romie ssion date/time d { Waleed ischarge date/time hospital service auth/cert status hospital area unit room/bed referring provider 08/24/2024 - communicati { Montie on in vanderbilt eye institute (continued) visit in { Paden formation provider information encounter provider berkowitz, sean, md, mba department name address phone fax van { Theodor de { Orie rbilt eye institute 2311 pierce ave 615-936-2020 615-936-1540 v { Salim anderbilt eye institute nashville tn 37232 medication list medication list 1 this report is for documentat { Melodi ion purposes { Jayven  only. the patient should not follow medication instructi { Briona ons wi { Akshay thin. f { Eduard or accurate instructions regarding medications, the patient should instead consult the { Destanie ir physician  { Kinleigh or after visit summary. { Genia  active at the { Meilani  end of visit medications last reviewed by scott, mary c, rn on 8/23/2024 1646 famotidine 20 mg tablet (pepcid) [reconciled by ferguson, { Uziel  sherri l, lpn on 1/11/2023 12 { Zakiyah 57] instructions: take 1 tablet (20 mg tota { Taylee l) by mouth every 12 hour { Olivier s. entered by: ferguson, sherri l, lpn entered on: 1/11/2023 atorvastatin 80 mg tablet (li { Ren pitor) { Carline  discontinu { Milah ed by { Augusta : { Ossie  mickey, lisa, lpn discontinued on: 10/2/2024 instructions:  { Concha take 1 tablet (80 mg total) by mouth { Courtnee  daily. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date: 8/8/20 { Delina 23 quantity: 90 tablet refill:  { Florance 3 refills by 8/7/2024 cycl { Shyanna obenzaprine 5 m { Londa g tablet (flexeril) [reconciled by maples, chantis on 9/5/2023 1522] printed on 10/3/24 7:12 am page 149, { Marchelle vumc eye  { Taelyn in { Nickey stitute nashville white, tyrone 2311 pierce { Celene  avenue mrn: 047717361, dob: 7/27/196 { Deon 9, legal sex: m nashville tn 37232 visit date: 8/24/2024 08/24/2024 - { Howard  communicatio { Larita n in vanderbilt eye insti { Zakia tute (c { Calvert ontinued) medication list (continued) instructions: take 1 tablet (5 mg total) by mouth every { Wheeler  8 hours as needed. entered by: maples, chantis entered { Paisleigh  on: 9/5/2023 start date: 7/22/2023 triamcinolone acetonide 55 mcg nasal spray aerosol (nasacort) discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discontinuation: duplicate order in { Laurin struction { Lisandra s: administer 2 sprays (110 mcg total) into each nos { Margene tril 2 times a day. authorized by: greenspan, debra l, aprn ordered on: 9/5/2023 start date: 9/5/2023 end date: 9/16/2024 quantity: 16.5 g refill: 11 refills by 9/4/2024 capsaicin 0.1 { Tempie  %  { Seneca topical cream discontinued by: gingrow, barbara, lpn discontinued on: 9/16/2024 reason for discont { Tobey inuation: duplicate order instructions: apply 1 application topica { Aleida lly daily for 90 { Kenadee  days. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 s { Aries tart da { Niklas te { Nyree : 9/7/2023 end date: 9/16/2024 action: patient not taking quantity: 42.5 g refill:  remaining aspirin 81 mg tablet,delayed rele { Raeanna ase instructions: take 1 tablet (81 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on: 9/7/2023 start date: 9/7/2023 quantity: 90  { Aleyna tablet refill: 3 refills by 9/6/2024 cetirizine 10 mg tablet (zyrtec)  { Lorianne instruction { Rahim s: take 1 tablet (10 mg total) by mouth once a day as needed fo { Dyanna r allergies. authorized by: de witte, anton jordan, md orde { Lonie red on: 10/4/2023 start date: 10/4/2023 quantity: 30 tablet refill: 9 refills by 10/3/2024 lidocaine 5 %  { Mallori topical patch (lidoderm) instructions: apply 1 patch { Rhona  topicall { Zachary y daily. apply to painful area 12 hours per day, remove for 12 hours. authorize { Izak d by: de witte, anton jordan, md ordered on: 10/17/2023 start date: 10/17/2023 end date: 10/16/2 { Brienna 024 quantity: 30 patch refill { Markia : 11 refills by 10/16/2024 nifedipine er 30 mg tablet,extended rele { Treasa ase (adalat cc) instruction { Kelan s: take 2 tablets (60 mg total) by mouth daily.  { Bradley authorized by: de witte, anton jordan, md ordered on: 4/23/2024 start date: 4/23/2024 quantity: 180 tablet refill: 3 ref { Keyara ills by 4/23/2025 lantus solostar u-100 insu { Malvina lin { Rochell  100 unit/ml (3 ml) subcutaneous pen (insulin glargine) instructions: inject 10 units under the skin 2 times a { Shenita  d { Jaison ay. authorized by: greenspan, debra l, aprn ordered on: 7/9/2024 start  { Jerred date: 7/9/2024 quantity: 18 ml refill: 3 refills by 7/9/2025 pantoprazole 20 mg tablet { Avigail ,delayed release (protonix) instructions: take 1 tablet (20 mg total) by m { Laquanda outh daily. auth { Maryssa orized by: greenspan, debra l, aprn ordered on: 7/9/2024 start date:  { Tameika 7/9/2024 end d { Akhil ate: 7/9/2025  { Ansh quantity: 30 tablet refill: 11 refills by 7/9/2025 prednisolone acetate 1 % eye drops,suspension (pred forte { Jessee ) instructions: after { Maricella  surgery, use 1 drop to the { Noemy  ri { Pearly ght eye ever { Shatara y 2 hours while awake unt { Ivelisse il bedtime. beginning the next day, dec { Lynnea rease to 1 drop to the right eye 4 times a da { Mitzie y for 1 week, then 3 times a day for 1 week, then  { Sarrah 2 times a day for 1 week, then printed o { Rigo n 10/3/2 { Alane 4 7:12 am page 150",
    {
        "entities": [
            [
                36,
                48,
                "PERSON"
            ],
            [
                100,
                108,
                "PERSON"
            ],
            [
                113,
                118,
                "PERSON"
            ],
            [
                128,
                136,
                "PERSON"
            ],
            [
                245,
                253,
                "PERSON"
            ],
            [
                257,
                264,
                "PERSON"
            ],
            [
                394,
                401,
                "PERSON"
            ],
            [
                557,
                566,
                "PERSON"
            ],
            [
                602,
                607,
                "PERSON"
            ],
            [
                681,
                687,
                "PERSON"
            ],
            [
                707,
                714,
                "PERSON"
            ],
            [
                841,
                848,
                "PERSON"
            ],
            [
                902,
                908,
                "PERSON"
            ],
            [
                1023,
                1031,
                "PERSON"
            ],
            [
                1036,
                1041,
                "PERSON"
            ],
            [
                1107,
                1113,
                "PERSON"
            ],
            [
                1222,
                1229,
                "PERSON"
            ],
            [
                1244,
                1251,
                "PERSON"
            ],
            [
                1311,
                1318,
                "PERSON"
            ],
            [
                1327,
                1334,
                "PERSON"
            ],
            [
                1344,
                1351,
                "PERSON"
            ],
            [
                1440,
                1449,
                "PERSON"
            ],
            [
                1465,
                1474,
                "PERSON"
            ],
            [
                1500,
                1506,
                "PERSON"
            ],
            [
                1523,
                1531,
                "PERSON"
            ],
            [
                1670,
                1676,
                "PERSON"
            ],
            [
                1709,
                1717,
                "PERSON"
            ],
            [
                1763,
                1770,
                "PERSON"
            ],
            [
                1798,
                1806,
                "PERSON"
            ],
            [
                1899,
                1903,
                "PERSON"
            ],
            [
                1912,
                1920,
                "PERSON"
            ],
            [
                1934,
                1940,
                "PERSON"
            ],
            [
                1948,
                1956,
                "PERSON"
            ],
            [
                1960,
                1966,
                "PERSON"
            ],
            [
                2029,
                2036,
                "PERSON"
            ],
            [
                2075,
                2084,
                "PERSON"
            ],
            [
                2172,
                2179,
                "PERSON"
            ],
            [
                2213,
                2222,
                "PERSON"
            ],
            [
                2251,
                2259,
                "PERSON"
            ],
            [
                2277,
                2283,
                "PERSON"
            ],
            [
                2391,
                2401,
                "PERSON"
            ],
            [
                2413,
                2420,
                "PERSON"
            ],
            [
                2425,
                2432,
                "PERSON"
            ],
            [
                2478,
                2485,
                "PERSON"
            ],
            [
                2525,
                2530,
                "PERSON"
            ],
            [
                2602,
                2609,
                "PERSON"
            ],
            [
                2625,
                2632,
                "PERSON"
            ],
            [
                2660,
                2666,
                "PERSON"
            ],
            [
                2676,
                2684,
                "PERSON"
            ],
            [
                2780,
                2788,
                "PERSON"
            ],
            [
                2846,
                2856,
                "PERSON"
            ],
            [
                3069,
                3076,
                "PERSON"
            ],
            [
                3088,
                3097,
                "PERSON"
            ],
            [
                3152,
                3160,
                "PERSON"
            ],
            [
                3346,
                3353,
                "PERSON"
            ],
            [
                3359,
                3366,
                "PERSON"
            ],
            [
                3467,
                3473,
                "PERSON"
            ],
            [
                3542,
                3549,
                "PERSON"
            ],
            [
                3568,
                3576,
                "PERSON"
            ],
            [
                3650,
                3656,
                "PERSON"
            ],
            [
                3666,
                3673,
                "PERSON"
            ],
            [
                3678,
                3684,
                "PERSON"
            ],
            [
                3814,
                3822,
                "PERSON"
            ],
            [
                3984,
                3991,
                "PERSON"
            ],
            [
                4064,
                4073,
                "PERSON"
            ],
            [
                4087,
                4093,
                "PERSON"
            ],
            [
                4159,
                4166,
                "PERSON"
            ],
            [
                4228,
                4234,
                "PERSON"
            ],
            [
                4342,
                4350,
                "PERSON"
            ],
            [
                4405,
                4411,
                "PERSON"
            ],
            [
                4423,
                4431,
                "PERSON"
            ],
            [
                4513,
                4518,
                "PERSON"
            ],
            [
                4617,
                4625,
                "PERSON"
            ],
            [
                4657,
                4664,
                "PERSON"
            ],
            [
                4734,
                4741,
                "PERSON"
            ],
            [
                4771,
                4777,
                "PERSON"
            ],
            [
                4828,
                4836,
                "PERSON"
            ],
            [
                4959,
                4966,
                "PERSON"
            ],
            [
                5013,
                5021,
                "PERSON"
            ],
            [
                5027,
                5035,
                "PERSON"
            ],
            [
                5148,
                5156,
                "PERSON"
            ],
            [
                5161,
                5168,
                "PERSON"
            ],
            [
                5242,
                5249,
                "PERSON"
            ],
            [
                5338,
                5346,
                "PERSON"
            ],
            [
                5423,
                5432,
                "PERSON"
            ],
            [
                5451,
                5459,
                "PERSON"
            ],
            [
                5531,
                5539,
                "PERSON"
            ],
            [
                5556,
                5562,
                "PERSON"
            ],
            [
                5579,
                5584,
                "PERSON"
            ],
            [
                5695,
                5702,
                "PERSON"
            ],
            [
                5726,
                5736,
                "PERSON"
            ],
            [
                5766,
                5772,
                "PERSON"
            ],
            [
                5778,
                5785,
                "PERSON"
            ],
            [
                5800,
                5808,
                "PERSON"
            ],
            [
                5836,
                5845,
                "PERSON"
            ],
            [
                5887,
                5894,
                "PERSON"
            ],
            [
                5942,
                5949,
                "PERSON"
            ],
            [
                6002,
                6009,
                "PERSON"
            ],
            [
                6052,
                6057,
                "PERSON"
            ],
            [
                6068,
                6074,
                "PERSON"
            ]
        ]
    }
),(
    "vumc he { Dorris nderso { Mylie nville - anderson white, { Ramsey  tyrone 128 n anderson ln mrn: 047717361, dob { Bernardino :  { Chayton 7/27/1969, legal sex: m hendersonville tn 37075 adm: 1/11/2023, d/c: 1/11/2023 01/11/2023 - xr general imaging in vanderbilt radiology hendersonville (continued) medication list (continued) atorvastatin 80 mg tabl { Diesel et (lipitor) discontinued by: lippard, giles a, aprn discontinued { Sasha  on: 8/8/2023 reason for discontinuation: reorder instructions: take 1 tablet (80 mg total) by mouth daily. authorized by: lippard, giles a, aprn ordered { Arlean   { Joely on: 10/25/2022 start date: 10/25/2022 quantity: 90 tablet refill { Dawayne : 3 refills by 10/25/2023 cetirizine 10 mg tablet (zyrtec) discontinued by: lippard, giles a, aprn discontinu { Zenia ed on: 8 { Caine /8/2023 reason for discontinuation: reorder instructions: tak { Camie e 1 tablet (10 mg total) by mouth once a day as needed for allergies. authorize { Elenore d by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quantity: 30 tablet refill: 11 refills  { Katharyn by 10/25/2023 docusate sodium 100 mg capsule ( { Myisha colace) discontinued by: de witte, anton jordan, md discontinued on: 12/6/2023 reason for discontinuation: cleanup(notavs) instructions: take one tablet tid prn constipation authorized by: lippard, giles a, aprn ordered on: 10/25/2022 start date: 10/25/2022 end date: 12/6/20 { Tommi 23 quantity: 60 capsule refill:  remaining fluticas { Rainey one propionate 50 mcg/actuation nasal spray,suspension (flonase) discontinued by: lippard, giles a, aprn discontinued on: 6/6/2023 reason fo { Sable r discontinuation: reorder instructions: administer 2 sprays { Hughie  into each nostril daily. authorized by: lippard, giles a, aprn ordered on:  { Issa 10/2 { Sina 5/2 { Ysabella 022 start date: 10/25/2 { Gavino 022 quantity: 16 g refill: 11 refills by 10/25 { Jalyn /2023 aze { Dolan lastine 137 mcg { Oris  (0.1 %) { Vasilios  nasa { Vaughan l spray aerosol (astelin) d { Julietta iscontinued by: lippard, giles a, aprn discontinued on: 8/8/2023 reason for discontinuation: reorder instructions: administer 1 spray in { Jaxx to each nostril 2 { Cheyann  times a day. use in each nostril as directed authorized by: lippard, giles a, aprn { Jemima  ordered on: 10/25/2022 start date: 10/25/2022 quan { Kennady tity: 30 ml refill: 12 refills by 10/25/2023 lidocaine 5 % topical patch (lidoderm) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for discontinuation: s { Zaynab top taking at d { Dotty ischarge (cancelrx) instructions: apply 1 patch topicall { Marisha y daily. apply to painful area 12 hours per day, remove for 12 hours. authorized by: lippard, giles  { Carolyn a { Renee , aprn ordered on: 10/25/2022 start date: 10/25/20 { Lashae 22 end date: 6/3/2023 action: patient not taking quantity: 30 patch refill: 11 refills by 10/25/2023 albuterol sulfate hfa 90 mcg/actuation aerosol  { Chloee inhaler discontinued by: lippard, giles a, aprn discontinued on: 8/8/202 { Kasia 3 reason for dis { Khai continuation: reorder instructions: inhale 2 puffs every { Albertina  4 hours as needed for wheezing. authorized by: lippard, gi { Clarine les a, aprn ordered on: 10/25/2022 start date: 10/25/2022 quantity: 18 g refill: 11 refills by 10/25 { Mathilde /2023 pantoprazole 20 mg tablet,delayed release  { Jalen (protonix) discontinued by: greenspan, debra l, aprn discont { Jo-Anne inued on: 7/9/2024 printed on 10/3/24 7:13 am page 2711,vumc hendersonville - anderson white, tyrone 128 n anderson ln mrn: 047717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 adm: 1/11/2023, d/c: 1/11/2023 01/11/2023 - xr genera { Kenadie l imaging in vanderbilt radiolo { Kiona gy hendersonville (continued) medication list (continued) reason for discontinuation: reorder instructions: take 1 tabl { Shereen et (20 mg total) by mouth da { Zulma ily. authorized { Mekayla  by: lippard, giles a, aprn ordered on: 11/29/2022 start date: 11/29/2022 end date: 7/9/2024 quantity: 30 tablet refill: 11 refills by 11/29/2023 sodium chloride 0.65% { Arletha  nasal spray  { Francina aerosol (ocean nasal) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for  { Teddi discontinuation: stop taking at discharge (cancelrx) instructions: administer 1 spray into each nostril as needed for rhinitis { Yolande . authorized by: lippard, giles a, aprn ordered on: 12/6/2022 start date:  { Jacquez 12/6/2022 end date: 6/3/2023 quantity: 15 ml refill: 12 refil { Kalob ls by 12/6/ { Vijay 2023 acetaminophen 325 mg tablet (tylenol) discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for discontinuation: reorder instructions: take 2 tablets (650 mg tot { Tyeisha al) by mouth eve { Marijane ry 6 hours as needed f { Xitlaly or mild pain. authorized b { Catarino y: lippard, giles a, aprn ordered on: 12/9/2022 start date: 12/9/2022 end date: 6/3/2023 action: patient not taking quantity: 30 tablet ref { Rebekkah ill:  remaining gaba { Benard pentin 300 mg capsule (neu { Geoffery ro { Eternity ntin) discontinued by: lipp { Ziva ard, giles a, aprn discontinued on: 2/9/2023 instructions: take 1 capsule by mou { Yeshua th three times a day authorized by: lippard,  { Roshan giles a, aprn orde { Kekoa red on: 1/4/2023 start date: 1/4/2023 quantity: 90 capsule refill:  remaining tr { Kenith ulicity 3 mg/0.5 ml subcutaneous pen injector (dulaglutide) discontinued by: wilson, danya horchi, pharmd discontinued on: 3/1/2023 reason for discontinuation: other (canc { Sumaya elrx) instruction { Dulcie s: inject 3 mg under the skin every 7 days. authorized by: lippard, giles a, aprn ordered on: 1/4/2023 start date: 1/4 { Yoshiko /2023 end date: 3/1/2023 quantity: 2 ml refill: 2  { Saleem refills by 1/4/2024 cyclobenzaprine 5 mg tablet (flexeril) [reco { Colene nciled by woehler, kristina, rn on 1/1 { Izetta 1/2023 0756] discontinued by: lehmann, melissa cary, pa-c discontinued on: 6/3/2023 reason for discontinuation: stop  { Willaim taking at discharge (cancelrx) instructions: take 1 tablet (5 mg total) by mouth every 8 { Clementina  hours as { Emma  needed. entered by: woehl { Dajon er, kristina, rn entered on: 1/11/2023 start date: 12/7/2022 end date: 6/3/2023 ibuprofen 600 mg tablet (advil,motrin) [reconciled by woehler, kristina, rn on 1/11/2023 0756] discontinued by: wang, zhijian, aprn discontinued on: 2/7/2023 reason for discontinuation: other  { Damir (cancelrx) instructions: take 1 tablet by  { Griffen mouth every 6 hours as nee { Eliott ded fo { Kairo r { Paxton  pain. do not exceed 4 doses in 24 hours. entered by: woehler, kristina, rn entered on: { Rebeccah  1/11/2023 start da { Jeremi te: 12/7/2022 end date: 2/7/2023 ryb { Laddie elsus 14 { Rufino  mg tablet (semaglutide) [reconciled by ferguson, sherri l, l { Kamdyn pn on 1/11/2023 1257] discontinued by: greenspan, debra l, aprn discon { Dayla tinued on: 2 { Kavya /14/2023 reason for discontinuation: discontinued by another clinician instructions: take 1 tablet (14 mg total) by mouth daily. entered by: ferguson, sherri l,  { Raeleigh lpn entered on: 1/11/2023 end date: 2/14/20 { Sherril 23 famotidine 20 mg tablet { Heather  (pep { Ledger cid) [reconciled by ferguson, sherri l, lpn on 1/11/2023 1257] printed on { Erline  10/3/24 7:13 am page 2712",
    {
        "entities": [
            [
                10,
                17,
                "PERSON"
            ],
            [
                26,
                32,
                "PERSON"
            ],
            [
                59,
                66,
                "PERSON"
            ],
            [
                114,
                125,
                "PERSON"
            ],
            [
                130,
                138,
                "PERSON"
            ],
            [
                354,
                361,
                "PERSON"
            ],
            [
                429,
                435,
                "PERSON"
            ],
            [
                591,
                598,
                "PERSON"
            ],
            [
                602,
                608,
                "PERSON"
            ],
            [
                675,
                683,
                "PERSON"
            ],
            [
                795,
                801,
                "PERSON"
            ],
            [
                812,
                818,
                "PERSON"
            ],
            [
                882,
                888,
                "PERSON"
            ],
            [
                970,
                978,
                "PERSON"
            ],
            [
                1095,
                1104,
                "PERSON"
            ],
            [
                1153,
                1160,
                "PERSON"
            ],
            [
                1438,
                1444,
                "PERSON"
            ],
            [
                1498,
                1505,
                "PERSON"
            ],
            [
                1648,
                1654,
                "PERSON"
            ],
            [
                1717,
                1724,
                "PERSON"
            ],
            [
                1803,
                1808,
                "PERSON"
            ],
            [
                1815,
                1820,
                "PERSON"
            ],
            [
                1826,
                1835,
                "PERSON"
            ],
            [
                1861,
                1868,
                "PERSON"
            ],
            [
                1917,
                1923,
                "PERSON"
            ],
            [
                1935,
                1941,
                "PERSON"
            ],
            [
                1959,
                1964,
                "PERSON"
            ],
            [
                1975,
                1984,
                "PERSON"
            ],
            [
                1992,
                2000,
                "PERSON"
            ],
            [
                2030,
                2039,
                "PERSON"
            ],
            [
                2178,
                2183,
                "PERSON"
            ],
            [
                2203,
                2211,
                "PERSON"
            ],
            [
                2297,
                2304,
                "PERSON"
            ],
            [
                2358,
                2366,
                "PERSON"
            ],
            [
                2553,
                2560,
                "PERSON"
            ],
            [
                2578,
                2584,
                "PERSON"
            ],
            [
                2643,
                2651,
                "PERSON"
            ],
            [
                2754,
                2762,
                "PERSON"
            ],
            [
                2766,
                2772,
                "PERSON"
            ],
            [
                2825,
                2832,
                "PERSON"
            ],
            [
                2983,
                2990,
                "PERSON"
            ],
            [
                3065,
                3071,
                "PERSON"
            ],
            [
                3090,
                3095,
                "PERSON"
            ],
            [
                3154,
                3164,
                "PERSON"
            ],
            [
                3226,
                3234,
                "PERSON"
            ],
            [
                3337,
                3346,
                "PERSON"
            ],
            [
                3397,
                3403,
                "PERSON"
            ],
            [
                3466,
                3474,
                "PERSON"
            ],
            [
                3718,
                3726,
                "PERSON"
            ],
            [
                3760,
                3766,
                "PERSON"
            ],
            [
                3888,
                3896,
                "PERSON"
            ],
            [
                3927,
                3933,
                "PERSON"
            ],
            [
                3951,
                3959,
                "PERSON"
            ],
            [
                4129,
                4137,
                "PERSON"
            ],
            [
                4153,
                4162,
                "PERSON"
            ],
            [
                4269,
                4275,
                "PERSON"
            ],
            [
                4404,
                4412,
                "PERSON"
            ],
            [
                4489,
                4497,
                "PERSON"
            ],
            [
                4561,
                4567,
                "PERSON"
            ],
            [
                4581,
                4587,
                "PERSON"
            ],
            [
                4780,
                4788,
                "PERSON"
            ],
            [
                4807,
                4816,
                "PERSON"
            ],
            [
                4841,
                4849,
                "PERSON"
            ],
            [
                4878,
                4887,
                "PERSON"
            ],
            [
                5029,
                5038,
                "PERSON"
            ],
            [
                5061,
                5068,
                "PERSON"
            ],
            [
                5097,
                5106,
                "PERSON"
            ],
            [
                5111,
                5120,
                "PERSON"
            ],
            [
                5150,
                5155,
                "PERSON"
            ],
            [
                5238,
                5245,
                "PERSON"
            ],
            [
                5293,
                5300,
                "PERSON"
            ],
            [
                5321,
                5327,
                "PERSON"
            ],
            [
                5410,
                5417,
                "PERSON"
            ],
            [
                5591,
                5598,
                "PERSON"
            ],
            [
                5618,
                5625,
                "PERSON"
            ],
            [
                5746,
                5754,
                "PERSON"
            ],
            [
                5807,
                5814,
                "PERSON"
            ],
            [
                5881,
                5888,
                "PERSON"
            ],
            [
                5929,
                5936,
                "PERSON"
            ],
            [
                6056,
                6064,
                "PERSON"
            ],
            [
                6155,
                6166,
                "PERSON"
            ],
            [
                6178,
                6183,
                "PERSON"
            ],
            [
                6212,
                6218,
                "PERSON"
            ],
            [
                6493,
                6499,
                "PERSON"
            ],
            [
                6544,
                6552,
                "PERSON"
            ],
            [
                6581,
                6588,
                "PERSON"
            ],
            [
                6597,
                6603,
                "PERSON"
            ],
            [
                6607,
                6614,
                "PERSON"
            ],
            [
                6704,
                6713,
                "PERSON"
            ],
            [
                6735,
                6742,
                "PERSON"
            ],
            [
                6781,
                6788,
                "PERSON"
            ],
            [
                6799,
                6806,
                "PERSON"
            ],
            [
                6870,
                6877,
                "PERSON"
            ],
            [
                6950,
                6956,
                "PERSON"
            ],
            [
                6971,
                6977,
                "PERSON"
            ],
            [
                7141,
                7150,
                "PERSON"
            ],
            [
                7196,
                7204,
                "PERSON"
            ],
            [
                7233,
                7241,
                "PERSON"
            ],
            [
                7249,
                7256,
                "PERSON"
            ],
            [
                7332,
                7339,
                "PERSON"
            ]
        ]
    }
),(
    "vumc eye institute nashville white, tyrone 2311 pierce avenue m { Yoselyn rn: 047717361, dob: 7/27/1969, legal se { Halima x: m nashville { Noland  tn 37232 visit date: 8/12/2024 08/12/2024 - proc { Stefon edure visit in vander { Batsheva bilt eye institute (continued)  { Shanee clinical notes (continued) patient underwent { Hamish  catarac { Osborne t extraction with intraocular lens placement of the left eye. see separate operative note for deta { Gabrial ils. electronically signed { Saeed  by valenzuela, daniel alejandro, md at 8/12/2024 12:36  { Sherree p { Terrell m printed on 10/3/24 7:12 am page 449,vum { Wayne c eye i { Link nstitute nashville white, tyrone 2311 pier { Hiba ce avenue mrn:  { Jesseca 047717361, dob: 7/27/1969, legal sex: m na { Fischer shville tn 37232 visit date: 8/12/2024 08/12/2024 - pr { Despina ocedure visit in vanderbilt eye institute (continued) lmr  { Marvin encounter level scans outpatient visit - sc { Otilia an on 8/12/2024 12:38 pm: e { Danny scmt op not { Kinzley e (effective from 8/12/2024) scan  { Iyonna (below) page of 2 eye surgery center of middle tennessee i 210 25th avenue north, suite 9 { Madelynne 20 i nashville, tn 37203 i p: 615-964-5912 f: 615-964-5913 i billing inquires: 877-620-9307 name: white, tyrone dos: 08/12/2024 eye surgery center acct #: 5059  { Azriel of middle tennessee dob: 07/27/1969 (55yr) address: { Harvie  1101 edge { Mischa hill ave, apt 705, nashville, tn 37203 ophthalmology - catara { Tanis ct  { Ander surgery operative repor { Lionell t attending was present and scrubbed for the entire procedure. attending: daniel valenzuela, m.d. pre-operative diagnosis: age-related combined sclerotic cataract (h25.12 { Natanael )  { Nisa post-operative diagnosis: same as pre-op diagnosis procedu { Sammy re: le { Taneka ft eye, phacoemulsification with insertion of iol (cpt 66982) indication: visually sig { Porscha nificant cataract causing decreased v { Cedar ision impairing activities or { Emaline  preventing  { Rashonda adequate ophthalmic evaluation due to opacity of media { Zacchaeus  an { Faustina esthe { Jene sia: topical and monitored by anesthesia service est { Loryn imat { Kylar ed blood loss: none implanted device(s) : - cnaoto - std-alcon lens i  { Jaylyn alcon size: 21:0 [ e { Jonpaul xp date #: 06/10/26 pupil siz { Serge e at start of case: 4 mm oper { Classie ative e { Karmyn vents: none systemic complications: none final  { Daymon iol position: in bag cde: 6.28 surgical proce { Jayquan dure: the p { Merwin atient  { Modesta was met in { Parris  the preoperative holding area where informed consent was verified, the operative eye was marked as  { Khalilah the op { Maizie erative site, and dilating drops were i { Almon nstilled. patient was transf { Carlisle erred to the opera { Irina tive suite and pla { Jhonny ced supine on the eye bed where cardiovascular m { Aleia onitoring was established. a t { Dru ime-out was performed. there was { Sammi  confirma { Quanisha tion of the { Teegan  patient, procedure, and eye. topical { Virgina  tetracaine was admi { Arabelle niste { Verenice red. the patient was prepped and draped in the typ { Brixton ical s { Kejuan terile { Rahsaan   { Aziza ophthalmic fashion. an eyelid speculum was placed in the operative eye. a para { Briann centesis incision was made using the sideport blade. 1%  { Little non- preserved lidocaine and epinephrin { Aubry e wa { Benigno s i { Jareth njected into the anterior chamber. the a { Janean nterior chamber was fil { Ladawn led  { Ananda with viscoelastic. a keratome { Asiah  blade was used to create the main temporal wound in a shelved fashion. a malyugin ring w { Minda as carefully ins { Shantal erted to expand the pupil for adequate visualization.  { Ronell the anterior c { Dede apsular tear was initiated using the pre-bent cys { Julie totome. a round continuous tear capsulorhexis was performed using a combination of the cystotome and the utrata forceps. balanced salt so { Mace lution was used to  { Rodriquez perform hydr { Romero odissection. the nucleus  { Bailie was divided into  { Gricelda quarters using the pr { Norman e-chopper { Bennet  and chopping technique and removed using ultrasound power and { Cletis  v { Riccardo acuum. the remaining { Slater  epin { Denese uclear material was removed using the phacoemulsific { Nyesha ation { Clemens  hand  { Carmon piece. op note generated by: { Gidget  amber armstrong 08/12/2024 11:40 printed on 10/3/24 7:12 am page 450",
    {
        "entities": [
            [
                66,
                74,
                "PERSON"
            ],
            [
                116,
                123,
                "PERSON"
            ],
            [
                140,
                147,
                "PERSON"
            ],
            [
                199,
                206,
                "PERSON"
            ],
            [
                230,
                239,
                "PERSON"
            ],
            [
                273,
                280,
                "PERSON"
            ],
            [
                327,
                334,
                "PERSON"
            ],
            [
                345,
                353,
                "PERSON"
            ],
            [
                454,
                462,
                "PERSON"
            ],
            [
                491,
                497,
                "PERSON"
            ],
            [
                556,
                564,
                "PERSON"
            ],
            [
                568,
                576,
                "PERSON"
            ],
            [
                620,
                626,
                "PERSON"
            ],
            [
                636,
                641,
                "PERSON"
            ],
            [
                686,
                691,
                "PERSON"
            ],
            [
                709,
                717,
                "PERSON"
            ],
            [
                762,
                770,
                "PERSON"
            ],
            [
                827,
                835,
                "PERSON"
            ],
            [
                896,
                903,
                "PERSON"
            ],
            [
                949,
                956,
                "PERSON"
            ],
            [
                986,
                992,
                "PERSON"
            ],
            [
                1006,
                1014,
                "PERSON"
            ],
            [
                1051,
                1058,
                "PERSON"
            ],
            [
                1150,
                1160,
                "PERSON"
            ],
            [
                1323,
                1330,
                "PERSON"
            ],
            [
                1384,
                1391,
                "PERSON"
            ],
            [
                1404,
                1411,
                "PERSON"
            ],
            [
                1475,
                1481,
                "PERSON"
            ],
            [
                1487,
                1493,
                "PERSON"
            ],
            [
                1519,
                1527,
                "PERSON"
            ],
            [
                1700,
                1709,
                "PERSON"
            ],
            [
                1714,
                1719,
                "PERSON"
            ],
            [
                1780,
                1786,
                "PERSON"
            ],
            [
                1795,
                1802,
                "PERSON"
            ],
            [
                1891,
                1899,
                "PERSON"
            ],
            [
                1939,
                1945,
                "PERSON"
            ],
            [
                1977,
                1985,
                "PERSON"
            ],
            [
                2000,
                2009,
                "PERSON"
            ],
            [
                2066,
                2076,
                "PERSON"
            ],
            [
                2082,
                2091,
                "PERSON"
            ],
            [
                2099,
                2104,
                "PERSON"
            ],
            [
                2159,
                2165,
                "PERSON"
            ],
            [
                2172,
                2178,
                "PERSON"
            ],
            [
                2251,
                2258,
                "PERSON"
            ],
            [
                2281,
                2289,
                "PERSON"
            ],
            [
                2321,
                2327,
                "PERSON"
            ],
            [
                2359,
                2367,
                "PERSON"
            ],
            [
                2377,
                2384,
                "PERSON"
            ],
            [
                2434,
                2441,
                "PERSON"
            ],
            [
                2489,
                2497,
                "PERSON"
            ],
            [
                2511,
                2518,
                "PERSON"
            ],
            [
                2528,
                2536,
                "PERSON"
            ],
            [
                2549,
                2556,
                "PERSON"
            ],
            [
                2659,
                2668,
                "PERSON"
            ],
            [
                2677,
                2684,
                "PERSON"
            ],
            [
                2726,
                2732,
                "PERSON"
            ],
            [
                2763,
                2772,
                "PERSON"
            ],
            [
                2793,
                2799,
                "PERSON"
            ],
            [
                2820,
                2827,
                "PERSON"
            ],
            [
                2878,
                2884,
                "PERSON"
            ],
            [
                2917,
                2921,
                "PERSON"
            ],
            [
                2956,
                2962,
                "PERSON"
            ],
            [
                2974,
                2983,
                "PERSON"
            ],
            [
                2997,
                3004,
                "PERSON"
            ],
            [
                3044,
                3052,
                "PERSON"
            ],
            [
                3075,
                3084,
                "PERSON"
            ],
            [
                3092,
                3101,
                "PERSON"
            ],
            [
                3154,
                3162,
                "PERSON"
            ],
            [
                3171,
                3178,
                "PERSON"
            ],
            [
                3187,
                3195,
                "PERSON"
            ],
            [
                3199,
                3205,
                "PERSON"
            ],
            [
                3286,
                3293,
                "PERSON"
            ],
            [
                3352,
                3359,
                "PERSON"
            ],
            [
                3401,
                3407,
                "PERSON"
            ],
            [
                3414,
                3422,
                "PERSON"
            ],
            [
                3428,
                3435,
                "PERSON"
            ],
            [
                3478,
                3485,
                "PERSON"
            ],
            [
                3511,
                3518,
                "PERSON"
            ],
            [
                3525,
                3532,
                "PERSON"
            ],
            [
                3564,
                3570,
                "PERSON"
            ],
            [
                3662,
                3668,
                "PERSON"
            ],
            [
                3687,
                3695,
                "PERSON"
            ],
            [
                3752,
                3759,
                "PERSON"
            ],
            [
                3776,
                3781,
                "PERSON"
            ],
            [
                3833,
                3839,
                "PERSON"
            ],
            [
                3979,
                3984,
                "PERSON"
            ],
            [
                4006,
                4016,
                "PERSON"
            ],
            [
                4031,
                4038,
                "PERSON"
            ],
            [
                4066,
                4073,
                "PERSON"
            ],
            [
                4093,
                4102,
                "PERSON"
            ],
            [
                4126,
                4133,
                "PERSON"
            ],
            [
                4145,
                4152,
                "PERSON"
            ],
            [
                4217,
                4224,
                "PERSON"
            ],
            [
                4229,
                4238,
                "PERSON"
            ],
            [
                4261,
                4268,
                "PERSON"
            ],
            [
                4276,
                4283,
                "PERSON"
            ],
            [
                4338,
                4345,
                "PERSON"
            ],
            [
                4353,
                4361,
                "PERSON"
            ],
            [
                4370,
                4377,
                "PERSON"
            ],
            [
                4408,
                4415,
                "PERSON"
            ]
        ]
    }
),(
    "vumc vis one hund { Tylar red oaks white, tyron { Illiana e 719 thompson ln mrn: 047717 { Kataleya 361, dob: 7/27/1969, legal sex: m vande { Dagmar rbilt imaging of one hundred o { Seneca aks visit date: 12/18/2023 nashville tn 37204 12/18/2023 - appointment in vanderbilt imaging services one hundred oaks ( { Akia continued) after visit summary (continued) after visit su { Honora mmary vanderbilt health tyrone white mrn: 04 { Larkin 7717361 instructions: yo { Sedona ur next steps go reason for hospitalization dec return 2:00 pm your  { Geoff primary diagnosis was: not on file 5 arrive by 1:45 pm a { Soloman nton jordan de witte, md vanderbilt { Alexandros  one hundred oaks care providers primary care north 719 thompson ln provider service role specialty suite 20400 peterson, general internal attending internal medicine nashville tn 37204 neeraja b, md medic { Alexi ine 615-936-2187 you have more future appointments. please allergies date revi { Shae ewed: 11/ { Jani 17 { Sherell / { Alvena 2023 review your full appointment list.  allergies what's next:  { Duran dec return with anton jordan de witte, vanderbilt one 5 md hu { Rapheal ndred oaks tuesday { Seven  dec 5, 2023 2:00 pm (arrive by primary care north 1:45 pm) 719 thompson ln suite 2040 { Katherina 0 nashvil { Marlin le tn 3 { Naomy 7204 615-936-2187 dec return with  { Shriya de { Keasia bra l { Lyndsie  greenspan, vanderb { Renetta ilt diabetes 12 aprn and endocrinology tuesday dec 12, 2023 2:00 { Lynden  p { Britteny m (arrive by 1215 21st ave s 1:45 pm) 8th fl, suite 8210 nashville tn 37232 615-34 { Clarisse 3-8332 d { Hawa ec mri lumbar spine wo vanderbilt imaging 18 contrast servic { Cameo es one monday dec 18, 2023 3:45 pm (arrive by hundred oaks 3:25 { Kathey  p { Michala m) 719 thompson ln adult in { Zinnia structions: suite 23300 if you have any implanted medical nash { Hanson ville t { Terresa n 37204 devices, plea { Halo se bring your identification  { Sherice 615-936-3606 card { Veva  with you. the card should include the ser { Geena ial number so that our team can ver { Luis ify that it is safe to enter the mri. my { Robbi  h { Hashim ealth a { Jalyssa t vanderbilt view yo { Sia ur  { Holly after visit summary an { Jennefer d more online at https:// myhealthatvanderbilt.com/ ty { Kasha rone white (mrn: 047717361) (7/27/1969) printed at 11/17/2023 1:54 pm page 1 of 2 epic printed on 10/3/24 { Naftali  7:12 am page 1365,vumc vis one hundred oaks white, tyrone  { Veer 71 { Burnice 9  { Ashby thomps { Benji on ln mrn: 047717361, dob: 7/27/196 { Arno 9, legal sex: m vanderbilt imaging of one hundred oaks vis { Dalene it date: 1 { Hermina 2/18/2023 nashville tn 37204 12/18/2023 - appointment in  { Jayci vanderbilt imaging services one hundred oaks (continued) after visit summary (co { Joannie ntinued { Channel ) what's next: (continued) dec return w { Banks ith anna mari { Derry e burgner, md vanderbilt nephrology/renal 20 wednesday dec 20, 2023 3:45 pm (arrive by 3:3 { Dondre 0 pm) transplant clinic 1301 medical center dr suite 2501 nashville tn 37232 615-343-7592 jan return with anton jordan de witte, md vanderbi { Jamieson lt one hundred oaks 23 tuesday jan 23, 2024 10:00 am (arrive by 9:45 am) primary care north 2024 719 thom { Laine pson ln suite 20400 nashville tn 37204 61 { Toni 5-936-2187 feb new patient wit { Jama h gretchen elizabeth schlosser covell, md va { Aayan nde { Jospeh rbilt neur { Jr ology 6 tuesday feb 6, 2024 2:20 pm (arrive by 2:05 pm) 134 pewitt dr 2 { Lary 024 suite 200 brentwood tn 37027 615 { Odie -936-006 { Shemika 0 your facility administered medication list notice cannot displa { Shylah y patient medications because the p { Collier a { Ula tien { Efrem t h { Connor as not yet been checked  { Rhylee in. medicat { Lyn ion list notice cannot display patient medications because the patient has not yet been checked in. { Ashlynne  press ganey survey we value your input. you will receive a su { Floria rvey from press ganey after your vi { Melannie sit. we use press ganey to { Aaryan  help us learn about your experience at vanderbilt. please fi { Bernell ll out the survey. it will only take a few minutes and can help us improve care for a { Raynaldo ll patien { Lyanna ts. we use your fee { Janet dback to recognize { Anushka  outstanding service from our t { Gaines eam members and to iden { Tuan tify opportunities to improve care for future pa { Kyley tients. tyrone white (mrn: 047717361) (7/27/1969) printed at 11/17/2023 1:54 p { Merlyn m page 2 of 2 epic referral mri/cat/pet sca { Stevi n  { Mildred #1636 { Niall 9116 [last edited by referral, end  { Angele of day on 1/1/2024 { Journie  0430] reason: specialty services req { Lavelle uired priority: routine { Randie  class: internal status: closed - system closed - expired printed on 10/3/24 7:12 am page 1366",
    {
        "entities": [
            [
                20,
                26,
                "PERSON"
            ],
            [
                50,
                58,
                "PERSON"
            ],
            [
                90,
                99,
                "PERSON"
            ],
            [
                141,
                148,
                "PERSON"
            ],
            [
                181,
                188,
                "PERSON"
            ],
            [
                311,
                316,
                "PERSON"
            ],
            [
                376,
                383,
                "PERSON"
            ],
            [
                430,
                437,
                "PERSON"
            ],
            [
                464,
                471,
                "PERSON"
            ],
            [
                542,
                548,
                "PERSON"
            ],
            [
                607,
                615,
                "PERSON"
            ],
            [
                653,
                664,
                "PERSON"
            ],
            [
                872,
                878,
                "PERSON"
            ],
            [
                959,
                964,
                "PERSON"
            ],
            [
                976,
                981,
                "PERSON"
            ],
            [
                986,
                994,
                "PERSON"
            ],
            [
                998,
                1005,
                "PERSON"
            ],
            [
                1072,
                1078,
                "PERSON"
            ],
            [
                1142,
                1150,
                "PERSON"
            ],
            [
                1171,
                1177,
                "PERSON"
            ],
            [
                1266,
                1276,
                "PERSON"
            ],
            [
                1288,
                1295,
                "PERSON"
            ],
            [
                1305,
                1311,
                "PERSON"
            ],
            [
                1348,
                1355,
                "PERSON"
            ],
            [
                1360,
                1367,
                "PERSON"
            ],
            [
                1375,
                1383,
                "PERSON"
            ],
            [
                1405,
                1413,
                "PERSON"
            ],
            [
                1480,
                1487,
                "PERSON"
            ],
            [
                1492,
                1501,
                "PERSON"
            ],
            [
                1586,
                1595,
                "PERSON"
            ],
            [
                1606,
                1611,
                "PERSON"
            ],
            [
                1674,
                1680,
                "PERSON"
            ],
            [
                1746,
                1753,
                "PERSON"
            ],
            [
                1758,
                1766,
                "PERSON"
            ],
            [
                1796,
                1803,
                "PERSON"
            ],
            [
                1868,
                1875,
                "PERSON"
            ],
            [
                1885,
                1893,
                "PERSON"
            ],
            [
                1917,
                1922,
                "PERSON"
            ],
            [
                1954,
                1962,
                "PERSON"
            ],
            [
                1982,
                1987,
                "PERSON"
            ],
            [
                2032,
                2038,
                "PERSON"
            ],
            [
                2076,
                2081,
                "PERSON"
            ],
            [
                2124,
                2130,
                "PERSON"
            ],
            [
                2135,
                2142,
                "PERSON"
            ],
            [
                2152,
                2160,
                "PERSON"
            ],
            [
                2183,
                2187,
                "PERSON"
            ],
            [
                2193,
                2199,
                "PERSON"
            ],
            [
                2224,
                2233,
                "PERSON"
            ],
            [
                2290,
                2296,
                "PERSON"
            ],
            [
                2404,
                2412,
                "PERSON"
            ],
            [
                2474,
                2479,
                "PERSON"
            ],
            [
                2484,
                2492,
                "PERSON"
            ],
            [
                2497,
                2503,
                "PERSON"
            ],
            [
                2512,
                2518,
                "PERSON"
            ],
            [
                2556,
                2561,
                "PERSON"
            ],
            [
                2622,
                2629,
                "PERSON"
            ],
            [
                2642,
                2650,
                "PERSON"
            ],
            [
                2710,
                2716,
                "PERSON"
            ],
            [
                2799,
                2807,
                "PERSON"
            ],
            [
                2817,
                2825,
                "PERSON"
            ],
            [
                2867,
                2873,
                "PERSON"
            ],
            [
                2889,
                2895,
                "PERSON"
            ],
            [
                2988,
                2995,
                "PERSON"
            ],
            [
                3138,
                3147,
                "PERSON"
            ],
            [
                3255,
                3261,
                "PERSON"
            ],
            [
                3305,
                3310,
                "PERSON"
            ],
            [
                3343,
                3348,
                "PERSON"
            ],
            [
                3395,
                3401,
                "PERSON"
            ],
            [
                3407,
                3414,
                "PERSON"
            ],
            [
                3427,
                3430,
                "PERSON"
            ],
            [
                3504,
                3509,
                "PERSON"
            ],
            [
                3548,
                3553,
                "PERSON"
            ],
            [
                3564,
                3572,
                "PERSON"
            ],
            [
                3640,
                3647,
                "PERSON"
            ],
            [
                3685,
                3693,
                "PERSON"
            ],
            [
                3697,
                3701,
                "PERSON"
            ],
            [
                3708,
                3714,
                "PERSON"
            ],
            [
                3720,
                3727,
                "PERSON"
            ],
            [
                3754,
                3761,
                "PERSON"
            ],
            [
                3775,
                3779,
                "PERSON"
            ],
            [
                3881,
                3890,
                "PERSON"
            ],
            [
                3955,
                3962,
                "PERSON"
            ],
            [
                4000,
                4009,
                "PERSON"
            ],
            [
                4038,
                4045,
                "PERSON"
            ],
            [
                4109,
                4117,
                "PERSON"
            ],
            [
                4205,
                4214,
                "PERSON"
            ],
            [
                4226,
                4233,
                "PERSON"
            ],
            [
                4255,
                4261,
                "PERSON"
            ],
            [
                4282,
                4290,
                "PERSON"
            ],
            [
                4324,
                4331,
                "PERSON"
            ],
            [
                4357,
                4362,
                "PERSON"
            ],
            [
                4413,
                4419,
                "PERSON"
            ],
            [
                4500,
                4507,
                "PERSON"
            ],
            [
                4553,
                4559,
                "PERSON"
            ],
            [
                4564,
                4572,
                "PERSON"
            ],
            [
                4580,
                4586,
                "PERSON"
            ],
            [
                4624,
                4631,
                "PERSON"
            ],
            [
                4652,
                4660,
                "PERSON"
            ],
            [
                4700,
                4708,
                "PERSON"
            ],
            [
                4734,
                4741,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical cen { Geovany ter east white { Park , tyrone 1211 medical cent { Reynolds er dr mrn: 047717361,  { Brayleigh dob: 7/27/1969, legal sex: m nashville tn 37232 v { Solange isit date: 9/5/2023 09/05/2023 - of { Maddix fice visit in vanderbilt diabetes and endocrinology (continued) letters (continued) it has been my pleasure to s { Ceara erve  { Bonifacio mr. white. he was last seen a { Edrick nd treated at our clinic on 9/5/2023. please s { Lyssa ee t { Calder he  { Jayshawn attached information rela { Leandre ted to that v { Alexie isit.  { Lasandra please contact us with { Amaiya  any ques { Amberlyn tions or concerns at the  { Diona number below, via careeveryw { Nader here or vanderbilt healt { Octavious h connect system. sincerely, debra  { Karalyn l greenspan, aprn dept: 615-343-8332 cc:  endo adult clinic return visit date of { Skyy  service: 9/5/2023 patient: tyrone white dob: 7/27/1969 m { Anh rn: 047717361 ref { Keiry erring provider: lippard, giles a, aprn primary care pr { Dashon ovider: giles a lippa { Jarek rd, aprn reason for visit: followup of diabetes { Desteny  mellitus subjective history of pr { Dustin esent illness: tyronne  { Kalin returns today with female friend and is most concerned about nasal drainage and chokin { Darry g. sta { Agustina tes he ran out of his n { Koren ose spray (azelastine last pres { Sumer cribed) { Blaize . he de { Matthews nie { Cailee s fevers, chills or  { Lauran other constitutional sx. he last saw me in april for his type 2 diabetes { Linh  mellitus. it appears his community health pharmd changed his rybelsus to trulicity 0.75mg qd a { Markie long with lantus 5 units qhs and he states his sugars are doing much better. h { Rio e uses his libre 2 sensor. . bg in  { Tula clinic 137. libre sensor data  { Rayann : printed on 10/3/24 { Baruch  7:1 { Burnett 3 am page 1817,vumc a { Sheryll dult medical center east white, tyrone 1211 medical center  { Teal dr mrn: 047717 { Dalvin 361, dob: 7/27/1969, legal sex: m nashvil { Kaelan le tn 37232 visit date: 9/5/2023 09/05/2023 - offi { Nissa ce visit in vanderbilt diabetes and endocrinology (continued) letters (continued) snapshot freest { Aksel yle libre august 2023 5 september 2023 (28 da { Khalif ys)  { Kimani glucose logge { Anglea d carbs { Asiya  average glucose daily carbs grams/day vera { Jia ge 350 lucose 141 mg/dl mg/ { Cornelio dl above target 20 % l { Lev ogged insulin 180 medien in target 76 % rapid-acting ins { Delaina u { Marye lin un { Sera its/day 70 below target 4 % long-acting insulin unit { Tenesha s/day 10th to 90th percentile 0 00:0 { Jonathen 0 06:00 12 { Emiko :00 { Leonia  18:00 00:00 total daily insulin { Azure  units { Evaline /day low glucose events low glucose 15 70 vents mg/dl 60 verag { Adreanna e duration 112 min 5 { Marline 0 40 00:00 06:00 12 { Aleya :0 { Liah 0 18:00 00:00 daily patterns (with ambulatory glucose profile) freestyle libre 9 august 2023 - 5 september 2023 (28 days) daily average 00:00 02:00 { Samarah  04:00 06:00 08 { Kordell :00 10:00 12:00 14:00 16:00 18:00 20:00 22:00 00:00 glucose 14 { Larue 1 146 158 143 129 129 143 139 147 135  { Alyana 141  { Dorthey 138 140 mg/dl 350 mg/dl 300 250 200 1 { Lashon 80 150 median target range 100 70 50 10th to 90th percentile 25th to 75th percentile 0 00:00 02:00 04:00 06:00 { Tadd  08:00 10:00 12:00 14:00 16:00 18:00 20:00 22:00 00:00 daily average average /number n { Marjean t down i note was seen in triage on 8/22 with co { Missouri mplaints of chestpain. workup show { Naimah ed  of isch { Solana aemic heart disease but labs did show worsening renal failure with efgr of 15 down from 27 the month before. bun u { Winslow p to 42 { Bebe . he admits to occasional naus { Corena e { Darlena a with vomiting and abdominal distention. also having urinary inc { Evalina on { Laveta tinence at night. he apparently signed out ama before he could be evaluated by nephrology. renal us in june show { Aspen ed . { Decker  patient is scheduled to see his pcp in the morning. rev { Ramel iew of systems review of syste { Jacquie ms constitutional: positive for  { Augusto decreased appetite and weight gain. negative for { Cashton  chills, diaphoresis, fever, malaise/fatigue a { Connell nd night sweats. hent: positiv { Stevenson e for hoarse voice. c/o constant clear nasal discharge a { Cammy nd post nasal drip causing hi { Gaynelle m to cough eyes: negative.  { Lovella printed on 10/3/24  { Nikkia 7:13 am page 18 { Georgie 18",
    {
        "entities": [
            [
                25,
                33,
                "PERSON"
            ],
            [
                50,
                55,
                "PERSON"
            ],
            [
                84,
                93,
                "PERSON"
            ],
            [
                118,
                128,
                "PERSON"
            ],
            [
                180,
                188,
                "PERSON"
            ],
            [
                226,
                233,
                "PERSON"
            ],
            [
                348,
                354,
                "PERSON"
            ],
            [
                362,
                372,
                "PERSON"
            ],
            [
                404,
                411,
                "PERSON"
            ],
            [
                460,
                466,
                "PERSON"
            ],
            [
                473,
                480,
                "PERSON"
            ],
            [
                486,
                495,
                "PERSON"
            ],
            [
                523,
                531,
                "PERSON"
            ],
            [
                547,
                554,
                "PERSON"
            ],
            [
                563,
                572,
                "PERSON"
            ],
            [
                597,
                604,
                "PERSON"
            ],
            [
                616,
                625,
                "PERSON"
            ],
            [
                653,
                659,
                "PERSON"
            ],
            [
                690,
                696,
                "PERSON"
            ],
            [
                723,
                733,
                "PERSON"
            ],
            [
                771,
                779,
                "PERSON"
            ],
            [
                862,
                867,
                "PERSON"
            ],
            [
                927,
                931,
                "PERSON"
            ],
            [
                951,
                957,
                "PERSON"
            ],
            [
                1015,
                1022,
                "PERSON"
            ],
            [
                1046,
                1052,
                "PERSON"
            ],
            [
                1102,
                1110,
                "PERSON"
            ],
            [
                1147,
                1154,
                "PERSON"
            ],
            [
                1180,
                1186,
                "PERSON"
            ],
            [
                1275,
                1281,
                "PERSON"
            ],
            [
                1290,
                1299,
                "PERSON"
            ],
            [
                1325,
                1331,
                "PERSON"
            ],
            [
                1365,
                1371,
                "PERSON"
            ],
            [
                1381,
                1388,
                "PERSON"
            ],
            [
                1398,
                1407,
                "PERSON"
            ],
            [
                1413,
                1420,
                "PERSON"
            ],
            [
                1443,
                1450,
                "PERSON"
            ],
            [
                1525,
                1530,
                "PERSON"
            ],
            [
                1628,
                1635,
                "PERSON"
            ],
            [
                1716,
                1720,
                "PERSON"
            ],
            [
                1758,
                1763,
                "PERSON"
            ],
            [
                1796,
                1803,
                "PERSON"
            ],
            [
                1826,
                1833,
                "PERSON"
            ],
            [
                1840,
                1848,
                "PERSON"
            ],
            [
                1872,
                1880,
                "PERSON"
            ],
            [
                1942,
                1947,
                "PERSON"
            ],
            [
                1964,
                1971,
                "PERSON"
            ],
            [
                2015,
                2022,
                "PERSON"
            ],
            [
                2075,
                2081,
                "PERSON"
            ],
            [
                2181,
                2187,
                "PERSON"
            ],
            [
                2235,
                2242,
                "PERSON"
            ],
            [
                2249,
                2256,
                "PERSON"
            ],
            [
                2272,
                2279,
                "PERSON"
            ],
            [
                2289,
                2295,
                "PERSON"
            ],
            [
                2341,
                2345,
                "PERSON"
            ],
            [
                2375,
                2384,
                "PERSON"
            ],
            [
                2409,
                2413,
                "PERSON"
            ],
            [
                2472,
                2480,
                "PERSON"
            ],
            [
                2484,
                2490,
                "PERSON"
            ],
            [
                2499,
                2504,
                "PERSON"
            ],
            [
                2559,
                2567,
                "PERSON"
            ],
            [
                2606,
                2615,
                "PERSON"
            ],
            [
                2628,
                2634,
                "PERSON"
            ],
            [
                2640,
                2647,
                "PERSON"
            ],
            [
                2682,
                2688,
                "PERSON"
            ],
            [
                2697,
                2705,
                "PERSON"
            ],
            [
                2770,
                2779,
                "PERSON"
            ],
            [
                2802,
                2810,
                "PERSON"
            ],
            [
                2832,
                2838,
                "PERSON"
            ],
            [
                2843,
                2848,
                "PERSON"
            ],
            [
                2998,
                3006,
                "PERSON"
            ],
            [
                3024,
                3032,
                "PERSON"
            ],
            [
                3097,
                3103,
                "PERSON"
            ],
            [
                3144,
                3151,
                "PERSON"
            ],
            [
                3158,
                3166,
                "PERSON"
            ],
            [
                3206,
                3213,
                "PERSON"
            ],
            [
                3326,
                3331,
                "PERSON"
            ],
            [
                3420,
                3428,
                "PERSON"
            ],
            [
                3479,
                3488,
                "PERSON"
            ],
            [
                3525,
                3532,
                "PERSON"
            ],
            [
                3546,
                3553,
                "PERSON"
            ],
            [
                3670,
                3678,
                "PERSON"
            ],
            [
                3688,
                3693,
                "PERSON"
            ],
            [
                3726,
                3733,
                "PERSON"
            ],
            [
                3737,
                3745,
                "PERSON"
            ],
            [
                3813,
                3821,
                "PERSON"
            ],
            [
                3826,
                3833,
                "PERSON"
            ],
            [
                3948,
                3954,
                "PERSON"
            ],
            [
                3961,
                3968,
                "PERSON"
            ],
            [
                4027,
                4033,
                "PERSON"
            ],
            [
                4066,
                4074,
                "PERSON"
            ],
            [
                4109,
                4117,
                "PERSON"
            ],
            [
                4168,
                4176,
                "PERSON"
            ],
            [
                4225,
                4233,
                "PERSON"
            ],
            [
                4266,
                4276,
                "PERSON"
            ],
            [
                4335,
                4341,
                "PERSON"
            ],
            [
                4373,
                4382,
                "PERSON"
            ],
            [
                4412,
                4420,
                "PERSON"
            ],
            [
                4442,
                4449,
                "PERSON"
            ],
            [
                4467,
                4475,
                "PERSON"
            ]
        ]
    }
),(
    "vumc h { Male endersonvi { Savon lle - anderson white, tyron { Donnetta e 1 { Mea 28 n an { Oaklee derson { Rogan  ln mrn: 047717361, dob: 7/27/1969, lega { Teague l  { Alaysha s { Toi ex { Eulalio : m hendersonville tn 370 { Garet 75 visi { Rivky t date: 4/20/2023 04/20/2023 - appointment in va { Samirah nderbilt primary car { Jamaica e hendersonville (co { Tyriq ntinued) referr { Aislyn al (continued) triage triage information decision: none schedule b { Emmi y date: 5/ { Christan 17/2023 coverages uhc commu { Curry nit { Kiernan y dual sn { Nova p plan: { Orlin  uhc { Damita  community dual covered: covered from: 1/1/2024 to: 1/31/2024 snp me { Dathan mber #: 127670950 humana me { Tray dicare hmo oon plan: humana { Aldon  medicare hmo covere { Milagro d: covered from: 3/1/2023 to: 5/31/2023 oon member # { Romelia : h75033225 a { Tyjuan merivantage wellpoint { Yancey  medicare plan: amerivantage c { Caylie overe { Joceline d: covered from: 6/1/2023 to:  { Linden 2/29/2024 amerigro { Saphira up we { Shamira llpoint ma member #: 768w12411 { Shanise  zz { Sharice zmcaid of ten { Algie nessee plan: medicaid supplem { Avon ental covered: cove { Chistopher red from: 6/1/2022 to: 12/31/2023 member #: td525606373 tc tenncare s { Elyjah elect pl { Karlton an: tc select { Vinh   covered: covered f { Myasia ro { Hassie m:  { Layan 8/30/2024 to: 8/30/2024  { Vertie member #: ze { Daunte dm { Jamall 13004089  { Micahel messages appointment  { Analiese rescheduled from to sent and delivered mychart, generic whit { Amadeo e, ty { Chloie rone 4/19 { Kemberly /2023 10:48 am last { Kloey  read in my health at vanderbi { Raychel lt not r { Derron ead  { Ruger appointmen { Tysen t information: visit type: return date: 4/2 { Aoife 0/2023 dept: vanderbilt primary  { Eisley care he { Agapito ndersonville provider: { Damonte  giles a lippar { Denim d { Juventino  time: { Page   { Treyson 10:40 am le { Telly ngt { Florrie h: 20 min app { Hala t s { Martin tatus: schedu { Stephannie led printed on 10/3/24 7:13 am { Alexcia  page { Kylea  2401, { Shanay vumc henders { Tonette onville - ander { Amauri son white, tyrone 128 n anderson ln mrn: 047717361, dob: 7/27/1969,  { Johney le { Steffen gal sex: m hendersonville tn 37075 visit { Bernie  date: 4/20 { Inaya /2023 04/20/2023 - appointment { Shenna  in vanderbi { Sultan lt primary care  { Hermine hende { Rima rsonville ( { Cavan contin { Montel ued) mess { Jeannetta ages (continued) original appoint { Keiara ment i { Lucienne nf { Detrick ormation: visit type: acute { Jarron  date { Alysse : 4/18/2023 dept: vanderbilt primary care hendersonville provider: gil { Fabiana es a { Lanelle  lippard time: 3:00 pm lengt { Lark h: 20 { Sir  m { Lannie in printed on 10/3/24 7 { Makinley :13 am page 24 { Nelida 02",
    {
        "entities": [
            [
                9,
                14,
                "PERSON"
            ],
            [
                27,
                33,
                "PERSON"
            ],
            [
                63,
                72,
                "PERSON"
            ],
            [
                78,
                82,
                "PERSON"
            ],
            [
                92,
                99,
                "PERSON"
            ],
            [
                108,
                114,
                "PERSON"
            ],
            [
                157,
                164,
                "PERSON"
            ],
            [
                169,
                177,
                "PERSON"
            ],
            [
                181,
                185,
                "PERSON"
            ],
            [
                190,
                198,
                "PERSON"
            ],
            [
                226,
                232,
                "PERSON"
            ],
            [
                242,
                248,
                "PERSON"
            ],
            [
                299,
                307,
                "PERSON"
            ],
            [
                330,
                338,
                "PERSON"
            ],
            [
                361,
                367,
                "PERSON"
            ],
            [
                385,
                392,
                "PERSON"
            ],
            [
                461,
                466,
                "PERSON"
            ],
            [
                479,
                488,
                "PERSON"
            ],
            [
                518,
                524,
                "PERSON"
            ],
            [
                530,
                538,
                "PERSON"
            ],
            [
                550,
                555,
                "PERSON"
            ],
            [
                565,
                571,
                "PERSON"
            ],
            [
                578,
                585,
                "PERSON"
            ],
            [
                656,
                663,
                "PERSON"
            ],
            [
                693,
                698,
                "PERSON"
            ],
            [
                728,
                734,
                "PERSON"
            ],
            [
                757,
                765,
                "PERSON"
            ],
            [
                820,
                828,
                "PERSON"
            ],
            [
                844,
                851,
                "PERSON"
            ],
            [
                875,
                882,
                "PERSON"
            ],
            [
                915,
                922,
                "PERSON"
            ],
            [
                930,
                939,
                "PERSON"
            ],
            [
                972,
                979,
                "PERSON"
            ],
            [
                1000,
                1008,
                "PERSON"
            ],
            [
                1016,
                1024,
                "PERSON"
            ],
            [
                1057,
                1065,
                "PERSON"
            ],
            [
                1071,
                1079,
                "PERSON"
            ],
            [
                1095,
                1101,
                "PERSON"
            ],
            [
                1133,
                1138,
                "PERSON"
            ],
            [
                1160,
                1171,
                "PERSON"
            ],
            [
                1243,
                1250,
                "PERSON"
            ],
            [
                1261,
                1269,
                "PERSON"
            ],
            [
                1285,
                1290,
                "PERSON"
            ],
            [
                1313,
                1320,
                "PERSON"
            ],
            [
                1325,
                1332,
                "PERSON"
            ],
            [
                1338,
                1344,
                "PERSON"
            ],
            [
                1371,
                1378,
                "PERSON"
            ],
            [
                1393,
                1400,
                "PERSON"
            ],
            [
                1405,
                1412,
                "PERSON"
            ],
            [
                1424,
                1432,
                "PERSON"
            ],
            [
                1456,
                1465,
                "PERSON"
            ],
            [
                1528,
                1535,
                "PERSON"
            ],
            [
                1543,
                1550,
                "PERSON"
            ],
            [
                1562,
                1571,
                "PERSON"
            ],
            [
                1593,
                1599,
                "PERSON"
            ],
            [
                1632,
                1640,
                "PERSON"
            ],
            [
                1651,
                1658,
                "PERSON"
            ],
            [
                1665,
                1671,
                "PERSON"
            ],
            [
                1684,
                1690,
                "PERSON"
            ],
            [
                1736,
                1742,
                "PERSON"
            ],
            [
                1777,
                1784,
                "PERSON"
            ],
            [
                1794,
                1802,
                "PERSON"
            ],
            [
                1827,
                1835,
                "PERSON"
            ],
            [
                1853,
                1859,
                "PERSON"
            ],
            [
                1863,
                1873,
                "PERSON"
            ],
            [
                1882,
                1887,
                "PERSON"
            ],
            [
                1891,
                1899,
                "PERSON"
            ],
            [
                1913,
                1919,
                "PERSON"
            ],
            [
                1925,
                1933,
                "PERSON"
            ],
            [
                1949,
                1954,
                "PERSON"
            ],
            [
                1960,
                1967,
                "PERSON"
            ],
            [
                1983,
                1994,
                "PERSON"
            ],
            [
                2027,
                2035,
                "PERSON"
            ],
            [
                2043,
                2049,
                "PERSON"
            ],
            [
                2058,
                2065,
                "PERSON"
            ],
            [
                2080,
                2088,
                "PERSON"
            ],
            [
                2106,
                2113,
                "PERSON"
            ],
            [
                2184,
                2191,
                "PERSON"
            ],
            [
                2196,
                2204,
                "PERSON"
            ],
            [
                2247,
                2254,
                "PERSON"
            ],
            [
                2268,
                2274,
                "PERSON"
            ],
            [
                2307,
                2314,
                "PERSON"
            ],
            [
                2329,
                2336,
                "PERSON"
            ],
            [
                2355,
                2363,
                "PERSON"
            ],
            [
                2371,
                2376,
                "PERSON"
            ],
            [
                2390,
                2396,
                "PERSON"
            ],
            [
                2405,
                2412,
                "PERSON"
            ],
            [
                2424,
                2434,
                "PERSON"
            ],
            [
                2470,
                2477,
                "PERSON"
            ],
            [
                2486,
                2495,
                "PERSON"
            ],
            [
                2500,
                2508,
                "PERSON"
            ],
            [
                2538,
                2545,
                "PERSON"
            ],
            [
                2553,
                2560,
                "PERSON"
            ],
            [
                2633,
                2641,
                "PERSON"
            ],
            [
                2648,
                2656,
                "PERSON"
            ],
            [
                2687,
                2692,
                "PERSON"
            ],
            [
                2700,
                2704,
                "PERSON"
            ],
            [
                2709,
                2716,
                "PERSON"
            ],
            [
                2742,
                2751,
                "PERSON"
            ],
            [
                2768,
                2775,
                "PERSON"
            ]
        ]
    }
),(
    "vumc the vanderbilt clinic white, tyrone 1301 medical center dr mrn: 04771736 { Waldemar 1, do { Zakariya b: 7/27/1969, legal sex: m the va { Nakeisha nderbil { Takara t clinic  { Kobi visit date: 9/16/202 { Armoni 4 nashville tn 37232-0028 09/16/2024 - appointmen { Latara t in vanderbilt nephrology/renal transplant clinic facesheet report patient demographics pati { Jonmichael ent name mrn legal do { Alexius b address phone white, tyrone 0477173 sex 7/27/1969 apt 705 615-260-229 { Indya 1  { Sadye (home) 61 { Antwaun  m 1101 edgehill ave 615-260-2291 (mobile) nashville { Marcoantonio  tn 37203 *preferred* hospital accou { Zaine nt name acct id class status primary coverage white, tyrone 1021453645 outpatient billed uhc community dual snp - uhc community dual snp guarantor account (for hospital account #102145364 { Ilda 5)  { Lashaunda re { Marysol lation to name pt service area  { Vayda active { Hussain ? ac { Jae ct type { Kayci  white, tyrone self vumc msa yes personal/family address phone ap { Eligio t 705 615-26 { Nicolai 0-22 { Margy 9 { Giovany 1 (h) 11 { Manfred 01 edgehill a { Terrica ve nashville, tn 37 { Quin 203 coverage information (for hospital acc { Djuan ount #102 { Marquell 1453645) f/o payor/plan precert # uhc community dual snp/uhc community dual s { Yechiel np subscriber subscriber # white, tyrone 1 { Louetta 27670950 address phone po box { Irine   { Kandyce 5220 kingston, ny 12402-5220 admission information current information attending prov { Jamere ider admitting  { Erminia provider admission type admission status freeman, genevieve clayton, elective u { Daphine nknown status aprn 6 { Darious 15-343-7592 a { Tahir dmission date/time  { Arlinda discharge date/time hospital se { Izabela rvice auth/cert status hospital area unit room/bed r { Delonte eferring provider de witte,  { Shamia anton jordan, md 09/16/2024 - appoint { Prescott ment in vanderbilt nephrology/renal transplant clinic (continued) visit inform { Ronney ation appointment information  { Arriana return canceled 9/16/2024 4:15 pm printed on 10/3/24  { Simcha 7:12 am page 15,vu { Antonietta mc the va { Jahaira nderb { Aayden ilt clinic white, tyrone 1301 med { Daulton ical cente { Jacee r dr mrn: 0477173 { Theodosia 61, dob: 7/27/1969, legal sex: m th { Torry e vanderbilt c { Charita linic visit date: 9/16/ { Meah 2024 nashville tn 37232-0028 09/16/2024 - appoint { Hartley ment in vanderbilt nephrology/renal transplant clinic { Morrison   { Adrina (contin { Christianne ued) visit information (continued) time provider department length 4:15 pm b { Shawnte urg { Webb ner, anna marie, md nephrology tvc 2 { Adalie  10 min refe { Kieran rral provider: de witte, anton jordan  { Taven enc fo { Jaxsen rm numb { Billiejo er: 30174819 notes: 2-3 month fuv history made on: 6/27/2024 3:35 pm by: gill, taylor d es c { Preslie anceled: 9/17/2 { Shawanna 024 2:59 pm by: napier, cynth { Sophronia ia a es cancel rsn: patient request communication tracking calls/messages text message on 9/16/2024 1415 phone number: 615-260-2291 message: result: phone not allowed medication list medicat { Shalynn ion list i this report is for docume { Taunya ntation purposes only. the patie { Dusten nt should not follow medication instructions within. { Hayle  for accurate instructions regarding medications, the patient should instead consult t { Veronique heir physician or after visit summary. active at the end of visit medications last reviewed by hicks, adam bradburn, dpm on  { Efraim 9/16/2024 { Keagan  09 { Temeka 36 famotidine 20 mg tablet (pepcid) [reconciled by ferguso { Apolonio n, sherri l, lpn on 1/11/2023 1 { Danthony 257] instructions: take 1 t { La ablet (20 mg total) by mouth every 12 hours. en { Sharell te { Kace red { Sedric  by: fergu { Terrion son, sherri l, lp { Alasia n entered on: 1/11/2023 montelukast 10 mg  { Ardelle tablet (singulair) ins { Briseyda tructions: take 1 tablet (10 mg total) by mouth every evening. authorized by: lippard, giles a, aprn ordered on: 8/8/2023 start date:  { Dalary 8/8/2023 quantity: 90 tablet refill: 3 r { Marrissa efills by 8/7/2024 cyclobenzaprine 5 mg tablet (flexeril) [reconciled by maples, chantis on 9/5/2023 1522] instructions: ta { Broden ke 1 tablet (5 mg total) by  { Charmayne mouth every 8 hours as needed. entered by: map { Gwenn les, chantis entered o { Jayce n: { Baylor  9/5/2023 start date: 7/22/2023 aspirin 81 mg tablet, { Kalen delayed release instructions: take 1 tablet (81 mg total) by mouth daily. authorized by: de witte, anton jordan, md ordered on { Kaydance : 9/ { Lynetta 7/2023 sta { Sianna rt date: 9/7/2023 quantity: 90  { Kalib tablet refill: 3 refills by 9/6/2024 printed on 10/3/24 7:12 am page 16",
    {
        "entities": [
            [
                80,
                89,
                "PERSON"
            ],
            [
                97,
                106,
                "PERSON"
            ],
            [
                142,
                151,
                "PERSON"
            ],
            [
                161,
                168,
                "PERSON"
            ],
            [
                180,
                185,
                "PERSON"
            ],
            [
                208,
                215,
                "PERSON"
            ],
            [
                267,
                274,
                "PERSON"
            ],
            [
                370,
                381,
                "PERSON"
            ],
            [
                405,
                413,
                "PERSON"
            ],
            [
                487,
                493,
                "PERSON"
            ],
            [
                498,
                504,
                "PERSON"
            ],
            [
                516,
                524,
                "PERSON"
            ],
            [
                579,
                592,
                "PERSON"
            ],
            [
                631,
                637,
                "PERSON"
            ],
            [
                827,
                832,
                "PERSON"
            ],
            [
                838,
                848,
                "PERSON"
            ],
            [
                853,
                861,
                "PERSON"
            ],
            [
                895,
                901,
                "PERSON"
            ],
            [
                910,
                918,
                "PERSON"
            ],
            [
                925,
                929,
                "PERSON"
            ],
            [
                939,
                945,
                "PERSON"
            ],
            [
                1013,
                1020,
                "PERSON"
            ],
            [
                1035,
                1043,
                "PERSON"
            ],
            [
                1050,
                1056,
                "PERSON"
            ],
            [
                1060,
                1068,
                "PERSON"
            ],
            [
                1079,
                1087,
                "PERSON"
            ],
            [
                1103,
                1111,
                "PERSON"
            ],
            [
                1133,
                1138,
                "PERSON"
            ],
            [
                1183,
                1189,
                "PERSON"
            ],
            [
                1201,
                1210,
                "PERSON"
            ],
            [
                1290,
                1298,
                "PERSON"
            ],
            [
                1343,
                1351,
                "PERSON"
            ],
            [
                1383,
                1389,
                "PERSON"
            ],
            [
                1393,
                1401,
                "PERSON"
            ],
            [
                1489,
                1496,
                "PERSON"
            ],
            [
                1514,
                1522,
                "PERSON"
            ],
            [
                1604,
                1612,
                "PERSON"
            ],
            [
                1635,
                1643,
                "PERSON"
            ],
            [
                1659,
                1665,
                "PERSON"
            ],
            [
                1687,
                1695,
                "PERSON"
            ],
            [
                1729,
                1737,
                "PERSON"
            ],
            [
                1792,
                1800,
                "PERSON"
            ],
            [
                1831,
                1838,
                "PERSON"
            ],
            [
                1878,
                1887,
                "PERSON"
            ],
            [
                1968,
                1975,
                "PERSON"
            ],
            [
                2008,
                2016,
                "PERSON"
            ],
            [
                2072,
                2079,
                "PERSON"
            ],
            [
                2100,
                2111,
                "PERSON"
            ],
            [
                2123,
                2131,
                "PERSON"
            ],
            [
                2139,
                2146,
                "PERSON"
            ],
            [
                2182,
                2190,
                "PERSON"
            ],
            [
                2203,
                2209,
                "PERSON"
            ],
            [
                2229,
                2239,
                "PERSON"
            ],
            [
                2277,
                2283,
                "PERSON"
            ],
            [
                2300,
                2308,
                "PERSON"
            ],
            [
                2334,
                2339,
                "PERSON"
            ],
            [
                2391,
                2399,
                "PERSON"
            ],
            [
                2455,
                2464,
                "PERSON"
            ],
            [
                2468,
                2475,
                "PERSON"
            ],
            [
                2485,
                2497,
                "PERSON"
            ],
            [
                2576,
                2584,
                "PERSON"
            ],
            [
                2590,
                2595,
                "PERSON"
            ],
            [
                2634,
                2641,
                "PERSON"
            ],
            [
                2656,
                2663,
                "PERSON"
            ],
            [
                2704,
                2710,
                "PERSON"
            ],
            [
                2719,
                2726,
                "PERSON"
            ],
            [
                2736,
                2745,
                "PERSON"
            ],
            [
                2840,
                2848,
                "PERSON"
            ],
            [
                2866,
                2875,
                "PERSON"
            ],
            [
                2907,
                2917,
                "PERSON"
            ],
            [
                3110,
                3118,
                "PERSON"
            ],
            [
                3157,
                3164,
                "PERSON"
            ],
            [
                3199,
                3206,
                "PERSON"
            ],
            [
                3261,
                3267,
                "PERSON"
            ],
            [
                3356,
                3366,
                "PERSON"
            ],
            [
                3493,
                3500,
                "PERSON"
            ],
            [
                3512,
                3519,
                "PERSON"
            ],
            [
                3525,
                3532,
                "PERSON"
            ],
            [
                3593,
                3602,
                "PERSON"
            ],
            [
                3636,
                3645,
                "PERSON"
            ],
            [
                3675,
                3678,
                "PERSON"
            ],
            [
                3728,
                3736,
                "PERSON"
            ],
            [
                3741,
                3746,
                "PERSON"
            ],
            [
                3752,
                3759,
                "PERSON"
            ],
            [
                3772,
                3780,
                "PERSON"
            ],
            [
                3800,
                3807,
                "PERSON"
            ],
            [
                3852,
                3860,
                "PERSON"
            ],
            [
                3885,
                3894,
                "PERSON"
            ],
            [
                4031,
                4038,
                "PERSON"
            ],
            [
                4081,
                4090,
                "PERSON"
            ],
            [
                4216,
                4223,
                "PERSON"
            ],
            [
                4254,
                4264,
                "PERSON"
            ],
            [
                4313,
                4319,
                "PERSON"
            ],
            [
                4344,
                4350,
                "PERSON"
            ],
            [
                4355,
                4362,
                "PERSON"
            ],
            [
                4418,
                4424,
                "PERSON"
            ],
            [
                4553,
                4562,
                "PERSON"
            ],
            [
                4569,
                4577,
                "PERSON"
            ],
            [
                4590,
                4597,
                "PERSON"
            ],
            [
                4631,
                4637,
                "PERSON"
            ]
        ]
    }
),(
    "vumc the vanderbilt clinic white, tyrone  { Huda 1301 m { Onyx edica { Waymond l center dr  { Nelia mrn: 047717361, dob: 7/27/1969, legal sex: m the vanderbilt clinic visit dat { Saba e: 2/7/2023 nashville tn 37232-0028 02/ { Berkeley 07/2023 - communica { Marly tion in vanderbilt nephrology/renal trans { Buddie plant clinic (conti { Fernand nued) medicati { Nathon on list (continued) authorize { Katrena d by: wang, zhijian, aprn ordered on: 2/7/2023 start date: 2/7/2023 end date: 6/3/2023 quantity { Wright : 12 capsule refill:  rema { Arian ining gabapentin 300 mg capsule { Ashante  (neurontin) d { Brienne isc { Dagny ont { Julieanna in { Sharlyn ued by: darks, tina m, aprn discontinued on: 3/13/2023 instr { Brooklyn uctions: t { Jerardo ake one capsule by mou { Anyah th three times a day authorized by: lippard, giles a,  { Lisset aprn ordered on: 2/9 { Ronisha /2023 start date: 2/9/2023 quanti { Kervin ty: 90 capsule refill:  remaining stopped in visit medicat { Ronal ions last reviewed { Alyanna  by wang, zhijian,  { Carmelina aprn on 2/7/2023 0853 ibuprofen 600 mg { Elysa  tablet (advil,motrin) [recon { Janyla ciled by woehler, krist { Miabella ina, rn on 1/11/2023 0756] discontin { Rowen ued  { Jarid by: wang, zhijian, aprn d { Taft iscontinued on: 2/7/2023 reason for discontinuation: other (cancelrx) clinical notes telephone encounter townsen { Allean d, pamela, rn at 2/7/2023 1628 author: townsend,  { Johanne pamela, rn serv { Mckinzie ice { Micheala : author type: registered nurse filed: 2/7/2023 4:28 { Zofia  pm en { Ethel counter date: 2/7/2023 status: s { Isaura igned editor: townsend, pamela, rn ( { Arlington registered nurse) message from zhijian wang, aprn sent at 2/7/2023 3:52 pm cst pam, please c { Jacolby all the pt i have revie { Jadin wed his today's lab 1. overall { Rajan  his renal function stable with egfr 25 today 2. glucose up, 315, please ask the  { Elfrieda pt to get { Keitha   { Loria better bs controlled 3. his vit { Temple d 1 { Byran 1 today, low, will add ergocalciferol 50,000 unit orally weekly x 12 weeks thanks electronically signed by townsend, pamela, rn at 2/7/2023 4:28 pm printed on 10/3/24 7:13 am pag { Saxon e 2643,vumc the vanderbilt clini { Jesusita c white, tyrone 1301 medical center dr mrn: 04 { Melda 7717361, dob: 7/27/1969, legal sex: m the vanderbilt clinic visit date: 2/7/2023 nashville tn 37232-0028 02/07/2023 - co { Teddy mmunicati { Bryn on in vanderbilt  { Lamon nephrology/renal transplant cli { Dava nic facesheet report patient demographics p { Torey atient name mrn legal dob address phon { Asad e white, tyrone 0477173 sex 7/27/1969 apt 705 615-260-2 { Etienne 291 (home) 61 m 1101 edgehill ave 615-260 { Kilian -2291 (mobile) nashvil { Anela le tn 37203 *preferred* hospital account not  { Christan on file admission information current informatio { Mikah n attending provider admitting provider admission typ { Zeus e admission st { Kambria a { Adair t { Corben us un { Darryle known status admission date/time discharge d { Elisabet ate/t { Wynne ime hospital { Tashawn  service auth/cert status hospital area unit room/bed referring provider 02/07/2023 - com { Silver munication in { Derric  vanderbilt nephrology/renal transplant clinic (continued) visi { Garey t information provider information encounter provider wang, zhijian, aprn department name address phone fax vanderbilt nephrology/renal 1301 medical center dr 615-343-7592 615-343-8216 transplant clinic suite 2501 nashville tn 37232 medication list medication list i this report is for documentation purposes only. the patient shoul { Lacey d not follow medication instructions within. for accurate instructions regarding medications, the patient should instead consult t { Britani heir ph { Conchita ysician or after visit summary. active at the end of visit m { Jamika edicati { Syeda ons last reviewed by wang, zhijian, aprn on 2/7/2023 0853 insulin g { Daylin largine (u-100) 100 unit/ml (3 ml) subc { Destry utaneous pen (lantus solosta { Alaura r,basaglar kwikpen) discontinued by:  { Earleen greenspan, debra l, aprn dis { Emmarie continue { Karsen d o { Daphney n: 2/14/2023 reason for discontinuation: discontinued by another clinician instructions: inject 25 units under the skin daily. aut { Haiden horized by: lippard, giles a, apr { Terah n  { Manford order { Annalyse ed on: 9 { Cheyanna /27/2022 start date: 9/27/2022 quantity: 9 ml refill: 11 refills by { Javien   { Fredda 9/27/2023 pioglitazone 30 mg tabl { Georganna et (actos) discontinued by: gree { Sam nspan, debra l, aprn discontinued on: 2/14/2023 reason  { Mickael for discontinuation: disco { Blondell ntinu { Burrell ed by another clinicia { Jacqueline n printed on 10/3/24 7:13 am page 26 { Javeon 44",
    {
        "entities": [
            [
                44,
                49,
                "PERSON"
            ],
            [
                58,
                63,
                "PERSON"
            ],
            [
                71,
                79,
                "PERSON"
            ],
            [
                94,
                100,
                "PERSON"
            ],
            [
                179,
                184,
                "PERSON"
            ],
            [
                226,
                235,
                "PERSON"
            ],
            [
                257,
                263,
                "PERSON"
            ],
            [
                307,
                314,
                "PERSON"
            ],
            [
                336,
                344,
                "PERSON"
            ],
            [
                361,
                368,
                "PERSON"
            ],
            [
                400,
                408,
                "PERSON"
            ],
            [
                506,
                513,
                "PERSON"
            ],
            [
                542,
                548,
                "PERSON"
            ],
            [
                582,
                590,
                "PERSON"
            ],
            [
                607,
                615,
                "PERSON"
            ],
            [
                621,
                627,
                "PERSON"
            ],
            [
                633,
                643,
                "PERSON"
            ],
            [
                648,
                656,
                "PERSON"
            ],
            [
                719,
                728,
                "PERSON"
            ],
            [
                741,
                749,
                "PERSON"
            ],
            [
                774,
                780,
                "PERSON"
            ],
            [
                837,
                844,
                "PERSON"
            ],
            [
                867,
                875,
                "PERSON"
            ],
            [
                911,
                918,
                "PERSON"
            ],
            [
                979,
                985,
                "PERSON"
            ],
            [
                1006,
                1014,
                "PERSON"
            ],
            [
                1036,
                1046,
                "PERSON"
            ],
            [
                1087,
                1093,
                "PERSON"
            ],
            [
                1125,
                1132,
                "PERSON"
            ],
            [
                1158,
                1167,
                "PERSON"
            ],
            [
                1206,
                1212,
                "PERSON"
            ],
            [
                1219,
                1225,
                "PERSON"
            ],
            [
                1253,
                1258,
                "PERSON"
            ],
            [
                1373,
                1380,
                "PERSON"
            ],
            [
                1432,
                1440,
                "PERSON"
            ],
            [
                1458,
                1467,
                "PERSON"
            ],
            [
                1473,
                1482,
                "PERSON"
            ],
            [
                1537,
                1543,
                "PERSON"
            ],
            [
                1552,
                1558,
                "PERSON"
            ],
            [
                1593,
                1600,
                "PERSON"
            ],
            [
                1639,
                1649,
                "PERSON"
            ],
            [
                1744,
                1752,
                "PERSON"
            ],
            [
                1778,
                1784,
                "PERSON"
            ],
            [
                1817,
                1823,
                "PERSON"
            ],
            [
                1907,
                1916,
                "PERSON"
            ],
            [
                1928,
                1935,
                "PERSON"
            ],
            [
                1939,
                1945,
                "PERSON"
            ],
            [
                1979,
                1986,
                "PERSON"
            ],
            [
                1992,
                1998,
                "PERSON"
            ],
            [
                2179,
                2185,
                "PERSON"
            ],
            [
                2220,
                2229,
                "PERSON"
            ],
            [
                2278,
                2284,
                "PERSON"
            ],
            [
                2407,
                2413,
                "PERSON"
            ],
            [
                2425,
                2430,
                "PERSON"
            ],
            [
                2450,
                2456,
                "PERSON"
            ],
            [
                2490,
                2495,
                "PERSON"
            ],
            [
                2541,
                2547,
                "PERSON"
            ],
            [
                2588,
                2593,
                "PERSON"
            ],
            [
                2651,
                2659,
                "PERSON"
            ],
            [
                2703,
                2710,
                "PERSON"
            ],
            [
                2735,
                2741,
                "PERSON"
            ],
            [
                2789,
                2798,
                "PERSON"
            ],
            [
                2849,
                2855,
                "PERSON"
            ],
            [
                2911,
                2916,
                "PERSON"
            ],
            [
                2933,
                2941,
                "PERSON"
            ],
            [
                2945,
                2951,
                "PERSON"
            ],
            [
                2955,
                2962,
                "PERSON"
            ],
            [
                2970,
                2978,
                "PERSON"
            ],
            [
                3025,
                3034,
                "PERSON"
            ],
            [
                3042,
                3048,
                "PERSON"
            ],
            [
                3063,
                3071,
                "PERSON"
            ],
            [
                3163,
                3170,
                "PERSON"
            ],
            [
                3186,
                3193,
                "PERSON"
            ],
            [
                3259,
                3265,
                "PERSON"
            ],
            [
                3600,
                3606,
                "PERSON"
            ],
            [
                3739,
                3747,
                "PERSON"
            ],
            [
                3757,
                3766,
                "PERSON"
            ],
            [
                3829,
                3836,
                "PERSON"
            ],
            [
                3846,
                3852,
                "PERSON"
            ],
            [
                3922,
                3929,
                "PERSON"
            ],
            [
                3971,
                3978,
                "PERSON"
            ],
            [
                4009,
                4016,
                "PERSON"
            ],
            [
                4056,
                4064,
                "PERSON"
            ],
            [
                4095,
                4103,
                "PERSON"
            ],
            [
                4114,
                4121,
                "PERSON"
            ],
            [
                4127,
                4135,
                "PERSON"
            ],
            [
                4268,
                4275,
                "PERSON"
            ],
            [
                4311,
                4317,
                "PERSON"
            ],
            [
                4322,
                4330,
                "PERSON"
            ],
            [
                4338,
                4347,
                "PERSON"
            ],
            [
                4358,
                4367,
                "PERSON"
            ],
            [
                4437,
                4444,
                "PERSON"
            ],
            [
                4448,
                4455,
                "PERSON"
            ],
            [
                4491,
                4501,
                "PERSON"
            ],
            [
                4536,
                4540,
                "PERSON"
            ],
            [
                4598,
                4606,
                "PERSON"
            ],
            [
                4635,
                4644,
                "PERSON"
            ],
            [
                4652,
                4660,
                "PERSON"
            ],
            [
                4685,
                4696,
                "PERSON"
            ],
            [
                4735,
                4742,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult hospital whi { Ricco te, tyro { Breauna ne  { Bryna 1211 medical center { Navya  dr. mrn: 047717361, dob: 7/27/ { Farhan 1969, leg { Steffan al sex: m nashville tn 37232-0004 adm: 7/26/2024, d/c: 7/26/2024 0 { Deondra 7/26/2024 - ed in vanderbilt emergency department (continued) after vis { Kelby it summary (continued) what's  { Gifford next (continued) aug pod#1 with daniel alejandro valen { Laree zuela vanderbilt eye institute 26 monday august 26 12:45 pm (arrive by 12:30 pm) 231 { Kruz 1 pierce ave 2024 save t { Marlowe ime. skip the line. vanderbilt eye instit { Quadir ute nashville tn 37232 you can now use self check-in for your visit. 61 { Addalynn 5-936-2020 use my health at vanderbilt to check-in for yo { Amarah ur visit right from your pho { Latifah ne. let us  { Tausha know you're here open th { Tisa e { Uma  mhav app on  { Alvino your phone when you arrive and click \"i { Cadence 'm here\" once you arrive at your appointment. sep af { Cordia t { Emalyn er surgery with daniel alejandro valenzuela vanderbilt eye institute 4 wedne { Gasper sday septembe { Griffith r 4 11:00 am (arrive by 10:45 am) 2311 pierce ave 2024 save time. skip the line. vanderbilt eye institute nashville tn 37232 you can now use self check-in fo { Kary r your visit. 615-936-2020 use my health at vanderbilt to check-in for  { Arlyne your visit right from your phone. let us know y { Evelia ou're here open the mhav app on your phone when you arrive and click \"i'm here\" once you arrive at your appointment. sep return with anna marie burgner vanderbilt nephrology/renal transplant 16 monday september { Indie  16 6 4:15  { Ozzy p { Virgilio m (arrive by 4:00 pm) clinic 2024 1301 me { Liya dical center dr save time. skip the line. suite 2501 you can now use self check-in { Marcus  for your visit. nashville tn 37232 use my health at vanderbil { Gavriel t to check-in for your { Yusra  v { Berlin isit right from { Jessiah  your 615-343-7592 phone. { Khamari  let us know you're here open the mhav { Linn  app o { Terran n your phone when you  { Aiko arrive an { Berna d click \"i'm here\" once you arrive at your appointment.  { Niamh tyrone white (mrn: 047717361) (7/27/1969) printed at 7/26/2024 5:07 am page 3 of 7 epic printed on 1 { Oneta 0/3/24 7:12 am page 565,vumc ad { Isaih ult hospital whi { Antonetta te, tyrone 1211 medical center dr. mrn: 047717361 { Bobette , dob: { Kynleigh  7/27/ { Meleah 1969, legal sex: m nashville tn 3723 { Natacha 2-0004 adm: 7/2 { Aneesa 6/ { Yanely 2024, d/c: 7/26/2024 07/26/ { Dayvon 2024 -  { Gracy ed in vanderbi { Okey lt emer { Samer gency de { Gurleen partment (continued) after vis { Rito it summary (continued) your medication list take these medications bd ultra-fine mini pen needle 31 { Jules  gauge x 3 { Kamia /16\" use one new pen needle two times a  { Nickie day for nee { Teo dle lantus generic { Treshawn  drug: pen needle, diabetic clotrimazole 1% cream apply 1 application topically 2 t { Avary imes a day for 30 co { Deysi mm { Kiyana only known as: lotrimin days. start freest { Zettie yle libre 2 sensor kit  { Noa 1 kit (1 each total) every 14 days. gene { Yariel ric { Genell  drug: flash glucose sensor lancets 33 gauge misc 1 lancet 2 times a d { Issabella ay. commonly known as: trueplus lancets lidocaine 2% mucosal jelly  { Saniah apply topically as needed for mild pain (apply to toe). commonly known as: xylocaine jelly start ask y { Glyn our doctor about these medica { Lebron tions ?  { Tommye acetaminophen 325 mg tablet { Armondo  take 2 tablets (650 mg total) by mouth every 6 hours c { Krystian ommonly known as: tylenol as needed for mild pain, moderate pain, headach { Emry es or ask fever. ask about: should | take this m { Shannah edication? ? { Tama  albuterol hfa 90 mcg/actuation inhaler inhale 2 puffs every 4 hou { Tandra rs as needed for wheezing. as { Keonte k ? aspirin 81 mg enteric coated t { Teddie ablet tak { Amyra e 1 tablet (81 mg total) by m { Breeann outh dail { Hertha y. ask ? atorvastatin 80 mg tablet take 1 tablet (80 mg total) by mouth daily. comm { Jelena only known as: lipitor ask ? azelastine 1 { Yoana 37 mcg (0.1 %) n { Brown asal spray administer 1 spr { Fitzgerald ay into each nostril 2 times a day as commonly known as: astelin needed for rh { Juaquin initis. use in each nostril { Eura  as directed ask ? c { Inara apsaicin 0.1%  { Keirra cream apply 1 app { Lilibeth lication topical { Hollie ly daily fo { Kemper r 90 days. ask tyrone whit { Kinzie e (mrn { Shakera : 0477173 { Tereasa 61) (7/27/1969) printed at 7/26/2024 5:07 am page 4 of 7 epic printed on 10/3/24 7:12 am page 566",
    {
        "entities": [
            [
                26,
                32,
                "PERSON"
            ],
            [
                43,
                51,
                "PERSON"
            ],
            [
                57,
                63,
                "PERSON"
            ],
            [
                85,
                91,
                "PERSON"
            ],
            [
                125,
                132,
                "PERSON"
            ],
            [
                144,
                152,
                "PERSON"
            ],
            [
                221,
                229,
                "PERSON"
            ],
            [
                303,
                309,
                "PERSON"
            ],
            [
                342,
                350,
                "PERSON"
            ],
            [
                407,
                413,
                "PERSON"
            ],
            [
                500,
                505,
                "PERSON"
            ],
            [
                532,
                540,
                "PERSON"
            ],
            [
                584,
                591,
                "PERSON"
            ],
            [
                665,
                674,
                "PERSON"
            ],
            [
                734,
                741,
                "PERSON"
            ],
            [
                772,
                780,
                "PERSON"
            ],
            [
                794,
                801,
                "PERSON"
            ],
            [
                828,
                833,
                "PERSON"
            ],
            [
                837,
                841,
                "PERSON"
            ],
            [
                857,
                864,
                "PERSON"
            ],
            [
                906,
                914,
                "PERSON"
            ],
            [
                969,
                976,
                "PERSON"
            ],
            [
                980,
                987,
                "PERSON"
            ],
            [
                1066,
                1073,
                "PERSON"
            ],
            [
                1089,
                1098,
                "PERSON"
            ],
            [
                1258,
                1263,
                "PERSON"
            ],
            [
                1337,
                1344,
                "PERSON"
            ],
            [
                1394,
                1401,
                "PERSON"
            ],
            [
                1614,
                1620,
                "PERSON"
            ],
            [
                1634,
                1639,
                "PERSON"
            ],
            [
                1643,
                1652,
                "PERSON"
            ],
            [
                1696,
                1701,
                "PERSON"
            ],
            [
                1786,
                1793,
                "PERSON"
            ],
            [
                1858,
                1866,
                "PERSON"
            ],
            [
                1891,
                1897,
                "PERSON"
            ],
            [
                1902,
                1909,
                "PERSON"
            ],
            [
                1927,
                1935,
                "PERSON"
            ],
            [
                1963,
                1971,
                "PERSON"
            ],
            [
                2012,
                2017,
                "PERSON"
            ],
            [
                2026,
                2033,
                "PERSON"
            ],
            [
                2058,
                2063,
                "PERSON"
            ],
            [
                2075,
                2081,
                "PERSON"
            ],
            [
                2140,
                2146,
                "PERSON"
            ],
            [
                2249,
                2255,
                "PERSON"
            ],
            [
                2289,
                2295,
                "PERSON"
            ],
            [
                2314,
                2324,
                "PERSON"
            ],
            [
                2376,
                2384,
                "PERSON"
            ],
            [
                2393,
                2402,
                "PERSON"
            ],
            [
                2411,
                2418,
                "PERSON"
            ],
            [
                2457,
                2465,
                "PERSON"
            ],
            [
                2483,
                2490,
                "PERSON"
            ],
            [
                2495,
                2502,
                "PERSON"
            ],
            [
                2532,
                2539,
                "PERSON"
            ],
            [
                2549,
                2555,
                "PERSON"
            ],
            [
                2572,
                2577,
                "PERSON"
            ],
            [
                2587,
                2593,
                "PERSON"
            ],
            [
                2604,
                2612,
                "PERSON"
            ],
            [
                2645,
                2650,
                "PERSON"
            ],
            [
                2752,
                2758,
                "PERSON"
            ],
            [
                2771,
                2777,
                "PERSON"
            ],
            [
                2820,
                2827,
                "PERSON"
            ],
            [
                2841,
                2845,
                "PERSON"
            ],
            [
                2866,
                2875,
                "PERSON"
            ],
            [
                2961,
                2967,
                "PERSON"
            ],
            [
                2990,
                2996,
                "PERSON"
            ],
            [
                3001,
                3008,
                "PERSON"
            ],
            [
                3053,
                3060,
                "PERSON"
            ],
            [
                3086,
                3090,
                "PERSON"
            ],
            [
                3133,
                3140,
                "PERSON"
            ],
            [
                3146,
                3153,
                "PERSON"
            ],
            [
                3226,
                3236,
                "PERSON"
            ],
            [
                3306,
                3313,
                "PERSON"
            ],
            [
                3418,
                3423,
                "PERSON"
            ],
            [
                3455,
                3462,
                "PERSON"
            ],
            [
                3473,
                3480,
                "PERSON"
            ],
            [
                3510,
                3518,
                "PERSON"
            ],
            [
                3576,
                3585,
                "PERSON"
            ],
            [
                3661,
                3666,
                "PERSON"
            ],
            [
                3717,
                3725,
                "PERSON"
            ],
            [
                3740,
                3745,
                "PERSON"
            ],
            [
                3814,
                3821,
                "PERSON"
            ],
            [
                3853,
                3860,
                "PERSON"
            ],
            [
                3897,
                3904,
                "PERSON"
            ],
            [
                3916,
                3922,
                "PERSON"
            ],
            [
                3954,
                3962,
                "PERSON"
            ],
            [
                3974,
                3981,
                "PERSON"
            ],
            [
                4067,
                4074,
                "PERSON"
            ],
            [
                4118,
                4124,
                "PERSON"
            ],
            [
                4143,
                4149,
                "PERSON"
            ],
            [
                4179,
                4190,
                "PERSON"
            ],
            [
                4271,
                4279,
                "PERSON"
            ],
            [
                4309,
                4314,
                "PERSON"
            ],
            [
                4337,
                4343,
                "PERSON"
            ],
            [
                4360,
                4367,
                "PERSON"
            ],
            [
                4387,
                4396,
                "PERSON"
            ],
            [
                4415,
                4422,
                "PERSON"
            ],
            [
                4436,
                4443,
                "PERSON"
            ],
            [
                4472,
                4479,
                "PERSON"
            ],
            [
                4488,
                4496,
                "PERSON"
            ],
            [
                4508,
                4516,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult medical cent { Parnell er east white, tyrone 121 { Aarna 1 medical center dr mrn { Jerika : 047717361, dob: 7/27 { Kareena /1969, legal sex: m nashville tn 37232 { Michaelene  visit date: 7/9/2024 07/09/20 { Rashelle 24 - offi { Bryer ce visit { Darek  in vanderbilt diabetes and endocrin { Mikell o { Avelina logy (continue { Brighton d) flowsheets (c { Calie ontinued) 7/9/2024 3:02 pm - rl a { Deb t 07/09/24 150 { Dezarae 3 weight change 85.03 kg -rl at since last visit  { Sneha 07/09/24 1503 fluid { Zakai  0 -rl at 07/09/24  { Erykah 1503 re { Lakota suscitation (#3) volume estimates f { Virgil luid 0 -rl at 0 { Aggie 7 { Debroah /09/24 1503 resuscitation (#4) ratio-based meal dosing approx predicted 5.9 -rl at 07/0 { Presleigh 9/24 1503 ratio (500/wt i in kg { Stanley ) rec start sliding 35 { Panagiotis .3 -rl  { Selwyn at 07/09/24 scale (3000 / wt 1503 in kg) basic  { Zuriel i { Sharhonda nfo { Delma rm { Haris ation approx predicted 42.5 -rl at 07/09/24 basal (0.5 * wt in 1503 kg) rec sta { Tammy rt basal 21.3  { Adrien -rl at 07/09/24 (.25 * wt in kg) 1503 rec s { Keelie tart fixed 7.1 -rl at 07/09/24 1503 mea { Christpher l (.08 * wt in kg) fixed m { Jawan eal insuli { Lian n { December  dosin { Abriella g approx predicted 17 { Bonni .6 -r { Voncile l at 07/09/24 correction (150 { Emeri 0 / 1503 wt in kg) encounter v { Mikaylah itals row name 07/09/24  { Rachal 1502 encounter vitals { Jathan  bp { Justis  136/79 -rl at 07/09/24 1503 pulse { Atha  91 -rl at 07/09/24 1503 weight 85 kg (187 lb 8 oz) - rl a { Camillia t 07/09/24 1503 lund-browder  { Lakeesha (adult) row nam { Maryan e 07/09/24 1502 volume estimates fluid 0 -rl at 07/09/ { Vincent 24 1 { Florence 503 re { Pa suscitation (#5) fluid 0 -rl at 07/09/24 1 { Leon 503 resusci { Niccole tation (#6) { Paulene  fluid 0 -rl { Xochilt  at 07/09/24 1503 resuscitation  { Arun (#7) fluid 0 -rl at 07/09/24 1503 resuscitation ( { Reinhold #8) fluid 0 -rl at 0 { Wilfrid 7/09/2 { Ahsley 4 1503 resuscitation printed on 10/3/24 7:12 a { Hindy m page 707,vu { Keilah mc adult medical center east white, tyrone  { Nida 1211 medical center d { Cicero r mrn: 047 { Berlin 717361, dob: 7/27/1969, le { Powell gal sex: m nashville tn 37232 visit date: 7/9/2024 07/09/2024 - office visit in vanderbilt dia { Verlyn betes and endocrinology ( { Briannah continued)  { Kadijah flows { Sakura heets (continued) (#9 { Sheyenne ) fluid 0 -rl  { Torian at 07/09/24 1503 resuscitation (#10) pain question { Leonard s row name 07/09/24 1459 pain assessmen { Teah t is the patient no -rl  { Darryll at 07/09/24 1459 havi { Irish ng pain { Shela  tod { Tona ay? vital signs row name 0 { Constantino 7/09/24 1502 other hr/pulse 91 -rl at 07/09/2 { Lillian 4 1503 user key (r) = recorded by, (t) =  { Ysidro taken by, (c) = cosigned by i { Delainey nitials name prov { Regine ider type discipline { Rowyn  rl lowe, rabecca ann, lpn li { Eliud censed  { Javontae nurse nurse letters letter by gr { Linkin eenspan, debra l, aprn on 7/11/2024 status: sent letter body: 7/9/2024 re: tyrone white dob: 7/27/1969 dear anton jordan de witte, md: it has been my pleasure to serve mr. white. he was last seen and { Nam  treated  { Rommel at our clinic on 7/9/2024. please see the attached information related to t { Edra hat vis { Florencia it. please contact us with any questions or { Korey  con { Taneshia cerns at the number below, via  { Ethyn careeveryw { Suanne h { Abdullahi ere or van { Darlyne derbilt health connect system. sincerely, de { Oaklyn bra l greenspan,  { Raena aprn dept: 615-343-8332 printed o { Vianna n 10/3/24 7:12 am page  { Derreck 708",
    {
        "entities": [
            [
                26,
                34,
                "PERSON"
            ],
            [
                62,
                68,
                "PERSON"
            ],
            [
                94,
                101,
                "PERSON"
            ],
            [
                126,
                134,
                "PERSON"
            ],
            [
                175,
                186,
                "PERSON"
            ],
            [
                219,
                228,
                "PERSON"
            ],
            [
                240,
                246,
                "PERSON"
            ],
            [
                257,
                263,
                "PERSON"
            ],
            [
                302,
                309,
                "PERSON"
            ],
            [
                313,
                321,
                "PERSON"
            ],
            [
                338,
                347,
                "PERSON"
            ],
            [
                366,
                372,
                "PERSON"
            ],
            [
                408,
                412,
                "PERSON"
            ],
            [
                429,
                437,
                "PERSON"
            ],
            [
                489,
                495,
                "PERSON"
            ],
            [
                517,
                523,
                "PERSON"
            ],
            [
                545,
                552,
                "PERSON"
            ],
            [
                562,
                569,
                "PERSON"
            ],
            [
                607,
                614,
                "PERSON"
            ],
            [
                632,
                638,
                "PERSON"
            ],
            [
                642,
                650,
                "PERSON"
            ],
            [
                740,
                750,
                "PERSON"
            ],
            [
                784,
                792,
                "PERSON"
            ],
            [
                817,
                828,
                "PERSON"
            ],
            [
                838,
                845,
                "PERSON"
            ],
            [
                895,
                902,
                "PERSON"
            ],
            [
                906,
                916,
                "PERSON"
            ],
            [
                922,
                928,
                "PERSON"
            ],
            [
                933,
                939,
                "PERSON"
            ],
            [
                1021,
                1027,
                "PERSON"
            ],
            [
                1044,
                1051,
                "PERSON"
            ],
            [
                1097,
                1104,
                "PERSON"
            ],
            [
                1146,
                1157,
                "PERSON"
            ],
            [
                1186,
                1192,
                "PERSON"
            ],
            [
                1205,
                1210,
                "PERSON"
            ],
            [
                1214,
                1223,
                "PERSON"
            ],
            [
                1232,
                1241,
                "PERSON"
            ],
            [
                1265,
                1271,
                "PERSON"
            ],
            [
                1279,
                1287,
                "PERSON"
            ],
            [
                1319,
                1325,
                "PERSON"
            ],
            [
                1358,
                1367,
                "PERSON"
            ],
            [
                1394,
                1401,
                "PERSON"
            ],
            [
                1425,
                1432,
                "PERSON"
            ],
            [
                1438,
                1445,
                "PERSON"
            ],
            [
                1482,
                1487,
                "PERSON"
            ],
            [
                1548,
                1557,
                "PERSON"
            ],
            [
                1589,
                1598,
                "PERSON"
            ],
            [
                1616,
                1623,
                "PERSON"
            ],
            [
                1680,
                1688,
                "PERSON"
            ],
            [
                1695,
                1704,
                "PERSON"
            ],
            [
                1713,
                1716,
                "PERSON"
            ],
            [
                1761,
                1766,
                "PERSON"
            ],
            [
                1780,
                1788,
                "PERSON"
            ],
            [
                1802,
                1810,
                "PERSON"
            ],
            [
                1825,
                1833,
                "PERSON"
            ],
            [
                1868,
                1873,
                "PERSON"
            ],
            [
                1925,
                1934,
                "PERSON"
            ],
            [
                1957,
                1965,
                "PERSON"
            ],
            [
                1974,
                1981,
                "PERSON"
            ],
            [
                2030,
                2036,
                "PERSON"
            ],
            [
                2052,
                2059,
                "PERSON"
            ],
            [
                2105,
                2110,
                "PERSON"
            ],
            [
                2134,
                2141,
                "PERSON"
            ],
            [
                2154,
                2161,
                "PERSON"
            ],
            [
                2190,
                2197,
                "PERSON"
            ],
            [
                2294,
                2301,
                "PERSON"
            ],
            [
                2329,
                2338,
                "PERSON"
            ],
            [
                2352,
                2360,
                "PERSON"
            ],
            [
                2368,
                2375,
                "PERSON"
            ],
            [
                2399,
                2408,
                "PERSON"
            ],
            [
                2425,
                2432,
                "PERSON"
            ],
            [
                2485,
                2493,
                "PERSON"
            ],
            [
                2535,
                2540,
                "PERSON"
            ],
            [
                2567,
                2575,
                "PERSON"
            ],
            [
                2599,
                2605,
                "PERSON"
            ],
            [
                2615,
                2621,
                "PERSON"
            ],
            [
                2628,
                2633,
                "PERSON"
            ],
            [
                2662,
                2674,
                "PERSON"
            ],
            [
                2722,
                2730,
                "PERSON"
            ],
            [
                2774,
                2781,
                "PERSON"
            ],
            [
                2813,
                2822,
                "PERSON"
            ],
            [
                2842,
                2849,
                "PERSON"
            ],
            [
                2872,
                2878,
                "PERSON"
            ],
            [
                2910,
                2916,
                "PERSON"
            ],
            [
                2926,
                2935,
                "PERSON"
            ],
            [
                2970,
                2977,
                "PERSON"
            ],
            [
                3179,
                3183,
                "PERSON"
            ],
            [
                3195,
                3202,
                "PERSON"
            ],
            [
                3280,
                3285,
                "PERSON"
            ],
            [
                3295,
                3305,
                "PERSON"
            ],
            [
                3351,
                3357,
                "PERSON"
            ],
            [
                3364,
                3373,
                "PERSON"
            ],
            [
                3407,
                3413,
                "PERSON"
            ],
            [
                3426,
                3433,
                "PERSON"
            ],
            [
                3437,
                3447,
                "PERSON"
            ],
            [
                3460,
                3468,
                "PERSON"
            ],
            [
                3515,
                3522,
                "PERSON"
            ],
            [
                3542,
                3548,
                "PERSON"
            ],
            [
                3584,
                3591,
                "PERSON"
            ],
            [
                3617,
                3625,
                "PERSON"
            ]
        ]
    }
),(
    "vumc hendersonvil { Karol le { Graci  - anderson white, tyrone 128 n anderson ln mrn: 0 { Kaiden 47717361, d { Coco ob: 7/27/1969, legal sex: m hend { Laiken ersonvill { Sanaya e tn 37075 vi { Chalmer sit date: 6/7/2023 06/07/2023 - communication in vand { Clearence e { Kee rbilt p { Ky rimary care hendersonville (continued) c { Levin linical notes (continued) dme order for cane was faxed electronically signed by ferguson, sh { Mamadou erri l, l { Carlena pn at 6/7/2023 4:11 pm other { Eleana  orders  { Michaelle general supply generic d { Ranee me (active) electro { Sharika nically sig { Diane ned by: l { Marvella ippard, giles a, aprn on 06/07/23 1453 status: active ordering user: lippard, giles a, aprn  { Delroy 06/07/23 1453 ordering provider: lippard, giles a { Kobie , aprn { Carlita  authorized by: lippard, giles a, aprn or { Kandra dering mode: standard frequency: { Kenadi  routine 0 { Kerin 6/07/23 - c { Arlon lass: clinic performed  { Bryar quantity: 1 released by: lippard, giles a, aprn 06/07/23 1453 diagnoses abnormal { Dijon  gait [r26.9] questionnair { Betzaida e question answer the face to { Zaniah  face  { Yale evaluation was per { Ameen formed on 6/7/2023 additional product info/specifications cane indications abnor { Bruce mal gait [r26.9 ( { Karima ic { Jabril d-10-c { Jamario m)]  { Alyshia printed on 10/3/24 7:13 am pag { Keiana e 2093,vumc hendersonville - ande { Jule rson white, tyrone 128 n anderson ln mrn: 0 { Keneth 4771 { Jaymi 7361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 vi { Horatio sit date: { Shivam  6/7/2023 06/07/2023 - refill in vanderbilt primary care he { Annita ndersonvill { Delsie e faceshee { Keylee t re { Naisha port patient demogr { Zariya aphics patient name mrn legal dob address phone white, tyrone 04771 { Keane 73 sex 7/27/1969 apt 705 615-260-2291 (home { Takoda ) 61 m 1101 edgehill ave 615-260-2291 (mobile) nashville tn 37203 *preferred* hospital account not on file admission information current information attending provider admitting provid { Erie e { Raelee r admission typ { Denise e admission s { Rhyan tatus unknown  { Ulyses status admission date/time discharge da { Cyndy te/time hospital servi { Deserae ce auth/cert status { Keandra  hospital area unit room/bed referring provider 06/07/2023 - refill in vanderbilt primary { Naia  care hendersonville { Jahiem  (continued) rea { Kori son for visit chief com { Pamela plaints [last edi { Ameena ted by nea { Finnley l, valerie d on 6/7/2023 1232] med refill pt is at the pharmacy, onset { Jalaya  date 6/7/ { Kamaya 2023 visit inform { Parthenia ation nursing asse { Wesly ssment  asse { Wesson ssment available fo { Arianny r this encounter. { Madalene  communica { Shifra tion tracking calls/messages interface (incomi { Herminio ng)  { Nery on 6/7/2023 1133 caller name: vanderbilt university pho { Niah ne number: 615- { Jamichael 322-2688 100  { Gypsy oaks - nashville, tn { Corby  - 719 thompson ln phone ( { Niel incoming) on 6/7/2023 1139 caller name: { Rasheen  vanderbilt u { Deziree niversity phone numb { Kendyll er: 615-322-2688 100 oaks - nashville, tn - 7 { Nidhi 19 thompson ln medication list medica { Kiyoshi tion list i this report is for documentation purposes only. th { Jazzlynn e patient shoul { Jameer d not follow m { Laya edication instructions within. for accurate instructions { Ashtyn  regard { Graig ing med { Sinclair ications, the p { Alaine atient shou { Crystle ld instead consult their p { Cutter hysic { Damarius ian or after visit su { Davontae mmary. { Spurgeon  a { Marlaina ctive at the en { Jamarr d of visit print { Severo ed on 10/3/24 7:13 am page  { Assunta 2094",
    {
        "entities": [
            [
                20,
                26,
                "PERSON"
            ],
            [
                31,
                37,
                "PERSON"
            ],
            [
                90,
                97,
                "PERSON"
            ],
            [
                111,
                116,
                "PERSON"
            ],
            [
                151,
                158,
                "PERSON"
            ],
            [
                170,
                177,
                "PERSON"
            ],
            [
                193,
                201,
                "PERSON"
            ],
            [
                257,
                267,
                "PERSON"
            ],
            [
                271,
                275,
                "PERSON"
            ],
            [
                285,
                288,
                "PERSON"
            ],
            [
                331,
                337,
                "PERSON"
            ],
            [
                432,
                440,
                "PERSON"
            ],
            [
                452,
                460,
                "PERSON"
            ],
            [
                491,
                498,
                "PERSON"
            ],
            [
                509,
                519,
                "PERSON"
            ],
            [
                546,
                552,
                "PERSON"
            ],
            [
                574,
                582,
                "PERSON"
            ],
            [
                596,
                602,
                "PERSON"
            ],
            [
                614,
                623,
                "PERSON"
            ],
            [
                718,
                725,
                "PERSON"
            ],
            [
                777,
                783,
                "PERSON"
            ],
            [
                792,
                800,
                "PERSON"
            ],
            [
                844,
                851,
                "PERSON"
            ],
            [
                886,
                893,
                "PERSON"
            ],
            [
                906,
                912,
                "PERSON"
            ],
            [
                926,
                932,
                "PERSON"
            ],
            [
                958,
                964,
                "PERSON"
            ],
            [
                1047,
                1053,
                "PERSON"
            ],
            [
                1082,
                1091,
                "PERSON"
            ],
            [
                1123,
                1130,
                "PERSON"
            ],
            [
                1139,
                1144,
                "PERSON"
            ],
            [
                1165,
                1171,
                "PERSON"
            ],
            [
                1254,
                1260,
                "PERSON"
            ],
            [
                1280,
                1287,
                "PERSON"
            ],
            [
                1292,
                1299,
                "PERSON"
            ],
            [
                1308,
                1316,
                "PERSON"
            ],
            [
                1323,
                1331,
                "PERSON"
            ],
            [
                1364,
                1371,
                "PERSON"
            ],
            [
                1407,
                1412,
                "PERSON"
            ],
            [
                1458,
                1465,
                "PERSON"
            ],
            [
                1472,
                1478,
                "PERSON"
            ],
            [
                1542,
                1550,
                "PERSON"
            ],
            [
                1562,
                1569,
                "PERSON"
            ],
            [
                1631,
                1638,
                "PERSON"
            ],
            [
                1652,
                1659,
                "PERSON"
            ],
            [
                1672,
                1679,
                "PERSON"
            ],
            [
                1686,
                1693,
                "PERSON"
            ],
            [
                1715,
                1722,
                "PERSON"
            ],
            [
                1792,
                1798,
                "PERSON"
            ],
            [
                1844,
                1851,
                "PERSON"
            ],
            [
                2038,
                2043,
                "PERSON"
            ],
            [
                2047,
                2054,
                "PERSON"
            ],
            [
                2072,
                2079,
                "PERSON"
            ],
            [
                2095,
                2101,
                "PERSON"
            ],
            [
                2118,
                2125,
                "PERSON"
            ],
            [
                2167,
                2173,
                "PERSON"
            ],
            [
                2198,
                2206,
                "PERSON"
            ],
            [
                2228,
                2236,
                "PERSON"
            ],
            [
                2328,
                2333,
                "PERSON"
            ],
            [
                2356,
                2363,
                "PERSON"
            ],
            [
                2382,
                2387,
                "PERSON"
            ],
            [
                2413,
                2420,
                "PERSON"
            ],
            [
                2440,
                2447,
                "PERSON"
            ],
            [
                2460,
                2468,
                "PERSON"
            ],
            [
                2541,
                2548,
                "PERSON"
            ],
            [
                2561,
                2568,
                "PERSON"
            ],
            [
                2588,
                2598,
                "PERSON"
            ],
            [
                2619,
                2625,
                "PERSON"
            ],
            [
                2640,
                2647,
                "PERSON"
            ],
            [
                2669,
                2677,
                "PERSON"
            ],
            [
                2697,
                2706,
                "PERSON"
            ],
            [
                2719,
                2726,
                "PERSON"
            ],
            [
                2775,
                2784,
                "PERSON"
            ],
            [
                2791,
                2796,
                "PERSON"
            ],
            [
                2854,
                2859,
                "PERSON"
            ],
            [
                2877,
                2887,
                "PERSON"
            ],
            [
                2903,
                2909,
                "PERSON"
            ],
            [
                2932,
                2938,
                "PERSON"
            ],
            [
                2967,
                2972,
                "PERSON"
            ],
            [
                3014,
                3022,
                "PERSON"
            ],
            [
                3038,
                3046,
                "PERSON"
            ],
            [
                3069,
                3077,
                "PERSON"
            ],
            [
                3125,
                3131,
                "PERSON"
            ],
            [
                3171,
                3179,
                "PERSON"
            ],
            [
                3244,
                3253,
                "PERSON"
            ],
            [
                3271,
                3278,
                "PERSON"
            ],
            [
                3295,
                3300,
                "PERSON"
            ],
            [
                3359,
                3366,
                "PERSON"
            ],
            [
                3376,
                3382,
                "PERSON"
            ],
            [
                3392,
                3401,
                "PERSON"
            ],
            [
                3419,
                3426,
                "PERSON"
            ],
            [
                3440,
                3448,
                "PERSON"
            ],
            [
                3477,
                3484,
                "PERSON"
            ],
            [
                3492,
                3501,
                "PERSON"
            ],
            [
                3525,
                3534,
                "PERSON"
            ],
            [
                3543,
                3552,
                "PERSON"
            ],
            [
                3557,
                3566,
                "PERSON"
            ],
            [
                3584,
                3591,
                "PERSON"
            ],
            [
                3610,
                3617,
                "PERSON"
            ],
            [
                3647,
                3655,
                "PERSON"
            ]
        ]
    }
),(
    "vumc hendersonville - anderson white, tyrone 128 n ande { Jackqueline rson ln m { Naiya rn: 0477173 { Niels 61, dob: 7/27/1969, legal sex: m hendersonville tn 37075 visit date: 8/7/2023 08/07/2023 - medication management in vanderbilt primary care hendersonville (continued) medication list (continued) discontinued by: de witte, anton jordan, md discontinued on: 10/17/2023 re { Elta ason for discontinuation: cleanup(notavs) { Phillip  instructions: administer 1 spr { Abbas ay into each nostril 2 times a day. use in each nostri { Aris l as directed authoriz { Emelda ed by: lippard, giles a, aprn  { Jaide ordered on: 8/8/2023 start date: 8/8/2023 quantity: 30 ml refill: 12 refills by 8/7/2024 cetirizin { Jalia e 10 mg tablet { Reema  (zyrtec) discontinued by: mickey, lisa, lpn discontinued on:  { Anabell 10/4/2023 reason for discontinuation: reorder instructions: take 1 { Ilah  tablet (10 mg total) { Murphy  by mouth once a day as needed for allergies.  { Nadeen authorized by: lippar { Otha d, giles a, aprn ordered on: 8/8/2023 start date: 8/8/ { Rozanne 2023 q { Victoriano uantity: 30 tablet refill: 11 refills by 8/7/2024 fluticasone propionate 50 mcg/actuation nasal spray,suspension (flonase) discontinued by: greenspan, debra l, aprn discontinued on: 9/5/2023 reason for discontinuation: other (cancelrx) instructions: administer 2 sprays into  { Arrianna each nostril daily. authorized by: lippard, giles a, ap { Arvella rn order { Bettyann ed on: 8/8/2023 start date: 8/8/2023 quantity: 16 g refill: 11 refills by 8/7/2024 montelu { Sayra kast 10 mg tablet (singulair) instructions: take 1 tablet (10 mg total)  { Ysabel by mouth every eveni { Zayra ng. authorized by: lippa { Zaylen rd, giles a, aprn ordered on:  { Caliyah 8/8/2023 start date: 8/8/2023 quantity: 90 tablet refill: 3 refills by 8/7/2024 nifedipine er 30 mg tablet,extended release (adalat cc) discontinued by: de witte, anton jordan, md discontinued on { Lilie : 4/23/2024 reason for discontinuation: reorder instructions:  { Malory take 1 { Remona  tablet (30 mg total) by mouth daily. authorized by: { Victory  lippard, giles a, aprn ordered on: 8/8/2023 { Azalee  start date: 8 { Suzi /8/2023 quantity: 90 tablet refill: 3 refills by 8/7/2024 stopped in visit medi { Tesa cations last reviewed by lippard, giles a, a { Tynesha prn on 6/9/202 { Sachin 3 113 { Ermelinda 0 dula { Gwynne glutide 1.5 mg/0.5 ml subcutaneous pen injector (trulicity) discontinued by: wils { Makenzy on { Necole , danya horc { Riva hi, pharmd discontinued on: 8/7/2023 reason for disc { Jermel ontinuat { Avia ion: dose { Sharolyn  adjustment (cancel { Adolf rx) clinical notes progress notes wils { Rosco on { Amna , danya { Atiya  horchi, pharm { Tobie d at 8/7/2023 1359 author { Trinitee : wilson, danya horch { Dyson i, pharmd service: a { Allisa utho { Amaryllis r type: pharmacist filed: { Daniya  8/8/2023 3:03 pm encounter date: 8/7/2023 status: signed editor: wilson, danya  { Laela horchi, pharmd (pharmacist) giles: pt called st { Alfonse ating he needs refills on nearly all his meds as he has been { Braxten  out for a few weeks. he is living at  { Callahan a new address { Thayne  in nashville so i updated this i { Analeigh n chart. he reports he does not h { Michayla ave { Teryn  a { Deward  car anym { Cleda ore so he cannot get to pharmacy easily. he has been off trulicity sev { Cozette eral weeks due to not being able to get it so i am sending in the 0.75mg dos { Sinead e to prevent side effects. prin { Diondre ted on 10/3/24 7:13 am page 2001,vumc hendersonville - anderson white, tyrone 128 n { Daysi  and { Carsyn erson ln mrn: 047717361, dob: 7/27/1969, legal sex: m hendersonville tn 37075 visit date: 8/7/2023 08/07/2023 - medication management in vanderbilt primary care hendersonville (continued) clinical notes (conti { Edgard nued) clinical staff: can t { Kyrin he following meds please be sent to vip so they { Bettylou  can be shipped to pt? albuterol atorvastatin azelastine cet { Laynie irizine flonase gabape { Carry ntin montelukast nifedipine thanks! danya h. wilson { Genevia , pharmd  { Wende populat { Aziz ion health clinical pharmacist 615-322-4663 electronically signed by  { Quan wilson, danya  { Twanda horchi, pharmd at 8/8/2023 3:03  { Ann pm lippard, giles a, aprn at 8/7/2023 1359 auth { Yoshio or: lipp { Mena ard, giles a, aprn service: - author type: nurse practitioner filed: 8/8/2023 3:03  { Makaylee pm encounter da { Meena te: 8/7/2023 status: signed editor: lippard, giles a, aprn (nurse practitioner) hi danya, i refilled everything ex { Monna c { Inger ept his gabapentin. i increased his do { Tamarah sage august 1 and sent in new prescription f { Aleksandr or that then. electronically signed by lippard, giles a, aprn at 8/8/202 { Benaiah 3 3:03 pm wilson, danya { Bliss  horchi, pharmd at 8/ { Marquisha 7/2023 1359 author: wilson, danya horchi, pharmd service: - author type: pha { Neve rmacist { Tiona  filed: 8/8/2023 3:17 p { Ioannis m encounter date: 8/7/2023  { Zakery status: signed ed { Bellamy itor: wilson, danya horchi, pharmd (pharmacist) retail support ph { Chere armacy (rsp), spoke with: { Lailani  pati { Mariko ent refills requested (please provide both medication name and strength): albuterol 9mcg atorvastatin 80mg azelastine 137mcg cetirizine 10mg flonase 50mcg gabapentin 300mg (at wag 615-327-1894, can this be tra { Nariyah nsferred) montelukast 10mg nifedipine er 30mg trulicity 0.75mg printed on 10/3/24 7:13 am page { Jerone  2002",
    {
        "entities": [
            [
                58,
                70,
                "PERSON"
            ],
            [
                82,
                88,
                "PERSON"
            ],
            [
                102,
                108,
                "PERSON"
            ],
            [
                380,
                385,
                "PERSON"
            ],
            [
                429,
                437,
                "PERSON"
            ],
            [
                471,
                477,
                "PERSON"
            ],
            [
                534,
                539,
                "PERSON"
            ],
            [
                564,
                571,
                "PERSON"
            ],
            [
                604,
                610,
                "PERSON"
            ],
            [
                711,
                717,
                "PERSON"
            ],
            [
                734,
                740,
                "PERSON"
            ],
            [
                805,
                813,
                "PERSON"
            ],
            [
                882,
                887,
                "PERSON"
            ],
            [
                911,
                918,
                "PERSON"
            ],
            [
                967,
                974,
                "PERSON"
            ],
            [
                998,
                1003,
                "PERSON"
            ],
            [
                1060,
                1068,
                "PERSON"
            ],
            [
                1077,
                1088,
                "PERSON"
            ],
            [
                1366,
                1375,
                "PERSON"
            ],
            [
                1433,
                1441,
                "PERSON"
            ],
            [
                1452,
                1461,
                "PERSON"
            ],
            [
                1554,
                1560,
                "PERSON"
            ],
            [
                1635,
                1642,
                "PERSON"
            ],
            [
                1665,
                1671,
                "PERSON"
            ],
            [
                1698,
                1705,
                "PERSON"
            ],
            [
                1738,
                1746,
                "PERSON"
            ],
            [
                1944,
                1950,
                "PERSON"
            ],
            [
                2015,
                2022,
                "PERSON"
            ],
            [
                2031,
                2038,
                "PERSON"
            ],
            [
                2093,
                2101,
                "PERSON"
            ],
            [
                2148,
                2155,
                "PERSON"
            ],
            [
                2172,
                2177,
                "PERSON"
            ],
            [
                2259,
                2264,
                "PERSON"
            ],
            [
                2311,
                2319,
                "PERSON"
            ],
            [
                2336,
                2343,
                "PERSON"
            ],
            [
                2351,
                2361,
                "PERSON"
            ],
            [
                2370,
                2377,
                "PERSON"
            ],
            [
                2461,
                2469,
                "PERSON"
            ],
            [
                2474,
                2481,
                "PERSON"
            ],
            [
                2496,
                2501,
                "PERSON"
            ],
            [
                2556,
                2563,
                "PERSON"
            ],
            [
                2574,
                2579,
                "PERSON"
            ],
            [
                2591,
                2600,
                "PERSON"
            ],
            [
                2622,
                2628,
                "PERSON"
            ],
            [
                2669,
                2675,
                "PERSON"
            ],
            [
                2680,
                2685,
                "PERSON"
            ],
            [
                2695,
                2701,
                "PERSON"
            ],
            [
                2718,
                2724,
                "PERSON"
            ],
            [
                2752,
                2761,
                "PERSON"
            ],
            [
                2785,
                2791,
                "PERSON"
            ],
            [
                2814,
                2821,
                "PERSON"
            ],
            [
                2828,
                2838,
                "PERSON"
            ],
            [
                2866,
                2873,
                "PERSON"
            ],
            [
                2956,
                2962,
                "PERSON"
            ],
            [
                3012,
                3020,
                "PERSON"
            ],
            [
                3083,
                3091,
                "PERSON"
            ],
            [
                3132,
                3141,
                "PERSON"
            ],
            [
                3157,
                3164,
                "PERSON"
            ],
            [
                3200,
                3209,
                "PERSON"
            ],
            [
                3245,
                3254,
                "PERSON"
            ],
            [
                3260,
                3266,
                "PERSON"
            ],
            [
                3271,
                3278,
                "PERSON"
            ],
            [
                3290,
                3296,
                "PERSON"
            ],
            [
                3369,
                3377,
                "PERSON"
            ],
            [
                3456,
                3463,
                "PERSON"
            ],
            [
                3497,
                3505,
                "PERSON"
            ],
            [
                3591,
                3597,
                "PERSON"
            ],
            [
                3604,
                3611,
                "PERSON"
            ],
            [
                3823,
                3830,
                "PERSON"
            ],
            [
                3860,
                3866,
                "PERSON"
            ],
            [
                3916,
                3925,
                "PERSON"
            ],
            [
                3988,
                3995,
                "PERSON"
            ],
            [
                4020,
                4026,
                "PERSON"
            ],
            [
                4080,
                4088,
                "PERSON"
            ],
            [
                4100,
                4106,
                "PERSON"
            ],
            [
                4116,
                4121,
                "PERSON"
            ],
            [
                4193,
                4198,
                "PERSON"
            ],
            [
                4215,
                4222,
                "PERSON"
            ],
            [
                4257,
                4261,
                "PERSON"
            ],
            [
                4311,
                4318,
                "PERSON"
            ],
            [
                4329,
                4334,
                "PERSON"
            ],
            [
                4420,
                4429,
                "PERSON"
            ],
            [
                4447,
                4453,
                "PERSON"
            ],
            [
                4570,
                4576,
                "PERSON"
            ],
            [
                4580,
                4586,
                "PERSON"
            ],
            [
                4627,
                4635,
                "PERSON"
            ],
            [
                4682,
                4692,
                "PERSON"
            ],
            [
                4767,
                4775,
                "PERSON"
            ],
            [
                4801,
                4807,
                "PERSON"
            ],
            [
                4831,
                4841,
                "PERSON"
            ],
            [
                4920,
                4925,
                "PERSON"
            ],
            [
                4935,
                4941,
                "PERSON"
            ],
            [
                4967,
                4975,
                "PERSON"
            ],
            [
                5005,
                5012,
                "PERSON"
            ],
            [
                5032,
                5040,
                "PERSON"
            ],
            [
                5108,
                5114,
                "PERSON"
            ],
            [
                5142,
                5150,
                "PERSON"
            ],
            [
                5158,
                5165,
                "PERSON"
            ],
            [
                5377,
                5385,
                "PERSON"
            ],
            [
                5482,
                5489,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adu { Cera lt hospital whi { Floyd te,  { Marolyn tyrone 1211 medical center dr. mrn: { Rosena  047717361, do { Shantay b: 7/27/1969, legal sex: m nashville tn 37232-000 { Gay 4 adm: 7/26/2024, d/c: 7/26/2024 07/26/2024 - ed in vanderbilt emergency department (conti { Jamaine nued) letters (continued) pauw, emily kathryn, md pri { Klaire nted on 10/3/24 7:12 am page 581,vumc adult hospital white, tyro { Lorean ne 1211 medical center dr. mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 372 { Nadean 32-0004 adm: 7 { Jadarius /26/2024, d/c: 7/26/2024 07/26/2024 - ed in vanderbilt emergency  { Johnathen de { Kamiah partment (continued) lmr encounter level scans { Antuan  consent for routine treatment (hospi { Kamar tal) - electronic signature on 7/26/2024 4:14 am (effecti { Perrin ve from 7 { Xavian /26/2024) - e-signe { Keondre d vanderbilt university { Artemis  medical ce { Jeslyn nter consent for healthcare services vanderbilt university medical center (vumc) is an academic medical ce { Margrett nter. o { Samiah ur mission is  { Yamil to: provid { Jahzara e high-quality care train health care { Kaeleigh  professionals  { Kathia perform  { Valinda research to improve { Maddex  h { Paige ealt { Race h care for y { Ajah ou and  { Roswell other patient { Holt s. i understand that during my care, v { Irene umc  { Aeris may:  { Carra treat me as my doctors a { Tijuana nd hea { Tytiana lth care team direct (in { Camille clu { Abbe ding giving medications, drawing blood { Daniyah  and performing other procedures) sedate me mildly with drugs, if needed for medical reason { Abdulaziz s test my b { Derrion lood for infections if someone is exposed to my blood or body fluids collec { Marcanthony t, dispose of, or use information, fluid { Romell , or tissue taken { Freyja  from me during tes { Jackelin ts, treatm { Jacquelene ent, or surg { Keirsten ery to  { Makia better un { Saydee dersta { Shireen nd my health and to { Aleksandar  improve he { Lenin alth care for others test my gene { Ardelia s or dna, if needed, to help wit { Kristeen h my med { Thorin ical issues  { Akeelah and find the { Sheronda  best ways to { Carmel  tre { Verdell at them take photos or { Italy  videos { Ferman  of me, if needed, to { Laken  treat or identify  { Marian me use treatments administered to me as an opportun { Gerrie ity to study and learn how to improve the care for me an { Seema d others contact me about research stu { Errick dies for which i might qualify u { Tou n { Adel l { Loralei ess i contact (615) 322-7343 to o { Rivers pt-out. i am awa { Akshara re that i have the ri { Kally ght to: refuse tests or treatment (as far as the l { Lester a { Marvis w allows) and to be told what might happen if i do ask { Donisha   { Mei for a copy of \"your rights  { Tiffini and { Tyshon  responsibilities as a patient,\" available in all vanderbilt { Avie  facilities, in handbooks for inpatients, and onl { Jesika ine at vanderbilthealth.com i u { Kit nderstand that  are ma { Laurinda de about the results of my treatment. vumc may  { Leighanne need { Reita  to share health information and leftover body fluids or tissue with o { Sada rganizati { Sharae ons other than vumc. vum { Tammera c must follow applicable privacy { Buelah  laws  { Chyenne when doing so, as described in  { Thania vumc's  { Abdallah notic { Ashtin e of privacy practices. i will not { Coltyn  be paid for any { Iverson  discov { Zephyr eries  { Twanna or inventi { Yaquelin ons that might result from the use of my health information and leftover fluid or tissue. prin { Syrus ted on 10/3/24 7:12 am page 582",
    {
        "entities": [
            [
                11,
                16,
                "PERSON"
            ],
            [
                34,
                40,
                "PERSON"
            ],
            [
                47,
                55,
                "PERSON"
            ],
            [
                93,
                100,
                "PERSON"
            ],
            [
                117,
                125,
                "PERSON"
            ],
            [
                177,
                181,
                "PERSON"
            ],
            [
                274,
                282,
                "PERSON"
            ],
            [
                338,
                345,
                "PERSON"
            ],
            [
                412,
                419,
                "PERSON"
            ],
            [
                510,
                517,
                "PERSON"
            ],
            [
                534,
                543,
                "PERSON"
            ],
            [
                611,
                621,
                "PERSON"
            ],
            [
                626,
                633,
                "PERSON"
            ],
            [
                682,
                689,
                "PERSON"
            ],
            [
                729,
                735,
                "PERSON"
            ],
            [
                795,
                802,
                "PERSON"
            ],
            [
                814,
                821,
                "PERSON"
            ],
            [
                843,
                851,
                "PERSON"
            ],
            [
                877,
                885,
                "PERSON"
            ],
            [
                899,
                906,
                "PERSON"
            ],
            [
                1015,
                1024,
                "PERSON"
            ],
            [
                1034,
                1041,
                "PERSON"
            ],
            [
                1058,
                1064,
                "PERSON"
            ],
            [
                1077,
                1085,
                "PERSON"
            ],
            [
                1125,
                1134,
                "PERSON"
            ],
            [
                1152,
                1159,
                "PERSON"
            ],
            [
                1170,
                1178,
                "PERSON"
            ],
            [
                1200,
                1207,
                "PERSON"
            ],
            [
                1212,
                1218,
                "PERSON"
            ],
            [
                1225,
                1230,
                "PERSON"
            ],
            [
                1245,
                1250,
                "PERSON"
            ],
            [
                1260,
                1268,
                "PERSON"
            ],
            [
                1284,
                1289,
                "PERSON"
            ],
            [
                1330,
                1336,
                "PERSON"
            ],
            [
                1343,
                1349,
                "PERSON"
            ],
            [
                1357,
                1363,
                "PERSON"
            ],
            [
                1390,
                1398,
                "PERSON"
            ],
            [
                1407,
                1415,
                "PERSON"
            ],
            [
                1442,
                1450,
                "PERSON"
            ],
            [
                1456,
                1461,
                "PERSON"
            ],
            [
                1502,
                1510,
                "PERSON"
            ],
            [
                1604,
                1614,
                "PERSON"
            ],
            [
                1628,
                1636,
                "PERSON"
            ],
            [
                1714,
                1726,
                "PERSON"
            ],
            [
                1769,
                1776,
                "PERSON"
            ],
            [
                1796,
                1803,
                "PERSON"
            ],
            [
                1825,
                1834,
                "PERSON"
            ],
            [
                1847,
                1858,
                "PERSON"
            ],
            [
                1873,
                1882,
                "PERSON"
            ],
            [
                1892,
                1898,
                "PERSON"
            ],
            [
                1910,
                1917,
                "PERSON"
            ],
            [
                1926,
                1934,
                "PERSON"
            ],
            [
                1956,
                1967,
                "PERSON"
            ],
            [
                1981,
                1987,
                "PERSON"
            ],
            [
                2023,
                2031,
                "PERSON"
            ],
            [
                2066,
                2075,
                "PERSON"
            ],
            [
                2086,
                2093,
                "PERSON"
            ],
            [
                2108,
                2116,
                "PERSON"
            ],
            [
                2131,
                2140,
                "PERSON"
            ],
            [
                2156,
                2163,
                "PERSON"
            ],
            [
                2170,
                2178,
                "PERSON"
            ],
            [
                2203,
                2209,
                "PERSON"
            ],
            [
                2219,
                2226,
                "PERSON"
            ],
            [
                2250,
                2256,
                "PERSON"
            ],
            [
                2278,
                2285,
                "PERSON"
            ],
            [
                2339,
                2346,
                "PERSON"
            ],
            [
                2405,
                2411,
                "PERSON"
            ],
            [
                2452,
                2459,
                "PERSON"
            ],
            [
                2494,
                2498,
                "PERSON"
            ],
            [
                2502,
                2507,
                "PERSON"
            ],
            [
                2511,
                2519,
                "PERSON"
            ],
            [
                2555,
                2562,
                "PERSON"
            ],
            [
                2581,
                2589,
                "PERSON"
            ],
            [
                2613,
                2619,
                "PERSON"
            ],
            [
                2672,
                2679,
                "PERSON"
            ],
            [
                2683,
                2690,
                "PERSON"
            ],
            [
                2747,
                2755,
                "PERSON"
            ],
            [
                2759,
                2763,
                "PERSON"
            ],
            [
                2793,
                2801,
                "PERSON"
            ],
            [
                2807,
                2814,
                "PERSON"
            ],
            [
                2877,
                2882,
                "PERSON"
            ],
            [
                2934,
                2941,
                "PERSON"
            ],
            [
                2975,
                2979,
                "PERSON"
            ],
            [
                3004,
                3013,
                "PERSON"
            ],
            [
                3063,
                3073,
                "PERSON"
            ],
            [
                3080,
                3086,
                "PERSON"
            ],
            [
                3159,
                3164,
                "PERSON"
            ],
            [
                3176,
                3183,
                "PERSON"
            ],
            [
                3210,
                3218,
                "PERSON"
            ],
            [
                3253,
                3260,
                "PERSON"
            ],
            [
                3269,
                3277,
                "PERSON"
            ],
            [
                3311,
                3318,
                "PERSON"
            ],
            [
                3328,
                3337,
                "PERSON"
            ],
            [
                3345,
                3352,
                "PERSON"
            ],
            [
                3389,
                3396,
                "PERSON"
            ],
            [
                3415,
                3423,
                "PERSON"
            ],
            [
                3433,
                3440,
                "PERSON"
            ],
            [
                3449,
                3456,
                "PERSON"
            ],
            [
                3469,
                3478,
                "PERSON"
            ],
            [
                3575,
                3581,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult  { Christinia medical center east white, tyro { Joslin ne 1211 medical center dr mrn: 047717361, dob: 7/27 { Leanora /1969, legal sex: m nashville tn 37232 visit date: 2/14/2023 02/14/20 { Naudia 23 - off { Edric ice visit in v { Einar anderbilt diabetes and end { Teegan ocrinology (continued) medication list (continued) discontinued by: lehmann, melissa cary, pa-c discontinued on: { Diana  6/3/2023 reason for discontinu { Mychael ation: stop taking at discharge (cancelrx) instructions: take 1 cap { Pavel sule (50,000 units total) by mouth weekly. authorized by: wang, zhijian, aprn ordered on: 2/7/2023 start date: 2/7/2023 { Breyanna  end date: 6/3/2023 q { Deven uantity: 12 capsule refill:  remaining gabapentin 300 m { Genny g capsule (neurontin) discontinued by: darks, tina m, aprn discontinued on: 3/13/2023 instruction { Maralyn s: take { Markeisha  one capsule by mouth three times a day authorized by: lippard, giles a, aprn ordered  { Zuleyka on { Jun : 2/9/2023 start  { Westyn date: 2/9/2023 quantity: 90 capsule refill:  remaining stopped in vi { Zen sit none { Aiesha  clinical notes ass { Lucianna essment  { Magdaline & plan note greenspan, debra l, aprn at 2/15/2023 1427 author: greenspan, debra l, aprn se { Nancee rvice: - author type: nurse practitioner filed: 2/15/2023 2:31 pm encounte { Nonie r date: 2/14/2023 status: written editor: greenspan, debra l, aprn (nurse practitioner) related problem: type  { Avarie 2 diabetes mellitus with chronic kidney disease, with long-term current use of insulin (cms/hcc) la { Denyse betes poorly controlled for a while. we spent a great { Hartley  deal of time discussing the importance of better control in order to slow { Lilyanne  down th { Majorie e decline in his ren { Tyrik al function. he states he will not go on dialysis. he also declines use o { Fredricka f insulin. we discussed giving up his pepsi an { Jennell d that this would go a long way to improving his glucose control. we discussed alternate beverages  { Kina and use of artificial sweetners as alte { Kynslee rnatives to his pepsi. we discussed the importance of nutrition to ma { Jasmine intaining his muscle which i believe has declined due to p { Jd oor dietary patterns and worsening renal failure with pr { Alyza otein lo { Baily ss. daughter has numero { Glen us questions about diet. we need a renal dietician to hel { Lillith p make recommendations. i suggested in the meantime, she can use plant protein su { Shakita pplements but would avoid milk based p { Zabrina rotein such as { Jakoby  whey protein to supplement his diet. we  { Icie di { Osie scussed the importance of at least 2 meals per day. he is willing to try { Toniann  this though it is apparent his d { Bear aughter will be doing most of the meal pl { Caspian anning, electronically signed by greenspan, debra l, aprn at 2 { Macklin /15/2023 2:31 pm greenspan, debra l, aprn at 2/15/2023 1432 author: greenspan, debra l, aprn { Naquan  service: author type: n { Alexzandra urse practiti { Bibiana oner filed: 2/15/2023 2:33 pm encounter date: 2/14/2023 st { Henriette atus: written editor: greenspan, debra l, aprn (nurse practitioner) related problem: vitamin d deficiency rec { Jessyca ommended adding otc vitamin d { Kalisha 3 2000 units per day. renal can decide if he should be on d2 instead. electronically signed by greenspan, debra l, aprn at 2/15/2023 2:33 pm greenspan, debra l { Kambree , aprn at 2/15/20 { Monette 23  { Sula 1433 printed on 10/3/24 7:13 am page 2581,vumc adult medical center east white, tyrone 1211 medical center dr mr { Clemon n: 047717361, dob: 7/27/1969, le { Diante gal { Clarinda  sex: m nashville tn 37232 visit date: 2/14/2023 02/14/2023 - office visit in vanderbilt diabetes and endocrinology (continued) clinical notes (cont { Grace inued) author: greenspan { Kahlan , debra l, ap { Katelynne r { Phillips n service: - author type: nurse practitioner filed: 2/15/2023 2:34 pm encounter date: 2/14/2023 { Danesha  status: writ { Maelynn te { Talaya n editor: greenspan, debra l, aprn (nurse practitioner) related problem: essential hypertension bp not adequately controlled. given his renal disease, will defer to nephrology which agent they want to increase. i did explain to tyronne t { Tyreke he impor { Gracee tance of good bp control to help slow  { Kimani down decline in kidney { Lucina s as well. electronica { Anirudh lly signed by greenspan, debra l, aprn at 2/15/2023 2:34 pm greenspan, debra l, aprn at 2/15/2023 1435 author: greenspan, debra l, aprn service: author type: nurse practitioner filed: 2/15/2023 2:36 pm encounter date: 2/14/2023 status: written editor: gree { Aine nspan, debra l, aprn (nurse practitioner) related problem: corns and callosities has large callouses on both feet. { Asa  ca { Gisella llous on r 5th mtp head appears to be getting ready ulcerate. refer to podiatry asap. electronically s { Karthik igned by greenspan, debra l, aprn a { Brina t 2/15/2023 2:36 pm progress n { Carolin otes greenspan, debra l, aprn at 2/14 { Olene /2023 1320 author: greenspan, debra l, aprn service: a { Delon uthor type: nurse practitioner filed: 2/15/2023 3:04 pm encounter date: 2/14/2023 status: signed ed { Issiah it { Shannel or: greenspan, debra l, aprn (nurse practitioner) endo adult clinic visit note date  { Jessi of service: 2/14/2023 patient: tyrone white dob: 7/2 { Laroy 7/1969 mrn: 047717361 referring provider: lippard,  { Alica giles a, aprn primary care pro { Kory vider: giles a lippard, aprn r { Evelena eas { Jeralyn on for visit: diabetes subjectiv { Karsyn e history of present illness: tyrone is a 53 y.o. who has been referred by lippard, giles a, aprn for further evaluation of type 2  { Saint diabetes mellitus. tyronne is accompanied t { Kadyn oday by { Meagen  his daughter, sahara, who lives { Ronica  with him currently. his chief concern today  { Cavin is painful feet. tyronne tells me { Drexel  that he was diagnosed with diabetes in his mid thirties in california at which time he was started on 70/30 insulin. he is unable to provide much history in a linear fashion today and has so { Marquese me difficulties exp { Briauna ressing him { Gadiel self, speech is slurred. he tells me he was a truck driver by trade. his daughter states he has been in and out of the hospital with either very high or very low su { Hallie gars. current treatment: trulicity 3mg weekly. printed on 10/3/24 7:13 am page 2582",
    {
        "entities": [
            [
                14,
                25,
                "PERSON"
            ],
            [
                59,
                66,
                "PERSON"
            ],
            [
                120,
                128,
                "PERSON"
            ],
            [
                200,
                207,
                "PERSON"
            ],
            [
                218,
                224,
                "PERSON"
            ],
            [
                241,
                247,
                "PERSON"
            ],
            [
                276,
                283,
                "PERSON"
            ],
            [
                398,
                404,
                "PERSON"
            ],
            [
                438,
                446,
                "PERSON"
            ],
            [
                516,
                522,
                "PERSON"
            ],
            [
                644,
                653,
                "PERSON"
            ],
            [
                677,
                683,
                "PERSON"
            ],
            [
                741,
                747,
                "PERSON"
            ],
            [
                847,
                855,
                "PERSON"
            ],
            [
                865,
                875,
                "PERSON"
            ],
            [
                964,
                972,
                "PERSON"
            ],
            [
                977,
                981,
                "PERSON"
            ],
            [
                1001,
                1008,
                "PERSON"
            ],
            [
                1079,
                1083,
                "PERSON"
            ],
            [
                1094,
                1101,
                "PERSON"
            ],
            [
                1123,
                1132,
                "PERSON"
            ],
            [
                1143,
                1153,
                "PERSON"
            ],
            [
                1246,
                1253,
                "PERSON"
            ],
            [
                1330,
                1336,
                "PERSON"
            ],
            [
                1449,
                1456,
                "PERSON"
            ],
            [
                1558,
                1565,
                "PERSON"
            ],
            [
                1621,
                1629,
                "PERSON"
            ],
            [
                1706,
                1715,
                "PERSON"
            ],
            [
                1726,
                1734,
                "PERSON"
            ],
            [
                1757,
                1763,
                "PERSON"
            ],
            [
                1839,
                1849,
                "PERSON"
            ],
            [
                1898,
                1906,
                "PERSON"
            ],
            [
                2008,
                2013,
                "PERSON"
            ],
            [
                2055,
                2063,
                "PERSON"
            ],
            [
                2135,
                2143,
                "PERSON"
            ],
            [
                2204,
                2207,
                "PERSON"
            ],
            [
                2266,
                2272,
                "PERSON"
            ],
            [
                2283,
                2289,
                "PERSON"
            ],
            [
                2315,
                2320,
                "PERSON"
            ],
            [
                2380,
                2388,
                "PERSON"
            ],
            [
                2472,
                2480,
                "PERSON"
            ],
            [
                2521,
                2529,
                "PERSON"
            ],
            [
                2546,
                2553,
                "PERSON"
            ],
            [
                2597,
                2602,
                "PERSON"
            ],
            [
                2607,
                2612,
                "PERSON"
            ],
            [
                2687,
                2695,
                "PERSON"
            ],
            [
                2731,
                2736,
                "PERSON"
            ],
            [
                2780,
                2788,
                "PERSON"
            ],
            [
                2853,
                2861,
                "PERSON"
            ],
            [
                2956,
                2963,
                "PERSON"
            ],
            [
                2990,
                3001,
                "PERSON"
            ],
            [
                3017,
                3025,
                "PERSON"
            ],
            [
                3086,
                3096,
                "PERSON"
            ],
            [
                3208,
                3216,
                "PERSON"
            ],
            [
                3248,
                3256,
                "PERSON"
            ],
            [
                3418,
                3426,
                "PERSON"
            ],
            [
                3446,
                3454,
                "PERSON"
            ],
            [
                3460,
                3465,
                "PERSON"
            ],
            [
                3580,
                3587,
                "PERSON"
            ],
            [
                3622,
                3629,
                "PERSON"
            ],
            [
                3635,
                3644,
                "PERSON"
            ],
            [
                3795,
                3801,
                "PERSON"
            ],
            [
                3828,
                3835,
                "PERSON"
            ],
            [
                3851,
                3861,
                "PERSON"
            ],
            [
                3865,
                3874,
                "PERSON"
            ],
            [
                3972,
                3980,
                "PERSON"
            ],
            [
                3996,
                4004,
                "PERSON"
            ],
            [
                4009,
                4016,
                "PERSON"
            ],
            [
                4256,
                4263,
                "PERSON"
            ],
            [
                4274,
                4281,
                "PERSON"
            ],
            [
                4322,
                4329,
                "PERSON"
            ],
            [
                4354,
                4361,
                "PERSON"
            ],
            [
                4386,
                4394,
                "PERSON"
            ],
            [
                4653,
                4658,
                "PERSON"
            ],
            [
                4775,
                4779,
                "PERSON"
            ],
            [
                4785,
                4793,
                "PERSON"
            ],
            [
                4898,
                4906,
                "PERSON"
            ],
            [
                4944,
                4950,
                "PERSON"
            ],
            [
                4983,
                4991,
                "PERSON"
            ],
            [
                5031,
                5037,
                "PERSON"
            ],
            [
                5094,
                5100,
                "PERSON"
            ],
            [
                5202,
                5209,
                "PERSON"
            ],
            [
                5214,
                5222,
                "PERSON"
            ],
            [
                5309,
                5315,
                "PERSON"
            ],
            [
                5370,
                5376,
                "PERSON"
            ],
            [
                5430,
                5436,
                "PERSON"
            ],
            [
                5469,
                5474,
                "PERSON"
            ],
            [
                5507,
                5515,
                "PERSON"
            ],
            [
                5521,
                5529,
                "PERSON"
            ],
            [
                5564,
                5571,
                "PERSON"
            ],
            [
                5705,
                5711,
                "PERSON"
            ],
            [
                5757,
                5763,
                "PERSON"
            ],
            [
                5773,
                5780,
                "PERSON"
            ],
            [
                5815,
                5822,
                "PERSON"
            ],
            [
                5870,
                5876,
                "PERSON"
            ],
            [
                5912,
                5919,
                "PERSON"
            ],
            [
                6113,
                6122,
                "PERSON"
            ],
            [
                6144,
                6152,
                "PERSON"
            ],
            [
                6166,
                6173,
                "PERSON"
            ],
            [
                6340,
                6347,
                "PERSON"
            ]
        ]
    }
),(
    "vumc adult one hundred oaks white, tyrone 719 thompson lane, { Jaidon  nashville mrn: 047717361, dob: 7/27/1969, legal se { Junie x: m nashville tn 37204 adm: 9/14/2023, d/c: 9/14/2023 09/14/2023 - full pft pre/post bronchodil in vanderbilt pulmonary  { Teara clinic (continued) imaging reports (continued) technician comments/assessments pt ha { Vasiliki s  { Cosme bell's palsy. pt did the best h { Taylen e could. p { Georgianne re fvc only one acceptable { Lynlee   { Teisha trial ou { Vennie t o { Zaire f various efforts. pre fvc does { Zayna  not repeat. post fvc ha { Bastian s 2 acceptable trials out of various efforts. post fvc does not repeat. dlco does not meet ats due to pt unable to inhale to at least 85% vc on both trials. pt received 2 puffs albuterol mdi via spacer with  reactions noted. prelimi { Colon nary description forced expiratory spirometry demonstrates a  { Silvester severe obstructive ventilato { Azia ry defect. there is  respo { Jazlene nse to inhaled bronchodilators. the total lung capacity by body { Katana  plethysmography is normal. the body plethysmograph rv is increased indicating early airway closure. the diffusing capacity is mildly reduced indic { Dasean ating a decreased { Sander  alveolar-capillary surface area for gas exchange. the inspired volume is less than th { Angeli e vital capacity indicating th { Junita at the actual dlco may be underestimated. the low peak flow raises the question of a fixed airway { Treveon  obstru { Zayvion ction { Anamarie  or poor effort. dia { Jaydah gnosis: sever { Meriam e obstructive v { Dolores entilatory defect. mild gas transfer defect. pf reference: spirometry (6-7): dockery/wang, (8-80): nhanesiii & knudson; lung volumes: crapo; dlco: miller calibratio { Kanye n date: 9/14/2023 calibration data: temp: 24°c pb { Larson ar: 752mmhg btps: 1.078 patient { Tayvon : whit { Raizel e, tyrone test date: 9/14/2023 page: 3 of 3 full pft pr { Daneen e and post bronchodilation resulted: 09/27/23 2045, result status: final result ordering provider: de witte, anton jordan, md 09/14/23 0802  { Caysen order status: c { Kennie ompleted filed by: interface, ancillary results and orders-general performed: 09/14/23 0838 - 09/14/23 09/27/23 { Ahuva  2046 accession num { Fleta ber: 7988 { Burnice 7266 resulting { Hogan  lab: v { Iran umc compas pft acknowledged by: de witte, anton jordan, md on 10/17/23 2005 components component value re { Lanisha ference range flag lab { Vittoria  printed on 10/3/24 7:13 am page { Zaya  1679,vumc a { Athanasios dult { Stevens  o { Deseree ne hundred oaks white, tyrone 719 thompson lane, nashville mrn: 047717361, dob: 7/27/19 { Brendyn 69, legal sex: m nashville tn 37204 adm: 9/14/2023, d/c: 9/14/2023 09/1 { Khristina 4/2023 - full pft pre/post bronchodil in vanderbilt pulmonary clinic (continued) imaging reports  { Kunal (continued) fvc p { Raquan redicted value pre bronchodilator 5.03 l compas f { Rodman vc actual value pre bronchodilator 3.26 l compas fvc % of predicted value pre 65 % compas bronchodila { Trysten tor  { Alejandrina fvc actual value post bronchodilator 3.45 l compas fvc % of predicted value post 69 % comp { Earlie as bronchodilato { Ova r fev1 predicted value pre  { Saleh bronchodilator 3.86 l compas fev1 ac { Armanda tual v { Arminda al { Jeffery ue pre bronchodilator 1.77 l compas fev1 % of predicted value pre 46 % compas bronch { Monisha odilator fev1 actual value post bronchodilator 1.64 l compas fev1 % of predicted value post 43 % compas bronchodilator fef25_75 predicted value { Yamile  pre 3.31 l/s compas bronchodilator fef25-75 actual value pre bronchodilator 1.44 l/s compas fef25_75 % o { Rasheem f predicted value pre 44 % compas bronchodilatory fef25-75 actual value post-bronchodilator 1.21 l/s compas fef25-75 { Gissell  % of predicted value post 36 % compas bronchodilator dlco predicted value 3 { Kareen 0.03 ml/m { Sara in/mmhg compas dlco actual value pre bronchodilator 18.37 ml/min/mmhg compa { Ardath s dlco % of pr { Jenevieve ed { Renate icted value pre 61 % compas bronchodilator dlco (hb) predic { Louanna ted value 30.03 ml/min/mmhg com { Nicholette pas dlco (hb) actual value pre bronchodil { Reginia ator  { Clorinda 19.94 ml/min/mmhg compas dlco (hb) % of predicted value 66 % compas dlco/va predicted va { Oda lue { Raniya  4.76 ml/min/mmhg/l - compas dlco/va actual value pre bronchodilator 4.14 ml/min/mmhg/l compas dlco/va % of predicted val { Crystal ue pre 8 { Isidoro 7 % compas bronchodilator hb predicted value pred { Iver _mean g/dl compas  { Kenzo hb actual val { Yuvraj ue rest1 g { Blanchie /dl compas tlc predicted value 7.07 l compas tlc actual pre bronchodilator 6.25 l compas tlc % of predicted value pre b 88 % compas rv predicted value pr 2.1 { Laquinta 5 l compas rv actual value 2.96 l compas rv % of pr { Shirl edicted value pre bronchodilator 138 % compas rv { Braxtyn /tlc predicted % 31 % compas rv { Corrie /tlc actual % pre bronchodilator 47 % compa { Kolbe s { Cleone  rv { Darice /tlc % of predicted pre bronchodilator 153 % compas  { Rylin interp - compas result: there is  present by fev1/fvc: however, there is an increase { Kofi  in r { Sione v and rv/tlc { Victoria  sug { Abbigayle gestive of gas trapping. there is  present. dlco data do n { Bobbijo ot meet criteria fo { Braylin r acceptability. the flow volume loop shows flattenin { Dominic g which may indicate large airway obstruction or may { Dot  be related to reduced effort. clinical correlation is advised. testing performed by lab - abbreviation name director address val { Gardenia id date range 190 compas vumc compas  { Reya unknown unknown 10/05/16 1421 - present pft indications printed on  { Zavian 10/3/24 7:13 { Deane  am page 1680",
    {
        "entities": [
            [
                63,
                70,
                "PERSON"
            ],
            [
                124,
                130,
                "PERSON"
            ],
            [
                254,
                260,
                "PERSON"
            ],
            [
                347,
                356,
                "PERSON"
            ],
            [
                361,
                367,
                "PERSON"
            ],
            [
                401,
                408,
                "PERSON"
            ],
            [
                421,
                432,
                "PERSON"
            ],
            [
                461,
                468,
                "PERSON"
            ],
            [
                472,
                479,
                "PERSON"
            ],
            [
                490,
                497,
                "PERSON"
            ],
            [
                503,
                509,
                "PERSON"
            ],
            [
                543,
                549,
                "PERSON"
            ],
            [
                576,
                584,
                "PERSON"
            ],
            [
                819,
                825,
                "PERSON"
            ],
            [
                889,
                899,
                "PERSON"
            ],
            [
                930,
                935,
                "PERSON"
            ],
            [
                964,
                972,
                "PERSON"
            ],
            [
                1038,
                1045,
                "PERSON"
            ],
            [
                1195,
                1202,
                "PERSON"
            ],
            [
                1222,
                1229,
                "PERSON"
            ],
            [
                1318,
                1325,
                "PERSON"
            ],
            [
                1358,
                1365,
                "PERSON"
            ],
            [
                1465,
                1473,
                "PERSON"
            ],
            [
                1483,
                1491,
                "PERSON"
            ],
            [
                1499,
                1508,
                "PERSON"
            ],
            [
                1531,
                1538,
                "PERSON"
            ],
            [
                1554,
                1561,
                "PERSON"
            ],
            [
                1579,
                1587,
                "PERSON"
            ],
            [
                1754,
                1760,
                "PERSON"
            ],
            [
                1812,
                1819,
                "PERSON"
            ],
            [
                1853,
                1860,
                "PERSON"
            ],
            [
                1869,
                1876,
                "PERSON"
            ],
            [
                1934,
                1941,
                "PERSON"
            ],
            [
                2084,
                2091,
                "PERSON"
            ],
            [
                2109,
                2116,
                "PERSON"
            ],
            [
                2230,
                2236,
                "PERSON"
            ],
            [
                2258,
                2264,
                "PERSON"
            ],
            [
                2276,
                2284,
                "PERSON"
            ],
            [
                2301,
                2307,
                "PERSON"
            ],
            [
                2317,
                2322,
                "PERSON"
            ],
            [
                2430,
                2438,
                "PERSON"
            ],
            [
                2463,
                2472,
                "PERSON"
            ],
            [
                2507,
                2512,
                "PERSON"
            ],
            [
                2527,
                2538,
                "PERSON"
            ],
            [
                2545,
                2553,
                "PERSON"
            ],
            [
                2558,
                2566,
                "PERSON"
            ],
            [
                2656,
                2664,
                "PERSON"
            ],
            [
                2738,
                2748,
                "PERSON"
            ],
            [
                2848,
                2854,
                "PERSON"
            ],
            [
                2874,
                2881,
                "PERSON"
            ],
            [
                2933,
                2940,
                "PERSON"
            ],
            [
                3044,
                3052,
                "PERSON"
            ],
            [
                3059,
                3071,
                "PERSON"
            ],
            [
                3164,
                3171,
                "PERSON"
            ],
            [
                3190,
                3194,
                "PERSON"
            ],
            [
                3224,
                3230,
                "PERSON"
            ],
            [
                3269,
                3277,
                "PERSON"
            ],
            [
                3286,
                3294,
                "PERSON"
            ],
            [
                3299,
                3307,
                "PERSON"
            ],
            [
                3394,
                3402,
                "PERSON"
            ],
            [
                3548,
                3555,
                "PERSON"
            ],
            [
                3663,
                3671,
                "PERSON"
            ],
            [
                3790,
                3798,
                "PERSON"
            ],
            [
                3877,
                3884,
                "PERSON"
            ],
            [
                3896,
                3901,
                "PERSON"
            ],
            [
                3979,
                3986,
                "PERSON"
            ],
            [
                4003,
                4013,
                "PERSON"
            ],
            [
                4018,
                4025,
                "PERSON"
            ],
            [
                4087,
                4095,
                "PERSON"
            ],
            [
                4129,
                4140,
                "PERSON"
            ],
            [
                4184,
                4192,
                "PERSON"
            ],
            [
                4200,
                4209,
                "PERSON"
            ],
            [
                4300,
                4304,
                "PERSON"
            ],
            [
                4310,
                4317,
                "PERSON"
            ],
            [
                4441,
                4449,
                "PERSON"
            ],
            [
                4460,
                4468,
                "PERSON"
            ],
            [
                4520,
                4525,
                "PERSON"
            ],
            [
                4546,
                4552,
                "PERSON"
            ],
            [
                4568,
                4575,
                "PERSON"
            ],
            [
                4588,
                4597,
                "PERSON"
            ],
            [
                4757,
                4766,
                "PERSON"
            ],
            [
                4820,
                4826,
                "PERSON"
            ],
            [
                4877,
                4885,
                "PERSON"
            ],
            [
                4919,
                4926,
                "PERSON"
            ],
            [
                4972,
                4978,
                "PERSON"
            ],
            [
                4982,
                4989,
                "PERSON"
            ],
            [
                4995,
                5002,
                "PERSON"
            ],
            [
                5057,
                5063,
                "PERSON"
            ],
            [
                5150,
                5155,
                "PERSON"
            ],
            [
                5163,
                5169,
                "PERSON"
            ],
            [
                5184,
                5193,
                "PERSON"
            ],
            [
                5200,
                5210,
                "PERSON"
            ],
            [
                5271,
                5279,
                "PERSON"
            ],
            [
                5301,
                5309,
                "PERSON"
            ],
            [
                5365,
                5373,
                "PERSON"
            ],
            [
                5428,
                5432,
                "PERSON"
            ],
            [
                5564,
                5573,
                "PERSON"
            ],
            [
                5613,
                5618,
                "PERSON"
            ],
            [
                5688,
                5695,
                "PERSON"
            ],
            [
                5710,
                5716,
                "PERSON"
            ]
        ]
    }
),(
    "vumc a { Sharri dult hospital white, tyrone 1211 medical ce { Avian nter dr. mrn: 0477 { Chazz 17361, do { Caroll b: 7/27/1969,  { Lashaun legal sex: m nash { Shaunta ville tn 37232-0004 visit date: 6/3/20 { Gerome 23 06/ { Shamir 03/2023 - procedure pass  { Aisling in vanderbilt university adult hospital facesheet report patient { Charlesetta  demographics p { Danasia atient { Aarron  name { Delos  mrn legal dob address { Sharod  phone white, tyrone 0477173 sex 7/ { Adair 27/1969 apt 705 615-260-2291 (home) 61 m 1101 edgehill ave 615-260 { Bridgit -2291 (mobile) nas { Yadhira hville tn 37203 *preferred* hospital  { Collis account not  { Salvadore on file admission information current informati { Lakayla on  { Marguerita attending provider admitting provider admission type admission status unknown status admission d { Geovani ate/time discharge date/time hospital service auth/cer { Kalan t s { Librado tatus hospital  { Solon area unit room/bed referring p { Lakeya rovider 06/03/2 { Eron 023 - procedure pass in vanderbilt universit { Jobe y adult hospital (continued) vi { Norma sit information admi { Patton ssion information arrival d { Darcel ate/time: admit date/time: 0 { Mikey 6/03/2023 ip adm. date/time: admission type: point of origin: admit category: means of arrival: primary service: secondary servic { Nicolo e: n/a transfer source: service are { Caressa a: unit: admit provider: attending provider:  { Halli referring provider: discharge information date/ti { Lahoma me: - disposition: - destination: - provider: - unit: -  { Mehki printed on 10/3/24 7:13 am page 2129,vumc adu { Amor lt hospita { Delorise l white, { Venetia  tyr { Clancy one 1211 medical cen { Tiffany ter dr. m { Jakiya rn: 047717361, dob: 7 { Malerie /27/1969, { Starlene  legal sex: m nas { Tayah hville tn { Ric  37232-0004 adm: 6/2/2023, d/c: 6/3/2023 06/02/2023 - ed to hosp-admission (discharged) in vande { Jerilynn rbilt university adult hospital facesheet report patient demograp { Marquitta hics patient name mrn legal dob address phone { Sakina  white, tyrone 0477173 sex 7/2 { Trixie 7/1969 apt 705 615 { Adil -26 { Antron 0-2291 (home) 61 m 1101 edgehil { Marquette l ave 615-260-2291 ( { Angelee mobile) nashville tn 37203 *preferred* hospita { Danell l  { Female account name acct id class stat { Khushi us primary coverage white, tyrone 1017062460  { Anil observation closed amerivantage wellpoint  { Lakota medicare -  { Kailie amerivantage amerigroup wellpoint ma guarantor account (for hospital account #1017 { Taren 06 { Samantha 2460) r { Adel elation to name pt service area active? acct type white, tyron { Amer e self vumc msa yes personal/fa { Rolla mily address phone apt 705 615-260-2291(h) 1101 edgehill ave nashville, tn 37203 coverage information (for hospital account #1017062460) 1. ame { Mileena riv { Apolinar antage wellpoint medicare/amerivantage amerigroup wellpoint { Mihir  ma f/o payor/plan precert # amerivantage wellpoint medicare/amerivantage amerigroup w { Chaney ellpoint  { Nori ma s { Alfred ubscrib { Gabriell er subscriber # white, { Jarrid  tyrone 768w { Joeann 12411 address phone tn claims po box 61010 virginia beach, va 23466 { Lorissa -1010 2. zzzmcaid of tennessee/medicaid supplem { Micaiah ental f/o payor/plan pre { Cameren cert # zzzmcaid of tennessee { Carlie / { Akayla medicaid supplemental subscriber subscriber # white, tyrone td525606373 address phone po b { Nataleigh ox 460 nashville, tn 37202-04 { Shondra 60 admi { Arnell ssion information current i { Doria nformat { Amaury ion attending at discha { Ames rge admitt { Ash ing provide { Mykala r admission type admission statu { Sarabeth s schwall, allis { Arnett on { Lem  leigh, md schwall,  { Aleeah allison leigh, md 61 { Bethanne 5- emergency confirmed discharge { Deangela  936-8219 { Ebone  admission date/time discharge date { Lanaya /time hospital servic { Tyonna e auth/cert { Verity  status 06/02/23 1423 06 { Damya /03/23 1756 general internal medicine incomplete printed on 10/3/24 7:13 am p { Marijo age 2130",
    {
        "entities": [
            [
                9,
                16,
                "PERSON"
            ],
            [
                62,
                68,
                "PERSON"
            ],
            [
                89,
                95,
                "PERSON"
            ],
            [
                107,
                114,
                "PERSON"
            ],
            [
                131,
                139,
                "PERSON"
            ],
            [
                159,
                167,
                "PERSON"
            ],
            [
                208,
                215,
                "PERSON"
            ],
            [
                224,
                231,
                "PERSON"
            ],
            [
                259,
                267,
                "PERSON"
            ],
            [
                334,
                346,
                "PERSON"
            ],
            [
                364,
                372,
                "PERSON"
            ],
            [
                381,
                388,
                "PERSON"
            ],
            [
                396,
                402,
                "PERSON"
            ],
            [
                427,
                434,
                "PERSON"
            ],
            [
                472,
                478,
                "PERSON"
            ],
            [
                547,
                555,
                "PERSON"
            ],
            [
                576,
                584,
                "PERSON"
            ],
            [
                624,
                631,
                "PERSON"
            ],
            [
                646,
                656,
                "PERSON"
            ],
            [
                706,
                714,
                "PERSON"
            ],
            [
                720,
                731,
                "PERSON"
            ],
            [
                830,
                838,
                "PERSON"
            ],
            [
                895,
                901,
                "PERSON"
            ],
            [
                907,
                915,
                "PERSON"
            ],
            [
                933,
                939,
                "PERSON"
            ],
            [
                972,
                979,
                "PERSON"
            ],
            [
                997,
                1002,
                "PERSON"
            ],
            [
                1049,
                1054,
                "PERSON"
            ],
            [
                1088,
                1094,
                "PERSON"
            ],
            [
                1117,
                1124,
                "PERSON"
            ],
            [
                1154,
                1161,
                "PERSON"
            ],
            [
                1192,
                1198,
                "PERSON"
            ],
            [
                1330,
                1337,
                "PERSON"
            ],
            [
                1375,
                1383,
                "PERSON"
            ],
            [
                1431,
                1437,
                "PERSON"
            ],
            [
                1489,
                1496,
                "PERSON"
            ],
            [
                1555,
                1561,
                "PERSON"
            ],
            [
                1609,
                1614,
                "PERSON"
            ],
            [
                1627,
                1636,
                "PERSON"
            ],
            [
                1647,
                1655,
                "PERSON"
            ],
            [
                1662,
                1669,
                "PERSON"
            ],
            [
                1692,
                1700,
                "PERSON"
            ],
            [
                1712,
                1719,
                "PERSON"
            ],
            [
                1743,
                1751,
                "PERSON"
            ],
            [
                1763,
                1772,
                "PERSON"
            ],
            [
                1792,
                1798,
                "PERSON"
            ],
            [
                1810,
                1814,
                "PERSON"
            ],
            [
                1913,
                1922,
                "PERSON"
            ],
            [
                1990,
                2000,
                "PERSON"
            ],
            [
                2048,
                2055,
                "PERSON"
            ],
            [
                2088,
                2095,
                "PERSON"
            ],
            [
                2116,
                2121,
                "PERSON"
            ],
            [
                2127,
                2134,
                "PERSON"
            ],
            [
                2168,
                2178,
                "PERSON"
            ],
            [
                2201,
                2209,
                "PERSON"
            ],
            [
                2258,
                2265,
                "PERSON"
            ],
            [
                2270,
                2277,
                "PERSON"
            ],
            [
                2311,
                2318,
                "PERSON"
            ],
            [
                2366,
                2371,
                "PERSON"
            ],
            [
                2416,
                2423,
                "PERSON"
            ],
            [
                2437,
                2444,
                "PERSON"
            ],
            [
                2529,
                2535,
                "PERSON"
            ],
            [
                2540,
                2549,
                "PERSON"
            ],
            [
                2559,
                2564,
                "PERSON"
            ],
            [
                2629,
                2634,
                "PERSON"
            ],
            [
                2668,
                2674,
                "PERSON"
            ],
            [
                2820,
                2828,
                "PERSON"
            ],
            [
                2834,
                2843,
                "PERSON"
            ],
            [
                2905,
                2911,
                "PERSON"
            ],
            [
                3000,
                3007,
                "PERSON"
            ],
            [
                3019,
                3024,
                "PERSON"
            ],
            [
                3031,
                3038,
                "PERSON"
            ],
            [
                3048,
                3057,
                "PERSON"
            ],
            [
                3082,
                3089,
                "PERSON"
            ],
            [
                3104,
                3111,
                "PERSON"
            ],
            [
                3181,
                3189,
                "PERSON"
            ],
            [
                3239,
                3247,
                "PERSON"
            ],
            [
                3274,
                3282,
                "PERSON"
            ],
            [
                3313,
                3320,
                "PERSON"
            ],
            [
                3324,
                3331,
                "PERSON"
            ],
            [
                3424,
                3434,
                "PERSON"
            ],
            [
                3466,
                3474,
                "PERSON"
            ],
            [
                3484,
                3491,
                "PERSON"
            ],
            [
                3521,
                3527,
                "PERSON"
            ],
            [
                3537,
                3544,
                "PERSON"
            ],
            [
                3570,
                3575,
                "PERSON"
            ],
            [
                3588,
                3592,
                "PERSON"
            ],
            [
                3606,
                3613,
                "PERSON"
            ],
            [
                3648,
                3657,
                "PERSON"
            ],
            [
                3676,
                3683,
                "PERSON"
            ],
            [
                3688,
                3692,
                "PERSON"
            ],
            [
                3715,
                3722,
                "PERSON"
            ],
            [
                3745,
                3754,
                "PERSON"
            ],
            [
                3789,
                3798,
                "PERSON"
            ],
            [
                3810,
                3816,
                "PERSON"
            ],
            [
                3854,
                3861,
                "PERSON"
            ],
            [
                3885,
                3892,
                "PERSON"
            ],
            [
                3906,
                3913,
                "PERSON"
            ],
            [
                3940,
                3946,
                "PERSON"
            ],
            [
                4026,
                4033,
                "PERSON"
            ]
        ]
    }
),(
    "vumc vi { Tryston s midtown white, tyrone 337 { Aven  22nd av { Charlean e n mrn: 047717361, dob: 7/27/1969, legal sex: m nashville tn 37203 adm: 4/16/2024 { Hillary , d/c: 4/16/2024 { Adalina  04/16/2024 - fl video swallow w speec { Sharde h in vand { Bakari erbilt imaging services midtown (continued) flowsheets (continued) noms swallowing 5  { Jennine -pd at 04/17/24 1333 recommendations/treatment ro { Roya w name 04/1 { Tempest 7/24  { Tait 1332 plan/recommendations diet regular; thin liquids recommendation -pd at 04/1 { Yousif 7/24 1333 additional consider neurology fol { Amberlynn low-up consult; outpatient recommendation { Audria  swallowing therapy { Kellen  s -pd at 04/17/24 1333 compen { Shamiya satory uprigh { Bram t as po { Chauncy ssib { Davante le swallowing for all ora { Jahari l strategies intake; small bites/sips;minimize distractions { Michaelangelo ;ea slowly;alternate soli { Ryen ds and liquids - pd at 0 { Jerrilyn 4/17/24 1333 r { Shirlie ecommended whole wi { Yecenia th liquids form of one at a time -pd at medications 04/17/24 1333 swallowing iii - emst -pd  { Emilyn at exercises 04/17/24 1333 vfs { Minta s row name 04/17/24 1325 vfss oral mechanism  weakness; decr { Refugia eas { Robynn e d labial mobility/strength;de creased l { Tamarra ingual mobil { Geremy ity/strength;ad { Loring  equate dentition;dysphonia dysart { Robyn hria;poor breath suppo { Timoteo rt; weak voliti { Lisabeth onal coug { Marcene h  { Maycie and/or throat clear - pd at 04/17/24 1 1332 respiratory ro { Roselee om air -pd at status 04/17/24 1332 utensils medicine cup;c { Ventura up;spoon -pd at 04/17/24 feeding patient fed se { Lakeysha lf -pd at 04/ { Micheline 17/24 1332 positioning seated in chair { Sami ; standing -pd at 04/17/24 133 { Vernia 2 projection  { Edan lateral;anterior/pos terior -pd at 04/17/24 133 { Tiernan 2 bolus -pd at pr { Henretta e { Jalayah sentation 04/17/24 1332 printed on  { Kaysha 10/3/2 { Kyrstin 4 7:12 am page 1073,v { Prentis umc vis midtown { Sandeep  white, tyrone 337 22nd ave n mrn: 047717361, dob: 7 { Thanh /27/196 { Kitana 9, legal sex: m nashville tn 37203 adm: 4/16 { Krislyn /2024, d/c: 4/16/2024 04/16/2024 - fl video swal { Larraine low w speech in vanderbi { Marielena lt imaging services midtown (continued) { Vannesa  flowsheets (continued) oral phase  loss of  { Jerrett findings bolus;hes { Elyana itation/red uction in ap propulsion; tongue pumping;pr { Jeanice olonged masticati { Jon on; piecem eal deglution;uncont { Barnett r { Daneil o iled bolus; decrease { Kiyan d palatal elevation with nasal penetration { Ryden ; lingual residue; palatal residue -pd at 04/17/2 { Tedd 4 1332 pharyngeal  findings aspiration; laryngeal vestibule penet { Morghan ration; delayed onset of swallow; vallecular residue mildly we { Rashell ak phary { Kage ngeal response. - pd at 04/17/24 1 13 { Alexsis 32 laryn { Breasia geal t { Mafalda hin liquid vestibule barium; ultra- { Nadiya thin penetration with  { Rella liquid barium (50 ml prepared thi { Rashaud n liquid  { Camry bar { Liddie ium and 50 ml water) -pd at 04/17/24 1332 compensatory alternating liq { Darick uids strategies and solids was atte { Jonte mpted effective in clearing vallecular residues { Judith . -pd at 04/17/24 1332 eso { Lige phageal  phase findings abnormalities; see radi { Shia ology report -pd at 04/17/24 13 { Jonie 32 anatomy grossly within  { Kanika normal limits -pd at 04/17/24 1332 user key (r) = recorded by, (t) = taken by, (c { Maelyn ) = cosigned by initials n { Memphis ame provider { Nahomi  type discipline pd duvall,  { Saralyn pamela e, slp speech and { Pearson  language patholog { Randall ist slp  { Shai sh henry, s { Mattew helly moni { Amairani que,  { Elizbeth techn { Divine ologist technician r.t.(r)(arrt) aa adt, auto  { Karrington compl { Kaytlynn ete hov - contacts messages after visit summary reminder from to se { Rion nt for delivery on mycha { Anijah rt, generic white, tyrone 4/17/2024 1:08 am 4/18/2024 last read i { Emilly n my health at vanderbilt not read printed on 10/3/24 7:12 am page 1074",
    {
        "entities": [
            [
                10,
                18,
                "PERSON"
            ],
            [
                48,
                53,
                "PERSON"
            ],
            [
                64,
                73,
                "PERSON"
            ],
            [
                158,
                166,
                "PERSON"
            ],
            [
                185,
                193,
                "PERSON"
            ],
            [
                234,
                241,
                "PERSON"
            ],
            [
                253,
                260,
                "PERSON"
            ],
            [
                348,
                356,
                "PERSON"
            ],
            [
                408,
                413,
                "PERSON"
            ],
            [
                427,
                435,
                "PERSON"
            ],
            [
                443,
                448,
                "PERSON"
            ],
            [
                530,
                537,
                "PERSON"
            ],
            [
                583,
                593,
                "PERSON"
            ],
            [
                637,
                644,
                "PERSON"
            ],
            [
                666,
                673,
                "PERSON"
            ],
            [
                706,
                714,
                "PERSON"
            ],
            [
                730,
                735,
                "PERSON"
            ],
            [
                745,
                753,
                "PERSON"
            ],
            [
                760,
                768,
                "PERSON"
            ],
            [
                796,
                803,
                "PERSON"
            ],
            [
                865,
                879,
                "PERSON"
            ],
            [
                907,
                912,
                "PERSON"
            ],
            [
                939,
                948,
                "PERSON"
            ],
            [
                965,
                973,
                "PERSON"
            ],
            [
                995,
                1003,
                "PERSON"
            ],
            [
                1098,
                1105,
                "PERSON"
            ],
            [
                1138,
                1144,
                "PERSON"
            ],
            [
                1207,
                1215,
                "PERSON"
            ],
            [
                1221,
                1228,
                "PERSON"
            ],
            [
                1272,
                1280,
                "PERSON"
            ],
            [
                1295,
                1302,
                "PERSON"
            ],
            [
                1320,
                1327,
                "PERSON"
            ],
            [
                1364,
                1370,
                "PERSON"
            ],
            [
                1395,
                1403,
                "PERSON"
            ],
            [
                1421,
                1430,
                "PERSON"
            ],
            [
                1442,
                1450,
                "PERSON"
            ],
            [
                1455,
                1462,
                "PERSON"
            ],
            [
                1523,
                1531,
                "PERSON"
            ],
            [
                1592,
                1600,
                "PERSON"
            ],
            [
                1650,
                1659,
                "PERSON"
            ],
            [
                1675,
                1685,
                "PERSON"
            ],
            [
                1726,
                1731,
                "PERSON"
            ],
            [
                1764,
                1771,
                "PERSON"
            ],
            [
                1787,
                1792,
                "PERSON"
            ],
            [
                1842,
                1850,
                "PERSON"
            ],
            [
                1870,
                1879,
                "PERSON"
            ],
            [
                1883,
                1891,
                "PERSON"
            ],
            [
                1929,
                1936,
                "PERSON"
            ],
            [
                1945,
                1953,
                "PERSON"
            ],
            [
                1977,
                1985,
                "PERSON"
            ],
            [
                2003,
                2011,
                "PERSON"
            ],
            [
                2066,
                2072,
                "PERSON"
            ],
            [
                2082,
                2089,
                "PERSON"
            ],
            [
                2136,
                2144,
                "PERSON"
            ],
            [
                2195,
                2204,
                "PERSON"
            ],
            [
                2231,
                2241,
                "PERSON"
            ],
            [
                2283,
                2291,
                "PERSON"
            ],
            [
                2338,
                2346,
                "PERSON"
            ],
            [
                2367,
                2374,
                "PERSON"
            ],
            [
                2431,
                2439,
                "PERSON"
            ],
            [
                2459,
                2463,
                "PERSON"
            ],
            [
                2497,
                2505,
                "PERSON"
            ],
            [
                2509,
                2516,
                "PERSON"
            ],
            [
                2541,
                2547,
                "PERSON"
            ],
            [
                2592,
                2598,
                "PERSON"
            ],
            [
                2650,
                2655,
                "PERSON"
            ],
            [
                2723,
                2731,
                "PERSON"
            ],
            [
                2796,
                2804,
                "PERSON"
            ],
            [
                2815,
                2820,
                "PERSON"
            ],
            [
                2860,
                2868,
                "PERSON"
            ],
            [
                2879,
                2887,
                "PERSON"
            ],
            [
                2896,
                2904,
                "PERSON"
            ],
            [
                2942,
                2949,
                "PERSON"
            ],
            [
                2974,
                2980,
                "PERSON"
            ],
            [
                3016,
                3024,
                "PERSON"
            ],
            [
                3036,
                3042,
                "PERSON"
            ],
            [
                3048,
                3055,
                "PERSON"
            ],
            [
                3128,
                3135,
                "PERSON"
            ],
            [
                3173,
                3179,
                "PERSON"
            ],
            [
                3229,
                3236,
                "PERSON"
            ],
            [
                3265,
                3270,
                "PERSON"
            ],
            [
                3320,
                3325,
                "PERSON"
            ],
            [
                3359,
                3365,
                "PERSON"
            ],
            [
                3394,
                3401,
                "PERSON"
            ],
            [
                3485,
                3492,
                "PERSON"
            ],
            [
                3521,
                3529,
                "PERSON"
            ],
            [
                3544,
                3551,
                "PERSON"
            ],
            [
                3582,
                3590,
                "PERSON"
            ],
            [
                3617,
                3625,
                "PERSON"
            ],
            [
                3646,
                3654,
                "PERSON"
            ],
            [
                3665,
                3670,
                "PERSON"
            ],
            [
                3684,
                3691,
                "PERSON"
            ],
            [
                3704,
                3713,
                "PERSON"
            ],
            [
                3721,
                3730,
                "PERSON"
            ],
            [
                3738,
                3745,
                "PERSON"
            ],
            [
                3794,
                3805,
                "PERSON"
            ],
            [
                3813,
                3822,
                "PERSON"
            ],
            [
                3892,
                3897,
                "PERSON"
            ],
            [
                3924,
                3931,
                "PERSON"
            ],
            [
                3999,
                4006,
                "PERSON"
            ]
        ]
    }
),(
    "vum { Journey c { Kodie  the van { Delcie derbilt clinic white, tyrone 1301 m { Evalena edical center dr mrn: { Jaelen  047717361, dob: { Yonathan   { Alysson 7/27/1969, l { Jakiyah egal sex: m the vanderbilt clinic visit date: 12/21 { Schuyler /2023 nashville tn 37232-0028 12/ { Elwanda 21/2023 - patien { Naja t messa { Luisangel ge { Amalie  in vanderbil { Memory t n { Neysa ephr { Tejas ol { Jonni ogy/re { Laylani nal { Shealyn  transplant clinic facesheet report patient { Aldair  demographics patient name mrn legal dob addr { Barrington ess phone white, tyrone 0477173 sex 7/27/19 { Duston 69 apt 705 615-2 { Dwan 60-2291 (home) 61 m 1101 edgehill a { Sahib ve { Brigit  615-260-2291 (mo { Emileigh bile) nashv { Lacresha ille tn  { Lynae 37203 *pr { Renna eferred { Sariya * hospital acc { Dayshawn ount not on f { Seferino ile admission  { Zeta information current information attendi { Bayron ng provider admitting provider admission type admi { Jarom ssi { Elexus on status unkn { Tiare own status admission date/time discharge  { Damario dat { Derian e/ti { Fahad me hospital service auth/cert status hospital area unit room/bed referring { Hagan  provider 12/21/2023 - patient message in vanderbilt nephrolo { Amiracle gy/renal transplant clinic (continu { Carmina ed) visit information  { Eleonora provider information encounter pr { Telisha ovider portalatin, gilda meli { Asael ssa, { Aundre  md d { Davi epartme { Desmon nt name address  { Aleeyah phone fax vanderbilt nephrology/renal 1301 m { Dodie edica { Pollie l  { Debera ce { Ottie nter { Saraya  dr 615-343-7592  { Vinita 615-343-8216  { Cillian transplant clinic suite 2501 nashville tn 37232 messages lab results from to sent and delivered gilda melissa portalatin, md white, tyrone 12/21/2023 7:47 pm last { Tylen  read in m { Ardyce y health at vanderbilt 12/21/2023 7:47 pm  { Lamia b { Torrey y rouse, rejean f (proxy for ty { Carmon rone w { Eoin hit { Shain e) hi mr. white, it was nice to meet you ye { Tywan s { Georganne terday in clinic. in  { Jameelah regards { Joshlyn  to your labs, your kidney { Lexa  function { Glennon  unfortunately still { Reco  r { Theresa emains low an { Thyra d for those  { Myrl re { Archie aso { Jaimi ns,  { Joselynn as  { Khloee discussed,  { Oona we have re { Sona ferred y { Barret ou for kidney transplant  { Gionni evaluation. in addition, { Ibraheem  we will like to see you in clinic again soon to discuss further dialysis options. please let us know if you { Matheus  have any questi { Aleyda ons or concern { Brooks s. happy holidays, gilda p { Fallyn ortalatin, m.d printed on 10/3/24 { Vernette  7:12 a { Devina m page 1321,vumc the vand { Kennedie erbilt clinic white, tyrone 1301 medical center dr mrn: 047717361 { Steward , dob: 7/27/1969, { Anaiyah  legal sex: m the va { Evita nderbilt clinic vi { Demar sit  { Martavious date: { Vander  12/21/2023 nashville tn 37232-0028 12/21/2023 - patient message in vanderbilt nephrology/r { Krystel enal transplant cli { Nicollette nic (continued) messages (continued) { Elson   { Kimbra printed on 10/3/24 7:12 am page 1322",
    {
        "entities": [
            [
                6,
                14,
                "PERSON"
            ],
            [
                18,
                24,
                "PERSON"
            ],
            [
                35,
                42,
                "PERSON"
            ],
            [
                80,
                88,
                "PERSON"
            ],
            [
                112,
                119,
                "PERSON"
            ],
            [
                138,
                147,
                "PERSON"
            ],
            [
                151,
                159,
                "PERSON"
            ],
            [
                174,
                182,
                "PERSON"
            ],
            [
                236,
                245,
                "PERSON"
            ],
            [
                281,
                289,
                "PERSON"
            ],
            [
                308,
                313,
                "PERSON"
            ],
            [
                323,
                333,
                "PERSON"
            ],
            [
                338,
                345,
                "PERSON"
            ],
            [
                361,
                368,
                "PERSON"
            ],
            [
                374,
                380,
                "PERSON"
            ],
            [
                387,
                393,
                "PERSON"
            ],
            [
                398,
                404,
                "PERSON"
            ],
            [
                413,
                421,
                "PERSON"
            ],
            [
                427,
                435,
                "PERSON"
            ],
            [
                481,
                488,
                "PERSON"
            ],
            [
                536,
                547,
                "PERSON"
            ],
            [
                593,
                600,
                "PERSON"
            ],
            [
                619,
                624,
                "PERSON"
            ],
            [
                662,
                668,
                "PERSON"
            ],
            [
                673,
                680,
                "PERSON"
            ],
            [
                700,
                709,
                "PERSON"
            ],
            [
                723,
                732,
                "PERSON"
            ],
            [
                743,
                749,
                "PERSON"
            ],
            [
                761,
                767,
                "PERSON"
            ],
            [
                777,
                784,
                "PERSON"
            ],
            [
                801,
                810,
                "PERSON"
            ],
            [
                826,
                835,
                "PERSON"
            ],
            [
                852,
                857,
                "PERSON"
            ],
            [
                899,
                906,
                "PERSON"
            ],
            [
                959,
                965,
                "PERSON"
            ],
            [
                971,
                978,
                "PERSON"
            ],
            [
                995,
                1001,
                "PERSON"
            ],
            [
                1045,
                1053,
                "PERSON"
            ],
            [
                1059,
                1066,
                "PERSON"
            ],
            [
                1073,
                1079,
                "PERSON"
            ],
            [
                1156,
                1162,
                "PERSON"
            ],
            [
                1226,
                1235,
                "PERSON"
            ],
            [
                1273,
                1281,
                "PERSON"
            ],
            [
                1306,
                1315,
                "PERSON"
            ],
            [
                1351,
                1359,
                "PERSON"
            ],
            [
                1391,
                1397,
                "PERSON"
            ],
            [
                1404,
                1411,
                "PERSON"
            ],
            [
                1419,
                1424,
                "PERSON"
            ],
            [
                1434,
                1441,
                "PERSON"
            ],
            [
                1460,
                1468,
                "PERSON"
            ],
            [
                1515,
                1521,
                "PERSON"
            ],
            [
                1529,
                1536,
                "PERSON"
            ],
            [
                1541,
                1548,
                "PERSON"
            ],
            [
                1553,
                1559,
                "PERSON"
            ],
            [
                1566,
                1573,
                "PERSON"
            ],
            [
                1593,
                1600,
                "PERSON"
            ],
            [
                1616,
                1624,
                "PERSON"
            ],
            [
                1789,
                1795,
                "PERSON"
            ],
            [
                1808,
                1815,
                "PERSON"
            ],
            [
                1860,
                1866,
                "PERSON"
            ],
            [
                1870,
                1877,
                "PERSON"
            ],
            [
                1911,
                1918,
                "PERSON"
            ],
            [
                1927,
                1932,
                "PERSON"
            ],
            [
                1938,
                1944,
                "PERSON"
            ],
            [
                1990,
                1996,
                "PERSON"
            ],
            [
                2000,
                2010,
                "PERSON"
            ],
            [
                2034,
                2043,
                "PERSON"
            ],
            [
                2053,
                2061,
                "PERSON"
            ],
            [
                2090,
                2095,
                "PERSON"
            ],
            [
                2107,
                2115,
                "PERSON"
            ],
            [
                2138,
                2143,
                "PERSON"
            ],
            [
                2148,
                2156,
                "PERSON"
            ],
            [
                2172,
                2178,
                "PERSON"
            ],
            [
                2193,
                2198,
                "PERSON"
            ],
            [
                2203,
                2210,
                "PERSON"
            ],
            [
                2216,
                2222,
                "PERSON"
            ],
            [
                2229,
                2238,
                "PERSON"
            ],
            [
                2244,
                2251,
                "PERSON"
            ],
            [
                2265,
                2270,
                "PERSON"
            ],
            [
                2283,
                2288,
                "PERSON"
            ],
            [
                2299,
                2306,
                "PERSON"
            ],
            [
                2334,
                2341,
                "PERSON"
            ],
            [
                2368,
                2377,
                "PERSON"
            ],
            [
                2488,
                2496,
                "PERSON"
            ],
            [
                2515,
                2522,
                "PERSON"
            ],
            [
                2539,
                2546,
                "PERSON"
            ],
            [
                2575,
                2582,
                "PERSON"
            ],
            [
                2618,
                2627,
                "PERSON"
            ],
            [
                2637,
                2644,
                "PERSON"
            ],
            [
                2672,
                2681,
                "PERSON"
            ],
            [
                2749,
                2757,
                "PERSON"
            ],
            [
                2777,
                2785,
                "PERSON"
            ],
            [
                2808,
                2814,
                "PERSON"
            ],
            [
                2835,
                2841,
                "PERSON"
            ],
            [
                2848,
                2859,
                "PERSON"
            ],
            [
                2867,
                2874,
                "PERSON"
            ],
            [
                2968,
                2976,
                "PERSON"
            ],
            [
                2998,
                3009,
                "PERSON"
            ],
            [
                3048,
                3054,
                "PERSON"
            ],
            [
                3058,
                3065,
                "PERSON"
            ]
        ]
    }
)]

In [58]:
import pandas as pd
import os
from tqdm import tqdm
import spacy
from spacy.tokens import DocBin

# Load a blank spaCy model
nlp = spacy.blank("en")  # Load blank English model

db = DocBin()

for text, annot in tqdm(TRAIN_DATA):
    doc = nlp.make_doc(text)
    ents = []
    for start, end, label in annot['entities']:
        span = doc.char_span(start, end, label=label, alignment_mode='contract')
        if span is None:
            print(f'Skipping entity in text: {text[start:end]}')
        else:
            ents.append(span)
    doc.ents = ents
    db.add(doc)

# Save the processed training data
db.to_disk('./train_data1.spacy')


100%|██████████| 100/100 [00:00<00:00, 178.47it/s]


In [59]:
! python -m spacy train config.cfg --output ./output_person --paths.train ./train_data1.spacy --paths.dev ./dev.spacy --gpu-id 0

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ℹ Saving to output directory: output_person
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
2025-02-11 02:09:04.880282: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-11 02:09:04.887175: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739268544.895140   69833 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739268544.897521   69833 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-

In [28]:
import spacy

model = spacy.load('/home/balaji/POC/POC/EasyOCR-ChatBot/output_person/model-best')
model.get_pipe('ner').labels

/home/balaji/miniconda3/envs/Python/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self._model.load_sta

('PERSON',)

In [56]:
# Sample input text
text = "sdasdasd was admitted on 01/15/2024 and discharged on 01/20/2024."

# Process the text using the trained model
doc = model(text)

# Print detected entities
for ent in doc.ents:
    print(f"Entity: {ent.text}, Label: {ent.label_}")
